In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, pathlib

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
DATASETS = os.path.join(BASE, "Datasets")
SRC = os.path.join(BASE, "src")
PROCESSED = os.path.join(BASE, "processed_data")
RESULTS = os.path.join(BASE, "results")
EXPERIMENTS = os.path.join(BASE, "experiments")

for p in [DATASETS, SRC, PROCESSED, RESULTS, EXPERIMENTS]:
    pathlib.Path(p).mkdir(parents=True, exist_ok=True)

if SRC not in sys.path:
    sys.path.append(SRC)

print("BASE =", BASE)
print("Exists BASE:", os.path.exists(BASE))
print("Datasets folder:", DATASETS, "exists:", os.path.exists(DATASETS))
print("SRC folder:", SRC, "exists:", os.path.exists(SRC))
print("Datasets content:", os.listdir(DATASETS))
print("SRC content:", os.listdir(SRC))

In [ ]:
import os

nab_path = os.path.join(DATASETS, "nab")
print("NAB path:", nab_path)
print("Exists:", os.path.exists(nab_path))

if os.path.exists(nab_path):
    print("NAB top-level:", os.listdir(nab_path)[:50])

In [ ]:
import os

expected_data = os.path.join(nab_path, "data")
expected_labels = os.path.join(nab_path, "labels", "combined_labels.json")

print("Has data/:", os.path.exists(expected_data), expected_data)
print("Has labels/combined_labels.json:", os.path.exists(expected_labels), expected_labels)

In [ ]:
import os, zipfile, pathlib

nab_path = os.path.join(DATASETS, "nab")
zip_path = os.path.join(nab_path, "archive.zip")

print("ZIP exists:", os.path.exists(zip_path), zip_path)

# Create a folder to extract into (keeps things clean)
extract_dir = os.path.join(nab_path, "extracted")
pathlib.Path(extract_dir).mkdir(parents=True, exist_ok=True)

# Extract
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print("Extraction done.")
print("Top-level extracted content:", os.listdir(extract_dir)[:30])

In [ ]:
import os

extract_dir = os.path.join(DATASETS, "nab", "extracted")

# Find where "data" and "labels" live
candidates = []
for root, dirs, files in os.walk(extract_dir):
    if "data" in dirs and "labels" in dirs:
        candidates.append(root)

print("Candidate NAB roots found:")
for c in candidates[:10]:
    print(" -", c)

if not candidates:
    print("No root with both data/ and labels/ found yet. We'll inspect manually.")

In [ ]:
import os

extract_dir = os.path.join(DATASETS, "nab", "extracted")
print("Extract dir:", extract_dir)

# List first-level folders/files
print("\nLevel 1:")
for name in sorted(os.listdir(extract_dir))[:200]:
    print(" -", name)

# List second-level for directories only
print("\nLevel 2 (directories only):")
for name in sorted(os.listdir(extract_dir))[:200]:
    p = os.path.join(extract_dir, name)
    if os.path.isdir(p):
        children = sorted(os.listdir(p))[:50]
        print(f" - {name}/ -> {children}")

In [ ]:
import os

extract_dir = os.path.join(DATASETS, "nab", "extracted")

found_data_dirs = []
found_labels_dirs = []
found_json_files = []

for root, dirs, files in os.walk(extract_dir):
    if os.path.basename(root).lower() == "data":
        found_data_dirs.append(root)
    if os.path.basename(root).lower() == "labels":
        found_labels_dirs.append(root)
    for f in files:
        if f.lower().endswith(".json") or "label" in f.lower():
            found_json_files.append(os.path.join(root, f))

print("DATA dirs found (first 10):")
for p in found_data_dirs[:10]:
    print(" -", p)

print("\nLABELS dirs found (first 10):")
for p in found_labels_dirs[:10]:
    print(" -", p)

print("\nJSON/label-related files found (first 20):")
for p in found_json_files[:20]:
    print(" -", p)

In [ ]:
import os
import pandas as pd

extract_dir = os.path.join(DATASETS, "nab", "extracted")

# Pick one folder, e.g., realKnownCause
folder = os.path.join(extract_dir, "realKnownCause")

print("Files inside realKnownCause:")
print(os.listdir(folder)[:10])

# Pick first CSV
first_csv = [f for f in os.listdir(folder) if f.endswith(".csv")][0]
csv_path = os.path.join(folder, first_csv)

print("\nLoading:", csv_path)
df = pd.read_csv(csv_path)

print("\nColumns:")
print(df.columns)

print("\nHead:")
print(df.head())

In [ ]:
import os

extract_dir = os.path.join(DATASETS, "nab", "extracted")

csv_dirs = []
for root, dirs, files in os.walk(extract_dir):
    if any(f.lower().endswith(".csv") for f in files):
        csv_dirs.append(root)

print("Number of directories containing CSVs:", len(csv_dirs))
print("First 10 CSV directories:")
for d in csv_dirs[:10]:
    print(" -", d)

In [ ]:
import os
import pandas as pd

extract_dir = os.path.join(DATASETS, "nab", "extracted")

# Find first directory containing CSVs
csv_dir = None
for root, dirs, files in os.walk(extract_dir):
    if any(f.lower().endswith(".csv") for f in files):
        csv_dir = root
        break

print("Using CSV dir:", csv_dir)

csv_files = sorted([f for f in os.listdir(csv_dir) if f.lower().endswith(".csv")])
print("First 10 CSV files:", csv_files[:10])

csv_path = os.path.join(csv_dir, csv_files[0])
df = pd.read_csv(csv_path)

print("\nLoaded:", csv_path)
print("\nColumns:", list(df.columns))
print("\nHead:")
print(df.head())

In [ ]:
import os
import pathlib

nab_path = os.path.join(DATASETS, "nab_official")
pathlib.Path(nab_path).mkdir(parents=True, exist_ok=True)

!git clone https://github.com/numenta/NAB.git "{nab_path}"

In [ ]:
print(os.listdir(nab_path))

In [ ]:
print(os.path.exists(os.path.join(nab_path, "labels", "combined_labels.json")))

In [ ]:
nab_root = os.path.join(DATASETS, "nab_official")

In [ ]:
import os

nab_loader_code = r'''
import os
import json
import pandas as pd

class NABLoader:

    def __init__(self, nab_root: str):
        self.nab_root = nab_root
        self.data_dir = os.path.join(nab_root, "data")
        self.labels_path = os.path.join(nab_root, "labels", "combined_labels.json")

        if not os.path.exists(self.data_dir):
            raise FileNotFoundError(f"Expected NAB data directory at: {self.data_dir}")

        if not os.path.exists(self.labels_path):
            raise FileNotFoundError(f"Expected combined_labels.json at: {self.labels_path}")

    def list_series(self):
        series = []
        for root, _, files in os.walk(self.data_dir):
            for f in files:
                if f.lower().endswith(".csv"):
                    full = os.path.join(root, f)
                    rel = os.path.relpath(full, self.data_dir)
                    series.append(rel.replace("\\\\", "/"))
        series.sort()
        return series

    def _load_labels(self):
        with open(self.labels_path, "r") as f:
            return json.load(f)

    def load_series(self, rel_csv_path: str):
        csv_path = os.path.join(self.data_dir, rel_csv_path)
        if not os.path.exists(csv_path):
            raise FileNotFoundError(csv_path)

        df = pd.read_csv(csv_path)
        df = df.rename(columns={df.columns[0]: "timestamp", df.columns[1]: "value"})
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
        df["value"] = pd.to_numeric(df["value"], errors="coerce")

        df = df.dropna(subset=["timestamp", "value"]).sort_values("timestamp").reset_index(drop=True)

        labels = self._load_labels()

        key = rel_csv_path.replace("\\\\", "/")
        anomaly_timestamps = set()

        if key in labels:
            for ts in labels[key]:
                t = pd.to_datetime(ts, errors="coerce", utc=True)
                if pd.notna(t):
                    anomaly_timestamps.add(t)

        df["label"] = df["timestamp"].isin(anomaly_timestamps).astype(int)

        return df
'''

file_path = os.path.join(SRC, "nab_loader.py")

with open(file_path, "w") as f:
    f.write(nab_loader_code)

print("Created:", file_path)
print("SRC content:", os.listdir(SRC))

In [ ]:
from nab_loader import NABLoader
import os

nab_root = os.path.join(DATASETS, "nab_official")
nab = NABLoader(nab_root)

series = nab.list_series()
print("Total NAB series:", len(series))
print("First 5:", series[:5])

df = nab.load_series(series[0])

print("\nColumns:", df.columns.tolist())
print("\nLabel distribution:")
print(df["label"].value_counts())

In [ ]:
preprocessing_code = r'''
import pandas as pd
import numpy as np

class Preprocessor:

    def __init__(self, window_duration="1H", step_duration="30min"):
        self.window_duration = pd.Timedelta(window_duration)
        self.step_duration = pd.Timedelta(step_duration)

    def clean(self, df: pd.DataFrame):
        df = df.copy()
        df = df.dropna(subset=["timestamp", "value"])
        df = df.sort_values("timestamp").drop_duplicates(subset="timestamp")
        df = df.reset_index(drop=True)
        return df

    def create_windows(self, df: pd.DataFrame):
        df = self.clean(df)

        start_time = df["timestamp"].min()
        end_time = df["timestamp"].max()

        windows = []
        labels = []

        current_start = start_time

        while current_start + self.window_duration <= end_time:
            current_end = current_start + self.window_duration

            mask = (df["timestamp"] >= current_start) & (df["timestamp"] < current_end)
            window_df = df.loc[mask]

            if len(window_df) > 0:
                features = self.extract_features(window_df)
                window_label = int(window_df["label"].max())
                windows.append(features)
                labels.append(window_label)

            current_start += self.step_duration

        return np.array(windows), np.array(labels)

    def extract_features(self, window_df: pd.DataFrame):
        values = window_df["value"].values

        features = [
            np.mean(values),
            np.std(values),
            np.min(values),
            np.max(values),
            np.median(values),
            np.percentile(values, 25),
            np.percentile(values, 75),
        ]

        return features
'''

file_path = os.path.join(SRC, "preprocessing.py")

with open(file_path, "w") as f:
    f.write(preprocessing_code)

print("Created:", file_path)
print("SRC content:", os.listdir(SRC))

In [ ]:
from preprocessing import Preprocessor

pre = Preprocessor(window_duration="1H", step_duration="30min")

df = nab.load_series(series[0])

X, y = pre.create_windows(df)

print("Feature matrix shape:", X.shape)
print("Labels shape:", y.shape)
print("First 5 labels:", y[:5])

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
# Find first series that contains at least one anomaly window
for s in series:
    df_tmp = nab.load_series(s)
    if df_tmp["label"].sum() > 0:
        print("Found anomaly series:", s)
        break

In [ ]:
df = nab.load_series(s)

X, y = pre.create_windows(df)

print("Total windows:", len(y))
print("Number of anomaly windows:", y.sum())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
from preprocessing import Preprocessor

pre2 = Preprocessor(window_duration="15min", step_duration="5min")

df = nab.load_series(s)  # same anomaly series found
X2, y2 = pre2.create_windows(df)

print("Total windows:", len(y2))
print("Anomaly windows:", y2.sum())

In [ ]:
import os
from datetime import datetime

exp_dir = os.path.join(BASE, "experiments")
os.makedirs(exp_dir, exist_ok=True)

exp_name = "exp_001_nab_windowing_analysis.md"
exp_path = os.path.join(exp_dir, exp_name)

content = f"""
# Experiment 001 — NAB Windowing Analysis
Date: {datetime.now()}

## Dataset
NAB Official
Series: artificialWithAnomaly/art_daily_flatmiddle.csv

## Experiment 1
Window: 1 hour
Step: 30 minutes

Total windows: 670
Anomaly windows: 2

Observation:
Severe class imbalance. Almost no positive samples.

---

## Experiment 2
Window: 15 minutes
Step: 5 minutes

Total windows: 4029
Anomaly windows: 3

Observation:
Increasing resolution increases total windows but anomaly density remains extremely low.

---

## Preliminary Conclusion
NAB anomaly markers are sparse. Window-level labeling yields severe imbalance.
Alternative strategies required:
- Smaller windows
- Label padding
- Series selection
- Different evaluation protocol
"""

with open(exp_path, "w") as f:
    f.write(content)

print("Experiment log created at:", exp_path)

In [ ]:
import os
from datetime import datetime

exp_dir = os.path.join(BASE, "experiments")
os.makedirs(exp_dir, exist_ok=True)

exp_path = os.path.join(exp_dir, "EXP_001_NAB_Windowing_Sensitivity.md")

content = f"""
# EXPERIMENT 001
## NAB Windowing Sensitivity Analysis

Date: {datetime.now().strftime("%Y-%m-%d %H:%M")}

---

# 1. Objective

To evaluate the impact of window duration and step size on anomaly density
when converting timestamp-level anomaly markers into window-level labels
for supervised anomaly detection.

This experiment investigates whether window-based feature extraction
produces a sufficient number of positive samples for supervised learning.

---

# 2. Dataset

Dataset: Numenta Anomaly Benchmark (Official GitHub Version)

Series Used:
artificialWithAnomaly/art_daily_flatmiddle.csv

Structure:
- timestamp (UTC)
- value
- anomaly timestamps provided via combined_labels.json

---

# 3. Preprocessing Pipeline

## Cleaning
- Drop NaN timestamps and values
- Convert timestamps to UTC datetime
- Convert value to numeric
- Sort by timestamp
- Remove duplicate timestamps

## Feature Extraction (per window)
Extracted statistical features:
1. Mean
2. Standard deviation
3. Minimum
4. Maximum
5. Median
6. 25th percentile
7. 75th percentile

Window label definition:
A window is labeled as anomalous (1) if it contains at least one
timestamp listed in combined_labels.json.
Otherwise, label = 0.

---

# 4. Experiment 1.1 — Baseline Window Configuration

Window Duration: 1 hour
Step Size: 30 minutes

Results:
Total windows: 670
Anomaly windows: 2

Class imbalance ratio:
2 / 670 ≈ 0.003

Supervised baseline (Random Forest):
Class 1 recall = 0.00
Class 1 F1-score = 0.00

Observation:
Extreme class imbalance.
Supervised classifier predicts only the majority class.

---

# 5. Experiment 1.2 — Higher Temporal Resolution

Window Duration: 15 minutes
Step Size: 5 minutes

Results:
Total windows: 4029
Anomaly windows: 3

Class imbalance ratio:
3 / 4029 ≈ 0.0007

Observation:
Reducing window size increases the total number of windows but does not
meaningfully increase anomaly density.

---

# 6. Analysis

Key Findings:

1. NAB anomaly markers are extremely sparse relative to window-based labeling.
2. Window-based conversion from timestamp anomalies to window-level labels
   produces severe class imbalance.
3. Supervised baseline models become unstable under such imbalance.
4. Window refinement alone does not sufficiently increase anomaly representation.

Implication:
Additional strategies are required, such as:
- Label padding (temporal expansion of anomaly windows)
- Aggregation across multiple series
- Alternative evaluation protocols
- Cost-sensitive learning

---

# 7. Conclusion

Window-based feature extraction over NAB results in severe anomaly sparsity.
This structural property must be addressed before supervised baselines
can serve as meaningful calibration tools.

This experiment justifies the need for controlled label expansion
or dataset aggregation in subsequent experiments.

"""

with open(exp_path, "w") as f:
    f.write(content)

print("Detailed experiment log created at:")
print(exp_path)

In [ ]:
import os

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
DATASETS = os.path.join(BASE, "Datasets")
SRC = os.path.join(BASE, "src")

print("DATASETS:", DATASETS)
print("Exists:", os.path.exists(DATASETS))

In [ ]:
import os

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
DATASETS = os.path.join(BASE, "Datasets")
nab_data_root = os.path.join(DATASETS, "nab_official", "data")

selected_domains = [
    "artificialWithAnomaly",
    "realAWSCloudwatch",
    "realTraffic",
    "realAdExchange",
    "realKnownCause"
]

selected_files = []

for domain in selected_domains:
    domain_path = os.path.join(nab_data_root, domain)
    csv_files = [f for f in os.listdir(domain_path) if f.endswith(".csv")]

    file_lengths = []

    for f in csv_files:
        full_path = os.path.join(domain_path, f)
        with open(full_path, "r") as fp:
            n_rows = sum(1 for _ in fp) - 1
        file_lengths.append((f, n_rows))

    file_lengths.sort(key=lambda x: x[1])

    if len(file_lengths) >= 3:
        short = file_lengths[0][0]
        median = file_lengths[len(file_lengths)//2][0]
        long = file_lengths[-1][0]

        selected_files.extend([
            os.path.join(domain, short),
            os.path.join(domain, median),
            os.path.join(domain, long)
        ])

print("Selected files:")
for f in selected_files:
    print("-", f)

In [ ]:
from nab_loader import NABLoader
from preprocessing import Preprocessor
import os
import pandas as pd

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
DATASETS = os.path.join(BASE, "Datasets")
nab_root = os.path.join(DATASETS, "nab_official")

nab = NABLoader(nab_root)

In [ ]:
import pandas as pd

window_configs = [
    ("1h", "30min"),
    ("30min", "10min"),
    ("15min", "5min"),
    ("5min", "1min")
]

results = []

for series_path in selected_files:
    print(f"\nProcessing: {series_path}")

    df = nab.load_series(series_path)

    for win, step in window_configs:
        pre = Preprocessor(window_duration=win, step_duration=step)
        X, y = pre.create_windows(df)

        total = len(y)
        anomalies = int(y.sum())
        ratio = anomalies / total if total > 0 else 0

        results.append({
            "series": series_path,
            "window": win,
            "step": step,
            "total_windows": total,
            "anomaly_windows": anomalies,
            "ratio": ratio
        })

profile_df = pd.DataFrame(results)

print("\nProfiling Complete.")
profile_df.head()

In [ ]:
summary = profile_df.groupby(["window", "step"])["ratio"].describe()
summary

In [ ]:
domain_summary = profile_df.copy()
domain_summary["domain"] = domain_summary["series"].apply(lambda x: x.split("/")[0])

domain_summary = domain_summary.groupby(["domain", "window"])["ratio"].median()
domain_summary

In [ ]:
import os
from datetime import datetime

import pandas as pd

# ---- Paths ----
BASE = "/content/drive/MyDrive/Master/Thesis/Code"
EXP_DIR = os.path.join(BASE, "experiments")
RES_DIR = os.path.join(BASE, "results")

os.makedirs(EXP_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

# ---- Assumptions: these already exist in your runtime ----
# profile_df: DataFrame with columns [series, window, step, total_windows, anomaly_windows, ratio]
# selected_files: list of 15 series paths used
# selected_domains: list of domains used (or reconstruct from series)

assert "profile_df" in globals(), "profile_df not found. Run profiling first."
assert "selected_files" in globals(), "selected_files not found. Run file selection first."

# ---- Save CSV artifact ----
ts = datetime.now().strftime("%Y%m%d_%H%M")
csv_path = os.path.join(RES_DIR, f"EXP_002_NAB_profiling_table_{ts}.csv")
profile_df.to_csv(csv_path, index=False)

# ---- Summaries for the log ----
summary = profile_df.groupby(["window", "step"])["ratio"].describe()

df2 = profile_df.copy()
df2["domain"] = df2["series"].apply(lambda x: x.split("/")[0])
domain_median = df2.groupby(["domain", "window", "step"])["ratio"].median().reset_index()

# Pivot for readability
domain_pivot = domain_median.pivot_table(
    index="domain",
    columns=["window", "step"],
    values="ratio",
    aggfunc="median"
)

# ---- Write Markdown Log ----
md_path = os.path.join(EXP_DIR, f"EXP_002_NAB_CrossDomain_WindowProfiling_{ts}.md")

window_configs = sorted(profile_df[["window", "step"]].drop_duplicates().values.tolist())

log = []
log.append("# EXPERIMENT 002")
log.append("## NAB Cross-Domain Window-Level Anomaly Density Profiling")
log.append(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
log.append("")
log.append("---")
log.append("")
log.append("# 1. Objective")
log.append("To assess whether converting timestamp-level anomaly markers into window-level labels produces sufficient anomaly density for supervised learning.")
log.append("This experiment profiles anomaly-window ratios across multiple NAB domains and multiple (window, step) configurations.")
log.append("")
log.append("---")
log.append("")
log.append("# 2. Dataset")
log.append("Dataset: Numenta Anomaly Benchmark (Official GitHub Version)")
log.append("Data source: `data/<domain>/*.csv`")
log.append("Labels source: `labels/combined_labels.json`")
log.append("")
log.append("---")
log.append("")
log.append("# 3. File Selection Strategy (Non-arbitrary)")
log.append("Domains profiled (5): artificialWithAnomaly, realAWSCloudwatch, realTraffic, realAdExchange, realKnownCause.")
log.append("Within each domain, three series were selected using the **short/median/long** rule based on number of rows:")
log.append("- Shortest series (0th percentile)")
log.append("- Median-length series (50th percentile)")
log.append("- Longest series (100th percentile)")
log.append("")
log.append("Selected series (15 total):")
for s in selected_files:
    log.append(f"- `{s}`")
log.append("")
log.append("---")
log.append("")
log.append("# 4. Label Conversion Definition (Window-Level)")
log.append("A window is labeled as anomalous (1) **iff** it contains at least one anomaly timestamp from `combined_labels.json` for that series.")
log.append("Otherwise, label = 0.")
log.append("")
log.append("---")
log.append("")
log.append("# 5. Windowing Configurations Tested")
for w, st in window_configs:
    log.append(f"- Window: `{w}` | Step: `{st}`")
log.append("")
log.append("---")
log.append("")
log.append("# 6. Outputs Computed Per Series and Configuration")
log.append("- `total_windows`")
log.append("- `anomaly_windows`")
log.append("- `ratio = anomaly_windows / total_windows`")
log.append("")
log.append("Full profiling table saved as CSV:")
log.append(f"- `{csv_path}`")
log.append("")
log.append("---")
log.append("")
log.append("# 7. Aggregate Results (Ratio Distribution by Configuration)")
log.append("Summary statistics for anomaly-window ratio across the 15 series:")
log.append("")
log.append("```")
log.append(summary.to_string())
log.append("```")
log.append("")
log.append("---")
log.append("")
log.append("# 8. Domain-Level Median Ratios")
log.append("Median anomaly-window ratio per domain for each configuration:")
log.append("")
log.append("```")
log.append(domain_pivot.to_string())
log.append("```")
log.append("")
log.append("---")
log.append("")
log.append("# 9. Key Findings")
log.append("1) Across domains, anomaly-window ratios are consistently very low (often below 1%).")
log.append("2) Reducing window size increases the number of windows but does **not** necessarily increase anomaly-window density under the 'contains ≥ 1 anomaly timestamp' rule.")
log.append("3) This indicates that supervised learning using naive window labels may be structurally limited without additional label modeling (e.g., padding/expansion) or aggregation.")
log.append("")
log.append("---")
log.append("")
log.append("# 10. Next Decision to Confirm with Supervisor")
log.append("Based on these profiling results, the next step should be chosen among:")
log.append("- (A) Controlled label expansion (padding) experiment")
log.append("- (B) Aggregation across multiple series for supervised calibration")
log.append("- (C) Shift evaluation emphasis toward unsupervised detection + post-hoc explainability, using NAB labels mainly as timestamp references")
log.append("")

with open(md_path, "w") as f:
    f.write("\n".join(log))

print("EXP-002 log created:", md_path)
print("EXP-002 table saved:", csv_path)

In [ ]:
import sys
import os

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
SRC = os.path.join(BASE, "src")

if SRC not in sys.path:
    sys.path.append(SRC)

print("SRC added to path:", SRC)

In [ ]:
from nab_loader import NABLoader

In [ ]:
import os
from datetime import datetime
import pandas as pd
import numpy as np

from nab_loader import NABLoader

# -------------------
# Paths
# -------------------
BASE = "/content/drive/MyDrive/Master/Thesis/Code"
DATASETS = os.path.join(BASE, "Datasets")
RES_DIR = os.path.join(BASE, "results")
EXP_DIR = os.path.join(BASE, "experiments")
os.makedirs(RES_DIR, exist_ok=True)
os.makedirs(EXP_DIR, exist_ok=True)

nab_root = os.path.join(DATASETS, "nab_official")
nab = NABLoader(nab_root)

# -------------------
# Configs
# -------------------
window_configs = [
    ("1h", "30min"),
    ("30min", "10min"),
    ("15min", "5min"),
    ("5min", "1min")
]

# Convert duration strings to Timedelta safely
def td(s: str) -> pd.Timedelta:
    # Use lowercase units to avoid FutureWarning about 'H'
    s = s.replace("H", "h").replace("MIN", "min").replace("Min", "min")
    return pd.Timedelta(s)

# -------------------
# Lightweight counting (no feature extraction)
# -------------------
def count_windows_and_anomalies(df: pd.DataFrame, window: str, step: str):
    w = td(window)
    st = td(step)

    df = df.dropna(subset=["timestamp", "value"]).copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df = df.dropna(subset=["timestamp"]).sort_values("timestamp")

    if df.empty:
        return 0, 0

    # Ensure label exists
    if "label" not in df.columns:
        df["label"] = 0

    # Set timestamp index for slicing
    df = df.set_index("timestamp")

    t0 = df.index.min()
    t1 = df.index.max()

    total = 0
    anomaly = 0

    start = t0
    while start + w <= t1:
        end = start + w
        window_df = df.loc[start:end]

        if not window_df.empty:
            total += 1
            if window_df["label"].sum() > 0:
                anomaly += 1

        start = start + st

    return total, anomaly

# -------------------
# Run profiling
# -------------------
all_series = nab.list_series()
print("Total NAB series:", len(all_series))

results = []
for i, s in enumerate(all_series, 1):
    df = nab.load_series(s)  # includes timestamp,value,label

    domain = s.split("/")[0] if "/" in s else "unknown"

    for win, step in window_configs:
        total, anom = count_windows_and_anomalies(df, win, step)
        ratio = (anom / total) if total > 0 else 0.0

        results.append({
            "series": s,
            "domain": domain,
            "window": win,
            "step": step,
            "total_windows": total,
            "anomaly_windows": anom,
            "ratio": ratio
        })

    if i % 10 == 0:
        print(f"Processed {i}/{len(all_series)} series...")

full_df = pd.DataFrame(results)

# -------------------
# Save artifacts
# -------------------
ts = datetime.now().strftime("%Y%m%d_%H%M")
table_path = os.path.join(RES_DIR, f"EXP_003_full_nab_profiling_table_{ts}.csv")
full_df.to_csv(table_path, index=False)

cfg_summary = full_df.groupby(["window", "step"])["ratio"].describe()
cfg_path = os.path.join(RES_DIR, f"EXP_003_summary_by_config_{ts}.csv")
cfg_summary.to_csv(cfg_path)

dom_summary = full_df.groupby(["domain", "window", "step"])["ratio"].median().reset_index()
dom_path = os.path.join(RES_DIR, f"EXP_003_summary_by_domain_{ts}.csv")
dom_summary.to_csv(dom_path, index=False)

print("\nSaved:")
print("Full table:", table_path)
print("Config summary:", cfg_path)
print("Domain summary:", dom_path)

# Print quick headline
print("\nHeadline medians by config:")
print(full_df.groupby(["window","step"])["ratio"].median().sort_values(ascending=False))

In [ ]:
Headline medians by config:

In [ ]:
import os
from datetime import datetime

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
EXP_DIR = os.path.join(BASE, "experiments")
RES_DIR = os.path.join(BASE, "results")

os.makedirs(EXP_DIR, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M")
log_path = os.path.join(EXP_DIR, f"EXP_003_FullNAB_WindowProfiling_{ts}.md")

# ---- Summaries ----
config_summary = full_df.groupby(["window", "step"])["ratio"].describe()
domain_summary = full_df.groupby(["domain", "window", "step"])["ratio"].median()

headline_median = full_df.groupby(["window","step"])["ratio"].median().sort_values(ascending=False)

# ---- Write Log ----
log = []

log.append("# EXPERIMENT 003")
log.append("## Full NAB Window-Level Anomaly Density Profiling")
log.append(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
log.append("")
log.append("---")
log.append("")
log.append("# 1. Objective")
log.append("To evaluate anomaly-window density across the entire NAB dataset (58 series) under multiple window/step configurations.")
log.append("This experiment verifies whether sparsity observed in EXP-002 is dataset-wide or sampling-dependent.")
log.append("")
log.append("---")
log.append("")
log.append("# 2. Dataset")
log.append("Dataset: Numenta Anomaly Benchmark (Official GitHub Repository)")
log.append("Total series analyzed: 58")
log.append("Labels: combined_labels.json (timestamp-level anomalies)")
log.append("")
log.append("---")
log.append("")
log.append("# 3. Window Label Definition")
log.append("A window is labeled as anomalous (1) if it contains at least one anomaly timestamp.")
log.append("Otherwise, label = 0.")
log.append("")
log.append("---")
log.append("")
log.append("# 4. Window Configurations Tested")
log.append("- 1h window / 30min step")
log.append("- 30min window / 10min step")
log.append("- 15min window / 5min step")
log.append("- 5min window / 1min step")
log.append("")
log.append("---")
log.append("")
log.append("# 5. Headline Median Anomaly Ratios (All 58 Series)")
log.append("")
log.append("```")
log.append(headline_median.to_string())
log.append("```")
log.append("")
log.append("---")
log.append("")
log.append("# 6. Aggregate Statistics by Configuration")
log.append("")
log.append("```")
log.append(config_summary.to_string())
log.append("```")
log.append("")
log.append("---")
log.append("")
log.append("# 7. Domain-Level Median Ratios")
log.append("")
log.append("```")
log.append(domain_summary.to_string())
log.append("```")
log.append("")
log.append("---")
log.append("")
log.append("# 8. Key Findings")
log.append("1) Anomaly-window ratios are consistently very low across all configurations.")
log.append("2) Median anomaly density decreases as window size decreases.")
log.append("3) The sparsity is dataset-wide and not due to subset selection.")
log.append("4) Under naive window labeling, supervised classification faces extreme class imbalance (<1% positives in most cases).")
log.append("")
log.append("---")
log.append("")
log.append("# 9. Interpretation")
log.append("The observed sparsity is a structural consequence of converting timestamp-level anomalies into fixed windows.")
log.append("This indicates that supervised window-level learning may require additional label modeling (e.g., temporal padding) or alternative evaluation strategies.")
log.append("")
log.append("---")
log.append("")
log.append("# 10. Next Step Decision")
log.append("To be determined:")
log.append("- Controlled anomaly padding experiment")
log.append("- Restrict supervised training to higher-density subsets")
log.append("- Shift toward unsupervised detection + explainability framework")
log.append("")

with open(log_path, "w") as f:
    f.write("\n".join(log))

print("EXP-003 log created at:")
print(log_path)

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
BASE = "/content/drive/MyDrive/Master/Thesis/Code"
DATASETS = os.path.join(BASE, "Datasets")
SRC = os.path.join(BASE, "src")

print("BASE:", BASE)
print("DATASETS exists:", os.path.exists(DATASETS))
print("SRC exists:", os.path.exists(SRC))




In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

**Publication safety change:** A historical cell containing `!rm -rf /content/drive` was disabled. It was not necessary for the documented experiments and could delete files. The unmodified original is preserved privately.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
DATASETS = os.path.join(BASE, "Datasets")
SRC = os.path.join(BASE, "src")

if SRC not in sys.path:
    sys.path.append(SRC)

print("DATASETS exists:", os.path.exists(DATASETS))
print("SRC exists:", os.path.exists(SRC))

if os.path.exists(SRC):
    print("SRC files:", os.listdir(SRC))

In [ ]:
from nab_loader import NABLoader
from preprocessing import Preprocessor

print("Imports successful.")

In [ ]:
import sys
import os
import importlib

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
SRC = os.path.join(BASE, "src")

# put src at the FRONT of the import path
if SRC in sys.path:
    sys.path.remove(SRC)
sys.path.insert(0, SRC)

# refresh Python's module cache
importlib.invalidate_caches()

print("SRC:", SRC)
print("Exists:", os.path.exists(SRC))
print("sys.path[0]:", sys.path[0])
print("Files:", os.listdir(SRC))

In [ ]:
import sys
import os
import importlib

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
SRC = os.path.join(BASE, "src")

# put src at the FRONT of the import path
if SRC in sys.path:
    sys.path.remove(SRC)
sys.path.insert(0, SRC)

# refresh Python's module cache
importlib.invalidate_caches()

print("SRC:", SRC)
print("Exists:", os.path.exists(SRC))
print("sys.path[0]:", sys.path[0])
print("Files:", os.listdir(SRC))

In [ ]:
from nab_loader import NABLoader
from preprocessing import Preprocessor

print("Imports successful.")

In [ ]:
import numpy as np
import os

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
DATASETS = os.path.join(BASE, "Datasets")

nab_root = os.path.join(DATASETS, "nab_official")
nab = NABLoader(nab_root)

series_list = nab.list_series()
print("Total NAB series:", len(series_list))

pre = Preprocessor(window_duration="15min", step_duration="5min")

X_all = []
y_all = []
series_meta = []

for i, s in enumerate(series_list, 1):
    df = nab.load_series(s)
    X, y = pre.create_windows(df)

    if len(y) == 0:
        continue

    X_all.append(X)
    y_all.append(y)
    series_meta.extend([s] * len(y))

    if i % 10 == 0 or i == len(series_list):
        print(f"Processed {i}/{len(series_list)} series")

X_all = np.vstack(X_all)
y_all = np.concatenate(y_all)
series_meta = np.array(series_meta)

print("\nFinal pooled dataset:")
print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)

unique, counts = np.unique(y_all, return_counts=True)
print("\nClass distribution:")
for u, c in zip(unique, counts):
    print(f"Class {u}: {c}")

print("\nAnomaly ratio:", y_all.mean())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.3,
    random_state=42,
    stratify=y_all
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

print("\nTrain anomaly ratio:", y_train.mean())
print("Test anomaly ratio:", y_test.mean())

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

dt_model = DecisionTreeClassifier(
    max_depth=5,
    class_weight="balanced",
    random_state=42
)

dt_model.fit(X_train_scaled, y_train)

y_pred_dt = dt_model.predict(X_test_scaled)

print("=== Decision Tree Classification Report ===")
print(classification_report(y_test, y_pred_dt, digits=4))

print("=== Decision Tree Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_dt))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import pandas as pd
import os

dt_results = {
    "model": "Decision Tree",
    "precision": precision_score(y_test, y_pred_dt, zero_division=0),
    "recall": recall_score(y_test, y_pred_dt, zero_division=0),
    "f1": f1_score(y_test, y_pred_dt, zero_division=0),
    "accuracy": accuracy_score(y_test, y_pred_dt)
}

dt_results_df = pd.DataFrame([dt_results])

results_path = os.path.join(RESULTS, "EXP_005_decision_tree_results.csv")
dt_results_df.to_csv(results_path, index=False)

print(dt_results_df)
print("\nSaved to:", results_path)

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(
    kernel="rbf",
    class_weight="balanced",
    random_state=42
)

svm_model.fit(X_train_scaled, y_train)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
DATASETS = os.path.join(BASE, "Datasets")
SRC = os.path.join(BASE, "src")
RESULTS = os.path.join(BASE, "results")
EXPERIMENTS = os.path.join(BASE, "experiments")

os.makedirs(RESULTS, exist_ok=True)
os.makedirs(EXPERIMENTS, exist_ok=True)

if SRC in sys.path:
    sys.path.remove(SRC)
sys.path.insert(0, SRC)

print("BASE:", BASE)
print("DATASETS exists:", os.path.exists(DATASETS))
print("SRC exists:", os.path.exists(SRC))
print("RESULTS exists:", os.path.exists(RESULTS))
print("EXPERIMENTS exists:", os.path.exists(EXPERIMENTS))
print("SRC files:", os.listdir(SRC))

In [ ]:
import importlib

importlib.invalidate_caches()

from nab_loader import NABLoader
from preprocessing import Preprocessor

print("Imports successful.")

In [ ]:
nab_root = os.path.join(DATASETS, "nab_official")
nab = NABLoader(nab_root)

series_list = nab.list_series()

print("Total NAB series:", len(series_list))
print("First 10 series:")
for s in series_list[:10]:
    print("-", s)

In [ ]:
import os

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
DATASETS = os.path.join(BASE, "Datasets")

print("DATASETS path:", DATASETS)
print("DATASETS exists:", os.path.exists(DATASETS))

if os.path.exists(DATASETS):
    print("\nContents of Datasets:")
    print(os.listdir(DATASETS))

    for item in os.listdir(DATASETS):
        item_path = os.path.join(DATASETS, item)
        if os.path.isdir(item_path):
            print(f"\n--- {item} ---")
            try:
                print(os.listdir(item_path)[:20])
            except Exception as e:
                print("Could not read:", e)

In [ ]:
import os

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
DATASETS = os.path.join(BASE, "Datasets")

nab_root = os.path.join(DATASETS, "nab_official")
print("nab_root:", nab_root)
print("nab_root exists:", os.path.exists(nab_root))
print("data exists:", os.path.exists(os.path.join(nab_root, "data")))
print("labels exists:", os.path.exists(os.path.join(nab_root, "labels")))
print("combined_labels exists:", os.path.exists(os.path.join(nab_root, "labels", "combined_labels.json")))

In [ ]:
nab = NABLoader(nab_root)

series_list = nab.list_series()

print("Total NAB series:", len(series_list))
print("First 5 series:")
for s in series_list[:5]:
    print("-", s)

In [ ]:
import numpy as np

pre = Preprocessor(window_duration="15min", step_duration="5min")

X_all = []
y_all = []
series_meta = []

for i, s in enumerate(series_list, 1):
    df = nab.load_series(s)
    X, y = pre.create_windows(df)

    if len(y) == 0:
        continue

    X_all.append(X)
    y_all.append(y)
    series_meta.extend([s] * len(y))

    if i % 10 == 0 or i == len(series_list):
        print(f"Processed {i}/{len(series_list)} series")

X_all = np.vstack(X_all)
y_all = np.concatenate(y_all)
series_meta = np.array(series_meta)

print("\nFinal pooled dataset:")
print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)

unique, counts = np.unique(y_all, return_counts=True)
print("\nClass distribution:")
for u, c in zip(unique, counts):
    print(f"Class {u}: {c}")

print("\nAnomaly ratio:", y_all.mean())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import os

# -------------------------
# Train / Test Split
# -------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.3,
    random_state=42,
    stratify=y_all
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("Train anomaly ratio:", y_train.mean())
print("Test anomaly ratio:", y_test.mean())


# -------------------------
# Feature Scaling
# -------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nScaling complete")
print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)


# -------------------------
# Linear SVM Model
# -------------------------

svm_model = LinearSVC(
    class_weight="balanced",
    max_iter=10000,
    random_state=42
)

print("\nTraining Linear SVM...")

svm_model.fit(X_train_scaled, y_train)

print("Training complete.")


# -------------------------
# Predictions
# -------------------------

y_pred = svm_model.predict(X_test_scaled)

print("\nPrediction complete.")
print("Predicted anomalies:", (y_pred == 1).sum())


# -------------------------
# Evaluation
# -------------------------

print("\n=== Linear SVM Classification Report ===")
print(classification_report(y_test, y_pred, digits=4))

print("\n=== Confusion Matrix ===")
cm = confusion_matrix(y_test, y_pred)
print(cm)


# -------------------------
# Save results
# -------------------------

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

results = {
    "model": "Linear SVM",
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "accuracy": accuracy_score(y_test, y_pred)
}

results_df = pd.DataFrame([results])

save_path = os.path.join(RESULTS, "EXP_005_linear_svm_results.csv")
results_df.to_csv(save_path, index=False)

print("\nSaved results to:", save_path)
print(results_df)

In [ ]:
import os
from datetime import datetime
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

# ===== Paths =====
BASE = "/content/drive/MyDrive/Master/Thesis/Code"
EXPERIMENTS = os.path.join(BASE, "experiments")
os.makedirs(EXPERIMENTS, exist_ok=True)

# ===== Required variables from your notebook =====
# X_all, y_all
# X_train, X_test, y_train, y_test
# X_train_scaled, X_test_scaled
# y_pred_svm
# RESULTS

# ===== Metrics =====
total_windows = len(y_all)
feature_dim = X_all.shape[1]
normal_samples = int((y_all == 0).sum())
anomaly_samples = int((y_all == 1).sum())
anomaly_ratio = float(y_all.mean())

train_ratio = float(y_train.mean())
test_ratio = float(y_test.mean())

precision = precision_score(y_test, y_pred_svm, zero_division=0)
recall = recall_score(y_test, y_pred_svm, zero_division=0)
f1 = f1_score(y_test, y_pred_svm, zero_division=0)
accuracy = accuracy_score(y_test, y_pred_svm)

cm = confusion_matrix(y_test, y_pred_svm)
report = classification_report(y_test, y_pred_svm, digits=4)

true_negatives = int(cm[0, 0])
false_positives = int(cm[0, 1])
false_negatives = int(cm[1, 0])
true_positives = int(cm[1, 1])

today_str = datetime.now().strftime("%Y-%m-%d %H:%M")
filename_date = datetime.now().strftime("%Y%m%d_%H%M")

report_text = f"""# EXP-006
## Linear SVM Classification on Pooled NAB Dataset

### Date
{today_str}

---

# 1. Objective

To evaluate the performance of a Linear Support Vector Machine (SVM) classifier on the pooled NAB dataset after converting timestamp-level anomaly labels into window-level supervised classification samples.

The goal of this experiment is to assess whether a linear supervised classifier can effectively distinguish anomalous windows from normal behavior in highly imbalanced time-series anomaly datasets.

---

# 2. Dataset

Dataset: Numenta Anomaly Benchmark (NAB)

Configuration:
- Total time series used: 58
- Window size: 15 minutes
- Step size: 5 minutes

After windowing and pooling across all series:

| Metric | Value |
|---|---|
| Total windows | {total_windows:,} |
| Feature dimension | {feature_dim} |
| Normal samples | {normal_samples:,} |
| Anomaly samples | {anomaly_samples:,} |
| Anomaly ratio | {anomaly_ratio:.10f} |

This confirms that the dataset is extremely imbalanced, with approximately 1 anomaly per {round(1/anomaly_ratio):,} windows.

---

# 3. Experimental Setup

## Train/Test Split

- Training samples: {len(y_train):,}
- Testing samples: {len(y_test):,}

Stratified split was used to preserve the anomaly ratio.

Training anomaly ratio:
`{train_ratio:.10f}`

Test anomaly ratio:
`{test_ratio:.10f}`

## Feature Scaling

Features were standardized using StandardScaler.

- X_train_scaled shape: {X_train_scaled.shape}
- X_test_scaled shape: {X_test_scaled.shape}

## Model

Classifier: Linear Support Vector Machine

Implementation:
`LinearSVC`

Parameters:
- class_weight = "balanced"
- max_iter = 10000
- random_state = 42

The linear SVM variant was selected due to its scalability on large datasets compared to kernel-based SVM.

---

# 4. Results

## Classification Report


In [ ]:
import os
from datetime import datetime
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
EXPERIMENTS = os.path.join(BASE, "experiments")
os.makedirs(EXPERIMENTS, exist_ok=True)

# metrics
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
accuracy = accuracy_score(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, digits=4)

now = datetime.now().strftime("%Y-%m-%d %H:%M")

lines = []

lines.append("# EXP-006")
lines.append("## Linear SVM Classification on Pooled NAB Dataset\n")

lines.append(f"Date: {now}\n")

lines.append("## Dataset Summary\n")
lines.append(f"Total windows: {len(y_all)}")
lines.append(f"Features: {X_all.shape[1]}")
lines.append(f"Normal samples: {(y_all==0).sum()}")
lines.append(f"Anomaly samples: {(y_all==1).sum()}")
lines.append(f"Anomaly ratio: {y_all.mean()}\n")

lines.append("## Train/Test Split\n")
lines.append(f"Train shape: {X_train.shape}")
lines.append(f"Test shape: {X_test.shape}\n")

lines.append("## Model\n")
lines.append("Linear Support Vector Machine (LinearSVC)")
lines.append("Parameters:")
lines.append("class_weight = balanced")
lines.append("max_iter = 10000")
lines.append("random_state = 42\n")

lines.append("## Classification Report\n")
lines.append("```")
lines.append(report)
lines.append("```\n")

lines.append("## Confusion Matrix\n")
lines.append("```")
lines.append(str(cm))
lines.append("```\n")

lines.append("## Metrics\n")
lines.append(f"Precision: {precision}")
lines.append(f"Recall: {recall}")
lines.append(f"F1 Score: {f1}")
lines.append(f"Accuracy: {accuracy}\n")

lines.append("## Observation\n")
lines.append("Linear SVM struggles to detect anomalies due to the extreme class imbalance in the NAB dataset.")
lines.append("The classifier correctly models normal behavior but detects very few anomaly windows.")

report_text = "\n".join(lines)

filename = f"EXP_006_linear_svm_{datetime.now().strftime('%Y%m%d_%H%M')}.md"
path = os.path.join(EXPERIMENTS, filename)

with open(path, "w") as f:
    f.write(report_text)

print("Report saved to:")
print(path)

In [ ]:
!pip install xgboost

In [ ]:
import os
from datetime import datetime
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
EXPERIMENTS = os.path.join(BASE, "experiments")
os.makedirs(EXPERIMENTS, exist_ok=True)

precision = precision_score(y_test, y_pred_xgb, zero_division=0)
recall = recall_score(y_test, y_pred_xgb, zero_division=0)
f1 = f1_score(y_test, y_pred_xgb, zero_division=0)
accuracy = accuracy_score(y_test, y_pred_xgb)

cm = confusion_matrix(y_test, y_pred_xgb)
report = classification_report(y_test, y_pred_xgb, digits=4)

now = datetime.now().strftime("%Y-%m-%d %H:%M")

lines = []

lines.append("# EXP-007")
lines.append("## XGBoost Classification on Pooled NAB Dataset\n")

lines.append(f"Date: {now}\n")

lines.append("## Dataset Summary\n")
lines.append(f"Total windows: {len(y_all)}")
lines.append(f"Features: {X_all.shape[1]}")
lines.append(f"Normal samples: {(y_all==0).sum()}")
lines.append(f"Anomaly samples: {(y_all==1).sum()}")
lines.append(f"Anomaly ratio: {y_all.mean()}\n")

lines.append("## Train/Test Split\n")
lines.append(f"Train shape: {X_train.shape}")
lines.append(f"Test shape: {X_test.shape}\n")

lines.append("## Model\n")
lines.append("XGBoost Classifier")
lines.append("Parameters:")
lines.append("n_estimators = 200")
lines.append("max_depth = 6")
lines.append("learning_rate = 0.1")
lines.append(f"scale_pos_weight = {scale_pos_weight}")
lines.append("objective = binary:logistic")
lines.append("eval_metric = logloss")
lines.append("random_state = 42")
lines.append("n_jobs = -1\n")

lines.append("## Classification Report\n")
lines.append("```")
lines.append(report)
lines.append("```\n")

lines.append("## Confusion Matrix\n")
lines.append("```")
lines.append(str(cm))
lines.append("```\n")

lines.append("## Metrics\n")
lines.append(f"Precision: {precision}")
lines.append(f"Recall: {recall}")
lines.append(f"F1 Score: {f1}")
lines.append(f"Accuracy: {accuracy}\n")

lines.append("## Observation\n")
lines.append("XGBoost was evaluated as a nonlinear tree-ensemble baseline on the same pooled NAB dataset.")
lines.append("Its performance will be compared directly against Decision Tree and Linear SVM under the same preprocessing and split conditions.")

report_text = "\n".join(lines)

filename = f"EXP_007_xgboost_{datetime.now().strftime('%Y%m%d_%H%M')}.md"
path = os.path.join(EXPERIMENTS, filename)

with open(path, "w") as f:
    f.write(report_text)

print("Report saved to:")
print(path)

In [ ]:
y_pred_xgb = xgb_model.predict(X_test_scaled)

print("Prediction complete.")
print("Predicted anomalies:", (y_pred_xgb == 1).sum())

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import pandas as pd
import os

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

print("scale_pos_weight:", scale_pos_weight)

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

print("\nTraining XGBoost...")
xgb_model.fit(X_train_scaled, y_train)
print("Training complete.")

y_pred_xgb = xgb_model.predict(X_test_scaled)

print("\nPrediction complete.")
print("Predicted anomalies:", (y_pred_xgb == 1).sum())

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred_xgb, digits=4))

print("\n=== Confusion Matrix ===")
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
print(cm_xgb)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import pandas as pd
import os

xgb_results = {
    "model": "XGBoost",
    "precision": precision_score(y_test, y_pred_xgb, zero_division=0),
    "recall": recall_score(y_test, y_pred_xgb, zero_division=0),
    "f1": f1_score(y_test, y_pred_xgb, zero_division=0),
    "accuracy": accuracy_score(y_test, y_pred_xgb)
}

xgb_results_df = pd.DataFrame([xgb_results])

results_path = os.path.join(RESULTS, "EXP_007_xgboost_results.csv")
xgb_results_df.to_csv(results_path, index=False)

preds_path = os.path.join(RESULTS, "EXP_007_xgboost_predictions.csv")
pd.DataFrame({
    "y_true": y_test,
    "y_pred": y_pred_xgb
}).to_csv(preds_path, index=False)

print("Saved metrics to:", results_path)
print("Saved predictions to:", preds_path)
print(xgb_results_df)

In [ ]:
import os
from datetime import datetime
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
EXPERIMENTS = os.path.join(BASE, "experiments")
os.makedirs(EXPERIMENTS, exist_ok=True)

precision = precision_score(y_test, y_pred_xgb, zero_division=0)
recall = recall_score(y_test, y_pred_xgb, zero_division=0)
f1 = f1_score(y_test, y_pred_xgb, zero_division=0)
accuracy = accuracy_score(y_test, y_pred_xgb)

cm = confusion_matrix(y_test, y_pred_xgb)
report = classification_report(y_test, y_pred_xgb, digits=4)

now = datetime.now().strftime("%Y-%m-%d %H:%M")

lines = []
lines.append("# EXP-007")
lines.append("## XGBoost Classification on Pooled NAB Dataset\n")
lines.append(f"Date: {now}\n")

lines.append("## Dataset Summary\n")
lines.append(f"Total windows: {len(y_all)}")
lines.append(f"Features: {X_all.shape[1]}")
lines.append(f"Normal samples: {(y_all==0).sum()}")
lines.append(f"Anomaly samples: {(y_all==1).sum()}")
lines.append(f"Anomaly ratio: {y_all.mean()}\n")

lines.append("## Train/Test Split\n")
lines.append(f"Train shape: {X_train.shape}")
lines.append(f"Test shape: {X_test.shape}\n")

lines.append("## Model\n")
lines.append("XGBoost Classifier")
lines.append("Parameters:")
lines.append("n_estimators = 200")
lines.append("max_depth = 6")
lines.append("learning_rate = 0.1")
lines.append(f"scale_pos_weight = {scale_pos_weight}")
lines.append("objective = binary:logistic")
lines.append("eval_metric = logloss")
lines.append("random_state = 42")
lines.append("n_jobs = -1\n")

lines.append("## Classification Report\n")
lines.append("```")
lines.append(report)
lines.append("```\n")

lines.append("## Confusion Matrix\n")
lines.append("```")
lines.append(str(cm))
lines.append("```\n")

lines.append("## Metrics\n")
lines.append(f"Precision: {precision}")
lines.append(f"Recall: {recall}")
lines.append(f"F1 Score: {f1}")
lines.append(f"Accuracy: {accuracy}\n")

lines.append("## Observation\n")
lines.append("XGBoost achieved the best balance so far among the tested models.")
lines.append("Compared with Decision Tree and Linear SVM, XGBoost improved the overall F1-score while maintaining moderate recall.")
lines.append("However, precision remains very low, indicating that the underlying dataset sparsity still strongly limits supervised performance.")

report_text = "\n".join(lines)

filename = f"EXP_007_xgboost_{datetime.now().strftime('%Y%m%d_%H%M')}.md"
path = os.path.join(EXPERIMENTS, filename)

with open(path, "w") as f:
    f.write(report_text)

print("Report saved to:")
print(path)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import pandas as pd
import os

mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=50,
    random_state=42
)

print("Training MLP...")
mlp_model.fit(X_train_scaled, y_train)
print("Training complete.")

y_pred_mlp = mlp_model.predict(X_test_scaled)

print("\n=== MLP Classification Report ===")
print(classification_report(y_test, y_pred_mlp, digits=4))

print("\n=== MLP Confusion Matrix ===")
cm_mlp = confusion_matrix(y_test, y_pred_mlp)
print(cm_mlp)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import pandas as pd
import os

mlp_results = {
    "model": "MLP",
    "precision": precision_score(y_test, y_pred_mlp, zero_division=0),
    "recall": recall_score(y_test, y_pred_mlp, zero_division=0),
    "f1": f1_score(y_test, y_pred_mlp, zero_division=0),
    "accuracy": accuracy_score(y_test, y_pred_mlp)
}

mlp_results_df = pd.DataFrame([mlp_results])

results_path = os.path.join(RESULTS, "EXP_008_mlp_results.csv")
mlp_results_df.to_csv(results_path, index=False)

print("Saved metrics to:", results_path)
print(mlp_results_df)

In [ ]:
import os
from datetime import datetime
from sklearn.metrics import classification_report, confusion_matrix

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
EXPERIMENTS = os.path.join(BASE, "experiments")
os.makedirs(EXPERIMENTS, exist_ok=True)

report = classification_report(y_test, y_pred_mlp, digits=4)
cm = confusion_matrix(y_test, y_pred_mlp)

now = datetime.now().strftime("%Y-%m-%d %H:%M")

lines = []
lines.append("# EXP-008")
lines.append("## MLP Neural Network Classification\n")
lines.append(f"Date: {now}\n")

lines.append("## Model")
lines.append("Multilayer Perceptron (MLPClassifier)")
lines.append("Hidden layers: (64, 32)")
lines.append("Activation: ReLU")
lines.append("Solver: Adam")
lines.append("Max iterations: 50\n")

lines.append("## Classification Report")
lines.append("```")
lines.append(report)
lines.append("```\n")

lines.append("## Confusion Matrix")
lines.append("```")
lines.append(str(cm))
lines.append("```\n")

lines.append("## Observation")
lines.append("The MLP classifier collapsed to the majority class and predicted all samples as normal.")
lines.append("This behavior is expected in extremely imbalanced datasets where anomaly samples represent less than 0.1% of the data.")
lines.append("The experiment highlights the limitations of neural networks trained with standard loss functions under extreme class imbalance.")

report_text = "\n".join(lines)

filename = f"EXP_008_mlp_{datetime.now().strftime('%Y%m%d_%H%M')}.md"
path = os.path.join(EXPERIMENTS, filename)

with open(path, "w") as f:
    f.write(report_text)

print("Report saved to:")
print(path)

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    ["Decision Tree", 0.0039, 0.6574, 0.0077],
    ["Linear SVM", 0.0075, 0.0463, 0.0129],
    ["XGBoost", 0.0127, 0.4352, 0.0247],
    ["MLP", 0.0000, 0.0000, 0.0000]
], columns=["Model", "Precision", "Recall", "F1"])

print(comparison)

comparison.to_csv(os.path.join(RESULTS, "EXP_model_comparison.csv"), index=False)

In [ ]:
import os

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
DATASETS = os.path.join(BASE, "Datasets")
blued_path = os.path.join(DATASETS, "blued")

print("BLUED path:", blued_path)
print("Exists:", os.path.exists(blued_path))

if os.path.exists(blued_path):
    print("\nTop-level BLUED contents:")
    for item in os.listdir(blued_path):
        print("-", item)

In [ ]:
import os

for root, dirs, files in os.walk(blued_path):
    depth = root.replace(blued_path, "").count(os.sep)
    if depth <= 2:
        print(f"\n{root}")
        for f in files[:20]:
            print("  -", f)

In [ ]:
!apt-get install unrar -y

In [ ]:
import os

extract_path = "/content/blued_extracted"
os.makedirs(extract_path, exist_ok=True)

print("Extraction folder:", extract_path)

In [ ]:
!unrar x "/content/drive/MyDrive/Master/Thesis/Code/Datasets/blued/location_001_dataset_001-015.rar" /content/blued_extracted/

In [ ]:
import os

for root, dirs, files in os.walk("/content/blued_extracted"):
    print("\n", root)
    for f in files[:20]:
        print("  ", f)

In [ ]:
import os

blued_root = "/content/blued_extracted/location_001_dataset_001"

all_files = sorted(os.listdir(blued_root))

print("Total files:", len(all_files))

print("\nMAT files:")
for f in all_files:
    if f.endswith(".mat.bz2"):
        print(" -", f)

print("\nFirst 30 TXT files:")
count = 0
for f in all_files:
    if f.endswith(".txt.bz2"):
        print(" -", f)
        count += 1
        if count >= 30:
            break

In [ ]:
import os
import bz2
import shutil

blued_root = "/content/blued_extracted/location_001_dataset_001"
mat_bz2 = os.path.join(blued_root, "location_001_matlab_1.mat.bz2")
mat_out = os.path.join(blued_root, "location_001_matlab_1.mat")

with bz2.open(mat_bz2, "rb") as f_in, open(mat_out, "wb") as f_out:
    shutil.copyfileobj(f_in, f_out)

print("Decompressed to:", mat_out)
print("Exists:", os.path.exists(mat_out))

In [ ]:
from scipy.io import loadmat

mat_data = loadmat("/content/blued_extracted/location_001_dataset_001/location_001_matlab_1.mat")

print("MAT keys:")
for k in mat_data.keys():
    if not k.startswith("__"):
        print("-", k)

In [ ]:
data = mat_data["data"]

print("Type:", type(data))
print("Shape:", data.shape)

In [ ]:
data_struct = data[0,0]

print("Fields inside data:")
print(data_struct.dtype.names)

In [ ]:
events = data_struct["event"]

print("Event type:", type(events))
print("Event shape:", events.shape)

print(events[:10])

In [ ]:
events = data_struct["events"]

print("Events type:", type(events))
print("Events shape:", events.shape)
print(events[:10])

In [ ]:
Pa = data_struct["Pa"]
Pb = data_struct["Pb"]
t = data_struct["t"]

print("Pa shape:", Pa.shape)
print("Pb shape:", Pb.shape)
print("t shape:", t.shape)

print("\nFirst 5 values of Pa:")
print(Pa[:5])

print("\nFirst 5 values of Pb:")
print(Pb[:5])

print("\nFirst 5 values of t:")
print(t[:5])

In [ ]:
import numpy as np

# ----- basic metadata -----
fs = data_struct["fs"]
start_date = data_struct["startDate"]
start_time = data_struct["startTime"]
last = data_struct["last"]
wdw_size = data_struct["wdw_size"]
window_shift = data_struct["window_shift"]

print("fs:", fs)
print("startDate:", start_date)
print("startTime:", start_time)
print("last:", last)
print("wdw_size:", wdw_size)
print("window_shift:", window_shift)

# ----- flatten signal -----
Pa_1d = data_struct["Pa"].flatten()
Pb_1d = data_struct["Pb"].flatten()
power = Pa_1d + Pb_1d

print("\nPa_1d shape:", Pa_1d.shape)
print("Pb_1d shape:", Pb_1d.shape)
print("power shape:", power.shape)

# ----- flatten events struct -----
events_struct = data_struct["events"][0, 0]

print("\nEvent fields:", events_struct.dtype.names)

event_indices = events_struct["index"]
event_times = events_struct["t"]
event_apps = events_struct["appName"]
event_phases = events_struct["phase"]

print("\nRaw event_indices type:", type(event_indices), "shape:", np.shape(event_indices))
print("Raw event_times type:", type(event_times), "shape:", np.shape(event_times))
print("Raw event_apps type:", type(event_apps), "shape:", np.shape(event_apps))
print("Raw event_phases type:", type(event_phases), "shape:", np.shape(event_phases))

print("\nFirst 10 event indices:")
print(event_indices[:10])

print("\nFirst 10 event times:")
print(event_times[:10])

print("\nFirst 10 app names:")
print(event_apps[:10])

print("\nFirst 10 phases:")
print(event_phases[:10])

In [ ]:
plugs = events_struct["Plugs"]

print("Type of plugs:", type(plugs))
print("Shape of plugs:", np.shape(plugs))

In [ ]:
plugs_struct = plugs[0, 0]

print("Fields inside plugs:")
print(plugs_struct.dtype.names)

In [ ]:
for field in plugs_struct.dtype.names:
    value = plugs_struct[field]
    print(f"\nField: {field}")
    print("Type:", type(value))
    print("Shape:", np.shape(value))
    try:
        print("First few values:")
        print(value[:10])
    except:
        print(value)

In [ ]:
import numpy as np
import pandas as pd

# flatten signals
power = (data_struct["Pa"].flatten() + data_struct["Pb"].flatten())

n = len(power)

print("Signal length:", n)

# create label vector
labels = np.zeros(n)

# extract event indices
event_indices = plugs_struct["index"].flatten()

print("Event indices:", event_indices)

# mark anomaly positions
for idx in event_indices:
    if idx < n:
        labels[int(idx)] = 1

print("Number of labeled events:", labels.sum())

In [ ]:
fs = data_struct["fs"][0][0]

timestamps = np.arange(n) / fs

print("Sampling frequency:", fs)
print("First timestamps:", timestamps[:5])

In [ ]:
df_blued = pd.DataFrame({
    "timestamp": timestamps,
    "value": power,
    "label": labels
})

print(df_blued.head())
print("\nClass distribution:")
print(df_blued["label"].value_counts())

In [ ]:
processed_path = "/content/drive/MyDrive/Master/Thesis/Code/processed_data/blued_processed.csv"

df_blued.to_csv(processed_path, index=False)

print("Saved BLUED dataset to:")
print(processed_path)

In [ ]:
import numpy as np
import pandas as pd

WINDOW_SIZE = 400
STEP_SIZE = 200

values = df_blued["value"].values
labels = df_blued["label"].values

X = []
y = []

for start in range(0, len(values) - WINDOW_SIZE, STEP_SIZE):

    end = start + WINDOW_SIZE

    window = values[start:end]
    window_labels = labels[start:end]

    features = [
        window.mean(),
        window.std(),
        window.min(),
        window.max(),
        np.median(window),
        np.percentile(window, 25),
        np.percentile(window, 75)
    ]

    label = 1 if np.any(window_labels == 1) else 0

    X.append(features)
    y.append(label)

X = np.array(X)
y = np.array(y)

print("Windowed dataset shape:", X.shape)
print("Labels shape:", y.shape)

print("\nClass distribution:")
print(pd.Series(y).value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("Train anomaly ratio:", y_train.mean())
print("Test anomaly ratio:", y_test.mean())

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete")
print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

tree_model = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

print("Training Decision Tree...")
tree_model.fit(X_train_scaled, y_train)

y_pred_tree = tree_model.predict(X_test_scaled)

print("\n=== Decision Tree Report ===")
print(classification_report(y_test, y_pred_tree, digits=4))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_tree))

In [ ]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(
    class_weight="balanced",
    max_iter=10000,
    random_state=42
)

print("Training Linear SVM...")
svm_model.fit(X_train_scaled, y_train)

y_pred_svm = svm_model.predict(X_test_scaled)

print("\n=== Linear SVM Report ===")
print(classification_report(y_test, y_pred_svm, digits=4))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_svm))

In [ ]:
import xgboost as xgb

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print("scale_pos_weight:", scale_pos_weight)

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost...")
xgb_model.fit(X_train_scaled, y_train)

y_pred_xgb = xgb_model.predict(X_test_scaled)

print("\n=== XGBoost Report ===")
print(classification_report(y_test, y_pred_xgb, digits=4))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_xgb))

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=50,
    random_state=42
)

print("Training MLP...")
mlp_model.fit(X_train_scaled, y_train)

y_pred_mlp = mlp_model.predict(X_test_scaled)

print("\n=== MLP Report ===")
print(classification_report(y_test, y_pred_mlp, digits=4))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_mlp))

In [ ]:
import numpy as np
import pandas as pd

# raw arrays
values = df_blued["value"].values
labels = df_blued["label"].values

n = len(values)
split_idx = int(n * 0.7)

values_train = values[:split_idx]
labels_train = labels[:split_idx]

values_test = values[split_idx:]
labels_test = labels[split_idx:]

print("Raw train length:", len(values_train))
print("Raw test length:", len(values_test))
print("Train anomalies:", labels_train.sum())
print("Test anomalies:", labels_test.sum())

In [ ]:
WINDOW_SIZE = 400
STEP_SIZE = 200

def make_windows(values, labels, window_size=400, step_size=200):
    X, y = [], []
    for start in range(0, len(values) - window_size, step_size):
        end = start + window_size
        window = values[start:end]
        window_labels = labels[start:end]

        features = [
            window.mean(),
            window.std(),
            window.min(),
            window.max(),
            np.median(window),
            np.percentile(window, 25),
            np.percentile(window, 75)
        ]

        label = 1 if np.any(window_labels == 1) else 0

        X.append(features)
        y.append(label)

    return np.array(X), np.array(y)

X_train, y_train = make_windows(values_train, labels_train, WINDOW_SIZE, STEP_SIZE)
X_test, y_test = make_windows(values_test, labels_test, WINDOW_SIZE, STEP_SIZE)

print("Train windows:", X_train.shape, "Anomalies:", y_train.sum())
print("Test windows:", X_test.shape, "Anomalies:", y_test.sum())

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

tree_model = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

print("Training Decision Tree...")
tree_model.fit(X_train_scaled, y_train)

y_pred_tree = tree_model.predict(X_test_scaled)

print("\n=== Decision Tree ===")
print(classification_report(y_test, y_pred_tree, digits=4))
print(confusion_matrix(y_test, y_pred_tree))

In [ ]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(
    class_weight="balanced",
    max_iter=10000,
    random_state=42
)

print("Training Linear SVM...")
svm_model.fit(X_train_scaled, y_train)

y_pred_svm = svm_model.predict(X_test_scaled)

print("\n=== Linear SVM ===")
print(classification_report(y_test, y_pred_svm, digits=4))
print(confusion_matrix(y_test, y_pred_svm))

In [ ]:
import xgboost as xgb

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print("scale_pos_weight:", scale_pos_weight)

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost...")
xgb_model.fit(X_train_scaled, y_train)

y_pred_xgb = xgb_model.predict(X_test_scaled)

print("\n=== XGBoost ===")
print(classification_report(y_test, y_pred_xgb, digits=4))
print(confusion_matrix(y_test, y_pred_xgb))

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp_model = MLPClassifier(
    hidden_layer_sizes=(64,32),
    activation="relu",
    solver="adam",
    max_iter=50,
    random_state=42
)

print("Training MLP...")
mlp_model.fit(X_train_scaled, y_train)

y_pred_mlp = mlp_model.predict(X_test_scaled)

print("\n=== MLP ===")
print(classification_report(y_test, y_pred_mlp, digits=4))
print(confusion_matrix(y_test, y_pred_mlp))

In [ ]:
import numpy as np

anomaly_indices = np.where(y == 1)[0]

print("Anomaly window indices:")
print(anomaly_indices)

distances = np.diff(anomaly_indices)

print("\nDistances between anomaly windows:")
print(distances)

print("\nAverage distance:", distances.mean())

In [ ]:
import pandas as pd

results = {
    "Model": ["Decision Tree", "Linear SVM", "MLP", "XGBoost"],
    "Precision": [1.0, 1.0, 1.0, 1.0],
    "Recall": [0.75, 0.75, 0.75, 1.0],
    "F1": [0.857, 0.857, 0.857, 1.0]
}

df_results = pd.DataFrame(results)

path = "/content/drive/MyDrive/Master/Thesis/Code/experiment_logs/blued_results.csv"

df_results.to_csv(path, index=False)

print("Results saved to:")
print(path)

In [ ]:
import os

folder = "/content/drive/MyDrive/Master/Thesis/Code/experiment_logs"
os.makedirs(folder, exist_ok=True)

print("Folder ready:", folder)

In [ ]:
import os

log_dir = "/content/drive/MyDrive/Master/Thesis/Code/experiment_logs"

os.makedirs(log_dir, exist_ok=True)

print("Experiment log folder ready:")
print(log_dir)

In [ ]:
import os

log_dir = "/content/drive/MyDrive/Master/Thesis/Code/experiment_logs"

os.makedirs(log_dir, exist_ok=True)

print("Experiment log folder ready:")
print(log_dir)

In [ ]:
metadata = {
    "dataset": "BLUED",
    "sampling_frequency_hz": 12000,
    "window_size": 400,
    "window_step": 200,
    "feature_count": 7,
    "split_method": "chronological",
    "train_windows": 2084,
    "test_windows": 892,
    "train_anomalies": 8,
    "test_anomalies": 4,
    "anomaly_ratio_percent": 0.4
}

df_meta = pd.DataFrame([metadata])

meta_path = os.path.join(log_dir, "BLUED_experiment_metadata.csv")
df_meta.to_csv(meta_path, index=False)

print("Saved experiment metadata:")
print(meta_path)

In [ ]:
import numpy as np

anomaly_indices = np.where(y == 1)[0]
distances = np.diff(anomaly_indices)

df_anomaly = pd.DataFrame({
    "anomaly_index": anomaly_indices
})

df_distances = pd.DataFrame({
    "distance_between_anomalies": distances
})

anomaly_path = os.path.join(log_dir, "BLUED_anomaly_indices.csv")
distance_path = os.path.join(log_dir, "BLUED_anomaly_distances.csv")

df_anomaly.to_csv(anomaly_path, index=False)
df_distances.to_csv(distance_path, index=False)

print("Saved anomaly indices:")
print(anomaly_path)

print("Saved anomaly spacing:")
print(distance_path)

In [ ]:
shape_info = {
    "total_windows": len(y),
    "feature_dimension": X.shape[1],
    "anomaly_windows": int(y.sum()),
    "normal_windows": int((y==0).sum()),
    "average_anomaly_distance": float(distances.mean())
}

df_shape = pd.DataFrame([shape_info])

shape_path = os.path.join(log_dir, "BLUED_dataset_statistics.csv")
df_shape.to_csv(shape_path, index=False)

print("Saved dataset statistics:")
print(shape_path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BASE = "/content/drive/MyDrive/Master/Thesis/Code"

DATASETS = os.path.join(BASE, "Datasets")
PROCESSED = os.path.join(BASE, "processed_data")
LOGS = os.path.join(BASE, "experiment_logs")

os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(LOGS, exist_ok=True)

print("Datasets folder:", DATASETS)
print("Processed folder:", PROCESSED)
print("Logs folder:", LOGS)

In [ ]:
print("Available datasets:\n")

for item in os.listdir(DATASETS):
    print("-", item)

In [ ]:
ead_path = os.path.join(DATASETS, "ead")

print("EAD path:", ead_path)
print("Exists:", os.path.exists(ead_path))

In [ ]:
for root, dirs, files in os.walk(ead_path):
    depth = root.replace(ead_path, "").count(os.sep)

    if depth <= 2:
        print("\n", root)

        for f in files[:20]:
            print("   ", f)

In [ ]:
from collections import Counter

ext_counter = Counter()

for root, dirs, files in os.walk(ead_path):
    for f in files:
        ext = os.path.splitext(f)[1].lower()
        ext_counter[ext] += 1

print("File types found:\n")

for ext, count in ext_counter.items():
    print(ext if ext else "[no extension]", ":", count)

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/ead/energy_activity_dataset.zip"
extract_path = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/ead"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("EAD dataset extracted.")

In [ ]:
for root, dirs, files in os.walk(extract_path):
    depth = root.replace(extract_path, "").count(os.sep)

    if depth <= 2:
        print("\n", root)

        for f in files[:20]:
            print("   ", f)

In [ ]:
from collections import Counter

ext_counter = Counter()

for root, dirs, files in os.walk(extract_path):
    for f in files:
        ext = os.path.splitext(f)[1].lower()
        ext_counter[ext] += 1

print("File types found:\n")

for ext, count in ext_counter.items():
    print(ext if ext else "[no extension]", ":", count)

In [ ]:
!pip install bson

In [ ]:
import bson
import pandas as pd

bson_path = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/ead/energy_activity_dataset/energy_activity_dataset/energy_activity.bson"

data = []

with open(bson_path, "rb") as f:
    for item in bson.decode_file_iter(f):
        data.append(item)

print("Records loaded:", len(data))

In [ ]:
!pip install pymongo

In [ ]:
from bson import decode_file_iter
import pandas as pd

In [ ]:
bson_path = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/ead/energy_activity_dataset/energy_activity_dataset/energy_activity.bson"

data = []

with open(bson_path, "rb") as f:
    for item in decode_file_iter(f):
        data.append(item)

print("Records loaded:", len(data))

In [ ]:
df_ead = pd.DataFrame(data)

print("Dataset shape:", df_ead.shape)
df_ead.head()

In [ ]:
print(df_ead.columns)

In [ ]:
import numpy as np

df_ead["power"] = df_ead["activePowers"].apply(lambda x: np.mean(x))

In [ ]:
df_ead[["power"]].head()

In [ ]:
df_ead["label"] = df_ead["event"].notna().astype(int)

print(df_ead["label"].value_counts())

In [ ]:
df_ead["timestamp"] = np.arange(len(df_ead))

In [ ]:
df_ead_clean = df_ead[["timestamp", "power", "label"]]

print(df_ead_clean.head())
print(df_ead_clean.shape)

In [ ]:
processed_path = "/content/drive/MyDrive/Master/Thesis/Code/processed_data/ead_processed.csv"

df_ead_clean.to_csv(processed_path, index=False)

print("Saved EAD dataset to:")
print(processed_path)

In [ ]:
print("Class distribution:")
print(df_ead_clean["label"].value_counts())

In [ ]:
df_ead_clean["power_diff"] = df_ead_clean["power"].diff().abs()

threshold = df_ead_clean["power_diff"].mean() + 2 * df_ead_clean["power_diff"].std()

df_ead_clean["label"] = (df_ead_clean["power_diff"] > threshold).astype(int)

In [ ]:
print(df_ead_clean["label"].value_counts())

In [ ]:
WINDOW = 100
STEP = 50

In [ ]:
import numpy as np

X = []
y = []

power = df_ead_clean["power"].values
labels = df_ead_clean["label"].values

for start in range(0, len(power) - WINDOW, STEP):

    end = start + WINDOW

    window = power[start:end]
    window_label = labels[start:end]

    features = [
        np.mean(window),
        np.std(window),
        np.min(window),
        np.max(window),
        np.median(window),
        np.percentile(window, 25),
        np.percentile(window, 75)
    ]

    X.append(features)
    y.append(1 if np.any(window_label == 1) else 0)

X = np.array(X)
y = np.array(y)

print("Windowed dataset shape:", X.shape)
print("Labels shape:", y.shape)

import pandas as pd
print("\nClass distribution:")
print(pd.Series(y).value_counts())

In [ ]:
WINDOW = 50
STEP = 25

In [ ]:
import numpy as np
import pandas as pd

WINDOW = 50
STEP = 25

X = []
y = []

power = df_ead_clean["power"].values
labels = df_ead_clean["label"].values

for start in range(0, len(power) - WINDOW, STEP):

    end = start + WINDOW

    window = power[start:end]
    window_label = labels[start:end]

    features = [
        np.mean(window),
        np.std(window),
        np.min(window),
        np.max(window),
        np.median(window),
        np.percentile(window, 25),
        np.percentile(window, 75)
    ]

    X.append(features)

    # window label rule
    y.append(1 if np.any(window_label == 1) else 0)

X = np.array(X)
y = np.array(y)

print("Windowed dataset shape:", X.shape)
print("Labels shape:", y.shape)

print("\nClass distribution:")
print(pd.Series(y).value_counts())

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
import numpy as np

models = {
    "Decision Tree": DecisionTreeClassifier(),
    "Linear SVM": LinearSVC(max_iter=10000),
    "MLP": MLPClassifier(hidden_layer_sizes=(32,16), max_iter=2000),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss")
}

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

results = {}

for name, model in models.items():

    precisions = []
    recalls = []
    f1s = []

    for train_idx, test_idx in skf.split(X, y):

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model.fit(X_train, y_train)

        preds = model.predict(X_test)

        precisions.append(precision_score(y_test, preds, zero_division=0))
        recalls.append(recall_score(y_test, preds, zero_division=0))
        f1s.append(f1_score(y_test, preds, zero_division=0))

    results[name] = {
        "Precision": np.mean(precisions),
        "Recall": np.mean(recalls),
        "F1": np.mean(f1s)
    }

print("\n=== EAD Results ===")

for m, r in results.items():
    print(m, r)

In [ ]:
import pandas as pd

results_table = [
["NAB","Decision Tree",0.0039,0.6574,0.0077],
["NAB","Linear SVM",0.0075,0.0463,0.0129],
["NAB","XGBoost",0.0127,0.4352,0.0247],
["NAB","MLP",0.0,0.0,0.0],

["BLUED","Decision Tree",1.0,0.75,0.857],
["BLUED","Linear SVM",1.0,0.75,0.857],
["BLUED","XGBoost",1.0,1.0,1.0],
["BLUED","MLP",1.0,0.75,0.857],

["EAD","Decision Tree",0.333,0.333,0.333],
["EAD","Linear SVM",0.0,0.0,0.0],
["EAD","XGBoost",0.0,0.0,0.0],
["EAD","MLP",0.0,0.0,0.0]
]

df = pd.DataFrame(results_table,
columns=["Dataset","Model","Precision","Recall","F1"])

df.to_csv("/content/drive/MyDrive/Master/Thesis/Code/experiment_logs/final_results.csv",index=False)

print(df)

In [ ]:
import os

BASE = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official"

cmd = f"""
python {BASE}/scripts/combine_labels.py \
--dataDir {BASE}/data \
--labelDir {BASE}/labels/raw \
--combinedLabelsPath {BASE}/labels/combined_labels.json \
--combinedWindowsPath {BASE}/labels/combined_windows.json \
--skipConfirmation
"""

print(cmd)
os.system(cmd)

In [ ]:
import os

BASE = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official/labels"

print(os.listdir(BASE))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive"):
    if "nab_official" in dirs:
        print(os.path.join(root, "nab_official"))
        break

In [ ]:
BASE = "PASTE_PATH_HERE/labels"

print(os.listdir(BASE))

In [ ]:
import os

paths = []

for root, dirs, files in os.walk("/content/drive"):
    if "nab_official" in dirs:
        paths.append(os.path.join(root, "nab_official"))

paths

In [ ]:
BASE = "/content/drive/MyDrive/Shared with me/nab_official/labels"

print(os.listdir(BASE))

In [ ]:
BASE_DATASETS = "/content/drive/MyDrive/Master/Thesis/Code/Datasets"

In [ ]:
import os

BASE = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official/labels"

print(os.listdir(BASE))

In [ ]:
import os

BASE = "/content/drive/MyDrive/Master/Thesis/Code/Datasets"

print(os.listdir(BASE))

In [ ]:
BASE = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official/labels"

print(os.listdir(BASE))

In [ ]:
import os

BASE = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official"

cmd = f"""
python {BASE}/scripts/combine_labels.py \
--dataDir {BASE}/data \
--labelDir {BASE}/labels/raw \
--combinedLabelsPath {BASE}/labels/combined_labels.json \
--combinedWindowsPath {BASE}/labels/combined_windows.json \
--skipConfirmation
"""

print(cmd)
os.system(cmd)

In [ ]:
BASE = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official/labels"

print(os.listdir(BASE))

In [ ]:
import subprocess

BASE = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official"

cmd = [
    "python",
    f"{BASE}/scripts/combine_labels.py",
    "--dataDir", f"{BASE}/data",
    "--labelDir", f"{BASE}/labels/raw",
    "--combinedLabelsPath", f"{BASE}/labels/combined_labels.json",
    "--combinedWindowsPath", f"{BASE}/labels/combined_windows.json",
    "--skipConfirmation",
]

result = subprocess.run(cmd, capture_output=True, text=True)

print("RETURN CODE:", result.returncode)
print("\nSTDOUT:\n", result.stdout)
print("\nSTDERR:\n", result.stderr)

In [ ]:
import os
BASE = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official"
print(os.path.exists(f"{BASE}/scripts/combine_labels.py"))
print(os.path.exists(f"{BASE}/data"))
print(os.path.exists(f"{BASE}/labels/raw"))

In [ ]:
!python --version

In [ ]:
import os
import subprocess

BASE = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official"

cmd = [
    "python",
    "scripts/combine_labels.py",
    "--dataDir", "data",
    "--labelDir", "labels/raw",
    "--combinedLabelsPath", "labels/combined_labels.json",
    "--combinedWindowsPath", "labels/combined_windows.json",
    "--skipConfirmation",
]

result = subprocess.run(cmd, cwd=BASE, capture_output=True, text=True)

print("RETURN CODE:", result.returncode)
print("\nSTDOUT:\n", result.stdout)
print("\nSTDERR:\n", result.stderr)

In [ ]:
import os

LABELS_DIR = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official/labels"
print(os.listdir(LABELS_DIR))

In [ ]:
import os
import subprocess

BASE = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official"

env = os.environ.copy()
env["PYTHONPATH"] = BASE + ":" + env.get("PYTHONPATH", "")

cmd = [
    "python",
    "scripts/combine_labels.py",
    "--dataDir", "data",
    "--labelDir", "labels/raw",
    "--combinedLabelsPath", "labels/combined_labels.json",
    "--combinedWindowsPath", "labels/combined_windows.json",
    "--skipConfirmation",
]

result = subprocess.run(
    cmd,
    cwd=BASE,
    env=env,
    capture_output=True,
    text=True
)

print("RETURN CODE:", result.returncode)
print("\nSTDOUT:\n", result.stdout)
print("\nSTDERR:\n", result.stderr)

In [ ]:
import os

LABELS_DIR = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official/labels"
print(os.listdir(LABELS_DIR))

In [ ]:
%cd /content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official

!PYTHONPATH=. python scripts/combine_labels.py \
    --dataDir data \
    --labelDir labels/raw \
    --combinedLabelsPath labels/combined_labels.json \
    --combinedWindowsPath labels/combined_windows.json \
    --skipConfirmation

In [ ]:
%cd /content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official

!ls
!ls nab

In [ ]:
# 1. Go to Colab local storage
%cd /content

# 2. Clone a fresh copy of NAB
!git clone https://github.com/numenta/NAB.git

# 3. Enter the repo
%cd /content/NAB

# 4. Generate combined labels
!python scripts/combine_labels.py \
    --dataDir data \
    --labelDir labels/raw \
    --combinedLabelsPath labels/combined_labels.json \
    --combinedWindowsPath labels/combined_windows.json \
    --skipConfirmation

# 5. Copy outputs back to your Drive dataset folder
!cp labels/combined_labels.json /content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official/labels/
!cp labels/combined_windows.json /content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official/labels/

print("Files successfully generated and copied to Drive.")

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/Master/Thesis/Code/Datasets/nab_official/labels"))

In [ ]:
import os

BASE = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/blued"

print(os.listdir(BASE))

In [ ]:
import pandas as pd

df = pd.read_csv("PATH_TO_ONE_FILE")

df.head()
df.columns

In [ ]:
!apt-get install unrar -y

In [ ]:
import os

BLUED_DIR = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/blued"

for file in os.listdir(BLUED_DIR):
    if file.endswith(".rar"):
        rar_path = os.path.join(BLUED_DIR, file)

        print("Extracting:", file)

        !unrar x -o+ "$rar_path" "$BLUED_DIR"

In [ ]:
print(os.listdir(BLUED_DIR))

In [ ]:
import pandas as pd
import os

for root, dirs, files in os.walk(BLUED_DIR):
    for f in files:
        if f.endswith(".csv"):
            sample_file = os.path.join(root, f)
            break

print("Sample file:", sample_file)

df = pd.read_csv(sample_file)

df.head()
df.columns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BLUED_DIR = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/blued"

print(os.listdir(BLUED_DIR))

In [ ]:
!apt-get install unrar -y

In [ ]:
import os

BLUED_DIR = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/blued"

rar_files = [f for f in os.listdir(BLUED_DIR) if f.endswith(".rar")]

print("Total RAR files:", len(rar_files))

for file in rar_files:

    rar_path = os.path.join(BLUED_DIR, file)

    print("\nExtracting:", file)

    !unrar x -o+ "$rar_path" "$BLUED_DIR"

print("\nExtraction completed.")

In [ ]:
for root, dirs, files in os.walk(BLUED_DIR):
    print(root)
    print("files:", len(files))
    print()
    break

In [ ]:
mat_files = 0

for root, dirs, files in os.walk(BLUED_DIR):
    for f in files:
        if f.endswith(".mat"):
            mat_files += 1

print("Total .mat files:", mat_files)

In [ ]:
!apt-get install bzip2 -y

In [ ]:
import os

BLUED_DIR = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/blued"

bz2_files = []

for root, dirs, files in os.walk(BLUED_DIR):
    for f in files:
        if f.endswith(".bz2"):
            bz2_files.append(os.path.join(root, f))

print("Total bz2 files:", len(bz2_files))

In [ ]:
for f in bz2_files:

    print("Decompressing:", f)

    !bzip2 -dk "$f"

print("\nAll bz2 files decompressed.")

In [ ]:
mat_files = 0

for root, dirs, files in os.walk(BLUED_DIR):
    for f in files:
        if f.endswith(".mat"):
            mat_files += 1

print("Total .mat files:", mat_files)

In [ ]:
import os

BLUED_DIR = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/blued"

deleted = 0

for f in os.listdir(BLUED_DIR):
    if f.endswith(".rar"):
        os.remove(os.path.join(BLUED_DIR, f))
        deleted += 1

print("Deleted RAR files:", deleted)

In [ ]:
deleted = 0

for root, dirs, files in os.walk(BLUED_DIR):
    for f in files:
        if f.endswith(".bz2"):
            os.remove(os.path.join(root, f))
            deleted += 1

print("Deleted bz2 files:", deleted)

In [ ]:
import os

AMPDS2_DIR = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/ampds2"

print(os.listdir(AMPDS2_DIR))

In [ ]:
import zipfile
import os

for f in os.listdir(AMPDS2_DIR):
    if f.endswith(".zip"):
        zip_path = os.path.join(AMPDS2_DIR, f)
        print("Extracting:", zip_path)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(AMPDS2_DIR)

In [ ]:
csv_files = []

for root, dirs, files in os.walk(AMPDS2_DIR):
    for f in files:
        if f.endswith(".csv"):
            csv_files.append(os.path.join(root, f))

print("Total CSV files:", len(csv_files))
print(csv_files[:10])

In [ ]:
import pandas as pd

sample_file = csv_files[0]
print("Sample file:", sample_file)

df = pd.read_csv(sample_file)

print(df.head())
print(df.columns)
print(df.shape)

In [ ]:
electric_files = []

for f in csv_files:

    name = f.lower()

    if "electric" in name or "power" in name:
        electric_files.append(f)

print("Electricity-related files:", len(electric_files))

for f in electric_files:
    print(f)

In [ ]:
sample_file = electric_files[0]

print("Sample electricity file:")
print(sample_file)

df = pd.read_csv(sample_file)

print(df.head())
print(df.columns)
print(df.shape)

In [ ]:
import os

UKDALE_DIR = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/ukdale"

print(os.listdir(UKDALE_DIR))

In [ ]:
import h5py

file_path = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/ukdale/ukdale.h5"

with h5py.File(file_path, "r") as f:

    def print_structure(name, obj):
        print(name)

    f.visititems(print_structure)

In [ ]:
import os

UKDALE_DIR = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/ukdale"

print("Top-level:")
print(os.listdir(UKDALE_DIR))

print("\nInside ukdale.h5 directory:")
print(os.listdir(os.path.join(UKDALE_DIR, "ukdale.h5")))

In [ ]:
import os

for root, dirs, files in os.walk(UKDALE_DIR):
    for f in files:
        if f.endswith(".h5"):
            print(os.path.join(root, f))

In [ ]:
import h5py

file_path = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/ukdale/ukdale.h5/ukdale.h5"

with h5py.File(file_path, "r") as f:
    def print_structure(name, obj):
        print(name)
    f.visititems(print_structure)

In [ ]:
import h5py

file_path = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/ukdale/ukdale.h5/ukdale.h5"

with h5py.File(file_path, "r") as f:
    print(list(f.keys()))

In [ ]:
import pandas as pd

with h5py.File(file_path, "r") as f:

    data = f["building1"]["elec"]["meter2"]["table"][:]

df = pd.DataFrame(data)

print(df.head())
print(df.columns)
print(df.shape)

In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/Master/Thesis/Code/Datasets/ukdale/ukdale.h5/ukdale.h5"

df = pd.read_hdf(
    file_path,
    key="/building1/elec/meter2"
)

print(df.head())
print(df.columns)
print(df.shape)

In [ ]:
df = df.reset_index()

df["timestamp"] = pd.to_datetime(df["index"], unit="s")

df["diff"] = df["timestamp"].diff().dt.total_seconds()

print(df["diff"].value_counts().head())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE_PATH = "/content/drive/MyDrive/Master/Thesis/Code"
DATASETS_PATH = f"{BASE_PATH}/Datasets"
RESULTS_PATH = f"{BASE_PATH}/results"

print(BASE_PATH)
print(DATASETS_PATH)

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))
print(os.listdir("/content/drive/MyDrive/Master"))
print(os.listdir("/content/drive/MyDrive/Master/Thesis"))
print(os.listdir(BASE_PATH))
print(os.listdir(DATASETS_PATH))

In [ ]:
UKDALE_PATH = f"{DATASETS_PATH}/ukdale"
print(os.listdir(UKDALE_PATH))

In [ ]:
import pandas as pd

ukdale_file = f"{UKDALE_PATH}/ukdale.h5"

with pd.HDFStore(ukdale_file, mode="r") as store:
    print(store.keys())

In [ ]:
ukdale_file = f"{DATASETS_PATH}/ukdale/ukdale.h5/ukdale.h5"

print(ukdale_file)

In [ ]:
import os

print(os.path.isfile(ukdale_file))

In [ ]:
import pandas as pd

with pd.HDFStore(ukdale_file, mode="r") as store:
    print(store.keys())

In [ ]:
ukdale_file = f"{DATASETS_PATH}/ukdale/ukdale.h5/ukdale.h5"

with pd.HDFStore(ukdale_file, mode="r") as store:
    print(store.keys())

In [ ]:
mains_key = "/building5/elec/meter1"

df = pd.read_hdf(ukdale_file, key=mains_key)

df.head()

In [ ]:
df.info()

In [ ]:
df.head(10)

In [ ]:
df.index.to_series().diff().value_counts().head()

In [ ]:
mains_key = "/building5/elec/meter1"

df = pd.read_hdf(ukdale_file, key=mains_key)

df.head()

In [ ]:
df.info()

In [ ]:
df.index.to_series().diff().value_counts().head()

In [ ]:
# Make a working copy
ukdale_df = df.copy()

# Flatten the multi-index column to a simple name
ukdale_df.columns = ["power"]

# Check
print(ukdale_df.head())
print(ukdale_df.columns)

In [ ]:
print("Shape:", ukdale_df.shape)
print("\nMissing values:")
print(ukdale_df.isna().sum())

In [ ]:
ukdale_1min = ukdale_df.resample("1min").agg({
    "power": ["mean", "max", "std"]
})

# Flatten column names
ukdale_1min.columns = ["mean_power", "max_power", "std_power"]

# Drop rows that became fully or partly invalid after resampling
ukdale_1min = ukdale_1min.dropna()

# Preview
print(ukdale_1min.head())
print("\nShape:", ukdale_1min.shape)

In [ ]:
ukdale_1min.index.to_series().diff().value_counts().head()

In [ ]:
ukdale_1min = ukdale_1min.reset_index()

print(ukdale_1min.head())
print(ukdale_1min.info())

In [ ]:
import os

PH2_PATH = f"{BASE_PATH}/processed_data/PH2"
os.makedirs(PH2_PATH, exist_ok=True)

print(PH2_PATH)

In [ ]:
ukdale_output_path = f"{PH2_PATH}/ukdale_building5_meter1_1min.csv"
ukdale_1min.to_csv(ukdale_output_path, index=False)

print("Saved to:", ukdale_output_path)

In [ ]:
ukdale_1min = ukdale_1min.rename(columns={"index": "timestamp"})
print(ukdale_1min.head())

In [ ]:
ukdale_1min.columns = ["timestamp", "mean_power", "max_power", "std_power"]
print(ukdale_1min.head())

In [ ]:
print("Original rows:", len(df))
print("Resampled rows:", len(ukdale_1min))

print("\nDate range:")
print("Start:", ukdale_1min["index"].min() if "index" in ukdale_1min.columns else ukdale_1min.iloc[:,0].min())
print("End:", ukdale_1min["index"].max() if "index" in ukdale_1min.columns else ukdale_1min.iloc[:,0].max())

In [ ]:
print("Original rows:", len(df))
print("Resampled rows:", len(ukdale_1min))
print("Start:", ukdale_1min["timestamp"].min())
print("End:", ukdale_1min["timestamp"].max())

In [ ]:
import os
import pandas as pd

# Copy original loaded dataframe
ukdale_df = df.copy()

# Flatten column name
ukdale_df.columns = ["power"]

# Resample to 1-minute features
ukdale_1min = ukdale_df.resample("1min").agg({
    "power": ["mean", "max", "std"]
})

# Flatten column names
ukdale_1min.columns = ["mean_power", "max_power", "std_power"]

# Drop invalid rows
ukdale_1min = ukdale_1min.dropna()

# Reset index
ukdale_1min = ukdale_1min.reset_index()

# Rename time column
ukdale_1min.columns = ["timestamp", "mean_power", "max_power", "std_power"]

# Create output folder
PH2_PATH = f"{BASE_PATH}/processed_data/PH2"
os.makedirs(PH2_PATH, exist_ok=True)

# Save
ukdale_output_path = f"{PH2_PATH}/ukdale_building5_meter1_1min.csv"
ukdale_1min.to_csv(ukdale_output_path, index=False)

# Checks
print(ukdale_1min.head())
print("\nResampled shape:", ukdale_1min.shape)
print("\nResolution check:")
print(ukdale_1min["timestamp"].diff().value_counts().head())
print("\nSaved to:", ukdale_output_path)

In [ ]:
LOG_PATH = f"{BASE_PATH}/experiment_logs/PH2"

import os
os.makedirs(LOG_PATH, exist_ok=True)

print(LOG_PATH)

In [ ]:
log_text = """
PHASE 2 — TIME STANDARDIZATION

Dataset:
UK-DALE

Building:
building5

Meter:
meter1 (mains aggregate)

Original resolution:
~6 seconds (irregular sampling intervals observed: 5–13 seconds)

Target resolution:
1 minute

Resampling method:
Aggregation per 1-minute interval using statistical descriptors.

Extracted features per timestamp:
- mean_power
- max_power
- std_power

Rationale:
A unified 1-minute temporal resolution ensures comparability across datasets with heterogeneous sampling rates (UK-DALE, AMPds2, BLUED, NAB).

Using statistical aggregation preserves transient spikes and variability that are relevant for anomaly detection.

Original number of observations:
1,763,101

Number of observations after resampling:
189,979

Missing values:
Removed after aggregation.

Output file:
processed_data/PH2/ukdale_building5_meter1_1min.csv

Status:
completed
"""

log_file = f"{LOG_PATH}/PH2_UKDALE_standardization.md"

with open(log_file, "w") as f:
    f.write(log_text)

print("Log saved to:")
print(log_file)

In [ ]:
AMPDS2_PATH = f"{DATASETS_PATH}/ampds2"

import os
print(os.listdir(AMPDS2_PATH))

In [ ]:
ampds2_file = f"{AMPDS2_PATH}/YOUR_FILE.csv"
import pandas as pd

ampds2_df = pd.read_csv(ampds2_file)
ampds2_df.head()

In [ ]:
ampds2_files = sorted([f for f in os.listdir(AMPDS2_PATH) if f.lower().startswith("electricity")])
print(ampds2_files)

In [ ]:
import pandas as pd

for fname in ampds2_files:
    if fname.endswith(".csv") and "billing" not in fname.lower():
        path = f"{AMPDS2_PATH}/{fname}"
        try:
            temp = pd.read_csv(path, nrows=5)
            print("\nFILE:", fname)
            print(temp.head())
            print("COLUMNS:", list(temp.columns))
        except Exception as e:
            print("\nFILE:", fname, "-> ERROR:", e)

In [ ]:
bme_file = f"{AMPDS2_PATH}/Electricity_BME.csv"
bme_df = pd.read_csv(bme_file)

print(bme_df.head())
print("\nColumns:", list(bme_df.columns))
print("\nShape:", bme_df.shape)

In [ ]:
bme_df.info()

In [ ]:
ampds2_files = sorted([f for f in os.listdir(AMPDS2_PATH) if f.lower().startswith("electricity")])
print(ampds2_files)

In [ ]:
import pandas as pd

for fname in ampds2_files:
    if fname.endswith(".csv") and "billing" not in fname.lower():
        path = f"{AMPDS2_PATH}/{fname}"
        try:
            temp = pd.read_csv(path, nrows=5)
            print("\nFILE:", fname)
            print(temp.head())
            print("COLUMNS:", list(temp.columns))
        except Exception as e:
            print("\nFILE:", fname, "-> ERROR:", e)

In [ ]:
ampds2_file = f"{AMPDS2_PATH}/Electricity_P.csv"

import pandas as pd

df = pd.read_csv(ampds2_file)

print(df.head())
print("\nColumns:", list(df.columns))
print("\nShape:", df.shape)

df.info()

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

df = df.sort_values('timestamp')

print(
    df['timestamp']
    .diff()
    .value_counts()
    .head()
)

In [ ]:
print(df.columns)

In [ ]:
df['timestamp'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])

In [ ]:
import pandas as pd
import os

ampds2_file = f"{AMPDS2_PATH}/Electricity_P.csv"

ampds2_df = pd.read_csv(ampds2_file)

# Build timestamp from UNIX_TS
ampds2_df["timestamp"] = pd.to_datetime(ampds2_df["UNIX_TS"], unit="s")

# Select the chosen whole-house-like real power channel
ampds2_df = ampds2_df[["timestamp", "WHE"]].copy()

# Rename to unified schema
ampds2_df = ampds2_df.rename(columns={"WHE": "power"})

# Sort by time
ampds2_df = ampds2_df.sort_values("timestamp")

# Check resolution
print(ampds2_df.head())
print("\nResolution check:")
print(ampds2_df["timestamp"].diff().value_counts().head())
print("\nShape:", ampds2_df.shape)

In [ ]:
ampds2_std = ampds2_df.copy()

ampds2_std["mean_power"] = ampds2_std["power"]
ampds2_std["max_power"] = ampds2_std["power"]
ampds2_std["std_power"] = 0.0

ampds2_std = ampds2_std[["timestamp", "mean_power", "max_power", "std_power"]]

print(ampds2_std.head())
print("\nShape:", ampds2_std.shape)

In [ ]:
ampds2_output_path = f"{BASE_PATH}/processed_data/PH2/ampds2_WHE_1min.csv"

ampds2_std.to_csv(ampds2_output_path, index=False)

print("Saved to:", ampds2_output_path)

In [ ]:
import os
from datetime import datetime

LOG_DIR = f"{BASE_PATH}/experiment_logs/PH2"
os.makedirs(LOG_DIR, exist_ok=True)

log_text = f"""
# PHASE 2 — TIME STANDARDIZATION

## Dataset
AMPds2

## Source file
Electricity_P.csv

## Selected channel
WHE (Whole House Electricity - real power)

## Timestamp construction
timestamp = pd.to_datetime(UNIX_TS, unit='s')

## Original temporal resolution
1 minute

## Target temporal resolution
1 minute

## Processing steps
1. Loaded Electricity_P.csv
2. Converted UNIX_TS into datetime format
3. Selected WHE as representative whole-house electricity signal
4. Sorted observations chronologically
5. Standardized schema to match PH2 format used across datasets

## Output schema
timestamp
mean_power
max_power
std_power

## Feature construction logic
mean_power = power
max_power = power
std_power = 0

## Output dataset characteristics
Rows: 1,051,200
Columns: 4

## Observations
AMPds2 already provides measurements at 1-minute resolution.
Therefore, no temporal resampling was required.
Only schema harmonization was applied to ensure cross-dataset consistency.

## Output file
processed_data/PH2/ampds2_WHE_1min.csv

## Status
completed

## Generated on
{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""

log_file_path = f"{LOG_DIR}/PH2_AMPDS2_standardization.md"

with open(log_file_path, "w") as f:
    f.write(log_text)

print("Log saved to:")
print(log_file_path)

In [ ]:
NAB_PATH = f"{DATASETS_PATH}/nab/data"

nab_file = f"{NAB_PATH}/artificialWithAnomaly/art_daily_flatmiddle.csv"

import pandas as pd

nab_df = pd.read_csv(nab_file)

print(nab_df.head())
print("\nColumns:", nab_df.columns)
print("\nShape:", nab_df.shape)

nab_df.info()

In [ ]:
print(os.listdir(f"{DATASETS_PATH}/nab"))

In [ ]:
NAB_PATH = f"{DATASETS_PATH}/nab_official"

import os
print(os.listdir(NAB_PATH))

In [ ]:
print(os.listdir(f"{NAB_PATH}/data"))

In [ ]:
print(os.listdir(f"{NAB_PATH}/data/artificialWithAnomaly"))

In [ ]:
nab_file = f"{NAB_PATH}/data/artificialWithAnomaly/art_daily_flatmiddle.csv"

import pandas as pd

nab_df = pd.read_csv(nab_file)

print(nab_df.head())
print("\nColumns:", nab_df.columns)
print("\nShape:", nab_df.shape)

nab_df.info()

In [ ]:
nab_df["timestamp"] = pd.to_datetime(nab_df["timestamp"])

nab_df = nab_df.sort_values("timestamp")

print(nab_df["timestamp"].diff().value_counts().head())

In [ ]:
import json

labels_path = f"{NAB_PATH}/labels/combined_labels.json"

with open(labels_path, "r") as f:
    labels_json = json.load(f)

print(type(labels_json))
print(len(labels_json))

# check this specific file key
target_key = "artificialWithAnomaly/art_daily_flatmiddle.csv"

print("\nExists in labels:", target_key in labels_json)

# preview timestamps of anomalies
labels_json[target_key][:5]

In [ ]:
anomaly_times = pd.to_datetime(labels_json[target_key])

nab_df["label"] = nab_df["timestamp"].isin(anomaly_times).astype(int)

print(nab_df.head())
print("\nLabel counts:")
print(nab_df["label"].value_counts())
print("\nAnomaly rows:")
print(nab_df[nab_df["label"] == 1])

In [ ]:
nab_std = nab_df[["timestamp", "value", "label"]].copy()

print(nab_std.head())
print("\nShape:", nab_std.shape)
print("\nLabel distribution:")
print(nab_std["label"].value_counts())

nab_output_path = f"{BASE_PATH}/processed_data/PH2/nab_art_daily_flatmiddle.csv"

nab_std.to_csv(nab_output_path, index=False)

print("\nSaved to:", nab_output_path)

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime

# -----------------------------
# Paths
# -----------------------------
NAB_PATH = f"{DATASETS_PATH}/nab_official"
NAB_DATA_PATH = f"{NAB_PATH}/data"
NAB_LABELS_PATH = f"{NAB_PATH}/labels/combined_labels.json"

NAB_OUTPUT_BASE = f"{BASE_PATH}/processed_data/PH2/nab_all_series"
NAB_LOG_DIR = f"{BASE_PATH}/experiment_logs/PH2"

os.makedirs(NAB_OUTPUT_BASE, exist_ok=True)
os.makedirs(NAB_LOG_DIR, exist_ok=True)

# -----------------------------
# Load labels
# -----------------------------
with open(NAB_LABELS_PATH, "r") as f:
    labels_json = json.load(f)

# -----------------------------
# Collect all NAB csv series
# -----------------------------
nab_csv_files = []

for root, dirs, files in os.walk(NAB_DATA_PATH):
    for file in files:
        if file.lower().endswith(".csv"):
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, NAB_DATA_PATH).replace("\\", "/")
            nab_csv_files.append((full_path, rel_path))

nab_csv_files = sorted(nab_csv_files)

print(f"Total NAB CSV series found: {len(nab_csv_files)}")
print("First few series:")
for _, rel in nab_csv_files[:10]:
    print(" -", rel)

# -----------------------------
# Process all series
# -----------------------------
summary_rows = []
failed_files = []

for full_path, rel_path in nab_csv_files:
    try:
        df = pd.read_csv(full_path)

        # Basic validation
        if "timestamp" not in df.columns or "value" not in df.columns:
            raise ValueError(f"Unexpected columns: {list(df.columns)}")

        # Normalize timestamp
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.sort_values("timestamp").reset_index(drop=True)

        # Resolution check
        diffs = df["timestamp"].diff().value_counts()
        dominant_resolution = str(diffs.index[0]) if len(diffs) > 0 else "N/A"

        # Load anomaly labels for this exact series if present
        anomaly_times = pd.to_datetime(labels_json.get(rel_path, []))
        df["label"] = df["timestamp"].isin(anomaly_times).astype(int)

        # Standardized output
        std_df = df[["timestamp", "value", "label"]].copy()

        # Save mirrored folder structure
        out_rel_path = rel_path
        out_full_path = os.path.join(NAB_OUTPUT_BASE, out_rel_path)
        os.makedirs(os.path.dirname(out_full_path), exist_ok=True)
        std_df.to_csv(out_full_path, index=False)

        # Summary
        total_rows = len(std_df)
        anomaly_rows = int(std_df["label"].sum())
        normal_rows = total_rows - anomaly_rows
        anomaly_ratio = anomaly_rows / total_rows if total_rows > 0 else 0.0

        summary_rows.append({
            "series": rel_path,
            "rows": total_rows,
            "normal_rows": normal_rows,
            "anomaly_rows": anomaly_rows,
            "anomaly_ratio": anomaly_ratio,
            "dominant_resolution": dominant_resolution,
            "output_file": out_full_path
        })

    except Exception as e:
        failed_files.append({
            "series": rel_path,
            "error": str(e)
        })

# -----------------------------
# Summary dataframe
# -----------------------------
summary_df = pd.DataFrame(summary_rows).sort_values(
    by=["anomaly_rows", "series"],
    ascending=[False, True]
).reset_index(drop=True)

print("\nProcessing complete.")
print(f"Successful series: {len(summary_df)}")
print(f"Failed series: {len(failed_files)}")

print("\nTop rows from summary:")
print(summary_df.head(10))

# Save summary CSV
summary_csv_path = f"{NAB_OUTPUT_BASE}/nab_ph2_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

print("\nSummary CSV saved to:")
print(summary_csv_path)

# Save failed file report if needed
failed_csv_path = None
if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_csv_path = f"{NAB_OUTPUT_BASE}/nab_ph2_failed_files.csv"
    failed_df.to_csv(failed_csv_path, index=False)
    print("\nFailed files CSV saved to:")
    print(failed_csv_path)

# -----------------------------
# Build markdown log
# -----------------------------
total_series = len(summary_df)
total_failed = len(failed_files)
total_rows_all = int(summary_df["rows"].sum()) if total_series > 0 else 0
total_anomalies_all = int(summary_df["anomaly_rows"].sum()) if total_series > 0 else 0

if total_series > 0:
    min_ratio = summary_df["anomaly_ratio"].min()
    max_ratio = summary_df["anomaly_ratio"].max()
else:
    min_ratio = 0
    max_ratio = 0

top_preview = summary_df.head(15)

preview_lines = []
for _, row in top_preview.iterrows():
    preview_lines.append(
        f"| {row['series']} | {row['rows']} | {row['anomaly_rows']} | {row['anomaly_ratio']:.6f} | {row['dominant_resolution']} |"
    )

preview_table = "\n".join(preview_lines)

log_text = f"""# PH2_LOG_003
## Dataset Group: NAB (All Available Series)

Date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

---

# 1. Objective

Standardize all NAB time-series files into the PH2 format:

timestamp | value | label

to support consistent downstream processing, feature extraction, windowing, and benchmarking.

---

# 2. Source

Dataset: Numenta Anomaly Benchmark (Official Repository)

Base path:
nab_official/data/

Labels source:
nab_official/labels/combined_labels.json

---

# 3. Processing Pipeline Applied

For each NAB CSV series:

1. Loaded the CSV file
2. Verified presence of required columns:
   - timestamp
   - value
3. Converted timestamp to datetime
4. Sorted observations chronologically
5. Retrieved anomaly timestamps from combined_labels.json using the relative series path
6. Created binary label column:
   - label = 1 for anomaly timestamp
   - label = 0 otherwise
7. Exported standardized file in mirrored folder structure

Output schema:
timestamp | value | label

---

# 4. Processing Summary

Total NAB series processed successfully: {total_series}
Total NAB series failed: {total_failed}

Total observations across all successful series: {total_rows_all}
Total anomaly points across all successful series: {total_anomalies_all}

Anomaly ratio range across successful series:
min = {min_ratio:.6f}
max = {max_ratio:.6f}

Summary CSV:
{summary_csv_path}

{"Failed files CSV:\n" + failed_csv_path if failed_csv_path else "No failed files were encountered."}

---

# 5. Preview of Processed Series

| Series | Rows | Anomaly Rows | Anomaly Ratio | Dominant Resolution |
|---|---:|---:|---:|---|
{preview_table}

---

# 6. Observations

1. NAB series are structurally clean and straightforward to standardize.
2. Labels are externally stored and must be joined by exact relative path.
3. Label sparsity remains a major issue in many NAB series.
4. This confirms the need for careful window-label construction in Phase 3.
5. The standardized NAB outputs are now ready for:
   - window creation
   - anomaly density profiling
   - supervised and benchmark experiments

---

# 7. Output Location

Standardized NAB files saved under:
{NAB_OUTPUT_BASE}

---

# 8. Status

Completed
"""

nab_log_path = f"{NAB_LOG_DIR}/PH2_NAB_all_series_standardization.md"
with open(nab_log_path, "w", encoding="utf-8") as f:
    f.write(log_text)

print("\nNAB log saved to:")
print(nab_log_path)

In [ ]:
BLUED_PATH = f"{DATASETS_PATH}/blued"

import os

for root, dirs, files in os.walk(BLUED_PATH):
    level = root.replace(BLUED_PATH, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")

    subindent = ' ' * 2 * (level + 1)
    for f in files[:10]:
        print(f"{subindent}{f}")

In [ ]:
BLUED_PATH = f"{DATASETS_PATH}/blued"

sample_file = f"{BLUED_PATH}/location_001_dataset_001/location_001_ivdata_001.txt"

import pandas as pd

blued_sample = pd.read_csv(sample_file, sep="\t")

print(blued_sample.head())
print("\nColumns:")
print(blued_sample.columns)

print("\nShape:")
print(blued_sample.shape)

blued_sample.info()

In [ ]:
sample_file = f"{BLUED_PATH}/location_001_dataset_001/location_001_ivdata_001.txt"

# find where actual data starts
with open(sample_file, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines[:50]):
    print(i, line.strip())

In [ ]:
sample_file = f"{BLUED_PATH}/location_001_dataset_001/location_001_ivdata_001.txt"

blued_sample = pd.read_csv(sample_file, skiprows=22)

print(blued_sample.head())
print("\nColumns:")
print(blued_sample.columns)
print("\nShape:")
print(blued_sample.shape)
print()
print(blued_sample.info())

In [ ]:
blued_sample = blued_sample.drop(columns=["Comment"])

blued_sample["power"] = blued_sample["VoltageA"] * blued_sample["Current A"]

print(blued_sample[["X_Value", "Current A", "VoltageA", "power"]].head())
print("\nPower stats:")
print(blued_sample["power"].describe())

In [ ]:
# reconstruct relative timestamp inside segment
sampling_interval = 8.333333e-5  # seconds from header

blued_sample["timestamp"] = pd.to_timedelta(
    blued_sample["X_Value"],
    unit="s"
)

# convert to 1-second bins (much smaller than 1-minute but safe first)
blued_features = (
    blued_sample
    .set_index("timestamp")
    .resample("1S")
    .agg(
        mean_power=("power", "mean"),
        max_power=("power", "max"),
        std_power=("power", "std")
    )
    .dropna()
    .reset_index()
)

print(blued_features.head())
print("\nShape:", blued_features.shape)
print("\nStats:")
print(blued_features.describe())

In [ ]:
import os
import pandas as pd
from datetime import datetime

BLUED_PATH = f"{DATASETS_PATH}/blued"
BLUED_OUTPUT_BASE = f"{BASE_PATH}/processed_data/PH2/blued_all_series"

os.makedirs(BLUED_OUTPUT_BASE, exist_ok=True)

def process_blued_file(file_path):

    df = pd.read_csv(file_path, skiprows=22)

    df = df.drop(columns=["Comment"], errors="ignore")

    df["power"] = df["VoltageA"] * df["Current A"]

    df["timestamp"] = pd.to_timedelta(df["X_Value"], unit="s")

    features = (
        df.set_index("timestamp")
        .resample("1s")
        .agg(
            mean_power=("power", "mean"),
            max_power=("power", "max"),
            std_power=("power", "std")
        )
        .dropna()
        .reset_index()
    )

    return features


summary_rows = []

for root, dirs, files in os.walk(BLUED_PATH):

    for file in files:

        if file.endswith(".txt"):

            full_path = os.path.join(root, file)

            rel_path = os.path.relpath(full_path, BLUED_PATH)

            try:

                features = process_blued_file(full_path)

                output_path = os.path.join(
                    BLUED_OUTPUT_BASE,
                    rel_path.replace(".txt", ".csv")
                )

                os.makedirs(os.path.dirname(output_path), exist_ok=True)

                features.to_csv(output_path, index=False)

                summary_rows.append({

                    "series": rel_path,
                    "rows": len(features),
                    "resolution": "1 second",
                    "output_file": output_path
                })

            except Exception as e:

                summary_rows.append({

                    "series": rel_path,
                    "rows": 0,
                    "resolution": "FAILED",
                    "output_file": str(e)
                })


summary_df = pd.DataFrame(summary_rows)

summary_path = f"{BLUED_OUTPUT_BASE}/blued_ph2_summary.csv"

summary_df.to_csv(summary_path, index=False)

print("Processed files:", len(summary_df))
print("Summary saved to:")
print(summary_path)

print(summary_df.head())

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os

print(os.path.exists(BASE_PATH))
print(os.path.exists(f"{BASE_PATH}/processed_data"))
print(os.path.exists(f"{BASE_PATH}/processed_data/PH2"))

In [ ]:
BLUED_OUTPUT_BASE = f"{BASE_PATH}/processed_data/PH2/blued_all_series"

print("Exists:", os.path.exists(BLUED_OUTPUT_BASE))

if os.path.exists(BLUED_OUTPUT_BASE):
    for root, dirs, files in os.walk(BLUED_OUTPUT_BASE):
        print(root, len(files))
        break

In [ ]:
import os
import pandas as pd

BLUED_PATH = f"{DATASETS_PATH}/blued"
BLUED_OUTPUT_BASE = f"{BASE_PATH}/processed_data/PH2/blued_all_series"

os.makedirs(BLUED_OUTPUT_BASE, exist_ok=True)

def process_blued_file(file_path):
    df = pd.read_csv(file_path, skiprows=22)
    df = df.drop(columns=["Comment"], errors="ignore")
    df["power"] = df["VoltageA"] * df["Current A"]
    df["timestamp"] = pd.to_timedelta(df["X_Value"], unit="s")

    features = (
        df.set_index("timestamp")
        .resample("1s")
        .agg(
            mean_power=("power", "mean"),
            max_power=("power", "max"),
            std_power=("power", "std")
        )
        .dropna()
        .reset_index()
    )
    return features

summary_rows = []

for root, dirs, files in os.walk(BLUED_PATH):
    for file in files:
        if file.endswith(".txt"):
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, BLUED_PATH)
            output_path = os.path.join(
                BLUED_OUTPUT_BASE,
                rel_path.replace(".txt", ".csv")
            )

            os.makedirs(os.path.dirname(output_path), exist_ok=True)

            # skip if already saved
            if os.path.exists(output_path):
                summary_rows.append({
                    "series": rel_path,
                    "rows": -1,
                    "resolution": "1 second",
                    "status": "SKIPPED_EXISTS",
                    "output_file": output_path
                })
                continue

            try:
                features = process_blued_file(full_path)
                features.to_csv(output_path, index=False)

                summary_rows.append({
                    "series": rel_path,
                    "rows": len(features),
                    "resolution": "1 second",
                    "status": "OK",
                    "output_file": output_path
                })

            except Exception as e:
                summary_rows.append({
                    "series": rel_path,
                    "rows": 0,
                    "resolution": "FAILED",
                    "status": str(e),
                    "output_file": ""
                })

summary_df = pd.DataFrame(summary_rows)

summary_path = f"{BLUED_OUTPUT_BASE}/blued_ph2_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("Processed entries:", len(summary_df))
print("Summary saved to:", summary_path)
print(summary_df["status"].value_counts(dropna=False))
print(summary_df.head())

In [ ]:
import os
import pandas as pd

BLUED_PATH = f"{DATASETS_PATH}/blued"
LOCAL_BLUED_OUTPUT_BASE = "/content/blued_all_series"

os.makedirs(LOCAL_BLUED_OUTPUT_BASE, exist_ok=True)

def process_blued_file(file_path):
    df = pd.read_csv(file_path, skiprows=22)
    df = df.drop(columns=["Comment"], errors="ignore")
    df["power"] = df["VoltageA"] * df["Current A"]
    df["timestamp"] = pd.to_timedelta(df["X_Value"], unit="s")

    features = (
        df.set_index("timestamp")
        .resample("1s")
        .agg(
            mean_power=("power", "mean"),
            max_power=("power", "max"),
            std_power=("power", "std")
        )
        .dropna()
        .reset_index()
    )
    return features

summary_rows = []

for root, dirs, files in os.walk(BLUED_PATH):
    for file in files:
        if file.endswith(".txt"):
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, BLUED_PATH)
            output_path = os.path.join(
                LOCAL_BLUED_OUTPUT_BASE,
                rel_path.replace(".txt", ".csv")
            )

            os.makedirs(os.path.dirname(output_path), exist_ok=True)

            try:
                features = process_blued_file(full_path)
                features.to_csv(output_path, index=False)

                summary_rows.append({
                    "series": rel_path,
                    "rows": len(features),
                    "resolution": "1 second",
                    "status": "OK",
                    "output_file": output_path
                })

            except Exception as e:
                summary_rows.append({
                    "series": rel_path,
                    "rows": 0,
                    "resolution": "FAILED",
                    "status": str(e),
                    "output_file": ""
                })

summary_df = pd.DataFrame(summary_rows)

local_summary_path = f"{LOCAL_BLUED_OUTPUT_BASE}/blued_ph2_summary.csv"
summary_df.to_csv(local_summary_path, index=False)

print("Processed entries:", len(summary_df))
print("Local summary saved to:", local_summary_path)
print(summary_df["status"].value_counts(dropna=False))
print(summary_df.head())

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
BASE_PATH = "/content/drive/MyDrive/Master/Thesis/Code"
DATASETS_PATH = f"{BASE_PATH}/Datasets"

import os

print(os.path.exists(BASE_PATH))
print(os.path.exists(DATASETS_PATH))
print(os.path.exists(f"{BASE_PATH}/processed_data/PH2"))
print(os.path.exists(f"{BASE_PATH}/experiment_logs/PH2"))

In [ ]:
BLUED_DRIVE_PATH = f"{BASE_PATH}/processed_data/PH2/blued_all_series"

print("Exists on Drive:", os.path.exists(BLUED_DRIVE_PATH))

if os.path.exists(BLUED_DRIVE_PATH):
    total_files = 0
    for root, dirs, files in os.walk(BLUED_DRIVE_PATH):
        total_files += len(files)
    print("Files found on Drive:", total_files)

In [ ]:
LOCAL_BLUED_PATH = "/content/blued_all_series"

print("Exists locally:", os.path.exists(LOCAL_BLUED_PATH))

if os.path.exists(LOCAL_BLUED_PATH):
    total_local_files = 0
    for root, dirs, files in os.walk(LOCAL_BLUED_PATH):
        total_local_files += len(files)
    print("Local files found:", total_local_files)

In [ ]:
import os
from datetime import datetime

LOG_DIR = f"{BASE_PATH}/experiment_logs/PH2"
os.makedirs(LOG_DIR, exist_ok=True)

blued_log_text = f"""
# PHASE 2 — BLUED STANDARDIZATION

## Dataset
BLUED

## Processing mode
Full-dataset batch preprocessing

## Raw format
LabVIEW waveform text files (.txt)

## Parsing details
- Header metadata skipped using skiprows=22
- Main waveform columns identified:
  - X_Value
  - Current A
  - Current B
  - VoltageA
- Comment column dropped when empty

## Power construction
power = VoltageA * Current A

## Temporal aggregation
Raw waveform samples were aggregated to 1-second statistical features:
- mean_power
- max_power
- std_power

## Batch execution result
Processed entries: 6574

### Status breakdown
- OK: 6430
- FAILED: 144

### Failure categories observed
- Missing VoltageA column: 47
- UTF-8 decode errors: 64
- No columns to parse from file: 33

## Success rate
6430 / 6574 = 97.8%

## Interpretation
The BLUED preprocessing pipeline was successfully validated and executed at scale.
A small subset of files failed due to raw file corruption, encoding inconsistencies, or missing waveform channels.
This failure rate is acceptable for research preprocessing and does not invalidate the dataset.

## Notes
The processed outputs were initially generated in local Colab storage:
/content/blued_all_series

The runtime disconnected before confirming permanent copy to Google Drive.
Drive persistence of the processed files must therefore be verified separately.

## Status
BLUED preprocessing completed successfully at execution level.
Persistence check pending after runtime reconnection.

## Logged on
{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""

blued_log_path = f"{LOG_DIR}/PH2_BLUED_standardization.md"

with open(blued_log_path, "w", encoding="utf-8") as f:
    f.write(blued_log_text)

print("BLUED log saved to:")
print(blued_log_path)

In [ ]:
import shutil
import os

LOCAL_BLUED_PATH = "/content/blued_all_series"
BLUED_DRIVE_PATH = f"{BASE_PATH}/processed_data/PH2/blued_all_series"

os.makedirs(BLUED_DRIVE_PATH, exist_ok=True)

shutil.copytree(
    LOCAL_BLUED_PATH,
    BLUED_DRIVE_PATH,
    dirs_exist_ok=True
)

print("BLUED files copied to Drive")

In [ ]:
total_files = 0
for root, dirs, files in os.walk(BLUED_DRIVE_PATH):
    total_files += len(files)

print("Files now saved on Drive:", total_files)

In [ ]:
print(
    os.path.exists(
        f"{BLUED_DRIVE_PATH}/blued_ph2_summary.csv"
    )
)

In [ ]:
import os
import pandas as pd

UKDALE_PATH = f"{DATASETS_PATH}/ukdale_h5"

UKDALE_OUTPUT_BASE = f"{BASE_PATH}/processed_data/PH2/ukdale_all_series"
os.makedirs(UKDALE_OUTPUT_BASE, exist_ok=True)

summary_rows = []

for file in os.listdir(UKDALE_PATH):

    if file.endswith(".h5"):

        building_name = file.replace(".h5", "")
        building_path = os.path.join(UKDALE_PATH, file)

        try:

            df = pd.read_hdf(building_path)

            # identify power columns
            power_cols = [c for c in df.columns if "power" in c.lower()]

            for col in power_cols:

                series = df[col].dropna()

                features = (
                    series
                    .resample("1min")
                    .agg(["mean","max","std"])
                    .dropna()
                    .reset_index()
                )

                features.columns = [
                    "timestamp",
                    "mean_power",
                    "max_power",
                    "std_power"
                ]

                output_file = (
                    f"{UKDALE_OUTPUT_BASE}/"
                    f"{building_name}_{col}_1min.csv"
                )

                features.to_csv(output_file, index=False)

                summary_rows.append({
                    "series": f"{building_name}/{col}",
                    "rows": len(features),
                    "resolution": "1 minute",
                    "status": "OK",
                    "output_file": output_file
                })

        except Exception as e:

            summary_rows.append({
                "series": building_name,
                "rows": 0,
                "resolution": "FAILED",
                "status": str(e),
                "output_file": ""
            })

summary_df = pd.DataFrame(summary_rows)

summary_path = f"{UKDALE_OUTPUT_BASE}/ukdale_ph2_summary.csv"

summary_df.to_csv(summary_path, index=False)

print("Processed series:", len(summary_df))
print(summary_df["status"].value_counts(dropna=False))
print(summary_df.head())
print("Summary saved to:")
print(summary_path)

In [ ]:
UKDALE_FILE = f"{DATASETS_PATH}/ukdale/ukdale.h5/ukdale.h5"

import os
import pandas as pd

print("Exists:", os.path.isfile(UKDALE_FILE))

with pd.HDFStore(UKDALE_FILE, mode="r") as store:
    keys = store.keys()

print("Total keys:", len(keys))
print("First 20 keys:")
for k in keys[:20]:
    print(k)

In [ ]:
import os
import pandas as pd

UKDALE_FILE = f"{DATASETS_PATH}/ukdale/ukdale.h5/ukdale.h5"
UKDALE_OUTPUT_BASE = f"{BASE_PATH}/processed_data/PH2/ukdale_all_series"

os.makedirs(UKDALE_OUTPUT_BASE, exist_ok=True)

with pd.HDFStore(UKDALE_FILE, mode="r") as store:
    all_keys = store.keys()

meter_keys = [
    k for k in all_keys
    if "/elec/meter" in k
]

print("Total meter series found:", len(meter_keys))
print("First 10 meter keys:")
for k in meter_keys[:10]:
    print(" -", k)

summary_rows = []

for key in meter_keys:
    try:
        series_df = pd.read_hdf(UKDALE_FILE, key=key)

        # flatten single-column dataframe
        if isinstance(series_df, pd.Series):
            series_df = series_df.to_frame(name="power")
        else:
            if series_df.shape[1] == 1:
                series_df.columns = ["power"]
            else:
                # if multi-index/single logical column, keep first column
                series_df = series_df.iloc[:, [0]].copy()
                series_df.columns = ["power"]

        series_df = series_df.dropna()

        if not isinstance(series_df.index, pd.DatetimeIndex):
            raise ValueError("Index is not DatetimeIndex")

        features = (
            series_df
            .resample("1min")
            .agg(
                mean_power=("power", "mean"),
                max_power=("power", "max"),
                std_power=("power", "std")
            )
            .dropna()
            .reset_index()
        )

        # safe filename from key
        safe_name = key.strip("/").replace("/", "_")
        output_file = f"{UKDALE_OUTPUT_BASE}/{safe_name}_1min.csv"

        features.to_csv(output_file, index=False)

        summary_rows.append({
            "series": key,
            "rows": len(features),
            "resolution": "1 minute",
            "status": "OK",
            "output_file": output_file
        })

    except Exception as e:
        summary_rows.append({
            "series": key,
            "rows": 0,
            "resolution": "FAILED",
            "status": str(e),
            "output_file": ""
        })

summary_df = pd.DataFrame(summary_rows)

summary_path = f"{UKDALE_OUTPUT_BASE}/ukdale_ph2_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\nProcessed series:", len(summary_df))
print(summary_df["status"].value_counts(dropna=False))
print("\nSummary saved to:")
print(summary_path)
print("\nPreview:")
print(summary_df.head())

In [ ]:
import os
from datetime import datetime

LOG_DIR = f"{BASE_PATH}/experiment_logs/PH2"
os.makedirs(LOG_DIR, exist_ok=True)

ukdale_log_text = f"""
# PHASE 2 — UK-DALE STANDARDIZATION

## Dataset
UK-DALE

## Source format
Single HDF5 container with internal meter-series keys

## Source file
Datasets/ukdale/ukdale.h5/ukdale.h5

## Processing scope
Full batch preprocessing over all electricity meter series found in the HDF5 container.

## Internal key structure
Example:
- /building5/elec/meter1
- /building5/elec/meter10
- /building5/elec/meter11

## Total meter series processed
111

## Processing steps
1. Opened the UK-DALE HDF5 container
2. Enumerated all internal electricity meter keys
3. Loaded each meter time series individually
4. Dropped missing values
5. Resampled each series to 1-minute resolution
6. Computed standardized statistical features:
   - mean_power
   - max_power
   - std_power
7. Exported one CSV per meter series
8. Generated batch summary file

## Output schema
timestamp
mean_power
max_power
std_power

## Target temporal resolution
1 minute

## Batch result
- Total processed series: 111
- Successful series: 111
- Failed series: 0

## Output location
processed_data/PH2/ukdale_all_series/

## Summary file
processed_data/PH2/ukdale_all_series/ukdale_ph2_summary.csv

## Observations
The UK-DALE Phase 2 preprocessing completed successfully for all detected meter series.
This replaces the earlier single-series pilot preprocessing run on building5/meter1.
The dataset is now fully standardized and ready for downstream Phase 3 window construction.

## Status
completed

## Logged on
{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""

ukdale_log_path = f"{LOG_DIR}/PH2_UKDALE_standardization.md"

with open(ukdale_log_path, "w", encoding="utf-8") as f:
    f.write(ukdale_log_text)

print("UK-DALE log updated:")
print(ukdale_log_path)

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

AMPDS2_PATH = f"{DATASETS_PATH}/ampds2"
AMPDS2_OUTPUT_BASE = f"{BASE_PATH}/processed_data/PH2/ampds2_all_series"

os.makedirs(AMPDS2_OUTPUT_BASE, exist_ok=True)

electricity_files = [
    f for f in os.listdir(AMPDS2_PATH)
    if f.startswith("Electricity_") and f.endswith(".csv")
]

print("Electricity files found:", len(electricity_files))

summary_rows = []

for file in tqdm(electricity_files):

    file_path = os.path.join(AMPDS2_PATH, file)

    try:
        df = pd.read_csv(file_path)

        # find timestamp column
        time_col = [c for c in df.columns if "time" in c.lower()][0]

        df[time_col] = pd.to_datetime(df[time_col])

        df = df.set_index(time_col)

        # keep numeric columns only
        numeric_cols = df.select_dtypes(include="number").columns

        for col in numeric_cols:

            series = df[col].dropna()

            features = (
                series
                .resample("1min")
                .agg(
                    mean_power="mean",
                    max_power="max",
                    std_power="std"
                )
                .dropna()
                .reset_index()
            )

            features.columns = [
                "timestamp",
                "mean_power",
                "max_power",
                "std_power"
            ]

            output_file = (
                f"{AMPDS2_OUTPUT_BASE}/"
                f"{file.replace('.csv','')}_{col}_1min.csv"
            )

            features.to_csv(output_file, index=False)

            summary_rows.append({
                "series": f"{file}/{col}",
                "rows": len(features),
                "resolution": "1 minute",
                "status": "OK",
                "output_file": output_file
            })

    except Exception as e:

        summary_rows.append({
            "series": file,
            "rows": 0,
            "resolution": "FAILED",
            "status": str(e),
            "output_file": ""
        })

summary_df = pd.DataFrame(summary_rows)

summary_path = f"{AMPDS2_OUTPUT_BASE}/ampds2_ph2_summary.csv"

summary_df.to_csv(summary_path, index=False)

print("\nProcessed series:", len(summary_df))
print(summary_df["status"].value_counts(dropna=False))
print("\nSummary saved to:")
print(summary_path)

summary_df.head()

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

AMPDS2_PATH = f"{DATASETS_PATH}/ampds2"
AMPDS2_OUTPUT_BASE = f"{BASE_PATH}/processed_data/PH2/ampds2_all_series"

os.makedirs(AMPDS2_OUTPUT_BASE, exist_ok=True)

def find_time_column(df):

    possible_names = [
        "unix_ts",
        "datetime",
        "Date/Time",
        "date",
        "time",
        "timestamp"
    ]

    for name in possible_names:
        if name in df.columns:
            return name

    return None


electricity_files = [
    f for f in os.listdir(AMPDS2_PATH)
    if f.startswith("Electricity_")
    and f.endswith(".csv")
    and "Billing" not in f
    and "Monthly" not in f
]

print("Electricity time-series files:", len(electricity_files))

summary_rows = []

for file in tqdm(electricity_files):

    try:

        path = os.path.join(AMPDS2_PATH, file)

        df = pd.read_csv(path)

        time_col = find_time_column(df)

        if time_col is None:
            raise ValueError("No timestamp column found")

        df[time_col] = pd.to_datetime(df[time_col])

        df = df.set_index(time_col)

        numeric_cols = df.select_dtypes(include="number").columns

        for col in numeric_cols:

            series = df[col].dropna()

            features = (
                series
                .resample("1min")
                .agg(
                    mean_power="mean",
                    max_power="max",
                    std_power="std"
                )
                .dropna()
                .reset_index()
            )

            features.columns = [
                "timestamp",
                "mean_power",
                "max_power",
                "std_power"
            ]

            output_file = (
                f"{AMPDS2_OUTPUT_BASE}/"
                f"{file.replace('.csv','')}_{col}_1min.csv"
            )

            features.to_csv(output_file, index=False)

            summary_rows.append({
                "series": f"{file}/{col}",
                "rows": len(features),
                "resolution": "1 minute",
                "status": "OK",
                "output_file": output_file
            })

    except Exception as e:

        summary_rows.append({
            "series": file,
            "rows": 0,
            "resolution": "FAILED",
            "status": str(e),
            "output_file": ""
        })

summary_df = pd.DataFrame(summary_rows)

summary_path = f"{AMPDS2_OUTPUT_BASE}/ampds2_ph2_summary.csv"

summary_df.to_csv(summary_path, index=False)

print("\nProcessed series:", len(summary_df))
print(summary_df["status"].value_counts(dropna=False))
print("\nSummary saved to:")
print(summary_path)

summary_df.head()

In [ ]:
import os
from datetime import datetime

LOG_DIR = f"{BASE_PATH}/experiment_logs/PH2"
os.makedirs(LOG_DIR, exist_ok=True)

ampds2_log_text = f"""
# PHASE 2 — AMPds2 STANDARDIZATION

## Dataset
AMPds2 (Almanac of Minutely Power Dataset)

## Source format
CSV time-series files with 1-minute sampling resolution

## Processing scope
All electricity-related time-series files:

Electricity_*.csv

Excluded files:
- Billing summaries
- Monthly summaries
- files without valid timestamp column

## Total electricity files detected
25

## Total processed time-series
235

## Processing steps
1. Loaded each electricity CSV file
2. Detected timestamp column automatically
3. Converted timestamps to pandas datetime index
4. Selected numeric power-related columns
5. Standardized feature representation using 1-minute aggregation:
   - mean_power
   - max_power
   - std_power
6. Exported one CSV per electricity channel

## Output schema
timestamp
mean_power
max_power
std_power

## Temporal resolution
1 minute (native dataset resolution)

## Batch result
- Total series processed: 235
- Successful series: 231
- Skipped series: 4

## Reason for skipped series
Some files did not contain valid timestamp columns or contained aggregated billing data rather than time-series measurements.

## Output location
processed_data/PH2/ampds2_all_series/

## Summary file
processed_data/PH2/ampds2_all_series/ampds2_ph2_summary.csv

## Observations
AMPds2 is already sampled at 1-minute resolution, requiring minimal transformation.
The dataset integrates cleanly into the unified Phase 2 schema.

## Status
completed

## Logged on
{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""

ampds2_log_path = f"{LOG_DIR}/PH2_AMPDS2_standardization.md"

with open(ampds2_log_path, "w", encoding="utf-8") as f:
    f.write(ampds2_log_text)

print("AMPds2 log saved:")
print(ampds2_log_path)

In [ ]:
import os
from datetime import datetime

LOG_DIR = f"{BASE_PATH}/experiment_logs/PH2"
os.makedirs(LOG_DIR, exist_ok=True)

ph2_master_log = f"""
# PHASE 2 — MASTER SUMMARY
DATA STANDARDIZATION AND HARMONIZATION

## Objective
Convert heterogeneous raw energy datasets into a unified time-series format
suitable for downstream window construction and anomaly detection experiments.

All datasets were transformed into a consistent schema:

timestamp
mean_power
max_power
std_power

with harmonized temporal resolution of 1 minute where applicable.

---

# DATASETS PROCESSED

## 1. NAB (Numenta Anomaly Benchmark)

### Source format
CSV time-series files with timestamp-level anomaly labels.

### Processing steps
- parsed timestamp column
- mapped anomaly labels from combined_labels.json
- resampled to 1-minute resolution
- standardized schema

### Result
All series successfully processed.

### Output location
processed_data/PH2/nab_all_series/

---

## 2. BLUED

### Source format
LabVIEW waveform text files (.txt)
high-frequency voltage and current measurements.

### Processing steps
- skipped metadata header rows
- constructed power signal:

power = VoltageA × Current A

- aggregated waveform samples to 1-second statistics
- later compatible with 1-minute harmonization

### Result
6574 files processed
6430 successful
144 discarded due to:
- encoding inconsistencies
- missing waveform channels
- corrupted files

success rate ≈ 97.8%

### Output location
processed_data/PH2/blued_all_series/

---

## 3. UK-DALE

### Source format
single HDF5 container containing appliance-level electricity meters.

internal structure example:

/building5/elec/meter1
/building5/elec/meter2

### Processing steps
- enumerated all electricity meter keys
- extracted each appliance time series
- resampled to 1-minute resolution
- standardized statistical features

### Result
111 meter series processed
111 successful
0 failures

### Output location
processed_data/PH2/ukdale_all_series/

---

## 4. AMPds2

### Source format
CSV files with 1-minute electricity measurements.

### Processing steps
- detected timestamp column automatically
- extracted numeric electricity channels
- standardized statistical features
- retained native 1-minute resolution

### Result
235 time-series processed
231 successful
4 skipped (non-timeseries files)

### Output location
processed_data/PH2/ampds2_all_series/

---

# FINAL STANDARDIZED DATA SCHEMA

timestamp
mean_power
max_power
std_power

This schema is consistent across all datasets and enables:

- comparable feature distributions
- unified window construction
- cross-dataset anomaly detection experiments

---

# PHASE 2 STATUS

completed

All selected datasets have been successfully standardized.

No further preprocessing required.

The data is now ready for Phase 3:

WINDOW CONSTRUCTION AND LABEL ALIGNMENT

---

# Logged on
{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""

ph2_master_path = f"{LOG_DIR}/PH2_MASTER_LOG.md"

with open(ph2_master_path, "w", encoding="utf-8") as f:
    f.write(ph2_master_log)

print("PHASE 2 MASTER LOG SAVED:")
print(ph2_master_path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/Master/Thesis/Code"
print("Exists:", os.path.exists(BASE_PATH))
print("Contents:", os.listdir(BASE_PATH)[:20] if os.path.exists(BASE_PATH) else "Path not found")

In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/Master/Thesis/Code"

PH2_PATH = os.path.join(BASE_PATH, "processed_data", "PH2")
PH3_PATH = os.path.join(BASE_PATH, "processed_data", "PH3")

DATASETS = ["NAB", "BLUED", "UK_DALE", "AMPds2"]

print("PH2 path:", PH2_PATH)
print("PH3 path:", PH3_PATH)
print("PH2 exists:", os.path.exists(PH2_PATH))

In [ ]:
folders = [
    os.path.join(PH3_PATH, "logs"),
    os.path.join(PH3_PATH, "configs"),
    os.path.join(PH3_PATH, "combined"),

    os.path.join(PH3_PATH, "NAB", "raw_windows", "sequence"),
    os.path.join(PH3_PATH, "NAB", "raw_windows", "features"),
    os.path.join(PH3_PATH, "NAB", "padded_windows", "pad_5min"),
    os.path.join(PH3_PATH, "NAB", "padded_windows", "pad_10min"),
    os.path.join(PH3_PATH, "NAB", "padded_windows", "pad_15min"),
    os.path.join(PH3_PATH, "NAB", "balanced"),
    os.path.join(PH3_PATH, "NAB", "summaries"),

    os.path.join(PH3_PATH, "BLUED", "raw_windows", "sequence"),
    os.path.join(PH3_PATH, "BLUED", "raw_windows", "features"),
    os.path.join(PH3_PATH, "BLUED", "balanced"),
    os.path.join(PH3_PATH, "BLUED", "summaries"),

    os.path.join(PH3_PATH, "UK_DALE", "windows", "sequence"),
    os.path.join(PH3_PATH, "UK_DALE", "windows", "features"),
    os.path.join(PH3_PATH, "UK_DALE", "summaries"),

    os.path.join(PH3_PATH, "AMPds2", "windows", "sequence"),
    os.path.join(PH3_PATH, "AMPds2", "windows", "features"),
    os.path.join(PH3_PATH, "AMPds2", "summaries"),
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Phase 3 folder structure created successfully.")

In [ ]:
for root, dirs, files in os.walk(PH3_PATH):
    level = root.replace(PH3_PATH, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level >= 2:
        continue

In [ ]:
log_text = """# PH3_LOG_001: Window Design and Feature Engineering Strategy

**Context:** Phase 3 Methodological Baseline
**Project:** Explainable Anomaly Detection in Smart Home Energy Systems

## 1. Window Architecture
The pipeline utilizes a sliding window approach designed to balance contextual capture of energy patterns with computational efficiency.

* **Window Length:** 60 minutes
* **Step Size:** 30 minutes
* **Overlap:** 50%
* **Representations:** Dual-format extraction per window
    * *Sequence Windows:* Raw time-series segments for deep learning baseline models.
    * *Feature Windows:* Extracted statistical aggregations for classical ML and downstream explainability tools (SHAP/LIME).

## 2. Dataset-Specific Labeling & Processing
Handling varying anomaly definitions across the multi-dataset setup.

* **NAB:** A window is labeled anomalous if any ground-truth anomaly point or range falls within its boundaries.
    * *Padding Strategy:* Optional ±5, ±10, and ±15 minute temporal padding to address label sparsity. The effect of this temporal label expansion on anomaly window density will be quantified systematically in EXP-004.
* **BLUED:** Labeled via event-window proxy.
* **UK-DALE & AMPds2:** Retained as unlabeled windows for unsupervised downstream experiments.

## 3. Data Splitting Rules
Due to the 50% window overlap, random data shuffling introduces severe data leakage risks where training and testing sets share adjacent window data.

* **Mandatory Splitting Strategy:** Strictly chronological.
* **Rule:** The earlier portion of the timeline is reserved exclusively for training, and the later portion is reserved exclusively for testing. Shuffling is strictly prohibited prior to the split.

## 4. Feature Engineering Baseline
To maintain clean explanations in the later SHAP/LIME phases, the initial feature set is kept compact.

**Core Features per Signal (Mean Power, Max Power, Std Power):**
* `mean`
* `std`
* `min`
* `max`
* `range`
* `median`
* `delta`
* `sum`
* `q25`
* `q75`
* `iqr`

**Total Feature Dimension:** 33 features per window (11 features × 3 signals).

## 5. Explainability Safeguards
Highly correlated features can dilute SHAP importance scores. Feature pruning will not occur during initial generation to avoid premature optimization.

* **Post-Generation Audit:** A correlation matrix will be computed on the complete 33-feature baseline table.
* **Action:** Highly correlated pairs (e.g., Pearson > 0.95) will be logged. A decision on correlation-based pruning will be deferred to the explainability phase.
* **Temporal Context:** Metadata features (e.g., `hour_of_day`) are excluded from the baseline to maintain cross-dataset uniformity but remain an optional enhancement for later experiments.
"""

log_path = os.path.join(PH3_PATH, "logs", "PH3_LOG_001_window_design.md")

with open(log_path, "w", encoding="utf-8") as f:
    f.write(log_text)

print("Saved:", log_path)

In [ ]:
PH3_CONFIG = {
    "window_size_minutes": 60,
    "step_size_minutes": 30,
    "overlap_percent": 50,
    "signals": ["mean_power", "max_power", "std_power"],
    "features_per_signal": ["mean", "std", "min", "max", "range", "median", "delta", "sum", "q25", "q75", "iqr"],
    "nab_padding_options_minutes": [0, 5, 10, 15],
    "label_for_unlabeled": -1,
    "split_strategy": "chronological"
}

PH3_CONFIG

In [ ]:
import os

for dataset in DATASETS:
    dataset_path = os.path.join(PH2_PATH, dataset)
    print(f"\n=== {dataset} ===")
    if os.path.exists(dataset_path):
        for item in os.listdir(dataset_path)[:15]:
            print(" -", item)
    else:
        print("Dataset folder not found")

In [ ]:
PH2_PATH = f"{BASE_PATH}/processed_data/PH2"

print("PH2 PATH:", PH2_PATH)
print("\n--- FOLDER TREE (depth ≤ 4) ---\n")

import os

for root, dirs, files in os.walk(PH2_PATH):
    level = root.replace(PH2_PATH, "").count(os.sep)

    if level <= 4:
        indent = "    " * level
        print(f"{indent}{os.path.basename(root)}/")

        for f in files[:8]:
            print(f"{indent}    {f}")

In [ ]:
DATASET_PATHS = {

    "nab":
        f"{PH2_PATH}/nab_all_series",

    "blued":
        f"{PH2_PATH}/blued_all_series",

    "ukdale":
        f"{PH2_PATH}/ukdale_all_series",

    "ampds2":
        f"{PH2_PATH}/ampds2_all_series",
}

In [ ]:
import os

def collect_csv_files(dataset_path):

    csv_files = []

    for root, dirs, files in os.walk(dataset_path):

        for f in files:

            if not f.endswith(".csv"):
                continue

            if "summary" in f.lower():
                continue

            csv_files.append(
                os.path.join(root, f)
            )

    return sorted(csv_files)

In [ ]:
for name, path in DATASET_PATHS.items():

    files = collect_csv_files(path)

    print("\n", name.upper())
    print("path:", path)
    print("n_files:", len(files))

    for f in files[:5]:
        print("  ", os.path.basename(f))

In [ ]:
import pandas as pd

def inspect_schema(file_path):

    df = pd.read_csv(file_path, nrows=5)

    print("\nFILE:", os.path.basename(file_path))
    print("columns:", df.columns.tolist())

    return df.columns.tolist()


for dataset, path in DATASET_PATHS.items():

    files = collect_csv_files(path)

    if len(files) > 0:

        inspect_schema(files[0])

In [ ]:
import pandas as pd

def harmonize_schema(df, dataset_name):

    df = df.copy()

    # ---------- UK-DALE timestamp fix ----------
    if "index" in df.columns:
        df = df.rename(columns={"index": "timestamp"})

    # ---------- NAB column normalization ----------
    if dataset_name == "nab":

        df = df.rename(columns={"value": "mean_power"})

        # create synthetic max/std so feature generator works uniformly
        df["max_power"] = df["mean_power"]
        df["std_power"] = 0

    # ---------- ensure timestamp ----------
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    # ---------- ensure required columns ----------
    required_cols = ["timestamp", "mean_power", "max_power", "std_power"]

    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing column {col}")

    # ---------- label handling ----------
    if "label" not in df.columns:
        df["label"] = -1

    # ---------- ordering ----------
    df = df[
        [
            "timestamp",
            "mean_power",
            "max_power",
            "std_power",
            "label",
        ]
    ]

    # ---------- sort ----------
    df = df.sort_values("timestamp")

    # ---------- drop invalid timestamps ----------
    df = df.dropna(subset=["timestamp"])

    return df

In [ ]:
for dataset, path in DATASET_PATHS.items():

    files = collect_csv_files(path)

    df = pd.read_csv(files[0])

    df_h = harmonize_schema(df, dataset)

    print("\n", dataset.upper())
    print(df_h.head())
    print(df_h.columns)

In [ ]:
import pandas as pd
import os

# pick one sample file from each problematic dataset
blued_file = collect_csv_files(DATASET_PATHS["blued"])[0]
ampds2_file = collect_csv_files(DATASET_PATHS["ampds2"])[0]

print("BLUED FILE:", blued_file)
df_blued_raw = pd.read_csv(blued_file)
print(df_blued_raw.head(10))
print(df_blued_raw.columns.tolist())
print(df_blued_raw.dtypes)

print("\n" + "="*80 + "\n")

print("AMPDS2 FILE:", ampds2_file)
df_ampds2_raw = pd.read_csv(ampds2_file)
print(df_ampds2_raw.head(10))
print(df_ampds2_raw.columns.tolist())
print(df_ampds2_raw.dtypes)

In [ ]:
print("Unique BLUED timestamp samples:")
print(df_blued_raw.iloc[:10, 0].tolist())

print("\nUnique AMPDS2 timestamp samples:")
print(df_ampds2_raw.iloc[:10, 0].tolist())

In [ ]:
import pandas as pd

BASE_START_TIME = pd.Timestamp("2000-01-01 00:00:00")


def harmonize_schema(df, dataset_name):

    df = df.copy()

    # ---------------------------
    # UK-DALE
    # ---------------------------
    if dataset_name == "ukdale":

        if "index" in df.columns:
            df = df.rename(columns={"index": "timestamp"})

        df["timestamp"] = pd.to_datetime(
            df["timestamp"],
            errors="coerce",
            utc=True
        )

    # ---------------------------
    # NAB
    # ---------------------------
    elif dataset_name == "nab":

        df = df.rename(columns={"value": "mean_power"})

        df["max_power"] = df["mean_power"]
        df["std_power"] = 0

        df["timestamp"] = pd.to_datetime(
            df["timestamp"],
            errors="coerce"
        )

    # ---------------------------
    # BLUED
    # ---------------------------
    elif dataset_name == "blued":

        # convert timedelta strings
        td = pd.to_timedelta(df["timestamp"])

        df["timestamp"] = BASE_START_TIME + td

    # ---------------------------
    # AMPds2
    # ---------------------------
    elif dataset_name == "ampds2":

        # rebuild timestamps from index
        df["timestamp"] = BASE_START_TIME + pd.to_timedelta(
            range(len(df)),
            unit="min"
        )

    # ---------------------------
    # label column
    # ---------------------------
    if "label" not in df.columns:
        df["label"] = -1

    # ---------------------------
    # ensure schema consistency
    # ---------------------------
    required_cols = [
        "timestamp",
        "mean_power",
        "max_power",
        "std_power",
        "label"
    ]

    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"{dataset_name} missing column {col}")

    df = df[required_cols]

    df = df.sort_values("timestamp")

    df = df.reset_index(drop=True)

    return df

In [ ]:
for dataset, path in DATASET_PATHS.items():

    files = collect_csv_files(path)

    df = pd.read_csv(files[0])

    df_h = harmonize_schema(df, dataset)

    print("\n", dataset.upper())
    print(df_h.head())
    print(df_h["timestamp"].diff().head())

In [ ]:
for dataset in ["ampds2"]:
    files = collect_csv_files(DATASET_PATHS[dataset])
    df = pd.read_csv(files[0])
    df_h = harmonize_schema(df, dataset)

    print("Rows:", len(df_h))
    print(df_h.head(5))
    print(df_h["timestamp"].tail(5))

In [ ]:
files = collect_csv_files(DATASET_PATHS["ampds2"])

row_counts = []

for f in files[:20]:

    df = pd.read_csv(f)

    row_counts.append(len(df))

    print(
        f.split("/")[-1],
        "rows:",
        len(df)
    )

print("\nmin rows:", min(row_counts))
print("max rows:", max(row_counts))

In [ ]:
RAW_AMPDS2_PATH = f"{BASE_PATH}/Datasets/ampds2"

import os
print("RAW_AMPDS2_PATH:", RAW_AMPDS2_PATH)
print(os.listdir(RAW_AMPDS2_PATH)[:30])

In [ ]:
import pandas as pd
import os

raw_files = [
    f for f in os.listdir(RAW_AMPDS2_PATH)
    if f.endswith(".csv")
]

print("CSV files:", raw_files[:10])

sample_file = os.path.join(RAW_AMPDS2_PATH, raw_files[0])
print("Sample file:", sample_file)

df_raw = pd.read_csv(sample_file)

print(df_raw.head())
print(df_raw.columns.tolist())
print(df_raw.dtypes)
print("Rows:", len(df_raw))

In [ ]:
import os
import pandas as pd

raw_files = sorted([
    f for f in os.listdir(RAW_AMPDS2_PATH)
    if f.endswith(".csv") and f.startswith("Electricity_") and "Billing" not in f
])

print("Electricity CSV files:")
print(raw_files[:20])

sample_file = os.path.join(RAW_AMPDS2_PATH, "Electricity_B1E.csv")
print("\nSample file:", sample_file)

df_raw = pd.read_csv(sample_file)

print("\nHEAD:")
print(df_raw.head())

print("\nCOLUMNS:")
print(df_raw.columns.tolist())

print("\nDTYPES:")
print(df_raw.dtypes)

print("\nROWS:", len(df_raw))

In [ ]:
import os
import pandas as pd
from glob import glob

RAW_AMPDS2_PATH = f"{BASE_PATH}/Datasets/ampds2"
AMPDS2_PH2_OUT = f"{PH2_PATH}/ampds2_all_series"

os.makedirs(AMPDS2_PH2_OUT, exist_ok=True)

electricity_files = sorted([
    f for f in glob(os.path.join(RAW_AMPDS2_PATH, "Electricity_*.csv"))
    if "Billing" not in os.path.basename(f) and "Monthly" not in os.path.basename(f)
])

print("Electricity files to process:", len(electricity_files))
for f in electricity_files[:10]:
    print("-", os.path.basename(f))

In [ ]:
def process_ampds2_electricity_file(file_path, output_dir):
    """
    AMPds2 raw electricity files are already minute-level.
    So we normalize each signal into PH2 unified schema without resampling.
    """
    df = pd.read_csv(file_path)

    if "unix_ts" not in df.columns:
        raise ValueError(f"'unix_ts' not found in {file_path}")

    df["timestamp"] = pd.to_datetime(df["unix_ts"], unit="s", utc=True)

    signal_cols = [c for c in df.columns if c not in ["unix_ts", "timestamp"]]

    base_name = os.path.splitext(os.path.basename(file_path))[0]

    saved_files = []

    for signal in signal_cols:
        out_df = pd.DataFrame({
            "timestamp": df["timestamp"],
            "mean_power": df[signal],
            "max_power": df[signal],
            "std_power": 0.0
        })

        out_name = f"{base_name}_{signal}_1min.csv"
        out_path = os.path.join(output_dir, out_name)

        out_df.to_csv(out_path, index=False)
        saved_files.append(out_path)

    return saved_files

In [ ]:
all_saved = []

for file_path in electricity_files:
    saved = process_ampds2_electricity_file(file_path, AMPDS2_PH2_OUT)
    all_saved.extend(saved)

print("Total saved AMPds2 PH2 files:", len(all_saved))
print("Sample outputs:")
for f in all_saved[:10]:
    print("-", os.path.basename(f))

In [ ]:
from tqdm.auto import tqdm
import os
import re
import pandas as pd
from glob import glob

In [ ]:
RAW_AMPDS2_PATH = f"{BASE_PATH}/Datasets/ampds2"
AMPDS2_PH2_OUT = f"{PH2_PATH}/ampds2_all_series"

os.makedirs(AMPDS2_PH2_OUT, exist_ok=True)

# Only keep raw electricity files like Electricity_B1E.csv, Electricity_B2E.csv, etc.
pattern = re.compile(r"^Electricity_[A-Z0-9]+E\.csv$")

electricity_files = sorted([
    os.path.join(RAW_AMPDS2_PATH, f)
    for f in os.listdir(RAW_AMPDS2_PATH)
    if pattern.match(f)
])

print("Raw AMPds2 electricity files to process:", len(electricity_files))
for f in electricity_files[:20]:
    print("-", os.path.basename(f))

In [ ]:
def process_ampds2_electricity_file(file_path, output_dir):
    """
    Process one raw AMPds2 electricity file.
    AMPds2 raw electricity files are already minute-level.
    We normalize each signal into:
    timestamp, mean_power, max_power, std_power
    """
    df = pd.read_csv(file_path)

    if "unix_ts" not in df.columns:
        raise ValueError(f"'unix_ts' not found in {file_path}")

    df["timestamp"] = pd.to_datetime(df["unix_ts"], unit="s", utc=True)

    signal_cols = [c for c in df.columns if c not in ["unix_ts", "timestamp"]]
    base_name = os.path.splitext(os.path.basename(file_path))[0]

    saved_files = []

    for signal in signal_cols:
        out_df = pd.DataFrame({
            "timestamp": df["timestamp"],
            "mean_power": df[signal],
            "max_power": df[signal],
            "std_power": 0.0
        })

        out_name = f"{base_name}_{signal}_1min.csv"
        out_path = os.path.join(output_dir, out_name)
        out_df.to_csv(out_path, index=False)
        saved_files.append(out_path)

    return saved_files

In [ ]:
all_saved = []
failed_files = []

for file_path in tqdm(electricity_files, desc="Processing AMPds2 raw files"):
    try:
        saved = process_ampds2_electricity_file(file_path, AMPDS2_PH2_OUT)
        all_saved.extend(saved)
    except Exception as e:
        failed_files.append({
            "file": file_path,
            "error": str(e)
        })

print("\nTotal saved AMPds2 PH2 files:", len(all_saved))
print("Failed files:", len(failed_files))

if failed_files:
    print("\nFailures:")
    for item in failed_files[:10]:
        print("-", os.path.basename(item["file"]), "->", item["error"])

print("\nSample outputs:")
for f in all_saved[:10]:
    print("-", os.path.basename(f))

In [ ]:
import os
import pandas as pd
from glob import glob
from tqdm.auto import tqdm

summary_rows = []

for csv_file in tqdm(sorted(glob(os.path.join(AMPDS2_PH2_OUT, "*.csv"))), desc="Building AMPds2 summary"):
    df = pd.read_csv(csv_file)

    summary_rows.append({
        "file_name": os.path.basename(csv_file),
        "rows": len(df),
        "columns": ", ".join(df.columns.tolist()),
        "start_time": df["timestamp"].iloc[0] if len(df) > 0 else None,
        "end_time": df["timestamp"].iloc[-1] if len(df) > 0 else None,
        "mean_of_mean_power": df["mean_power"].mean() if len(df) > 0 else None,
        "std_of_mean_power": df["mean_power"].std() if len(df) > 0 else None
    })

ampds2_summary_df = pd.DataFrame(summary_rows)
ampds2_summary_path = os.path.join(AMPDS2_PH2_OUT, "ampds2_ph2_summary.csv")
ampds2_summary_df.to_csv(ampds2_summary_path, index=False)

print("Saved summary:", ampds2_summary_path)
print(ampds2_summary_df.head())

In [ ]:
import os
import pandas as pd
from glob import glob

all_csvs = sorted(glob(os.path.join(AMPDS2_PH2_OUT, "*.csv")))

bad_files = []

for csv_file in all_csvs:
    try:
        df = pd.read_csv(csv_file, nrows=3)
        cols = df.columns.tolist()

        if "timestamp" not in cols:
            bad_files.append((os.path.basename(csv_file), cols))
    except Exception as e:
        bad_files.append((os.path.basename(csv_file), f"READ ERROR: {e}"))

print("Total CSVs found:", len(all_csvs))
print("Bad files:")
for item in bad_files:
    print(item)

In [ ]:
import os
import pandas as pd
from glob import glob
from tqdm.auto import tqdm

summary_rows = []
skipped_files = []

all_csvs = sorted(glob(os.path.join(AMPDS2_PH2_OUT, "*.csv")))

for csv_file in tqdm(all_csvs, desc="Building AMPds2 summary"):
    try:
        df = pd.read_csv(csv_file)

        required_cols = {"timestamp", "mean_power", "max_power", "std_power"}

        if not required_cols.issubset(df.columns):
            skipped_files.append({
                "file_name": os.path.basename(csv_file),
                "reason": f"Missing required columns: {df.columns.tolist()}"
            })
            continue

        summary_rows.append({
            "file_name": os.path.basename(csv_file),
            "rows": len(df),
            "columns": ", ".join(df.columns.tolist()),
            "start_time": df["timestamp"].iloc[0] if len(df) > 0 else None,
            "end_time": df["timestamp"].iloc[-1] if len(df) > 0 else None,
            "mean_of_mean_power": df["mean_power"].mean() if len(df) > 0 else None,
            "std_of_mean_power": df["mean_power"].std() if len(df) > 0 else None
        })

    except Exception as e:
        skipped_files.append({
            "file_name": os.path.basename(csv_file),
            "reason": str(e)
        })

ampds2_summary_df = pd.DataFrame(summary_rows)
ampds2_summary_path = os.path.join(AMPDS2_PH2_OUT, "ampds2_ph2_summary.csv")
ampds2_summary_df.to_csv(ampds2_summary_path, index=False)

print("Saved summary:", ampds2_summary_path)
print("Valid files summarized:", len(ampds2_summary_df))
print("Skipped files:", len(skipped_files))

if skipped_files:
    skipped_df = pd.DataFrame(skipped_files)
    skipped_log_path = os.path.join(AMPDS2_PH2_OUT, "ampds2_summary_skipped_files.csv")
    skipped_df.to_csv(skipped_log_path, index=False)
    print("Saved skipped-files log:", skipped_log_path)
    print(skipped_df.head())

In [ ]:
def collect_valid_ph2_signal_csvs(folder_path):
    files = []
    for f in sorted(glob(os.path.join(folder_path, "*.csv"))):
        name = os.path.basename(f).lower()
        if "summary" in name or "skipped" in name or "failed" in name:
            continue
        files.append(f)
    return files

In [ ]:
ampds2_files = collect_valid_ph2_signal_csvs(AMPDS2_PH2_OUT)

print("AMPds2 valid signal files:", len(ampds2_files))

sample_ampds2 = ampds2_files[0]
df_check = pd.read_csv(sample_ampds2)

print("Sample file:", os.path.basename(sample_ampds2))
print(df_check.head())
print(df_check.columns.tolist())
print("Rows:", len(df_check))

In [ ]:
import os

def collect_csv_files(dataset_path):
    csv_files = []

    for root, dirs, files in os.walk(dataset_path):
        for f in files:
            if not f.endswith(".csv"):
                continue

            name = f.lower()

            if "summary" in name or "skipped" in name or "failed" in name:
                continue

            csv_files.append(os.path.join(root, f))

    return sorted(csv_files)

In [ ]:
for dataset, path in DATASET_PATHS.items():
    files = collect_csv_files(path)
    df = pd.read_csv(files[0])
    df_h = harmonize_schema(df, dataset)

    print(f"\n{dataset.upper()}")
    print(df_h.head())
    print(df_h.columns.tolist())
    print("Rows:", len(df_h))
    print("Timestamp diff:")
    print(df_h["timestamp"].diff().head())

In [ ]:
for dataset, path in DATASET_PATHS.items():
    files = collect_csv_files(path)
    df = pd.read_csv(files[0])
    df_h = harmonize_schema(df, dataset)

    windows = generate_window_indices(
        df_h,
        window_size_minutes=PH3_CONFIG["window_size_minutes"],
        step_size_minutes=PH3_CONFIG["step_size_minutes"]
    )

    print(f"\n{dataset.upper()}")
    print("sample file:", os.path.basename(files[0]))
    print("rows:", len(df_h))
    print("sampling_minutes:", infer_sampling_minutes(df_h))
    print("n_windows:", len(windows))
    print("first_3_windows:", windows[:3])

In [ ]:
import numpy as np
import pandas as pd

def infer_sampling_minutes(df):
    diffs = df["timestamp"].sort_values().diff().dropna()
    if len(diffs) == 0:
        return None
    return diffs.mode().iloc[0].total_seconds() / 60.0


def generate_window_indices(df, window_size_minutes=60, step_size_minutes=30):
    """
    Returns a list of (start_idx, end_idx) tuples, where end_idx is exclusive.
    """
    sampling_minutes = infer_sampling_minutes(df)

    if sampling_minutes is None:
        return []

    window_rows = int(round(window_size_minutes / sampling_minutes))
    step_rows = int(round(step_size_minutes / sampling_minutes))

    if window_rows <= 0 or step_rows <= 0:
        raise ValueError("Invalid window_rows or step_rows")

    n = len(df)
    windows = []

    for start in range(0, n - window_rows + 1, step_rows):
        end = start + window_rows
        windows.append((start, end))

    return windows

In [ ]:
import pandas as pd

BASE_START_TIME = pd.Timestamp("2000-01-01 00:00:00")

def harmonize_schema(df, dataset_name):
    df = df.copy()

    if dataset_name == "ukdale":
        if "index" in df.columns:
            df = df.rename(columns={"index": "timestamp"})

        df["timestamp"] = pd.to_datetime(
            df["timestamp"],
            errors="coerce",
            utc=True
        )

    elif dataset_name == "nab":
        df = df.rename(columns={"value": "mean_power"})
        df["max_power"] = df["mean_power"]
        df["std_power"] = 0

        df["timestamp"] = pd.to_datetime(
            df["timestamp"],
            errors="coerce"
        )

    elif dataset_name == "blued":
        td = pd.to_timedelta(df["timestamp"])
        df["timestamp"] = BASE_START_TIME + td

    elif dataset_name == "ampds2":
        # Use real timestamps if present
        if "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(
                df["timestamp"],
                errors="coerce",
                utc=True
            )
        else:
            # fallback only if timestamp is missing
            df["timestamp"] = BASE_START_TIME + pd.to_timedelta(
                range(len(df)),
                unit="min"
            )

    if "label" not in df.columns:
        df["label"] = -1

    required_cols = ["timestamp", "mean_power", "max_power", "std_power", "label"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"{dataset_name} missing column {col}")

    df = df[required_cols]
    df = df.dropna(subset=["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)

    return df

In [ ]:
for dataset, path in DATASET_PATHS.items():
    files = collect_csv_files(path)
    df = pd.read_csv(files[0])
    df_h = harmonize_schema(df, dataset)

    windows = generate_window_indices(
        df_h,
        window_size_minutes=PH3_CONFIG["window_size_minutes"],
        step_size_minutes=PH3_CONFIG["step_size_minutes"]
    )

    print(f"\n{dataset.upper()}")
    print("sample file:", os.path.basename(files[0]))
    print("rows:", len(df_h))
    print("sampling_minutes:", infer_sampling_minutes(df_h))
    print("n_windows:", len(windows))
    print("first_3_windows:", windows[:3])

In [ ]:
import numpy as np
import pandas as pd

def extract_window_features(window_df, signals=None):
    if signals is None:
        signals = ["mean_power", "max_power", "std_power"]

    features = {}

    for signal in signals:
        s = pd.to_numeric(window_df[signal], errors="coerce").astype(float)

        features[f"{signal}_mean"] = s.mean()
        features[f"{signal}_std"] = s.std()
        features[f"{signal}_min"] = s.min()
        features[f"{signal}_max"] = s.max()
        features[f"{signal}_range"] = s.max() - s.min()
        features[f"{signal}_median"] = s.median()
        features[f"{signal}_delta"] = s.iloc[-1] - s.iloc[0]
        features[f"{signal}_sum"] = s.sum()
        features[f"{signal}_q25"] = s.quantile(0.25)
        features[f"{signal}_q75"] = s.quantile(0.75)
        features[f"{signal}_iqr"] = s.quantile(0.75) - s.quantile(0.25)

    return features

In [ ]:
def assign_window_label(window_df, dataset_name):
    """
    Main baseline rule:
    - NAB: anomalous if any label==1 exists inside the window
    - unlabeled datasets: -1
    """
    if dataset_name == "nab":
        if "label" in window_df.columns:
            return int((window_df["label"] == 1).any())
        return 0

    return -1

In [ ]:
def build_feature_windows_for_file(
    file_path,
    dataset_name,
    window_size_minutes=60,
    step_size_minutes=30,
    signals=None
):
    if signals is None:
        signals = ["mean_power", "max_power", "std_power"]

    df = pd.read_csv(file_path)
    df_h = harmonize_schema(df, dataset_name)

    windows = generate_window_indices(
        df_h,
        window_size_minutes=window_size_minutes,
        step_size_minutes=step_size_minutes
    )

    rows = []

    for window_id, (start_idx, end_idx) in enumerate(windows):
        window_df = df_h.iloc[start_idx:end_idx].copy()

        feature_row = {
            "dataset_name": dataset_name,
            "series_id": os.path.basename(file_path),
            "window_id": window_id,
            "start_time": window_df["timestamp"].iloc[0],
            "end_time": window_df["timestamp"].iloc[-1],
            "window_size_minutes": window_size_minutes,
            "step_size_minutes": step_size_minutes,
            "sampling_minutes": infer_sampling_minutes(df_h),
            "label": assign_window_label(window_df, dataset_name),
        }

        feature_row.update(
            extract_window_features(window_df, signals=signals)
        )

        rows.append(feature_row)

    return pd.DataFrame(rows)

In [ ]:
MAIN_PHASE3_DATASETS = ["nab", "ukdale", "ampds2"]

for dataset in MAIN_PHASE3_DATASETS:
    files = collect_csv_files(DATASET_PATHS[dataset])
    sample_file = files[0]

    feature_df = build_feature_windows_for_file(
        file_path=sample_file,
        dataset_name=dataset,
        window_size_minutes=PH3_CONFIG["window_size_minutes"],
        step_size_minutes=PH3_CONFIG["step_size_minutes"],
        signals=PH3_CONFIG["signals"]
    )

    print(f"\n{dataset.upper()}")
    print("sample file:", os.path.basename(sample_file))
    print("feature windows shape:", feature_df.shape)
    print(feature_df.head(2))
    print("label distribution:")
    print(feature_df["label"].value_counts(dropna=False))

## Checkpoint Note
Phase 3 baseline design and data harmonization were validated successfully.

Confirmed:
- NAB supports 60-minute windows at 5-minute sampling
- UK-DALE supports 60-minute windows at 1-minute sampling
- AMPds2 supports 60-minute windows at 1-minute sampling after PH2 repair
- BLUED segmented files are too short for the main 60-minute baseline and will be treated separately

Current status:
- Feature extraction logic is correct
- Current implementation is too slow for large files
- Next step is to refactor feature-window generation using a capped pilot mode, progress bars, and incremental saving

## Checkpoint Update
Phase 3 preprocessing validation completed.

Verified:
- PH2 folder structure resolved successfully
- Schema harmonization works for NAB, BLUED, UK-DALE, and AMPds2
- AMPds2 PH2 bug was identified and repaired
- Main 60-minute / 30-minute sliding-window baseline is valid for:
  - NAB
  - UK-DALE
  - AMPds2
- BLUED segmented files are too short for the main 60-minute baseline and will be handled separately

Observed issue:
- Initial feature-window generation implementation is correct logically but too slow for large files

Next action:
- Refactor Phase 3 feature generation using:
  - capped pilot mode
  - progress bars
  - incremental saving
  - dataset-by-dataset execution

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from glob import glob
from tqdm.auto import tqdm

BASE_PATH = "/content/drive/MyDrive/Master/Thesis/Code"

PH2_PATH = os.path.join(BASE_PATH, "processed_data", "PH2")
PH3_PATH = os.path.join(BASE_PATH, "processed_data", "PH3")

DATASET_PATHS = {
    "nab":    os.path.join(PH2_PATH, "nab_all_series"),
    "blued":  os.path.join(PH2_PATH, "blued_all_series"),
    "ukdale": os.path.join(PH2_PATH, "ukdale_all_series"),
    "ampds2": os.path.join(PH2_PATH, "ampds2_all_series"),
}

PH3_CONFIG = {
    "window_size_minutes": 60,
    "step_size_minutes": 30,
    "overlap_percent": 50,
    "signals": ["mean_power", "max_power", "std_power"],
    "features_per_signal": ["mean", "std", "min", "max", "range", "median", "delta", "sum", "q25", "q75", "iqr"],
    "nab_padding_options_minutes": [0, 5, 10, 15],
    "label_for_unlabeled": -1,
    "split_strategy": "chronological",
}

BASE_START_TIME = pd.Timestamp("2000-01-01 00:00:00")

def collect_csv_files(dataset_path):
    csv_files = []
    for root, dirs, files in os.walk(dataset_path):
        for f in files:
            if not f.endswith(".csv"):
                continue
            name = f.lower()
            if "summary" in name or "skipped" in name or "failed" in name:
                continue
            csv_files.append(os.path.join(root, f))
    return sorted(csv_files)

def harmonize_schema(df, dataset_name):
    df = df.copy()

    if dataset_name == "ukdale":
        if "index" in df.columns:
            df = df.rename(columns={"index": "timestamp"})
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)

    elif dataset_name == "nab":
        df = df.rename(columns={"value": "mean_power"})
        df["max_power"] = df["mean_power"]
        df["std_power"] = 0.0
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    elif dataset_name == "blued":
        td = pd.to_timedelta(df["timestamp"])
        df["timestamp"] = BASE_START_TIME + td

    elif dataset_name == "ampds2":
        if "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
        else:
            df["timestamp"] = BASE_START_TIME + pd.to_timedelta(range(len(df)), unit="min")

    if "label" not in df.columns:
        df["label"] = -1

    required_cols = ["timestamp", "mean_power", "max_power", "std_power", "label"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"{dataset_name} missing column {col}")

    df = df[required_cols]
    df = df.dropna(subset=["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    return df

def infer_sampling_minutes(df):
    diffs = df["timestamp"].sort_values().diff().dropna()
    if len(diffs) == 0:
        return None
    return diffs.mode().iloc[0].total_seconds() / 60.0

def generate_window_indices(df, window_size_minutes=60, step_size_minutes=30):
    sampling_minutes = infer_sampling_minutes(df)
    if sampling_minutes is None:
        return []

    window_rows = int(round(window_size_minutes / sampling_minutes))
    step_rows = int(round(step_size_minutes / sampling_minutes))

    if window_rows <= 0 or step_rows <= 0:
        raise ValueError("Invalid window_rows or step_rows")

    n = len(df)
    windows = []
    for start in range(0, n - window_rows + 1, step_rows):
        end = start + window_rows
        windows.append((start, end))
    return windows

print("PH2 exists:", os.path.exists(PH2_PATH))
print("PH3 exists:", os.path.exists(PH3_PATH))

In [ ]:
checkpoint_note = """

## Checkpoint Update
Phase 3 preprocessing validation completed.

Verified:
- PH2 folder structure resolved successfully
- Schema harmonization works for NAB, BLUED, UK-DALE, and AMPds2
- AMPds2 PH2 bug was identified and repaired
- Main 60-minute / 30-minute sliding-window baseline is valid for:
  - NAB
  - UK-DALE
  - AMPds2
- BLUED segmented files are too short for the main 60-minute baseline and will be handled separately

Observed issue:
- Initial feature-window generation implementation is correct logically but too slow for large files

Next action:
- Refactor Phase 3 feature generation using:
  - capped pilot mode
  - progress bars
  - incremental saving
  - dataset-by-dataset execution
"""

log_path = os.path.join(PH3_PATH, "logs", "PH3_LOG_002_window_generation.md")

with open(log_path, "a", encoding="utf-8") as f:
    f.write(checkpoint_note)

print("Checkpoint appended to:", log_path)

In [ ]:
import os

# Rebuild PH3 folders if this runtime is fresh
os.makedirs(PH3_PATH, exist_ok=True)
os.makedirs(os.path.join(PH3_PATH, "logs"), exist_ok=True)
os.makedirs(os.path.join(PH3_PATH, "combined"), exist_ok=True)

log_path = os.path.join(PH3_PATH, "logs", "PH3_LOG_002_window_generation.md")

checkpoint_note = """

## Checkpoint Update
Phase 3 preprocessing validation completed.

Verified:
- PH2 folder structure resolved successfully
- Schema harmonization works for NAB, BLUED, UK-DALE, and AMPds2
- AMPds2 PH2 bug was identified and repaired
- Main 60-minute / 30-minute sliding-window baseline is valid for:
  - NAB
  - UK-DALE
  - AMPds2
- BLUED segmented files are too short for the main 60-minute baseline and will be handled separately

Observed issue:
- Initial feature-window generation implementation is correct logically but too slow for large files

Next action:
- Refactor Phase 3 feature generation using:
  - capped pilot mode
  - progress bars
  - incremental saving
  - dataset-by-dataset execution
"""

with open(log_path, "a", encoding="utf-8") as f:
    f.write(checkpoint_note)

print("Checkpoint appended to:", log_path)

In [ ]:
def safe_std(x):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) <= 1:
        return 0.0
    return float(np.std(x, ddof=1))

def extract_signal_features_np(arr, signal_name):
    arr = np.asarray(arr, dtype=float)

    return {
        f"{signal_name}_mean": float(np.nanmean(arr)),
        f"{signal_name}_std": safe_std(arr),
        f"{signal_name}_min": float(np.nanmin(arr)),
        f"{signal_name}_max": float(np.nanmax(arr)),
        f"{signal_name}_range": float(np.nanmax(arr) - np.nanmin(arr)),
        f"{signal_name}_median": float(np.nanmedian(arr)),
        f"{signal_name}_delta": float(arr[-1] - arr[0]),
        f"{signal_name}_sum": float(np.nansum(arr)),
        f"{signal_name}_q25": float(np.nanquantile(arr, 0.25)),
        f"{signal_name}_q75": float(np.nanquantile(arr, 0.75)),
        f"{signal_name}_iqr": float(np.nanquantile(arr, 0.75) - np.nanquantile(arr, 0.25)),
    }

def assign_window_label_np(label_arr, start_idx, end_idx, dataset_name):
    if dataset_name == "nab":
        return int(np.any(label_arr[start_idx:end_idx] == 1))
    return -1

def build_feature_windows_for_file_fast(
    file_path,
    dataset_name,
    window_size_minutes=60,
    step_size_minutes=30,
    signals=None,
    max_windows=None,
    show_progress=True,
):
    if signals is None:
        signals = ["mean_power", "max_power", "std_power"]

    df = pd.read_csv(file_path)
    df_h = harmonize_schema(df, dataset_name)

    windows = generate_window_indices(
        df_h,
        window_size_minutes=window_size_minutes,
        step_size_minutes=step_size_minutes
    )

    if max_windows is not None:
        windows = windows[:max_windows]

    if len(windows) == 0:
        return pd.DataFrame()

    timestamps = df_h["timestamp"].to_numpy()
    label_arr = df_h["label"].to_numpy()

    signal_arrays = {
        sig: pd.to_numeric(df_h[sig], errors="coerce").to_numpy(dtype=float)
        for sig in signals
    }

    sampling_minutes = infer_sampling_minutes(df_h)
    rows = []

    iterator = enumerate(windows)
    if show_progress:
        iterator = tqdm(iterator, total=len(windows), desc=f"{dataset_name}:{os.path.basename(file_path)}")

    for window_id, (start_idx, end_idx) in iterator:
        row = {
            "dataset_name": dataset_name,
            "series_id": os.path.basename(file_path),
            "window_id": window_id,
            "start_time": timestamps[start_idx],
            "end_time": timestamps[end_idx - 1],
            "window_size_minutes": window_size_minutes,
            "step_size_minutes": step_size_minutes,
            "sampling_minutes": sampling_minutes,
            "label": assign_window_label_np(label_arr, start_idx, end_idx, dataset_name),
        }

        for sig in signals:
            arr = signal_arrays[sig][start_idx:end_idx]
            row.update(extract_signal_features_np(arr, sig))

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
MAIN_PHASE3_DATASETS = ["nab", "ukdale", "ampds2"]
PILOT_MAX_WINDOWS = 500

pilot_out_dir = os.path.join(PH3_PATH, "combined", "pilot_feature_windows")
os.makedirs(pilot_out_dir, exist_ok=True)

pilot_results = {}

for dataset in MAIN_PHASE3_DATASETS:
    files = collect_csv_files(DATASET_PATHS[dataset])
    sample_file = files[0]

    print(f"\n=== {dataset.upper()} ===")
    print("Sample file:", os.path.basename(sample_file))

    feature_df = build_feature_windows_for_file_fast(
        file_path=sample_file,
        dataset_name=dataset,
        window_size_minutes=PH3_CONFIG["window_size_minutes"],
        step_size_minutes=PH3_CONFIG["step_size_minutes"],
        signals=PH3_CONFIG["signals"],
        max_windows=PILOT_MAX_WINDOWS,
        show_progress=True
    )

    pilot_results[dataset] = feature_df

    out_path = os.path.join(
        pilot_out_dir,
        f"{dataset}_pilot_{os.path.basename(sample_file).replace('.csv', '')}_features.csv"
    )
    feature_df.to_csv(out_path, index=False)

    print("Saved:", out_path)
    print("Shape:", feature_df.shape)
    if not feature_df.empty:
        print(feature_df[["dataset_name", "series_id", "window_id", "start_time", "end_time", "label"]].head(2))
        print("Label distribution:")
        print(feature_df["label"].value_counts(dropna=False))

In [ ]:
pilot_summary_rows = []

for dataset, df in pilot_results.items():
    if df.empty:
        pilot_summary_rows.append({
            "dataset": dataset,
            "n_windows": 0,
            "n_columns": 0,
            "label_counts": "EMPTY"
        })
    else:
        pilot_summary_rows.append({
            "dataset": dataset,
            "n_windows": len(df),
            "n_columns": len(df.columns),
            "label_counts": df["label"].value_counts(dropna=False).to_dict()
        })

pilot_summary_df = pd.DataFrame(pilot_summary_rows)
pilot_summary_path = os.path.join(pilot_out_dir, "pilot_summary.csv")
pilot_summary_df.to_csv(pilot_summary_path, index=False)

print("Pilot summary saved:", pilot_summary_path)
print(pilot_summary_df)

In [ ]:
import os
import json
import pandas as pd
from tqdm.auto import tqdm

MAIN_FEATURE_OUTPUTS = {
    "nab":    os.path.join(PH3_PATH, "NAB", "raw_windows", "features"),
    "ukdale": os.path.join(PH3_PATH, "UK_DALE", "windows", "features"),
    "ampds2": os.path.join(PH3_PATH, "AMPds2", "windows", "features"),
}

for path in MAIN_FEATURE_OUTPUTS.values():
    os.makedirs(path, exist_ok=True)

os.makedirs(os.path.join(PH3_PATH, "combined"), exist_ok=True)

print(MAIN_FEATURE_OUTPUTS)

In [ ]:
def generate_feature_windows_for_dataset(
    dataset_name,
    input_dir,
    output_dir,
    window_size_minutes=60,
    step_size_minutes=30,
    signals=None,
    skip_existing=True,
    max_files=None
):
    if signals is None:
        signals = ["mean_power", "max_power", "std_power"]

    input_files = collect_csv_files(input_dir)

    if max_files is not None:
        input_files = input_files[:max_files]

    summary_rows = []
    failed_rows = []

    outer_bar = tqdm(input_files, desc=f"Processing {dataset_name}", position=0)

    for file_path in outer_bar:
        base_name = os.path.splitext(os.path.basename(file_path))[0]
        out_file = os.path.join(output_dir, f"{base_name}_feature_windows.csv")

        if skip_existing and os.path.exists(out_file):
            summary_rows.append({
                "dataset_name": dataset_name,
                "series_id": os.path.basename(file_path),
                "status": "skipped_existing",
                "n_windows": None,
                "n_columns": None,
                "label_counts": None,
                "output_file": out_file
            })
            continue

        try:
            feature_df = build_feature_windows_for_file_fast(
                file_path=file_path,
                dataset_name=dataset_name,
                window_size_minutes=window_size_minutes,
                step_size_minutes=step_size_minutes,
                signals=signals,
                max_windows=None,
                show_progress=False
            )

            feature_df.to_csv(out_file, index=False)

            label_counts = {}
            if not feature_df.empty and "label" in feature_df.columns:
                label_counts = feature_df["label"].value_counts(dropna=False).to_dict()

            summary_rows.append({
                "dataset_name": dataset_name,
                "series_id": os.path.basename(file_path),
                "status": "success",
                "n_windows": len(feature_df),
                "n_columns": len(feature_df.columns) if not feature_df.empty else 0,
                "label_counts": json.dumps(label_counts),
                "output_file": out_file
            })

        except Exception as e:
            failed_rows.append({
                "dataset_name": dataset_name,
                "series_id": os.path.basename(file_path),
                "error": str(e)
            })

    summary_df = pd.DataFrame(summary_rows)
    failed_df = pd.DataFrame(failed_rows)

    summary_path = os.path.join(output_dir, f"{dataset_name}_feature_generation_summary.csv")
    summary_df.to_csv(summary_path, index=False)

    failed_path = os.path.join(output_dir, f"{dataset_name}_feature_generation_failed.csv")
    if not failed_df.empty:
        failed_df.to_csv(failed_path, index=False)

    return summary_df, failed_df, summary_path, failed_path

In [ ]:
nab_summary_df, nab_failed_df, nab_summary_path, nab_failed_path = generate_feature_windows_for_dataset(
    dataset_name="nab",
    input_dir=DATASET_PATHS["nab"],
    output_dir=MAIN_FEATURE_OUTPUTS["nab"],
    window_size_minutes=PH3_CONFIG["window_size_minutes"],
    step_size_minutes=PH3_CONFIG["step_size_minutes"],
    signals=PH3_CONFIG["signals"],
    skip_existing=True,
    max_files=None
)

print("NAB summary saved to:", nab_summary_path)
print("NAB failures:", len(nab_failed_df))
print(nab_summary_df.head())

In [ ]:
print("Total NAB series processed:", len(nab_summary_df))
print("Status counts:")
print(nab_summary_df["status"].value_counts(dropna=False))

successful_nab = nab_summary_df[nab_summary_df["status"] == "success"]
print("\nSuccessful NAB files:", len(successful_nab))

print("\nSample summary rows:")
print(successful_nab.head())

nab_output_files = [
    f for f in os.listdir(MAIN_FEATURE_OUTPUTS["nab"])
    if f.endswith("_feature_windows.csv")
]

print("\nGenerated NAB feature files:", len(nab_output_files))
print(nab_output_files[:5])

In [ ]:
sample_nab_out = os.path.join(MAIN_FEATURE_OUTPUTS["nab"], nab_output_files[0])
df_nab_out = pd.read_csv(sample_nab_out)

print("Sample output file:", sample_nab_out)
print(df_nab_out.head())
print(df_nab_out.shape)
print(df_nab_out["label"].value_counts(dropna=False))

In [ ]:
ukdale_summary_df, ukdale_failed_df, ukdale_summary_path, ukdale_failed_path = generate_feature_windows_for_dataset(
    dataset_name="ukdale",
    input_dir=DATASET_PATHS["ukdale"],
    output_dir=MAIN_FEATURE_OUTPUTS["ukdale"],
    window_size_minutes=PH3_CONFIG["window_size_minutes"],
    step_size_minutes=PH3_CONFIG["step_size_minutes"],
    signals=PH3_CONFIG["signals"],
    skip_existing=True,
    max_files=None
)

print("UK-DALE summary saved to:", ukdale_summary_path)
print("UK-DALE failures:", len(ukdale_failed_df))
print(ukdale_summary_df.head())

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/Master/Thesis/Code"
print("Mounted:", os.path.exists("/content/drive/MyDrive"))
print("BASE_PATH exists:", os.path.exists(BASE_PATH))
print(os.listdir(BASE_PATH)[:20] if os.path.exists(BASE_PATH) else "Path not found")

In [ ]:
import os
import pandas as pd

ukdale_summary_path = "/content/drive/MyDrive/Master/Thesis/Code/processed_data/PH3/UK_DALE/windows/features/ukdale_feature_generation_summary.csv"
nab_summary_path = "/content/drive/MyDrive/Master/Thesis/Code/processed_data/PH3/NAB/raw_windows/features/nab_feature_generation_summary.csv"
nab_failed_path = "/content/drive/MyDrive/Master/Thesis/Code/processed_data/PH3/NAB/raw_windows/features/nab_feature_generation_failed.csv"

print("UK-DALE summary exists:", os.path.exists(ukdale_summary_path))
print("NAB summary exists:", os.path.exists(nab_summary_path))
print("NAB failed log exists:", os.path.exists(nab_failed_path))

ukdale_summary_df = pd.read_csv(ukdale_summary_path)
nab_summary_df = pd.read_csv(nab_summary_path)

print("\nUK-DALE status counts:")
print(ukdale_summary_df["status"].value_counts(dropna=False))

print("\nNAB status counts:")
print(nab_summary_df["status"].value_counts(dropna=False))

In [ ]:
if os.path.exists(nab_failed_path):
    nab_failed_df = pd.read_csv(nab_failed_path)
    print(nab_failed_df.head(20))
    print("Total NAB failures:", len(nab_failed_df))

In [ ]:
import os

log_path = "/content/drive/MyDrive/Master/Thesis/Code/processed_data/PH3/logs/PH3_LOG_002_window_generation.md"
os.makedirs(os.path.dirname(log_path), exist_ok=True)

nab_note = """

## NAB Baseline Compatibility Note
During full Phase 3 feature-window generation for NAB, 7 series were excluded from the main baseline because their native sampling interval was too coarse for the fixed 60-minute window / 30-minute step configuration.

Observed failed series:
- exchange-2_cpc_results.csv
- exchange-2_cpm_results.csv
- exchange-3_cpc_results.csv
- exchange-3_cpm_results.csv
- exchange-4_cpc_results.csv
- exchange-4_cpm_results.csv
- ambient_temperature_system_failure.csv

Reason:
- main baseline window configuration produced invalid window_rows or step_rows due to coarse temporal resolution

Action:
- these series are excluded from the main Phase 3 baseline
- retained NAB-compatible series continue to serve as the labeled benchmark branch
"""

with open(log_path, "a", encoding="utf-8") as f:
    f.write(nab_note)

print("NAB compatibility note appended to:", log_path)

In [ ]:
ampds2_summary_df, ampds2_failed_df, ampds2_summary_path, ampds2_failed_path = generate_feature_windows_for_dataset(
    dataset_name="ampds2",
    input_dir=DATASET_PATHS["ampds2"],
    output_dir=MAIN_FEATURE_OUTPUTS["ampds2"],
    window_size_minutes=PH3_CONFIG["window_size_minutes"],
    step_size_minutes=PH3_CONFIG["step_size_minutes"],
    signals=PH3_CONFIG["signals"],
    skip_existing=True,
    max_files=None
)

print("AMPds2 summary saved to:", ampds2_summary_path)
print("AMPds2 failures:", len(ampds2_failed_df))
print(ampds2_summary_df.head())

In [ ]:
import os
import pandas as pd

print("AMPds2 status counts:")
print(ampds2_summary_df["status"].value_counts(dropna=False))

sample_ampds2_outputs = [
    f for f in os.listdir(MAIN_FEATURE_OUTPUTS["ampds2"])
    if f.endswith("_feature_windows.csv")
]

print("Generated AMPds2 feature files:", len(sample_ampds2_outputs))
print(sample_ampds2_outputs[:5])

sample_ampds2_file = os.path.join(MAIN_FEATURE_OUTPUTS["ampds2"], sample_ampds2_outputs[0])
df_ampds2_out = pd.read_csv(sample_ampds2_file)

print("Sample output file:", sample_ampds2_file)
print(df_ampds2_out.head())
print(df_ampds2_out.shape)
print(df_ampds2_out["label"].value_counts(dropna=False))

In [ ]:
import os
import pandas as pd

nab_summary_path = os.path.join(PH3_PATH, "NAB", "raw_windows", "features", "nab_feature_generation_summary.csv")
ukdale_summary_path = os.path.join(PH3_PATH, "UK_DALE", "windows", "features", "ukdale_feature_generation_summary.csv")
ampds2_summary_path = os.path.join(PH3_PATH, "AMPds2", "windows", "features", "ampds2_feature_generation_summary.csv")

nab_df = pd.read_csv(nab_summary_path)
ukdale_df = pd.read_csv(ukdale_summary_path)
ampds2_df = pd.read_csv(ampds2_summary_path)

combined_summary_df = pd.concat([nab_df, ukdale_df, ampds2_df], ignore_index=True)

combined_summary_path = os.path.join(PH3_PATH, "combined", "ph3_main_baseline_feature_generation_summary.csv")
combined_summary_df.to_csv(combined_summary_path, index=False)

print("Saved:", combined_summary_path)
print(combined_summary_df["dataset_name"].value_counts())
print(combined_summary_df["status"].value_counts(dropna=False))
print(combined_summary_df.head())

In [ ]:
log_path = os.path.join(PH3_PATH, "logs", "PH3_LOG_002_window_generation.md")
os.makedirs(os.path.dirname(log_path), exist_ok=True)

final_note = """

## Phase 3 Main Baseline Completion
Main feature-window generation completed successfully for the fixed 60-minute window / 30-minute step baseline.

Completed datasets:
- NAB: 51 compatible series generated successfully
- UK-DALE: 111 series generated successfully
- AMPds2: 231 series generated successfully

Compatibility note:
- 7 NAB series were excluded due to coarse temporal sampling incompatible with the fixed baseline
- BLUED segmented files remain outside the main baseline branch because they are too short for 60-minute windows

Output properties:
- one feature-window CSV generated per source time-series
- unified feature schema with 42 columns
- labeled benchmark branch preserved for NAB
- unlabeled baseline branch preserved for UK-DALE and AMPds2

Next step:
- perform feature-space audit and combined Phase 3 validation
- then proceed to Phase 4 modeling on the generated window-level feature tables
"""

with open(log_path, "a", encoding="utf-8") as f:
    f.write(final_note)

print("Updated log:", log_path)

In [ ]:
audit_rows = []

for dataset_name, folder in {
    "nab": os.path.join(PH3_PATH, "NAB", "raw_windows", "features"),
    "ukdale": os.path.join(PH3_PATH, "UK_DALE", "windows", "features"),
    "ampds2": os.path.join(PH3_PATH, "AMPds2", "windows", "features"),
}.items():
    files = [f for f in os.listdir(folder) if f.endswith("_feature_windows.csv")]
    if not files:
        continue

    sample_file = os.path.join(folder, files[0])
    df = pd.read_csv(sample_file)

    audit_rows.append({
        "dataset_name": dataset_name,
        "sample_file": files[0],
        "rows": len(df),
        "columns": len(df.columns),
        "has_nulls": int(df.isna().sum().sum()),
        "label_counts": df["label"].value_counts(dropna=False).to_dict() if "label" in df.columns else {}
    })

audit_df = pd.DataFrame(audit_rows)
audit_path = os.path.join(PH3_PATH, "combined", "ph3_feature_window_audit.csv")
audit_df.to_csv(audit_path, index=False)

print("Saved audit:", audit_path)
print(audit_df)

In [ ]:
import os
import pandas as pd
from glob import glob

BASE_PATH = "/content/drive/MyDrive/Master/Thesis/Code"
PH3_PATH = os.path.join(BASE_PATH, "processed_data", "PH3")

paths_to_check = {
    "PH3 main combined summary": os.path.join(PH3_PATH, "combined", "ph3_main_baseline_feature_generation_summary.csv"),
    "PH3 feature audit": os.path.join(PH3_PATH, "combined", "ph3_feature_window_audit.csv"),
    "PH3 log": os.path.join(PH3_PATH, "logs", "PH3_LOG_002_window_generation.md"),

    "NAB summary": os.path.join(PH3_PATH, "NAB", "raw_windows", "features", "nab_feature_generation_summary.csv"),
    "UK-DALE summary": os.path.join(PH3_PATH, "UK_DALE", "windows", "features", "ukdale_feature_generation_summary.csv"),
    "AMPds2 summary": os.path.join(PH3_PATH, "AMPds2", "windows", "features", "ampds2_feature_generation_summary.csv"),

    "NAB failed log": os.path.join(PH3_PATH, "NAB", "raw_windows", "features", "nab_feature_generation_failed.csv"),
}

print("=== FILE EXISTENCE CHECK ===")
for label, path in paths_to_check.items():
    print(f"{label}: {os.path.exists(path)}")
    if os.path.exists(path):
        print("   ", path)

print("\n=== FEATURE FILE COUNTS ===")
nab_feature_files = sorted(glob(os.path.join(PH3_PATH, "NAB", "raw_windows", "features", "*_feature_windows.csv")))
ukdale_feature_files = sorted(glob(os.path.join(PH3_PATH, "UK_DALE", "windows", "features", "*_feature_windows.csv")))
ampds2_feature_files = sorted(glob(os.path.join(PH3_PATH, "AMPds2", "windows", "features", "*_feature_windows.csv")))

print("NAB feature files:", len(nab_feature_files))
print("UK-DALE feature files:", len(ukdale_feature_files))
print("AMPds2 feature files:", len(ampds2_feature_files))

print("\n=== SUMMARY CONTENT CHECK ===")
combined_summary = pd.read_csv(paths_to_check["PH3 main combined summary"])
audit_df = pd.read_csv(paths_to_check["PH3 feature audit"])
nab_summary = pd.read_csv(paths_to_check["NAB summary"])
ukdale_summary = pd.read_csv(paths_to_check["UK-DALE summary"])
ampds2_summary = pd.read_csv(paths_to_check["AMPds2 summary"])

print("Combined summary rows:", len(combined_summary))
print(combined_summary["dataset_name"].value_counts())
print("\nCombined status counts:")
print(combined_summary["status"].value_counts(dropna=False))

print("\nAudit table:")
print(audit_df)

print("\nNAB status counts:")
print(nab_summary["status"].value_counts(dropna=False))

print("\nUK-DALE status counts:")
print(ukdale_summary["status"].value_counts(dropna=False))

print("\nAMPds2 status counts:")
print(ampds2_summary["status"].value_counts(dropna=False))

print("\n=== SAMPLE OUTPUT FILE CHECK ===")
sample_paths = {
    "NAB sample": nab_feature_files[0] if nab_feature_files else None,
    "UK-DALE sample": ukdale_feature_files[0] if ukdale_feature_files else None,
    "AMPds2 sample": ampds2_feature_files[0] if ampds2_feature_files else None,
}

for label, sample_path in sample_paths.items():
    print(f"\n{label}:")
    if sample_path is None:
        print("  Missing")
        continue

    df = pd.read_csv(sample_path, nrows=5)
    print("  File:", sample_path)
    print("  Columns:", len(df.columns))
    print("  Column names:", df.columns.tolist())
    print("  Head:")
    print(df.head())

print("\n=== FINAL CHECKPOINT STATUS ===")
ok = (
    os.path.exists(paths_to_check["PH3 main combined summary"]) and
    os.path.exists(paths_to_check["PH3 feature audit"]) and
    os.path.exists(paths_to_check["PH3 log"]) and
    len(nab_feature_files) == 51 and
    len(ukdale_feature_files) == 111 and
    len(ampds2_feature_files) == 231
)

print("PH3 main baseline saved correctly:", ok)

In [ ]:
# =========================
# PH4 - CELL 1: Mount + paths + folders
# =========================

from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Base paths from the approved Phase 4 handoff
BASE_PATH = "/content/drive/MyDrive/Master/Thesis/Code"
PROCESSED_DATA_PATH = f"{BASE_PATH}/processed_data"
PH3_PATH = f"{PROCESSED_DATA_PATH}/PH3"
PH4_PATH = f"{PROCESSED_DATA_PATH}/PH4"

EXPERIMENT_LOGS_PATH = f"{BASE_PATH}/experiment_logs"
PH4_LOGS_PATH = f"{EXPERIMENT_LOGS_PATH}/PH4"

# Phase 3 saved inputs
PH3_COMBINED_SUMMARY = f"{PH3_PATH}/combined/ph3_main_baseline_feature_generation_summary.csv"
PH3_FEATURE_AUDIT = f"{PH3_PATH}/combined/ph3_feature_window_audit.csv"

NAB_FEATURES_DIR = f"{PH3_PATH}/NAB/raw_windows/features"
UKDALE_FEATURES_DIR = f"{PH3_PATH}/UK_DALE/windows/features"
AMPDS2_FEATURES_DIR = f"{PH3_PATH}/AMPds2/windows/features"

NAB_FEATURE_SUMMARY = f"{NAB_FEATURES_DIR}/nab_feature_generation_summary.csv"
UKDALE_FEATURE_SUMMARY = f"{UKDALE_FEATURES_DIR}/ukdale_feature_generation_summary.csv"
AMPDS2_FEATURE_SUMMARY = f"{AMPDS2_FEATURES_DIR}/ampds2_feature_generation_summary.csv"

NAB_FAILED_LOG = f"{NAB_FEATURES_DIR}/nab_feature_generation_failed.csv"

# Phase 4 output structure
PH4_DIRS = [
    f"{PH4_PATH}/supervised/NAB/inputs",
    f"{PH4_PATH}/supervised/NAB/splits",
    f"{PH4_PATH}/supervised/NAB/models",
    f"{PH4_PATH}/supervised/NAB/predictions",
    f"{PH4_PATH}/supervised/NAB/metrics",
    f"{PH4_PATH}/supervised/BLUED",  # deferred branch
    f"{PH4_PATH}/unsupervised/NAB",
    f"{PH4_PATH}/unsupervised/UK_DALE",
    f"{PH4_PATH}/unsupervised/AMPds2",
    f"{PH4_PATH}/combined",
    f"{PH4_PATH}/configs",
    PH4_LOGS_PATH,
]

for d in PH4_DIRS:
    os.makedirs(d, exist_ok=True)

print("Phase 4 folders are ready.")
for d in PH4_DIRS:
    print("-", d)

In [ ]:
# =========================
# PH4 - CELL 2: Imports + NAB feature-window loading
# =========================

!pip -q install xgboost

import glob
import json
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import joblib

RANDOM_STATE = 42
TARGET_COL = "label"

METADATA_COLS = [
    "dataset_name",
    "series_id",
    "window_id",
    "start_time",
    "end_time",
    "window_size_minutes",
    "step_size_minutes",
    "sampling_minutes",
]

EXCLUDE_FILE_KEYWORDS = [
    "summary",
    "failed",
    "audit",
    "metrics",
    "predictions",
    "splits",
]

def discover_feature_files(root_dir):
    all_csvs = glob.glob(os.path.join(root_dir, "*.csv"))
    feature_csvs = []
    for f in all_csvs:
        name = os.path.basename(f).lower()
        if not any(k in name for k in EXCLUDE_FILE_KEYWORDS):
            feature_csvs.append(f)
    return sorted(feature_csvs)

nab_feature_files = discover_feature_files(NAB_FEATURES_DIR)
print(f"NAB feature-window files found: {len(nab_feature_files)}")

if len(nab_feature_files) == 0:
    raise FileNotFoundError("No NAB feature-window CSVs found in the Phase 3 NAB features folder.")

nab_dfs = []
for fp in nab_feature_files:
    df = pd.read_csv(fp)
    df["source_file"] = os.path.basename(fp)
    nab_dfs.append(df)

nab_df = pd.concat(nab_dfs, ignore_index=True)

# Basic type cleanup
for c in ["start_time", "end_time"]:
    if c in nab_df.columns:
        nab_df[c] = pd.to_datetime(nab_df[c], errors="coerce")

# Label cleanup
if TARGET_COL not in nab_df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found.")

nab_df = nab_df.dropna(subset=[TARGET_COL]).copy()
nab_df[TARGET_COL] = nab_df[TARGET_COL].astype(int)
nab_df = nab_df[nab_df[TARGET_COL].isin([0, 1])].copy()

print("Loaded NAB dataframe shape:", nab_df.shape)
print("Columns:", list(nab_df.columns))
print("Label distribution:")
print(nab_df[TARGET_COL].value_counts(dropna=False))
print("Unique series count:", nab_df["series_id"].nunique() if "series_id" in nab_df.columns else "series_id missing")

In [ ]:
# =========================
# PH4 - CELL 3: Purged chronological split per series
# =========================

def purged_chronological_split(df, target_col="label", train_ratio=0.70):
    required_cols = ["series_id", "start_time", "end_time", target_col]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns for split: {missing}")

    split_frames = []

    for series_id, g in df.groupby("series_id", sort=False):
        g = g.sort_values("start_time").reset_index(drop=True)

        n = len(g)
        if n < 3:
            tmp = g.copy()
            tmp["split"] = "dropped_too_short"
            split_frames.append(tmp)
            continue

        cut_idx = max(1, int(np.floor(train_ratio * n)))
        cut_idx = min(cut_idx, n - 1)

        train_part = g.iloc[:cut_idx].copy()
        test_part  = g.iloc[cut_idx:].copy()

        last_train_end = train_part["end_time"].max()
        keep_test_mask = test_part["start_time"] >= last_train_end

        purged_test = test_part.loc[keep_test_mask].copy()
        dropped_purge = test_part.loc[~keep_test_mask].copy()

        train_part["split"] = "train"
        purged_test["split"] = "test"
        dropped_purge["split"] = "dropped_purge"

        split_frames.extend([train_part, purged_test, dropped_purge])

    out = pd.concat(split_frames, ignore_index=True)
    return out

split_df = purged_chronological_split(nab_df, target_col=TARGET_COL, train_ratio=0.70)

# Save split audit
split_audit_path = f"{PH4_PATH}/supervised/NAB/splits/nab_supervised_split_audit.csv"
split_df.to_csv(split_audit_path, index=False)

train_df = split_df[split_df["split"] == "train"].copy()
test_df  = split_df[split_df["split"] == "test"].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Dropped purge rows:", (split_df["split"] == "dropped_purge").sum())
print("Train label counts:")
print(train_df[TARGET_COL].value_counts())
print("Test label counts:")
print(test_df[TARGET_COL].value_counts())

# Feature schema
candidate_features = [
    c for c in train_df.columns
    if c not in (METADATA_COLS + [TARGET_COL, "source_file", "split"])
]

numeric_feature_cols = [
    c for c in candidate_features
    if pd.api.types.is_numeric_dtype(train_df[c])
]

# Drop all-NaN or constant columns from train
usable_feature_cols = []
for c in numeric_feature_cols:
    series = train_df[c]
    if series.notna().sum() == 0:
        continue
    if series.nunique(dropna=True) <= 1:
        continue
    usable_feature_cols.append(c)

print(f"Usable feature count: {len(usable_feature_cols)}")
print("First 20 usable features:", usable_feature_cols[:20])

feature_schema = {
    "dataset": "NAB",
    "target_col": TARGET_COL,
    "metadata_cols": METADATA_COLS,
    "excluded_non_feature_cols": ["source_file", "split"],
    "feature_cols": usable_feature_cols,
    "split_strategy": "per-series purged chronological holdout",
    "train_ratio": 0.70,
}

schema_path = f"{PH4_PATH}/configs/nab_supervised_feature_schema.json"
with open(schema_path, "w") as f:
    json.dump(feature_schema, f, indent=2)

X_train = train_df[usable_feature_cols].copy()
y_train = train_df[TARGET_COL].copy()
X_test  = test_df[usable_feature_cols].copy()
y_test  = test_df[TARGET_COL].copy()

# Save model inputs for reproducibility
train_input_path = f"{PH4_PATH}/supervised/NAB/inputs/nab_train_inputs.csv"
test_input_path = f"{PH4_PATH}/supervised/NAB/inputs/nab_test_inputs.csv"

train_df.to_csv(train_input_path, index=False)
test_df.to_csv(test_input_path, index=False)

print("Saved:")
print("-", split_audit_path)
print("-", schema_path)
print("-", train_input_path)
print("-", test_input_path)

In [ ]:
# =========================
# PH4 - CELL 4: Train baseline supervised models on NAB
# =========================

from sklearn.base import clone

pos_count = int((y_train == 1).sum())
neg_count = int((y_train == 0).sum())
scale_pos_weight = (neg_count / pos_count) if pos_count > 0 else 1.0

models = {
    "RandomForest": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=1,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
    "LogisticRegression": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),
    "XGBoost": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
}

metrics_rows = []
all_predictions = []

for model_name, model in models.items():
    print(f"\nTraining {model_name} ...")
    fitted = clone(model)
    fitted.fit(X_train, y_train)

    # score/probability
    if hasattr(fitted, "predict_proba"):
        y_score = fitted.predict_proba(X_test)[:, 1]
    else:
        y_score = fitted.decision_function(X_test)

    y_pred = (y_score >= 0.5).astype(int)

    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    try:
        roc_auc = roc_auc_score(y_test, y_score)
    except Exception:
        roc_auc = np.nan

    metrics_rows.append({
        "dataset": "NAB",
        "model_name": model_name,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "n_features": len(usable_feature_cols),
        "positive_train": int((y_train == 1).sum()),
        "positive_test": int((y_test == 1).sum()),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    })

    pred_df = test_df[[
        c for c in test_df.columns
        if c in ["dataset_name", "series_id", "window_id", "start_time", "end_time", "source_file", TARGET_COL]
    ]].copy()
    pred_df["y_true"] = y_test.values
    pred_df["y_score"] = y_score
    pred_df["y_pred"] = y_pred
    pred_df["model_name"] = model_name

    all_predictions.append(pred_df)

    model_path = f"{PH4_PATH}/supervised/NAB/models/{model_name}_nab_baseline.joblib"
    joblib.dump(fitted, model_path)
    print(f"Saved model -> {model_path}")

metrics_df = pd.DataFrame(metrics_rows).sort_values("f1", ascending=False).reset_index(drop=True)
predictions_df = pd.concat(all_predictions, ignore_index=True)

metrics_path = f"{PH4_PATH}/supervised/NAB/metrics/nab_supervised_model_metrics.csv"
predictions_path = f"{PH4_PATH}/supervised/NAB/predictions/nab_supervised_test_predictions.csv"

metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_csv(predictions_path, index=False)

print("\n=== METRICS ===")
print(metrics_df)

print("\nSaved:")
print("-", metrics_path)
print("-", predictions_path)

In [ ]:
# =========================
# PH4 - CELL 5: Write Phase 4 logs
# =========================

from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

log1 = f"""# PH4_LOG_001_supervised_baseline_design

## Timestamp
{timestamp}

## Decision
First supervised Phase 4 baseline starts with NAB.

## Justification
- BLUED is deferred to a later short-window branch.
- NAB is already available from the validated main Phase 3 baseline.
- Goal is to validate supervised modeling pipeline before unsupervised expansion.

## Models
- Random Forest
- XGBoost
- Logistic Regression

## Split strategy
- Per-series chronological split
- 70% train / 30% test
- Purged boundary overlap using start_time and end_time

## Target
- label

## Feature schema
- Exclude metadata columns:
  dataset_name, series_id, window_id, start_time, end_time,
  window_size_minutes, step_size_minutes, sampling_minutes
- Use remaining numeric engineered features

## Evaluation metrics
- Precision
- Recall
- F1-score
- ROC-AUC

## Output folders
- supervised/NAB/inputs
- supervised/NAB/splits
- supervised/NAB/models
- supervised/NAB/predictions
- supervised/NAB/metrics
"""

with open(f"{PH4_LOGS_PATH}/PH4_LOG_001_supervised_baseline_design.md", "w") as f:
    f.write(log1)

log2 = f"""# PH4_LOG_002_supervised_nab_baseline

## Timestamp
{timestamp}

## Dataset
NAB

## Train/Test sizes
- Train rows: {len(train_df)}
- Test rows: {len(test_df)}

## Label counts
### Train
{y_train.value_counts().to_string()}

### Test
{y_test.value_counts().to_string()}

## Feature count
{len(usable_feature_cols)}

## Feature columns
{chr(10).join(['- ' + c for c in usable_feature_cols])}

## Metrics
{metrics_df.to_markdown(index=False)}
"""

with open(f"{PH4_LOGS_PATH}/PH4_LOG_002_supervised_nab_baseline.md", "w") as f:
    f.write(log2)

print("Phase 4 logs written successfully.")

In [ ]:
# PH4 verification checkpoint

print("=== Metrics ===")
display(metrics_df)

print("\n=== Train/Test Shapes ===")
print("Train:", train_df.shape)
print("Test:", test_df.shape)

print("\n=== Label Counts ===")
print("Train labels:")
print(y_train.value_counts())

print("\nTest labels:")
print(y_test.value_counts())

print("\n=== Saved Outputs Check ===")
paths_to_check = [
    f"{PH4_PATH}/supervised/NAB/metrics/nab_supervised_model_metrics.csv",
    f"{PH4_PATH}/supervised/NAB/predictions/nab_supervised_test_predictions.csv",
    f"{PH4_PATH}/supervised/NAB/splits/nab_supervised_split_audit.csv",
    f"{PH4_PATH}/configs/nab_supervised_feature_schema.json",
    f"{PH4_LOGS_PATH}/PH4_LOG_001_supervised_baseline_design.md",
    f"{PH4_LOGS_PATH}/PH4_LOG_002_supervised_nab_baseline.md",
]

for p in paths_to_check:
    print("FOUND" if os.path.exists(p) else "MISSING", "->", p)

In [ ]:
from sklearn.metrics import precision_recall_curve

def find_best_threshold(y_true, y_scores):
    precision, recall, thresholds = precision_recall_curve(y_true, y_scores)

    f1_scores = (2 * precision * recall) / (precision + recall + 1e-10)

    best_idx = f1_scores.argmax()

    return {
        "best_threshold": thresholds[best_idx],
        "best_f1": f1_scores[best_idx],
        "precision": precision[best_idx],
        "recall": recall[best_idx]
    }

threshold_results = []

for model_name in predictions_df["model_name"].unique():
    df_m = predictions_df[predictions_df["model_name"] == model_name]

    res = find_best_threshold(df_m["y_true"], df_m["y_score"])
    res["model_name"] = model_name

    threshold_results.append(res)

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df)

In [ ]:
# =========================
# PH4 - LOG SUPERVISED NAB BASELINE WITH THRESHOLD TUNING
# =========================

from datetime import datetime
import os

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

log_path = f"{PH4_LOGS_PATH}/PH4_LOG_002_supervised_nab_baseline.md"

log_content = f"""# PH4_LOG_002_supervised_nab_baseline

## Experiment ID
PH4_SUP_NAB_001

## Timestamp
{timestamp}

## Experiment Title
Supervised Baseline Modeling on NAB Feature Windows

## Objective
To validate the Phase 4 supervised modeling pipeline using the already generated Phase 3 NAB feature-window dataset.

The objective was not to achieve final thesis-level performance, but to verify:
- feature-window compatibility
- chronological splitting
- model training
- prediction generation
- metric computation
- threshold-based decision evaluation

## Dataset
Numenta Anomaly Benchmark (NAB)

## Input Source
Phase 3 saved feature-window outputs:

`{NAB_FEATURES_DIR}`

## Phase 3 Baseline Status
The Phase 3 main baseline is treated as closed and was not regenerated.

## Window Configuration
- Window size: 60 minutes
- Step size: 30 minutes
- Overlap: 50%

## Feature Schema
Target column:

`label`

Excluded metadata columns:
- dataset_name
- series_id
- window_id
- start_time
- end_time
- window_size_minutes
- step_size_minutes
- sampling_minutes
- source_file
- split

Number of usable numeric features:

{len(usable_feature_cols)}

Feature columns:
{chr(10).join(["- " + c for c in usable_feature_cols])}

## Split Strategy
A per-series chronological split was used.

Initial split:
- 70% training
- 30% testing

A purging step was applied at the train/test boundary to reduce overlap leakage caused by the 60-minute window and 30-minute step.

## Train/Test Summary
Training rows:

{len(train_df)}

Testing rows:

{len(test_df)}

Training label distribution:

{y_train.value_counts().to_string()}

Testing label distribution:

{y_test.value_counts().to_string()}

## Models
The following supervised models were trained:
- Random Forest
- Logistic Regression
- XGBoost

## Initial Evaluation
The first evaluation used the default threshold of 0.5.

Initial model metrics:

{metrics_df.to_markdown(index=False)}

## Threshold Optimization
Because the anomaly class was extremely rare, threshold tuning was applied using the precision-recall curve.

Threshold tuning selected the threshold that maximized F1-score for each model.

Threshold optimization results:

{threshold_df.to_markdown(index=False)}

## Interpretation
The supervised baseline executed successfully, but the results show that supervised classification is limited under the current NAB window-label formulation.

The anomaly class is extremely sparse:
- Training positives: {int((y_train == 1).sum())}
- Testing positives: {int((y_test == 1).sum())}

This creates a severe class imbalance problem.

The results indicate that:
- Logistic Regression achieved relatively high recall but very low precision.
- Random Forest produced a more conservative decision pattern.
- XGBoost achieved the best tuned F1-score among the tested models.

However, the overall F1-scores remained low after threshold optimization.

This suggests that the main limitation is not code failure, but the weak supervision problem introduced by converting sparse timestamp-level anomalies into 60-minute window-level labels.

## Methodological Conclusion
The supervised baseline is valid as a pipeline validation experiment.

It confirms that:
- Phase 3 feature windows can be used for modeling.
- Chronological splitting works.
- Model training and prediction export work.
- Evaluation metrics are generated correctly.
- Threshold optimization is necessary for anomaly detection under class imbalance.

However, supervised learning is not sufficient as the main modeling strategy under the current NAB label sparsity.

This supports moving to unsupervised anomaly detection models, which better align with the thesis scope and the lack of reliable anomaly labels in real smart home datasets.

## Limitations
- NAB is not a smart-home-specific dataset.
- Labels are sparse at the window level.
- The 60-minute window may dilute short anomaly events.
- No hyperparameter optimization was performed.
- No synthetic anomaly injection was used in this experiment.
- BLUED was not used because it is deferred to a later short-window branch.

## Decision
Close PH4_SUP_NAB_001 as the supervised NAB baseline experiment.

Proceed to unsupervised anomaly detection on NAB using:
- Isolation Forest
- One-Class SVM
- Local Outlier Factor
- Autoencoder

## Saved Outputs
Metrics:

`{PH4_PATH}/supervised/NAB/metrics/nab_supervised_model_metrics.csv`

Predictions:

`{PH4_PATH}/supervised/NAB/predictions/nab_supervised_test_predictions.csv`

Split audit:

`{PH4_PATH}/supervised/NAB/splits/nab_supervised_split_audit.csv`

Feature schema:

`{PH4_PATH}/configs/nab_supervised_feature_schema.json`

Models folder:

`{PH4_PATH}/supervised/NAB/models`
"""

with open(log_path, "w", encoding="utf-8") as f:
    f.write(log_content)

print("Saved experiment log:")
print(log_path)

In [ ]:
# =========================
# PH4 - Progress bar setup
# =========================

from tqdm.auto import tqdm
import time

tqdm.pandas()

print("Progress bar support is ready.")

In [ ]:
# =========================
# PH4_UNSUP_NAB_001 - Isolation Forest (Per-Series)
# =========================

from sklearn.ensemble import IsolationForest
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import os

# Output storage
all_results = []

series_ids = split_df["series_id"].unique()

print(f"Total series: {len(series_ids)}")

for series_id in tqdm(series_ids, desc="Isolation Forest per-series"):

    # --- Split per series ---
    df_series = split_df[split_df["series_id"] == series_id]

    train_series = df_series[df_series["split"] == "train"]
    test_series  = df_series[df_series["split"] == "test"]

    # Skip empty
    if len(train_series) == 0 or len(test_series) == 0:
        continue

    # --- TRAIN ONLY ON NORMAL DATA ---
    train_normal = train_series[train_series["label"] == 0]

    # Safety check
    if len(train_normal) < 50:
        continue

    X_train = train_normal[usable_feature_cols]
    X_test  = test_series[usable_feature_cols]
    y_test  = test_series["label"]

    # --- Model ---
    model = IsolationForest(
        n_estimators=100,
        contamination=0.01,   # small anomaly assumption
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train)

    # --- Scores ---
    scores = -model.decision_function(X_test)  # higher = more anomalous

    # Store results
    df_out = test_series.copy()
    df_out["anomaly_score"] = scores
    df_out["model"] = "IsolationForest"

    all_results.append(df_out)

# Combine all
unsup_results_df = pd.concat(all_results, ignore_index=True)

# Save raw scores
unsup_scores_path = f"{PH4_PATH}/unsupervised/NAB/isolation_forest_scores.csv"
unsup_results_df.to_csv(unsup_scores_path, index=False)

print("Saved:", unsup_scores_path)
print("Shape:", unsup_results_df.shape)

In [ ]:
# =========================
# PH4_UNSUP_NAB_001 - Evaluation
# =========================

from sklearn.metrics import precision_recall_curve, f1_score, precision_score, recall_score, roc_auc_score

y_true = unsup_results_df["label"].values
y_score = unsup_results_df["anomaly_score"].values

# --- Find best threshold ---
precision, recall, thresholds = precision_recall_curve(y_true, y_score)
f1_scores = (2 * precision * recall) / (precision + recall + 1e-10)

best_idx = np.argmax(f1_scores)

best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]
best_precision = precision[best_idx]
best_recall = recall[best_idx]

# --- Apply threshold ---
y_pred = (y_score >= best_threshold).astype(int)

roc_auc = roc_auc_score(y_true, y_score)

print("=== Isolation Forest Results ===")
print("Best Threshold:", best_threshold)
print("F1:", best_f1)
print("Precision:", best_precision)
print("Recall:", best_recall)
print("ROC-AUC:", roc_auc)

In [ ]:
# =========================
# Save metrics
# =========================

metrics_unsup_df = pd.DataFrame([{
    "dataset": "NAB",
    "model": "IsolationForest",
    "best_threshold": best_threshold,
    "f1": best_f1,
    "precision": best_precision,
    "recall": best_recall,
    "roc_auc": roc_auc
}])

metrics_path = f"{PH4_PATH}/unsupervised/NAB/isolation_forest_metrics.csv"
metrics_unsup_df.to_csv(metrics_path, index=False)

print("Saved:", metrics_path)
display(metrics_unsup_df)

In [ ]:
# =========================
# PH4 LOG - Unsupervised NAB Isolation Forest
# =========================

from datetime import datetime

log_path = f"{PH4_LOGS_PATH}/PH4_LOG_004_unsupervised_nab.md"

log = f"""# PH4_LOG_004_unsupervised_nab

## Experiment ID
PH4_UNSUP_NAB_001

## Timestamp
{datetime.now()}

## Model
Isolation Forest

## Strategy
Per-series modeling

## Training
- Used ONLY normal windows (label = 0)
- Labels NOT used during training

## Testing
- Full test set (normal + anomaly)

## Results
F1: {best_f1}
Precision: {best_precision}
Recall: {best_recall}
ROC-AUC: {roc_auc}

## Interpretation
Unsupervised anomaly detection is more suitable than supervised learning under sparse anomaly conditions.

Isolation Forest captures anomaly structure without relying on labels.

## Decision
Proceed with:
- One-Class SVM
- LOF
"""

with open(log_path, "w") as f:
    f.write(log)

print("Log saved:", log_path)

In [ ]:
# =========================
# Append final interpretation to PH4_UNSUP_NAB_001 log
# =========================

append_text = f"""

## Final Interpretation Addendum

Isolation Forest produced a strong improvement over the supervised NAB baseline.

Comparison against the best supervised model:
- Best supervised model: XGBoost
- Supervised F1-score: approximately 0.047
- Supervised ROC-AUC: approximately 0.68
- Isolation Forest F1-score: {best_f1:.6f}
- Isolation Forest ROC-AUC: {roc_auc:.6f}

This indicates that the unsupervised formulation is more suitable for the current problem setting.

The improvement is methodologically important because NAB anomaly labels are sparse at the window level. Supervised classifiers depend directly on these weak labels, while Isolation Forest learns the normal behavior distribution and detects deviations without using labels during training.

The high ROC-AUC indicates that the model has strong anomaly-ranking ability, while the relatively low precision remains expected because the anomaly class is extremely rare.

This result supports the thesis direction: anomaly detection in smart home energy systems should primarily be modeled using unsupervised methods, especially when real anomaly labels are sparse, incomplete, or unavailable.

## Status
PH4_UNSUP_NAB_001 is closed successfully.

Next experiment:
PH4_UNSUP_NAB_002 — One-Class SVM using the same per-series unsupervised protocol.
"""

log_path = f"{PH4_LOGS_PATH}/PH4_LOG_004_unsupervised_nab.md"

with open(log_path, "a", encoding="utf-8") as f:
    f.write(append_text)

print("Updated log:")
print(log_path)

In [ ]:
# =========================
# PH4_UNSUP_NAB_002 - One-Class SVM Per-Series
# =========================

from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import os

all_ocsvm_results = []

series_ids = split_df["series_id"].unique()

print(f"Total series: {len(series_ids)}")

for series_id in tqdm(series_ids, desc="One-Class SVM per-series"):

    df_series = split_df[split_df["series_id"] == series_id]

    train_series = df_series[df_series["split"] == "train"]
    test_series  = df_series[df_series["split"] == "test"]

    if len(train_series) == 0 or len(test_series) == 0:
        continue

    # Train without labels: normal windows only
    train_normal = train_series[train_series["label"] == 0]

    if len(train_normal) < 50:
        continue

    X_train = train_normal[usable_feature_cols]
    X_test  = test_series[usable_feature_cols]

    model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ocsvm", OneClassSVM(
            kernel="rbf",
            gamma="scale",
            nu=0.01
        ))
    ])

    model.fit(X_train)

    # decision_function: lower = more anomalous
    scores = -model.decision_function(X_test)

    df_out = test_series.copy()
    df_out["anomaly_score"] = scores
    df_out["model"] = "OneClassSVM"

    all_ocsvm_results.append(df_out)

ocsvm_results_df = pd.concat(all_ocsvm_results, ignore_index=True)

ocsvm_scores_path = f"{PH4_PATH}/unsupervised/NAB/one_class_svm_scores.csv"
ocsvm_results_df.to_csv(ocsvm_scores_path, index=False)

print("Saved:", ocsvm_scores_path)
print("Shape:", ocsvm_results_df.shape)

In [ ]:
# =========================
# PH4_UNSUP_NAB_002 - Evaluation
# =========================

from sklearn.metrics import precision_recall_curve, roc_auc_score

y_true = ocsvm_results_df["label"].values
y_score = ocsvm_results_df["anomaly_score"].values

precision, recall, thresholds = precision_recall_curve(y_true, y_score)
f1_scores = (2 * precision * recall) / (precision + recall + 1e-10)

best_idx = np.argmax(f1_scores)

best_threshold_ocsvm = thresholds[best_idx]
best_f1_ocsvm = f1_scores[best_idx]
best_precision_ocsvm = precision[best_idx]
best_recall_ocsvm = recall[best_idx]
roc_auc_ocsvm = roc_auc_score(y_true, y_score)

ocsvm_metrics_df = pd.DataFrame([{
    "dataset": "NAB",
    "model": "OneClassSVM",
    "best_threshold": best_threshold_ocsvm,
    "f1": best_f1_ocsvm,
    "precision": best_precision_ocsvm,
    "recall": best_recall_ocsvm,
    "roc_auc": roc_auc_ocsvm
}])

ocsvm_metrics_path = f"{PH4_PATH}/unsupervised/NAB/one_class_svm_metrics.csv"
ocsvm_metrics_df.to_csv(ocsvm_metrics_path, index=False)

print("Saved:", ocsvm_metrics_path)
display(ocsvm_metrics_df)

In [ ]:
# =========================
# PH4 Master Results Tracker
# =========================

import os
import pandas as pd
from datetime import datetime

master_results_path = f"{PH4_PATH}/combined/ph4_experiment_results_tracker.xlsx"

rows = []

# 1) Supervised default-threshold metrics
if "metrics_df" in globals():
    for _, r in metrics_df.iterrows():
        rows.append({
            "experiment_id": "PH4_SUP_NAB_001",
            "phase": "PH4",
            "dataset": "NAB",
            "model": r["model_name"],
            "modeling_type": "Supervised",
            "protocol": "Per-series purged chronological split; default threshold = 0.5",
            "train_label_usage": "Labels used for supervised training",
            "evaluation_label_usage": "Labels used for evaluation",
            "threshold_type": "Default 0.5",
            "best_threshold": 0.5,
            "precision": r["precision"],
            "recall": r["recall"],
            "f1": r["f1"],
            "roc_auc": r["roc_auc"],
            "n_train": r["n_train"],
            "n_test": r["n_test"],
            "n_features": r["n_features"],
            "notes": "Initial supervised baseline; weak due to sparse window-level labels."
        })

# 2) Supervised threshold-tuned metrics
if "threshold_df" in globals() and "metrics_df" in globals():
    roc_lookup = dict(zip(metrics_df["model_name"], metrics_df["roc_auc"]))
    for _, r in threshold_df.iterrows():
        rows.append({
            "experiment_id": "PH4_SUP_NAB_001_TUNED",
            "phase": "PH4",
            "dataset": "NAB",
            "model": r["model_name"],
            "modeling_type": "Supervised",
            "protocol": "Per-series purged chronological split; threshold optimized by F1",
            "train_label_usage": "Labels used for supervised training",
            "evaluation_label_usage": "Labels used for evaluation",
            "threshold_type": "F1-optimized",
            "best_threshold": r["best_threshold"],
            "precision": r["precision"],
            "recall": r["recall"],
            "f1": r["best_f1"],
            "roc_auc": roc_lookup.get(r["model_name"], None),
            "n_train": len(train_df),
            "n_test": len(test_df),
            "n_features": len(usable_feature_cols),
            "notes": "Threshold tuning improved F1 but performance remained limited by label sparsity."
        })

# 3) Isolation Forest
if os.path.exists(f"{PH4_PATH}/unsupervised/NAB/isolation_forest_metrics.csv"):
    iso_df = pd.read_csv(f"{PH4_PATH}/unsupervised/NAB/isolation_forest_metrics.csv")
    for _, r in iso_df.iterrows():
        rows.append({
            "experiment_id": "PH4_UNSUP_NAB_001",
            "phase": "PH4",
            "dataset": "NAB",
            "model": r["model"],
            "modeling_type": "Unsupervised",
            "protocol": "Per-series; trained on normal windows only; labels used only for evaluation",
            "train_label_usage": "Labels used only to select normal training windows",
            "evaluation_label_usage": "Labels used only after scoring",
            "threshold_type": "F1-optimized",
            "best_threshold": r["best_threshold"],
            "precision": r["precision"],
            "recall": r["recall"],
            "f1": r["f1"],
            "roc_auc": r["roc_auc"],
            "n_train": len(train_df),
            "n_test": len(test_df),
            "n_features": len(usable_feature_cols),
            "notes": "Strongest result so far; high ROC-AUC and better F1 than supervised models."
        })

# 4) One-Class SVM
if os.path.exists(f"{PH4_PATH}/unsupervised/NAB/one_class_svm_metrics.csv"):
    ocsvm_df = pd.read_csv(f"{PH4_PATH}/unsupervised/NAB/one_class_svm_metrics.csv")
    for _, r in ocsvm_df.iterrows():
        rows.append({
            "experiment_id": "PH4_UNSUP_NAB_002",
            "phase": "PH4",
            "dataset": "NAB",
            "model": r["model"],
            "modeling_type": "Unsupervised",
            "protocol": "Per-series; trained on normal windows only; labels used only for evaluation",
            "train_label_usage": "Labels used only to select normal training windows",
            "evaluation_label_usage": "Labels used only after scoring",
            "threshold_type": "F1-optimized",
            "best_threshold": r["best_threshold"],
            "precision": r["precision"],
            "recall": r["recall"],
            "f1": r["f1"],
            "roc_auc": r["roc_auc"],
            "n_train": len(train_df),
            "n_test": len(test_df),
            "n_features": len(usable_feature_cols),
            "notes": "Good ROC-AUC but weaker recall and F1 than Isolation Forest."
        })

results_tracker_df = pd.DataFrame(rows)

summary_df = (
    results_tracker_df
    .sort_values(["dataset", "f1"], ascending=[True, False])
    .reset_index(drop=True)
)

best_by_dataset_df = (
    results_tracker_df
    .sort_values(["dataset", "f1"], ascending=[True, False])
    .groupby("dataset", as_index=False)
    .first()
)

with pd.ExcelWriter(master_results_path, engine="openpyxl") as writer:
    results_tracker_df.to_excel(writer, sheet_name="All_Experiments", index=False)
    summary_df.to_excel(writer, sheet_name="Ranked_By_F1", index=False)
    best_by_dataset_df.to_excel(writer, sheet_name="Best_By_Dataset", index=False)

print("Master Excel tracker saved:")
print(master_results_path)

display(summary_df)

In [ ]:
# =========================
# Log PH4_UNSUP_NAB_002 after reviewing results
# =========================

from datetime import datetime

log_path = f"{PH4_LOGS_PATH}/PH4_LOG_004_unsupervised_nab.md"

log_append = f"""

---

# PH4_UNSUP_NAB_002

## Timestamp
{datetime.now()}

## Experiment Title
Unsupervised One-Class SVM on NAB Feature Windows

## Objective
To evaluate One-Class SVM as a per-series unsupervised anomaly detection model using the same Phase 3 feature-window representation.

## Dataset
NAB

## Model
One-Class SVM

## Modeling Strategy
A separate One-Class SVM model was trained for each NAB series.

## Training Protocol
Labels were not used as prediction targets.

Training used only normal windows from the training split:

`label = 0`

Labels were used only to identify normal windows for training and later for evaluation.

## Testing Protocol
The model was tested on the full test split containing both normal and anomalous windows.

## Feature Schema
Same usable numeric Phase 3 feature-window schema used in the supervised baseline and Isolation Forest experiment.

Number of features:

{len(usable_feature_cols)}

## Hyperparameters
- kernel: rbf
- gamma: scale
- nu: 0.01
- preprocessing: median imputation + standard scaling

## Results
- Best threshold: {best_threshold_ocsvm}
- Precision: {best_precision_ocsvm}
- Recall: {best_recall_ocsvm}
- F1-score: {best_f1_ocsvm}
- ROC-AUC: {roc_auc_ocsvm}

## Interpretation
One-Class SVM achieved a strong ROC-AUC, indicating that it can rank anomalous windows higher than normal windows with reasonable effectiveness.

However, its F1-score and recall were lower than Isolation Forest. This suggests that, under the current NAB feature-window representation, the boundary-based One-Class SVM approach is less effective than the isolation-based approach.

The result remains useful because it confirms that different unsupervised models produce different anomaly-ranking behavior on the same feature windows. This supports the need for model comparison in Phase 4.

## Comparison with Isolation Forest
Isolation Forest remains the stronger unsupervised model so far.

Isolation Forest:
- F1-score: {best_f1}
- ROC-AUC: {roc_auc}

One-Class SVM:
- F1-score: {best_f1_ocsvm}
- ROC-AUC: {roc_auc_ocsvm}

## Saved Outputs
Scores:

`{ocsvm_scores_path}`

Metrics:

`{ocsvm_metrics_path}`

Master Results Tracker:

`{master_results_path}`

## Status
PH4_UNSUP_NAB_002 completed and logged.

## Next Experiment
PH4_UNSUP_NAB_003 — Local Outlier Factor using the same per-series unsupervised protocol.
"""

with open(log_path, "a", encoding="utf-8") as f:
    f.write(log_append)

print("Updated log:")
print(log_path)

In [ ]:
# =========================
# PH4_UNSUP_NAB_003 - Local Outlier Factor (Per-Series)
# =========================

from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import os

all_lof_results = []

series_ids = split_df["series_id"].unique()

print(f"Total series: {len(series_ids)}")

for series_id in tqdm(series_ids, desc="LOF per-series"):

    df_series = split_df[split_df["series_id"] == series_id]

    train_series = df_series[df_series["split"] == "train"]
    test_series  = df_series[df_series["split"] == "test"]

    if len(train_series) == 0 or len(test_series) == 0:
        continue

    train_normal = train_series[train_series["label"] == 0]

    if len(train_normal) < 50:
        continue

    X_train = train_normal[usable_feature_cols]
    X_test  = test_series[usable_feature_cols]

    model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("lof", LocalOutlierFactor(
            n_neighbors=20,
            contamination=0.01,
            novelty=True
        ))
    ])

    model.fit(X_train)

    # LOF: lower scores = more anomalous → invert
    scores = -model.decision_function(X_test)

    df_out = test_series.copy()
    df_out["anomaly_score"] = scores
    df_out["model"] = "LOF"

    all_lof_results.append(df_out)

lof_results_df = pd.concat(all_lof_results, ignore_index=True)

lof_scores_path = f"{PH4_PATH}/unsupervised/NAB/lof_scores.csv"
lof_results_df.to_csv(lof_scores_path, index=False)

print("Saved:", lof_scores_path)
print("Shape:", lof_results_df.shape)

In [ ]:
# =========================
# PH4_UNSUP_NAB_003 - Evaluation
# =========================

from sklearn.metrics import precision_recall_curve, roc_auc_score

y_true = lof_results_df["label"].values
y_score = lof_results_df["anomaly_score"].values

precision, recall, thresholds = precision_recall_curve(y_true, y_score)
f1_scores = (2 * precision * recall) / (precision + recall + 1e-10)

best_idx = np.argmax(f1_scores)

best_threshold_lof = thresholds[best_idx]
best_f1_lof = f1_scores[best_idx]
best_precision_lof = precision[best_idx]
best_recall_lof = recall[best_idx]
roc_auc_lof = roc_auc_score(y_true, y_score)

lof_metrics_df = pd.DataFrame([{
    "dataset": "NAB",
    "model": "LOF",
    "best_threshold": best_threshold_lof,
    "f1": best_f1_lof,
    "precision": best_precision_lof,
    "recall": best_recall_lof,
    "roc_auc": roc_auc_lof
}])

lof_metrics_path = f"{PH4_PATH}/unsupervised/NAB/lof_metrics.csv"
lof_metrics_df.to_csv(lof_metrics_path, index=False)

print("Saved:", lof_metrics_path)
display(lof_metrics_df)

In [ ]:
from datetime import datetime

log_path = f"{PH4_LOGS_PATH}/PH4_LOG_004_unsupervised_nab.md"

log_append = f"""

---

# PH4_UNSUP_NAB_003

## Timestamp
{datetime.now()}

## Experiment Title
Unsupervised Local Outlier Factor on NAB Feature Windows

## Objective
To evaluate Local Outlier Factor as a per-series unsupervised anomaly detection model using the same Phase 3 feature-window representation used for Isolation Forest and One-Class SVM.

## Dataset
NAB

## Model
Local Outlier Factor

## Modeling Strategy
A separate LOF model was trained for each NAB series.

## Training Protocol
Training used only normal windows from the training split:

`label = 0`

Labels were not used as prediction targets. Labels were used only to select normal training windows and later for evaluation.

## Testing Protocol
The model was tested on the full test split containing both normal and anomalous windows.

## Feature Schema
Same usable numeric Phase 3 feature-window schema used in the previous Phase 4 experiments.

Number of features:

{len(usable_feature_cols)}

## Hyperparameters
- n_neighbors: 20
- contamination: 0.01
- novelty: True
- preprocessing: median imputation + standard scaling

## Results
- Best threshold: {best_threshold_lof}
- Precision: {best_precision_lof}
- Recall: {best_recall_lof}
- F1-score: {best_f1_lof}
- ROC-AUC: {roc_auc_lof}

## Interpretation
LOF achieved the highest recall among the unsupervised models tested so far, but with lower precision than Isolation Forest.

This indicates that LOF behaves as a more aggressive anomaly detector: it identifies more true anomalies but also produces more false positives.

Compared with Isolation Forest, LOF is weaker overall because it has lower F1-score and lower ROC-AUC.

Compared with One-Class SVM, LOF has stronger recall and slightly higher F1-score, but lower ROC-AUC.

## Model Comparison Summary
Isolation Forest remains the strongest overall unsupervised model so far.

- Isolation Forest: F1 = {best_f1}, ROC-AUC = {roc_auc}
- One-Class SVM: F1 = {best_f1_ocsvm}, ROC-AUC = {roc_auc_ocsvm}
- LOF: F1 = {best_f1_lof}, ROC-AUC = {roc_auc_lof}

## Methodological Decision
LOF is retained as a comparison model, not as the current best model.

The current ranking is:
1. Isolation Forest
2. LOF
3. One-Class SVM

## Saved Outputs
Scores:

`{lof_scores_path}`

Metrics:

`{lof_metrics_path}`

## Status
PH4_UNSUP_NAB_003 completed and logged.

## Next Step
Proceed to the next Phase 4 modeling step.
"""

with open(log_path, "a", encoding="utf-8") as f:
    f.write(log_append)

print("LOF log added:")
print(log_path)

In [ ]:
import pandas as pd
import os

tracker_df = pd.read_excel(master_results_path, sheet_name="All_Experiments")

tracker_df = tracker_df[tracker_df["experiment_id"] != "PH4_UNSUP_NAB_003"].copy()

lof_new_row = pd.DataFrame([{
    "experiment_id": "PH4_UNSUP_NAB_003",
    "phase": "PH4",
    "dataset": "NAB",
    "model": "LOF",
    "modeling_type": "Unsupervised",
    "protocol": "Per-series; trained on normal windows only; labels used only for evaluation",
    "train_label_usage": "Labels used only to select normal training windows",
    "evaluation_label_usage": "Labels used only after scoring",
    "threshold_type": "F1-optimized",
    "best_threshold": best_threshold_lof,
    "precision": best_precision_lof,
    "recall": best_recall_lof,
    "f1": best_f1_lof,
    "roc_auc": roc_auc_lof,
    "n_train": len(train_df),
    "n_test": len(test_df),
    "n_features": len(usable_feature_cols),
    "notes": "Aggressive detector; high recall but lower precision than Isolation Forest."
}])

tracker_df = pd.concat([tracker_df, lof_new_row], ignore_index=True)

summary_df = tracker_df.sort_values(["dataset", "f1"], ascending=[True, False]).reset_index(drop=True)
best_df = summary_df.groupby("dataset", as_index=False).first()

with pd.ExcelWriter(master_results_path, engine="openpyxl") as writer:
    tracker_df.to_excel(writer, sheet_name="All_Experiments", index=False)
    summary_df.to_excel(writer, sheet_name="Ranked_By_F1", index=False)
    best_df.to_excel(writer, sheet_name="Best_By_Dataset", index=False)

print("Master results tracker updated:")
print(master_results_path)

display(summary_df)

In [ ]:
# =========================
# PH4_UNSUP_NAB_004 - Autoencoder Per-Series
# =========================

import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(42)
np.random.seed(42)

all_ae_results = []

series_ids = split_df["series_id"].unique()

print(f"Total series: {len(series_ids)}")

def build_autoencoder(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(input_dim, activation="linear")
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model

for series_id in tqdm(series_ids, desc="Autoencoder per-series"):

    df_series = split_df[split_df["series_id"] == series_id]

    train_series = df_series[df_series["split"] == "train"]
    test_series  = df_series[df_series["split"] == "test"]

    if len(train_series) == 0 or len(test_series) == 0:
        continue

    train_normal = train_series[train_series["label"] == 0]

    if len(train_normal) < 100:
        continue

    X_train_raw = train_normal[usable_feature_cols]
    X_test_raw  = test_series[usable_feature_cols]

    preprocessor = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    X_train = preprocessor.fit_transform(X_train_raw)
    X_test  = preprocessor.transform(X_test_raw)

    input_dim = X_train.shape[1]

    ae = build_autoencoder(input_dim)

    early_stop = callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    ae.fit(
        X_train,
        X_train,
        epochs=50,
        batch_size=128,
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=0
    )

    X_test_reconstructed = ae.predict(X_test, verbose=0)

    reconstruction_error = np.mean(
        np.square(X_test - X_test_reconstructed),
        axis=1
    )

    df_out = test_series.copy()
    df_out["anomaly_score"] = reconstruction_error
    df_out["model"] = "Autoencoder"

    all_ae_results.append(df_out)

ae_results_df = pd.concat(all_ae_results, ignore_index=True)

ae_scores_path = f"{PH4_PATH}/unsupervised/NAB/autoencoder_scores.csv"
ae_results_df.to_csv(ae_scores_path, index=False)

print("Saved:", ae_scores_path)
print("Shape:", ae_results_df.shape)

In [ ]:
# =========================
# PH4_UNSUP_NAB_004 - Evaluation
# =========================

from sklearn.metrics import precision_recall_curve, roc_auc_score

y_true = ae_results_df["label"].values
y_score = ae_results_df["anomaly_score"].values

precision, recall, thresholds = precision_recall_curve(y_true, y_score)
f1_scores = (2 * precision * recall) / (precision + recall + 1e-10)

best_idx = np.argmax(f1_scores)

best_threshold_ae = thresholds[best_idx]
best_f1_ae = f1_scores[best_idx]
best_precision_ae = precision[best_idx]
best_recall_ae = recall[best_idx]
roc_auc_ae = roc_auc_score(y_true, y_score)

ae_metrics_df = pd.DataFrame([{
    "dataset": "NAB",
    "model": "Autoencoder",
    "best_threshold": best_threshold_ae,
    "f1": best_f1_ae,
    "precision": best_precision_ae,
    "recall": best_recall_ae,
    "roc_auc": roc_auc_ae
}])

ae_metrics_path = f"{PH4_PATH}/unsupervised/NAB/autoencoder_metrics.csv"
ae_metrics_df.to_csv(ae_metrics_path, index=False)

print("Saved:", ae_metrics_path)
display(ae_metrics_df)

In [ ]:
# =========================
# PH4_UNSUP_NAB_004 - Log Autoencoder + Update Excel + Close NAB
# =========================

from datetime import datetime
import pandas as pd

# ---------- 1) Append Autoencoder log ----------
log_path = f"{PH4_LOGS_PATH}/PH4_LOG_004_unsupervised_nab.md"

log_append = f"""

---

# PH4_UNSUP_NAB_004

## Timestamp
{datetime.now()}

## Experiment Title
Unsupervised Autoencoder on NAB Feature Windows

## Objective
To evaluate a reconstruction-based unsupervised anomaly detection model using the same Phase 3 feature-window representation.

## Dataset
NAB

## Model
Tabular Autoencoder

## Modeling Strategy
A separate Autoencoder was trained for each NAB series.

## Training Protocol
Training used only normal windows from the training split:

`label = 0`

Labels were not used as prediction targets. Labels were used only to select normal training windows and later for evaluation.

## Testing Protocol
The model was tested on the full test split containing both normal and anomalous windows.

## Feature Schema
Same usable numeric Phase 3 feature-window schema.

Number of features:

{len(usable_feature_cols)}

## Autoencoder Architecture
- Input layer: number of usable features
- Dense 32 ReLU
- Dense 16 ReLU
- Dense 8 ReLU bottleneck
- Dense 16 ReLU
- Dense 32 ReLU
- Linear output layer

## Training Setup
- Loss: Mean Squared Error
- Optimizer: Adam
- Epochs: 50 maximum
- Batch size: 128
- Validation split: 0.1
- Early stopping patience: 5

## Results
- Best threshold: {best_threshold_ae}
- Precision: {best_precision_ae}
- Recall: {best_recall_ae}
- F1-score: {best_f1_ae}
- ROC-AUC: {roc_auc_ae}

## Interpretation
The Autoencoder achieved the highest F1-score among the NAB unsupervised models, indicating the strongest thresholded anomaly-decision performance.

However, Isolation Forest achieved the highest ROC-AUC, indicating stronger ranking ability across thresholds.

This distinction is important: Autoencoder is currently the best decision model after threshold selection, while Isolation Forest remains the strongest anomaly-ranking model.

## Final NAB Unsupervised Model Comparison
- Isolation Forest: F1 = {best_f1}, ROC-AUC = {roc_auc}
- One-Class SVM: F1 = {best_f1_ocsvm}, ROC-AUC = {roc_auc_ocsvm}
- LOF: F1 = {best_f1_lof}, ROC-AUC = {roc_auc_lof}
- Autoencoder: F1 = {best_f1_ae}, ROC-AUC = {roc_auc_ae}

## Methodological Decision
NAB modeling is complete.

Autoencoder is retained as the best thresholded decision model on NAB.
Isolation Forest is retained as the strongest anomaly-ranking model on NAB.

## Saved Outputs
Scores:

`{ae_scores_path}`

Metrics:

`{ae_metrics_path}`

## Status
PH4_UNSUP_NAB_004 completed and logged.
"""

with open(log_path, "a", encoding="utf-8") as f:
    f.write(log_append)

print("Autoencoder log added:", log_path)


# ---------- 2) Update Excel tracker ----------
tracker_df = pd.read_excel(master_results_path, sheet_name="All_Experiments")

# Avoid duplicate Autoencoder rows if cell is rerun
tracker_df = tracker_df[tracker_df["experiment_id"] != "PH4_UNSUP_NAB_004"].copy()

ae_new_row = pd.DataFrame([{
    "experiment_id": "PH4_UNSUP_NAB_004",
    "phase": "PH4",
    "dataset": "NAB",
    "model": "Autoencoder",
    "modeling_type": "Unsupervised",
    "protocol": "Per-series; trained on normal windows only; labels used only for evaluation",
    "train_label_usage": "Labels used only to select normal training windows",
    "evaluation_label_usage": "Labels used only after scoring",
    "threshold_type": "F1-optimized",
    "best_threshold": best_threshold_ae,
    "precision": best_precision_ae,
    "recall": best_recall_ae,
    "f1": best_f1_ae,
    "roc_auc": roc_auc_ae,
    "n_train": len(train_df),
    "n_test": len(test_df),
    "n_features": len(usable_feature_cols),
    "notes": "Best thresholded F1 on NAB; reconstruction-based anomaly detection."
}])

tracker_df = pd.concat([tracker_df, ae_new_row], ignore_index=True)

summary_df = tracker_df.sort_values(["dataset", "f1"], ascending=[True, False]).reset_index(drop=True)
best_df = summary_df.groupby("dataset", as_index=False).first()

with pd.ExcelWriter(master_results_path, engine="openpyxl") as writer:
    tracker_df.to_excel(writer, sheet_name="All_Experiments", index=False)
    summary_df.to_excel(writer, sheet_name="Ranked_By_F1", index=False)
    best_df.to_excel(writer, sheet_name="Best_By_Dataset", index=False)

print("Master Excel tracker updated:", master_results_path)
display(summary_df)


# ---------- 3) Create final NAB closure log ----------
closure_log_path = f"{PH4_LOGS_PATH}/PH4_LOG_004A_nab_modeling_closure.md"

closure_log = f"""# PH4_LOG_004A_nab_modeling_closure

## Timestamp
{datetime.now()}

## Closure Scope
This log formally closes NAB modeling in Phase 4.

## Completed NAB Experiments
1. PH4_SUP_NAB_001 — Supervised baseline
2. PH4_UNSUP_NAB_001 — Isolation Forest
3. PH4_UNSUP_NAB_002 — One-Class SVM
4. PH4_UNSUP_NAB_003 — Local Outlier Factor
5. PH4_UNSUP_NAB_004 — Autoencoder

## Final Unsupervised NAB Results

| Model | F1-score | Precision | Recall | ROC-AUC |
|---|---:|---:|---:|---:|
| Autoencoder | {best_f1_ae:.6f} | {best_precision_ae:.6f} | {best_recall_ae:.6f} | {roc_auc_ae:.6f} |
| Isolation Forest | {best_f1:.6f} | {best_precision:.6f} | {best_recall:.6f} | {roc_auc:.6f} |
| LOF | {best_f1_lof:.6f} | {best_precision_lof:.6f} | {best_recall_lof:.6f} | {roc_auc_lof:.6f} |
| One-Class SVM | {best_f1_ocsvm:.6f} | {best_precision_ocsvm:.6f} | {best_recall_ocsvm:.6f} | {roc_auc_ocsvm:.6f} |

## Final Interpretation
NAB results show that unsupervised anomaly detection is more suitable than the supervised baseline under sparse window-level anomaly labels.

The Autoencoder achieved the highest F1-score, making it the strongest thresholded decision model on NAB.

Isolation Forest achieved the highest ROC-AUC, making it the strongest anomaly-ranking model on NAB.

LOF showed aggressive detection behavior with high recall but lower precision.

One-Class SVM showed good ranking ability but weaker thresholded detection performance.

## Methodological Conclusion
NAB has served its role as a labeled benchmark for validating the Phase 4 modeling and evaluation protocol.

NAB modeling is now closed.

## Next Step
Proceed to unlabeled smart home datasets:
1. UK-DALE
2. AMPds2

For UK-DALE and AMPds2, labels are unavailable, so evaluation will focus on:
- anomaly score distributions
- consistency across models
- comparative model behavior
- synthetic anomaly injection later only if needed
"""

with open(closure_log_path, "w", encoding="utf-8") as f:
    f.write(closure_log)

print("NAB closure log saved:", closure_log_path)

In [ ]:
# =========================
# FINAL NAB CLOSURE LOG
# =========================

from datetime import datetime

closure_log_path = f"{PH4_LOGS_PATH}/PH4_LOG_004A_nab_modeling_closure.md"

closure_log = f"""# PH4_LOG_004A_nab_modeling_closure

## Timestamp
{datetime.now()}

## Scope
This document formally closes all NAB experiments in Phase 4.

## Completed Experiments

### Supervised
- PH4_SUP_NAB_001 — Supervised baseline (RF, XGBoost, Logistic Regression)

### Unsupervised
- PH4_UNSUP_NAB_001 — Isolation Forest
- PH4_UNSUP_NAB_002 — One-Class SVM
- PH4_UNSUP_NAB_003 — Local Outlier Factor
- PH4_UNSUP_NAB_004 — Autoencoder

## Final Results Summary

| Model | F1-score | Precision | Recall | ROC-AUC |
|---|---:|---:|---:|---:|
| Autoencoder | {best_f1_ae:.6f} | {best_precision_ae:.6f} | {best_recall_ae:.6f} | {roc_auc_ae:.6f} |
| Isolation Forest | {best_f1:.6f} | {best_precision:.6f} | {best_recall:.6f} | {roc_auc:.6f} |
| LOF | {best_f1_lof:.6f} | {best_precision_lof:.6f} | {best_recall_lof:.6f} | {roc_auc_lof:.6f} |
| One-Class SVM | {best_f1_ocsvm:.6f} | {best_precision_ocsvm:.6f} | {best_recall_ocsvm:.6f} | {roc_auc_ocsvm:.6f} |

## Key Findings

1. Supervised models performed poorly due to:
   - extreme class imbalance (~0.3%)
   - weak window-level labeling

2. Unsupervised models significantly improved performance.

3. Model behavior differences:
   - Isolation Forest → best overall balance
   - LOF → high recall (aggressive detection)
   - One-Class SVM → conservative detection
   - Autoencoder → best F1-score

## Final Model Decision

- **Primary model (decision-based): Autoencoder**
- **Primary model (ranking-based): Isolation Forest**

Both are retained for further evaluation on real datasets.

## Methodological Conclusion

NAB successfully served as:
- a labeled benchmark dataset
- a validation step for modeling and evaluation pipeline

The results confirm that:
> anomaly detection in this problem is better formulated as an unsupervised task

## Status

NAB modeling is **complete and closed**.

No further experiments will be performed on NAB within the main pipeline.

## Next Step

Proceed to:
PH4_UNSUP_UKDALE_001 — Isolation Forest

Evaluation will change because:
- no labels are available
- focus will be on anomaly score distributions and model consistency
"""

with open(closure_log_path, "w", encoding="utf-8") as f:
    f.write(closure_log)

print("NAB closure log saved:")
print(closure_log_path)

In [ ]:
# =========================
# FINAL UPDATE OF MASTER TRACKER
# =========================

tracker_df = pd.read_excel(master_results_path, sheet_name="All_Experiments")

summary_df = tracker_df.sort_values(["dataset", "f1"], ascending=[True, False]).reset_index(drop=True)

best_df = summary_df.groupby("dataset", as_index=False).first()

with pd.ExcelWriter(master_results_path, engine="openpyxl") as writer:
    tracker_df.to_excel(writer, sheet_name="All_Experiments", index=False)
    summary_df.to_excel(writer, sheet_name="Ranked_By_F1", index=False)
    best_df.to_excel(writer, sheet_name="Best_By_Dataset", index=False)

print("Final NAB results saved in Excel tracker")
display(summary_df)

In [ ]:
# =========================
# PH4_UNSUP_UKDALE_001 - Load UK-DALE PH3 Feature Windows Correctly
# =========================

import os
import glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

UKDALE_FEATURES_DIR = f"{PH3_PATH}/UK_DALE/windows/features"

EXCLUDE_FILE_KEYWORDS = [
    "summary",
    "failed",
    "audit",
    "metrics",
    "predictions",
    "splits",
]

def discover_feature_files(root_dir):
    all_csvs = glob.glob(os.path.join(root_dir, "*.csv"))
    feature_csvs = []
    for f in all_csvs:
        name = os.path.basename(f).lower()
        if not any(k in name for k in EXCLUDE_FILE_KEYWORDS):
            feature_csvs.append(f)
    return sorted(feature_csvs)

ukdale_feature_files = discover_feature_files(UKDALE_FEATURES_DIR)

print("UK-DALE feature files found:", len(ukdale_feature_files))

if len(ukdale_feature_files) == 0:
    raise FileNotFoundError("No UK-DALE feature-window files found.")

ukdale_dfs = []

for fp in tqdm(ukdale_feature_files, desc="Loading UK-DALE feature files"):
    df = pd.read_csv(fp)
    df["source_file"] = os.path.basename(fp)
    ukdale_dfs.append(df)

ukdale_df = pd.concat(ukdale_dfs, ignore_index=True)

# Parse dates
for c in ["start_time", "end_time"]:
    if c in ukdale_df.columns:
        ukdale_df[c] = pd.to_datetime(ukdale_df[c], errors="coerce")

print("UK-DALE loaded shape:", ukdale_df.shape)
print("Columns:", list(ukdale_df.columns))
print("Unique series:", ukdale_df["series_id"].nunique())

In [ ]:
# =========================
# PH4_UNSUP_UKDALE_001 - Chronological Split for Unlabeled UK-DALE
# =========================

def chronological_split_unlabeled(df, train_ratio=0.70):
    split_frames = []

    for series_id, g in tqdm(df.groupby("series_id", sort=False), desc="Splitting UK-DALE series"):
        g = g.sort_values("start_time").reset_index(drop=True)

        n = len(g)
        if n < 3:
            g["split"] = "dropped_too_short"
            split_frames.append(g)
            continue

        cut_idx = max(1, int(np.floor(train_ratio * n)))
        cut_idx = min(cut_idx, n - 1)

        train_part = g.iloc[:cut_idx].copy()
        test_part = g.iloc[cut_idx:].copy()

        # Purge overlap boundary if end_time exists
        last_train_end = train_part["end_time"].max()
        keep_test_mask = test_part["start_time"] >= last_train_end

        purged_test = test_part.loc[keep_test_mask].copy()
        dropped_purge = test_part.loc[~keep_test_mask].copy()

        train_part["split"] = "train"
        purged_test["split"] = "test"
        dropped_purge["split"] = "dropped_purge"

        split_frames.extend([train_part, purged_test, dropped_purge])

    return pd.concat(split_frames, ignore_index=True)

ukdale_split_df = chronological_split_unlabeled(ukdale_df)

print("UK-DALE split counts:")
print(ukdale_split_df["split"].value_counts())

print("UK-DALE train shape:", ukdale_split_df[ukdale_split_df["split"] == "train"].shape)
print("UK-DALE test shape:", ukdale_split_df[ukdale_split_df["split"] == "test"].shape)

In [ ]:
# =========================
# PH4_UNSUP_UKDALE_001 - Isolation Forest Corrected
# =========================

from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

all_ukdale_results = []

metadata_cols = [
    "dataset_name",
    "series_id",
    "window_id",
    "start_time",
    "end_time",
    "window_size_minutes",
    "step_size_minutes",
    "sampling_minutes",
    "source_file",
    "split",
]

ukdale_feature_cols = [
    c for c in ukdale_split_df.columns
    if c not in metadata_cols
    and pd.api.types.is_numeric_dtype(ukdale_split_df[c])
]

# Remove all-null or constant columns
usable_ukdale_features = []
train_check = ukdale_split_df[ukdale_split_df["split"] == "train"]

for c in ukdale_feature_cols:
    if train_check[c].notna().sum() == 0:
        continue
    if train_check[c].nunique(dropna=True) <= 1:
        continue
    usable_ukdale_features.append(c)

print("Usable UK-DALE features:", len(usable_ukdale_features))

series_ids = ukdale_split_df["series_id"].unique()
print(f"Total UK-DALE series: {len(series_ids)}")

for series_id in tqdm(series_ids, desc="UK-DALE Isolation Forest per-series"):

    df_series = ukdale_split_df[ukdale_split_df["series_id"] == series_id]

    train_series = df_series[df_series["split"] == "train"]
    test_series = df_series[df_series["split"] == "test"]

    if len(train_series) < 50 or len(test_series) == 0:
        continue

    X_train = train_series[usable_ukdale_features]
    X_test = test_series[usable_ukdale_features]

    model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("iforest", IsolationForest(
            n_estimators=100,
            contamination=0.01,
            random_state=42,
            n_jobs=-1
        ))
    ])

    model.fit(X_train)

    scores = -model.decision_function(X_test)

    df_out = test_series.copy()
    df_out["anomaly_score"] = scores
    df_out["model"] = "IsolationForest"

    all_ukdale_results.append(df_out)

if len(all_ukdale_results) == 0:
    raise ValueError("No UK-DALE results generated. Check series lengths and feature columns.")

ukdale_results_df = pd.concat(all_ukdale_results, ignore_index=True)

ukdale_scores_path = f"{PH4_PATH}/unsupervised/UK_DALE/isolation_forest_scores.csv"
ukdale_results_df.to_csv(ukdale_scores_path, index=False)

print("Saved:", ukdale_scores_path)
print("Shape:", ukdale_results_df.shape)

In [ ]:
print("UK-DALE IF results shape:", ukdale_results_df.shape)
print("Saved path:", ukdale_scores_path)
print("Score summary:")
display(ukdale_results_df["anomaly_score"].describe())

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.hist(ukdale_results_df["anomaly_score"], bins=100)
plt.title("UK-DALE Isolation Forest Anomaly Score Distribution")
plt.xlabel("Anomaly Score")
plt.ylabel("Frequency")
plt.show()

top_ukdale_if = (
    ukdale_results_df
    .sort_values("anomaly_score", ascending=False)
    .head(20)
)

display(top_ukdale_if[[
    "series_id",
    "start_time",
    "end_time",
    "anomaly_score",
    "source_file"
]])

In [ ]:
ukdale_if_series_summary = (
    ukdale_results_df
    .groupby("series_id")["anomaly_score"]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
    .sort_values("max", ascending=False)
)

display(ukdale_if_series_summary.head(20))

In [ ]:
# =========================
# Verify UK-DALE house/building coverage
# =========================

import re

def extract_building(series_id):
    match = re.search(r"building(\d+)", str(series_id))
    return f"building{match.group(1)}" if match else "unknown"

ukdale_df["building_id"] = ukdale_df["series_id"].apply(extract_building)
ukdale_split_df["building_id"] = ukdale_split_df["series_id"].apply(extract_building)
ukdale_results_df["building_id"] = ukdale_results_df["series_id"].apply(extract_building)

print("Buildings in loaded UK-DALE PH3 data:")
print(sorted(ukdale_df["building_id"].unique()))

print("\nNumber of series per building:")
display(
    ukdale_df[["building_id", "series_id"]]
    .drop_duplicates()
    .groupby("building_id")
    .size()
    .reset_index(name="num_series")
)

print("\nTest-result rows per building:")
display(
    ukdale_results_df
    .groupby("building_id")
    .size()
    .reset_index(name="num_test_windows")
)

In [ ]:
# =========================
# Log PH4_UNSUP_UKDALE_001 - Isolation Forest
# =========================

from datetime import datetime

ukdale_log_path = f"{PH4_LOGS_PATH}/PH4_LOG_005_unsupervised_ukdale.md"

score_summary = ukdale_results_df["anomaly_score"].describe()

building_series_summary = (
    ukdale_df[["building_id", "series_id"]]
    .drop_duplicates()
    .groupby("building_id")
    .size()
    .reset_index(name="num_series")
)

building_test_summary = (
    ukdale_results_df
    .groupby("building_id")
    .size()
    .reset_index(name="num_test_windows")
)

top_series_summary = (
    ukdale_results_df
    .groupby("series_id")["anomaly_score"]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
    .sort_values("max", ascending=False)
    .head(10)
)

top_anomalies_for_log = (
    ukdale_results_df
    .sort_values("anomaly_score", ascending=False)
    .head(20)[["series_id", "start_time", "end_time", "anomaly_score", "source_file"]]
)

log_content = f"""# PH4_LOG_005_unsupervised_ukdale

## Experiment ID
PH4_UNSUP_UKDALE_001

## Timestamp
{datetime.now()}

## Experiment Title
Unsupervised Isolation Forest on UK-DALE Feature Windows

## Objective
To apply Isolation Forest to the unlabeled UK-DALE feature-window dataset and analyze anomaly-score behavior using a real smart-home energy dataset.

## Dataset
UK-DALE

## Dataset Role
UK-DALE is treated as an unlabeled real-world smart home energy dataset.

No precision, recall, F1-score, or ROC-AUC are computed because explicit anomaly labels are unavailable.

## Granularity Clarification
The current UK-DALE Phase 3 files represent sub-meter / appliance-level or circuit-level series, not only whole-house aggregate readings.

Evidence:
- The loaded data contains 111 series across the five UK-DALE buildings.
- A purely house-level setup would contain approximately one aggregate series per building.
- Series names such as building1_elec_meter44 and building5_elec_meter12 indicate meter-level readings.

This is consistent with the thesis scope because the work focuses on appliance-level or sub-meter-level anomaly detection.

## Building Coverage
Buildings included in the loaded UK-DALE data:

{sorted(ukdale_df["building_id"].unique())}

## Number of Series per Building
{building_series_summary.to_markdown(index=False)}

## Test Windows per Building
{building_test_summary.to_markdown(index=False)}

## Input Source
`{UKDALE_FEATURES_DIR}`

## Loaded Data Summary
- Feature files loaded: {len(ukdale_feature_files)}
- Loaded rows: {len(ukdale_df)}
- Unique series: {ukdale_df["series_id"].nunique()}

## Split Strategy
A per-series chronological split was used:
- 70% train
- 30% test
- boundary purging applied to reduce overlap leakage

Split counts:

{ukdale_split_df["split"].value_counts().to_string()}

## Model
Isolation Forest

## Modeling Strategy
Per-series modeling.

A separate Isolation Forest model was trained for each UK-DALE meter-level series.

## Training Protocol
The model was trained on the training portion of each series.

No anomaly labels were used.

## Testing Protocol
The model generated anomaly scores for the test portion of each series.

## Feature Schema
Number of usable numeric features:

{len(usable_ukdale_features)}

## Hyperparameters
- n_estimators: 100
- contamination: 0.01
- random_state: 42
- n_jobs: -1
- preprocessing: median imputation

## Output
Anomaly scores were generated per test window.

Saved score file:

`{ukdale_scores_path}`

## Score Distribution Summary
{score_summary.to_string()}

## Top 20 Scored Windows
{top_anomalies_for_log.to_markdown(index=False)}

## Top Series by Maximum Anomaly Score
{top_series_summary.to_markdown(index=False)}

## Interpretation
The anomaly-score distribution is right-skewed, with most windows concentrated in the lower-score region and a smaller number of high-score windows forming the anomalous tail.

The highest anomaly scores appear in temporally adjacent windows, which is consistent with sustained unusual consumption behavior rather than isolated random noise.

The highest-scoring windows are concentrated in specific meter-level series, especially building5_elec_meter12 and building1_elec_meter44. This suggests that anomaly behavior is series-dependent, supporting the use of per-series modeling rather than one global model.

## Methodological Decision
PH4_UNSUP_UKDALE_001 is accepted.

The experiment confirms that Isolation Forest can generate interpretable anomaly-score distributions for unlabeled UK-DALE sub-meter data.

## Next Step
Proceed to additional unsupervised UK-DALE models to compare anomaly-score consistency across models:
- One-Class SVM
- Local Outlier Factor
- Autoencoder

## Status
Completed and logged.
"""

with open(ukdale_log_path, "w", encoding="utf-8") as f:
    f.write(log_content)

print("UK-DALE Isolation Forest log saved:")
print(ukdale_log_path)

In [ ]:
# =========================
# PH4_UNSUP_UKDALE_002 - One-Class SVM
# =========================

from sklearn.svm import OneClassSVM
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import os

all_ukdale_ocsvm_results = []

series_ids = ukdale_split_df["series_id"].unique()

print(f"Total UK-DALE series: {len(series_ids)}")

for series_id in tqdm(series_ids, desc="UK-DALE OCSVM per-series"):

    df_series = ukdale_split_df[ukdale_split_df["series_id"] == series_id]

    train_series = df_series[df_series["split"] == "train"]
    test_series  = df_series[df_series["split"] == "test"]

    if len(train_series) < 50 or len(test_series) == 0:
        continue

    X_train = train_series[usable_ukdale_features]
    X_test  = test_series[usable_ukdale_features]

    model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ocsvm", OneClassSVM(
            kernel="rbf",
            gamma="scale",
            nu=0.01
        ))
    ])

    model.fit(X_train)

    # OCSVM: lower score = more anomalous → invert
    scores = -model.decision_function(X_test)

    df_out = test_series.copy()
    df_out["anomaly_score"] = scores
    df_out["model"] = "OneClassSVM"

    all_ukdale_ocsvm_results.append(df_out)

if len(all_ukdale_ocsvm_results) == 0:
    raise ValueError("No results generated.")

ukdale_ocsvm_df = pd.concat(all_ukdale_ocsvm_results, ignore_index=True)

ukdale_ocsvm_path = f"{PH4_PATH}/unsupervised/UK_DALE/ocsvm_scores.csv"
ukdale_ocsvm_df.to_csv(ukdale_ocsvm_path, index=False)

print("Saved:", ukdale_ocsvm_path)
print("Shape:", ukdale_ocsvm_df.shape)

In [ ]:
# =========================
# Compare IF vs OCSVM distributions
# =========================

import matplotlib.pyplot as plt

plt.figure()
plt.hist(ukdale_results_df["anomaly_score"], bins=100, alpha=0.5, label="Isolation Forest")
plt.hist(ukdale_ocsvm_df["anomaly_score"], bins=100, alpha=0.5, label="OCSVM")
plt.legend()
plt.title("UK-DALE: IF vs OCSVM Score Distribution")
plt.xlabel("Anomaly Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
top_ocsvm = (
    ukdale_ocsvm_df
    .sort_values("anomaly_score", ascending=False)
    .head(20)
)

display(top_ocsvm[[
    "series_id",
    "start_time",
    "end_time",
    "anomaly_score"
]])

In [ ]:
print("OCSVM score summary:")
display(ukdale_ocsvm_df["anomaly_score"].describe())

In [ ]:
# =========================
# Log PH4_UNSUP_UKDALE_002 - One-Class SVM
# =========================

from datetime import datetime

ukdale_log_path = f"{PH4_LOGS_PATH}/PH4_LOG_005_unsupervised_ukdale.md"

ocsvm_score_summary = ukdale_ocsvm_df["anomaly_score"].describe()

top_ocsvm = (
    ukdale_ocsvm_df
    .sort_values("anomaly_score", ascending=False)
    .head(20)[["series_id", "start_time", "end_time", "anomaly_score"]]
)

top_series_ocsvm = (
    ukdale_ocsvm_df
    .groupby("series_id")["anomaly_score"]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
    .sort_values("max", ascending=False)
    .head(10)
)

log_append = f"""

---

# PH4_UNSUP_UKDALE_002

## Timestamp
{datetime.now()}

## Experiment Title
Unsupervised One-Class SVM on UK-DALE Feature Windows

## Objective
To evaluate One-Class SVM on unlabeled UK-DALE data and compare anomaly-score behavior against Isolation Forest.

## Model
One-Class SVM (RBF kernel)

## Modeling Strategy
Per-series modeling using the same feature representation and split strategy as PH4_UNSUP_UKDALE_001.

## Training Protocol
- Trained on full training windows (no label filtering)
- No labels used

## Output
Anomaly scores generated per test window.

Saved file:

`{ukdale_ocsvm_path}`

## Score Summary
{ocsvm_score_summary.to_string()}

## Top 20 Anomalies
{top_ocsvm.to_markdown(index=False)}

## Top Series by Max Score
{top_series_ocsvm.to_markdown(index=False)}

## Interpretation

One-Class SVM produced a highly dispersed anomaly-score distribution with extreme values.

Observed behavior:
- Very large score range (min ≈ {ocsvm_score_summary["min"]:.2f}, max ≈ {ocsvm_score_summary["max"]:.2f})
- High standard deviation (≈ {ocsvm_score_summary["std"]:.2f})
- Strong concentration of top anomalies in a single series (e.g., building1_elec_meter44)

This indicates instability in the learned boundary under high-dimensional feature conditions.

Compared to Isolation Forest:
- OCSVM produces less stable and less interpretable scores
- Sensitive to feature scaling and distribution variations

## Methodological Conclusion

OCSVM is retained as a comparison model but is not suitable as the primary model for UK-DALE.

## Status
Completed and logged.

## Next Step
Proceed to:
PH4_UNSUP_UKDALE_003 — Local Outlier Factor
"""

with open(ukdale_log_path, "a", encoding="utf-8") as f:
    f.write(log_append)

print("OCSVM log appended:")
print(ukdale_log_path)

In [ ]:
# =========================
# PH4_UNSUP_UKDALE_003 - LOF
# =========================

from sklearn.neighbors import LocalOutlierFactor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from tqdm.auto import tqdm
import numpy as np
import pandas as pd

all_ukdale_lof_results = []

series_ids = ukdale_split_df["series_id"].unique()

print(f"Total UK-DALE series: {len(series_ids)}")

for series_id in tqdm(series_ids, desc="UK-DALE LOF per-series"):

    df_series = ukdale_split_df[ukdale_split_df["series_id"] == series_id]

    train_series = df_series[df_series["split"] == "train"]
    test_series  = df_series[df_series["split"] == "test"]

    if len(train_series) < 50 or len(test_series) == 0:
        continue

    X_train = train_series[usable_ukdale_features]
    X_test  = test_series[usable_ukdale_features]

    model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("lof", LocalOutlierFactor(
            n_neighbors=20,
            contamination=0.01,
            novelty=True
        ))
    ])

    model.fit(X_train)

    # LOF: lower = more anomalous → invert
    scores = -model.decision_function(X_test)

    df_out = test_series.copy()
    df_out["anomaly_score"] = scores
    df_out["model"] = "LOF"

    all_ukdale_lof_results.append(df_out)

if len(all_ukdale_lof_results) == 0:
    raise ValueError("No LOF results generated.")

ukdale_lof_df = pd.concat(all_ukdale_lof_results, ignore_index=True)

ukdale_lof_path = f"{PH4_PATH}/unsupervised/UK_DALE/lof_scores.csv"
ukdale_lof_df.to_csv(ukdale_lof_path, index=False)

print("Saved:", ukdale_lof_path)
print("Shape:", ukdale_lof_df.shape)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()

plt.hist(ukdale_results_df["anomaly_score"], bins=100, alpha=0.4, label="IF")
plt.hist(ukdale_ocsvm_df["anomaly_score"], bins=100, alpha=0.4, label="OCSVM")
plt.hist(ukdale_lof_df["anomaly_score"], bins=100, alpha=0.4, label="LOF")

plt.legend()
plt.title("UK-DALE: IF vs OCSVM vs LOF")
plt.xlabel("Anomaly Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
top_lof = (
    ukdale_lof_df
    .sort_values("anomaly_score", ascending=False)
    .head(20)
)

display(top_lof[[
    "series_id",
    "start_time",
    "end_time",
    "anomaly_score"
]])

In [ ]:
print("LOF score summary:")
display(ukdale_lof_df["anomaly_score"].describe())

In [ ]:
# =========================
# PH4 Figure Output Folder
# =========================

import os
import matplotlib.pyplot as plt

PH4_FIGURES_PATH = f"{PH4_PATH}/figures"
UKDALE_FIGURES_PATH = f"{PH4_FIGURES_PATH}/UK_DALE"

os.makedirs(UKDALE_FIGURES_PATH, exist_ok=True)

print("Figure folder ready:")
print(UKDALE_FIGURES_PATH)

In [ ]:
# =========================
# Save UK-DALE Score Distribution Plots
# =========================

# 1) Isolation Forest distribution
plt.figure(figsize=(8, 5))
plt.hist(ukdale_results_df["anomaly_score"], bins=100)
plt.title("UK-DALE Isolation Forest Anomaly Score Distribution")
plt.xlabel("Anomaly Score")
plt.ylabel("Frequency")
plt.tight_layout()
if_plot_path = f"{UKDALE_FIGURES_PATH}/ukdale_if_score_distribution.png"
plt.savefig(if_plot_path, dpi=300, bbox_inches="tight")
plt.show()

# 2) OCSVM distribution
plt.figure(figsize=(8, 5))
plt.hist(ukdale_ocsvm_df["anomaly_score"], bins=100)
plt.title("UK-DALE One-Class SVM Anomaly Score Distribution")
plt.xlabel("Anomaly Score")
plt.ylabel("Frequency")
plt.tight_layout()
ocsvm_plot_path = f"{UKDALE_FIGURES_PATH}/ukdale_ocsvm_score_distribution.png"
plt.savefig(ocsvm_plot_path, dpi=300, bbox_inches="tight")
plt.show()

# 3) LOF raw distribution
plt.figure(figsize=(8, 5))
plt.hist(ukdale_lof_df["anomaly_score"], bins=100)
plt.title("UK-DALE LOF Raw Anomaly Score Distribution")
plt.xlabel("Anomaly Score")
plt.ylabel("Frequency")
plt.tight_layout()
lof_raw_plot_path = f"{UKDALE_FIGURES_PATH}/ukdale_lof_raw_score_distribution.png"
plt.savefig(lof_raw_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved plots:")
print(if_plot_path)
print(ocsvm_plot_path)
print(lof_raw_plot_path)

In [ ]:
# =========================
# Diagnose UK-DALE Window Duration Issues
# =========================

ukdale_lof_diag = ukdale_lof_df.copy()

ukdale_lof_diag["start_time"] = pd.to_datetime(ukdale_lof_diag["start_time"], errors="coerce")
ukdale_lof_diag["end_time"] = pd.to_datetime(ukdale_lof_diag["end_time"], errors="coerce")

ukdale_lof_diag["duration_minutes_actual"] = (
    ukdale_lof_diag["end_time"] - ukdale_lof_diag["start_time"]
).dt.total_seconds() / 60

print("Actual window duration summary:")
display(ukdale_lof_diag["duration_minutes_actual"].describe())

print("Number of windows with duration > 60 minutes:")
print((ukdale_lof_diag["duration_minutes_actual"] > 60).sum())

print("Number of windows with duration > 120 minutes:")
print((ukdale_lof_diag["duration_minutes_actual"] > 120).sum())

print("Top abnormal-duration windows:")
display(
    ukdale_lof_diag
    .sort_values("duration_minutes_actual", ascending=False)
    .head(20)[[
        "series_id",
        "start_time",
        "end_time",
        "duration_minutes_actual",
        "anomaly_score",
        "source_file"
    ]]
)

In [ ]:
# =========================
# Check whether high scores overlap with abnormal window durations
# =========================

def add_duration(df):
    temp = df.copy()
    temp["start_time"] = pd.to_datetime(temp["start_time"], errors="coerce")
    temp["end_time"] = pd.to_datetime(temp["end_time"], errors="coerce")
    temp["duration_minutes_actual"] = (
        temp["end_time"] - temp["start_time"]
    ).dt.total_seconds() / 60
    return temp

ukdale_if_diag = add_duration(ukdale_results_df)
ukdale_ocsvm_diag = add_duration(ukdale_ocsvm_df)

print("IF top 20 duration check:")
display(
    ukdale_if_diag
    .sort_values("anomaly_score", ascending=False)
    .head(20)[["series_id", "start_time", "end_time", "duration_minutes_actual", "anomaly_score"]]
)

print("OCSVM top 20 duration check:")
display(
    ukdale_ocsvm_diag
    .sort_values("anomaly_score", ascending=False)
    .head(20)[["series_id", "start_time", "end_time", "duration_minutes_actual", "anomaly_score"]]
)

In [ ]:
# =========================
# Filter valid windows (≈60 min only)
# =========================

def filter_valid_windows(df):
    temp = df.copy()
    temp["start_time"] = pd.to_datetime(temp["start_time"], errors="coerce")
    temp["end_time"] = pd.to_datetime(temp["end_time"], errors="coerce")

    temp["duration_minutes_actual"] = (
        temp["end_time"] - temp["start_time"]
    ).dt.total_seconds() / 60

    valid_df = temp[
        (temp["duration_minutes_actual"] >= 55) &
        (temp["duration_minutes_actual"] <= 65)
    ].copy()

    return valid_df

ukdale_lof_filtered_df = filter_valid_windows(ukdale_lof_df)

print("Original:", ukdale_lof_df.shape)
print("Filtered:", ukdale_lof_filtered_df.shape)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

lof_path = "/content/drive/MyDrive/Master/Thesis/Code/processed_data/PH4/unsupervised/UK_DALE/lof_scores.csv"

# Load minimal columns first (memory efficient)
cols_needed = ["series_id", "start_time", "end_time", "anomaly_score"]

ukdale_lof_df = pd.read_csv(lof_path, usecols=cols_needed)

print("Loaded:", ukdale_lof_df.shape)

In [ ]:
# =========================
# Memory-safe filtering
# =========================

import numpy as np

ukdale_lof_df["start_time"] = pd.to_datetime(ukdale_lof_df["start_time"], errors="coerce")
ukdale_lof_df["end_time"] = pd.to_datetime(ukdale_lof_df["end_time"], errors="coerce")

duration = (ukdale_lof_df["end_time"] - ukdale_lof_df["start_time"]).dt.total_seconds() / 60

mask = (duration >= 55) & (duration <= 65)

# Apply mask WITHOUT copying everything
ukdale_lof_filtered_df = ukdale_lof_df.loc[mask]

print("Original:", len(ukdale_lof_df))
print("Filtered:", len(ukdale_lof_filtered_df))

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.hist(ukdale_lof_filtered_df["anomaly_score"], bins=100)
plt.title("UK-DALE LOF (Filtered)")
plt.xlabel("Anomaly Score")
plt.ylabel("Frequency")
plt.show()

print("Filtered summary:")
display(ukdale_lof_filtered_df["anomaly_score"].describe())

In [ ]:
filtered_path = "/content/drive/MyDrive/Master/Thesis/Code/processed_data/PH4/unsupervised/UK_DALE/lof_scores_filtered.csv"

ukdale_lof_filtered_df.to_csv(filtered_path, index=False)

print("Saved filtered LOF:", filtered_path)

In [ ]:
# =========================
# Log PH4_UNSUP_UKDALE_003 - LOF
# =========================

from datetime import datetime

lof_log_append = f"""

---

# PH4_UNSUP_UKDALE_003

## Timestamp
{datetime.now()}

## Experiment Title
Unsupervised Local Outlier Factor on UK-DALE Feature Windows

## Objective
To evaluate LOF behavior on unlabeled UK-DALE data and compare it with other unsupervised models.

## Data Quality Correction
Initial LOF results revealed extreme anomaly scores due to invalid temporal windows.

A filtering step was applied:
- kept only windows with duration between 55–65 minutes

This removed corrupted windows from evaluation.

## Results (After Filtering)

Summary statistics:
- mean: {ukdale_lof_filtered_df["anomaly_score"].mean()}
- std: {ukdale_lof_filtered_df["anomaly_score"].std()}
- min: {ukdale_lof_filtered_df["anomaly_score"].min()}
- max: {ukdale_lof_filtered_df["anomaly_score"].max()}

## Interpretation

Even after correcting window duration inconsistencies, LOF produced highly unstable anomaly scores with extremely large values.

This behavior is attributed to:
- sensitivity of LOF to high-dimensional feature spaces
- density-based nature of the algorithm
- variation in feature distributions across appliance-level time series

Compared to:
- Isolation Forest → stable and interpretable
- One-Class SVM → unstable but bounded

LOF shows the highest numerical instability.

## Methodological Conclusion

LOF is not suitable as a primary anomaly detection model for this dataset.

However, it is retained as a comparison model to highlight differences between:
- tree-based methods (Isolation Forest)
- boundary-based methods (OCSVM)
- density-based methods (LOF)

## Status
Completed and logged.

## Next Step
PH4_UNSUP_UKDALE_004 — Autoencoder
"""

with open(ukdale_log_path, "a", encoding="utf-8") as f:
    f.write(lof_log_append)

print("LOF log appended.")

In [ ]:
# =========================
# Recover UK-DALE log path after runtime restart
# =========================

BASE_PATH = "/content/drive/MyDrive/Master/Thesis/Code"
PROCESSED_DATA_PATH = f"{BASE_PATH}/processed_data"
PH4_PATH = f"{PROCESSED_DATA_PATH}/PH4"
EXPERIMENT_LOGS_PATH = f"{BASE_PATH}/experiment_logs"
PH4_LOGS_PATH = f"{EXPERIMENT_LOGS_PATH}/PH4"

ukdale_log_path = f"{PH4_LOGS_PATH}/PH4_LOG_005_unsupervised_ukdale.md"

print("UK-DALE log path restored:")
print(ukdale_log_path)

In [ ]:
# =========================
# PH4_UNSUP_UKDALE_004 - Autoencoder, memory-safe, valid windows only
# =========================

import os, glob, gc
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(42)
np.random.seed(42)

BASE_PATH = "/content/drive/MyDrive/Master/Thesis/Code"
PH3_PATH = f"{BASE_PATH}/processed_data/PH3"
PH4_PATH = f"{BASE_PATH}/processed_data/PH4"

UKDALE_FEATURES_DIR = f"{PH3_PATH}/UK_DALE/windows/features"
UKDALE_AE_PATH = f"{PH4_PATH}/unsupervised/UK_DALE/autoencoder_scores.csv"

EXCLUDE_FILE_KEYWORDS = ["summary", "failed", "audit", "metrics", "predictions", "splits"]

ukdale_files = [
    f for f in glob.glob(os.path.join(UKDALE_FEATURES_DIR, "*.csv"))
    if not any(k in os.path.basename(f).lower() for k in EXCLUDE_FILE_KEYWORDS)
]

print("UK-DALE files:", len(ukdale_files))

metadata_cols = [
    "dataset_name", "series_id", "window_id", "start_time", "end_time",
    "window_size_minutes", "step_size_minutes", "sampling_minutes",
    "label", "source_file", "split", "duration_minutes_actual"
]

def build_autoencoder(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(input_dim, activation="linear")
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

def prepare_valid_windows(df):
    df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
    df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")

    df["duration_minutes_actual"] = (
        df["end_time"] - df["start_time"]
    ).dt.total_seconds() / 60

    df = df[
        (df["duration_minutes_actual"] >= 55) &
        (df["duration_minutes_actual"] <= 65)
    ].copy()

    df = df.sort_values("start_time").reset_index(drop=True)
    return df

def chronological_split_single_series(df, train_ratio=0.70):
    n = len(df)
    cut_idx = max(1, int(np.floor(train_ratio * n)))
    cut_idx = min(cut_idx, n - 1)

    train_df = df.iloc[:cut_idx].copy()
    test_df = df.iloc[cut_idx:].copy()

    last_train_end = train_df["end_time"].max()
    test_df = test_df[test_df["start_time"] >= last_train_end].copy()

    return train_df, test_df

# Start fresh output file
if os.path.exists(UKDALE_AE_PATH):
    os.remove(UKDALE_AE_PATH)

first_write = True
processed_series = 0
skipped_series = 0

for fp in tqdm(ukdale_files, desc="UK-DALE Autoencoder per-series"):
    try:
        df = pd.read_csv(fp)
        df["source_file"] = os.path.basename(fp)

        df = prepare_valid_windows(df)

        if len(df) < 200:
            skipped_series += 1
            continue

        train_df, test_df = chronological_split_single_series(df, train_ratio=0.70)

        if len(train_df) < 100 or len(test_df) == 0:
            skipped_series += 1
            continue

        candidate_features = [
            c for c in df.columns
            if c not in metadata_cols and pd.api.types.is_numeric_dtype(df[c])
        ]

        usable_features = []
        for c in candidate_features:
            if train_df[c].notna().sum() == 0:
                continue
            if train_df[c].nunique(dropna=True) <= 1:
                continue
            usable_features.append(c)

        if len(usable_features) == 0:
            skipped_series += 1
            continue

        X_train_raw = train_df[usable_features]
        X_test_raw = test_df[usable_features]

        preprocessor = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])

        X_train = preprocessor.fit_transform(X_train_raw)
        X_test = preprocessor.transform(X_test_raw)

        ae = build_autoencoder(X_train.shape[1])

        early_stop = callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        )

        ae.fit(
            X_train, X_train,
            epochs=30,
            batch_size=256,
            validation_split=0.1,
            callbacks=[early_stop],
            verbose=0
        )

        X_pred = ae.predict(X_test, verbose=0)
        scores = np.mean(np.square(X_test - X_pred), axis=1)

        out = test_df[[
            "dataset_name", "series_id", "window_id",
            "start_time", "end_time", "source_file",
            "duration_minutes_actual"
        ]].copy()

        out["anomaly_score"] = scores
        out["model"] = "Autoencoder"

        out.to_csv(
            UKDALE_AE_PATH,
            mode="w" if first_write else "a",
            header=first_write,
            index=False
        )

        first_write = False
        processed_series += 1

        del df, train_df, test_df, X_train_raw, X_test_raw, X_train, X_test, X_pred, out, ae
        tf.keras.backend.clear_session()
        gc.collect()

    except Exception as e:
        skipped_series += 1
        print("Skipped:", os.path.basename(fp), "| Error:", str(e))

print("Saved:", UKDALE_AE_PATH)
print("Processed series:", processed_series)
print("Skipped series:", skipped_series)

In [ ]:
# =========================
# UK-DALE Autoencoder Summary + Plot
# =========================

import os
import pandas as pd
import matplotlib.pyplot as plt

PH4_FIGURES_PATH = f"{PH4_PATH}/figures"
UKDALE_FIGURES_PATH = f"{PH4_FIGURES_PATH}/UK_DALE"
os.makedirs(UKDALE_FIGURES_PATH, exist_ok=True)

ukdale_ae_df = pd.read_csv(UKDALE_AE_PATH)

print("Autoencoder output shape:", ukdale_ae_df.shape)
print("Autoencoder score summary:")
display(ukdale_ae_df["anomaly_score"].describe())

plt.figure(figsize=(8, 5))
plt.hist(ukdale_ae_df["anomaly_score"], bins=100)
plt.title("UK-DALE Autoencoder Anomaly Score Distribution")
plt.xlabel("Reconstruction Error")
plt.ylabel("Frequency")
plt.tight_layout()

ae_plot_path = f"{UKDALE_FIGURES_PATH}/ukdale_autoencoder_score_distribution.png"
plt.savefig(ae_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved plot:", ae_plot_path)

top_ae = (
    ukdale_ae_df
    .sort_values("anomaly_score", ascending=False)
    .head(20)
)

display(top_ae[[
    "series_id", "start_time", "end_time",
    "duration_minutes_actual", "anomaly_score", "source_file"
]])

In [ ]:
# =========================
# Log PH4_UNSUP_UKDALE_004 - Autoencoder
# =========================

from datetime import datetime

ukdale_log_path = f"{PH4_LOGS_PATH}/PH4_LOG_005_unsupervised_ukdale.md"

ae_summary = ukdale_ae_df["anomaly_score"].describe()

top_ae_for_log = (
    ukdale_ae_df
    .sort_values("anomaly_score", ascending=False)
    .head(20)[[
        "series_id", "start_time", "end_time",
        "duration_minutes_actual", "anomaly_score", "source_file"
    ]]
)

ae_log_append = f"""

---

# PH4_UNSUP_UKDALE_004

## Timestamp
{datetime.now()}

## Experiment Title
Unsupervised Autoencoder on UK-DALE Feature Windows

## Objective
To evaluate a reconstruction-based unsupervised anomaly detection model on the unlabeled UK-DALE dataset.

## Model
Tabular Autoencoder

## Modeling Strategy
A separate Autoencoder was trained per UK-DALE meter-level series.

## Data Filtering
Only valid 55–65 minute windows were used to avoid temporal-gap artifacts.

## Training Protocol
The Autoencoder was trained on the training portion of each valid series.

No anomaly labels were used.

## Output
Reconstruction error was used as the anomaly score.

Saved score file:

`{UKDALE_AE_PATH}`

## Process Summary
- Processed series: 107
- Skipped series: 4
- Output rows: {len(ukdale_ae_df)}

## Score Distribution Summary
{ae_summary.to_string()}

## Top 20 Anomalies
{top_ae_for_log.to_markdown(index=False)}

## Interpretation
The Autoencoder produced a strongly right-skewed reconstruction-error distribution.

Most windows have very low reconstruction error, while a small number of windows produce extremely high reconstruction error. This indicates that the model learned regular appliance-level behavior for most windows, while rare windows were reconstructed poorly and therefore treated as highly anomalous.

Unlike LOF, the highest Autoencoder anomaly scores occur in valid 59-minute windows, suggesting that these high scores are not caused by corrupted temporal window lengths.

## Methodological Conclusion
The Autoencoder is accepted as a valid reconstruction-based anomaly detection model for UK-DALE.

It is retained as a comparison model alongside Isolation Forest, One-Class SVM, and LOF.

## Status
Completed and logged.
"""

with open(ukdale_log_path, "a", encoding="utf-8") as f:
    f.write(ae_log_append)

print("Autoencoder log appended:")
print(ukdale_log_path)

In [ ]:
# =========================
# Create / update UK-DALE unlabeled score analysis Excel
# =========================

import pandas as pd
import os

ukdale_score_excel_path = f"{PH4_PATH}/combined/ph4_ukdale_unlabeled_score_analysis.xlsx"

def score_row(dataset, model, df, notes):
    s = df["anomaly_score"]
    dominant_series = (
        df.groupby("series_id")["anomaly_score"]
        .max()
        .sort_values(ascending=False)
        .index[0]
    )
    return {
        "dataset": dataset,
        "model": model,
        "n_windows": len(df),
        "mean_score": s.mean(),
        "std_score": s.std(),
        "min_score": s.min(),
        "p25_score": s.quantile(0.25),
        "median_score": s.quantile(0.50),
        "p75_score": s.quantile(0.75),
        "p95_score": s.quantile(0.95),
        "p99_score": s.quantile(0.99),
        "max_score": s.max(),
        "dominant_top_series": dominant_series,
        "notes": notes
    }

ukdale_score_rows = []

ukdale_score_rows.append(score_row(
    "UK-DALE", "IsolationForest", ukdale_results_df,
    "Stable and interpretable anomaly-score distribution."
))

ukdale_score_rows.append(score_row(
    "UK-DALE", "OneClassSVM", ukdale_ocsvm_df,
    "Heavy-tailed and less stable than Isolation Forest."
))

ukdale_score_rows.append(score_row(
    "UK-DALE", "LOF_Filtered", ukdale_lof_filtered_df,
    "Highly unstable even after filtering invalid-duration windows."
))

ukdale_score_rows.append(score_row(
    "UK-DALE", "Autoencoder", ukdale_ae_df,
    "Valid reconstruction-based model; strongly right-skewed reconstruction errors."
))

ukdale_score_analysis_df = pd.DataFrame(ukdale_score_rows)

with pd.ExcelWriter(ukdale_score_excel_path, engine="openpyxl") as writer:
    ukdale_score_analysis_df.to_excel(writer, sheet_name="Score_Distribution", index=False)

print("UK-DALE score analysis saved:")
print(ukdale_score_excel_path)

display(ukdale_score_analysis_df)

In [ ]:
# =========================
# Recover UK-DALE score DataFrames after runtime restart
# =========================

import os
import pandas as pd
import matplotlib.pyplot as plt

BASE_PATH = "/content/drive/MyDrive/Master/Thesis/Code"
PH4_PATH = f"{BASE_PATH}/processed_data/PH4"

UKDALE_DIR = f"{PH4_PATH}/unsupervised/UK_DALE"
UKDALE_FIGURES_PATH = f"{PH4_PATH}/figures/UK_DALE"
os.makedirs(UKDALE_FIGURES_PATH, exist_ok=True)

if_path = f"{UKDALE_DIR}/isolation_forest_scores.csv"
ocsvm_path = f"{UKDALE_DIR}/ocsvm_scores.csv"
lof_filtered_path = f"{UKDALE_DIR}/lof_scores_filtered.csv"
ae_path = f"{UKDALE_DIR}/autoencoder_scores.csv"

cols_common = ["series_id", "start_time", "end_time", "anomaly_score"]

ukdale_results_df = pd.read_csv(if_path, usecols=lambda c: c in cols_common)
ukdale_ocsvm_df = pd.read_csv(ocsvm_path, usecols=lambda c: c in cols_common)
ukdale_lof_filtered_df = pd.read_csv(lof_filtered_path, usecols=lambda c: c in cols_common)
ukdale_ae_df = pd.read_csv(ae_path, usecols=lambda c: c in cols_common + ["duration_minutes_actual", "source_file"])

print("IF:", ukdale_results_df.shape)
print("OCSVM:", ukdale_ocsvm_df.shape)
print("LOF filtered:", ukdale_lof_filtered_df.shape)
print("AE:", ukdale_ae_df.shape)

In [ ]:
# =========================
# Save UK-DALE distribution plots
# =========================

def save_hist(df, model_name, xlabel, filename):
    plt.figure(figsize=(8, 5))
    plt.hist(df["anomaly_score"], bins=100)
    plt.title(f"UK-DALE {model_name} Anomaly Score Distribution")
    plt.xlabel(xlabel)
    plt.ylabel("Frequency")
    plt.tight_layout()
    path = f"{UKDALE_FIGURES_PATH}/{filename}"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

save_hist(ukdale_results_df, "Isolation Forest", "Anomaly Score", "ukdale_if_score_distribution.png")
save_hist(ukdale_ocsvm_df, "One-Class SVM", "Anomaly Score", "ukdale_ocsvm_score_distribution.png")
save_hist(ukdale_lof_filtered_df, "LOF Filtered", "Anomaly Score", "ukdale_lof_filtered_score_distribution.png")
save_hist(ukdale_ae_df, "Autoencoder", "Reconstruction Error", "ukdale_autoencoder_score_distribution.png")

In [ ]:
# =========================
# Create / update UK-DALE unlabeled score analysis Excel
# =========================

ukdale_score_excel_path = f"{PH4_PATH}/combined/ph4_ukdale_unlabeled_score_analysis.xlsx"

def score_row(dataset, model, df, notes):
    s = df["anomaly_score"]
    dominant_series = (
        df.groupby("series_id")["anomaly_score"]
        .max()
        .sort_values(ascending=False)
        .index[0]
    )
    return {
        "dataset": dataset,
        "model": model,
        "n_windows": len(df),
        "mean_score": s.mean(),
        "std_score": s.std(),
        "min_score": s.min(),
        "p25_score": s.quantile(0.25),
        "median_score": s.quantile(0.50),
        "p75_score": s.quantile(0.75),
        "p95_score": s.quantile(0.95),
        "p99_score": s.quantile(0.99),
        "max_score": s.max(),
        "dominant_top_series": dominant_series,
        "notes": notes
    }

ukdale_score_analysis_df = pd.DataFrame([
    score_row("UK-DALE", "IsolationForest", ukdale_results_df,
              "Stable and interpretable anomaly-score distribution."),
    score_row("UK-DALE", "OneClassSVM", ukdale_ocsvm_df,
              "Heavy-tailed and less stable than Isolation Forest."),
    score_row("UK-DALE", "LOF_Filtered", ukdale_lof_filtered_df,
              "Highly unstable even after filtering invalid-duration windows."),
    score_row("UK-DALE", "Autoencoder", ukdale_ae_df,
              "Valid reconstruction-based model; strongly right-skewed reconstruction errors.")
])

with pd.ExcelWriter(ukdale_score_excel_path, engine="openpyxl") as writer:
    ukdale_score_analysis_df.to_excel(writer, sheet_name="Score_Distribution", index=False)

print("UK-DALE score analysis saved:")
print(ukdale_score_excel_path)

display(ukdale_score_analysis_df)

In [ ]:
# =========================
# PH4_LOG_006 — UK-DALE CLOSURE
# =========================

from datetime import datetime

ukdale_closure_log_path = f"{PH4_LOGS_PATH}/PH4_LOG_006_ukdale_closure.md"

# Basic summaries
def s(df):
    return df["anomaly_score"].describe()

if_summary = s(ukdale_results_df)
ocsvm_summary = s(ukdale_ocsvm_df)
lof_summary = s(ukdale_lof_filtered_df)
ae_summary = s(ukdale_ae_df)

# Dominant series per model
def dominant_series(df):
    return (
        df.groupby("series_id")["anomaly_score"]
        .max()
        .sort_values(ascending=False)
        .index[0]
    )

dom_if = dominant_series(ukdale_results_df)
dom_ocsvm = dominant_series(ukdale_ocsvm_df)
dom_lof = dominant_series(ukdale_lof_filtered_df)
dom_ae = dominant_series(ukdale_ae_df)

closure_log = f"""# PH4_LOG_006_ukdale_closure

## Timestamp
{datetime.now()}

## Scope
Formal closure of UK-DALE modeling (Phase 4 — Unsupervised).

## Dataset Role
UK-DALE is used as an **unlabeled real-world dataset**.
Evaluation is based on **anomaly-score distributions and cross-model consistency**, not classification metrics.

## Data Quality Note
A subset of windows had invalid durations (>> 60 minutes).
A filtering step (55–65 minutes) was applied for LOF and Autoencoder to remove temporal-gap artifacts.

## Models Evaluated
1) Isolation Forest
2) One-Class SVM
3) Local Outlier Factor (filtered)
4) Autoencoder

---

## Score Distribution Summaries

### Isolation Forest
{if_summary.to_string()}

### One-Class SVM
{ocsvm_summary.to_string()}

### LOF (Filtered)
{lof_summary.to_string()}

### Autoencoder
{ae_summary.to_string()}

---

## Dominant High-Anomaly Series

- Isolation Forest: {dom_if}
- One-Class SVM: {dom_ocsvm}
- LOF (Filtered): {dom_lof}
- Autoencoder: {dom_ae}

---

## Cross-Model Observations

- Isolation Forest produced **stable, bounded, and interpretable** scores.
- One-Class SVM produced **heavy-tailed and unstable** distributions.
- LOF remained **numerically unstable** even after correcting window durations, indicating sensitivity to feature space.
- Autoencoder produced **strongly right-skewed reconstruction errors**, capturing rare abnormal patterns.

High anomaly scores often appear in **clusters of consecutive windows**, indicating **sustained abnormal consumption behavior**, not isolated noise.

---

## Methodological Conclusions

- **Isolation Forest** is the most reliable baseline for UK-DALE.
- **Autoencoder** is a valid complementary model (reconstruction-based).
- **One-Class SVM and LOF** are retained as comparison models to illustrate instability across methods.

The results demonstrate that:
> Model choice significantly affects anomaly-score behavior on real-world smart home energy data.

---

## Outputs

Scores:
- isolation_forest_scores.csv
- ocsvm_scores.csv
- lof_scores_filtered.csv
- autoencoder_scores.csv

Figures:
- ukdale_if_score_distribution.png
- ukdale_ocsvm_score_distribution.png
- ukdale_lof_filtered_score_distribution.png
- ukdale_autoencoder_score_distribution.png

Excel:
- ph4_ukdale_unlabeled_score_analysis.xlsx

---

## Status
UK-DALE modeling is **complete and closed**.

## Next Step
Proceed to:
PH4_UNSUP_AMPDS2_001 — Isolation Forest
"""

with open(ukdale_closure_log_path, "w", encoding="utf-8") as f:
    f.write(closure_log)

print("UK-DALE closure log saved:")
print(ukdale_closure_log_path)

In [ ]:
# =========================
# PH4_SUP_NAB_002 - Setup
# Clean supervised validation with 4 models:
# RF, XGBoost, Logistic Regression, MLP
# =========================

from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import json
import joblib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

BASE_PATH = "/content/drive/MyDrive/Master/Thesis/Code"
PH3_PATH = f"{BASE_PATH}/processed_data/PH3"
PH4_PATH = f"{BASE_PATH}/processed_data/PH4"
PH4_LOGS_PATH = f"{BASE_PATH}/experiment_logs/PH4"

NAB_FEATURES_DIR = f"{PH3_PATH}/NAB/raw_windows/features"

PH4_SUP_NAB_PATH = f"{PH4_PATH}/supervised/NAB"
PH4_SUP_NAB_MODELS_PATH = f"{PH4_SUP_NAB_PATH}/models"
PH4_SUP_NAB_METRICS_PATH = f"{PH4_SUP_NAB_PATH}/metrics"
PH4_SUP_NAB_PREDS_PATH = f"{PH4_SUP_NAB_PATH}/predictions"
PH4_SUP_NAB_SPLITS_PATH = f"{PH4_SUP_NAB_PATH}/splits"
PH4_SUP_NAB_INPUTS_PATH = f"{PH4_SUP_NAB_PATH}/inputs"

for p in [
    PH4_SUP_NAB_MODELS_PATH,
    PH4_SUP_NAB_METRICS_PATH,
    PH4_SUP_NAB_PREDS_PATH,
    PH4_SUP_NAB_SPLITS_PATH,
    PH4_SUP_NAB_INPUTS_PATH,
    PH4_LOGS_PATH
]:
    os.makedirs(p, exist_ok=True)

RANDOM_STATE = 42
TARGET_COL = "label"

METADATA_COLS = [
    "dataset_name",
    "series_id",
    "window_id",
    "start_time",
    "end_time",
    "window_size_minutes",
    "step_size_minutes",
    "sampling_minutes",
]

In [ ]:
# =========================
# Load NAB PH3 feature windows
# =========================

EXCLUDE_FILE_KEYWORDS = [
    "summary",
    "failed",
    "audit",
    "metrics",
    "predictions",
    "splits",
]

def discover_feature_files(root_dir):
    all_csvs = glob.glob(os.path.join(root_dir, "*.csv"))
    return sorted([
        f for f in all_csvs
        if not any(k in os.path.basename(f).lower() for k in EXCLUDE_FILE_KEYWORDS)
    ])

nab_feature_files = discover_feature_files(NAB_FEATURES_DIR)

print("NAB feature files found:", len(nab_feature_files))

if len(nab_feature_files) == 0:
    raise FileNotFoundError("No NAB feature-window files found.")

dfs = []

for fp in tqdm(nab_feature_files, desc="Loading NAB feature files"):
    df = pd.read_csv(fp)
    df["source_file"] = os.path.basename(fp)
    dfs.append(df)

nab_df = pd.concat(dfs, ignore_index=True)

for c in ["start_time", "end_time"]:
    nab_df[c] = pd.to_datetime(nab_df[c], errors="coerce")

nab_df = nab_df.dropna(subset=[TARGET_COL]).copy()
nab_df[TARGET_COL] = nab_df[TARGET_COL].astype(int)
nab_df = nab_df[nab_df[TARGET_COL].isin([0, 1])].copy()

print("NAB shape:", nab_df.shape)
print("Unique series:", nab_df["series_id"].nunique())
print("Label distribution:")
print(nab_df[TARGET_COL].value_counts())

In [ ]:
# =========================
# PH4_SUP_NAB_002 - Official 80/20 chronological split
# with internal validation carved only from training region
# =========================

def chronological_80_20_with_internal_validation(
    df,
    train_test_ratio=0.80,
    internal_val_ratio_within_train=0.20
):
    split_frames = []

    for series_id, g in tqdm(df.groupby("series_id", sort=False), desc="Splitting NAB series"):
        g = g.sort_values("start_time").reset_index(drop=True)

        n = len(g)
        if n < 5:
            temp = g.copy()
            temp["split"] = "dropped_too_short"
            split_frames.append(temp)
            continue

        # Official 80/20 train-test boundary
        train_region_end = int(np.floor(train_test_ratio * n))
        train_region_end = max(2, min(train_region_end, n - 1))

        train_region = g.iloc[:train_region_end].copy()
        test_part = g.iloc[train_region_end:].copy()

        # Internal validation only inside the 80% training region
        internal_train_end = int(
            np.floor((1 - internal_val_ratio_within_train) * len(train_region))
        )
        internal_train_end = max(1, min(internal_train_end, len(train_region) - 1))

        train_part = train_region.iloc[:internal_train_end].copy()
        val_part = train_region.iloc[internal_train_end:].copy()

        # Purge validation windows overlapping training boundary
        last_train_end = train_part["end_time"].max()
        val_keep = val_part["start_time"] >= last_train_end
        dropped_val = val_part.loc[~val_keep].copy()
        val_part = val_part.loc[val_keep].copy()

        # Purge test windows overlapping validation/training boundary
        if len(val_part) > 0:
            last_val_end = val_part["end_time"].max()
        else:
            last_val_end = last_train_end

        test_keep = test_part["start_time"] >= last_val_end
        dropped_test = test_part.loc[~test_keep].copy()
        test_part = test_part.loc[test_keep].copy()

        train_part["split"] = "train"
        val_part["split"] = "internal_validation"
        test_part["split"] = "test"
        dropped_val["split"] = "dropped_purge_validation"
        dropped_test["split"] = "dropped_purge_test"

        split_frames.extend([
            train_part,
            val_part,
            test_part,
            dropped_val,
            dropped_test
        ])

    return pd.concat(split_frames, ignore_index=True)


nab_split_df = chronological_80_20_with_internal_validation(
    nab_df,
    train_test_ratio=0.80,
    internal_val_ratio_within_train=0.20
)

split_audit_path = f"{PH4_SUP_NAB_SPLITS_PATH}/nab_supervised_80_20_split_with_internal_validation_audit.csv"
nab_split_df.to_csv(split_audit_path, index=False)

print("Split counts:")
print(nab_split_df["split"].value_counts())

train_df = nab_split_df[nab_split_df["split"] == "train"].copy()
val_df = nab_split_df[nab_split_df["split"] == "internal_validation"].copy()
test_df = nab_split_df[nab_split_df["split"] == "test"].copy()

print("\nTrain shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain labels:")
print(train_df[TARGET_COL].value_counts())

print("\nInternal validation labels:")
print(val_df[TARGET_COL].value_counts())

print("\nTest labels:")
print(test_df[TARGET_COL].value_counts())

print("\nSaved split audit:")
print(split_audit_path)

In [ ]:
# =========================
# PH4_SUP_NAB_002 - Feature schema
# Official 80/20 split with internal validation inside training region
# =========================

excluded_cols = METADATA_COLS + [TARGET_COL, "source_file", "split"]

candidate_features = [
    c for c in train_df.columns
    if c not in excluded_cols
    and pd.api.types.is_numeric_dtype(train_df[c])
]

usable_features = []

for c in candidate_features:
    if train_df[c].notna().sum() == 0:
        continue
    if train_df[c].nunique(dropna=True) <= 1:
        continue
    usable_features.append(c)

print("Usable feature count:", len(usable_features))
print(usable_features)

schema = {
    "experiment_id": "PH4_SUP_NAB_002",
    "dataset": "NAB",
    "target_col": TARGET_COL,
    "feature_cols": usable_features,
    "excluded_cols": excluded_cols,
    "split_strategy": "official 80/20 per-series chronological train-test split with internal validation carved from training region",
    "official_train_ratio": 0.80,
    "official_test_ratio": 0.20,
    "internal_validation_source": "validation subset taken only from the 80% training region",
    "internal_validation_ratio_within_training_region": 0.20,
    "leakage_control": "boundary purging applied between train-validation and validation-test regions"
}

schema_path = f"{PH4_PATH}/configs/nab_supervised_4model_feature_schema_80_20.json"
os.makedirs(f"{PH4_PATH}/configs", exist_ok=True)

with open(schema_path, "w") as f:
    json.dump(schema, f, indent=2)

X_train = train_df[usable_features]
y_train = train_df[TARGET_COL]

X_val = val_df[usable_features]
y_val = val_df[TARGET_COL]

X_test = test_df[usable_features]
y_test = test_df[TARGET_COL]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nLabel counts:")
print("Train:")
print(y_train.value_counts())
print("\nInternal validation:")
print(y_val.value_counts())
print("\nTest:")
print(y_test.value_counts())

print("\nSaved schema:")
print(schema_path)

In [ ]:
# =========================
# Train classical supervised models
# =========================

!pip -q install xgboost

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve
from sklearn.base import clone
from xgboost import XGBClassifier

pos_train = int((y_train == 1).sum())
neg_train = int((y_train == 0).sum())
scale_pos_weight = neg_train / max(pos_train, 1)

classical_models = {
    "RandomForest": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    "LogisticRegression": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "XGBoost": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
}

def find_best_threshold_on_validation(y_true, y_score):
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    f1_scores = (2 * precision * recall) / (precision + recall + 1e-10)

    # thresholds has length len(precision)-1
    if len(thresholds) == 0:
        return 0.5, 0.0, 0.0, 0.0

    best_idx = np.argmax(f1_scores[:-1])
    return thresholds[best_idx], f1_scores[best_idx], precision[best_idx], recall[best_idx]

def evaluate_with_threshold(y_true, y_score, threshold):
    y_pred = (y_score >= threshold).astype(int)

    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score)
    }, y_pred

classical_metrics = []
classical_predictions = []

for model_name, model in tqdm(classical_models.items(), desc="Training classical supervised models"):
    print(f"\nTraining {model_name}")

    fitted = clone(model)
    fitted.fit(X_train, y_train)

    val_score = fitted.predict_proba(X_val)[:, 1]
    test_score = fitted.predict_proba(X_test)[:, 1]

    best_threshold, val_f1, val_precision, val_recall = find_best_threshold_on_validation(
        y_val, val_score
    )

    test_metrics_default, test_pred_default = evaluate_with_threshold(
        y_test, test_score, 0.5
    )

    test_metrics_tuned, test_pred_tuned = evaluate_with_threshold(
        y_test, test_score, best_threshold
    )

    classical_metrics.append({
        "experiment_id": "PH4_SUP_NAB_002",
        "dataset": "NAB",
        "model": model_name,
        "threshold_source": "default_0.5",
        "threshold": 0.5,
        "validation_f1_at_selected_threshold": None,
        "test_precision": test_metrics_default["precision"],
        "test_recall": test_metrics_default["recall"],
        "test_f1": test_metrics_default["f1"],
        "test_roc_auc": test_metrics_default["roc_auc"],
        "n_train": len(X_train),
        "n_validation": len(X_val),
        "n_test": len(X_test),
        "n_features": len(usable_features),
        "positive_train": int((y_train == 1).sum()),
        "positive_validation": int((y_val == 1).sum()),
        "positive_test": int((y_test == 1).sum())
    })

    classical_metrics.append({
        "experiment_id": "PH4_SUP_NAB_002",
        "dataset": "NAB",
        "model": model_name,
        "threshold_source": "validation_f1_optimized",
        "threshold": best_threshold,
        "validation_f1_at_selected_threshold": val_f1,
        "test_precision": test_metrics_tuned["precision"],
        "test_recall": test_metrics_tuned["recall"],
        "test_f1": test_metrics_tuned["f1"],
        "test_roc_auc": test_metrics_tuned["roc_auc"],
        "n_train": len(X_train),
        "n_validation": len(X_val),
        "n_test": len(X_test),
        "n_features": len(usable_features),
        "positive_train": int((y_train == 1).sum()),
        "positive_validation": int((y_val == 1).sum()),
        "positive_test": int((y_test == 1).sum())
    })

    pred_df = test_df[[
        "dataset_name", "series_id", "window_id",
        "start_time", "end_time", TARGET_COL, "source_file"
    ]].copy()

    pred_df["model"] = model_name
    pred_df["y_true"] = y_test.values
    pred_df["y_score"] = test_score
    pred_df["y_pred_default_0_5"] = test_pred_default
    pred_df["y_pred_validation_tuned"] = test_pred_tuned
    pred_df["validation_selected_threshold"] = best_threshold

    classical_predictions.append(pred_df)

    model_path = f"{PH4_SUP_NAB_MODELS_PATH}/{model_name}_PH4_SUP_NAB_002.joblib"
    joblib.dump(fitted, model_path)
    print("Saved model:", model_path)

classical_metrics_df = pd.DataFrame(classical_metrics)
classical_predictions_df = pd.concat(classical_predictions, ignore_index=True)

display(classical_metrics_df.sort_values("test_f1", ascending=False))

In [ ]:
# =========================
# Train MLP / simple neural network
# =========================

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Preprocess manually for Keras
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_np = scaler.fit_transform(imputer.fit_transform(X_train))
X_val_np = scaler.transform(imputer.transform(X_val))
X_test_np = scaler.transform(imputer.transform(X_test))

y_train_np = y_train.values.astype(int)
y_val_np = y_val.values.astype(int)
y_test_np = y_test.values.astype(int)

classes = np.array([0, 1])
class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_np
)

class_weight = {
    0: class_weights_values[0],
    1: class_weights_values[1]
}

print("Class weights:", class_weight)

def build_mlp(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    return model

mlp = build_mlp(X_train_np.shape[1])

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = mlp.fit(
    X_train_np,
    y_train_np,
    validation_data=(X_val_np, y_val_np),
    epochs=50,
    batch_size=256,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)

val_score_mlp = mlp.predict(X_val_np, verbose=0).ravel()
test_score_mlp = mlp.predict(X_test_np, verbose=0).ravel()

best_threshold_mlp, val_f1_mlp, val_precision_mlp, val_recall_mlp = find_best_threshold_on_validation(
    y_val_np, val_score_mlp
)

test_metrics_mlp_default, test_pred_mlp_default = evaluate_with_threshold(
    y_test_np, test_score_mlp, 0.5
)

test_metrics_mlp_tuned, test_pred_mlp_tuned = evaluate_with_threshold(
    y_test_np, test_score_mlp, best_threshold_mlp
)

mlp_metrics_df = pd.DataFrame([
    {
        "experiment_id": "PH4_SUP_NAB_002",
        "dataset": "NAB",
        "model": "MLP",
        "threshold_source": "default_0.5",
        "threshold": 0.5,
        "validation_f1_at_selected_threshold": None,
        "test_precision": test_metrics_mlp_default["precision"],
        "test_recall": test_metrics_mlp_default["recall"],
        "test_f1": test_metrics_mlp_default["f1"],
        "test_roc_auc": test_metrics_mlp_default["roc_auc"],
        "n_train": len(X_train),
        "n_validation": len(X_val),
        "n_test": len(X_test),
        "n_features": len(usable_features),
        "positive_train": int((y_train == 1).sum()),
        "positive_validation": int((y_val == 1).sum()),
        "positive_test": int((y_test == 1).sum())
    },
    {
        "experiment_id": "PH4_SUP_NAB_002",
        "dataset": "NAB",
        "model": "MLP",
        "threshold_source": "validation_f1_optimized",
        "threshold": best_threshold_mlp,
        "validation_f1_at_selected_threshold": val_f1_mlp,
        "test_precision": test_metrics_mlp_tuned["precision"],
        "test_recall": test_metrics_mlp_tuned["recall"],
        "test_f1": test_metrics_mlp_tuned["f1"],
        "test_roc_auc": test_metrics_mlp_tuned["roc_auc"],
        "n_train": len(X_train),
        "n_validation": len(X_val),
        "n_test": len(X_test),
        "n_features": len(usable_features),
        "positive_train": int((y_train == 1).sum()),
        "positive_validation": int((y_val == 1).sum()),
        "positive_test": int((y_test == 1).sum())
    }
])

mlp_pred_df = test_df[[
    "dataset_name", "series_id", "window_id",
    "start_time", "end_time", TARGET_COL, "source_file"
]].copy()

mlp_pred_df["model"] = "MLP"
mlp_pred_df["y_true"] = y_test_np
mlp_pred_df["y_score"] = test_score_mlp
mlp_pred_df["y_pred_default_0_5"] = test_pred_mlp_default
mlp_pred_df["y_pred_validation_tuned"] = test_pred_mlp_tuned
mlp_pred_df["validation_selected_threshold"] = best_threshold_mlp

mlp_model_path = f"{PH4_SUP_NAB_MODELS_PATH}/MLP_PH4_SUP_NAB_002.keras"
mlp.save(mlp_model_path)

print("Saved MLP:", mlp_model_path)
display(mlp_metrics_df)

In [ ]:
# =========================
# Save PH4 supervised 4-model NAB outputs
# =========================

supervised_4model_metrics_df = pd.concat(
    [classical_metrics_df, mlp_metrics_df],
    ignore_index=True
)

supervised_4model_predictions_df = pd.concat(
    [classical_predictions_df, mlp_pred_df],
    ignore_index=True
)

metrics_path = f"{PH4_SUP_NAB_METRICS_PATH}/nab_supervised_4model_metrics_train_val_test.csv"
preds_path = f"{PH4_SUP_NAB_PREDS_PATH}/nab_supervised_4model_predictions_train_val_test.csv"

supervised_4model_metrics_df.to_csv(metrics_path, index=False)
supervised_4model_predictions_df.to_csv(preds_path, index=False)

excel_path = f"{PH4_PATH}/combined/ph4_nab_8model_tracker.xlsx"

# Optional: include old unsupervised results later if available, but Phase 4 sheet is supervised only
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    supervised_4model_metrics_df.sort_values(
        ["threshold_source", "test_f1"],
        ascending=[True, False]
    ).to_excel(writer, sheet_name="PH4_Supervised_NAB", index=False)

print("Saved metrics:", metrics_path)
print("Saved predictions:", preds_path)
print("Saved Excel:", excel_path)

display(
    supervised_4model_metrics_df
    .sort_values("test_f1", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
# =========================
# PH4_LOG_002 - Supervised NAB 4-model validation
# =========================

from datetime import datetime

ph4_supervised_log_path = f"{PH4_LOGS_PATH}/PH4_LOG_002_supervised_nab_4model_validation.md"

best_supervised = (
    supervised_4model_metrics_df
    .sort_values("test_f1", ascending=False)
    .iloc[0]
)

best_auc = (
    supervised_4model_metrics_df
    .sort_values("test_roc_auc", ascending=False)
    .iloc[0]
)

log_content = f"""# PH4_LOG_002_supervised_nab_4model_validation

## Timestamp
{datetime.now()}

## Experiment ID
PH4_SUP_NAB_002

## Experiment Title
Supervised Validation on NAB Using Four Models

## Objective
To perform the official Phase 4 supervised validation experiment using the four models requested by the supervisor.

The purpose of this phase is to validate:
- the Phase 3 feature-window representation
- the supervised modeling pipeline
- the chronological split protocol
- the evaluation protocol

## Dataset
NAB

## Dataset Role
NAB is used as a labeled time-series benchmark dataset for supervised validation.

## Models
The following supervised models were implemented:

1. Random Forest
2. XGBoost
3. Logistic Regression
4. Multilayer Perceptron (MLP)

## Split Strategy
An official 80/20 chronological train-test split was used.

To avoid tuning the decision threshold directly on the test set, an internal validation subset was carved only from the 80% training region.

Final split sizes:
- Training rows: {len(train_df)}
- Internal validation rows: {len(val_df)}
- Test rows: {len(test_df)}

Label distribution:

Training:
{y_train.value_counts().to_string()}

Internal validation:
{y_val.value_counts().to_string()}

Test:
{y_test.value_counts().to_string()}

## Feature Schema
Number of usable numeric features:

{len(usable_features)}

Features:
{chr(10).join(["- " + c for c in usable_features])}

## Evaluation Metrics
The following metrics were computed on the final test set:
- Precision
- Recall
- F1-score
- ROC-AUC

Two threshold settings were reported:
1. Default threshold = 0.5
2. Validation-optimized threshold selected using F1-score on the internal validation subset

## Results

{supervised_4model_metrics_df.sort_values("test_f1", ascending=False).to_markdown(index=False)}

## Best Model by F1-score
- Model: {best_supervised["model"]}
- Threshold source: {best_supervised["threshold_source"]}
- Test F1-score: {best_supervised["test_f1"]}
- Test Precision: {best_supervised["test_precision"]}
- Test Recall: {best_supervised["test_recall"]}
- Test ROC-AUC: {best_supervised["test_roc_auc"]}

## Best Model by ROC-AUC
- Model: {best_auc["model"]}
- Threshold source: {best_auc["threshold_source"]}
- Test ROC-AUC: {best_auc["test_roc_auc"]}
- Test F1-score: {best_auc["test_f1"]}

## Interpretation
The supervised validation experiment executed successfully.

Random Forest achieved the best F1-score after validation-based threshold tuning, while XGBoost achieved the highest ROC-AUC.

However, the overall supervised performance remains weak. This is mainly due to the extreme imbalance and sparsity of anomaly labels at the window level.

The default 0.5 threshold was ineffective for most models, confirming that fixed-threshold supervised classification is not suitable under severe anomaly sparsity.

Validation-based threshold tuning improved detection behavior but did not fully overcome the weak supervision problem.

## Methodological Conclusion
The supervised Phase 4 experiment validates that the feature-window pipeline and supervised model workflow are operational.

However, the low supervised F1-scores show that sparse window-level labels limit supervised classification performance.

This finding supports the need for later unsupervised anomaly detection experiments in Phase 5, while keeping Phase 4 strictly focused on supervised validation.

## Saved Outputs
Metrics:

`{metrics_path}`

Predictions:

`{preds_path}`

Excel tracker:

`{excel_path}`

Split audit:

`{split_audit_path}`

Feature schema:

`{schema_path}`

## Status
PH4_SUP_NAB_002 completed and logged.

## Next Step
Update the Overleaf Phase 4 supervised validation section and present these results in the Monday meeting.
"""

with open(ph4_supervised_log_path, "w", encoding="utf-8") as f:
    f.write(log_content)

print("Phase 4 supervised NAB log saved:")
print(ph4_supervised_log_path)

In [ ]:
# =========================
# Phase 4 supervised NAB comparison plot
# =========================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_PATH = "/content/drive/MyDrive/Master/Thesis/Code"
PH4_PATH = f"{BASE_PATH}/processed_data/PH4"

metrics_path = f"{PH4_PATH}/supervised/NAB/metrics/nab_supervised_4model_metrics_train_val_test.csv"
figures_path = f"{PH4_PATH}/figures/NAB"
combined_path = f"{PH4_PATH}/combined"

os.makedirs(figures_path, exist_ok=True)
os.makedirs(combined_path, exist_ok=True)

df = pd.read_csv(metrics_path)

# Keep only validation-optimized threshold rows
plot_df = df[df["threshold_source"] == "validation_f1_optimized"].copy()

# Keep the 4 supervised models in a clean order
model_order = ["RandomForest", "XGBoost", "MLP", "LogisticRegression"]
plot_df["model"] = pd.Categorical(plot_df["model"], categories=model_order, ordered=True)
plot_df = plot_df.sort_values("model").reset_index(drop=True)

# Save simplified summary CSV
summary_cols = [
    "model",
    "threshold",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_roc_auc"
]
summary_df = plot_df[summary_cols].copy()

summary_csv_path = f"{combined_path}/phase4_supervised_nab_summary_tuned.csv"
summary_df.to_csv(summary_csv_path, index=False)

print("Saved summary CSV:")
print(summary_csv_path)
display(summary_df)

# Plot grouped bar chart
models = plot_df["model"].tolist()
precision_vals = plot_df["test_precision"].tolist()
recall_vals = plot_df["test_recall"].tolist()
f1_vals = plot_df["test_f1"].tolist()
roc_vals = plot_df["test_roc_auc"].tolist()

x = np.arange(len(models))
width = 0.2

plt.figure(figsize=(10, 6))
plt.bar(x - 1.5*width, precision_vals, width, label="Precision")
plt.bar(x - 0.5*width, recall_vals, width, label="Recall")
plt.bar(x + 0.5*width, f1_vals, width, label="F1-score")
plt.bar(x + 1.5*width, roc_vals, width, label="ROC-AUC")

plt.xticks(x, models, rotation=15)
plt.ylabel("Score")
plt.title("Phase 4 Supervised Validation on NAB (Validation-Optimized Thresholds)")
plt.legend()
plt.tight_layout()

plot_path = f"{figures_path}/phase4_supervised_nab_comparison.png"
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved plot:")
print(plot_path)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

BASE_PATH = "/content/drive/MyDrive/Master/Thesis/Code"
PH4_PATH = f"{BASE_PATH}/processed_data/PH4"
figures_path = f"{PH4_PATH}/figures/NAB"
os.makedirs(figures_path, exist_ok=True)

data = [
    ["Random Forest", "Supervised", 0.017404, 0.500000, 0.033637, 0.702491],
    ["XGBoost", "Supervised", 0.014525, 0.270833, 0.027572, 0.725816],
    ["MLP", "Supervised", 0.006897, 0.041667, 0.011834, 0.544555],
    ["Logistic Regression", "Supervised", 0.000000, 0.000000, 0.000000, 0.541310],
    ["Autoencoder", "Anomaly Detection", 0.090000, 0.284000, 0.136000, 0.821000],
    ["Isolation Forest", "Anomaly Detection", 0.067000, 0.649000, 0.122000, 0.920000],
    ["LOF", "Anomaly Detection", 0.056000, 0.676000, 0.104000, 0.813000],
    ["One-Class SVM", "Anomaly Detection", 0.064000, 0.243000, 0.102000, 0.909000],
]

df = pd.DataFrame(data, columns=["Model", "Type", "Precision", "Recall", "F1-score", "ROC-AUC"])

x = np.arange(len(df))
width = 0.2

plt.figure(figsize=(13, 6))
plt.bar(x - 1.5*width, df["Precision"], width, label="Precision")
plt.bar(x - 0.5*width, df["Recall"], width, label="Recall")
plt.bar(x + 0.5*width, df["F1-score"], width, label="F1-score")
plt.bar(x + 1.5*width, df["ROC-AUC"], width, label="ROC-AUC")

plt.xticks(x, df["Model"], rotation=30, ha="right")
plt.ylabel("Score")
plt.title("Extended NAB Benchmark: Supervised vs Anomaly-Detection Models")
plt.legend()
plt.tight_layout()

plot_path = f"{figures_path}/nab_extended_8model_benchmark.png"
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved plot:", plot_path)

In [ ]:
# =========================
# PH4_SUP_NAB_003 - Feature Set V2 Setup
# =========================

from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import json
import joblib
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

BASE_PATH = "/content/drive/MyDrive/Master/Thesis/Code"
PROCESSED_DATA_PATH = f"{BASE_PATH}/processed_data"
PH2_PATH = f"{PROCESSED_DATA_PATH}/PH2"
PH3_PATH = f"{PROCESSED_DATA_PATH}/PH3"
PH4_PATH = f"{PROCESSED_DATA_PATH}/PH4"
PH4_LOGS_PATH = f"{BASE_PATH}/experiment_logs/PH4"

# Existing PH3 NAB V1 feature windows
NAB_PH3_FEATURES_DIR = f"{PH3_PATH}/NAB/raw_windows/features"

# PH4 V2 output folders
PH4_NAB_SUP_PATH = f"{PH4_PATH}/supervised/NAB"
PH4_NAB_V2_PATH = f"{PH4_NAB_SUP_PATH}/feature_set_v2"

V2_INPUTS_PATH = f"{PH4_NAB_V2_PATH}/inputs"
V2_SPLITS_PATH = f"{PH4_NAB_V2_PATH}/splits"
V2_MODELS_PATH = f"{PH4_NAB_V2_PATH}/models"
V2_PREDS_PATH = f"{PH4_NAB_V2_PATH}/predictions"
V2_METRICS_PATH = f"{PH4_NAB_V2_PATH}/metrics"
V2_CONFIGS_PATH = f"{PH4_NAB_V2_PATH}/configs"
V2_FIGURES_PATH = f"{PH4_PATH}/figures/NAB/feature_set_v2"
V2_COMBINED_PATH = f"{PH4_PATH}/combined"

for p in [
    V2_INPUTS_PATH,
    V2_SPLITS_PATH,
    V2_MODELS_PATH,
    V2_PREDS_PATH,
    V2_METRICS_PATH,
    V2_CONFIGS_PATH,
    V2_FIGURES_PATH,
    V2_COMBINED_PATH,
    PH4_LOGS_PATH
]:
    os.makedirs(p, exist_ok=True)

RANDOM_STATE = 42
TARGET_COL = "label"

SIGNAL_COLS = ["mean_power", "max_power", "std_power"]

METADATA_COLS = [
    "dataset_name",
    "series_id",
    "window_id",
    "start_time",
    "end_time",
    "window_size_minutes",
    "step_size_minutes",
    "sampling_minutes",
    "source_file"
]

print("Feature Set V2 folders are ready.")

In [ ]:
# =========================
# Inspect existing PH3 NAB feature-window files
# =========================

EXCLUDE_FILE_KEYWORDS = [
    "summary",
    "failed",
    "audit",
    "metrics",
    "predictions",
    "splits"
]

def discover_feature_files(root_dir):
    all_csvs = glob.glob(os.path.join(root_dir, "*.csv"))
    return sorted([
        f for f in all_csvs
        if not any(k in os.path.basename(f).lower() for k in EXCLUDE_FILE_KEYWORDS)
    ])

nab_ph3_files = discover_feature_files(NAB_PH3_FEATURES_DIR)

print("PH3 NAB feature-window files found:", len(nab_ph3_files))

if len(nab_ph3_files) == 0:
    raise FileNotFoundError("No PH3 NAB feature-window files found.")

# Inspect columns across files
all_columns = set()
sample_shapes = []

for fp in tqdm(nab_ph3_files, desc="Inspecting PH3 NAB files"):
    temp = pd.read_csv(fp, nrows=5)
    all_columns.update(temp.columns.tolist())
    sample_shapes.append((os.path.basename(fp), temp.shape[1]))

all_columns = sorted(all_columns)

print("\nCurrent PH3 available columns:")
for c in all_columns:
    print("-", c)

numeric_feature_like_cols = [
    c for c in all_columns
    if c not in METADATA_COLS + [TARGET_COL, "split"]
]

print("\nCurrent PH3 non-metadata columns:")
print(len(numeric_feature_like_cols))
print(numeric_feature_like_cols)

In [ ]:
# =========================
# Discover PH2 standardized NAB files
# =========================

required_cols = {"timestamp", "mean_power", "max_power", "std_power", "label"}

all_csvs = glob.glob(os.path.join(PH2_PATH, "**", "*.csv"), recursive=True)

nab_ph2_files = []

for fp in tqdm(all_csvs, desc="Searching PH2 CSV files"):
    path_lower = fp.lower()
    if "nab" not in path_lower:
        continue

    try:
        temp = pd.read_csv(fp, nrows=5)
        cols = set(temp.columns)
        if required_cols.issubset(cols):
            nab_ph2_files.append(fp)
    except Exception:
        pass

nab_ph2_files = sorted(nab_ph2_files)

print("PH2 standardized NAB files found:", len(nab_ph2_files))

if len(nab_ph2_files) == 0:
    raise FileNotFoundError(
        "No PH2 standardized NAB files found with timestamp, mean_power, max_power, std_power, label."
    )

print("\nFirst 10 files:")
for fp in nab_ph2_files[:10]:
    print("-", fp)

In [ ]:
# =========================
# Diagnose PH2 CSV schemas
# =========================

import os
import glob
import pandas as pd
from collections import Counter, defaultdict
from tqdm.auto import tqdm

BASE_PATH = "/content/drive/MyDrive/Master/Thesis/Code"
PROCESSED_DATA_PATH = f"{BASE_PATH}/processed_data"
PH2_PATH = f"{PROCESSED_DATA_PATH}/PH2"
PH3_PATH = f"{PROCESSED_DATA_PATH}/PH3"

all_csvs = glob.glob(os.path.join(PH2_PATH, "**", "*.csv"), recursive=True)

print("Total PH2 CSV files found:", len(all_csvs))

schema_counter = Counter()
schema_examples = defaultdict(list)

for fp in tqdm(all_csvs, desc="Inspecting PH2 CSV schemas"):
    try:
        temp = pd.read_csv(fp, nrows=5)
        cols = tuple(temp.columns.tolist())
        schema_counter[cols] += 1
        if len(schema_examples[cols]) < 5:
            schema_examples[cols].append(fp)
    except Exception as e:
        pass

print("\nTop PH2 schemas:")
for i, (cols, count) in enumerate(schema_counter.most_common(20), start=1):
    print(f"\nSchema {i} | Count: {count}")
    print(cols)
    print("Examples:")
    for ex in schema_examples[cols]:
        print(" -", ex)

In [ ]:
# =========================
# Flexible NAB candidate discovery
# =========================

candidate_files = []

for fp in tqdm(all_csvs, desc="Finding possible NAB/labelled time-series files"):
    try:
        temp = pd.read_csv(fp, nrows=5)
        cols = set(temp.columns.str.lower())

        has_time = any(c in cols for c in ["timestamp", "time", "datetime", "date"])
        has_label = "label" in cols
        has_numeric_signal = any(c in cols for c in [
            "value", "power", "mean_power", "max_power", "std_power",
            "consumption", "energy"
        ])

        # do not require 'nab' in path yet
        if has_time and has_label and has_numeric_signal:
            candidate_files.append(fp)

    except Exception:
        pass

print("Candidate labelled time-series files:", len(candidate_files))

print("\nFirst 30 candidates:")
for fp in candidate_files[:30]:
    print("-", fp)
    print("  columns:", pd.read_csv(fp, nrows=1).columns.tolist())

In [ ]:
# =========================
# Corrected PH2 NAB discovery
# NAB schema is: timestamp, value, label
# =========================

import os
import glob
import pandas as pd
from tqdm.auto import tqdm

required_cols = {"timestamp", "value", "label"}

all_csvs = glob.glob(os.path.join(PH2_PATH, "**", "*.csv"), recursive=True)

nab_ph2_files = []

for fp in tqdm(all_csvs, desc="Finding NAB PH2 files"):
    try:
        if "nab" not in fp.lower():
            continue

        temp = pd.read_csv(fp, nrows=5)
        cols = set(temp.columns)

        if required_cols.issubset(cols):
            nab_ph2_files.append(fp)

    except Exception:
        pass

nab_ph2_files = sorted(nab_ph2_files)

print("NAB PH2 files found:", len(nab_ph2_files))

if len(nab_ph2_files) == 0:
    raise FileNotFoundError("No NAB PH2 files found with timestamp, value, label.")

print("\nFirst 10 NAB PH2 files:")
for fp in nab_ph2_files[:10]:
    print("-", fp)

In [ ]:
# =========================
# Corrected Feature Set V2 Functions for NAB single value signal
# =========================

import numpy as np
import pandas as pd

VALUE_COL = "value"

def safe_skew(x):
    x = pd.Series(x).dropna()
    if len(x) < 3:
        return 0.0
    val = x.skew()
    return 0.0 if pd.isna(val) else float(val)

def safe_kurtosis(x):
    x = pd.Series(x).dropna()
    if len(x) < 4:
        return 0.0
    val = x.kurtosis()
    return 0.0 if pd.isna(val) else float(val)

def shannon_entropy(x, bins=10):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    if len(x) < 2:
        return 0.0
    if np.nanmax(x) == np.nanmin(x):
        return 0.0

    hist, _ = np.histogram(x, bins=bins)
    probs = hist / max(hist.sum(), 1)
    probs = probs[probs > 0]
    return float(-np.sum(probs * np.log2(probs)))

def autocorr_lag(x, lag=1):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    if len(x) <= lag + 1:
        return 0.0

    x1 = x[:-lag]
    x2 = x[lag:]

    if np.std(x1) == 0 or np.std(x2) == 0:
        return 0.0

    val = np.corrcoef(x1, x2)[0, 1]
    return 0.0 if np.isnan(val) else float(val)

def simple_peak_count(x):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)

    if len(x) < 3:
        return 0

    mu = np.mean(x)
    sigma = np.std(x)
    threshold = mu + 2 * sigma

    middle = x[1:-1]
    left = x[:-2]
    right = x[2:]

    peaks = (middle > left) & (middle > right) & (middle > threshold)
    return int(np.sum(peaks))

def slope_feature(x):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)

    if len(x) < 2:
        return 0.0

    t = np.arange(len(x))

    try:
        slope = np.polyfit(t, x, 1)[0]
        return float(slope)
    except Exception:
        return 0.0

def value_signal_features(values, prefix="value"):
    x = pd.Series(values).astype(float).dropna()

    if len(x) == 0:
        return {f"{prefix}_empty": 1}

    arr = x.values

    diffs = np.diff(arr) if len(arr) > 1 else np.array([0.0])
    abs_diffs = np.abs(diffs)

    denom = np.abs(arr[:-1]) + 1e-6 if len(arr) > 1 else np.array([1.0])
    rate_change = diffs / denom if len(arr) > 1 else np.array([0.0])
    rate_change = np.clip(rate_change, -1e6, 1e6)

    rolling_var = x.rolling(window=5, min_periods=2).var().dropna()
    if len(rolling_var) == 0:
        rolling_var = pd.Series([0.0])

    rolling_mean = x.rolling(window=5, min_periods=2).mean().dropna()
    if len(rolling_mean) == 0:
        rolling_mean = pd.Series([float(np.mean(arr))])

    mean_val = float(np.mean(arr))
    std_val = float(np.std(arr))
    min_val = float(np.min(arr))
    max_val = float(np.max(arr))

    features = {
        # Base statistical descriptors
        f"{prefix}_mean": mean_val,
        f"{prefix}_std": std_val,
        f"{prefix}_var": float(np.var(arr)),
        f"{prefix}_min": min_val,
        f"{prefix}_max": max_val,
        f"{prefix}_range": max_val - min_val,
        f"{prefix}_median": float(np.median(arr)),
        f"{prefix}_sum": float(np.sum(arr)),
        f"{prefix}_energy_sum": float(np.sum(np.abs(arr))),
        f"{prefix}_q25": float(np.quantile(arr, 0.25)),
        f"{prefix}_q75": float(np.quantile(arr, 0.75)),
        f"{prefix}_iqr": float(np.quantile(arr, 0.75) - np.quantile(arr, 0.25)),
        f"{prefix}_delta": float(arr[-1] - arr[0]) if len(arr) > 1 else 0.0,

        # Shape descriptors
        f"{prefix}_skewness": safe_skew(arr),
        f"{prefix}_kurtosis": safe_kurtosis(arr),
        f"{prefix}_cv": float(std_val / (abs(mean_val) + 1e-6)),

        # Rolling descriptors
        f"{prefix}_rolling_var_mean": float(rolling_var.mean()),
        f"{prefix}_rolling_var_std": float(rolling_var.std()) if len(rolling_var) > 1 else 0.0,
        f"{prefix}_rolling_var_max": float(rolling_var.max()),
        f"{prefix}_rolling_mean_mean": float(rolling_mean.mean()),
        f"{prefix}_rolling_mean_std": float(rolling_mean.std()) if len(rolling_mean) > 1 else 0.0,

        # Change descriptors
        f"{prefix}_diff_mean": float(np.mean(diffs)),
        f"{prefix}_diff_std": float(np.std(diffs)),
        f"{prefix}_diff_min": float(np.min(diffs)),
        f"{prefix}_diff_max": float(np.max(diffs)),
        f"{prefix}_diff_abs_mean": float(np.mean(abs_diffs)),
        f"{prefix}_diff_abs_max": float(np.max(abs_diffs)),
        f"{prefix}_diff_abs_sum": float(np.sum(abs_diffs)),

        # Rate-of-change descriptors
        f"{prefix}_rate_change_mean": float(np.mean(rate_change)),
        f"{prefix}_rate_change_std": float(np.std(rate_change)),
        f"{prefix}_rate_change_abs_mean": float(np.mean(np.abs(rate_change))),
        f"{prefix}_rate_change_max": float(np.max(rate_change)),
        f"{prefix}_rate_change_min": float(np.min(rate_change)),

        # Peak descriptors
        f"{prefix}_peak_count": simple_peak_count(arr),
        f"{prefix}_peak_ratio": float(simple_peak_count(arr) / max(len(arr), 1)),

        # Entropy and autocorrelation
        f"{prefix}_entropy": shannon_entropy(arr, bins=10),
        f"{prefix}_autocorr_lag1": autocorr_lag(arr, lag=1),
        f"{prefix}_autocorr_lag2": autocorr_lag(arr, lag=2),
        f"{prefix}_autocorr_lag5": autocorr_lag(arr, lag=5),

        # Trend and activity
        f"{prefix}_slope": slope_feature(arr),
        f"{prefix}_zero_ratio": float(np.mean(arr == 0)),
        f"{prefix}_nonzero_ratio": float(np.mean(arr != 0)),
        f"{prefix}_above_mean_ratio": float(np.mean(arr > mean_val)),
        f"{prefix}_below_mean_ratio": float(np.mean(arr < mean_val)),
    }

    return features

def compute_window_features(window_df):
    return value_signal_features(window_df["value"], prefix="value")

In [ ]:
# =========================
# Generate NAB Feature Set V2 from actual PH2 schema: timestamp, value, label
# =========================

WINDOW_MINUTES = 60
STEP_MINUTES = 30

v2_rows = []
generation_summary = []

for fp in tqdm(nab_ph2_files, desc="Generating NAB Feature Set V2"):
    try:
        df = pd.read_csv(fp)

        missing = required_cols - set(df.columns)
        if missing:
            generation_summary.append({
                "source_file": os.path.basename(fp),
                "status": "failed_missing_columns",
                "reason": str(missing),
                "n_windows": 0
            })
            continue

        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        df["label"] = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int)
        df["label"] = (df["label"] > 0).astype(int)

        df = df.dropna(subset=["timestamp", "value"]).copy()
        df = df.sort_values("timestamp").reset_index(drop=True)

        if len(df) < 5:
            generation_summary.append({
                "source_file": os.path.basename(fp),
                "status": "skipped_too_short",
                "reason": "too few rows",
                "n_windows": 0
            })
            continue

        # Estimate sampling interval dynamically
        time_diffs = df["timestamp"].diff().dt.total_seconds().dropna() / 60
        median_sampling_minutes = float(time_diffs.median()) if len(time_diffs) > 0 else 1.0

        if pd.isna(median_sampling_minutes) or median_sampling_minutes <= 0:
            median_sampling_minutes = 1.0

        expected_samples = max(2, int(np.floor(WINDOW_MINUTES / median_sampling_minutes)))
        min_valid_samples = max(2, int(np.ceil(0.5 * expected_samples)))

        series_id = os.path.basename(fp)
        dataset_name = "NAB"

        start_time = df["timestamp"].min()
        final_time = df["timestamp"].max()

        current_start = start_time
        window_id = 0
        file_window_count = 0

        while current_start + pd.Timedelta(minutes=WINDOW_MINUTES) <= final_time + pd.Timedelta(minutes=median_sampling_minutes):
            current_end_exclusive = current_start + pd.Timedelta(minutes=WINDOW_MINUTES)

            window_df = df[
                (df["timestamp"] >= current_start) &
                (df["timestamp"] < current_end_exclusive)
            ].copy()

            if len(window_df) >= min_valid_samples:
                features = compute_window_features(window_df)

                row = {
                    "dataset_name": dataset_name,
                    "series_id": series_id,
                    "window_id": window_id,
                    "start_time": current_start,
                    "end_time": window_df["timestamp"].max(),
                    "window_size_minutes": WINDOW_MINUTES,
                    "step_size_minutes": STEP_MINUTES,
                    "sampling_minutes": median_sampling_minutes,
                    "valid_sample_count": len(window_df),
                    "expected_sample_count": expected_samples,
                    "duration_minutes_actual": (
                        window_df["timestamp"].max() - current_start
                    ).total_seconds() / 60,
                    # any-point overlap label
                    "label": int(window_df["label"].max() > 0),
                    "source_file": os.path.basename(fp)
                }

                row.update(features)
                v2_rows.append(row)
                file_window_count += 1

            window_id += 1
            current_start = current_start + pd.Timedelta(minutes=STEP_MINUTES)

        generation_summary.append({
            "source_file": os.path.basename(fp),
            "status": "success",
            "reason": "",
            "n_windows": file_window_count,
            "n_rows_input": len(df),
            "sampling_minutes": median_sampling_minutes,
            "expected_samples_per_window": expected_samples,
            "min_valid_samples": min_valid_samples,
            "start_time": start_time,
            "end_time": final_time
        })

    except Exception as e:
        generation_summary.append({
            "source_file": os.path.basename(fp),
            "status": "failed_exception",
            "reason": str(e),
            "n_windows": 0
        })

nab_v2_df = pd.DataFrame(v2_rows)
v2_generation_summary_df = pd.DataFrame(generation_summary)

v2_features_path = f"{V2_INPUTS_PATH}/nab_feature_set_v2_windows.csv"
v2_generation_summary_path = f"{V2_INPUTS_PATH}/nab_feature_set_v2_generation_summary.csv"

nab_v2_df.to_csv(v2_features_path, index=False)
v2_generation_summary_df.to_csv(v2_generation_summary_path, index=False)

print("Saved V2 feature windows:")
print(v2_features_path)

print("\nSaved V2 generation summary:")
print(v2_generation_summary_path)

print("\nV2 shape:", nab_v2_df.shape)
print("\nV2 label distribution:")
print(nab_v2_df["label"].value_counts(dropna=False))

print("\nV2 columns:", len(nab_v2_df.columns))

print("\nGeneration status:")
print(v2_generation_summary_df["status"].value_counts())

In [ ]:
# =========================
# V2 Fairness Audit: align V2 with V1-compatible PH3 NAB series
# =========================

import os
import glob
import pandas as pd

NAB_PH3_FEATURES_DIR = f"{PH3_PATH}/NAB/raw_windows/features"

EXCLUDE_FILE_KEYWORDS = [
    "summary",
    "failed",
    "audit",
    "metrics",
    "predictions",
    "splits",
]

def discover_feature_files(root_dir):
    all_csvs = glob.glob(os.path.join(root_dir, "*.csv"))
    return sorted([
        f for f in all_csvs
        if not any(k in os.path.basename(f).lower() for k in EXCLUDE_FILE_KEYWORDS)
    ])

def normalize_series_key(filename):
    name = os.path.basename(str(filename))
    name = name.replace("_feature_windows.csv", "")
    name = name.replace(".csv", "")

    # handles possible duplicate naming like nab_art_daily_flatmiddle.csv
    if name.startswith("nab_"):
        name = name[4:]

    return name

# V1 Phase 3-compatible series
v1_files = discover_feature_files(NAB_PH3_FEATURES_DIR)
v1_keys = sorted(set(normalize_series_key(f) for f in v1_files))

# V2 generated series
nab_v2_df["series_key"] = nab_v2_df["source_file"].apply(normalize_series_key)
v2_keys = sorted(nab_v2_df["series_key"].unique())

common_keys = sorted(set(v1_keys).intersection(set(v2_keys)))
v1_only = sorted(set(v1_keys) - set(v2_keys))
v2_only = sorted(set(v2_keys) - set(v1_keys))

print("V1 PH3-compatible series:", len(v1_keys))
print("V2 generated series:", len(v2_keys))
print("Common aligned series:", len(common_keys))
print("V1-only series:", len(v1_only))
print("V2-only series:", len(v2_only))

print("\nV2-only series:")
for x in v2_only:
    print("-", x)

print("\nV1-only series:")
for x in v1_only:
    print("-", x)

# Create aligned V2 dataset for fair V1 vs V2 comparison
nab_v2_aligned_df = nab_v2_df[nab_v2_df["series_key"].isin(common_keys)].copy()

v2_aligned_path = f"{V2_INPUTS_PATH}/nab_feature_set_v2_windows_aligned_to_v1.csv"
nab_v2_aligned_df.to_csv(v2_aligned_path, index=False)

print("\nV2 all shape:", nab_v2_df.shape)
print("V2 aligned shape:", nab_v2_aligned_df.shape)

print("\nV2 all label distribution:")
print(nab_v2_df["label"].value_counts())

print("\nV2 aligned label distribution:")
print(nab_v2_aligned_df["label"].value_counts())

print("\nSaved aligned V2:")
print(v2_aligned_path)

In [ ]:
# =========================
# Use aligned V2 as the official Feature Set V2 experiment dataset
# =========================

nab_v2_df = nab_v2_aligned_df.copy()

print("Official V2 dataset for training is now V1-aligned.")
print("Shape:", nab_v2_df.shape)
print("Label distribution:")
print(nab_v2_df["label"].value_counts())

In [ ]:
# =========================
# Feature Set V2 schema - corrected metadata exclusion
# =========================

excluded_cols = METADATA_COLS + [
    "label",
    "split",
    "series_key",
    "duration_minutes_actual",
    "valid_sample_count",
    "expected_sample_count"
]

candidate_features = [
    c for c in train_df.columns
    if c not in excluded_cols
    and pd.api.types.is_numeric_dtype(train_df[c])
]

usable_v2_features = []

for c in candidate_features:
    if train_df[c].notna().sum() == 0:
        continue
    if train_df[c].nunique(dropna=True) <= 1:
        continue
    usable_v2_features.append(c)

print("Usable V2 feature count:", len(usable_v2_features))
print(usable_v2_features)

v2_schema = {
    "experiment_id": "PH4_SUP_NAB_003",
    "dataset": "NAB",
    "feature_set": "V2",
    "target_col": "label",
    "source": "Generated from PH2 standardized NAB series with schema timestamp, value, label",
    "comparison_scope": "V2 aligned to the same 51 PH3-compatible NAB series used in V1",
    "reason_for_regeneration": (
        "Skewness, kurtosis, entropy, autocorrelation, peak count, rolling variance, "
        "and rate-of-change descriptors require within-window time-series values."
    ),
    "window_size_minutes": 60,
    "step_size_minutes": 30,
    "split_strategy": "official 80/20 chronological train-test split with internal validation carved from training region",
    "official_train_ratio": 0.80,
    "official_test_ratio": 0.20,
    "internal_validation_ratio_within_training_region": 0.20,
    "feature_cols": usable_v2_features,
    "n_features": len(usable_v2_features),
    "excluded_cols": excluded_cols
}

v2_schema_path = f"{V2_CONFIGS_PATH}/nab_feature_set_v2_schema.json"

with open(v2_schema_path, "w") as f:
    json.dump(v2_schema, f, indent=2, default=str)

X_train = train_df[usable_v2_features]
y_train = train_df["label"]

X_val = val_df[usable_v2_features]
y_val = val_df["label"]

X_test = test_df[usable_v2_features]
y_test = test_df["label"]

print("Saved schema:", v2_schema_path)
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nLabel counts:")
print("Train:")
print(y_train.value_counts())
print("\nValidation:")
print(y_val.value_counts())
print("\nTest:")
print(y_test.value_counts())

In [ ]:
# =========================
# Feature Set V2 official 80/20 split with internal validation
# =========================

nab_v2_df["start_time"] = pd.to_datetime(nab_v2_df["start_time"], errors="coerce")
nab_v2_df["end_time"] = pd.to_datetime(nab_v2_df["end_time"], errors="coerce")
nab_v2_df["label"] = nab_v2_df["label"].astype(int)

def chronological_80_20_with_internal_validation(
    df,
    train_test_ratio=0.80,
    internal_val_ratio_within_train=0.20
):
    split_frames = []

    for series_id, g in tqdm(df.groupby("series_id", sort=False), desc="Splitting V2 NAB series"):
        g = g.sort_values("start_time").reset_index(drop=True)

        n = len(g)
        if n < 5:
            temp = g.copy()
            temp["split"] = "dropped_too_short"
            split_frames.append(temp)
            continue

        train_region_end = int(np.floor(train_test_ratio * n))
        train_region_end = max(2, min(train_region_end, n - 1))

        train_region = g.iloc[:train_region_end].copy()
        test_part = g.iloc[train_region_end:].copy()

        internal_train_end = int(
            np.floor((1 - internal_val_ratio_within_train) * len(train_region))
        )
        internal_train_end = max(1, min(internal_train_end, len(train_region) - 1))

        train_part = train_region.iloc[:internal_train_end].copy()
        val_part = train_region.iloc[internal_train_end:].copy()

        last_train_end = train_part["end_time"].max()
        val_keep = val_part["start_time"] >= last_train_end
        dropped_val = val_part.loc[~val_keep].copy()
        val_part = val_part.loc[val_keep].copy()

        if len(val_part) > 0:
            last_val_end = val_part["end_time"].max()
        else:
            last_val_end = last_train_end

        test_keep = test_part["start_time"] >= last_val_end
        dropped_test = test_part.loc[~test_keep].copy()
        test_part = test_part.loc[test_keep].copy()

        train_part["split"] = "train"
        val_part["split"] = "internal_validation"
        test_part["split"] = "test"
        dropped_val["split"] = "dropped_purge_validation"
        dropped_test["split"] = "dropped_purge_test"

        split_frames.extend([
            train_part,
            val_part,
            test_part,
            dropped_val,
            dropped_test
        ])

    return pd.concat(split_frames, ignore_index=True)

nab_v2_split_df = chronological_80_20_with_internal_validation(nab_v2_df)

v2_split_audit_path = f"{V2_SPLITS_PATH}/nab_feature_set_v2_80_20_split_audit.csv"
nab_v2_split_df.to_csv(v2_split_audit_path, index=False)

train_df = nab_v2_split_df[nab_v2_split_df["split"] == "train"].copy()
val_df = nab_v2_split_df[nab_v2_split_df["split"] == "internal_validation"].copy()
test_df = nab_v2_split_df[nab_v2_split_df["split"] == "test"].copy()

print("Split counts:")
print(nab_v2_split_df["split"].value_counts())

print("\nTrain:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

print("\nSaved split audit:")
print(v2_split_audit_path)

In [ ]:
# =========================
# Diagnose duplicate V2 series keys after alignment
# =========================

dup_check = (
    nab_v2_df[["series_key", "series_id", "source_file"]]
    .drop_duplicates()
    .groupby("series_key")
    .agg(
        n_series_id=("series_id", "nunique"),
        n_source_file=("source_file", "nunique"),
        series_ids=("series_id", lambda x: sorted(set(x))),
        source_files=("source_file", lambda x: sorted(set(x)))
    )
    .reset_index()
)

duplicates = dup_check[dup_check["n_series_id"] > 1]

print("Unique series_key:", nab_v2_df["series_key"].nunique())
print("Unique series_id:", nab_v2_df["series_id"].nunique())
print("Duplicate normalized series keys:", len(duplicates))

display(duplicates)

In [ ]:
# =========================
# Clean duplicate normalized series keys for fair V1 vs V2 comparison
# =========================

# Add priority: prefer files from nab_all_series if path/source info is available
# If only source_file exists, this still safely keeps the first sorted source per series_key.

dedup_parts = []

for key, g in nab_v2_df.groupby("series_key", sort=False):
    unique_sources = sorted(g["source_file"].dropna().unique())

    # Keep first source deterministically
    chosen_source = unique_sources[0]

    # If there are multiple, prefer non-prefixed source over duplicate root-style source if applicable
    # Example: prefer art_daily_flatmiddle.csv over nab_art_daily_flatmiddle.csv
    non_prefixed = [s for s in unique_sources if not str(s).startswith("nab_")]
    if len(non_prefixed) > 0:
        chosen_source = sorted(non_prefixed)[0]

    dedup_parts.append(g[g["source_file"] == chosen_source].copy())

nab_v2_df = pd.concat(dedup_parts, ignore_index=True)

print("After duplicate cleanup:")
print("Shape:", nab_v2_df.shape)
print("Unique series_key:", nab_v2_df["series_key"].nunique())
print("Unique series_id:", nab_v2_df["series_id"].nunique())
print("Label distribution:")
print(nab_v2_df["label"].value_counts())

v2_aligned_dedup_path = f"{V2_INPUTS_PATH}/nab_feature_set_v2_windows_aligned_to_v1_deduplicated.csv"
nab_v2_df.to_csv(v2_aligned_dedup_path, index=False)

print("Saved deduplicated aligned V2:")
print(v2_aligned_dedup_path)

In [ ]:
# =========================
# Cell A - Feature Set V2 official 80/20 split after duplicate cleanup
# =========================

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

nab_v2_df["start_time"] = pd.to_datetime(nab_v2_df["start_time"], errors="coerce")
nab_v2_df["end_time"] = pd.to_datetime(nab_v2_df["end_time"], errors="coerce")
nab_v2_df["label"] = nab_v2_df["label"].astype(int)

def chronological_80_20_with_internal_validation(
    df,
    train_test_ratio=0.80,
    internal_val_ratio_within_train=0.20
):
    split_frames = []

    for series_id, g in tqdm(df.groupby("series_key", sort=False), desc="Splitting V2 NAB series"):
        g = g.sort_values("start_time").reset_index(drop=True)

        n = len(g)
        if n < 5:
            temp = g.copy()
            temp["split"] = "dropped_too_short"
            split_frames.append(temp)
            continue

        train_region_end = int(np.floor(train_test_ratio * n))
        train_region_end = max(2, min(train_region_end, n - 1))

        train_region = g.iloc[:train_region_end].copy()
        test_part = g.iloc[train_region_end:].copy()

        internal_train_end = int(
            np.floor((1 - internal_val_ratio_within_train) * len(train_region))
        )
        internal_train_end = max(1, min(internal_train_end, len(train_region) - 1))

        train_part = train_region.iloc[:internal_train_end].copy()
        val_part = train_region.iloc[internal_train_end:].copy()

        last_train_end = train_part["end_time"].max()
        val_keep = val_part["start_time"] >= last_train_end
        dropped_val = val_part.loc[~val_keep].copy()
        val_part = val_part.loc[val_keep].copy()

        if len(val_part) > 0:
            last_val_end = val_part["end_time"].max()
        else:
            last_val_end = last_train_end

        test_keep = test_part["start_time"] >= last_val_end
        dropped_test = test_part.loc[~test_keep].copy()
        test_part = test_part.loc[test_keep].copy()

        train_part["split"] = "train"
        val_part["split"] = "internal_validation"
        test_part["split"] = "test"
        dropped_val["split"] = "dropped_purge_validation"
        dropped_test["split"] = "dropped_purge_test"

        split_frames.extend([
            train_part,
            val_part,
            test_part,
            dropped_val,
            dropped_test
        ])

    return pd.concat(split_frames, ignore_index=True)

nab_v2_split_df = chronological_80_20_with_internal_validation(nab_v2_df)

v2_split_audit_path = f"{V2_SPLITS_PATH}/nab_feature_set_v2_80_20_split_audit_deduplicated.csv"
nab_v2_split_df.to_csv(v2_split_audit_path, index=False)

train_df = nab_v2_split_df[nab_v2_split_df["split"] == "train"].copy()
val_df = nab_v2_split_df[nab_v2_split_df["split"] == "internal_validation"].copy()
test_df = nab_v2_split_df[nab_v2_split_df["split"] == "test"].copy()

print("Split counts:")
print(nab_v2_split_df["split"].value_counts())

print("\nTrain:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

print("\nUnique series_key:", nab_v2_df["series_key"].nunique())
print("Saved split audit:")
print(v2_split_audit_path)

In [ ]:
# =========================
# Cell B - Feature Set V2 schema after duplicate cleanup
# =========================

import json
import os

excluded_cols = METADATA_COLS + [
    "label",
    "split",
    "series_key",
    "duration_minutes_actual",
    "valid_sample_count",
    "expected_sample_count"
]

candidate_features = [
    c for c in train_df.columns
    if c not in excluded_cols
    and pd.api.types.is_numeric_dtype(train_df[c])
]

usable_v2_features = []

for c in candidate_features:
    if train_df[c].notna().sum() == 0:
        continue
    if train_df[c].nunique(dropna=True) <= 1:
        continue
    usable_v2_features.append(c)

print("Usable V2 feature count:", len(usable_v2_features))
print(usable_v2_features)

v2_schema = {
    "experiment_id": "PH4_SUP_NAB_003",
    "dataset": "NAB",
    "feature_set": "V2",
    "target_col": "label",
    "source": "Generated from PH2 standardized NAB series with schema timestamp, value, label",
    "comparison_scope": "V2 aligned to the same 51 PH3-compatible NAB series used in Feature Set V1",
    "reason_for_regeneration": (
        "Skewness, kurtosis, entropy, autocorrelation, peak count, rolling variance, "
        "and rate-of-change descriptors require within-window time-series values."
    ),
    "window_size_minutes": 60,
    "step_size_minutes": 30,
    "split_strategy": "official 80/20 chronological train-test split with internal validation carved from training region",
    "official_train_ratio": 0.80,
    "official_test_ratio": 0.20,
    "internal_validation_ratio_within_training_region": 0.20,
    "feature_cols": usable_v2_features,
    "n_features": len(usable_v2_features),
    "excluded_cols": excluded_cols
}

v2_schema_path = f"{V2_CONFIGS_PATH}/nab_feature_set_v2_schema_deduplicated.json"

with open(v2_schema_path, "w") as f:
    json.dump(v2_schema, f, indent=2, default=str)

X_train = train_df[usable_v2_features]
y_train = train_df["label"]

X_val = val_df[usable_v2_features]
y_val = val_df["label"]

X_test = test_df[usable_v2_features]
y_test = test_df["label"]

print("Saved schema:", v2_schema_path)
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nLabel counts:")
print("Train:")
print(y_train.value_counts())
print("\nValidation:")
print(y_val.value_counts())
print("\nTest:")
print(y_test.value_counts())

print("\nSanity check:")
print("Unique series_key:", nab_v2_df["series_key"].nunique())
print("Number of V2 features:", len(usable_v2_features))

In [ ]:
# =========================
# Cell C - Train Feature Set V2 classical supervised models
# =========================

!pip -q install xgboost

import joblib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)
from sklearn.base import clone
from xgboost import XGBClassifier

pos_train = int((y_train == 1).sum())
neg_train = int((y_train == 0).sum())
scale_pos_weight = neg_train / max(pos_train, 1)

classical_models = {
    "RandomForest": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    "LogisticRegression": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "XGBoost": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
}

def find_best_threshold_on_validation(y_true, y_score):
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    f1_scores = (2 * precision * recall) / (precision + recall + 1e-10)

    if len(thresholds) == 0:
        return 0.5, 0.0, 0.0, 0.0

    best_idx = np.argmax(f1_scores[:-1])
    return thresholds[best_idx], f1_scores[best_idx], precision[best_idx], recall[best_idx]

def evaluate_with_threshold(y_true, y_score, threshold):
    y_pred = (y_score >= threshold).astype(int)

    try:
        roc_auc = roc_auc_score(y_true, y_score)
    except Exception:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(y_true, y_score)
    except Exception:
        pr_auc = np.nan

    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc,
        "pr_auc_average_precision": pr_auc
    }, y_pred

v2_classical_metrics = []
v2_classical_predictions = []

for model_name, model in tqdm(classical_models.items(), desc="Training V2 classical models"):
    print(f"\nTraining {model_name}")

    fitted = clone(model)
    fitted.fit(X_train, y_train)

    val_score = fitted.predict_proba(X_val)[:, 1]
    test_score = fitted.predict_proba(X_test)[:, 1]

    best_threshold, val_f1, val_precision, val_recall = find_best_threshold_on_validation(
        y_val, val_score
    )

    test_metrics_default, test_pred_default = evaluate_with_threshold(
        y_test, test_score, 0.5
    )

    test_metrics_tuned, test_pred_tuned = evaluate_with_threshold(
        y_test, test_score, best_threshold
    )

    for threshold_source, threshold, val_f1_selected, metrics, preds in [
        ("default_0.5", 0.5, None, test_metrics_default, test_pred_default),
        ("validation_f1_optimized", best_threshold, val_f1, test_metrics_tuned, test_pred_tuned)
    ]:
        v2_classical_metrics.append({
            "experiment_id": "PH4_SUP_NAB_003",
            "dataset": "NAB",
            "feature_set": "V2",
            "model": model_name,
            "threshold_source": threshold_source,
            "threshold": threshold,
            "validation_f1_at_selected_threshold": val_f1_selected,
            "test_precision": metrics["precision"],
            "test_recall": metrics["recall"],
            "test_f1": metrics["f1"],
            "test_roc_auc": metrics["roc_auc"],
            "test_pr_auc_average_precision": metrics["pr_auc_average_precision"],
            "n_train": len(X_train),
            "n_validation": len(X_val),
            "n_test": len(X_test),
            "n_features": len(usable_v2_features),
            "positive_train": int((y_train == 1).sum()),
            "positive_validation": int((y_val == 1).sum()),
            "positive_test": int((y_test == 1).sum())
        })

    pred_df = test_df[[
        "dataset_name", "series_id", "series_key", "window_id",
        "start_time", "end_time", "label", "source_file"
    ]].copy()

    pred_df["feature_set"] = "V2"
    pred_df["model"] = model_name
    pred_df["y_true"] = y_test.values
    pred_df["y_score"] = test_score
    pred_df["y_pred_default_0_5"] = test_pred_default
    pred_df["y_pred_validation_tuned"] = test_pred_tuned
    pred_df["validation_selected_threshold"] = best_threshold

    v2_classical_predictions.append(pred_df)

    model_path = f"{V2_MODELS_PATH}/{model_name}_PH4_SUP_NAB_003_deduplicated.joblib"
    joblib.dump(fitted, model_path)
    print("Saved:", model_path)

v2_classical_metrics_df = pd.DataFrame(v2_classical_metrics)
v2_classical_predictions_df = pd.concat(v2_classical_predictions, ignore_index=True)

display(v2_classical_metrics_df.sort_values("test_f1", ascending=False))

In [ ]:
# =========================
# Cell D - Train Feature Set V2 MLP
# =========================

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_np = scaler.fit_transform(imputer.fit_transform(X_train))
X_val_np = scaler.transform(imputer.transform(X_val))
X_test_np = scaler.transform(imputer.transform(X_test))

y_train_np = y_train.values.astype(int)
y_val_np = y_val.values.astype(int)
y_test_np = y_test.values.astype(int)

classes = np.array([0, 1])
class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_np
)

class_weight = {
    0: class_weights_values[0],
    1: class_weights_values[1]
}

print("MLP class weights:", class_weight)

def build_mlp(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")]
    )

    return model

mlp = build_mlp(X_train_np.shape[1])

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = mlp.fit(
    X_train_np,
    y_train_np,
    validation_data=(X_val_np, y_val_np),
    epochs=50,
    batch_size=256,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)

val_score_mlp = mlp.predict(X_val_np, verbose=0).ravel()
test_score_mlp = mlp.predict(X_test_np, verbose=0).ravel()

best_threshold_mlp, val_f1_mlp, val_precision_mlp, val_recall_mlp = find_best_threshold_on_validation(
    y_val_np, val_score_mlp
)

test_metrics_mlp_default, test_pred_mlp_default = evaluate_with_threshold(
    y_test_np, test_score_mlp, 0.5
)

test_metrics_mlp_tuned, test_pred_mlp_tuned = evaluate_with_threshold(
    y_test_np, test_score_mlp, best_threshold_mlp
)

v2_mlp_metrics_df = pd.DataFrame([
    {
        "experiment_id": "PH4_SUP_NAB_003",
        "dataset": "NAB",
        "feature_set": "V2",
        "model": "MLP",
        "threshold_source": "default_0.5",
        "threshold": 0.5,
        "validation_f1_at_selected_threshold": None,
        "test_precision": test_metrics_mlp_default["precision"],
        "test_recall": test_metrics_mlp_default["recall"],
        "test_f1": test_metrics_mlp_default["f1"],
        "test_roc_auc": test_metrics_mlp_default["roc_auc"],
        "test_pr_auc_average_precision": test_metrics_mlp_default["pr_auc_average_precision"],
        "n_train": len(X_train),
        "n_validation": len(X_val),
        "n_test": len(X_test),
        "n_features": len(usable_v2_features),
        "positive_train": int((y_train == 1).sum()),
        "positive_validation": int((y_val == 1).sum()),
        "positive_test": int((y_test == 1).sum())
    },
    {
        "experiment_id": "PH4_SUP_NAB_003",
        "dataset": "NAB",
        "feature_set": "V2",
        "model": "MLP",
        "threshold_source": "validation_f1_optimized",
        "threshold": best_threshold_mlp,
        "validation_f1_at_selected_threshold": val_f1_mlp,
        "test_precision": test_metrics_mlp_tuned["precision"],
        "test_recall": test_metrics_mlp_tuned["recall"],
        "test_f1": test_metrics_mlp_tuned["f1"],
        "test_roc_auc": test_metrics_mlp_tuned["roc_auc"],
        "test_pr_auc_average_precision": test_metrics_mlp_tuned["pr_auc_average_precision"],
        "n_train": len(X_train),
        "n_validation": len(X_val),
        "n_test": len(X_test),
        "n_features": len(usable_v2_features),
        "positive_train": int((y_train == 1).sum()),
        "positive_validation": int((y_val == 1).sum()),
        "positive_test": int((y_test == 1).sum())
    }
])

v2_mlp_pred_df = test_df[[
    "dataset_name", "series_id", "series_key", "window_id",
    "start_time", "end_time", "label", "source_file"
]].copy()

v2_mlp_pred_df["feature_set"] = "V2"
v2_mlp_pred_df["model"] = "MLP"
v2_mlp_pred_df["y_true"] = y_test_np
v2_mlp_pred_df["y_score"] = test_score_mlp
v2_mlp_pred_df["y_pred_default_0_5"] = test_pred_mlp_default
v2_mlp_pred_df["y_pred_validation_tuned"] = test_pred_mlp_tuned
v2_mlp_pred_df["validation_selected_threshold"] = best_threshold_mlp

mlp_model_path = f"{V2_MODELS_PATH}/MLP_PH4_SUP_NAB_003_deduplicated.keras"
mlp.save(mlp_model_path)

print("Saved MLP:", mlp_model_path)
display(v2_mlp_metrics_df)

In [ ]:
# =========================
# Cell E - Save V2 outputs + compare V1 vs V2
# =========================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

v2_metrics_df = pd.concat(
    [v2_classical_metrics_df, v2_mlp_metrics_df],
    ignore_index=True
)

v2_predictions_df = pd.concat(
    [v2_classical_predictions_df, v2_mlp_pred_df],
    ignore_index=True
)

v2_metrics_path = f"{V2_METRICS_PATH}/nab_supervised_feature_set_v2_metrics_deduplicated.csv"
v2_predictions_path = f"{V2_PREDS_PATH}/nab_supervised_feature_set_v2_predictions_deduplicated.csv"

v2_metrics_df.to_csv(v2_metrics_path, index=False)
v2_predictions_df.to_csv(v2_predictions_path, index=False)

v1_metrics_path = f"{PH4_NAB_SUP_PATH}/metrics/nab_supervised_4model_metrics_train_val_test.csv"

if os.path.exists(v1_metrics_path):
    v1_metrics_df = pd.read_csv(v1_metrics_path)
    v1_metrics_df["feature_set"] = "V1"
else:
    v1_metrics_df = pd.DataFrame()

v2_metrics_df["feature_set"] = "V2"

v2_tuned = v2_metrics_df[v2_metrics_df["threshold_source"] == "validation_f1_optimized"].copy()

if not v1_metrics_df.empty:
    v1_tuned = v1_metrics_df[v1_metrics_df["threshold_source"] == "validation_f1_optimized"].copy()
    v1_tuned["feature_set"] = "V1"

    comparison_df = pd.merge(
        v1_tuned[[
            "model", "test_precision", "test_recall", "test_f1",
            "test_roc_auc"
        ]].rename(columns={
            "test_precision": "v1_precision",
            "test_recall": "v1_recall",
            "test_f1": "v1_f1",
            "test_roc_auc": "v1_roc_auc"
        }),
        v2_tuned[[
            "model", "test_precision", "test_recall", "test_f1",
            "test_roc_auc", "test_pr_auc_average_precision"
        ]].rename(columns={
            "test_precision": "v2_precision",
            "test_recall": "v2_recall",
            "test_f1": "v2_f1",
            "test_roc_auc": "v2_roc_auc",
            "test_pr_auc_average_precision": "v2_pr_auc_average_precision"
        }),
        on="model",
        how="outer"
    )

    comparison_df["delta_f1_v2_minus_v1"] = comparison_df["v2_f1"] - comparison_df["v1_f1"]
    comparison_df["delta_roc_auc_v2_minus_v1"] = comparison_df["v2_roc_auc"] - comparison_df["v1_roc_auc"]
else:
    comparison_df = pd.DataFrame()

excel_tracker_path = f"{V2_COMBINED_PATH}/ph4_nab_feature_set_v1_vs_v2_tracker_deduplicated.xlsx"

with pd.ExcelWriter(excel_tracker_path, engine="openpyxl") as writer:
    if not v1_metrics_df.empty:
        v1_metrics_df.to_excel(writer, sheet_name="Feature_Set_V1", index=False)
    v2_metrics_df.to_excel(writer, sheet_name="Feature_Set_V2", index=False)
    v2_tuned.to_excel(writer, sheet_name="V2_Tuned_Only", index=False)
    if not comparison_df.empty:
        comparison_df.to_excel(writer, sheet_name="V1_vs_V2", index=False)

print("Saved V2 metrics:", v2_metrics_path)
print("Saved V2 predictions:", v2_predictions_path)
print("Saved Excel tracker:", excel_tracker_path)

print("\nV2 metrics:")
display(v2_metrics_df.sort_values("test_f1", ascending=False).reset_index(drop=True))

print("\nV1 vs V2 comparison:")
display(comparison_df)

# Plot 1: V2 model comparison
plot_df = v2_tuned.copy()
model_order = ["RandomForest", "XGBoost", "MLP", "LogisticRegression"]
plot_df["model"] = pd.Categorical(plot_df["model"], categories=model_order, ordered=True)
plot_df = plot_df.sort_values("model").reset_index(drop=True)

x = np.arange(len(plot_df))
width = 0.18

plt.figure(figsize=(11, 6))
plt.bar(x - 1.5*width, plot_df["test_precision"], width, label="Precision")
plt.bar(x - 0.5*width, plot_df["test_recall"], width, label="Recall")
plt.bar(x + 0.5*width, plot_df["test_f1"], width, label="F1-score")
plt.bar(x + 1.5*width, plot_df["test_roc_auc"], width, label="ROC-AUC")

plt.xticks(x, plot_df["model"], rotation=15)
plt.ylabel("Score")
plt.title("NAB Supervised Validation: Feature Set V2")
plt.legend()
plt.tight_layout()

v2_plot_path = f"{V2_FIGURES_PATH}/nab_supervised_feature_set_v2_model_comparison_deduplicated.png"
plt.savefig(v2_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved V2 model comparison plot:", v2_plot_path)

# Plot 2: V1 vs V2 comparison
if not comparison_df.empty:
    comp = comparison_df.copy()
    comp["model"] = pd.Categorical(comp["model"], categories=model_order, ordered=True)
    comp = comp.sort_values("model").reset_index(drop=True)

    x = np.arange(len(comp))
    width = 0.2

    plt.figure(figsize=(11, 6))
    plt.bar(x - 1.5*width, comp["v1_f1"], width, label="V1 F1")
    plt.bar(x - 0.5*width, comp["v2_f1"], width, label="V2 F1")
    plt.bar(x + 0.5*width, comp["v1_roc_auc"], width, label="V1 ROC-AUC")
    plt.bar(x + 1.5*width, comp["v2_roc_auc"], width, label="V2 ROC-AUC")

    plt.xticks(x, comp["model"], rotation=15)
    plt.ylabel("Score")
    plt.title("NAB Supervised Validation: Feature Set V1 vs V2")
    plt.legend()
    plt.tight_layout()

    v1_v2_plot_path = f"{V2_FIGURES_PATH}/nab_feature_set_v1_vs_v2_comparison_deduplicated.png"
    plt.savefig(v1_v2_plot_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved V1 vs V2 comparison plot:", v1_v2_plot_path)

In [ ]:
# =========================
# Log PH4_SUP_NAB_003 - Feature Set V2
# =========================

from datetime import datetime
import numpy as np

v2_log_path = f"{PH4_LOGS_PATH}/PH4_LOG_003_supervised_nab_feature_set_v2.md"

v2_tuned = v2_metrics_df[v2_metrics_df["threshold_source"] == "validation_f1_optimized"].copy()

best_v2_f1 = v2_tuned.sort_values("test_f1", ascending=False).iloc[0]
best_v2_auc = v2_tuned.sort_values("test_roc_auc", ascending=False).iloc[0]
best_v2_pr = v2_tuned.sort_values("test_pr_auc_average_precision", ascending=False).iloc[0]

avg_delta_f1 = comparison_df["delta_f1_v2_minus_v1"].mean()
avg_delta_auc = comparison_df["delta_roc_auc_v2_minus_v1"].mean()

if avg_delta_f1 > 0:
    f1_statement = "Feature Set V2 produced a small average F1-score improvement across the four supervised models."
elif avg_delta_f1 < 0:
    f1_statement = "Feature Set V2 did not improve average F1-score across the four supervised models."
else:
    f1_statement = "Feature Set V2 produced no average F1-score change across the four supervised models."

if avg_delta_auc > 0:
    auc_statement = "Feature Set V2 improved ROC-AUC across the supervised models, indicating improved ranking ability."
else:
    auc_statement = "Feature Set V2 did not improve ROC-AUC across the supervised models."

log_content = f"""# PH4_LOG_003_supervised_nab_feature_set_v2

## Timestamp
{datetime.now()}

## Experiment ID
PH4_SUP_NAB_003

## Experiment Title
Phase 4 Supervised NAB Validation Using Enhanced Feature Set V2

## Motivation
The supervisor noted that Feature Set V1 was too limited for energy time-series anomaly detection.

Feature Set V1 was based on a smaller set of aggregate statistical descriptors. Feature Set V2 extends the feature representation using richer temporal, statistical, and distributional descriptors.

## Dataset
NAB

## Dataset Role
NAB is used as a labeled benchmark dataset for supervised validation.

## Feature Set V1 Baseline
Feature Set V1 used the existing Phase 3 feature-window outputs.

The previous supervised baseline used:
- 22 usable numeric features
- Random Forest
- XGBoost
- Logistic Regression
- MLP
- Official 80/20 chronological split with internal validation

## Feature Set V2 Construction
The Phase 2 NAB files were inspected and confirmed to use the schema:

`timestamp, value, label`

Therefore, Feature Set V2 was generated from the within-window `value` sequence rather than from the existing Phase 3 aggregate window files.

This was necessary because the following descriptors require access to the actual within-window sequence:
- rolling variance
- skewness
- kurtosis
- rate of change
- peak count
- entropy
- autocorrelation
- temporal difference descriptors

## Feature Set V2 Features
Feature Set V2 includes:
- mean
- standard deviation
- variance
- minimum
- maximum
- range
- median
- sum
- energy sum
- quartiles and IQR
- delta
- skewness
- kurtosis
- coefficient of variation
- rolling variance descriptors
- rolling mean descriptors
- first-difference descriptors
- absolute-difference descriptors
- rate-of-change descriptors
- peak count
- peak ratio
- entropy
- autocorrelation at lags 1, 2, and 5
- slope
- zero and nonzero ratios
- above/below mean ratios

Number of usable V2 features:

{len(usable_v2_features)}

## Fairness Alignment
Feature Set V2 was aligned to the same 51 PH3-compatible NAB series used by Feature Set V1.

A duplicate normalized series key was detected and removed before training to ensure fair comparison.

Final Feature Set V2 split:
- Training rows: {len(train_df)}
- Internal validation rows: {len(val_df)}
- Test rows: {len(test_df)}

Label distribution:

Training:
{y_train.value_counts().to_string()}

Internal validation:
{y_val.value_counts().to_string()}

Test:
{y_test.value_counts().to_string()}

## Split Strategy
The same official Phase 4 split protocol was used:
- 80% chronological training region
- 20% final test region
- internal validation subset carved only from the training region
- boundary purging applied to reduce overlap leakage

## Models
The same four supervised models were trained:
1. Random Forest
2. XGBoost
3. Logistic Regression
4. MLP

## Evaluation Metrics
The following metrics were reported:
- Precision
- Recall
- F1-score
- ROC-AUC
- PR-AUC / Average Precision

## Feature Set V2 Results

{v2_tuned.sort_values("test_f1", ascending=False).to_markdown(index=False)}

## Best V2 Model by F1-score
- Model: {best_v2_f1["model"]}
- F1-score: {best_v2_f1["test_f1"]}
- Precision: {best_v2_f1["test_precision"]}
- Recall: {best_v2_f1["test_recall"]}
- ROC-AUC: {best_v2_f1["test_roc_auc"]}
- PR-AUC: {best_v2_f1["test_pr_auc_average_precision"]}

## Best V2 Model by ROC-AUC
- Model: {best_v2_auc["model"]}
- ROC-AUC: {best_v2_auc["test_roc_auc"]}
- F1-score: {best_v2_auc["test_f1"]}

## Best V2 Model by PR-AUC
- Model: {best_v2_pr["model"]}
- PR-AUC: {best_v2_pr["test_pr_auc_average_precision"]}
- F1-score: {best_v2_pr["test_f1"]}

## Feature Set V1 vs V2 Comparison

{comparison_df.to_markdown(index=False)}

## Interpretation
{f1_statement}

{auc_statement}

The results show that Feature Set V2 improves the representational richness of the supervised pipeline and improves ROC-AUC for all four supervised models. This suggests that the richer temporal and statistical descriptors improve anomaly-ranking ability.

However, F1-score does not improve uniformly. MLP and Logistic Regression improve in F1-score, while Random Forest slightly decreases and XGBoost produces zero thresholded F1-score under the validation-selected threshold.

This indicates that richer features alone do not fully solve the supervised detection problem. The main limitations remain:
- extreme class imbalance
- sparse anomaly labels
- weak window-level supervision
- threshold sensitivity under rare positive labels

## Methodological Conclusion
Feature Set V2 is accepted as the revised supervised feature representation for Phase 4.

The previous Feature Set V1 results are retained as a baseline. Feature Set V2 should be reported as the supervisor-requested enhanced feature experiment.

The results should be presented carefully: V2 improves feature richness and ranking behavior, but supervised classification remains limited by label sparsity.

## Saved Outputs
Metrics CSV:

`{v2_metrics_path}`

Predictions CSV:

`{v2_predictions_path}`

Feature schema JSON:

`{v2_schema_path}`

Split audit CSV:

`{v2_split_audit_path}`

Excel tracker:

`{excel_tracker_path}`

Plots:
- `{v2_plot_path}`
- `{v1_v2_plot_path}`

## Status
PH4_SUP_NAB_003 completed and logged.
"""

with open(v2_log_path, "w", encoding="utf-8") as f:
    f.write(log_content)

print("Feature Set V2 log saved:")
print(v2_log_path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import sys
from pathlib import Path
from datetime import datetime

BASE = "/content/drive/MyDrive/Master/Thesis/Code"
DATASETS = os.path.join(BASE, "Datasets")
NAB_ROOT = os.path.join(DATASETS, "nab_official")

SRC = os.path.join(BASE, "src")
PROCESSED = os.path.join(BASE, "processed_data")
RESULTS = os.path.join(BASE, "results")
EXPERIMENTS = os.path.join(BASE, "experiments")

EXP_NAME = "EXP_NAB_60min_stride15"
EXP_PROCESSED = os.path.join(PROCESSED, "PH4", EXP_NAME)
EXP_RESULTS = os.path.join(RESULTS, EXP_NAME)
EXP_LOGS = os.path.join(EXPERIMENTS, EXP_NAME)

for p in [SRC, PROCESSED, RESULTS, EXPERIMENTS, EXP_PROCESSED, EXP_RESULTS, EXP_LOGS]:
    Path(p).mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("NAB_ROOT:", NAB_ROOT)
print("NAB exists:", os.path.exists(NAB_ROOT))
print("Data exists:", os.path.exists(os.path.join(NAB_ROOT, "data")))
print("Labels exists:", os.path.exists(os.path.join(NAB_ROOT, "labels")))

In [ ]:
import json
import subprocess

data_dir = os.path.join(NAB_ROOT, "data")
labels_dir = os.path.join(NAB_ROOT, "labels")
combined_labels_path = os.path.join(labels_dir, "combined_labels.json")
combined_windows_path = os.path.join(labels_dir, "combined_windows.json")

print("combined_labels exists:", os.path.exists(combined_labels_path))
print("combined_windows exists:", os.path.exists(combined_windows_path))

# Generate combined labels/windows if missing
if not os.path.exists(combined_labels_path) or not os.path.exists(combined_windows_path):
    print("Combined label/window files missing. Trying to regenerate using NAB script...")

    cmd = [
        "python",
        "scripts/combine_labels.py",
        "--dataDir", "data",
        "--labelDir", "labels/raw",
        "--combinedLabelsPath", "labels/combined_labels.json",
        "--combinedWindowsPath", "labels/combined_windows.json",
        "--skipConfirmation",
    ]

    result = subprocess.run(
        cmd,
        cwd=NAB_ROOT,
        capture_output=True,
        text=True
    )

    print("Return code:", result.returncode)
    print("STDOUT:", result.stdout[:1000])
    print("STDERR:", result.stderr[:1000])

print("Final combined_labels exists:", os.path.exists(combined_labels_path))
print("Final combined_windows exists:", os.path.exists(combined_windows_path))

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, roc_auc_score, average_precision_score
)
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

np.random.seed(42)

WINDOW_DURATION = pd.Timedelta(minutes=60)
STRIDE_DURATION = pd.Timedelta(minutes=15)

print("Window:", WINDOW_DURATION)
print("Stride:", STRIDE_DURATION)

In [ ]:
with open(combined_labels_path, "r") as f:
    combined_labels = json.load(f)

if os.path.exists(combined_windows_path):
    with open(combined_windows_path, "r") as f:
        combined_windows = json.load(f)
else:
    combined_windows = {}

print("Number of series in combined_labels:", len(combined_labels))
print("Number of series in combined_windows:", len(combined_windows))

def parse_ts(x):
    return pd.to_datetime(x, errors="coerce", utc=True)

def get_anomaly_intervals(series_key):
    """
    Prefer official NAB anomaly windows.
    Fallback to timestamp labels as zero-width intervals.
    """
    intervals = []

    if series_key in combined_windows:
        for pair in combined_windows[series_key]:
            if isinstance(pair, list) and len(pair) == 2:
                start = parse_ts(pair[0])
                end = parse_ts(pair[1])
                if pd.notna(start) and pd.notna(end):
                    intervals.append((start, end))

    elif series_key in combined_labels:
        for ts in combined_labels[series_key]:
            t = parse_ts(ts)
            if pd.notna(t):
                intervals.append((t, t))

    return intervals

def window_overlaps_anomaly(window_start, window_end, intervals):
    for a_start, a_end in intervals:
        if window_start <= a_end and window_end >= a_start:
            return 1
    return 0

In [ ]:
def extract_window_features(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return None

    diffs = np.diff(values)

    # robust safety
    if len(diffs) == 0:
        diffs = np.array([0.0])

    s = pd.Series(values)

    features = {
        "mean": np.mean(values),
        "std": np.std(values),
        "min": np.min(values),
        "max": np.max(values),
        "median": np.median(values),
        "q25": np.percentile(values, 25),
        "q75": np.percentile(values, 75),
        "iqr": np.percentile(values, 75) - np.percentile(values, 25),
        "range": np.max(values) - np.min(values),
        "sum": np.sum(values),
        "energy": np.sum(values ** 2),
        "rms": np.sqrt(np.mean(values ** 2)),
        "var": np.var(values),
        "mad": np.mean(np.abs(values - np.mean(values))),
        "skew": s.skew() if len(values) > 2 else 0.0,
        "kurtosis": s.kurtosis() if len(values) > 3 else 0.0,
        "mean_abs_diff": np.mean(np.abs(diffs)),
        "max_abs_diff": np.max(np.abs(diffs)),
        "std_diff": np.std(diffs),
        "first_last_diff": values[-1] - values[0],
        "slope": np.polyfit(np.arange(len(values)), values, 1)[0] if len(values) > 1 else 0.0,
        "peak_count": int(np.sum(values > (np.mean(values) + 2 * np.std(values)))) if np.std(values) > 0 else 0,
    }

    return features

In [ ]:
def list_nab_series():
    series = []
    for root, _, files in os.walk(data_dir):
        for f in files:
            if f.lower().endswith(".csv"):
                full_path = os.path.join(root, f)
                rel_path = os.path.relpath(full_path, data_dir).replace("\\", "/")
                series.append(rel_path)
    return sorted(series)

def load_nab_series(series_key):
    path = os.path.join(data_dir, series_key)
    df = pd.read_csv(path)

    df = df.rename(columns={df.columns[0]: "timestamp", df.columns[1]: "value"})
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    df = (
        df.dropna(subset=["timestamp", "value"])
          .sort_values("timestamp")
          .drop_duplicates(subset=["timestamp"])
          .reset_index(drop=True)
    )

    return df

def create_windows_for_series(series_key):
    df = load_nab_series(series_key)
    intervals = get_anomaly_intervals(series_key)

    if df.empty:
        return []

    t_min = df["timestamp"].min()
    t_max = df["timestamp"].max()

    rows = []
    current_start = t_min

    while current_start + WINDOW_DURATION <= t_max:
        current_end = current_start + WINDOW_DURATION

        mask = (df["timestamp"] >= current_start) & (df["timestamp"] < current_end)
        window_df = df.loc[mask]

        if len(window_df) > 0:
            features = extract_window_features(window_df["value"].values)

            if features is not None:
                label = window_overlaps_anomaly(current_start, current_end, intervals)

                row = {
                    "series": series_key,
                    "domain": series_key.split("/")[0],
                    "window_start": current_start,
                    "window_end": current_end,
                    "label": label,
                    "n_points": len(window_df),
                }
                row.update(features)
                rows.append(row)

        current_start += STRIDE_DURATION

    return rows

series_list = list_nab_series()
print("Total NAB series:", len(series_list))
print("First 5:", series_list[:5])

all_rows = []

for i, s in enumerate(series_list, 1):
    try:
        rows = create_windows_for_series(s)
        all_rows.extend(rows)
    except Exception as e:
        print("FAILED:", s, "ERROR:", e)

    if i % 10 == 0 or i == len(series_list):
        print(f"Processed {i}/{len(series_list)} series. Windows so far: {len(all_rows)}")

windows_df = pd.DataFrame(all_rows)

print("\nFinal window dataset shape:", windows_df.shape)
print("\nLabel distribution:")
print(windows_df["label"].value_counts())
print("\nAnomaly ratio:")
print(windows_df["label"].mean())

windows_path = os.path.join(EXP_PROCESSED, "nab_windows_60min_stride15.csv")
windows_df.to_csv(windows_path, index=False)

print("\nSaved windows to:")
print(windows_path)

In [ ]:
def chronological_split_by_series(df, train_ratio=0.60, val_ratio=0.20):
    train_parts = []
    val_parts = []
    test_parts = []

    for series_key, g in df.groupby("series"):
        g = g.sort_values("window_start").reset_index(drop=True)

        n = len(g)
        if n < 5:
            continue

        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))

        train_parts.append(g.iloc[:train_end])
        val_parts.append(g.iloc[train_end:val_end])
        test_parts.append(g.iloc[val_end:])

    train_df = pd.concat(train_parts, ignore_index=True)
    val_df = pd.concat(val_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)

    return train_df, val_df, test_df

train_df, val_df, test_df = chronological_split_by_series(windows_df)

print("Train:", train_df.shape, train_df["label"].value_counts().to_dict(), "ratio:", train_df["label"].mean())
print("Val:", val_df.shape, val_df["label"].value_counts().to_dict(), "ratio:", val_df["label"].mean())
print("Test:", test_df.shape, test_df["label"].value_counts().to_dict(), "ratio:", test_df["label"].mean())

meta_cols = ["series", "domain", "window_start", "window_end", "label"]
feature_cols = [c for c in windows_df.columns if c not in meta_cols]

# keep only numeric feature columns
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(windows_df[c]) and c != "label"]

print("Feature count:", len(feature_cols))
print(feature_cols)

X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
X_val = val_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = test_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

y_train = train_df["label"].values
y_val = val_df["label"].values
y_test = test_df["label"].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaled shapes:")
print(X_train_scaled.shape, X_val_scaled.shape, X_test_scaled.shape)

In [ ]:
def compute_metrics(y_true, y_pred, scores=None):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    roc_auc = np.nan
    pr_auc = np.nan

    if scores is not None and len(np.unique(y_true)) > 1:
        try:
            roc_auc = roc_auc_score(y_true, scores)
        except Exception:
            pass
        try:
            pr_auc = average_precision_score(y_true, scores)
        except Exception:
            pass

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
        "fpr": fpr,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    }

def select_threshold_by_val_f1(y_val, val_scores, percentiles=None):
    if percentiles is None:
        percentiles = [80, 85, 90, 92, 94, 95, 96, 97, 98, 99, 99.2, 99.5, 99.7, 99.9]

    best = None

    for p in percentiles:
        threshold = np.percentile(val_scores, p)
        y_pred = (val_scores >= threshold).astype(int)
        metrics = compute_metrics(y_val, y_pred, val_scores)

        candidate = {
            "percentile": p,
            "threshold": threshold,
            **metrics
        }

        if best is None or candidate["f1"] > best["f1"]:
            best = candidate

    return best

def evaluate_model_scores(model_name, val_scores, test_scores):
    best = select_threshold_by_val_f1(y_val, val_scores)

    test_pred = (test_scores >= best["threshold"]).astype(int)
    test_metrics = compute_metrics(y_test, test_pred, test_scores)

    result = {
        "model": model_name,
        "selected_percentile": best["percentile"],
        "selected_threshold": best["threshold"],
        "val_precision": best["precision"],
        "val_recall": best["recall"],
        "val_f1": best["f1"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_f1": test_metrics["f1"],
        "test_accuracy": test_metrics["accuracy"],
        "test_fpr": test_metrics["fpr"],
        "test_roc_auc": test_metrics["roc_auc"],
        "test_pr_auc": test_metrics["pr_auc"],
        "tn": test_metrics["tn"],
        "fp": test_metrics["fp"],
        "fn": test_metrics["fn"],
        "tp": test_metrics["tp"],
    }

    return result, test_pred

In [ ]:
iso = IsolationForest(
    n_estimators=500,
    max_samples="auto",
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

print("Training Isolation Forest...")
iso.fit(X_train_scaled)

# Higher score = more anomalous
val_scores_iso = -iso.decision_function(X_val_scaled)
test_scores_iso = -iso.decision_function(X_test_scaled)

iso_result, iso_pred = evaluate_model_scores("Isolation Forest", val_scores_iso, test_scores_iso)

print(iso_result)

In [ ]:
lof = LocalOutlierFactor(
    n_neighbors=35,
    novelty=True,
    contamination="auto"
)

print("Training LOF...")
lof.fit(X_train_scaled)

# Higher score = more anomalous
val_scores_lof = -lof.decision_function(X_val_scaled)
test_scores_lof = -lof.decision_function(X_test_scaled)

lof_result, lof_pred = evaluate_model_scores("LOF", val_scores_lof, test_scores_lof)

print(lof_result)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(42)

input_dim = X_train_scaled.shape[1]

ae = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(input_dim, activation="linear")
])

ae.compile(optimizer="adam", loss="mse")

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

print("Training Autoencoder...")
history = ae.fit(
    X_train_scaled,
    X_train_scaled,
    validation_data=(X_val_scaled, X_val_scaled),
    epochs=100,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)

val_recon = ae.predict(X_val_scaled, verbose=0)
test_recon = ae.predict(X_test_scaled, verbose=0)

val_scores_ae = np.mean((X_val_scaled - val_recon) ** 2, axis=1)
test_scores_ae = np.mean((X_test_scaled - test_recon) ** 2, axis=1)

ae_result, ae_pred = evaluate_model_scores("Autoencoder", val_scores_ae, test_scores_ae)

print(ae_result)

In [ ]:
metrics_df = pd.DataFrame([iso_result, lof_result, ae_result])

metrics_path = os.path.join(EXP_RESULTS, "metrics_nab_60min_stride15.csv")
metrics_df.to_csv(metrics_path, index=False)

print("Saved metrics:")
print(metrics_path)
display(metrics_df)

predictions_df = test_df[["series", "domain", "window_start", "window_end", "label"]].copy()
predictions_df["iso_score"] = test_scores_iso
predictions_df["iso_pred"] = iso_pred
predictions_df["lof_score"] = test_scores_lof
predictions_df["lof_pred"] = lof_pred
predictions_df["ae_score"] = test_scores_ae
predictions_df["ae_pred"] = ae_pred

predictions_path = os.path.join(EXP_RESULTS, "test_predictions_nab_60min_stride15.csv")
predictions_df.to_csv(predictions_path, index=False)

print("Saved predictions:")
print(predictions_path)

In [ ]:
now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

log_text = f"""
# {EXP_NAME}

## Objective

Redo the NAB experiment using the supervisor-approved windowing configuration:

- Window size: 60 minutes
- Stride: 15 minutes
- Overlap: 75%

The goal is to increase overlap, improve anomaly coverage, and evaluate whether anomaly detection performance improves despite the extreme imbalance in NAB.

## Dataset

Dataset: Numenta Anomaly Benchmark (NAB)

Source folder:
`{NAB_ROOT}`

Data folder:
`{data_dir}`

Labels:
`combined_labels.json`
`combined_windows.json`

## Labeling rule

A window is labeled anomalous if it overlaps an official NAB anomaly window from `combined_windows.json`.

If official anomaly windows are unavailable for a series, the code falls back to exact anomaly timestamps from `combined_labels.json`.

## Window construction

- Window duration: {WINDOW_DURATION}
- Stride duration: {STRIDE_DURATION}
- Feature count: {len(feature_cols)}
- Features: {feature_cols}

## Dataset after windowing

Total windows: {len(windows_df)}
Normal windows: {int((windows_df["label"] == 0).sum())}
Anomaly windows: {int((windows_df["label"] == 1).sum())}
Anomaly ratio: {windows_df["label"].mean():.8f}

## Split protocol

Chronological split by series:

- Train: first 60%
- Validation: next 20%
- Test: final 20%

This avoids random leakage between highly overlapping windows.

## Models

1. Isolation Forest
2. Local Outlier Factor
3. Autoencoder

## Threshold selection

Thresholds were selected on the validation set using F1-score over a percentile grid of anomaly scores.

The final metrics were computed on the test set only.

## Results

{metrics_df.to_markdown(index=False)}

## Output files

Window dataset:
`{windows_path}`

Metrics:
`{metrics_path}`

Predictions:
`{predictions_path}`

## Interpretation placeholder

Interpretation should be added after reviewing the results. Focus on:

- Whether the 15-minute stride increased anomaly-window coverage
- Whether recall improved
- Whether false positive rate became acceptable
- Which model gives the best F1/recall trade-off
- Whether this configuration should replace the previous NAB baseline

## Status

Completed.
Generated on: {now}
"""

log_path = os.path.join(EXP_LOGS, "EXP_NAB_60min_stride15_log.md")

with open(log_path, "w", encoding="utf-8") as f:
    f.write(log_text)

print("Experiment log saved:")
print(log_path)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np
import os
from datetime import datetime

def evaluate_supervised_classifier(model_name, val_scores, test_scores):
    """
    val_scores/test_scores must be anomaly probabilities or anomaly confidence scores.
    Higher score = more anomalous.
    Threshold is selected on validation set using F1.
    """
    best = select_threshold_by_val_f1(y_val, val_scores)

    test_pred = (test_scores >= best["threshold"]).astype(int)
    test_metrics = compute_metrics(y_test, test_pred, test_scores)

    result = {
        "model": model_name,
        "selected_percentile": best["percentile"],
        "selected_threshold": best["threshold"],
        "val_precision": best["precision"],
        "val_recall": best["recall"],
        "val_f1": best["f1"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_f1": test_metrics["f1"],
        "test_accuracy": test_metrics["accuracy"],
        "test_fpr": test_metrics["fpr"],
        "test_roc_auc": test_metrics["roc_auc"],
        "test_pr_auc": test_metrics["pr_auc"],
        "tn": test_metrics["tn"],
        "fp": test_metrics["fp"],
        "fn": test_metrics["fn"],
        "tp": test_metrics["tp"],
    }

    return result, test_pred

def print_result_summary(name, result):
    print(f"\n=== {name} ===")
    print(f"Test Precision: {result['test_precision']:.4f}")
    print(f"Test Recall:    {result['test_recall']:.4f}")
    print(f"Test F1:        {result['test_f1']:.4f}")
    print(f"Test FPR:       {result['test_fpr']:.4f}")
    print(f"Test ROC-AUC:   {result['test_roc_auc']:.4f}")
    print(f"Test PR-AUC:    {result['test_pr_auc']:.4f}")
    print(f"TP={result['tp']} | FP={result['fp']} | FN={result['fn']} | TN={result['tn']}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest...")
rf_model.fit(X_train_scaled, y_train)

val_scores_rf = rf_model.predict_proba(X_val_scaled)[:, 1]
test_scores_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

rf_result, rf_pred = evaluate_supervised_classifier(
    "Random Forest",
    val_scores_rf,
    test_scores_rf
)

print_result_summary("Random Forest", rf_result)

In [ ]:
!pip install -q xgboost

from xgboost import XGBClassifier

neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
scale_pos_weight = neg_count / max(pos_count, 1)

print("Negative train samples:", neg_count)
print("Positive train samples:", pos_count)
print("scale_pos_weight:", scale_pos_weight)

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost...")
xgb_model.fit(X_train_scaled, y_train)

val_scores_xgb = xgb_model.predict_proba(X_val_scaled)[:, 1]
test_scores_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]

xgb_result, xgb_pred = evaluate_supervised_classifier(
    "XGBoost",
    val_scores_xgb,
    test_scores_xgb
)

print_result_summary("XGBoost", xgb_result)

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg_model = LogisticRegression(
    class_weight="balanced",
    solver="liblinear",
    max_iter=5000,
    random_state=42
)

print("Training Logistic Regression...")
logreg_model.fit(X_train_scaled, y_train)

val_scores_logreg = logreg_model.predict_proba(X_val_scaled)[:, 1]
test_scores_logreg = logreg_model.predict_proba(X_test_scaled)[:, 1]

logreg_result, logreg_pred = evaluate_supervised_classifier(
    "Logistic Regression",
    val_scores_logreg,
    test_scores_logreg
)

print_result_summary("Logistic Regression", logreg_result)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(42)

classes = np.array([0, 1])
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight_dict = {
    0: class_weights_array[0],
    1: class_weights_array[1]
}

print("Class weights:", class_weight_dict)

input_dim = X_train_scaled.shape[1]

mlp_model = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.25),

    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.25),

    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

mlp_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

print("Training MLP...")
history_mlp = mlp_model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=512,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

val_scores_mlp = mlp_model.predict(X_val_scaled, verbose=0).ravel()
test_scores_mlp = mlp_model.predict(X_test_scaled, verbose=0).ravel()

mlp_result, mlp_pred = evaluate_supervised_classifier(
    "MLP",
    val_scores_mlp,
    test_scores_mlp
)

print_result_summary("MLP", mlp_result)

In [ ]:
old_metrics_path = os.path.join(EXP_RESULTS, "metrics_nab_60min_stride15.csv")

old_metrics_df = pd.read_csv(old_metrics_path)

supervised_metrics_df = pd.DataFrame([
    rf_result,
    xgb_result,
    logreg_result,
    mlp_result
])

all_metrics_df = pd.concat(
    [old_metrics_df, supervised_metrics_df],
    ignore_index=True
)

all_metrics_path = os.path.join(
    EXP_RESULTS,
    "metrics_nab_60min_stride15_all_models.csv"
)

all_metrics_df.to_csv(all_metrics_path, index=False)

print("Saved all model metrics to:")
print(all_metrics_path)

display(
    all_metrics_df.sort_values(
        by=["test_f1", "test_recall", "test_pr_auc"],
        ascending=False
    )
)

In [ ]:
supervised_predictions_df = test_df[["series", "domain", "window_start", "window_end", "label"]].copy()

supervised_predictions_df["rf_score"] = test_scores_rf
supervised_predictions_df["rf_pred"] = rf_pred

supervised_predictions_df["xgb_score"] = test_scores_xgb
supervised_predictions_df["xgb_pred"] = xgb_pred

supervised_predictions_df["logreg_score"] = test_scores_logreg
supervised_predictions_df["logreg_pred"] = logreg_pred

supervised_predictions_df["mlp_score"] = test_scores_mlp
supervised_predictions_df["mlp_pred"] = mlp_pred

supervised_predictions_path = os.path.join(
    EXP_RESULTS,
    "test_predictions_nab_60min_stride15_supervised_models.csv"
)

supervised_predictions_df.to_csv(supervised_predictions_path, index=False)

print("Saved supervised predictions to:")
print(supervised_predictions_path)

In [ ]:
now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

supervised_log_text = f"""
# EXP_NAB_60min_stride15_SUPERVISED_BASELINES

## Objective

Run supervisor-requested supervised benchmark models on the NAB 60-minute window / 15-minute stride experiment.

## Dataset

Dataset: NAB

Window configuration:
- Window size: 60 minutes
- Stride: 15 minutes
- Overlap: 75%

The same train/validation/test split from EXP_NAB_60min_stride15 was reused.

## Models

1. Random Forest
2. XGBoost
3. Logistic Regression
4. MLP

## Important methodological note

These models are used as supervised benchmark baselines because NAB provides anomaly labels.

They are not treated as the final thesis methodology for unlabeled smart-home datasets. The final thesis remains focused on unsupervised anomaly detection and XAI.

## Imbalance handling

- Random Forest: class_weight = balanced_subsample
- XGBoost: scale_pos_weight = {scale_pos_weight}
- Logistic Regression: class_weight = balanced
- MLP: Keras class_weight = {class_weight_dict}

## Thresholding

Each model produced anomaly probabilities or anomaly scores.

The final classification threshold was selected on the validation set using F1-score over a percentile grid.

Final reported metrics were computed on the test set only.

## Results

{supervised_metrics_df.to_markdown(index=False)}

## Combined all-model results

{all_metrics_df.sort_values(by=["test_f1", "test_recall", "test_pr_auc"], ascending=False).to_markdown(index=False)}

## Output files

Supervised metrics:
`{all_metrics_path}`

Supervised predictions:
`{supervised_predictions_path}`

## Status

Completed.

Generated on:
{now}
"""

supervised_log_path = os.path.join(
    EXP_LOGS,
    "EXP_NAB_60min_stride15_supervised_baselines_log.md"
)

with open(supervised_log_path, "w", encoding="utf-8") as f:
    f.write(supervised_log_text)

print("Saved supervised experiment log:")
print(supervised_log_path)

In [ ]:
# ============================================================
# EXP_NAB_60min_stride30
# Fair comparison against 60min / 15min experiment
# Same features, same labels, same split logic, same models
# ============================================================

EXP_NAME_6030 = "EXP_NAB_60min_stride30"

EXP_PROCESSED_6030 = os.path.join(PROCESSED, "PH4", EXP_NAME_6030)
EXP_RESULTS_6030 = os.path.join(RESULTS, EXP_NAME_6030)
EXP_LOGS_6030 = os.path.join(EXPERIMENTS, EXP_NAME_6030)

for p in [EXP_PROCESSED_6030, EXP_RESULTS_6030, EXP_LOGS_6030]:
    os.makedirs(p, exist_ok=True)

WINDOW_DURATION_6030 = pd.Timedelta(minutes=60)
STRIDE_DURATION_6030 = pd.Timedelta(minutes=30)

def create_windows_for_series_config(series_key, window_duration, stride_duration):
    df = load_nab_series(series_key)
    intervals = get_anomaly_intervals(series_key)

    if df.empty:
        return []

    t_min = df["timestamp"].min()
    t_max = df["timestamp"].max()

    rows = []
    current_start = t_min

    while current_start + window_duration <= t_max:
        current_end = current_start + window_duration

        mask = (df["timestamp"] >= current_start) & (df["timestamp"] < current_end)
        window_df = df.loc[mask]

        if len(window_df) > 0:
            features = extract_window_features(window_df["value"].values)

            if features is not None:
                label = window_overlaps_anomaly(current_start, current_end, intervals)

                row = {
                    "series": series_key,
                    "domain": series_key.split("/")[0],
                    "window_start": current_start,
                    "window_end": current_end,
                    "label": label,
                    "n_points": len(window_df),
                }

                row.update(features)
                rows.append(row)

        current_start += stride_duration

    return rows

all_rows_6030 = []

for i, s in enumerate(series_list, 1):
    try:
        rows = create_windows_for_series_config(
            s,
            WINDOW_DURATION_6030,
            STRIDE_DURATION_6030
        )
        all_rows_6030.extend(rows)

    except Exception as e:
        print("FAILED:", s, "ERROR:", e)

    if i % 10 == 0 or i == len(series_list):
        print(f"Processed {i}/{len(series_list)} series. Windows so far: {len(all_rows_6030)}")

windows_df_6030 = pd.DataFrame(all_rows_6030)

print("\n60/30 window dataset shape:", windows_df_6030.shape)
print("\nLabel distribution:")
print(windows_df_6030["label"].value_counts())
print("\nAnomaly ratio:")
print(windows_df_6030["label"].mean())

windows_path_6030 = os.path.join(EXP_PROCESSED_6030, "nab_windows_60min_stride30.csv")
windows_df_6030.to_csv(windows_path_6030, index=False)

print("\nSaved 60/30 windows to:")
print(windows_path_6030)

In [ ]:
train_df_6030, val_df_6030, test_df_6030 = chronological_split_by_series(windows_df_6030)

print("Train:", train_df_6030.shape, train_df_6030["label"].value_counts().to_dict(), "ratio:", train_df_6030["label"].mean())
print("Val:", val_df_6030.shape, val_df_6030["label"].value_counts().to_dict(), "ratio:", val_df_6030["label"].mean())
print("Test:", test_df_6030.shape, test_df_6030["label"].value_counts().to_dict(), "ratio:", test_df_6030["label"].mean())

meta_cols = ["series", "domain", "window_start", "window_end", "label"]
feature_cols_6030 = [c for c in windows_df_6030.columns if c not in meta_cols]
feature_cols_6030 = [
    c for c in feature_cols_6030
    if pd.api.types.is_numeric_dtype(windows_df_6030[c]) and c != "label"
]

print("Feature count:", len(feature_cols_6030))
print(feature_cols_6030)

X_train_6030 = train_df_6030[feature_cols_6030].replace([np.inf, -np.inf], np.nan).fillna(0)
X_val_6030 = val_df_6030[feature_cols_6030].replace([np.inf, -np.inf], np.nan).fillna(0)
X_test_6030 = test_df_6030[feature_cols_6030].replace([np.inf, -np.inf], np.nan).fillna(0)

y_train_6030 = train_df_6030["label"].values
y_val_6030 = val_df_6030["label"].values
y_test_6030 = test_df_6030["label"].values

scaler_6030 = StandardScaler()
X_train_scaled_6030 = scaler_6030.fit_transform(X_train_6030)
X_val_scaled_6030 = scaler_6030.transform(X_val_6030)
X_test_scaled_6030 = scaler_6030.transform(X_test_6030)

print("Scaled shapes:")
print(X_train_scaled_6030.shape, X_val_scaled_6030.shape, X_test_scaled_6030.shape)

In [ ]:
def compute_metrics_local(y_true, y_pred, scores=None):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    roc_auc = np.nan
    pr_auc = np.nan

    if scores is not None and len(np.unique(y_true)) > 1:
        try:
            roc_auc = roc_auc_score(y_true, scores)
        except Exception:
            pass

        try:
            pr_auc = average_precision_score(y_true, scores)
        except Exception:
            pass

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
        "fpr": fpr,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    }

def select_threshold_by_val_f1_local(y_val_local, val_scores, percentiles=None):
    if percentiles is None:
        percentiles = [80, 85, 90, 92, 94, 95, 96, 97, 98, 99, 99.2, 99.5, 99.7, 99.9]

    best = None

    for p in percentiles:
        threshold = np.percentile(val_scores, p)
        y_pred = (val_scores >= threshold).astype(int)
        metrics = compute_metrics_local(y_val_local, y_pred, val_scores)

        candidate = {
            "percentile": p,
            "threshold": threshold,
            **metrics
        }

        if best is None or candidate["f1"] > best["f1"]:
            best = candidate

    return best

def evaluate_scores_local(model_name, y_val_local, y_test_local, val_scores, test_scores):
    best = select_threshold_by_val_f1_local(y_val_local, val_scores)

    test_pred = (test_scores >= best["threshold"]).astype(int)
    test_metrics = compute_metrics_local(y_test_local, test_pred, test_scores)

    result = {
        "model": model_name,
        "selected_percentile": best["percentile"],
        "selected_threshold": best["threshold"],
        "val_precision": best["precision"],
        "val_recall": best["recall"],
        "val_f1": best["f1"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_f1": test_metrics["f1"],
        "test_accuracy": test_metrics["accuracy"],
        "test_fpr": test_metrics["fpr"],
        "test_roc_auc": test_metrics["roc_auc"],
        "test_pr_auc": test_metrics["pr_auc"],
        "tn": test_metrics["tn"],
        "fp": test_metrics["fp"],
        "fn": test_metrics["fn"],
        "tp": test_metrics["tp"],
    }

    return result, test_pred

In [ ]:
# ============================================================
# Isolation Forest
# ============================================================

iso_6030 = IsolationForest(
    n_estimators=500,
    max_samples="auto",
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

print("Training Isolation Forest 60/30...")
iso_6030.fit(X_train_scaled_6030)

val_scores_iso_6030 = -iso_6030.decision_function(X_val_scaled_6030)
test_scores_iso_6030 = -iso_6030.decision_function(X_test_scaled_6030)

iso_result_6030, iso_pred_6030 = evaluate_scores_local(
    "Isolation Forest",
    y_val_6030,
    y_test_6030,
    val_scores_iso_6030,
    test_scores_iso_6030
)

# ============================================================
# LOF
# ============================================================

lof_6030 = LocalOutlierFactor(
    n_neighbors=35,
    novelty=True,
    contamination="auto"
)

print("Training LOF 60/30...")
lof_6030.fit(X_train_scaled_6030)

val_scores_lof_6030 = -lof_6030.decision_function(X_val_scaled_6030)
test_scores_lof_6030 = -lof_6030.decision_function(X_test_scaled_6030)

lof_result_6030, lof_pred_6030 = evaluate_scores_local(
    "LOF",
    y_val_6030,
    y_test_6030,
    val_scores_lof_6030,
    test_scores_lof_6030
)

# ============================================================
# Autoencoder
# ============================================================

tf.random.set_seed(42)

input_dim_6030 = X_train_scaled_6030.shape[1]

ae_6030 = models.Sequential([
    layers.Input(shape=(input_dim_6030,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(input_dim_6030, activation="linear")
])

ae_6030.compile(optimizer="adam", loss="mse")

early_stop_ae_6030 = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

print("Training Autoencoder 60/30...")
ae_6030.fit(
    X_train_scaled_6030,
    X_train_scaled_6030,
    validation_data=(X_val_scaled_6030, X_val_scaled_6030),
    epochs=100,
    batch_size=256,
    callbacks=[early_stop_ae_6030],
    verbose=1
)

val_recon_6030 = ae_6030.predict(X_val_scaled_6030, verbose=0)
test_recon_6030 = ae_6030.predict(X_test_scaled_6030, verbose=0)

val_scores_ae_6030 = np.mean((X_val_scaled_6030 - val_recon_6030) ** 2, axis=1)
test_scores_ae_6030 = np.mean((X_test_scaled_6030 - test_recon_6030) ** 2, axis=1)

ae_result_6030, ae_pred_6030 = evaluate_scores_local(
    "Autoencoder",
    y_val_6030,
    y_test_6030,
    val_scores_ae_6030,
    test_scores_ae_6030
)

# ============================================================
# Random Forest
# ============================================================

rf_6030 = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest 60/30...")
rf_6030.fit(X_train_scaled_6030, y_train_6030)

val_scores_rf_6030 = rf_6030.predict_proba(X_val_scaled_6030)[:, 1]
test_scores_rf_6030 = rf_6030.predict_proba(X_test_scaled_6030)[:, 1]

rf_result_6030, rf_pred_6030 = evaluate_scores_local(
    "Random Forest",
    y_val_6030,
    y_test_6030,
    val_scores_rf_6030,
    test_scores_rf_6030
)

# ============================================================
# XGBoost
# ============================================================

neg_6030 = int((y_train_6030 == 0).sum())
pos_6030 = int((y_train_6030 == 1).sum())
scale_pos_weight_6030 = neg_6030 / max(pos_6030, 1)

xgb_6030 = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight_6030,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost 60/30...")
xgb_6030.fit(X_train_scaled_6030, y_train_6030)

val_scores_xgb_6030 = xgb_6030.predict_proba(X_val_scaled_6030)[:, 1]
test_scores_xgb_6030 = xgb_6030.predict_proba(X_test_scaled_6030)[:, 1]

xgb_result_6030, xgb_pred_6030 = evaluate_scores_local(
    "XGBoost",
    y_val_6030,
    y_test_6030,
    val_scores_xgb_6030,
    test_scores_xgb_6030
)

# ============================================================
# Logistic Regression
# ============================================================

logreg_6030 = LogisticRegression(
    class_weight="balanced",
    solver="liblinear",
    max_iter=5000,
    random_state=42
)

print("Training Logistic Regression 60/30...")
logreg_6030.fit(X_train_scaled_6030, y_train_6030)

val_scores_logreg_6030 = logreg_6030.predict_proba(X_val_scaled_6030)[:, 1]
test_scores_logreg_6030 = logreg_6030.predict_proba(X_test_scaled_6030)[:, 1]

logreg_result_6030, logreg_pred_6030 = evaluate_scores_local(
    "Logistic Regression",
    y_val_6030,
    y_test_6030,
    val_scores_logreg_6030,
    test_scores_logreg_6030
)

# ============================================================
# MLP
# ============================================================

classes = np.array([0, 1])
class_weights_6030 = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_6030
)

class_weight_dict_6030 = {
    0: class_weights_6030[0],
    1: class_weights_6030[1]
}

tf.random.set_seed(42)

mlp_6030 = models.Sequential([
    layers.Input(shape=(input_dim_6030,)),
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.25),

    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.25),

    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

mlp_6030.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

early_stop_mlp_6030 = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

print("Training MLP 60/30...")
mlp_6030.fit(
    X_train_scaled_6030,
    y_train_6030,
    validation_data=(X_val_scaled_6030, y_val_6030),
    epochs=100,
    batch_size=512,
    class_weight=class_weight_dict_6030,
    callbacks=[early_stop_mlp_6030],
    verbose=1
)

val_scores_mlp_6030 = mlp_6030.predict(X_val_scaled_6030, verbose=0).ravel()
test_scores_mlp_6030 = mlp_6030.predict(X_test_scaled_6030, verbose=0).ravel()

mlp_result_6030, mlp_pred_6030 = evaluate_scores_local(
    "MLP",
    y_val_6030,
    y_test_6030,
    val_scores_mlp_6030,
    test_scores_mlp_6030
)

# ============================================================
# Save 60/30 metrics
# ============================================================

metrics_6030_df = pd.DataFrame([
    iso_result_6030,
    lof_result_6030,
    ae_result_6030,
    rf_result_6030,
    xgb_result_6030,
    logreg_result_6030,
    mlp_result_6030
])

metrics_6030_path = os.path.join(EXP_RESULTS_6030, "metrics_nab_60min_stride30_all_models.csv")
metrics_6030_df.to_csv(metrics_6030_path, index=False)

print("Saved 60/30 metrics to:")
print(metrics_6030_path)

display(metrics_6030_df.sort_values(by=["test_f1", "test_recall", "test_pr_auc"], ascending=False))

In [ ]:
metrics_6015_path = os.path.join(
    RESULTS,
    "EXP_NAB_60min_stride15",
    "metrics_nab_60min_stride15_all_models.csv"
)

metrics_6030_path = os.path.join(
    RESULTS,
    "EXP_NAB_60min_stride30",
    "metrics_nab_60min_stride30_all_models.csv"
)

metrics_6015 = pd.read_csv(metrics_6015_path)
metrics_6030 = pd.read_csv(metrics_6030_path)

metrics_6015["configuration"] = "60min_window_15min_stride"
metrics_6030["configuration"] = "60min_window_30min_stride"

comparison_long = pd.concat([metrics_6030, metrics_6015], ignore_index=True)

comparison_long_path = os.path.join(
    RESULTS,
    "EXP_NAB_60min_stride_comparison_long.csv"
)

comparison_long.to_csv(comparison_long_path, index=False)

comparison_wide = metrics_6015.merge(
    metrics_6030,
    on="model",
    suffixes=("_60_15", "_60_30")
)

key_metrics = [
    "test_precision",
    "test_recall",
    "test_f1",
    "test_fpr",
    "test_roc_auc",
    "test_pr_auc"
]

summary_rows = []

for _, row in comparison_wide.iterrows():
    item = {"model": row["model"]}

    for metric in key_metrics:
        v_15 = row[f"{metric}_60_15"]
        v_30 = row[f"{metric}_60_30"]
        delta = v_15 - v_30

        item[f"{metric}_60_30"] = v_30
        item[f"{metric}_60_15"] = v_15
        item[f"{metric}_delta"] = delta

    summary_rows.append(item)

comparison_summary = pd.DataFrame(summary_rows)

comparison_summary_path = os.path.join(
    RESULTS,
    "EXP_NAB_60min_stride15_vs_stride30_summary.csv"
)

comparison_summary.to_csv(comparison_summary_path, index=False)

print("Saved long comparison to:")
print(comparison_long_path)

print("Saved summary comparison to:")
print(comparison_summary_path)

display(
    comparison_summary.sort_values(
        by=["test_f1_delta", "test_recall_delta"],
        ascending=False
    )
)

In [ ]:
best_6015 = metrics_6015.sort_values(
    by=["test_f1", "test_recall", "test_pr_auc"],
    ascending=False
).iloc[0]

best_6030 = metrics_6030.sort_values(
    by=["test_f1", "test_recall", "test_pr_auc"],
    ascending=False
).iloc[0]

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

comparison_log = f"""
# NAB Window-Stride Comparison
## 60-minute Window: 15-minute Stride vs 30-minute Stride

Generated on: {now}

---

## Objective

Compare whether increasing window overlap improves NAB anomaly detection performance.

The comparison uses:

- Same dataset: NAB
- Same window size: 60 minutes
- Different stride:
  - 30 minutes = 50% overlap
  - 15 minutes = 75% overlap
- Same feature extraction logic
- Same official NAB anomaly-window overlap labeling
- Same chronological split by series
- Same model set
- Same validation-based threshold selection

---

## Models Compared

1. Isolation Forest
2. Local Outlier Factor
3. Autoencoder
4. Random Forest
5. XGBoost
6. Logistic Regression
7. MLP

---

## Best Model: 60/30 Configuration

Model: {best_6030["model"]}

- Test Precision: {best_6030["test_precision"]:.4f}
- Test Recall: {best_6030["test_recall"]:.4f}
- Test F1: {best_6030["test_f1"]:.4f}
- Test FPR: {best_6030["test_fpr"]:.4f}
- Test PR-AUC: {best_6030["test_pr_auc"]:.4f}

---

## Best Model: 60/15 Configuration

Model: {best_6015["model"]}

- Test Precision: {best_6015["test_precision"]:.4f}
- Test Recall: {best_6015["test_recall"]:.4f}
- Test F1: {best_6015["test_f1"]:.4f}
- Test FPR: {best_6015["test_fpr"]:.4f}
- Test PR-AUC: {best_6015["test_pr_auc"]:.4f}

---

## Full Comparison Table

{comparison_summary.sort_values(by=["test_f1_delta", "test_recall_delta"], ascending=False).to_markdown(index=False)}

---

## Interpretation Template

To be finalized after reviewing the table:

- If test F1 improves with 15-minute stride:
  The increased overlap helped capture more anomaly context and improved detection performance.

- If recall improves but FPR also increases:
  The 15-minute stride made the detector more sensitive, but at the cost of more false positives.

- If performance does not improve:
  Increased overlap alone is insufficient to overcome NAB's extreme imbalance. Further improvements should focus on threshold tuning, anomaly-window padding, feature selection, or model-specific optimization.

---

## Output Files

60/15 metrics:
{metrics_6015_path}

60/30 metrics:
{metrics_6030_path}

Long comparison:
{comparison_long_path}

Summary comparison:
{comparison_summary_path}
"""

comparison_log_path = os.path.join(
    EXPERIMENTS,
    "EXP_NAB_60min_stride15_vs_stride30_comparison.md"
)

with open(comparison_log_path, "w", encoding="utf-8") as f:
    f.write(comparison_log)

print("Comparison report saved to:")
print(comparison_log_path)

In [ ]:
# ============================================================
# NAB INTERPRETATION AND STABILITY ANALYSIS
# Supervisor feedback response
# ============================================================

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import sys
from pathlib import Path
from datetime import datetime

BASE = "/content/drive/MyDrive/Master/Thesis/Code"

DATASETS = os.path.join(BASE, "Datasets")
PROCESSED = os.path.join(BASE, "processed_data")
RESULTS = os.path.join(BASE, "results")
EXPERIMENTS = os.path.join(BASE, "experiments")

NAB_ROOT = os.path.join(DATASETS, "nab_official")

EXP_6015 = "EXP_NAB_60min_stride15"
EXP_6030 = "EXP_NAB_60min_stride30"

PATH_6015_WINDOWS = os.path.join(PROCESSED, "PH4", EXP_6015, "nab_windows_60min_stride15.csv")
PATH_6030_WINDOWS = os.path.join(PROCESSED, "PH4", EXP_6030, "nab_windows_60min_stride30.csv")

PATH_6015_METRICS = os.path.join(RESULTS, EXP_6015, "metrics_nab_60min_stride15_all_models.csv")
PATH_6030_METRICS = os.path.join(RESULTS, EXP_6030, "metrics_nab_60min_stride30_all_models.csv")

INTERP_RESULTS = os.path.join(RESULTS, "EXP_NAB_interpretation_stability")
INTERP_LOGS = os.path.join(EXPERIMENTS, "EXP_NAB_interpretation_stability")

for p in [INTERP_RESULTS, INTERP_LOGS]:
    Path(p).mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("60/15 windows exist:", os.path.exists(PATH_6015_WINDOWS))
print("60/30 windows exist:", os.path.exists(PATH_6030_WINDOWS))
print("60/15 metrics exist:", os.path.exists(PATH_6015_METRICS))
print("60/30 metrics exist:", os.path.exists(PATH_6030_METRICS))

In [ ]:
import numpy as np
import pandas as pd

windows_6015 = pd.read_csv(PATH_6015_WINDOWS)
windows_6030 = pd.read_csv(PATH_6030_WINDOWS)

metrics_6015 = pd.read_csv(PATH_6015_METRICS)
metrics_6030 = pd.read_csv(PATH_6030_METRICS)

# Convert timestamps if present
for df in [windows_6015, windows_6030]:
    df["window_start"] = pd.to_datetime(df["window_start"], errors="coerce")
    df["window_end"] = pd.to_datetime(df["window_end"], errors="coerce")

print("60/15 windows:", windows_6015.shape)
print("60/30 windows:", windows_6030.shape)

print("\n60/15 label distribution:")
print(windows_6015["label"].value_counts())

print("\n60/30 label distribution:")
print(windows_6030["label"].value_counts())

print("\n60/15 anomaly ratio:", windows_6015["label"].mean())
print("60/30 anomaly ratio:", windows_6030["label"].mean())

In [ ]:
def representation_summary(df, config_name):
    total = len(df)
    anomaly = int(df["label"].sum())
    normal = total - anomaly
    anomaly_ratio = anomaly / total if total > 0 else 0

    unique_series = df["series"].nunique()
    unique_domains = df["domain"].nunique() if "domain" in df.columns else np.nan

    return {
        "configuration": config_name,
        "total_windows": total,
        "normal_windows": normal,
        "anomaly_windows": anomaly,
        "anomaly_ratio": anomaly_ratio,
        "unique_series": unique_series,
        "unique_domains": unique_domains,
        "windows_per_anomaly": total / anomaly if anomaly > 0 else np.nan,
    }

rep_summary = pd.DataFrame([
    representation_summary(windows_6030, "60min_window_30min_stride"),
    representation_summary(windows_6015, "60min_window_15min_stride"),
])

rep_summary["delta_vs_60_30"] = np.nan

for col in ["total_windows", "normal_windows", "anomaly_windows", "anomaly_ratio", "windows_per_anomaly"]:
    rep_summary.loc[1, f"{col}_delta"] = rep_summary.loc[1, col] - rep_summary.loc[0, col]

rep_path = os.path.join(INTERP_RESULTS, "window_representation_summary.csv")
rep_summary.to_csv(rep_path, index=False)

display(rep_summary)
print("Saved:", rep_path)

In [ ]:
def chronological_split_by_series(df, train_ratio=0.60, val_ratio=0.20):
    train_parts = []
    val_parts = []
    test_parts = []

    for series_key, g in df.groupby("series"):
        g = g.sort_values("window_start").reset_index(drop=True)

        n = len(g)
        if n < 5:
            continue

        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))

        train_parts.append(g.iloc[:train_end])
        val_parts.append(g.iloc[train_end:val_end])
        test_parts.append(g.iloc[val_end:])

    train_df = pd.concat(train_parts, ignore_index=True)
    val_df = pd.concat(val_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)

    return train_df, val_df, test_df

train_6030, val_6030, test_6030 = chronological_split_by_series(windows_6030)
train_6015, val_6015, test_6015 = chronological_split_by_series(windows_6015)

def split_summary(split_name, df, config):
    total = len(df)
    anomaly = int(df["label"].sum())
    normal = total - anomaly
    return {
        "configuration": config,
        "split": split_name,
        "total_windows": total,
        "normal_windows": normal,
        "anomaly_windows": anomaly,
        "anomaly_ratio": anomaly / total if total > 0 else 0,
        "series_count": df["series"].nunique(),
    }

split_summary_df = pd.DataFrame([
    split_summary("train", train_6030, "60/30"),
    split_summary("validation", val_6030, "60/30"),
    split_summary("test", test_6030, "60/30"),

    split_summary("train", train_6015, "60/15"),
    split_summary("validation", val_6015, "60/15"),
    split_summary("test", test_6015, "60/15"),
])

split_path = os.path.join(INTERP_RESULTS, "split_level_representation_summary.csv")
split_summary_df.to_csv(split_path, index=False)

display(split_summary_df)
print("Saved:", split_path)

In [ ]:
meta_cols = ["series", "domain", "window_start", "window_end", "label"]
feature_cols_6015 = [
    c for c in windows_6015.columns
    if c not in meta_cols and pd.api.types.is_numeric_dtype(windows_6015[c])
]

feature_cols_6030 = [
    c for c in windows_6030.columns
    if c not in meta_cols and pd.api.types.is_numeric_dtype(windows_6030[c])
]

common_features = sorted(list(set(feature_cols_6015).intersection(set(feature_cols_6030))))

print("Common feature count:", len(common_features))
print(common_features)

def cohen_d(normal_values, anomaly_values):
    normal_values = np.asarray(normal_values, dtype=float)
    anomaly_values = np.asarray(anomaly_values, dtype=float)

    normal_values = normal_values[np.isfinite(normal_values)]
    anomaly_values = anomaly_values[np.isfinite(anomaly_values)]

    if len(normal_values) < 2 or len(anomaly_values) < 2:
        return np.nan

    mean_diff = np.mean(anomaly_values) - np.mean(normal_values)
    pooled_std = np.sqrt((np.var(normal_values) + np.var(anomaly_values)) / 2)

    if pooled_std == 0:
        return 0

    return mean_diff / pooled_std

def feature_suitability(df, config_name, features):
    rows = []

    for feat in features:
        normal = df.loc[df["label"] == 0, feat].replace([np.inf, -np.inf], np.nan).dropna()
        anomaly = df.loc[df["label"] == 1, feat].replace([np.inf, -np.inf], np.nan).dropna()

        rows.append({
            "configuration": config_name,
            "feature": feat,
            "normal_median": normal.median(),
            "anomaly_median": anomaly.median(),
            "median_difference": anomaly.median() - normal.median(),
            "normal_mean": normal.mean(),
            "anomaly_mean": anomaly.mean(),
            "mean_difference": anomaly.mean() - normal.mean(),
            "cohen_d": cohen_d(normal, anomaly),
            "abs_cohen_d": abs(cohen_d(normal, anomaly)),
        })

    return pd.DataFrame(rows).sort_values("abs_cohen_d", ascending=False)

feat_6030 = feature_suitability(windows_6030, "60/30", common_features)
feat_6015 = feature_suitability(windows_6015, "60/15", common_features)

feature_suitability_df = pd.concat([feat_6030, feat_6015], ignore_index=True)

feature_path = os.path.join(INTERP_RESULTS, "feature_suitability_effect_sizes.csv")
feature_suitability_df.to_csv(feature_path, index=False)

display(feature_suitability_df.head(20))
print("Saved:", feature_path)

In [ ]:
import matplotlib.pyplot as plt

top_features = (
    feature_suitability_df
    .groupby("feature")["abs_cohen_d"]
    .mean()
    .sort_values(ascending=False)
    .head(8)
    .index
    .tolist()
)

print("Top features:", top_features)

for feat in top_features:
    plt.figure(figsize=(6, 4))

    normal_values = windows_6015.loc[windows_6015["label"] == 0, feat].replace([np.inf, -np.inf], np.nan).dropna()
    anomaly_values = windows_6015.loc[windows_6015["label"] == 1, feat].replace([np.inf, -np.inf], np.nan).dropna()

    plt.boxplot(
        [normal_values, anomaly_values],
        labels=["Normal", "Anomaly"],
        showfliers=False
    )

    plt.title(f"Feature separation in 60/15: {feat}")
    plt.ylabel(feat)
    plt.tight_layout()

    fig_path = os.path.join(INTERP_RESULTS, f"feature_boxplot_6015_{feat}.png")
    plt.savefig(fig_path, dpi=200)
    plt.show()

    print("Saved:", fig_path)

In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, roc_auc_score, average_precision_score
)

pred_6015_path = os.path.join(
    RESULTS,
    EXP_6015,
    "test_predictions_nab_60min_stride15.csv"
)

pred_6015 = pd.read_csv(pred_6015_path)

print("Prediction file exists:", os.path.exists(pred_6015_path))
print(pred_6015.columns)
print(pred_6015.shape)

In [ ]:
def metrics_at_percentiles(y_true, scores, model, config, percentiles=None):
    if percentiles is None:
        percentiles = np.arange(50, 99.6, 2.5)

    rows = []

    for p in percentiles:
        threshold = np.percentile(scores, p)
        y_pred = (scores >= threshold).astype(int)

        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

        rows.append({
            "configuration": config,
            "model": model,
            "percentile": p,
            "threshold": threshold,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "fpr": fpr,
            "tp": int(tp),
            "fp": int(fp),
            "fn": int(fn),
            "tn": int(tn),
        })

    return pd.DataFrame(rows)

threshold_rows = []

y_true_6015 = pred_6015["label"].values

score_columns = {
    "Isolation Forest": "iso_score",
    "LOF": "lof_score",
    "Autoencoder": "ae_score",
}

for model, score_col in score_columns.items():
    if score_col in pred_6015.columns:
        threshold_rows.append(
            metrics_at_percentiles(
                y_true_6015,
                pred_6015[score_col].values,
                model,
                "60/15"
            )
        )

threshold_6015_df = pd.concat(threshold_rows, ignore_index=True)

threshold_6015_path = os.path.join(INTERP_RESULTS, "threshold_sensitivity_60_15_unsupervised.csv")
threshold_6015_df.to_csv(threshold_6015_path, index=False)

display(threshold_6015_df.sort_values(["model", "f1"], ascending=[True, False]).groupby("model").head(5))
print("Saved:", threshold_6015_path)

In [ ]:
for model in threshold_6015_df["model"].unique():
    g = threshold_6015_df[threshold_6015_df["model"] == model].sort_values("percentile")

    plt.figure(figsize=(7, 4))
    plt.plot(g["percentile"], g["f1"], marker="o", label="F1")
    plt.plot(g["percentile"], g["recall"], marker="o", label="Recall")
    plt.plot(g["percentile"], g["fpr"], marker="o", label="FPR")

    plt.title(f"Threshold sensitivity — {model} — 60/15")
    plt.xlabel("Score percentile threshold")
    plt.ylabel("Metric value")
    plt.legend()
    plt.tight_layout()

    fig_path = os.path.join(INTERP_RESULTS, f"threshold_sensitivity_60_15_{model.replace(' ', '_')}.png")
    plt.savefig(fig_path, dpi=200)
    plt.show()

    print("Saved:", fig_path)

In [ ]:
best_features_6015 = feat_6015.head(10)[["feature", "cohen_d", "abs_cohen_d"]]
best_features_6030 = feat_6030.head(10)[["feature", "cohen_d", "abs_cohen_d"]]

best_thresholds = (
    threshold_6015_df
    .sort_values(["model", "f1"], ascending=[True, False])
    .groupby("model")
    .head(1)
)

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

report_text = f"""
# NAB Interpretation and Stability Analysis

Generated on: {now}

---

## 1. Purpose

This analysis responds to the supervisor feedback that the 60/30 vs 60/15 experiment should be interpreted as a sampling/window-representation analysis rather than direct model improvement.

The goal is to examine:

1. What is driving performance.
2. Whether the evaluation setup is stable.
3. Whether the features and windowing are suitable under extreme imbalance.

---

## 2. Window Representation Summary

{rep_summary.to_markdown(index=False)}

---

## 3. Split-Level Representation Summary

{split_summary_df.to_markdown(index=False)}

---

## 4. Feature Suitability

Top features by average normal/anomaly separation were computed using effect size.

### Top 60/30 Features

{best_features_6030.to_markdown(index=False)}

### Top 60/15 Features

{best_features_6015.to_markdown(index=False)}

Interpretation placeholder:
- Features with higher absolute effect size are more likely to drive model decisions.
- If only a few spike-related features dominate, the models may be detecting abrupt changes rather than broader anomalous behavior.
- If feature separation is weak, the current feature representation may be insufficient.

---

## 5. Threshold Sensitivity — 60/15

Best threshold point per model:

{best_thresholds.to_markdown(index=False)}

Interpretation placeholder:
- A model should not be selected only from a single F1 value.
- Stability across nearby thresholds should be considered.
- Models with lower FPR and stable F1 are more defensible as baselines.

---

## 6. Revised Interpretation

The change from 60/30 to 60/15 should not be interpreted as direct model improvement.
It is a change in the sampling strategy that changes the window-level representation of the data.

The previous result suggests that LOF is more sensitive to the increased-overlap representation, while Isolation Forest appears more stable across the two tested stride settings.

However, because the metric differences are small and NAB is extremely imbalanced, baseline selection should not be based only on marginal F1 differences.
It should also consider:

- false positive rate,
- threshold sensitivity,
- feature suitability,
- anomaly-window distribution,
- and stability of the evaluation setup.

---

## 7. Current Next Step

Before further tuning, the next step should be to review:

1. whether 60/15 materially changes anomaly-window representation,
2. whether the selected thresholds are stable,
3. whether the current features separate normal and anomaly windows sufficiently.

Only after this interpretation step should model tuning or SHAP/LIME explanation be applied.

"""

report_path = os.path.join(INTERP_LOGS, "NAB_interpretation_and_stability_analysis.md")

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print("Saved interpretation report:")
print(report_path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
from pathlib import Path
from datetime import datetime

BASE = "/content/drive/MyDrive/Master/Thesis/Code"

DATASETS = os.path.join(BASE, "Datasets")
PROCESSED = os.path.join(BASE, "processed_data")
RESULTS = os.path.join(BASE, "results")
EXPERIMENTS = os.path.join(BASE, "experiments")

EXP_6015 = "EXP_NAB_60min_stride15"
EXP_6030 = "EXP_NAB_60min_stride30"

PATH_6015_WINDOWS = os.path.join(PROCESSED, "PH4", EXP_6015, "nab_windows_60min_stride15.csv")
PATH_6030_WINDOWS = os.path.join(PROCESSED, "PH4", EXP_6030, "nab_windows_60min_stride30.csv")

PATH_6015_PRED = os.path.join(RESULTS, EXP_6015, "test_predictions_nab_60min_stride15.csv")
PATH_6030_PRED = os.path.join(RESULTS, EXP_6030, "test_predictions_nab_60min_stride30.csv")

INTERP_RESULTS = os.path.join(RESULTS, "EXP_NAB_threshold_stability")
INTERP_LOGS = os.path.join(EXPERIMENTS, "EXP_NAB_threshold_stability")

for p in [INTERP_RESULTS, INTERP_LOGS]:
    Path(p).mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("60/15 windows:", os.path.exists(PATH_6015_WINDOWS))
print("60/30 windows:", os.path.exists(PATH_6030_WINDOWS))
print("60/15 predictions:", os.path.exists(PATH_6015_PRED))
print("60/30 predictions:", os.path.exists(PATH_6030_PRED))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
windows_6015 = pd.read_csv(PATH_6015_WINDOWS)
windows_6030 = pd.read_csv(PATH_6030_WINDOWS)

for df in [windows_6015, windows_6030]:
    df["window_start"] = pd.to_datetime(df["window_start"], errors="coerce")
    df["window_end"] = pd.to_datetime(df["window_end"], errors="coerce")

print("60/15:", windows_6015.shape)
print("60/30:", windows_6030.shape)

print("\n60/15 label distribution:")
print(windows_6015["label"].value_counts())

print("\n60/30 label distribution:")
print(windows_6030["label"].value_counts())

In [ ]:
def chronological_split_by_series(df, train_ratio=0.60, val_ratio=0.20):
    train_parts = []
    val_parts = []
    test_parts = []

    for series_key, g in df.groupby("series"):
        g = g.sort_values("window_start").reset_index(drop=True)

        n = len(g)
        if n < 5:
            continue

        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))

        train_parts.append(g.iloc[:train_end])
        val_parts.append(g.iloc[train_end:val_end])
        test_parts.append(g.iloc[val_end:])

    train_df = pd.concat(train_parts, ignore_index=True)
    val_df = pd.concat(val_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)

    return train_df, val_df, test_df


train_6030, val_6030, test_6030 = chronological_split_by_series(windows_6030)
train_6015, val_6015, test_6015 = chronological_split_by_series(windows_6015)

print("60/30 train/val/test:", train_6030.shape, val_6030.shape, test_6030.shape)
print("60/15 train/val/test:", train_6015.shape, val_6015.shape, test_6015.shape)

print("\n60/30 test labels:")
print(test_6030["label"].value_counts())

print("\n60/15 test labels:")
print(test_6015["label"].value_counts())

In [ ]:
meta_cols = ["series", "domain", "window_start", "window_end", "label"]

feature_cols_6030 = [
    c for c in windows_6030.columns
    if c not in meta_cols and pd.api.types.is_numeric_dtype(windows_6030[c])
]

feature_cols_6015 = [
    c for c in windows_6015.columns
    if c not in meta_cols and pd.api.types.is_numeric_dtype(windows_6015[c])
]

common_features = sorted(list(set(feature_cols_6030).intersection(set(feature_cols_6015))))

print("Common feature count:", len(common_features))
print(common_features)

def prepare_xy(train_df, val_df, test_df, features):
    X_train = train_df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
    X_val = val_df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = test_df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

    y_train = train_df["label"].values
    y_val = val_df["label"].values
    y_test = test_df["label"].values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test


X_train_6030, X_val_6030, X_test_6030, y_train_6030, y_val_6030, y_test_6030 = prepare_xy(
    train_6030, val_6030, test_6030, common_features
)

X_train_6015, X_val_6015, X_test_6015, y_train_6015, y_val_6015, y_test_6015 = prepare_xy(
    train_6015, val_6015, test_6015, common_features
)

print("60/30 scaled:", X_train_6030.shape, X_val_6030.shape, X_test_6030.shape)
print("60/15 scaled:", X_train_6015.shape, X_val_6015.shape, X_test_6015.shape)

In [ ]:
def train_unsupervised_and_get_scores(X_train, X_val, X_test, config_name):
    results = {}

    # Isolation Forest
    iso = IsolationForest(
        n_estimators=500,
        max_samples="auto",
        contamination="auto",
        random_state=42,
        n_jobs=-1
    )

    print(f"Training Isolation Forest for {config_name}...")
    iso.fit(X_train)

    results["iso_val_score"] = -iso.decision_function(X_val)
    results["iso_test_score"] = -iso.decision_function(X_test)

    # LOF
    lof = LocalOutlierFactor(
        n_neighbors=35,
        novelty=True,
        contamination="auto"
    )

    print(f"Training LOF for {config_name}...")
    lof.fit(X_train)

    results["lof_val_score"] = -lof.decision_function(X_val)
    results["lof_test_score"] = -lof.decision_function(X_test)

    # Autoencoder
    tf.random.set_seed(42)

    input_dim = X_train.shape[1]

    ae = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(input_dim, activation="linear")
    ])

    ae.compile(optimizer="adam", loss="mse")

    early_stop = callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    print(f"Training Autoencoder for {config_name}...")
    ae.fit(
        X_train,
        X_train,
        validation_data=(X_val, X_val),
        epochs=100,
        batch_size=256,
        callbacks=[early_stop],
        verbose=1
    )

    val_recon = ae.predict(X_val, verbose=0)
    test_recon = ae.predict(X_test, verbose=0)

    results["ae_val_score"] = np.mean((X_val - val_recon) ** 2, axis=1)
    results["ae_test_score"] = np.mean((X_test - test_recon) ** 2, axis=1)

    return results


scores_6030 = train_unsupervised_and_get_scores(
    X_train_6030,
    X_val_6030,
    X_test_6030,
    "60/30"
)

pred_6030 = test_6030[["series", "domain", "window_start", "window_end", "label"]].copy()
pred_6030["iso_score"] = scores_6030["iso_test_score"]
pred_6030["lof_score"] = scores_6030["lof_test_score"]
pred_6030["ae_score"] = scores_6030["ae_test_score"]

# temporary binary predictions using 80th percentile just for file completeness;
# threshold analysis below will recompute properly across many percentiles.
for score_col, pred_col in [
    ("iso_score", "iso_pred"),
    ("lof_score", "lof_pred"),
    ("ae_score", "ae_pred")
]:
    threshold = np.percentile(pred_6030[score_col], 80)
    pred_6030[pred_col] = (pred_6030[score_col] >= threshold).astype(int)

PATH_6030_PRED = os.path.join(RESULTS, EXP_6030, "test_predictions_nab_60min_stride30.csv")
pred_6030.to_csv(PATH_6030_PRED, index=False)

print("Saved 60/30 predictions to:")
print(PATH_6030_PRED)
print(pred_6030.shape)
print(pred_6030.head())

In [ ]:
pred_6015 = pd.read_csv(PATH_6015_PRED)
pred_6030 = pd.read_csv(PATH_6030_PRED)

print("60/15 prediction columns:")
print(pred_6015.columns)

print("\n60/30 prediction columns:")
print(pred_6030.columns)

print("\n60/15 shape:", pred_6015.shape)
print("60/30 shape:", pred_6030.shape)

In [ ]:
def threshold_sensitivity(y_true, scores, model_name, config_name, percentiles=None):
    if percentiles is None:
        percentiles = np.arange(50, 99.6, 2.5)

    rows = []

    for p in percentiles:
        threshold = np.percentile(scores, p)
        y_pred = (scores >= threshold).astype(int)

        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        accuracy = accuracy_score(y_true, y_pred)
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

        roc_auc = np.nan
        pr_auc = np.nan

        if len(np.unique(y_true)) > 1:
            try:
                roc_auc = roc_auc_score(y_true, scores)
            except Exception:
                pass

            try:
                pr_auc = average_precision_score(y_true, scores)
            except Exception:
                pass

        rows.append({
            "configuration": config_name,
            "model": model_name,
            "percentile": p,
            "threshold": threshold,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "accuracy": accuracy,
            "fpr": fpr,
            "roc_auc": roc_auc,
            "pr_auc": pr_auc,
            "tp": int(tp),
            "fp": int(fp),
            "fn": int(fn),
            "tn": int(tn)
        })

    return pd.DataFrame(rows)

In [ ]:
score_map = {
    "Isolation Forest": "iso_score",
    "LOF": "lof_score",
    "Autoencoder": "ae_score"
}

threshold_dfs = []

for model_name, score_col in score_map.items():
    if score_col in pred_6030.columns:
        threshold_dfs.append(
            threshold_sensitivity(
                pred_6030["label"].values,
                pred_6030[score_col].values,
                model_name,
                "60/30"
            )
        )

    if score_col in pred_6015.columns:
        threshold_dfs.append(
            threshold_sensitivity(
                pred_6015["label"].values,
                pred_6015[score_col].values,
                model_name,
                "60/15"
            )
        )

threshold_all_df = pd.concat(threshold_dfs, ignore_index=True)

threshold_all_path = os.path.join(
    INTERP_RESULTS,
    "threshold_sensitivity_60_30_vs_60_15_unsupervised.csv"
)

threshold_all_df.to_csv(threshold_all_path, index=False)

print("Saved threshold sensitivity:")
print(threshold_all_path)

display(
    threshold_all_df
    .sort_values(["configuration", "model", "f1"], ascending=[True, True, False])
    .groupby(["configuration", "model"])
    .head(5)
)

In [ ]:
def stability_summary_for_group(g):
    best = g.sort_values("f1", ascending=False).iloc[0]

    # Stable region: thresholds where F1 is within 95% of best F1
    stable = g[g["f1"] >= 0.95 * best["f1"]]

    return pd.Series({
        "best_percentile": best["percentile"],
        "best_f1": best["f1"],
        "precision_at_best_f1": best["precision"],
        "recall_at_best_f1": best["recall"],
        "fpr_at_best_f1": best["fpr"],
        "roc_auc": best["roc_auc"],
        "pr_auc": best["pr_auc"],
        "stable_percentile_min": stable["percentile"].min(),
        "stable_percentile_max": stable["percentile"].max(),
        "stable_percentile_width": stable["percentile"].max() - stable["percentile"].min(),
        "stable_points_count": len(stable),
        "f1_std_across_thresholds": g["f1"].std(),
        "recall_std_across_thresholds": g["recall"].std(),
        "fpr_std_across_thresholds": g["fpr"].std(),
    })


stability_summary = (
    threshold_all_df
    .groupby(["configuration", "model"])
    .apply(stability_summary_for_group)
    .reset_index()
)

stability_summary_path = os.path.join(
    INTERP_RESULTS,
    "threshold_stability_summary_60_30_vs_60_15.csv"
)

stability_summary.to_csv(stability_summary_path, index=False)

print("Saved stability summary:")
print(stability_summary_path)

display(
    stability_summary.sort_values(
        by=["best_f1", "fpr_at_best_f1"],
        ascending=[False, True]
    )
)

In [ ]:
for config_name in threshold_all_df["configuration"].unique():
    for model_name in threshold_all_df["model"].unique():
        g = threshold_all_df[
            (threshold_all_df["configuration"] == config_name) &
            (threshold_all_df["model"] == model_name)
        ].sort_values("percentile")

        if g.empty:
            continue

        plt.figure(figsize=(7, 4))
        plt.plot(g["percentile"], g["f1"], marker="o", label="F1")
        plt.plot(g["percentile"], g["recall"], marker="o", label="Recall")
        plt.plot(g["percentile"], g["fpr"], marker="o", label="FPR")

        plt.title(f"Threshold sensitivity — {model_name} — {config_name}")
        plt.xlabel("Score percentile threshold")
        plt.ylabel("Metric value")
        plt.legend()
        plt.tight_layout()

        safe_model = model_name.replace(" ", "_")
        safe_config = config_name.replace("/", "_")

        fig_path = os.path.join(
            INTERP_RESULTS,
            f"threshold_sensitivity_{safe_config}_{safe_model}.png"
        )

        plt.savefig(fig_path, dpi=200)
        plt.show()

        print("Saved:", fig_path)

In [ ]:
now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

best_table = stability_summary.sort_values(
    by=["best_f1", "fpr_at_best_f1"],
    ascending=[False, True]
)

report_text = f"""
# EXP_NAB_Threshold_Stability_60_30_vs_60_15

Generated on: {now}

---

## 1. Objective

This experiment evaluates whether the NAB model results are stable across anomaly-score thresholds.

The supervisor feedback emphasized that the 60/30 vs 60/15 comparison should not be interpreted only as model improvement.
Therefore, this experiment analyzes threshold sensitivity for the main anomaly-detection models:

1. Isolation Forest
2. LOF
3. Autoencoder

under both windowing representations:

- 60-minute window / 30-minute stride
- 60-minute window / 15-minute stride

---

## 2. Method

For each model and configuration, anomaly scores were evaluated across percentile thresholds from 50% to 97.5%.

At each threshold, the following were computed:

- precision
- recall
- F1-score
- false positive rate
- ROC-AUC
- PR-AUC
- confusion matrix counts

A stability region was defined as the range of thresholds where F1 remains within 95% of the best F1 for that model/configuration.

---

## 3. Threshold Stability Summary

{best_table.to_markdown(index=False)}

---

## 4. Interpretation

This experiment should be interpreted as a stability analysis, not a tuning experiment.

If a model achieves a good F1-score only at a narrow threshold range, then its performance is threshold-sensitive and less stable.

If a model maintains similar F1-score across a wider threshold range while keeping false positive rate controlled, then it is more stable and more defensible as a baseline.

---

## 5. Key Points to Review

The main questions to answer from this table are:

1. Which model has the highest best F1?
2. Which model has the lowest false positive rate at its best F1?
3. Which model has the widest stable percentile range?
4. Does 60/15 actually improve stability, or only increase overlap?
5. Is LOF's improvement stable, or threshold-dependent?
6. Is Isolation Forest more stable across thresholds?

---

## 6. Output Files

Full threshold curve table:

`{threshold_all_path}`

Stability summary:

`{stability_summary_path}`

Plots are saved in:

`{INTERP_RESULTS}`

---

## 7. Status

Completed.
"""

report_path = os.path.join(
    INTERP_LOGS,
    "EXP_NAB_threshold_stability_60_30_vs_60_15.md"
)

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print("Saved report:")
print(report_path)

In [ ]:
# ============================================================
# FPR-constrained operating point analysis
# ============================================================

fpr_limits = [0.10, 0.15, 0.20, 0.25, 0.30]

constrained_rows = []

for fpr_limit in fpr_limits:
    for (config, model), g in threshold_all_df.groupby(["configuration", "model"]):
        eligible = g[g["fpr"] <= fpr_limit].copy()

        if eligible.empty:
            constrained_rows.append({
                "fpr_limit": fpr_limit,
                "configuration": config,
                "model": model,
                "status": "no eligible threshold",
                "best_percentile": None,
                "precision": None,
                "recall": None,
                "f1": None,
                "fpr": None,
                "roc_auc": g["roc_auc"].iloc[0],
                "pr_auc": g["pr_auc"].iloc[0],
            })
        else:
            best = eligible.sort_values(
                by=["f1", "recall", "precision"],
                ascending=False
            ).iloc[0]

            constrained_rows.append({
                "fpr_limit": fpr_limit,
                "configuration": config,
                "model": model,
                "status": "eligible",
                "best_percentile": best["percentile"],
                "precision": best["precision"],
                "recall": best["recall"],
                "f1": best["f1"],
                "fpr": best["fpr"],
                "roc_auc": best["roc_auc"],
                "pr_auc": best["pr_auc"],
            })

fpr_constrained_df = pd.DataFrame(constrained_rows)

fpr_constrained_path = os.path.join(
    INTERP_RESULTS,
    "fpr_constrained_operating_points.csv"
)

fpr_constrained_df.to_csv(fpr_constrained_path, index=False)

print("Saved FPR-constrained operating points:")
print(fpr_constrained_path)

display(
    fpr_constrained_df[fpr_constrained_df["status"] == "eligible"]
    .sort_values(["fpr_limit", "f1"], ascending=[True, False])
)

# EXP_NAB_Representation_Improvement

Goal: Improve the NAB window representation before further model evaluation and SHAP/LIME.
Focus:
1. Per-series normalization
2. Robust normalization
3. Feature refinement/selection
4. Robust window-level features

In [ ]:
# ============================================================
# EXP_NAB_Representation_Improvement
# Cell 1: Setup and file discovery
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import pandas as pd
import numpy as np
import os
from datetime import datetime

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

EXP_DIR = BASE / "experiments" / "EXP_NAB_Representation_Improvement"
EXP_DIR.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("Experiment directory:", EXP_DIR)
print("BASE exists:", BASE.exists())

# Search only inside the thesis code folder to avoid scanning the whole Drive
valid_suffixes = {".csv", ".parquet", ".pkl", ".pickle"}

keywords = [
    "nab",
    "window",
    "windows",
    "feature",
    "features",
    "60",
    "30",
    "15",
    "lof",
    "iforest",
    "isolation",
    "autoencoder"
]

candidates = []

for path in BASE.rglob("*"):
    if path.is_file() and path.suffix.lower() in valid_suffixes:
        name_lower = path.name.lower()
        full_lower = str(path).lower()

        score = sum(1 for k in keywords if k in full_lower)

        if "nab" in full_lower or score >= 3:
            try:
                size_mb = path.stat().st_size / (1024 * 1024)
                modified = datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
            except Exception:
                size_mb = None
                modified = None

            candidates.append({
                "score": score,
                "filename": path.name,
                "suffix": path.suffix,
                "size_mb": round(size_mb, 3) if size_mb is not None else None,
                "modified": modified,
                "path": str(path)
            })

candidates_df = pd.DataFrame(candidates)

if candidates_df.empty:
    print("No candidate files found. Check BASE path.")
else:
    candidates_df = candidates_df.sort_values(
        by=["score", "modified", "size_mb"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    print("Candidate files found:", len(candidates_df))
    display(candidates_df.head(80))

In [ ]:
# ============================================================
# EXP_NAB_Representation_Improvement
# Cell 2: Load NAB 60/30 and 60/15 window files + inspect schema
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")
EXP_DIR = BASE / "experiments" / "EXP_NAB_Representation_Improvement"
EXP_DIR.mkdir(parents=True, exist_ok=True)

# Locate the exact files
file_6030 = list(BASE.rglob("nab_windows_60min_stride30.csv"))
file_6015 = list(BASE.rglob("nab_windows_60min_stride15.csv"))

assert len(file_6030) > 0, "Could not find nab_windows_60min_stride30.csv"
assert len(file_6015) > 0, "Could not find nab_windows_60min_stride15.csv"

file_6030 = file_6030[0]
file_6015 = file_6015[0]

print("60/30 file:", file_6030)
print("60/15 file:", file_6015)

df_6030 = pd.read_csv(file_6030)
df_6015 = pd.read_csv(file_6015)

print("\nLoaded:")
print("60/30 shape:", df_6030.shape)
print("60/15 shape:", df_6015.shape)

def detect_columns(df):
    label_candidates = [
        "label", "is_anomaly", "anomaly", "window_label",
        "target", "y", "is_anomalous"
    ]

    series_candidates = [
        "series", "series_id", "file", "filename", "source_file",
        "stream", "metric", "metric_name", "domain", "entity"
    ]

    time_candidates = [
        "window_start", "start_time", "start", "timestamp",
        "time", "window_end", "end_time", "end"
    ]

    label_col = next((c for c in label_candidates if c in df.columns), None)
    series_col = next((c for c in series_candidates if c in df.columns), None)
    time_col = next((c for c in time_candidates if c in df.columns), None)

    return label_col, series_col, time_col

for name, df in [("60/30", df_6030), ("60/15", df_6015)]:
    label_col, series_col, time_col = detect_columns(df)

    print("\n" + "="*80)
    print(f"Dataset: {name}")
    print("="*80)
    print("Shape:", df.shape)
    print("Detected label column:", label_col)
    print("Detected series column:", series_col)
    print("Detected time column:", time_col)

    if label_col is not None:
        print("\nLabel distribution:")
        print(df[label_col].value_counts(dropna=False).sort_index())
        print("Anomaly ratio:", df[label_col].mean())

    if series_col is not None:
        print("\nNumber of unique series:", df[series_col].nunique())
        print("Example series values:")
        print(df[series_col].dropna().astype(str).unique()[:10])

    print("\nColumns:")
    print(list(df.columns))

    print("\nHead:")
    display(df.head())

In [ ]:
# ============================================================
# EXP_NAB_Representation_Improvement
# Cell 3: Representation scaling experiment
# Models: Isolation Forest + LOF
# Variants:
#   1. global_standard
#   2. global_robust
#   3. per_series_zscore
#   4. per_series_robust
# ============================================================

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# Keep LOF manageable on CPU.
# For final thesis runs, we can increase this or set to None if runtime allows.
LOF_TRAIN_MAX = 60000

FEATURE_PRIORITY = [
    "energy", "first_last_diff", "iqr", "kurtosis", "mad", "max",
    "max_abs_diff", "mean", "mean_abs_diff", "median", "min",
    "n_points", "peak_count", "q25", "q75", "range", "rms",
    "skew", "slope", "std", "std_diff", "sum", "var"
]

def detect_columns(df):
    label_candidates = [
        "label", "is_anomaly", "anomaly", "window_label",
        "target", "y", "is_anomalous"
    ]

    series_candidates = [
        "series", "series_id", "file", "filename", "source_file",
        "stream", "metric", "metric_name", "domain", "entity"
    ]

    time_candidates = [
        "window_start", "start_time", "start", "timestamp",
        "time", "window_end", "end_time", "end"
    ]

    label_col = next((c for c in label_candidates if c in df.columns), None)
    series_col = next((c for c in series_candidates if c in df.columns), None)
    time_col = next((c for c in time_candidates if c in df.columns), None)

    if label_col is None:
        raise ValueError("Could not detect label column.")

    if series_col is None:
        df["_series_id_auto"] = "all_series"
        series_col = "_series_id_auto"

    return label_col, series_col, time_col


def detect_feature_columns(df, label_col):
    priority_features = [c for c in FEATURE_PRIORITY if c in df.columns]
    if len(priority_features) >= 5:
        return priority_features

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    exclude_patterns = [
        "label", "target", "anomaly", "id", "index",
        "start", "end", "time", "timestamp",
        "window_size", "step", "stride", "sampling"
    ]

    feature_cols = []
    for c in numeric_cols:
        if c == label_col:
            continue
        c_lower = c.lower()
        if any(p in c_lower for p in exclude_patterns):
            continue
        feature_cols.append(c)

    return feature_cols


def add_chronological_split(df, series_col, time_col=None):
    df = df.copy()

    if time_col is not None:
        try:
            df[time_col] = pd.to_datetime(df[time_col])
        except Exception:
            pass
        df = df.sort_values([series_col, time_col]).reset_index(drop=True)
    else:
        df = df.sort_values([series_col]).reset_index(drop=True)

    df["_split"] = "train"

    for _, idx in df.groupby(series_col, sort=False).groups.items():
        idx = np.array(list(idx))
        n = len(idx)
        n_train = int(np.floor(0.60 * n))
        n_val = int(np.floor(0.20 * n))

        train_idx = idx[:n_train]
        val_idx = idx[n_train:n_train+n_val]
        test_idx = idx[n_train+n_val:]

        df.loc[train_idx, "_split"] = "train"
        df.loc[val_idx, "_split"] = "val"
        df.loc[test_idx, "_split"] = "test"

    return df


def safe_scale(scale):
    scale = scale.copy()
    scale = scale.replace([np.inf, -np.inf], np.nan)
    scale = scale.fillna(1.0)
    scale[scale == 0] = 1.0
    return scale


def transform_features(df, feature_cols, series_col, variant):
    train_df = df[df["_split"] == "train"].copy()

    X_raw = df[feature_cols].apply(pd.to_numeric, errors="coerce")
    X_train_raw = train_df[feature_cols].apply(pd.to_numeric, errors="coerce")

    global_median = X_train_raw.median()
    X_raw = X_raw.fillna(global_median)
    X_train_raw = X_train_raw.fillna(global_median)

    if variant == "global_standard":
        center = X_train_raw.mean()
        scale = safe_scale(X_train_raw.std())
        X = (X_raw - center) / scale
        return X.astype(float)

    if variant == "global_robust":
        center = X_train_raw.median()
        q75 = X_train_raw.quantile(0.75)
        q25 = X_train_raw.quantile(0.25)
        scale = safe_scale(q75 - q25)
        X = (X_raw - center) / scale
        return X.astype(float)

    if variant in ["per_series_zscore", "per_series_robust"]:
        X_out = pd.DataFrame(index=df.index, columns=feature_cols, dtype=float)

        global_mean = X_train_raw.mean()
        global_std = safe_scale(X_train_raw.std())
        global_med = X_train_raw.median()
        global_iqr = safe_scale(X_train_raw.quantile(0.75) - X_train_raw.quantile(0.25))

        for series_value, idx in df.groupby(series_col, sort=False).groups.items():
            idx = list(idx)
            train_idx = df[(df[series_col] == series_value) & (df["_split"] == "train")].index

            if len(train_idx) == 0:
                if variant == "per_series_zscore":
                    center = global_mean
                    scale = global_std
                else:
                    center = global_med
                    scale = global_iqr
            else:
                train_part = X_raw.loc[train_idx, feature_cols]
                if variant == "per_series_zscore":
                    center = train_part.mean()
                    scale = safe_scale(train_part.std())
                else:
                    center = train_part.median()
                    scale = safe_scale(train_part.quantile(0.75) - train_part.quantile(0.25))

            X_out.loc[idx, feature_cols] = (X_raw.loc[idx, feature_cols] - center) / scale

        return X_out.astype(float)

    raise ValueError(f"Unknown variant: {variant}")


def metric_dict(y_true, scores, threshold):
    y_pred = (scores >= threshold).astype(int)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fpr": fpr,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn
    }


def evaluate_thresholds(y_val, scores_val, y_test, scores_test):
    percentiles = np.arange(50, 98, 2.5)
    thresholds = np.unique(np.percentile(scores_val, percentiles))

    rows = []
    for thr in thresholds:
        val_m = metric_dict(y_val, scores_val, thr)
        rows.append({
            "threshold": thr,
            "val_precision": val_m["precision"],
            "val_recall": val_m["recall"],
            "val_f1": val_m["f1"],
            "val_fpr": val_m["fpr"]
        })

    grid = pd.DataFrame(rows)

    # Best validation F1, tie-breaker: lower validation FPR
    best_row = grid.sort_values(["val_f1", "val_fpr"], ascending=[False, True]).iloc[0]
    best_thr = best_row["threshold"]

    test_m = metric_dict(y_test, scores_test, best_thr)

    try:
        roc_auc = roc_auc_score(y_test, scores_test)
    except Exception:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(y_test, scores_test)
    except Exception:
        pr_auc = np.nan

    best_result = {
        "selected_threshold": best_thr,
        "val_f1": best_row["val_f1"],
        "val_fpr": best_row["val_fpr"],
        "test_precision": test_m["precision"],
        "test_recall": test_m["recall"],
        "test_f1": test_m["f1"],
        "test_fpr": test_m["fpr"],
        "test_roc_auc": roc_auc,
        "test_pr_auc": pr_auc,
        "test_tp": test_m["tp"],
        "test_fp": test_m["fp"],
        "test_tn": test_m["tn"],
        "test_fn": test_m["fn"]
    }

    fpr_rows = []
    for limit in [0.10, 0.15, 0.20, 0.25, 0.30]:
        allowed = grid[grid["val_fpr"] <= limit].copy()
        if allowed.empty:
            chosen = grid.sort_values(["val_fpr", "val_f1"], ascending=[True, False]).iloc[0]
            feasible = False
        else:
            chosen = allowed.sort_values(["val_f1", "val_fpr"], ascending=[False, True]).iloc[0]
            feasible = True

        thr = chosen["threshold"]
        test_c = metric_dict(y_test, scores_test, thr)

        fpr_rows.append({
            "fpr_limit": limit,
            "feasible_on_val": feasible,
            "selected_threshold": thr,
            "val_f1": chosen["val_f1"],
            "val_fpr": chosen["val_fpr"],
            "test_precision": test_c["precision"],
            "test_recall": test_c["recall"],
            "test_f1": test_c["f1"],
            "test_fpr": test_c["fpr"]
        })

    return best_result, pd.DataFrame(fpr_rows), grid


def fit_score_model(model_name, X_train, X_val, X_test):
    if model_name == "IsolationForest":
        model = IsolationForest(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            contamination="auto"
        )
        model.fit(X_train)
        scores_val = -model.decision_function(X_val)
        scores_test = -model.decision_function(X_test)
        return scores_val, scores_test

    if model_name == "LOF":
        X_fit = X_train

        if LOF_TRAIN_MAX is not None and len(X_fit) > LOF_TRAIN_MAX:
            rng = np.random.default_rng(RANDOM_STATE)
            sample_idx = rng.choice(len(X_fit), size=LOF_TRAIN_MAX, replace=False)
            X_fit = X_fit.iloc[sample_idx]
            print(f"LOF trained on sampled subset: {len(X_fit):,} rows")

        model = LocalOutlierFactor(
            n_neighbors=35,
            novelty=True,
            contamination="auto",
            n_jobs=-1
        )
        model.fit(X_fit)
        scores_val = -model.decision_function(X_val)
        scores_test = -model.decision_function(X_test)
        return scores_val, scores_test

    raise ValueError(f"Unknown model: {model_name}")


def run_representation_experiment(df, window_name):
    df = df.copy()

    label_col, series_col, time_col = detect_columns(df)
    feature_cols = detect_feature_columns(df, label_col)

    print("\n" + "="*90)
    print(f"Running window configuration: {window_name}")
    print("="*90)
    print("Label column:", label_col)
    print("Series column:", series_col)
    print("Time column:", time_col)
    print("Number of features:", len(feature_cols))
    print("Features:", feature_cols)

    df = add_chronological_split(df, series_col, time_col)

    print("\nSplit distribution:")
    print(df.groupby("_split")[label_col].agg(["count", "sum", "mean"]))

    y_train = df.loc[df["_split"] == "train", label_col].astype(int).values
    y_val = df.loc[df["_split"] == "val", label_col].astype(int).values
    y_test = df.loc[df["_split"] == "test", label_col].astype(int).values

    all_results = []
    all_fpr_results = []

    variants = [
        "global_standard",
        "global_robust",
        "per_series_zscore",
        "per_series_robust"
    ]

    models = ["IsolationForest", "LOF"]

    for variant in variants:
        print(f"\n--- Representation variant: {variant} ---")
        X_all = transform_features(df, feature_cols, series_col, variant)

        # Safety cleanup
        X_all = X_all.replace([np.inf, -np.inf], np.nan)
        X_all = X_all.fillna(0.0)

        X_train = X_all.loc[df["_split"] == "train"]
        X_val = X_all.loc[df["_split"] == "val"]
        X_test = X_all.loc[df["_split"] == "test"]

        for model_name in models:
            print(f"Training/evaluating: {model_name}")

            scores_val, scores_test = fit_score_model(model_name, X_train, X_val, X_test)
            best_result, fpr_df, grid_df = evaluate_thresholds(
                y_val, scores_val, y_test, scores_test
            )

            result_row = {
                "window_config": window_name,
                "variant": variant,
                "model": model_name,
                **best_result
            }
            all_results.append(result_row)

            fpr_df.insert(0, "model", model_name)
            fpr_df.insert(0, "variant", variant)
            fpr_df.insert(0, "window_config", window_name)
            all_fpr_results.append(fpr_df)

    results_df = pd.DataFrame(all_results)
    fpr_results_df = pd.concat(all_fpr_results, ignore_index=True)

    return results_df, fpr_results_df


# Run both window configurations
results_6030, fpr_6030 = run_representation_experiment(df_6030, "60/30")
results_6015, fpr_6015 = run_representation_experiment(df_6015, "60/15")

all_results = pd.concat([results_6030, results_6015], ignore_index=True)
all_fpr_results = pd.concat([fpr_6030, fpr_6015], ignore_index=True)

# Save outputs
results_path = EXP_DIR / "representation_scaling_best_f1_results.csv"
fpr_path = EXP_DIR / "representation_scaling_fpr_constrained_results.csv"

all_results.to_csv(results_path, index=False)
all_fpr_results.to_csv(fpr_path, index=False)

print("\nSaved:")
print(results_path)
print(fpr_path)

print("\nBest-F1 threshold selection results, sorted by test F1:")
display(
    all_results.sort_values(
        ["test_f1", "test_fpr"],
        ascending=[False, True]
    )
)

print("\nFPR-constrained results, sorted by FPR limit then test F1:")
display(
    all_fpr_results.sort_values(
        ["fpr_limit", "test_f1"],
        ascending=[True, False]
    )
)

In [ ]:
# ============================================================
# EXP_NAB_Representation_Improvement
# Cell 4: Focused result summary for interpretation
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")
EXP_DIR = BASE / "experiments" / "EXP_NAB_Representation_Improvement"

best_path = EXP_DIR / "representation_scaling_best_f1_results.csv"
fpr_path = EXP_DIR / "representation_scaling_fpr_constrained_results.csv"

best_df = pd.read_csv(best_path)
fpr_df = pd.read_csv(fpr_path)

print("Loaded:")
print(best_path)
print(fpr_path)

# ------------------------------------------------------------
# 1. Best-F1 summary
# ------------------------------------------------------------

best_cols = [
    "window_config", "variant", "model",
    "test_precision", "test_recall", "test_f1", "test_fpr",
    "test_roc_auc", "test_pr_auc"
]

best_summary = (
    best_df[best_cols]
    .sort_values(["test_f1", "test_fpr"], ascending=[False, True])
    .reset_index(drop=True)
)

print("\nTop Best-F1 Results:")
display(best_summary.head(12))

# ------------------------------------------------------------
# 2. FPR <= 0.10 summary
# ------------------------------------------------------------

fpr10 = fpr_df[fpr_df["fpr_limit"] == 0.10].copy()

fpr10_cols = [
    "window_config", "variant", "model",
    "test_precision", "test_recall", "test_f1", "test_fpr"
]

fpr10_summary = (
    fpr10[fpr10_cols]
    .sort_values(["test_f1", "test_fpr"], ascending=[False, True])
    .reset_index(drop=True)
)

print("\nBest Results Under FPR <= 0.10:")
display(fpr10_summary.head(12))

# ------------------------------------------------------------
# 3. Isolation Forest comparison only
# ------------------------------------------------------------

if_compare = fpr10[
    fpr10["model"] == "IsolationForest"
].copy()

if_compare = if_compare[
    [
        "window_config", "variant",
        "test_precision", "test_recall", "test_f1", "test_fpr"
    ]
].sort_values(["window_config", "test_f1"], ascending=[True, False])

print("\nIsolation Forest Representation Comparison Under FPR <= 0.10:")
display(if_compare)

# ------------------------------------------------------------
# 4. Quantify improvement over current Chapter 4 baseline
#    Current Chapter 4 IF 60/30 at FPR <= 0.10:
#    Precision=0.2267, Recall=0.1848, F1=0.2036, FPR=0.0882
# ------------------------------------------------------------

old_baseline = {
    "precision": 0.2267,
    "recall": 0.1848,
    "f1": 0.2036,
    "fpr": 0.0882
}

new_row = fpr10[
    (fpr10["window_config"] == "60/30") &
    (fpr10["variant"] == "per_series_zscore") &
    (fpr10["model"] == "IsolationForest")
].iloc[0]

improvement = pd.DataFrame([
    {
        "metric": "Precision",
        "old_if_6030": old_baseline["precision"],
        "new_if_6030_per_series_zscore": new_row["test_precision"],
        "absolute_change": new_row["test_precision"] - old_baseline["precision"]
    },
    {
        "metric": "Recall",
        "old_if_6030": old_baseline["recall"],
        "new_if_6030_per_series_zscore": new_row["test_recall"],
        "absolute_change": new_row["test_recall"] - old_baseline["recall"]
    },
    {
        "metric": "F1",
        "old_if_6030": old_baseline["f1"],
        "new_if_6030_per_series_zscore": new_row["test_f1"],
        "absolute_change": new_row["test_f1"] - old_baseline["f1"]
    },
    {
        "metric": "FPR",
        "old_if_6030": old_baseline["fpr"],
        "new_if_6030_per_series_zscore": new_row["test_fpr"],
        "absolute_change": new_row["test_fpr"] - old_baseline["fpr"]
    }
])

print("\nImprovement Over Current Chapter 4 IF 60/30 Baseline:")
display(improvement)

# ------------------------------------------------------------
# 5. Save clean summaries
# ------------------------------------------------------------

best_summary_path = EXP_DIR / "summary_best_f1_results_clean.csv"
fpr10_summary_path = EXP_DIR / "summary_fpr010_results_clean.csv"
if_compare_path = EXP_DIR / "summary_if_representation_comparison_fpr010.csv"
improvement_path = EXP_DIR / "summary_if6030_improvement_over_ch4_baseline.csv"

best_summary.to_csv(best_summary_path, index=False)
fpr10_summary.to_csv(fpr10_summary_path, index=False)
if_compare.to_csv(if_compare_path, index=False)
improvement.to_csv(improvement_path, index=False)

print("\nSaved clean summary files:")
print(best_summary_path)
print(fpr10_summary_path)
print(if_compare_path)
print(improvement_path)

In [ ]:
# ============================================================
# EXP_NAB_Representation_Improvement
# Cell 5: Feature refinement / selection on per-series z-score
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")
EXP_DIR = BASE / "experiments" / "EXP_NAB_Representation_Improvement"
EXP_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Helper: correlation-based feature pruning using training only
# ------------------------------------------------------------

def correlation_prune_features(X_train, threshold=0.95):
    """
    Label-free correlation pruning.
    Uses training features only to remove highly redundant features.
    """
    corr = X_train.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    mean_corr = corr.mean(axis=1)
    to_drop = set()

    for col in upper.columns:
        high_corr_with = upper.index[upper[col] > threshold].tolist()
        for row in high_corr_with:
            if row in to_drop or col in to_drop:
                continue

            # Drop the feature that is more correlated on average with all others
            if mean_corr[col] >= mean_corr[row]:
                to_drop.add(col)
            else:
                to_drop.add(row)

    kept = [c for c in X_train.columns if c not in to_drop]
    return kept, sorted(list(to_drop))


def near_constant_filter(X_train, min_std=1e-8):
    stds = X_train.std()
    kept = stds[stds > min_std].index.tolist()
    dropped = stds[stds <= min_std].index.tolist()
    return kept, dropped


def get_domain_feature_sets(all_features):
    """
    Domain-informed, label-free feature subsets.
    These are based on feature meaning, not label optimization.
    """
    sets = {}

    sets["all_features"] = list(all_features)

    change_features = [
        "mean_abs_diff", "max_abs_diff", "std_diff", "first_last_diff",
        "slope", "peak_count"
    ]

    variability_features = [
        "std", "var", "iqr", "range", "mad", "skew", "kurtosis"
    ]

    level_energy_features = [
        "mean", "median", "min", "max", "q25", "q75",
        "sum", "energy", "rms"
    ]

    robust_mixed_features = [
        "mean_abs_diff", "max_abs_diff", "std_diff", "first_last_diff",
        "slope", "peak_count",
        "std", "iqr", "range", "mad",
        "energy", "rms", "skew", "kurtosis"
    ]

    compact_features = [
        "mean_abs_diff", "std_diff", "mad", "iqr", "range",
        "energy", "rms", "slope", "peak_count", "skew"
    ]

    sets["change_only"] = [f for f in change_features if f in all_features]
    sets["variability_only"] = [f for f in variability_features if f in all_features]
    sets["level_energy_only"] = [f for f in level_energy_features if f in all_features]
    sets["robust_mixed"] = [f for f in robust_mixed_features if f in all_features]
    sets["compact_domain"] = [f for f in compact_features if f in all_features]

    return sets


def run_feature_selection_experiment(df, window_name):
    df = df.copy()

    label_col, series_col, time_col = detect_columns(df)
    feature_cols = detect_feature_columns(df, label_col)

    print("\n" + "="*90)
    print(f"Feature selection experiment: {window_name}")
    print("="*90)
    print("Detected features:", feature_cols)

    df = add_chronological_split(df, series_col, time_col)

    y_val = df.loc[df["_split"] == "val", label_col].astype(int).values
    y_test = df.loc[df["_split"] == "test", label_col].astype(int).values

    # Use the winning representation from previous experiment
    X_all = transform_features(
        df=df,
        feature_cols=feature_cols,
        series_col=series_col,
        variant="per_series_zscore"
    )

    X_all = X_all.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    X_train_full = X_all.loc[df["_split"] == "train"]
    X_val_full = X_all.loc[df["_split"] == "val"]
    X_test_full = X_all.loc[df["_split"] == "test"]

    # --------------------------------------------------------
    # Label-free feature subsets
    # --------------------------------------------------------

    subset_dict = get_domain_feature_sets(feature_cols)

    # Near-constant filter
    non_constant_features, dropped_constant = near_constant_filter(X_train_full)
    subset_dict["non_constant"] = non_constant_features

    # Correlation pruning at multiple thresholds
    for thr in [0.95, 0.90, 0.85, 0.80]:
        kept, dropped = correlation_prune_features(
            X_train_full[non_constant_features],
            threshold=thr
        )
        subset_dict[f"corr_pruned_{thr}"] = kept
        print(f"Correlation pruning {thr}: kept={len(kept)}, dropped={len(dropped)}")

    # Remove empty or duplicate subsets
    clean_subsets = {}
    seen = set()

    for name, feats in subset_dict.items():
        feats = [f for f in feats if f in X_all.columns]
        key = tuple(feats)
        if len(feats) == 0:
            continue
        if key in seen:
            continue
        seen.add(key)
        clean_subsets[name] = feats

    print("\nFeature subsets to evaluate:")
    for name, feats in clean_subsets.items():
        print(f"{name}: {len(feats)} features -> {feats}")

    models = ["IsolationForest", "LOF"]
    rows = []
    fpr_rows_all = []

    for subset_name, feats in clean_subsets.items():
        print(f"\n--- Subset: {subset_name} ({len(feats)} features) ---")

        X_train = X_train_full[feats]
        X_val = X_val_full[feats]
        X_test = X_test_full[feats]

        for model_name in models:
            print(f"Training/evaluating: {model_name}")

            scores_val, scores_test = fit_score_model(
                model_name=model_name,
                X_train=X_train,
                X_val=X_val,
                X_test=X_test
            )

            best_result, fpr_df_sub, grid_df = evaluate_thresholds(
                y_val=y_val,
                scores_val=scores_val,
                y_test=y_test,
                scores_test=scores_test
            )

            rows.append({
                "window_config": window_name,
                "representation": "per_series_zscore",
                "feature_subset": subset_name,
                "n_features": len(feats),
                "features": "|".join(feats),
                "model": model_name,
                **best_result
            })

            fpr_df_sub.insert(0, "model", model_name)
            fpr_df_sub.insert(0, "features", "|".join(feats))
            fpr_df_sub.insert(0, "n_features", len(feats))
            fpr_df_sub.insert(0, "feature_subset", subset_name)
            fpr_df_sub.insert(0, "representation", "per_series_zscore")
            fpr_df_sub.insert(0, "window_config", window_name)
            fpr_rows_all.append(fpr_df_sub)

    results_df = pd.DataFrame(rows)
    fpr_results_df = pd.concat(fpr_rows_all, ignore_index=True)

    return results_df, fpr_results_df


# Run feature selection experiment on both window configurations
fs_results_6030, fs_fpr_6030 = run_feature_selection_experiment(df_6030, "60/30")
fs_results_6015, fs_fpr_6015 = run_feature_selection_experiment(df_6015, "60/15")

fs_results_all = pd.concat([fs_results_6030, fs_results_6015], ignore_index=True)
fs_fpr_all = pd.concat([fs_fpr_6030, fs_fpr_6015], ignore_index=True)

# Save outputs
fs_results_path = EXP_DIR / "feature_selection_best_f1_results.csv"
fs_fpr_path = EXP_DIR / "feature_selection_fpr_constrained_results.csv"

fs_results_all.to_csv(fs_results_path, index=False)
fs_fpr_all.to_csv(fs_fpr_path, index=False)

print("\nSaved:")
print(fs_results_path)
print(fs_fpr_path)

print("\nTop feature-selection results by test F1:")
display(
    fs_results_all.sort_values(
        ["test_f1", "test_fpr"],
        ascending=[False, True]
    ).head(20)
)

print("\nFeature-selection results under validation FPR <= 0.10:")
fs_fpr10 = fs_fpr_all[fs_fpr_all["fpr_limit"] == 0.10].copy()
display(
    fs_fpr10.sort_values(
        ["test_f1", "test_fpr"],
        ascending=[False, True]
    ).head(20)
)

In [ ]:
# ============================================================
# EXP_NAB_Representation_Improvement
# Cell 6: Export clean tables, figures, and interpretation notes
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")
EXP_DIR = BASE / "experiments" / "EXP_NAB_Representation_Improvement"
FIG_DIR = EXP_DIR / "figures"
TABLE_DIR = EXP_DIR / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Load experiment outputs
# ------------------------------------------------------------

scaling_best = pd.read_csv(EXP_DIR / "representation_scaling_best_f1_results.csv")
scaling_fpr = pd.read_csv(EXP_DIR / "representation_scaling_fpr_constrained_results.csv")

fs_best = pd.read_csv(EXP_DIR / "feature_selection_best_f1_results.csv")
fs_fpr = pd.read_csv(EXP_DIR / "feature_selection_fpr_constrained_results.csv")

print("Loaded scaling and feature-selection outputs.")

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def round_metrics(df, cols=None, decimals=4):
    df = df.copy()
    if cols is None:
        cols = df.select_dtypes(include=[np.number]).columns
    for c in cols:
        if c in df.columns:
            df[c] = df[c].round(decimals)
    return df


def save_latex_table(df, path, caption, label):
    latex = df.to_latex(
        index=False,
        escape=True,
        caption=caption,
        label=label
    )
    path.write_text(latex, encoding="utf-8")
    print("Saved LaTeX table:", path)


def save_csv(df, path):
    df.to_csv(path, index=False)
    print("Saved CSV:", path)


# ------------------------------------------------------------
# 1. Scaling experiment: IF comparison under validation FPR <= 0.10
# ------------------------------------------------------------

fpr10 = scaling_fpr[scaling_fpr["fpr_limit"] == 0.10].copy()

if_fpr10 = fpr10[fpr10["model"] == "IsolationForest"].copy()
if_fpr10 = if_fpr10[
    [
        "window_config", "variant", "test_precision",
        "test_recall", "test_f1", "test_fpr"
    ]
].sort_values(["window_config", "test_f1"], ascending=[True, False])

if_fpr10_clean = round_metrics(if_fpr10)

save_csv(if_fpr10_clean, TABLE_DIR / "if_representation_comparison_fpr010.csv")
save_latex_table(
    if_fpr10_clean,
    TABLE_DIR / "if_representation_comparison_fpr010.tex",
    "Isolation Forest representation comparison under validation FPR $\\leq$ 0.10.",
    "tab:if_representation_comparison_fpr010"
)

display(if_fpr10_clean)

# Plot: IF F1 by representation variant and window config
plot_df = if_fpr10.copy()
plot_df["config_variant"] = plot_df["window_config"] + "\n" + plot_df["variant"]

plt.figure(figsize=(11, 5))
plt.bar(plot_df["config_variant"], plot_df["test_f1"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Test F1-score")
plt.title("Isolation Forest Representation Comparison under Validation FPR ≤ 0.10")
plt.tight_layout()

fig_path = FIG_DIR / "if_representation_comparison_fpr010.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure:", fig_path)

# ------------------------------------------------------------
# 2. Old Chapter 4 baseline vs improved per-series z-score
# ------------------------------------------------------------

old_new = pd.DataFrame([
    {
        "metric": "Precision",
        "chapter4_if_6030": 0.2267,
        "improved_if_6030_per_series_zscore": 0.294830,
        "absolute_change": 0.294830 - 0.2267
    },
    {
        "metric": "Recall",
        "chapter4_if_6030": 0.1848,
        "improved_if_6030_per_series_zscore": 0.286950,
        "absolute_change": 0.286950 - 0.1848
    },
    {
        "metric": "F1",
        "chapter4_if_6030": 0.2036,
        "improved_if_6030_per_series_zscore": 0.290837,
        "absolute_change": 0.290837 - 0.2036
    },
    {
        "metric": "FPR",
        "chapter4_if_6030": 0.0882,
        "improved_if_6030_per_series_zscore": 0.095939,
        "absolute_change": 0.095939 - 0.0882
    }
])

old_new_clean = round_metrics(old_new)

save_csv(old_new_clean, TABLE_DIR / "if6030_old_vs_improved_per_series_zscore.csv")
save_latex_table(
    old_new_clean,
    TABLE_DIR / "if6030_old_vs_improved_per_series_zscore.tex",
    "Comparison between the Chapter 4 Isolation Forest 60/30 baseline and the improved per-series z-score representation.",
    "tab:if6030_old_vs_improved_per_series_zscore"
)

display(old_new_clean)

# Plot: old vs new metrics
x = np.arange(len(old_new))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, old_new["chapter4_if_6030"], width, label="Chapter 4 baseline")
plt.bar(x + width/2, old_new["improved_if_6030_per_series_zscore"], width, label="Per-series z-score")
plt.xticks(x, old_new["metric"])
plt.ylabel("Metric value")
plt.title("Chapter 4 Baseline vs Improved Representation")
plt.legend()
plt.tight_layout()

fig_path = FIG_DIR / "if6030_old_vs_improved_per_series_zscore.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure:", fig_path)

# ------------------------------------------------------------
# 3. Feature-selection summary under validation FPR <= 0.10
# ------------------------------------------------------------

fs_fpr10 = fs_fpr[fs_fpr["fpr_limit"] == 0.10].copy()

fs_top = fs_fpr10[
    [
        "window_config", "feature_subset", "n_features", "model",
        "test_precision", "test_recall", "test_f1", "test_fpr"
    ]
].sort_values(["test_f1", "test_fpr"], ascending=[False, True]).head(12)

fs_top_clean = round_metrics(fs_top)

save_csv(fs_top_clean, TABLE_DIR / "feature_selection_top_fpr010.csv")
save_latex_table(
    fs_top_clean,
    TABLE_DIR / "feature_selection_top_fpr010.tex",
    "Top feature-selection results under validation FPR $\\leq$ 0.10.",
    "tab:feature_selection_top_fpr010"
)

display(fs_top_clean)

# Plot: top feature-selection F1 with FPR labels
plot_df = fs_top.copy()
plot_df["label"] = (
    plot_df["window_config"] + "\n" +
    plot_df["feature_subset"] + "\n" +
    plot_df["model"]
)

plt.figure(figsize=(12, 6))
plt.bar(plot_df["label"], plot_df["test_f1"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Test F1-score")
plt.title("Top Feature-Selection Results under Validation FPR ≤ 0.10")
plt.tight_layout()

fig_path = FIG_DIR / "feature_selection_top_fpr010.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure:", fig_path)

# ------------------------------------------------------------
# 4. Best-F1 feature-selection summary, independent of FPR preference
# ------------------------------------------------------------

fs_best_top = fs_best[
    [
        "window_config", "feature_subset", "n_features", "model",
        "test_precision", "test_recall", "test_f1", "test_fpr",
        "test_roc_auc", "test_pr_auc"
    ]
].sort_values(["test_f1", "test_fpr"], ascending=[False, True]).head(12)

fs_best_top_clean = round_metrics(fs_best_top)

save_csv(fs_best_top_clean, TABLE_DIR / "feature_selection_top_bestf1.csv")
save_latex_table(
    fs_best_top_clean,
    TABLE_DIR / "feature_selection_top_bestf1.tex",
    "Top feature-selection results ranked by test F1-score.",
    "tab:feature_selection_top_bestf1"
)

display(fs_best_top_clean)

# ------------------------------------------------------------
# 5. Interpretation notes for thesis writing
# ------------------------------------------------------------

notes = """
EXP_NAB_Representation_Improvement -- Interpretation Notes

1. Representation scaling result:
Per-series z-score normalization produced the strongest and most defensible improvement, especially with Isolation Forest. Under the 60/30 setting, per-series z-score normalization with all 23 features achieved Precision = 0.2948, Recall = 0.2870, F1 = 0.2908, and FPR = 0.0959.

2. Comparison with Chapter 4 baseline:
Compared with the earlier Chapter 4 Isolation Forest 60/30 baseline, the improved representation increased Precision by +0.0681, Recall by +0.1022, and F1 by +0.0872, while FPR increased only slightly by +0.0077. This supports the interpretation that cross-series scale variation was limiting the earlier representation.

3. Stride interpretation:
After per-series z-score normalization, the 60/30 and 60/15 configurations produced very similar Isolation Forest results. This supports the revised interpretation that denser overlap alone is not the main source of improvement. The representation normalization is more important than stride reduction.

4. Feature-selection result:
Feature selection did not clearly outperform the all-feature per-series z-score representation under a strict false-positive interpretation. The 60/15 corr_pruned_0.95 configuration achieved a slightly higher F1, but the test FPR increased above 0.10 and ROC-AUC/PR-AUC were lower than the all-feature configuration. Therefore, it should not replace the main baseline.

5. Level-energy feature subset:
The level_energy_only subset improved recall and F1, but at the cost of higher FPR. This makes it useful as a recall-sensitive representation, not as the main baseline.

6. Provisional main configuration:
The current provisional main configuration is:
- 60-minute window / 30-minute stride
- per-series z-score normalization
- all 23 window-level features
- Isolation Forest
This configuration is stable, interpretable, and defensible under FPR-aware evaluation.

7. Next step:
The next experiment should test robust window-level features or apply the selected representation to the next dataset branch before SHAP/LIME. SHAP and LIME should be applied only after the stable representation/model configuration is finalized.
"""

notes_path = EXP_DIR / "representation_improvement_interpretation_notes.txt"
notes_path.write_text(textwrap.dedent(notes).strip(), encoding="utf-8")

print("Saved notes:", notes_path)

print("\nAll exports completed.")
print("Figures saved to:", FIG_DIR)
print("Tables saved to:", TABLE_DIR)

# EXP_NAB_Robust_Window_Features

Goal: Rebuild NAB windows with additional robust window-level features.

This experiment continues the representation improvement stage by adding more robust descriptors that may better capture spikes, drops, local irregularity, and sustained deviations.

Previous result:
- Best provisional setup: 60/30 + per-series z-score + all 23 features + Isolation Forest.
- Next question: can more robust within-window features improve the representation further?

In [ ]:
# ============================================================
# EXP_NAB_Robust_Window_Features
# Cell 1: Discover standardized/raw NAB time-series files
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import pandas as pd
import numpy as np
import os
from datetime import datetime

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

EXP_DIR = BASE / "experiments" / "EXP_NAB_Robust_Window_Features"
EXP_DIR.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("Experiment directory:", EXP_DIR)
print("BASE exists:", BASE.exists())

# Search for likely NAB raw/standardized files and label files
valid_suffixes = {".csv", ".json", ".parquet", ".txt"}

keywords = [
    "nab",
    "numenta",
    "standardized",
    "standardised",
    "raw",
    "real",
    "artificial",
    "labels",
    "combined",
    "anomaly",
    "time_series",
    "timeseries"
]

candidates = []

for path in BASE.rglob("*"):
    if path.is_file() and path.suffix.lower() in valid_suffixes:
        full_lower = str(path).lower()
        name_lower = path.name.lower()

        score = sum(1 for k in keywords if k in full_lower)

        # Strongly include NAB-related files
        include = (
            "nab" in full_lower
            or "artificial" in full_lower
            or "realtraffic" in full_lower
            or "realadexchange" in full_lower
            or "realmicrosoftcloud" in full_lower
            or "realawscloudwatch" in full_lower
            or "labels" in name_lower
        )

        if include:
            try:
                size_mb = path.stat().st_size / (1024 * 1024)
                modified = datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
            except Exception:
                size_mb = None
                modified = None

            candidates.append({
                "score": score,
                "filename": path.name,
                "suffix": path.suffix,
                "size_mb": round(size_mb, 3) if size_mb is not None else None,
                "modified": modified,
                "path": str(path)
            })

candidates_df = pd.DataFrame(candidates)

if candidates_df.empty:
    print("No candidates found. We may need to inspect the folder structure manually.")
else:
    candidates_df = candidates_df.sort_values(
        by=["score", "modified", "size_mb"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    print("Candidate files found:", len(candidates_df))
    display(candidates_df.head(120))

In [ ]:
# ============================================================
# EXP_NAB_Robust_Window_Features
# Cell 2: Inspect candidate folders and sample file schemas
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from collections import Counter

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

# Rebuild candidate list in case previous variable is not available
valid_suffixes = {".csv", ".json"}

keywords = [
    "nab", "numenta", "artificial", "realtraffic", "realadexchange",
    "realmicrosoftcloud", "realawscloudwatch", "labels", "combined"
]

candidates = []

for path in BASE.rglob("*"):
    if path.is_file() and path.suffix.lower() in valid_suffixes:
        full_lower = str(path).lower()
        name_lower = path.name.lower()
        include = any(k in full_lower for k in keywords)
        if include:
            try:
                size_mb = path.stat().st_size / (1024 * 1024)
            except Exception:
                size_mb = None

            candidates.append({
                "filename": path.name,
                "suffix": path.suffix,
                "size_mb": round(size_mb, 4) if size_mb is not None else None,
                "parent": str(path.parent),
                "path": str(path)
            })

cand = pd.DataFrame(candidates)

print("Total candidates:", len(cand))

# ------------------------------------------------------------
# 1. Show parent folders with counts
# ------------------------------------------------------------

folder_summary = (
    cand.groupby("parent")
    .agg(
        file_count=("filename", "count"),
        total_mb=("size_mb", "sum")
    )
    .reset_index()
    .sort_values(["file_count", "total_mb"], ascending=[False, False])
)

print("\nTop candidate folders:")
display(folder_summary.head(40))

# ------------------------------------------------------------
# 2. Show likely label files
# ------------------------------------------------------------

label_candidates = cand[
    cand["filename"].str.lower().str.contains("label|combined|anomaly", regex=True)
].copy()

print("\nLikely label/anomaly files:")
display(label_candidates.sort_values(["suffix", "filename"]).head(80))

# ------------------------------------------------------------
# 3. Show sample CSV schemas from top folders
# ------------------------------------------------------------

print("\nSample CSV schemas from likely folders:")

sample_csvs = cand[cand["suffix"].str.lower() == ".csv"].copy()

# Prefer files that look like raw NAB series, but inspect numentaTM too
preferred = sample_csvs[
    sample_csvs["filename"].str.lower().str.contains(
        "art_|machine_temperature|ec2_|twitter_|traffic|exchange|cpu|ambient|occupancy|speed",
        regex=True
    )
].head(20)

for i, row in preferred.iterrows():
    p = Path(row["path"])
    print("\n" + "="*100)
    print("File:", p.name)
    print("Parent:", p.parent)
    print("Path:", p)
    try:
        df_sample = pd.read_csv(p, nrows=5)
        print("Shape preview:", df_sample.shape)
        print("Columns:", list(df_sample.columns))
        display(df_sample.head())
    except Exception as e:
        print("Could not read:", e)

# ------------------------------------------------------------
# 4. Try reading JSON label files to understand structure
# ------------------------------------------------------------

json_labels = label_candidates[label_candidates["suffix"].str.lower() == ".json"].copy()

print("\nJSON label file structure preview:")

for i, row in json_labels.head(10).iterrows():
    p = Path(row["path"])
    print("\n" + "="*100)
    print("JSON:", p.name)
    print("Path:", p)
    try:
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)

        print("Type:", type(data))
        if isinstance(data, dict):
            print("Number of keys:", len(data))
            print("First keys:", list(data.keys())[:10])
            first_key = list(data.keys())[0] if len(data) > 0 else None
            if first_key is not None:
                print("First key example:", first_key)
                print("First value example:", data[first_key][:3] if isinstance(data[first_key], list) else data[first_key])
        elif isinstance(data, list):
            print("Length:", len(data))
            print("First item:", data[0] if len(data) > 0 else None)

    except Exception as e:
        print("Could not read JSON:", e)

In [ ]:
# ============================================================
# EXP_NAB_Robust_Window_Features
# Cell 3: Rebuild NAB windows from raw official files
#         with baseline + robust window-level features
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import math
import warnings
warnings.filterwarnings("ignore")

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

DATA_ROOT = BASE / "Datasets" / "nab_official" / "data"
LABEL_PATH = BASE / "Datasets" / "nab_official" / "labels" / "combined_labels.json"

EXP_DIR = BASE / "experiments" / "EXP_NAB_Robust_Window_Features"
OUT_DIR = BASE / "processed_data" / "PH4" / "EXP_NAB_Robust_Window_Features"
EXP_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("LABEL_PATH:", LABEL_PATH)
print("DATA_ROOT exists:", DATA_ROOT.exists())
print("LABEL_PATH exists:", LABEL_PATH.exists())

with open(LABEL_PATH, "r", encoding="utf-8") as f:
    labels_dict = json.load(f)

print("Label keys:", len(labels_dict))


def safe_float(x):
    try:
        if pd.isna(x):
            return 0.0
        if np.isinf(x):
            return 0.0
        return float(x)
    except Exception:
        return 0.0


def entropy_from_values(x, bins=10):
    x = np.asarray(x, dtype=float)
    if len(x) < 2 or np.nanstd(x) == 0:
        return 0.0

    counts, _ = np.histogram(x, bins=bins)
    total = counts.sum()
    if total == 0:
        return 0.0

    p = counts[counts > 0] / total
    return float(-(p * np.log2(p)).sum())


def autocorr_lag1(x):
    x = np.asarray(x, dtype=float)
    if len(x) < 3:
        return 0.0

    x1 = x[:-1]
    x2 = x[1:]

    if np.std(x1) == 0 or np.std(x2) == 0:
        return 0.0

    return safe_float(np.corrcoef(x1, x2)[0, 1])


def linear_slope(x):
    x = np.asarray(x, dtype=float)
    if len(x) < 2:
        return 0.0

    t = np.arange(len(x), dtype=float)
    try:
        return safe_float(np.polyfit(t, x, 1)[0])
    except Exception:
        return 0.0


def local_max_count(x):
    x = np.asarray(x, dtype=float)
    if len(x) < 3:
        return 0

    count = 0
    for i in range(1, len(x) - 1):
        if x[i] > x[i - 1] and x[i] > x[i + 1]:
            count += 1
    return int(count)


def extract_robust_window_features(values):
    x = np.asarray(values, dtype=float)
    x = x[~np.isnan(x)]

    if len(x) == 0:
        return None

    n = len(x)

    mean = np.mean(x)
    std = np.std(x, ddof=1) if n > 1 else 0.0
    min_v = np.min(x)
    max_v = np.max(x)
    median = np.median(x)
    q25 = np.percentile(x, 25)
    q75 = np.percentile(x, 75)
    iqr = q75 - q25
    range_v = max_v - min_v
    sum_v = np.sum(x)
    energy = np.sum(x ** 2)
    rms = np.sqrt(np.mean(x ** 2))
    var = np.var(x, ddof=1) if n > 1 else 0.0
    mad = np.median(np.abs(x - median))

    s = pd.Series(x)
    skew = safe_float(s.skew()) if n > 2 else 0.0
    kurtosis = safe_float(s.kurt()) if n > 3 else 0.0

    diffs = np.diff(x)
    abs_diffs = np.abs(diffs)

    mean_abs_diff = np.mean(abs_diffs) if len(abs_diffs) > 0 else 0.0
    max_abs_diff = np.max(abs_diffs) if len(abs_diffs) > 0 else 0.0
    std_diff = np.std(diffs, ddof=1) if len(diffs) > 1 else 0.0
    first_last_diff = x[-1] - x[0] if n > 1 else 0.0
    slope = linear_slope(x)
    peak_count = local_max_count(x)

    # Robust scale. 1.4826*MAD estimates std under normality.
    robust_scale = 1.4826 * mad
    if robust_scale == 0 or np.isnan(robust_scale):
        robust_scale = iqr / 1.349 if iqr != 0 else 1.0
    if robust_scale == 0 or np.isnan(robust_scale):
        robust_scale = 1.0

    robust_z = np.abs((x - median) / robust_scale)

    # Robust spike/drop descriptors
    robust_z_max = np.max(robust_z)
    robust_z_mean = np.mean(robust_z)
    robust_z_std = np.std(robust_z, ddof=1) if n > 1 else 0.0
    robust_z_q90 = np.percentile(robust_z, 90)
    robust_z_q95 = np.percentile(robust_z, 95)
    robust_outlier_count_z3 = int(np.sum(robust_z > 3.0))
    robust_outlier_ratio_z3 = robust_outlier_count_z3 / n

    # Delta-based descriptors
    if len(diffs) > 0:
        delta_abs_q90 = np.percentile(abs_diffs, 90)
        delta_abs_q95 = np.percentile(abs_diffs, 95)
        delta_abs_iqr = np.percentile(abs_diffs, 75) - np.percentile(abs_diffs, 25)

        positive_jump_count = int(np.sum(diffs > delta_abs_q90))
        negative_jump_count = int(np.sum(diffs < -delta_abs_q90))
        positive_diff_ratio = float(np.mean(diffs > 0))
        negative_diff_ratio = float(np.mean(diffs < 0))
        zero_diff_ratio = float(np.mean(diffs == 0))

        signs = np.sign(diffs)
        nonzero_signs = signs[signs != 0]
        sign_change_count = int(np.sum(nonzero_signs[1:] != nonzero_signs[:-1])) if len(nonzero_signs) > 1 else 0
    else:
        delta_abs_q90 = 0.0
        delta_abs_q95 = 0.0
        delta_abs_iqr = 0.0
        positive_jump_count = 0
        negative_jump_count = 0
        positive_diff_ratio = 0.0
        negative_diff_ratio = 0.0
        zero_diff_ratio = 0.0
        sign_change_count = 0

    # Distributional ratios
    above_median_ratio = float(np.mean(x > median))
    above_q75_ratio = float(np.mean(x > q75))
    below_q25_ratio = float(np.mean(x < q25))
    zero_value_ratio = float(np.mean(x == 0))
    unique_value_ratio = float(len(np.unique(x)) / n)

    # Local instability using rolling median
    if n >= 3:
        roll_med = pd.Series(x).rolling(window=3, center=True, min_periods=1).median().values
        local_dev = np.abs(x - roll_med)
        local_dev_mean = np.mean(local_dev)
        local_dev_max = np.max(local_dev)
    else:
        local_dev_mean = 0.0
        local_dev_max = 0.0

    trend_strength = abs(slope) / (std + 1e-9)
    normalized_range_mad = range_v / (robust_scale + 1e-9)

    features = {
        # Original 23-style features
        "n_points": n,
        "mean": mean,
        "std": std,
        "min": min_v,
        "max": max_v,
        "median": median,
        "q25": q25,
        "q75": q75,
        "iqr": iqr,
        "range": range_v,
        "sum": sum_v,
        "energy": energy,
        "rms": rms,
        "var": var,
        "mad": mad,
        "skew": skew,
        "kurtosis": kurtosis,
        "mean_abs_diff": mean_abs_diff,
        "max_abs_diff": max_abs_diff,
        "std_diff": std_diff,
        "first_last_diff": first_last_diff,
        "slope": slope,
        "peak_count": peak_count,

        # New robust features
        "robust_z_max": robust_z_max,
        "robust_z_mean": robust_z_mean,
        "robust_z_std": robust_z_std,
        "robust_z_q90": robust_z_q90,
        "robust_z_q95": robust_z_q95,
        "robust_outlier_count_z3": robust_outlier_count_z3,
        "robust_outlier_ratio_z3": robust_outlier_ratio_z3,
        "delta_abs_q90": delta_abs_q90,
        "delta_abs_q95": delta_abs_q95,
        "delta_abs_iqr": delta_abs_iqr,
        "positive_jump_count": positive_jump_count,
        "negative_jump_count": negative_jump_count,
        "positive_diff_ratio": positive_diff_ratio,
        "negative_diff_ratio": negative_diff_ratio,
        "zero_diff_ratio": zero_diff_ratio,
        "sign_change_count": sign_change_count,
        "above_median_ratio": above_median_ratio,
        "above_q75_ratio": above_q75_ratio,
        "below_q25_ratio": below_q25_ratio,
        "zero_value_ratio": zero_value_ratio,
        "unique_value_ratio": unique_value_ratio,
        "local_dev_mean": local_dev_mean,
        "local_dev_max": local_dev_max,
        "trend_strength": trend_strength,
        "normalized_range_mad": normalized_range_mad,
        "entropy": entropy_from_values(x),
        "autocorr_lag1": autocorr_lag1(x)
    }

    return {k: safe_float(v) for k, v in features.items()}


def build_windows_for_series(series_key, csv_path, anomaly_times, window_minutes=60, stride_minutes=30):
    df = pd.read_csv(csv_path)

    if "timestamp" not in df.columns or "value" not in df.columns:
        return []

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["timestamp", "value"]).sort_values("timestamp").reset_index(drop=True)

    if df.empty:
        return []

    anomaly_times = pd.to_datetime(anomaly_times) if len(anomaly_times) > 0 else pd.to_datetime([])

    start_min = df["timestamp"].min()
    start_max = df["timestamp"].max()

    window_delta = pd.Timedelta(minutes=window_minutes)
    stride_delta = pd.Timedelta(minutes=stride_minutes)

    rows = []
    current_start = start_min
    window_id = 0

    # End-exclusive windowing, matching previous n_points=12 for 5-minute NAB files.
    while current_start + window_delta <= start_max:
        current_end = current_start + window_delta

        mask = (df["timestamp"] >= current_start) & (df["timestamp"] < current_end)
        values = df.loc[mask, "value"].values

        if len(values) >= 2:
            feats = extract_robust_window_features(values)

            if feats is not None:
                if len(anomaly_times) > 0:
                    label = int(((anomaly_times >= current_start) & (anomaly_times < current_end)).any())
                else:
                    label = 0

                domain = series_key.split("/")[0]

                row = {
                    "series": series_key,
                    "domain": domain,
                    "window_id": window_id,
                    "window_start": current_start,
                    "window_end": current_end,
                    "label": label,
                    **feats
                }

                rows.append(row)

        current_start += stride_delta
        window_id += 1

    return rows


def build_nab_robust_windows(window_minutes=60, stride_minutes=30):
    all_rows = []

    for i, series_key in enumerate(sorted(labels_dict.keys())):
        csv_path = DATA_ROOT / series_key

        if not csv_path.exists():
            print("Missing:", series_key)
            continue

        anomaly_times = labels_dict.get(series_key, [])
        rows = build_windows_for_series(
            series_key=series_key,
            csv_path=csv_path,
            anomaly_times=anomaly_times,
            window_minutes=window_minutes,
            stride_minutes=stride_minutes
        )

        all_rows.extend(rows)

        if (i + 1) % 10 == 0:
            print(f"Processed {i+1}/{len(labels_dict)} series. Rows so far: {len(all_rows):,}")

    out = pd.DataFrame(all_rows)

    return out


# Build both configurations
robust_6030 = build_nab_robust_windows(window_minutes=60, stride_minutes=30)
robust_6015 = build_nab_robust_windows(window_minutes=60, stride_minutes=15)

print("\nBuilt robust windows:")
print("60/30:", robust_6030.shape)
print("60/15:", robust_6015.shape)

print("\nLabel distribution 60/30:")
print(robust_6030["label"].value_counts().sort_index())
print("Anomaly ratio:", robust_6030["label"].mean())

print("\nLabel distribution 60/15:")
print(robust_6015["label"].value_counts().sort_index())
print("Anomaly ratio:", robust_6015["label"].mean())

# Save
path_6030 = OUT_DIR / "nab_robust_windows_60min_stride30.csv"
path_6015 = OUT_DIR / "nab_robust_windows_60min_stride15.csv"

robust_6030.to_csv(path_6030, index=False)
robust_6015.to_csv(path_6015, index=False)

print("\nSaved:")
print(path_6030)
print(path_6015)

print("\nColumns:")
print(list(robust_6030.columns))

display(robust_6030.head())

In [ ]:
# ============================================================
# EXP_NAB_Robust_Window_Features
# Cell 4: Rebuild robust features using existing Chapter 4 window templates
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

DATA_ROOT = BASE / "Datasets" / "nab_official" / "data"

TEMPLATE_6030 = BASE / "processed_data" / "PH4" / "EXP_NAB_60min_stride30" / "nab_windows_60min_stride30.csv"
TEMPLATE_6015 = BASE / "processed_data" / "PH4" / "EXP_NAB_60min_stride15" / "nab_windows_60min_stride15.csv"

OUT_DIR = BASE / "processed_data" / "PH4" / "EXP_NAB_Robust_Window_Features"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("TEMPLATE_6030:", TEMPLATE_6030)
print("TEMPLATE_6015:", TEMPLATE_6015)
print("DATA_ROOT exists:", DATA_ROOT.exists())
print("TEMPLATE_6030 exists:", TEMPLATE_6030.exists())
print("TEMPLATE_6015 exists:", TEMPLATE_6015.exists())

# This cell uses extract_robust_window_features() from the previous cell.
# Check that it exists.
if "extract_robust_window_features" not in globals():
    raise RuntimeError("extract_robust_window_features() is not defined. Run Cell 3 first.")


# ------------------------------------------------------------
# Feature names expected from extract_robust_window_features
# ------------------------------------------------------------

FEATURE_NAMES = [
    "n_points", "mean", "std", "min", "max", "median", "q25", "q75",
    "iqr", "range", "sum", "energy", "rms", "var", "mad", "skew",
    "kurtosis", "mean_abs_diff", "max_abs_diff", "std_diff",
    "first_last_diff", "slope", "peak_count",
    "robust_z_max", "robust_z_mean", "robust_z_std", "robust_z_q90",
    "robust_z_q95", "robust_outlier_count_z3", "robust_outlier_ratio_z3",
    "delta_abs_q90", "delta_abs_q95", "delta_abs_iqr",
    "positive_jump_count", "negative_jump_count",
    "positive_diff_ratio", "negative_diff_ratio", "zero_diff_ratio",
    "sign_change_count", "above_median_ratio", "above_q75_ratio",
    "below_q25_ratio", "zero_value_ratio", "unique_value_ratio",
    "local_dev_mean", "local_dev_max", "trend_strength",
    "normalized_range_mad", "entropy", "autocorr_lag1"
]

ZERO_FEATURES = {k: 0.0 for k in FEATURE_NAMES}


# ------------------------------------------------------------
# Raw series cache
# ------------------------------------------------------------

raw_cache = {}

def load_raw_series(series_key):
    if series_key in raw_cache:
        return raw_cache[series_key]

    path = DATA_ROOT / series_key

    if not path.exists():
        raise FileNotFoundError(f"Raw NAB file not found: {path}")

    df = pd.read_csv(path)

    if "timestamp" not in df.columns or "value" not in df.columns:
        raise ValueError(f"Unexpected schema in {path}: {list(df.columns)}")

    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    df = df.dropna(subset=["timestamp", "value"]).sort_values("timestamp").reset_index(drop=True)

    t_int = df["timestamp"].astype("int64").values
    values = df["value"].astype(float).values

    raw_cache[series_key] = (t_int, values)
    return raw_cache[series_key]


def rebuild_from_template(template_path, out_path, window_name):
    print("\n" + "="*90)
    print(f"Rebuilding robust features using template: {window_name}")
    print("="*90)

    template = pd.read_csv(template_path)

    required_cols = ["series", "domain", "window_start", "window_end", "label"]
    missing = [c for c in required_cols if c not in template.columns]
    if missing:
        raise ValueError(f"Template missing required columns: {missing}")

    template["window_start"] = pd.to_datetime(template["window_start"], utc=True)
    template["window_end"] = pd.to_datetime(template["window_end"], utc=True)

    print("Template shape:", template.shape)
    print("Template label distribution:")
    print(template["label"].value_counts().sort_index())
    print("Template anomaly ratio:", template["label"].mean())

    rows = []
    total = len(template)
    processed = 0
    missing_value_windows = 0

    for series_key, group in template.groupby("series", sort=False):
        t_int, values = load_raw_series(series_key)

        group = group.copy()
        starts = group["window_start"].astype("int64").values
        ends = group["window_end"].astype("int64").values

        left_idx = np.searchsorted(t_int, starts, side="left")
        right_idx = np.searchsorted(t_int, ends, side="left")

        for row_idx, (_, row) in enumerate(group.iterrows()):
            lo = left_idx[row_idx]
            hi = right_idx[row_idx]

            window_values = values[lo:hi]

            if len(window_values) >= 2:
                feats = extract_robust_window_features(window_values)
                if feats is None:
                    feats = ZERO_FEATURES.copy()
                    missing_value_windows += 1
            else:
                feats = ZERO_FEATURES.copy()
                feats["n_points"] = float(len(window_values))
                missing_value_windows += 1

            out_row = {
                "series": row["series"],
                "domain": row["domain"],
                "window_start": row["window_start"],
                "window_end": row["window_end"],
                "label": int(row["label"]),
                **feats
            }

            rows.append(out_row)

        processed += len(group)

        if processed % 20000 < len(group):
            print(f"Processed {processed:,}/{total:,} windows...")

    out = pd.DataFrame(rows)

    # Keep datetime as string for CSV compatibility
    out["window_start"] = out["window_start"].astype(str)
    out["window_end"] = out["window_end"].astype(str)

    out.to_csv(out_path, index=False)

    print("\nSaved:", out_path)
    print("Output shape:", out.shape)
    print("Output label distribution:")
    print(out["label"].value_counts().sort_index())
    print("Output anomaly ratio:", out["label"].mean())
    print("Missing/short-value windows:", missing_value_windows)
    print("Columns:", len(out.columns))

    print("\nN-points distribution:")
    print(out["n_points"].value_counts().sort_index().head(20))

    return out


# Build robust feature files aligned with old Chapter 4 templates
ROBUST_TEMPLATE_6030 = OUT_DIR / "nab_robust_template_windows_60min_stride30.csv"
ROBUST_TEMPLATE_6015 = OUT_DIR / "nab_robust_template_windows_60min_stride15.csv"

robust_template_6030 = rebuild_from_template(
    TEMPLATE_6030,
    ROBUST_TEMPLATE_6030,
    "60/30"
)

robust_template_6015 = rebuild_from_template(
    TEMPLATE_6015,
    ROBUST_TEMPLATE_6015,
    "60/15"
)

print("\nFinal saved files:")
print(ROBUST_TEMPLATE_6030)
print(ROBUST_TEMPLATE_6015)

print("\nPreview 60/30:")
display(robust_template_6030.head())

In [ ]:
# ============================================================
# EXP_NAB_Robust_Window_Features
# Cell 5: Sanity check original vs robust-template window alignment
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

TEMPLATE_6030 = BASE / "processed_data" / "PH4" / "EXP_NAB_60min_stride30" / "nab_windows_60min_stride30.csv"
TEMPLATE_6015 = BASE / "processed_data" / "PH4" / "EXP_NAB_60min_stride15" / "nab_windows_60min_stride15.csv"

ROBUST_6030 = BASE / "processed_data" / "PH4" / "EXP_NAB_Robust_Window_Features" / "nab_robust_template_windows_60min_stride30.csv"
ROBUST_6015 = BASE / "processed_data" / "PH4" / "EXP_NAB_Robust_Window_Features" / "nab_robust_template_windows_60min_stride15.csv"

orig_6030 = pd.read_csv(TEMPLATE_6030)
orig_6015 = pd.read_csv(TEMPLATE_6015)

rob_6030 = pd.read_csv(ROBUST_6030)
rob_6015 = pd.read_csv(ROBUST_6015)

def check_alignment(orig, rob, name):
    print("\n" + "="*80)
    print(f"Alignment check: {name}")
    print("="*80)

    print("Original shape:", orig.shape)
    print("Robust shape:", rob.shape)

    print("\nOriginal label distribution:")
    print(orig["label"].value_counts().sort_index())
    print("Original anomaly ratio:", orig["label"].mean())

    print("\nRobust label distribution:")
    print(rob["label"].value_counts().sort_index())
    print("Robust anomaly ratio:", rob["label"].mean())

    same_series = (orig["series"].astype(str).values == rob["series"].astype(str).values).all()
    same_labels = (orig["label"].astype(int).values == rob["label"].astype(int).values).all()

    print("\nSame series order:", same_series)
    print("Same labels:", same_labels)

    print("\nOriginal n_points distribution:")
    print(orig["n_points"].value_counts().sort_index().head(30))

    print("\nRobust n_points distribution:")
    print(rob["n_points"].value_counts().sort_index().head(30))

    if "n_points" in orig.columns:
        diff = orig["n_points"].astype(float).values - rob["n_points"].astype(float).values
        print("\nN-points difference summary:")
        print(pd.Series(diff).describe())

check_alignment(orig_6030, rob_6030, "60/30")
check_alignment(orig_6015, rob_6015, "60/15")

In [ ]:
# ============================================================
# EXP_NAB_Robust_Window_Features
# Cell 6: Evaluate robust window-level features
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, average_precision_score
)

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

EXP_DIR = BASE / "experiments" / "EXP_NAB_Robust_Window_Features"
EXP_DIR.mkdir(parents=True, exist_ok=True)

ROBUST_6030 = BASE / "processed_data" / "PH4" / "EXP_NAB_Robust_Window_Features" / "nab_robust_template_windows_60min_stride30.csv"
ROBUST_6015 = BASE / "processed_data" / "PH4" / "EXP_NAB_Robust_Window_Features" / "nab_robust_template_windows_60min_stride15.csv"

print("ROBUST_6030 exists:", ROBUST_6030.exists())
print("ROBUST_6015 exists:", ROBUST_6015.exists())

df_6030 = pd.read_csv(ROBUST_6030)
df_6015 = pd.read_csv(ROBUST_6015)

print("60/30:", df_6030.shape)
print("60/15:", df_6015.shape)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def detect_feature_columns(df):
    excluded = {
        "series", "domain", "window_start", "window_end",
        "label", "_split", "window_id"
    }
    return [
        c for c in df.columns
        if c not in excluded and pd.api.types.is_numeric_dtype(df[c])
    ]


def add_chronological_split(df, series_col="series", time_col="window_start"):
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col], utc=True)
    df["_split"] = None

    for series, g in df.groupby(series_col, sort=False):
        idx = g.sort_values(time_col).index.to_numpy()
        n = len(idx)

        train_end = int(n * 0.60)
        val_end = int(n * 0.80)

        df.loc[idx[:train_end], "_split"] = "train"
        df.loc[idx[train_end:val_end], "_split"] = "val"
        df.loc[idx[val_end:], "_split"] = "test"

    return df


def per_series_zscore_transform(df, feature_cols, series_col="series"):
    """
    Per-series z-score using training-region statistics only.
    """
    X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).copy()
    X_out = pd.DataFrame(index=df.index, columns=feature_cols, dtype=float)

    for series, idx_all in df.groupby(series_col, sort=False).groups.items():
        idx_all = list(idx_all)
        idx_train = df.loc[idx_all].index[df.loc[idx_all, "_split"] == "train"].tolist()

        train_values = X.loc[idx_train, feature_cols]

        means = train_values.mean()
        stds = train_values.std().replace(0, 1.0).fillna(1.0)

        X_out.loc[idx_all, feature_cols] = (X.loc[idx_all, feature_cols] - means) / stds

    X_out = X_out.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return X_out


def compute_metrics(y_true, y_pred, scores):
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    try:
        roc_auc = roc_auc_score(y_true, scores)
    except Exception:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(y_true, scores)
    except Exception:
        pr_auc = np.nan

    return {
        "test_precision": precision,
        "test_recall": recall,
        "test_f1": f1,
        "test_fpr": fpr,
        "test_roc_auc": roc_auc,
        "test_pr_auc": pr_auc,
        "test_tp": tp,
        "test_fp": fp,
        "test_tn": tn,
        "test_fn": fn
    }


def fit_score_model(model_name, X_train, X_val, X_test, random_state=42):
    if model_name == "IsolationForest":
        model = IsolationForest(
            n_estimators=300,
            contamination="auto",
            random_state=random_state,
            n_jobs=-1
        )
        model.fit(X_train)

        # Higher score = more anomalous
        scores_val = -model.decision_function(X_val)
        scores_test = -model.decision_function(X_test)

    elif model_name == "LOF":
        # Keep LOF manageable on Colab CPU
        LOF_TRAIN_MAX = 60000

        if len(X_train) > LOF_TRAIN_MAX:
            X_train_fit = X_train.sample(LOF_TRAIN_MAX, random_state=random_state)
            print(f"LOF trained on sampled subset: {LOF_TRAIN_MAX:,} rows")
        else:
            X_train_fit = X_train

        model = LocalOutlierFactor(
            n_neighbors=35,
            novelty=True,
            contamination="auto",
            n_jobs=-1
        )
        model.fit(X_train_fit)

        # Higher score = more anomalous
        scores_val = -model.decision_function(X_val)
        scores_test = -model.decision_function(X_test)

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return scores_val, scores_test


def evaluate_threshold_grid(y_val, scores_val, y_test, scores_test):
    percentiles = np.arange(50, 98, 2.5)

    grid_rows = []
    best = None

    for p in percentiles:
        threshold = np.percentile(scores_val, p)

        pred_val = (scores_val >= threshold).astype(int)
        pred_test = (scores_test >= threshold).astype(int)

        val_precision = precision_score(y_val, pred_val, zero_division=0)
        val_recall = recall_score(y_val, pred_val, zero_division=0)
        val_f1 = f1_score(y_val, pred_val, zero_division=0)

        tn, fp, fn, tp = confusion_matrix(y_val, pred_val, labels=[0, 1]).ravel()
        val_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

        test_metrics = compute_metrics(y_test, pred_test, scores_test)

        row = {
            "percentile": p,
            "selected_threshold": threshold,
            "val_precision": val_precision,
            "val_recall": val_recall,
            "val_f1": val_f1,
            "val_fpr": val_fpr,
            **test_metrics
        }

        grid_rows.append(row)

        if best is None or val_f1 > best["val_f1"]:
            best = row.copy()

    return best, pd.DataFrame(grid_rows)


def evaluate_fpr_constrained(y_val, scores_val, y_test, scores_test, limits=[0.10, 0.15, 0.20, 0.25, 0.30]):
    percentiles = np.arange(50, 98, 2.5)

    candidate_rows = []

    for p in percentiles:
        threshold = np.percentile(scores_val, p)

        pred_val = (scores_val >= threshold).astype(int)
        pred_test = (scores_test >= threshold).astype(int)

        val_precision = precision_score(y_val, pred_val, zero_division=0)
        val_recall = recall_score(y_val, pred_val, zero_division=0)
        val_f1 = f1_score(y_val, pred_val, zero_division=0)

        tn, fp, fn, tp = confusion_matrix(y_val, pred_val, labels=[0, 1]).ravel()
        val_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

        test_metrics = compute_metrics(y_test, pred_test, scores_test)

        candidate_rows.append({
            "percentile": p,
            "selected_threshold": threshold,
            "val_precision": val_precision,
            "val_recall": val_recall,
            "val_f1": val_f1,
            "val_fpr": val_fpr,
            **test_metrics
        })

    candidates = pd.DataFrame(candidate_rows)

    selected_rows = []

    for limit in limits:
        feasible = candidates[candidates["val_fpr"] <= limit].copy()

        if len(feasible) == 0:
            chosen = candidates.sort_values("val_fpr").iloc[0].copy()
            feasible_flag = False
        else:
            chosen = feasible.sort_values(["val_f1", "val_fpr"], ascending=[False, True]).iloc[0].copy()
            feasible_flag = True

        chosen["fpr_limit"] = limit
        chosen["feasible_on_val"] = feasible_flag
        selected_rows.append(chosen)

    return pd.DataFrame(selected_rows)


def run_robust_eval(df, window_name):
    df = add_chronological_split(df)

    all_features = detect_feature_columns(df)

    original_23 = [
        "energy", "first_last_diff", "iqr", "kurtosis", "mad", "max",
        "max_abs_diff", "mean", "mean_abs_diff", "median", "min",
        "n_points", "peak_count", "q25", "q75", "range", "rms",
        "skew", "slope", "std", "std_diff", "sum", "var"
    ]
    original_23 = [f for f in original_23 if f in all_features]

    robust_new = [
        f for f in all_features
        if f not in original_23
    ]

    robust_compact = [
        "mean_abs_diff", "max_abs_diff", "std_diff", "first_last_diff",
        "slope", "peak_count", "mad", "iqr", "range", "energy", "rms",
        "robust_z_max", "robust_z_mean", "robust_z_q90", "robust_z_q95",
        "robust_outlier_count_z3", "robust_outlier_ratio_z3",
        "delta_abs_q90", "delta_abs_q95", "delta_abs_iqr",
        "positive_jump_count", "negative_jump_count",
        "sign_change_count", "local_dev_mean", "local_dev_max",
        "trend_strength", "normalized_range_mad", "entropy", "autocorr_lag1"
    ]
    robust_compact = [f for f in robust_compact if f in all_features]

    feature_sets = {
        "original_23_recomputed": original_23,
        "robust_new_only": robust_new,
        "original_plus_robust_all": original_23 + robust_new,
        "robust_compact": robust_compact
    }

    print("\n" + "="*90)
    print(f"Running robust feature evaluation: {window_name}")
    print("="*90)
    print("Split distribution:")
    print(df.groupby("_split")["label"].agg(["count", "sum", "mean"]))

    y_val = df.loc[df["_split"] == "val", "label"].astype(int).values
    y_test = df.loc[df["_split"] == "test", "label"].astype(int).values

    results = []
    fpr_results = []

    for feature_set_name, features in feature_sets.items():
        print(f"\n--- Feature set: {feature_set_name} ({len(features)} features) ---")

        X_all = per_series_zscore_transform(df, features)

        X_train = X_all.loc[df["_split"] == "train"]
        X_val = X_all.loc[df["_split"] == "val"]
        X_test = X_all.loc[df["_split"] == "test"]

        for model_name in ["IsolationForest", "LOF"]:
            print(f"Training/evaluating: {model_name}")

            scores_val, scores_test = fit_score_model(
                model_name=model_name,
                X_train=X_train,
                X_val=X_val,
                X_test=X_test
            )

            best, grid = evaluate_threshold_grid(
                y_val=y_val,
                scores_val=scores_val,
                y_test=y_test,
                scores_test=scores_test
            )

            fpr_df = evaluate_fpr_constrained(
                y_val=y_val,
                scores_val=scores_val,
                y_test=y_test,
                scores_test=scores_test
            )

            results.append({
                "window_config": window_name,
                "normalization": "per_series_zscore",
                "feature_set": feature_set_name,
                "n_features": len(features),
                "model": model_name,
                **best
            })

            fpr_df.insert(0, "model", model_name)
            fpr_df.insert(0, "n_features", len(features))
            fpr_df.insert(0, "feature_set", feature_set_name)
            fpr_df.insert(0, "normalization", "per_series_zscore")
            fpr_df.insert(0, "window_config", window_name)

            fpr_results.append(fpr_df)

    return pd.DataFrame(results), pd.concat(fpr_results, ignore_index=True)


# Run
robust_best_6030, robust_fpr_6030 = run_robust_eval(df_6030, "60/30")
robust_best_6015, robust_fpr_6015 = run_robust_eval(df_6015, "60/15")

robust_best_all = pd.concat([robust_best_6030, robust_best_6015], ignore_index=True)
robust_fpr_all = pd.concat([robust_fpr_6030, robust_fpr_6015], ignore_index=True)

# Save
best_path = EXP_DIR / "robust_feature_best_f1_results.csv"
fpr_path = EXP_DIR / "robust_feature_fpr_constrained_results.csv"

robust_best_all.to_csv(best_path, index=False)
robust_fpr_all.to_csv(fpr_path, index=False)

print("\nSaved:")
print(best_path)
print(fpr_path)

print("\nTop robust-feature results by test F1:")
display(
    robust_best_all.sort_values(
        ["test_f1", "test_fpr"],
        ascending=[False, True]
    ).head(20)
)

print("\nRobust-feature results under validation FPR <= 0.10:")
display(
    robust_fpr_all[robust_fpr_all["fpr_limit"] == 0.10]
    .sort_values(["test_f1", "test_fpr"], ascending=[False, True])
    .head(20)
)

60-minute window / 30-minute stride
per-series or per-appliance z-score normalization
23 window-level features
Isolation Forest as main model
LOF and Autoencoder as comparison baselines later
synthetic anomaly injection only for evaluation if real labels are unavailable

In [ ]:
# ============================================================
# EXP_UKDALE_Selected_Pipeline
# Cell 1: Setup and discover UK-DALE files
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
import os

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

EXP_DIR = BASE / "experiments" / "EXP_UKDALE_Selected_Pipeline"
OUT_DIR = BASE / "processed_data" / "UKDALE_Selected_Pipeline"

EXP_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("Experiment directory:", EXP_DIR)
print("Output directory:", OUT_DIR)
print("BASE exists:", BASE.exists())

# ------------------------------------------------------------
# Confirm NAB result files are saved
# ------------------------------------------------------------

nab_files_to_check = [
    BASE / "experiments" / "EXP_NAB_Representation_Improvement" / "representation_scaling_best_f1_results.csv",
    BASE / "experiments" / "EXP_NAB_Representation_Improvement" / "representation_scaling_fpr_constrained_results.csv",
    BASE / "experiments" / "EXP_NAB_Representation_Improvement" / "feature_selection_best_f1_results.csv",
    BASE / "experiments" / "EXP_NAB_Representation_Improvement" / "feature_selection_fpr_constrained_results.csv",
    BASE / "experiments" / "EXP_NAB_Robust_Window_Features" / "robust_feature_best_f1_results.csv",
    BASE / "experiments" / "EXP_NAB_Robust_Window_Features" / "robust_feature_fpr_constrained_results.csv",
]

print("\nNAB saved-result check:")
for p in nab_files_to_check:
    print(p.name, "->", p.exists())

# ------------------------------------------------------------
# Discover UK-DALE files
# ------------------------------------------------------------

valid_suffixes = {".csv", ".h5", ".hdf5", ".yaml", ".yml", ".txt", ".json"}

keywords = [
    "ukdale", "uk-dale", "uk_dale",
    "building", "meter", "appliance",
    "channel", "mains", "kettle", "fridge",
    "washing", "dishwasher", "microwave"
]

candidates = []

for path in BASE.rglob("*"):
    if path.is_file() and path.suffix.lower() in valid_suffixes:
        full_lower = str(path).lower()
        name_lower = path.name.lower()

        if any(k in full_lower for k in keywords):
            try:
                size_mb = path.stat().st_size / (1024 * 1024)
                modified = datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
            except Exception:
                size_mb = None
                modified = None

            candidates.append({
                "filename": path.name,
                "suffix": path.suffix,
                "size_mb": round(size_mb, 3) if size_mb is not None else None,
                "modified": modified,
                "parent": str(path.parent),
                "path": str(path)
            })

candidates_df = pd.DataFrame(candidates)

if candidates_df.empty:
    print("\nNo UK-DALE candidates found.")
else:
    candidates_df = candidates_df.sort_values(
        by=["suffix", "size_mb"],
        ascending=[True, False]
    ).reset_index(drop=True)

    print("\nUK-DALE candidate files found:", len(candidates_df))
    display(candidates_df.head(120))

In [ ]:
# ============================================================
# EXP_UKDALE_Selected_Pipeline
# Cell 2: Inspect UK-DALE processed files and metadata
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
from datetime import datetime

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

valid_suffixes = {".csv", ".h5", ".hdf5", ".yaml", ".yml", ".json", ".txt"}

# ------------------------------------------------------------
# 1. Collect UK-DALE-related files, excluding old model score files
# ------------------------------------------------------------

exclude_terms = [
    "scores",
    "score",
    "prediction",
    "predictions",
    "result",
    "results",
    "anomaly_score"
]

include_terms = [
    "ukdale", "uk-dale", "uk_dale",
    "building", "meter", "appliance",
    "inventory", "channel", "mains",
    "kettle", "fridge", "washing", "dishwasher", "microwave"
]

records = []

for path in BASE.rglob("*"):
    if path.is_file() and path.suffix.lower() in valid_suffixes:
        full_lower = str(path).lower()
        name_lower = path.name.lower()

        include = any(t in full_lower for t in include_terms)
        exclude = any(t in name_lower for t in exclude_terms)

        if include and not exclude:
            try:
                size_mb = path.stat().st_size / (1024 * 1024)
                modified = datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
            except Exception:
                size_mb = None
                modified = None

            records.append({
                "filename": path.name,
                "suffix": path.suffix,
                "size_mb": round(size_mb, 3) if size_mb is not None else None,
                "modified": modified,
                "parent": str(path.parent),
                "path": str(path)
            })

uk_files = pd.DataFrame(records)

print("Filtered UK-DALE candidate files:", len(uk_files))

if not uk_files.empty:
    uk_files = uk_files.sort_values(["suffix", "size_mb"], ascending=[True, False]).reset_index(drop=True)

    print("\nTop folders:")
    folder_summary = (
        uk_files.groupby("parent")
        .agg(file_count=("filename", "count"), total_mb=("size_mb", "sum"))
        .reset_index()
        .sort_values(["file_count", "total_mb"], ascending=[False, False])
    )
    display(folder_summary.head(30))

    print("\nMetadata / mapping candidates:")
    meta = uk_files[
        uk_files["filename"].str.lower().str.contains(
            "map|mapping|inventory|summary|yaml|yml|meter_devices|building",
            regex=True
        )
    ].copy()
    display(meta.head(80))

    print("\nLikely 1-minute meter/appliance CSV files:")
    one_min = uk_files[
        uk_files["filename"].str.lower().str.contains(
            r"building\d+_elec_meter\d+_1min\.csv",
            regex=True
        )
    ].copy()

    # Extract building and meter number
    def extract_building_meter(name):
        m = re.search(r"building(\d+)_elec_meter(\d+)_1min\.csv", name.lower())
        if m:
            return int(m.group(1)), int(m.group(2))
        return None, None

    if not one_min.empty:
        bm = one_min["filename"].apply(extract_building_meter)
        one_min["building"] = [x[0] for x in bm]
        one_min["meter"] = [x[1] for x in bm]
        one_min = one_min.sort_values(["building", "meter"]).reset_index(drop=True)

        print("1-minute CSV count:", len(one_min))
        display(one_min.head(120))

        print("\n1-minute files by building:")
        display(
            one_min.groupby("building")
            .agg(file_count=("filename", "count"), total_mb=("size_mb", "sum"))
            .reset_index()
        )
    else:
        print("No building*_elec_meter*_1min.csv files found.")

# ------------------------------------------------------------
# 2. Inspect schemas of representative 1-minute files
# ------------------------------------------------------------

print("\nRepresentative CSV schemas:")

if "one_min" in globals() and not one_min.empty:
    sample_files = one_min.head(8)

    for _, row in sample_files.iterrows():
        p = Path(row["path"])
        print("\n" + "="*100)
        print("File:", p.name)
        print("Path:", p)
        print("Size MB:", row["size_mb"])

        try:
            df_sample = pd.read_csv(p, nrows=5)
            print("Columns:", list(df_sample.columns))
            print("Shape preview:", df_sample.shape)
            display(df_sample)
        except Exception as e:
            print("Could not read file:", e)

# ------------------------------------------------------------
# 3. Try loading likely map/inventory CSVs
# ------------------------------------------------------------

print("\nTrying to preview likely mapping/inventory CSV files:")

if not uk_files.empty:
    mapping_candidates = uk_files[
        (uk_files["suffix"].str.lower() == ".csv") &
        (uk_files["filename"].str.lower().str.contains("map|mapping|inventory|summary|appliance", regex=True))
    ].copy()

    display(mapping_candidates.head(30))

    for _, row in mapping_candidates.head(10).iterrows():
        p = Path(row["path"])
        print("\n" + "="*100)
        print("Metadata file:", p.name)
        print("Path:", p)

        try:
            df_meta = pd.read_csv(p, nrows=20)
            print("Columns:", list(df_meta.columns))
            display(df_meta.head(20))
        except Exception as e:
            print("Could not read metadata file:", e)

In [ ]:
# ============================================================
# EXP_UKDALE_Selected_Pipeline
# Cell 3: Find UK-DALE meter-to-appliance metadata
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
from datetime import datetime

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

valid_suffixes = {".csv", ".yaml", ".yml", ".json", ".txt", ".h5", ".hdf5"}

metadata_terms = [
    "ukdale",
    "uk-dale",
    "uk_dale",
    "meter_devices",
    "building1",
    "building2",
    "building3",
    "building4",
    "building5",
    "appliance",
    "appliances",
    "inventory",
    "map",
    "mapping",
    "metadata",
    "channel"
]

records = []

for path in BASE.rglob("*"):
    if path.is_file() and path.suffix.lower() in valid_suffixes:
        full_lower = str(path).lower()
        name_lower = path.name.lower()

        if any(t in full_lower for t in metadata_terms):
            try:
                size_mb = path.stat().st_size / (1024 * 1024)
                modified = datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
            except Exception:
                size_mb = None
                modified = None

            records.append({
                "filename": path.name,
                "suffix": path.suffix,
                "size_mb": round(size_mb, 3) if size_mb is not None else None,
                "modified": modified,
                "parent": str(path.parent),
                "path": str(path)
            })

meta_files = pd.DataFrame(records)

print("Metadata-related files found:", len(meta_files))

if meta_files.empty:
    print("No metadata files found.")
else:
    meta_files = meta_files.sort_values(["suffix", "size_mb"], ascending=[True, False]).reset_index(drop=True)

    print("\nAll metadata-related candidates:")
    display(meta_files.head(150))

    print("\nLikely YAML metadata files:")
    yaml_files = meta_files[meta_files["suffix"].str.lower().isin([".yaml", ".yml"])].copy()
    display(yaml_files)

    print("\nLikely CSV metadata/map files:")
    csv_meta = meta_files[
        (meta_files["suffix"].str.lower() == ".csv") &
        (meta_files["filename"].str.lower().str.contains("map|mapping|inventory|summary|appliance|channel|meter", regex=True))
    ].copy()
    display(csv_meta.head(80))

    print("\nLikely HDF5 files:")
    h5_files = meta_files[meta_files["suffix"].str.lower().isin([".h5", ".hdf5"])].copy()
    display(h5_files)

# ------------------------------------------------------------
# Preview readable metadata CSVs
# ------------------------------------------------------------

print("\nPreview likely metadata CSV files:")

if "csv_meta" in globals() and not csv_meta.empty:
    for _, row in csv_meta.head(20).iterrows():
        p = Path(row["path"])
        print("\n" + "="*100)
        print("CSV:", p.name)
        print("Path:", p)
        try:
            df = pd.read_csv(p, nrows=30)
            print("Columns:", list(df.columns))
            display(df.head(30))
        except Exception as e:
            print("Could not read:", e)

# ------------------------------------------------------------
# Preview YAML files as raw text
# ------------------------------------------------------------

print("\nPreview YAML files:")

if "yaml_files" in globals() and not yaml_files.empty:
    for _, row in yaml_files.head(20).iterrows():
        p = Path(row["path"])
        print("\n" + "="*100)
        print("YAML:", p.name)
        print("Path:", p)
        try:
            with open(p, "r", encoding="utf-8") as f:
                txt = f.read(3000)
            print(txt)
        except Exception as e:
            print("Could not read:", e)

In [ ]:
# ============================================================
# EXP_UKDALE_Selected_Pipeline
# Cell 4: Build full UK-DALE meter-to-appliance inventory
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import json

try:
    import yaml
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyyaml"])
    import yaml

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

UKDALE_META_DIR = BASE / "Datasets" / "ukdale"
UKDALE_1MIN_DIR = BASE / "processed_data" / "PH2" / "ukdale_all_series"
PH2_SUMMARY = UKDALE_1MIN_DIR / "ukdale_ph2_summary.csv"

OUT_DIR = BASE / "processed_data" / "UKDALE_Selected_Pipeline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("UKDALE_META_DIR:", UKDALE_META_DIR)
print("UKDALE_1MIN_DIR:", UKDALE_1MIN_DIR)
print("PH2_SUMMARY:", PH2_SUMMARY)
print("Metadata dir exists:", UKDALE_META_DIR.exists())
print("1-min dir exists:", UKDALE_1MIN_DIR.exists())
print("PH2 summary exists:", PH2_SUMMARY.exists())

# ------------------------------------------------------------
# Load PH2 summary if available
# ------------------------------------------------------------

ph2 = None
if PH2_SUMMARY.exists():
    ph2 = pd.read_csv(PH2_SUMMARY)
    print("\nPH2 summary shape:", ph2.shape)
    display(ph2.head())

    # Extract building and meter from series path like /building5/elec/meter1
    def extract_from_series(series):
        m = re.search(r"/building(\d+)/elec/meter(\d+)", str(series))
        if m:
            return int(m.group(1)), int(m.group(2))
        return None, None

    extracted = ph2["series"].apply(extract_from_series)
    ph2["building"] = [x[0] for x in extracted]
    ph2["meter"] = [x[1] for x in extracted]
else:
    print("No PH2 summary found. Inventory will still be created from YAML + file existence.")


# ------------------------------------------------------------
# Parse YAML metadata
# ------------------------------------------------------------

inventory_rows = []

for building_id in range(1, 6):
    yaml_path = UKDALE_META_DIR / f"building{building_id}.yaml"

    if not yaml_path.exists():
        print("Missing YAML:", yaml_path)
        continue

    with open(yaml_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    appliances = data.get("appliances", [])

    for app in appliances:
        meters = app.get("meters", [])
        if not isinstance(meters, list):
            meters = [meters]

        for meter in meters:
            filename = f"building{building_id}_elec_meter{meter}_1min.csv"
            csv_path = UKDALE_1MIN_DIR / filename

            row = {
                "dataset": "UK-DALE",
                "building": building_id,
                "meter": int(meter) if meter is not None else None,
                "filename": filename,
                "csv_path": str(csv_path),
                "csv_exists": csv_path.exists(),
                "original_name": app.get("original_name", None),
                "appliance_type": app.get("type", None),
                "instance": app.get("instance", None),
                "room": app.get("room", None),
                "manufacturer": app.get("manufacturer", None),
                "max_power": app.get("max_power", None),
                "on_power_threshold": app.get("on_power_threshold", None),
                "min_on_duration": app.get("min_on_duration", None),
                "min_off_duration": app.get("min_off_duration", None),
            }

            inventory_rows.append(row)

inventory = pd.DataFrame(inventory_rows)

# ------------------------------------------------------------
# Add row counts from PH2 summary
# ------------------------------------------------------------

if ph2 is not None:
    ph2_small = ph2[["building", "meter", "rows", "resolution", "status", "output_file"]].copy()
    inventory = inventory.merge(
        ph2_small,
        on=["building", "meter"],
        how="left"
    )
else:
    inventory["rows"] = np.nan
    inventory["resolution"] = np.nan
    inventory["status"] = np.nan
    inventory["output_file"] = np.nan

# ------------------------------------------------------------
# Mark thesis-relevant target appliances
# ------------------------------------------------------------

target_terms = [
    "kettle",
    "fridge",
    "freezer",
    "washing",
    "washer",
    "dishwasher",
    "dish_washer",
    "dish washer",
    "microwave"
]

def normalize_text(x):
    return str(x).lower().replace("-", "_").replace(" ", "_")

def is_target_appliance(row):
    text = normalize_text(row.get("original_name", "")) + " " + normalize_text(row.get("appliance_type", ""))
    return any(t.replace(" ", "_") in text for t in target_terms)

inventory["selected_candidate"] = inventory.apply(is_target_appliance, axis=1)

# Remove duplicate appliance-meter rows only if exact duplicates exist
inventory = inventory.drop_duplicates(
    subset=["building", "meter", "original_name", "appliance_type"]
).reset_index(drop=True)

# Sort
inventory = inventory.sort_values(["building", "meter", "original_name"]).reset_index(drop=True)

# Save full inventory
inventory_path = OUT_DIR / "ukdale_full_meter_appliance_inventory.csv"
inventory.to_csv(inventory_path, index=False)

print("\nFull UK-DALE inventory shape:", inventory.shape)
print("Saved:", inventory_path)

print("\nInventory by building:")
display(
    inventory.groupby("building")
    .agg(
        appliance_rows=("original_name", "count"),
        unique_meters=("meter", "nunique"),
        existing_csvs=("csv_exists", "sum"),
        selected_candidates=("selected_candidate", "sum")
    )
    .reset_index()
)

print("\nSelected candidate appliances:")
selected = inventory[
    (inventory["selected_candidate"] == True) &
    (inventory["csv_exists"] == True)
].copy()

display(
    selected[
        [
            "building", "meter", "original_name", "appliance_type",
            "rows", "csv_exists", "filename", "csv_path"
        ]
    ].sort_values(["building", "meter"])
)

selected_path = OUT_DIR / "ukdale_selected_candidate_appliances.csv"
selected.to_csv(selected_path, index=False)

print("\nSaved selected candidates:")
print(selected_path)

print("\nFull inventory preview:")
display(inventory.head(80))

In [ ]:
# ============================================================
# EXP_UKDALE_Selected_Pipeline
# Cell 5: Create experiment-ready selected appliance-meter list
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

OUT_DIR = BASE / "processed_data" / "UKDALE_Selected_Pipeline"
INVENTORY_PATH = OUT_DIR / "ukdale_full_meter_appliance_inventory.csv"

print("INVENTORY_PATH:", INVENTORY_PATH)
print("Exists:", INVENTORY_PATH.exists())

inventory = pd.read_csv(INVENTORY_PATH)

# Keep only selected candidate rows with existing CSV files
cand = inventory[
    (inventory["selected_candidate"] == True) &
    (inventory["csv_exists"] == True)
].copy()

print("Raw selected candidate rows:", cand.shape)

def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def join_unique(values):
    vals = []
    for v in values:
        v = clean_text(v)
        if v and v.lower() != "nan":
            vals.append(v)
    return "; ".join(sorted(set(vals)))

def infer_primary_category(text):
    t = text.lower().replace("-", "_").replace(" ", "_")

    if "kettle" in t:
        return "kettle"
    if "fridge" in t or "freezer" in t:
        return "fridge_freezer"
    if "dishwasher" in t or "dish_washer" in t or "dish_washer" in t or "dish" in t:
        return "dishwasher"
    if "washing" in t or "washer" in t:
        return "washing_machine"
    if "microwave" in t:
        return "microwave"

    return "other"

# Deduplicate by physical meter
group_cols = [
    "building", "meter", "filename", "csv_path", "rows",
    "csv_exists", "resolution", "status", "output_file"
]

dedup = (
    cand.groupby(group_cols, dropna=False)
    .agg(
        original_names=("original_name", join_unique),
        appliance_types=("appliance_type", join_unique),
        instances=("instance", lambda x: join_unique(x.astype(str))),
        rooms=("room", join_unique),
        manufacturers=("manufacturer", join_unique),
        max_power=("max_power", "max"),
        on_power_threshold=("on_power_threshold", "max"),
        min_on_duration=("min_on_duration", "max"),
        min_off_duration=("min_off_duration", "max"),
        n_metadata_rows=("original_name", "count")
    )
    .reset_index()
)

# Determine whether a meter is shared/ambiguous
dedup["combined_text"] = (
    dedup["original_names"].fillna("") + " " + dedup["appliance_types"].fillna("")
)

dedup["primary_category"] = dedup["combined_text"].apply(infer_primary_category)

# Shared meter if multiple metadata rows or text suggests mixed appliances
dedup["is_shared_or_ambiguous"] = False

for idx, row in dedup.iterrows():
    text = row["combined_text"].lower()

    # More than one metadata row often means same physical channel is assigned to multiple appliances/types
    if row["n_metadata_rows"] > 1:
        dedup.loc[idx, "is_shared_or_ambiguous"] = True

    # Explicit mixed names
    mixed_indicators = [
        "radio",
        "food processor",
        "toasted sandwich",
        "breadmaker",
        "tv",
        "dvd",
        "lamp"
    ]

    if any(m in text for m in mixed_indicators):
        dedup.loc[idx, "is_shared_or_ambiguous"] = True

# Main clean set: selected target meters that are not shared/ambiguous
dedup["use_in_main_experiment"] = ~dedup["is_shared_or_ambiguous"]

# Sort
dedup = dedup.sort_values(["building", "meter"]).reset_index(drop=True)

# Save
experiment_list_path = OUT_DIR / "ukdale_experiment_ready_appliance_meters.csv"
clean_main_path = OUT_DIR / "ukdale_clean_main_experiment_meters.csv"
ambiguous_path = OUT_DIR / "ukdale_shared_or_ambiguous_candidate_meters.csv"

dedup.to_csv(experiment_list_path, index=False)
dedup[dedup["use_in_main_experiment"] == True].to_csv(clean_main_path, index=False)
dedup[dedup["is_shared_or_ambiguous"] == True].to_csv(ambiguous_path, index=False)

print("\nExperiment-ready unique selected meters:", dedup.shape)
print("Saved:", experiment_list_path)

print("\nClean main-experiment meters:", dedup["use_in_main_experiment"].sum())
print("Saved:", clean_main_path)

print("\nShared/ambiguous candidate meters:", dedup["is_shared_or_ambiguous"].sum())
print("Saved:", ambiguous_path)

print("\nExperiment-ready selected meters:")
display(
    dedup[
        [
            "building", "meter", "primary_category",
            "original_names", "appliance_types",
            "rows", "n_metadata_rows",
            "is_shared_or_ambiguous", "use_in_main_experiment",
            "filename"
        ]
    ]
)

print("\nSummary by category:")
display(
    dedup.groupby(["primary_category", "use_in_main_experiment"])
    .agg(
        meters=("meter", "count"),
        total_rows=("rows", "sum")
    )
    .reset_index()
    .sort_values(["primary_category", "use_in_main_experiment"])
)

In [ ]:
# ============================================================
# EXP_UKDALE_Selected_Pipeline
# Cell 6: Build combined UK-DALE clean feature-window dataset
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import json

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

OUT_DIR = BASE / "processed_data" / "UKDALE_Selected_Pipeline"
FEATURE_DIR = BASE / "processed_data" / "PH3" / "UK_DALE" / "windows" / "features"

CLEAN_METERS_PATH = OUT_DIR / "ukdale_clean_main_experiment_meters.csv"

print("FEATURE_DIR:", FEATURE_DIR)
print("FEATURE_DIR exists:", FEATURE_DIR.exists())
print("CLEAN_METERS_PATH:", CLEAN_METERS_PATH)
print("CLEAN_METERS_PATH exists:", CLEAN_METERS_PATH.exists())

clean_meters = pd.read_csv(CLEAN_METERS_PATH)

print("\nClean meters shape:", clean_meters.shape)
display(
    clean_meters[
        [
            "building", "meter", "primary_category",
            "original_names", "appliance_types",
            "rows", "filename"
        ]
    ]
)

# ------------------------------------------------------------
# Selected 23 features from the NAB-selected pipeline
# ------------------------------------------------------------

SELECTED_23_FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def feature_window_filename(raw_filename):
    return raw_filename.replace(".csv", "_feature_windows.csv")

def find_time_columns(df):
    candidates = [
        "window_start", "start_time", "start", "timestamp",
        "index", "time", "datetime"
    ]
    return [c for c in candidates if c in df.columns]

# ------------------------------------------------------------
# Inspect feature-window availability and schemas
# ------------------------------------------------------------

records = []
parts = []
schema_examples = []

for _, row in clean_meters.iterrows():
    raw_filename = row["filename"]
    fw_name = feature_window_filename(raw_filename)
    fw_path = FEATURE_DIR / fw_name

    exists = fw_path.exists()

    rec = {
        "building": row["building"],
        "meter": row["meter"],
        "primary_category": row["primary_category"],
        "raw_filename": raw_filename,
        "feature_window_file": fw_name,
        "feature_window_path": str(fw_path),
        "exists": exists
    }

    if exists:
        try:
            df = pd.read_csv(fw_path, nrows=5)
            rec["preview_rows"] = len(df)
            rec["n_columns_preview"] = df.shape[1]
            rec["columns"] = list(df.columns)

            missing_features = [f for f in SELECTED_23_FEATURES if f not in df.columns]
            rec["missing_selected_features"] = missing_features
            rec["n_missing_selected_features"] = len(missing_features)

            schema_examples.append((fw_name, df.copy()))
        except Exception as e:
            rec["error"] = str(e)

    records.append(rec)

availability = pd.DataFrame(records)

print("\nFeature-window availability:")
display(
    availability[
        [
            "building", "meter", "primary_category",
            "feature_window_file",
            "exists", "n_columns_preview", "n_missing_selected_features",
            "missing_selected_features"
        ]
    ]
)

missing_files = availability[availability["exists"] == False]
if len(missing_files) > 0:
    print("\nWARNING: Some feature-window files are missing:")
    display(missing_files)

# Show schema examples
print("\nSchema examples:")
for fname, sample_df in schema_examples[:3]:
    print("\n" + "="*100)
    print(fname)
    print("Columns:", list(sample_df.columns))
    display(sample_df.head())

# ------------------------------------------------------------
# Load and combine feature-window files
# ------------------------------------------------------------

print("\nLoading and combining clean UK-DALE feature windows...")

for _, row in clean_meters.iterrows():
    raw_filename = row["filename"]
    fw_name = feature_window_filename(raw_filename)
    fw_path = FEATURE_DIR / fw_name

    if not fw_path.exists():
        print("Skipping missing:", fw_name)
        continue

    df = pd.read_csv(fw_path)

    # Ensure selected features exist
    missing_features = [f for f in SELECTED_23_FEATURES if f not in df.columns]
    if missing_features:
        print(f"Skipping {fw_name}; missing selected features:", missing_features)
        continue

    # Keep useful existing metadata columns if present
    possible_meta_cols = [
        "dataset_name", "series_id", "window_start", "window_end",
        "start_time", "end_time", "timestamp", "label"
    ]
    existing_meta_cols = [c for c in possible_meta_cols if c in df.columns]

    keep_cols = existing_meta_cols + SELECTED_23_FEATURES
    df = df[keep_cols].copy()

    # Add clean metadata
    df["dataset"] = "UK-DALE"
    df["building"] = int(row["building"])
    df["meter"] = int(row["meter"])
    df["primary_category"] = row["primary_category"]
    df["original_names"] = row["original_names"]
    df["appliance_types"] = row["appliance_types"]
    df["source_raw_file"] = raw_filename
    df["source_feature_window_file"] = fw_name

    # UK-DALE is naturally unlabeled here
    if "label" not in df.columns:
        df["label"] = -1

    df["is_synthetic"] = False

    parts.append(df)

    print(f"Loaded {fw_name}: {df.shape}")

if len(parts) == 0:
    raise RuntimeError("No feature-window files were loaded. Check FEATURE_DIR and filenames.")

combined = pd.concat(parts, ignore_index=True)

# Normalize label for original UK-DALE windows
combined["label"] = combined["label"].fillna(-1).astype(int)

# Save
combined_path = OUT_DIR / "ukdale_clean_main_60min_stride30_feature_windows.csv"
combined.to_csv(combined_path, index=False)

print("\nCombined UK-DALE clean feature-window dataset:")
print("Shape:", combined.shape)
print("Saved:", combined_path)

print("\nColumns:")
print(list(combined.columns))

print("\nWindow counts by building and appliance category:")
summary_building_category = (
    combined.groupby(["building", "primary_category"])
    .agg(
        n_windows=("label", "count"),
        labels=("label", lambda x: dict(pd.Series(x).value_counts().sort_index()))
    )
    .reset_index()
    .sort_values(["building", "primary_category"])
)
display(summary_building_category)

summary_category = (
    combined.groupby("primary_category")
    .agg(
        n_windows=("label", "count"),
        n_meters=("source_raw_file", "nunique")
    )
    .reset_index()
    .sort_values("primary_category")
)

print("\nWindow counts by appliance category:")
display(summary_category)

# Save summaries
summary_building_category_path = OUT_DIR / "ukdale_clean_main_window_summary_by_building_category.csv"
summary_category_path = OUT_DIR / "ukdale_clean_main_window_summary_by_category.csv"

summary_building_category.to_csv(summary_building_category_path, index=False)
summary_category.to_csv(summary_category_path, index=False)

print("\nSaved summaries:")
print(summary_building_category_path)
print(summary_category_path)

In [ ]:
# ============================================================
# EXP_UKDALE_Selected_Pipeline
# Cell 6B: Rebuild UK-DALE 60/30 windows with selected 23 features
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import warnings
warnings.filterwarnings("ignore")

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

OUT_DIR = BASE / "processed_data" / "UKDALE_Selected_Pipeline"
UKDALE_1MIN_DIR = BASE / "processed_data" / "PH2" / "ukdale_all_series"

CLEAN_METERS_PATH = OUT_DIR / "ukdale_clean_main_experiment_meters.csv"

WINDOW_SIZE = 60       # 60 minutes
STRIDE = 30            # 30 minutes

print("OUT_DIR:", OUT_DIR)
print("UKDALE_1MIN_DIR:", UKDALE_1MIN_DIR)
print("CLEAN_METERS_PATH:", CLEAN_METERS_PATH)
print("CLEAN_METERS_PATH exists:", CLEAN_METERS_PATH.exists())

clean_meters = pd.read_csv(CLEAN_METERS_PATH)

print("\nClean meters selected for UK-DALE main experiment:", clean_meters.shape)
display(
    clean_meters[
        [
            "building", "meter", "primary_category",
            "original_names", "appliance_types", "rows", "filename"
        ]
    ]
)

SELECTED_23_FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]


# ------------------------------------------------------------
# Vectorized 60/30 feature extraction
# ------------------------------------------------------------

def extract_selected_23_features(values, timestamps, window_size=60, stride=30):
    """
    Extracts selected 23 NAB-style window features from a 1-minute power series.
    Uses row-based 60-sample windows with 30-sample stride because the PH2 files
    are already resampled to 1-minute resolution.
    """

    values = pd.to_numeric(pd.Series(values), errors="coerce").astype(float).to_numpy()
    values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)

    n = len(values)

    if n < window_size:
        return pd.DataFrame()

    start_idx = np.arange(0, n - window_size + 1, stride)
    end_idx = start_idx + window_size - 1

    # Sliding windows: shape = (n_windows, 60)
    windows = sliding_window_view(values, window_shape=window_size)[::stride]

    # Basic statistics
    mean = windows.mean(axis=1)
    std = windows.std(axis=1, ddof=0)
    var = windows.var(axis=1, ddof=0)
    min_v = windows.min(axis=1)
    max_v = windows.max(axis=1)
    range_v = max_v - min_v
    sum_v = windows.sum(axis=1)
    energy = sum_v / 60.0   # Wh approximation from W readings at 1-minute resolution

    median = np.median(windows, axis=1)
    q25 = np.percentile(windows, 25, axis=1)
    q75 = np.percentile(windows, 75, axis=1)
    iqr = q75 - q25

    mad = np.median(np.abs(windows - median[:, None]), axis=1)
    rms = np.sqrt(np.mean(windows ** 2, axis=1))

    first_last_diff = windows[:, -1] - windows[:, 0]

    diffs = np.diff(windows, axis=1)
    abs_diffs = np.abs(diffs)

    mean_abs_diff = abs_diffs.mean(axis=1)
    max_abs_diff = abs_diffs.max(axis=1)
    std_diff = diffs.std(axis=1, ddof=0)

    # Local peak count
    if window_size >= 3:
        peak_count = (
            (windows[:, 1:-1] > windows[:, :-2]) &
            (windows[:, 1:-1] > windows[:, 2:])
        ).sum(axis=1)
    else:
        peak_count = np.zeros(windows.shape[0])

    # Skewness and excess kurtosis
    centered = windows - mean[:, None]

    with np.errstate(divide="ignore", invalid="ignore"):
        m3 = np.mean(centered ** 3, axis=1)
        m4 = np.mean(centered ** 4, axis=1)

        skew = np.where(std > 0, m3 / (std ** 3), 0.0)
        kurtosis = np.where(std > 0, (m4 / (std ** 4)) - 3.0, 0.0)

    skew = np.nan_to_num(skew, nan=0.0, posinf=0.0, neginf=0.0)
    kurtosis = np.nan_to_num(kurtosis, nan=0.0, posinf=0.0, neginf=0.0)

    # Linear slope over the window
    t = np.arange(window_size)
    t_centered = t - t.mean()
    denom = np.sum(t_centered ** 2)

    slope = ((windows - mean[:, None]) * t_centered[None, :]).sum(axis=1) / denom

    # Timestamp metadata
    timestamps = pd.to_datetime(pd.Series(timestamps), utc=True)
    start_times = timestamps.iloc[start_idx].to_numpy()
    end_times = timestamps.iloc[end_idx].to_numpy()

    duration_minutes = (
        (pd.to_datetime(end_times, utc=True) - pd.to_datetime(start_times, utc=True))
        / np.timedelta64(1, "m")
    ) + 1

    out = pd.DataFrame({
        "window_start": start_times,
        "window_end": end_times,
        "duration_minutes": duration_minutes,
        "window_size_minutes": window_size,
        "step_size_minutes": stride,

        "energy": energy,
        "first_last_diff": first_last_diff,
        "iqr": iqr,
        "kurtosis": kurtosis,
        "mad": mad,
        "max": max_v,
        "max_abs_diff": max_abs_diff,
        "mean": mean,
        "mean_abs_diff": mean_abs_diff,
        "median": median,
        "min": min_v,
        "n_points": window_size,
        "peak_count": peak_count,
        "q25": q25,
        "q75": q75,
        "range": range_v,
        "rms": rms,
        "skew": skew,
        "slope": slope,
        "std": std,
        "std_diff": std_diff,
        "sum": sum_v,
        "var": var
    })

    return out


# ------------------------------------------------------------
# Build windows for all clean UK-DALE appliance meters
# ------------------------------------------------------------

all_parts = []
summary_rows = []

for i, row in clean_meters.iterrows():
    filename = row["filename"]
    csv_path = Path(row["csv_path"])

    if not csv_path.exists():
        # fallback if csv_path in CSV is stale
        csv_path = UKDALE_1MIN_DIR / filename

    print("\n" + "="*100)
    print(f"[{i+1}/{len(clean_meters)}] Processing:", filename)
    print("Path:", csv_path)
    print("Exists:", csv_path.exists())

    if not csv_path.exists():
        print("Skipping missing file:", filename)
        summary_rows.append({
            "building": row["building"],
            "meter": row["meter"],
            "primary_category": row["primary_category"],
            "filename": filename,
            "status": "missing_csv",
            "input_rows": 0,
            "n_windows": 0
        })
        continue

    raw = pd.read_csv(csv_path, usecols=["index", "mean_power"])

    raw["index"] = pd.to_datetime(raw["index"], utc=True, errors="coerce")
    raw["mean_power"] = pd.to_numeric(raw["mean_power"], errors="coerce")

    raw = raw.dropna(subset=["index", "mean_power"])
    raw = raw.sort_values("index").drop_duplicates(subset=["index"]).reset_index(drop=True)

    print("Input rows after cleaning:", len(raw))

    features = extract_selected_23_features(
        values=raw["mean_power"].values,
        timestamps=raw["index"].values,
        window_size=WINDOW_SIZE,
        stride=STRIDE
    )

    if features.empty:
        print("No windows generated.")
        summary_rows.append({
            "building": row["building"],
            "meter": row["meter"],
            "primary_category": row["primary_category"],
            "filename": filename,
            "status": "too_short",
            "input_rows": len(raw),
            "n_windows": 0
        })
        continue

    # Add metadata
    features.insert(0, "dataset", "UK-DALE")
    features.insert(1, "building", int(row["building"]))
    features.insert(2, "meter", int(row["meter"]))
    features.insert(3, "primary_category", row["primary_category"])
    features.insert(4, "original_names", row["original_names"])
    features.insert(5, "appliance_types", row["appliance_types"])
    features.insert(6, "source_raw_file", filename)

    # Original UK-DALE windows are unlabeled
    features["label"] = -1
    features["is_synthetic"] = False

    all_parts.append(features)

    print("Generated windows:", features.shape)

    summary_rows.append({
        "building": row["building"],
        "meter": row["meter"],
        "primary_category": row["primary_category"],
        "filename": filename,
        "status": "success",
        "input_rows": len(raw),
        "n_windows": len(features),
        "duration_min": features["duration_minutes"].min(),
        "duration_median": features["duration_minutes"].median(),
        "duration_max": features["duration_minutes"].max()
    })

if len(all_parts) == 0:
    raise RuntimeError("No UK-DALE windows were generated. Check the clean meter list and PH2 files.")

combined = pd.concat(all_parts, ignore_index=True)
summary = pd.DataFrame(summary_rows)

# Ensure selected 23 features exist
missing_final = [f for f in SELECTED_23_FEATURES if f not in combined.columns]
print("\nMissing selected features in combined dataset:", missing_final)

# Save outputs
combined_path = OUT_DIR / "ukdale_clean_main_60min_stride30_selected23_windows.csv"
summary_path = OUT_DIR / "ukdale_clean_main_60min_stride30_selected23_summary.csv"

combined.to_csv(combined_path, index=False)
summary.to_csv(summary_path, index=False)

print("\n" + "="*100)
print("UK-DALE selected 23-feature window dataset created")
print("="*100)
print("Combined shape:", combined.shape)
print("Saved combined:", combined_path)
print("Saved summary:", summary_path)

print("\nGeneration summary:")
display(summary)

print("\nWindow counts by building and appliance category:")
summary_building_category = (
    combined.groupby(["building", "primary_category"])
    .agg(
        n_windows=("label", "count"),
        n_source_files=("source_raw_file", "nunique"),
        duration_median=("duration_minutes", "median"),
        duration_max=("duration_minutes", "max")
    )
    .reset_index()
    .sort_values(["building", "primary_category"])
)
display(summary_building_category)

print("\nWindow counts by appliance category:")
summary_category = (
    combined.groupby("primary_category")
    .agg(
        n_windows=("label", "count"),
        n_meters=("source_raw_file", "nunique"),
        duration_median=("duration_minutes", "median"),
        duration_max=("duration_minutes", "max")
    )
    .reset_index()
    .sort_values("primary_category")
)
display(summary_category)

# Save summaries
summary_building_category_path = OUT_DIR / "ukdale_selected23_window_summary_by_building_category.csv"
summary_category_path = OUT_DIR / "ukdale_selected23_window_summary_by_category.csv"

summary_building_category.to_csv(summary_building_category_path, index=False)
summary_category.to_csv(summary_category_path, index=False)

print("\nSaved extra summaries:")
print(summary_building_category_path)
print(summary_category_path)

print("\nCombined dataset preview:")
display(combined.head())

In [ ]:
# ============================================================
# EXP_UKDALE_Selected_Pipeline
# Cell 7: Validate and filter continuous 60-minute UK-DALE windows
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

OUT_DIR = BASE / "processed_data" / "UKDALE_Selected_Pipeline"

WINDOWS_PATH = OUT_DIR / "ukdale_clean_main_60min_stride30_selected23_windows.csv"

print("WINDOWS_PATH:", WINDOWS_PATH)
print("Exists:", WINDOWS_PATH.exists())

df = pd.read_csv(WINDOWS_PATH)

print("\nOriginal combined shape:", df.shape)

# Convert timestamps safely
df["window_start"] = pd.to_datetime(df["window_start"], utc=True, errors="coerce")
df["window_end"] = pd.to_datetime(df["window_end"], utc=True, errors="coerce")

# Recompute duration to be safe
df["duration_minutes_recomputed"] = (
    (df["window_end"] - df["window_start"]) / np.timedelta64(1, "m")
) + 1

print("\nDuration statistics before filtering:")
display(df["duration_minutes_recomputed"].describe())

print("\nDuration value counts for common durations:")
display(df["duration_minutes_recomputed"].value_counts().sort_index().head(20))

# Valid windows should represent exactly 60 continuous minutes.
# Allow tiny floating tolerance.
valid_mask = df["duration_minutes_recomputed"].between(59.9, 60.1)

valid = df[valid_mask].copy()
removed = df[~valid_mask].copy()

print("\nValid continuous windows:", valid.shape)
print("Removed gap-crossing windows:", removed.shape)
print("Removed percentage:", round(len(removed) / len(df) * 100, 3), "%")

print("\nRemoved duration statistics:")
if len(removed) > 0:
    display(removed["duration_minutes_recomputed"].describe())
else:
    print("No invalid windows removed.")

# Replace duration_minutes with validated recomputed duration
valid["duration_minutes"] = valid["duration_minutes_recomputed"]
valid = valid.drop(columns=["duration_minutes_recomputed"])

# Save valid filtered dataset
VALID_PATH = OUT_DIR / "ukdale_clean_main_60min_stride30_selected23_valid_windows.csv"
REMOVED_PATH = OUT_DIR / "ukdale_removed_gap_crossing_windows_summary.csv"

valid.to_csv(VALID_PATH, index=False)

# Summaries
before_summary = (
    df.groupby(["building", "primary_category", "source_raw_file"])
    .agg(before_windows=("label", "count"))
    .reset_index()
)

after_summary = (
    valid.groupby(["building", "primary_category", "source_raw_file"])
    .agg(after_windows=("label", "count"))
    .reset_index()
)

quality_summary = before_summary.merge(
    after_summary,
    on=["building", "primary_category", "source_raw_file"],
    how="left"
)

quality_summary["after_windows"] = quality_summary["after_windows"].fillna(0).astype(int)
quality_summary["removed_windows"] = quality_summary["before_windows"] - quality_summary["after_windows"]
quality_summary["removed_percent"] = (
    quality_summary["removed_windows"] / quality_summary["before_windows"] * 100
).round(3)

quality_summary.to_csv(REMOVED_PATH, index=False)

print("\nSaved valid continuous-window dataset:")
print(VALID_PATH)

print("\nSaved quality summary:")
print(REMOVED_PATH)

print("\nQuality summary by source file:")
display(quality_summary.sort_values(["building", "primary_category", "source_raw_file"]))

print("\nFinal valid window counts by building and appliance category:")
summary_building_category = (
    valid.groupby(["building", "primary_category"])
    .agg(
        n_windows=("label", "count"),
        n_source_files=("source_raw_file", "nunique"),
        duration_median=("duration_minutes", "median"),
        duration_max=("duration_minutes", "max")
    )
    .reset_index()
    .sort_values(["building", "primary_category"])
)
display(summary_building_category)

print("\nFinal valid window counts by appliance category:")
summary_category = (
    valid.groupby("primary_category")
    .agg(
        n_windows=("label", "count"),
        n_meters=("source_raw_file", "nunique"),
        duration_median=("duration_minutes", "median"),
        duration_max=("duration_minutes", "max")
    )
    .reset_index()
    .sort_values("primary_category")
)
display(summary_category)

# Save final summaries
summary_building_category_path = OUT_DIR / "ukdale_valid_window_summary_by_building_category.csv"
summary_category_path = OUT_DIR / "ukdale_valid_window_summary_by_category.csv"

summary_building_category.to_csv(summary_building_category_path, index=False)
summary_category.to_csv(summary_category_path, index=False)

print("\nSaved final summaries:")
print(summary_building_category_path)
print(summary_category_path)

print("\nFinal dataset preview:")
display(valid.head())

In [ ]:
# ============================================================
# EXP_UKDALE_Selected_Pipeline
# Cell 8: UK-DALE real-data Isolation Forest scoring
# No synthetic injection
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

OUT_DIR = BASE / "processed_data" / "UKDALE_Selected_Pipeline"
EXP_DIR = BASE / "experiments" / "EXP_UKDALE_Selected_Pipeline"

EXP_DIR.mkdir(parents=True, exist_ok=True)

VALID_WINDOWS_PATH = OUT_DIR / "ukdale_clean_main_60min_stride30_selected23_valid_windows.csv"

print("VALID_WINDOWS_PATH:", VALID_WINDOWS_PATH)
print("Exists:", VALID_WINDOWS_PATH.exists())

df = pd.read_csv(VALID_WINDOWS_PATH)

print("\nLoaded UK-DALE valid windows:", df.shape)

# ------------------------------------------------------------
# Selected 23 features from final NAB representation
# ------------------------------------------------------------

FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

missing = [f for f in FEATURES if f not in df.columns]
print("Missing features:", missing)

if missing:
    raise RuntimeError(f"Missing selected features: {missing}")

# ------------------------------------------------------------
# Clean numerical feature values
# ------------------------------------------------------------

for f in FEATURES:
    df[f] = pd.to_numeric(df[f], errors="coerce")

df[FEATURES] = df[FEATURES].replace([np.inf, -np.inf], np.nan)
df[FEATURES] = df[FEATURES].fillna(0.0)

df["window_start"] = pd.to_datetime(df["window_start"], utc=True, errors="coerce")
df["window_end"] = pd.to_datetime(df["window_end"], utc=True, errors="coerce")

# Sort chronologically per meter
df = df.sort_values(["building", "meter", "window_start"]).reset_index(drop=True)

# Unique physical series id
df["series_key"] = (
    "building" + df["building"].astype(str) +
    "_meter" + df["meter"].astype(str) +
    "_" + df["primary_category"].astype(str)
)

print("\nSeries included:")
display(
    df.groupby(["series_key", "building", "meter", "primary_category"])
    .agg(n_windows=("label", "count"))
    .reset_index()
    .sort_values(["building", "meter"])
)

# ------------------------------------------------------------
# Chronological split per meter: 60/20/20
# ------------------------------------------------------------

split_parts = []

for series_key, g in df.groupby("series_key", sort=False):
    g = g.sort_values("window_start").copy()
    n = len(g)

    train_end = int(n * 0.60)
    val_end = int(n * 0.80)

    split = np.array(["test"] * n, dtype=object)
    split[:train_end] = "train"
    split[train_end:val_end] = "val"

    g["split"] = split
    split_parts.append(g)

df_split = pd.concat(split_parts, ignore_index=True)

print("\nSplit summary:")
display(
    df_split.groupby(["split"])
    .agg(n_windows=("label", "count"))
    .reset_index()
)

print("\nSplit summary by category:")
display(
    df_split.groupby(["primary_category", "split"])
    .agg(n_windows=("label", "count"))
    .reset_index()
    .sort_values(["primary_category", "split"])
)

# ------------------------------------------------------------
# Per-meter z-score normalization using TRAIN statistics only
# ------------------------------------------------------------

scaled_parts = []

for series_key, g in df_split.groupby("series_key", sort=False):
    g = g.copy()

    train_mask = g["split"] == "train"

    scaler = StandardScaler()
    scaler.fit(g.loc[train_mask, FEATURES])

    X_scaled = scaler.transform(g[FEATURES])

    scaled_cols = [f"{f}_z" for f in FEATURES]
    scaled_df = pd.DataFrame(X_scaled, columns=scaled_cols, index=g.index)

    g = pd.concat([g, scaled_df], axis=1)

    scaled_parts.append(g)

df_scaled = pd.concat(scaled_parts, ignore_index=True)

SCALED_FEATURES = [f"{f}_z" for f in FEATURES]

print("\nScaled dataset shape:", df_scaled.shape)
print("Scaled features:", len(SCALED_FEATURES))

# ------------------------------------------------------------
# Train one Isolation Forest model on all TRAIN windows
# This follows the selected NAB model choice but applies it to UK-DALE.
# ------------------------------------------------------------

train_df = df_scaled[df_scaled["split"] == "train"].copy()
val_df = df_scaled[df_scaled["split"] == "val"].copy()
test_df = df_scaled[df_scaled["split"] == "test"].copy()

print("\nTrain/Val/Test shapes:")
print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

X_train = train_df[SCALED_FEATURES].values
X_all = df_scaled[SCALED_FEATURES].values

if_model = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

print("\nTraining Isolation Forest on original UK-DALE train windows only...")
if_model.fit(X_train)

# In sklearn, lower decision_function = more anomalous.
# Convert to anomaly score where higher = more anomalous.
df_scaled["if_decision_function"] = if_model.decision_function(X_all)
df_scaled["anomaly_score"] = -df_scaled["if_decision_function"]

# Threshold from validation score distribution.
# Since no labels exist, this is an operating threshold, not an evaluated optimum.
val_scores = df_scaled.loc[df_scaled["split"] == "val", "anomaly_score"]

THRESHOLD_PERCENTILE = 95
threshold = np.percentile(val_scores, THRESHOLD_PERCENTILE)

df_scaled["if_threshold_percentile"] = THRESHOLD_PERCENTILE
df_scaled["if_threshold"] = threshold
df_scaled["predicted_anomaly"] = (df_scaled["anomaly_score"] >= threshold).astype(int)

print("\nIsolation Forest scoring complete.")
print("Validation percentile threshold:", THRESHOLD_PERCENTILE)
print("Threshold value:", threshold)

# ------------------------------------------------------------
# Summaries
# ------------------------------------------------------------

overall_summary = (
    df_scaled.groupby("split")
    .agg(
        n_windows=("predicted_anomaly", "count"),
        flagged_anomalies=("predicted_anomaly", "sum"),
        flagged_ratio=("predicted_anomaly", "mean"),
        score_mean=("anomaly_score", "mean"),
        score_std=("anomaly_score", "std"),
        score_min=("anomaly_score", "min"),
        score_median=("anomaly_score", "median"),
        score_max=("anomaly_score", "max")
    )
    .reset_index()
)

overall_summary["flagged_ratio"] = overall_summary["flagged_ratio"].round(4)

category_summary = (
    df_scaled.groupby(["primary_category", "split"])
    .agg(
        n_windows=("predicted_anomaly", "count"),
        flagged_anomalies=("predicted_anomaly", "sum"),
        flagged_ratio=("predicted_anomaly", "mean"),
        score_mean=("anomaly_score", "mean"),
        score_median=("anomaly_score", "median"),
        score_max=("anomaly_score", "max")
    )
    .reset_index()
    .sort_values(["primary_category", "split"])
)

category_summary["flagged_ratio"] = category_summary["flagged_ratio"].round(4)

series_summary = (
    df_scaled.groupby(["series_key", "building", "meter", "primary_category", "split"])
    .agg(
        n_windows=("predicted_anomaly", "count"),
        flagged_anomalies=("predicted_anomaly", "sum"),
        flagged_ratio=("predicted_anomaly", "mean"),
        score_mean=("anomaly_score", "mean"),
        score_median=("anomaly_score", "median"),
        score_max=("anomaly_score", "max")
    )
    .reset_index()
    .sort_values(["building", "meter", "split"])
)

series_summary["flagged_ratio"] = series_summary["flagged_ratio"].round(4)

print("\nOverall scoring summary:")
display(overall_summary)

print("\nCategory-level scoring summary:")
display(category_summary)

print("\nSeries-level scoring summary:")
display(series_summary)

# Top anomalous windows, mostly for inspection and later XAI examples
top_anomalies = (
    df_scaled
    .sort_values("anomaly_score", ascending=False)
    .head(1000)
    .copy()
)

top_cols = [
    "dataset", "building", "meter", "primary_category",
    "original_names", "appliance_types", "source_raw_file",
    "window_start", "window_end", "split",
    "anomaly_score", "if_threshold", "predicted_anomaly"
] + FEATURES

print("\nTop anomalous windows preview:")
display(top_anomalies[top_cols].head(30))

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

scored_path = OUT_DIR / "ukdale_clean_main_if_scored_no_injection.csv"
overall_summary_path = EXP_DIR / "ukdale_if_no_injection_overall_summary.csv"
category_summary_path = EXP_DIR / "ukdale_if_no_injection_category_summary.csv"
series_summary_path = EXP_DIR / "ukdale_if_no_injection_series_summary.csv"
top_anomalies_path = EXP_DIR / "ukdale_if_no_injection_top_anomalous_windows.csv"

df_scaled.to_csv(scored_path, index=False)
overall_summary.to_csv(overall_summary_path, index=False)
category_summary.to_csv(category_summary_path, index=False)
series_summary.to_csv(series_summary_path, index=False)
top_anomalies[top_cols].to_csv(top_anomalies_path, index=False)

print("\nSaved outputs:")
print(scored_path)
print(overall_summary_path)
print(category_summary_path)
print(series_summary_path)
print(top_anomalies_path)

In [ ]:
# ============================================================
# EXP_UKDALE_Selected_Pipeline
# Cell 9: UK-DALE no-injection result summaries and XAI candidates
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

OUT_DIR = BASE / "processed_data" / "UKDALE_Selected_Pipeline"
EXP_DIR = BASE / "experiments" / "EXP_UKDALE_Selected_Pipeline"
FIG_DIR = EXP_DIR / "figures"
TABLE_DIR = EXP_DIR / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

SCORED_PATH = OUT_DIR / "ukdale_clean_main_if_scored_no_injection.csv"

print("SCORED_PATH:", SCORED_PATH)
print("Exists:", SCORED_PATH.exists())

df = pd.read_csv(SCORED_PATH)

print("\nLoaded scored UK-DALE dataset:", df.shape)

df["window_start"] = pd.to_datetime(df["window_start"], utc=True, errors="coerce")
df["window_end"] = pd.to_datetime(df["window_end"], utc=True, errors="coerce")

# ------------------------------------------------------------
# 1. Overall summary table
# ------------------------------------------------------------

overall_summary = (
    df.groupby("split")
    .agg(
        n_windows=("predicted_anomaly", "count"),
        flagged_windows=("predicted_anomaly", "sum"),
        flagged_ratio=("predicted_anomaly", "mean"),
        score_mean=("anomaly_score", "mean"),
        score_median=("anomaly_score", "median"),
        score_max=("anomaly_score", "max")
    )
    .reset_index()
)

overall_summary["flagged_ratio_percent"] = (overall_summary["flagged_ratio"] * 100).round(2)

overall_path = TABLE_DIR / "ukdale_no_injection_overall_summary.csv"
overall_summary.to_csv(overall_path, index=False)

print("\nOverall summary:")
display(overall_summary)

# ------------------------------------------------------------
# 2. Category summary table
# ------------------------------------------------------------

category_summary = (
    df.groupby(["primary_category", "split"])
    .agg(
        n_windows=("predicted_anomaly", "count"),
        flagged_windows=("predicted_anomaly", "sum"),
        flagged_ratio=("predicted_anomaly", "mean"),
        score_mean=("anomaly_score", "mean"),
        score_median=("anomaly_score", "median"),
        score_max=("anomaly_score", "max")
    )
    .reset_index()
    .sort_values(["primary_category", "split"])
)

category_summary["flagged_ratio_percent"] = (category_summary["flagged_ratio"] * 100).round(2)

category_path = TABLE_DIR / "ukdale_no_injection_category_summary.csv"
category_summary.to_csv(category_path, index=False)

print("\nCategory summary:")
display(category_summary)

# ------------------------------------------------------------
# 3. Test-only category summary
# ------------------------------------------------------------

test_category = category_summary[category_summary["split"] == "test"].copy()
test_category = test_category.sort_values("flagged_ratio", ascending=False)

test_category_path = TABLE_DIR / "ukdale_no_injection_test_category_summary.csv"
test_category.to_csv(test_category_path, index=False)

print("\nTest category summary:")
display(test_category)

# ------------------------------------------------------------
# 4. XAI candidate windows
#    Select top anomalous windows from TEST split:
#    - top 5 overall
#    - top 3 per appliance category
# ------------------------------------------------------------

test_df = df[df["split"] == "test"].copy()

top_overall = (
    test_df.sort_values("anomaly_score", ascending=False)
    .head(20)
    .copy()
)

top_by_category = (
    test_df.sort_values("anomaly_score", ascending=False)
    .groupby("primary_category", group_keys=False)
    .head(5)
    .copy()
    .sort_values(["primary_category", "anomaly_score"], ascending=[True, False])
)

xai_cols = [
    "dataset", "building", "meter", "primary_category",
    "original_names", "appliance_types", "source_raw_file",
    "window_start", "window_end", "split",
    "anomaly_score", "if_threshold", "predicted_anomaly"
]

top_overall_path = TABLE_DIR / "ukdale_xai_candidate_top_overall_test_windows.csv"
top_by_category_path = TABLE_DIR / "ukdale_xai_candidate_top_by_category_test_windows.csv"

top_overall[xai_cols].to_csv(top_overall_path, index=False)
top_by_category[xai_cols].to_csv(top_by_category_path, index=False)

print("\nTop overall TEST anomalous windows:")
display(top_overall[xai_cols].head(20))

print("\nTop TEST anomalous windows by category:")
display(top_by_category[xai_cols])

# ------------------------------------------------------------
# 5. Figure: overall flagged ratio by split
# ------------------------------------------------------------

fig1_path = FIG_DIR / "ukdale_if_no_injection_flagged_ratio_by_split.png"

plot_df = overall_summary.copy()
plot_df = plot_df.set_index("split").loc[["train", "val", "test"]].reset_index()

plt.figure(figsize=(7, 4))
plt.bar(plot_df["split"], plot_df["flagged_ratio_percent"])
plt.xlabel("Data split")
plt.ylabel("Flagged windows (%)")
plt.title("UK-DALE Isolation Forest Flagged Ratio by Split")
plt.tight_layout()
plt.savefig(fig1_path, dpi=300)
plt.show()

# ------------------------------------------------------------
# 6. Figure: test flagged ratio by appliance category
# ------------------------------------------------------------

fig2_path = FIG_DIR / "ukdale_if_no_injection_test_flagged_ratio_by_category.png"

plt.figure(figsize=(9, 4.5))
plt.bar(test_category["primary_category"], test_category["flagged_ratio_percent"])
plt.xlabel("Appliance category")
plt.ylabel("Flagged test windows (%)")
plt.title("UK-DALE Test Flagged Ratio by Appliance Category")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(fig2_path, dpi=300)
plt.show()

# ------------------------------------------------------------
# 7. Figure: anomaly score boxplot by category for TEST split
# ------------------------------------------------------------

fig3_path = FIG_DIR / "ukdale_if_no_injection_test_score_distribution_by_category.png"

categories = sorted(test_df["primary_category"].unique())
score_data = [
    test_df[test_df["primary_category"] == cat]["anomaly_score"].values
    for cat in categories
]

plt.figure(figsize=(9, 4.5))
plt.boxplot(score_data, labels=categories, showfliers=False)
plt.xlabel("Appliance category")
plt.ylabel("Anomaly score")
plt.title("UK-DALE Test Anomaly Score Distribution by Appliance Category")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(fig3_path, dpi=300)
plt.show()

print("\nSaved tables:")
print(overall_path)
print(category_path)
print(test_category_path)
print(top_overall_path)
print(top_by_category_path)

print("\nSaved figures:")
print(fig1_path)
print(fig2_path)
print(fig3_path)

In [ ]:
# ============================================================
# EXP_UKDALE_Selected_Pipeline
# Cell 10: SHAP + LIME explanations for selected UK-DALE flagged windows
# No synthetic injection
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Install SHAP and LIME if needed
try:
    import shap
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap"])
    import shap

try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lime"])
    from lime.lime_tabular import LimeTabularExplainer

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

OUT_DIR = BASE / "processed_data" / "UKDALE_Selected_Pipeline"
EXP_DIR = BASE / "experiments" / "EXP_UKDALE_Selected_Pipeline"
XAI_DIR = EXP_DIR / "xai"
TABLE_DIR = EXP_DIR / "tables"

XAI_DIR.mkdir(parents=True, exist_ok=True)

SCORED_PATH = OUT_DIR / "ukdale_clean_main_if_scored_no_injection.csv"
CANDIDATE_PATH = TABLE_DIR / "ukdale_xai_candidate_top_by_category_test_windows.csv"

print("SCORED_PATH:", SCORED_PATH)
print("Exists:", SCORED_PATH.exists())
print("CANDIDATE_PATH:", CANDIDATE_PATH)
print("Exists:", CANDIDATE_PATH.exists())

df = pd.read_csv(SCORED_PATH)
candidates = pd.read_csv(CANDIDATE_PATH)

df["window_start"] = pd.to_datetime(df["window_start"], utc=True, errors="coerce")
df["window_end"] = pd.to_datetime(df["window_end"], utc=True, errors="coerce")

candidates["window_start"] = pd.to_datetime(candidates["window_start"], utc=True, errors="coerce")
candidates["window_end"] = pd.to_datetime(candidates["window_end"], utc=True, errors="coerce")

print("\nLoaded scored dataset:", df.shape)
print("Loaded XAI candidates:", candidates.shape)

# ------------------------------------------------------------
# Selected 23 final representation features
# ------------------------------------------------------------

FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

missing = [f for f in FEATURES if f not in df.columns]
print("Missing features:", missing)
if missing:
    raise RuntimeError(f"Missing features: {missing}")

# ------------------------------------------------------------
# Select one top TEST flagged window per appliance category
# ------------------------------------------------------------

selected_candidates = (
    candidates[candidates["predicted_anomaly"] == 1]
    .sort_values(["primary_category", "anomaly_score"], ascending=[True, False])
    .groupby("primary_category", group_keys=False)
    .head(1)
    .copy()
    .reset_index(drop=True)
)

print("\nSelected one XAI candidate per appliance category:")
display(selected_candidates[
    [
        "building", "meter", "primary_category",
        "window_start", "window_end",
        "anomaly_score", "if_threshold"
    ]
])

# ------------------------------------------------------------
# Rebuild series_key and prepare train/test data
# ------------------------------------------------------------

df["series_key"] = (
    "building" + df["building"].astype(str) +
    "_meter" + df["meter"].astype(str) +
    "_" + df["primary_category"].astype(str)
)

# Use only original, no-injection data
for f in FEATURES:
    df[f] = pd.to_numeric(df[f], errors="coerce")

df[FEATURES] = df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0)

# The scored file already contains split column from previous stage
if "split" not in df.columns:
    raise RuntimeError("Column 'split' not found. Re-run Cell 8 first.")

# ------------------------------------------------------------
# Per-meter z-score normalization using TRAIN statistics only
# Recreate the exact modeling setup used in Cell 8
# ------------------------------------------------------------

scaled_parts = []

for series_key, g in df.groupby("series_key", sort=False):
    g = g.copy()
    train_mask = g["split"] == "train"

    scaler = StandardScaler()
    scaler.fit(g.loc[train_mask, FEATURES])

    X_scaled = scaler.transform(g[FEATURES])
    scaled_cols = [f"{f}_z" for f in FEATURES]

    scaled_df = pd.DataFrame(X_scaled, columns=scaled_cols, index=g.index)
    g = pd.concat([g, scaled_df], axis=1)

    scaled_parts.append(g)

df_scaled = pd.concat(scaled_parts, ignore_index=True)

SCALED_FEATURES = [f"{f}_z" for f in FEATURES]

train_df = df_scaled[df_scaled["split"] == "train"].copy()
X_train = train_df[SCALED_FEATURES].values

print("\nTrain shape for XAI model rebuild:", X_train.shape)

# ------------------------------------------------------------
# Re-train Isolation Forest using the same setup as Cell 8
# ------------------------------------------------------------

if_model = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

print("\nRe-training Isolation Forest for explanation consistency...")
if_model.fit(X_train)

def anomaly_score_fn(X):
    """
    Higher output = more anomalous.
    Returns a 1D anomaly-score array.
    """
    return -if_model.decision_function(np.asarray(X))

def anomaly_score_fn_2d(X):
    """
    LIME expects 2D class-like output.
    We convert anomaly score into two columns:
    [normal_score, anomaly_score]
    This is not probability; it is only for local explanation.
    """
    scores = anomaly_score_fn(X)
    return np.column_stack([-scores, scores])

# ------------------------------------------------------------
# Match selected candidates back to full scaled dataframe
# ------------------------------------------------------------

def find_matching_row(candidate, full_df):
    mask = (
        (full_df["building"] == candidate["building"]) &
        (full_df["meter"] == candidate["meter"]) &
        (full_df["primary_category"] == candidate["primary_category"]) &
        (full_df["window_start"] == candidate["window_start"]) &
        (full_df["window_end"] == candidate["window_end"]) &
        (full_df["split"] == "test")
    )

    matched = full_df[mask].copy()

    if matched.empty:
        raise RuntimeError(
            f"No match found for candidate: "
            f"{candidate['primary_category']} "
            f"{candidate['window_start']} - {candidate['window_end']}"
        )

    return matched.iloc[0]

selected_rows = []

for _, cand in selected_candidates.iterrows():
    selected_rows.append(find_matching_row(cand, df_scaled))

selected_xai_df = pd.DataFrame(selected_rows).reset_index(drop=True)

print("\nMatched selected XAI rows:")
display(selected_xai_df[
    [
        "building", "meter", "primary_category",
        "window_start", "window_end",
        "anomaly_score", "predicted_anomaly"
    ]
])

# ------------------------------------------------------------
# SHAP KernelExplainer
# Use a small background sample from training to keep runtime reasonable.
# ------------------------------------------------------------

rng = np.random.default_rng(42)

BACKGROUND_SIZE = 100
background_idx = rng.choice(len(X_train), size=min(BACKGROUND_SIZE, len(X_train)), replace=False)
X_background = X_train[background_idx]

X_explain = selected_xai_df[SCALED_FEATURES].values

print("\nRunning SHAP KernelExplainer...")
print("Background shape:", X_background.shape)
print("Explain shape:", X_explain.shape)

shap_explainer = shap.KernelExplainer(anomaly_score_fn, X_background)

# nsamples controls approximation quality/runtime
shap_values = shap_explainer.shap_values(X_explain, nsamples=300)

shap_values = np.asarray(shap_values)

if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 0]

print("SHAP values shape:", shap_values.shape)

# Save SHAP explanations in long format
shap_records = []

for i, row in selected_xai_df.iterrows():
    vals = shap_values[i]

    temp = pd.DataFrame({
        "feature_scaled": SCALED_FEATURES,
        "feature": FEATURES,
        "shap_value": vals,
        "abs_shap_value": np.abs(vals),
        "feature_value_scaled": X_explain[i],
        "feature_value_original": row[FEATURES].values.astype(float)
    })

    temp["dataset"] = "UK-DALE"
    temp["building"] = row["building"]
    temp["meter"] = row["meter"]
    temp["primary_category"] = row["primary_category"]
    temp["window_start"] = row["window_start"]
    temp["window_end"] = row["window_end"]
    temp["anomaly_score"] = row["anomaly_score"]

    temp = temp.sort_values("abs_shap_value", ascending=False)

    shap_records.append(temp)

shap_long = pd.concat(shap_records, ignore_index=True)

shap_path = XAI_DIR / "ukdale_shap_explanations_selected_test_windows.csv"
shap_long.to_csv(shap_path, index=False)

print("\nTop SHAP features per selected window:")
display(
    shap_long.groupby("primary_category", group_keys=False)
    .head(8)[
        [
            "primary_category", "building", "meter",
            "window_start", "anomaly_score",
            "feature", "feature_value_original",
            "shap_value", "abs_shap_value"
        ]
    ]
)

print("\nSaved SHAP explanations:")
print(shap_path)

# ------------------------------------------------------------
# LIME explanations
# ------------------------------------------------------------

print("\nRunning LIME explanations...")

lime_explainer = LimeTabularExplainer(
    training_data=X_train,
    feature_names=SCALED_FEATURES,
    class_names=["normal_score", "anomaly_score"],
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

lime_records = []

for i, row in selected_xai_df.iterrows():
    x = X_explain[i]

    explanation = lime_explainer.explain_instance(
        data_row=x,
        predict_fn=anomaly_score_fn_2d,
        num_features=10,
        labels=[1]
    )

    lime_list = explanation.as_list(label=1)

    for rank, (condition, weight) in enumerate(lime_list, start=1):
        lime_records.append({
            "dataset": "UK-DALE",
            "building": row["building"],
            "meter": row["meter"],
            "primary_category": row["primary_category"],
            "window_start": row["window_start"],
            "window_end": row["window_end"],
            "anomaly_score": row["anomaly_score"],
            "rank": rank,
            "lime_condition": condition,
            "lime_weight": weight,
            "abs_lime_weight": abs(weight)
        })

lime_df = pd.DataFrame(lime_records)

lime_path = XAI_DIR / "ukdale_lime_explanations_selected_test_windows.csv"
lime_df.to_csv(lime_path, index=False)

print("\nTop LIME features per selected window:")
display(lime_df)

print("\nSaved LIME explanations:")
print(lime_path)

# ------------------------------------------------------------
# Compact comparison table: top 5 SHAP + top 5 LIME per category
# ------------------------------------------------------------

shap_top5 = (
    shap_long.groupby("primary_category", group_keys=False)
    .head(5)
    .copy()
)

lime_top5 = (
    lime_df.sort_values(["primary_category", "rank"])
    .groupby("primary_category", group_keys=False)
    .head(5)
    .copy()
)

shap_top5_path = XAI_DIR / "ukdale_shap_top5_per_category.csv"
lime_top5_path = XAI_DIR / "ukdale_lime_top5_per_category.csv"

shap_top5.to_csv(shap_top5_path, index=False)
lime_top5.to_csv(lime_top5_path, index=False)

print("\nSaved compact XAI tables:")
print(shap_top5_path)
print(lime_top5_path)

print("\nXAI step completed successfully.")

In [ ]:
# ============================================================
# EXP_UKDALE_Selected_Pipeline
# Cell 10B: Fixed SHAP + LIME explanations for UK-DALE flagged windows
# No synthetic injection
# Fixes duplicate *_z columns issue
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Install SHAP and LIME if needed
try:
    import shap
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap"])
    import shap

try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lime"])
    from lime.lime_tabular import LimeTabularExplainer

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

OUT_DIR = BASE / "processed_data" / "UKDALE_Selected_Pipeline"
EXP_DIR = BASE / "experiments" / "EXP_UKDALE_Selected_Pipeline"
XAI_DIR = EXP_DIR / "xai"
TABLE_DIR = EXP_DIR / "tables"

XAI_DIR.mkdir(parents=True, exist_ok=True)

SCORED_PATH = OUT_DIR / "ukdale_clean_main_if_scored_no_injection.csv"
CANDIDATE_PATH = TABLE_DIR / "ukdale_xai_candidate_top_by_category_test_windows.csv"

print("SCORED_PATH:", SCORED_PATH)
print("Exists:", SCORED_PATH.exists())
print("CANDIDATE_PATH:", CANDIDATE_PATH)
print("Exists:", CANDIDATE_PATH.exists())

df = pd.read_csv(SCORED_PATH)
candidates = pd.read_csv(CANDIDATE_PATH)

df["window_start"] = pd.to_datetime(df["window_start"], utc=True, errors="coerce")
df["window_end"] = pd.to_datetime(df["window_end"], utc=True, errors="coerce")

candidates["window_start"] = pd.to_datetime(candidates["window_start"], utc=True, errors="coerce")
candidates["window_end"] = pd.to_datetime(candidates["window_end"], utc=True, errors="coerce")

print("\nLoaded scored dataset:", df.shape)
print("Loaded XAI candidates:", candidates.shape)

# ------------------------------------------------------------
# Selected 23 final representation features
# ------------------------------------------------------------

FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

missing = [f for f in FEATURES if f not in df.columns]
print("Missing features:", missing)

if missing:
    raise RuntimeError(f"Missing features: {missing}")

# ------------------------------------------------------------
# Clean duplicate / old scaled columns
# ------------------------------------------------------------

# Drop any previously saved *_z columns to avoid duplicate names
old_scaled_cols = [c for c in df.columns if c.endswith("_z")]
print("Existing scaled columns to drop:", len(old_scaled_cols))

df = df.drop(columns=old_scaled_cols, errors="ignore")

# Remove duplicate column names if any remain
df = df.loc[:, ~df.columns.duplicated()].copy()

print("Dataset shape after removing duplicate/scaled columns:", df.shape)

# ------------------------------------------------------------
# Select one top TEST flagged window per appliance category
# ------------------------------------------------------------

selected_candidates = (
    candidates[candidates["predicted_anomaly"] == 1]
    .sort_values(["primary_category", "anomaly_score"], ascending=[True, False])
    .groupby("primary_category", group_keys=False)
    .head(1)
    .copy()
    .reset_index(drop=True)
)

print("\nSelected one XAI candidate per appliance category:")
display(selected_candidates[
    [
        "building", "meter", "primary_category",
        "window_start", "window_end",
        "anomaly_score", "if_threshold"
    ]
])

# ------------------------------------------------------------
# Prepare modeling data
# ------------------------------------------------------------

df["series_key"] = (
    "building" + df["building"].astype(str) +
    "_meter" + df["meter"].astype(str) +
    "_" + df["primary_category"].astype(str)
)

for f in FEATURES:
    df[f] = pd.to_numeric(df[f], errors="coerce")

df[FEATURES] = df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0)

if "split" not in df.columns:
    raise RuntimeError("Column 'split' not found. Re-run Cell 8 first.")

# ------------------------------------------------------------
# Per-meter z-score normalization using TRAIN statistics only
# ------------------------------------------------------------

scaled_parts = []

for series_key, g in df.groupby("series_key", sort=False):
    g = g.copy()
    train_mask = g["split"] == "train"

    scaler = StandardScaler()
    scaler.fit(g.loc[train_mask, FEATURES])

    X_scaled = scaler.transform(g[FEATURES])
    scaled_cols = [f"{f}_z" for f in FEATURES]

    scaled_df = pd.DataFrame(X_scaled, columns=scaled_cols, index=g.index)
    g = pd.concat([g, scaled_df], axis=1)

    scaled_parts.append(g)

df_scaled = pd.concat(scaled_parts, ignore_index=True)

# Remove duplicate column names again, just in case
df_scaled = df_scaled.loc[:, ~df_scaled.columns.duplicated()].copy()

SCALED_FEATURES = [f"{f}_z" for f in FEATURES]

print("\nScaled feature count:", len(SCALED_FEATURES))
print("Scaled dataset shape:", df_scaled.shape)

train_df = df_scaled[df_scaled["split"] == "train"].copy()
X_train = train_df[SCALED_FEATURES].values

print("X_train shape:", X_train.shape)

if X_train.shape[1] != 23:
    raise RuntimeError(f"Expected 23 scaled features, got {X_train.shape[1]}")

# ------------------------------------------------------------
# Re-train Isolation Forest using same setup as Cell 8
# ------------------------------------------------------------

if_model = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

print("\nRe-training Isolation Forest for explanation consistency...")
if_model.fit(X_train)

def anomaly_score_fn(X):
    """
    Higher output = more anomalous.
    """
    return -if_model.decision_function(np.asarray(X))

# ------------------------------------------------------------
# Match selected candidates back to full scaled dataframe
# ------------------------------------------------------------

selected_rows = []

for _, cand in selected_candidates.iterrows():
    mask = (
        (df_scaled["building"] == cand["building"]) &
        (df_scaled["meter"] == cand["meter"]) &
        (df_scaled["primary_category"] == cand["primary_category"]) &
        (df_scaled["window_start"] == cand["window_start"]) &
        (df_scaled["window_end"] == cand["window_end"]) &
        (df_scaled["split"] == "test")
    )

    matched = df_scaled[mask].copy()

    if matched.empty:
        raise RuntimeError(
            f"No match found for {cand['primary_category']} "
            f"{cand['window_start']} - {cand['window_end']}"
        )

    # Convert to dict to avoid pandas Series duplicate-index issues
    selected_rows.append(matched.iloc[0].to_dict())

selected_xai_df = pd.DataFrame(selected_rows).reset_index(drop=True)

print("\nMatched selected XAI rows:")
display(selected_xai_df[
    [
        "building", "meter", "primary_category",
        "window_start", "window_end",
        "anomaly_score", "predicted_anomaly"
    ]
])

X_explain = selected_xai_df[SCALED_FEATURES].values

print("\nX_explain shape:", X_explain.shape)

# ------------------------------------------------------------
# SHAP KernelExplainer
# ------------------------------------------------------------

rng = np.random.default_rng(42)

BACKGROUND_SIZE = 100
background_idx = rng.choice(len(X_train), size=min(BACKGROUND_SIZE, len(X_train)), replace=False)
X_background = X_train[background_idx]

print("\nRunning SHAP KernelExplainer...")
print("Background shape:", X_background.shape)
print("Explain shape:", X_explain.shape)

shap_explainer = shap.KernelExplainer(anomaly_score_fn, X_background)
shap_values = shap_explainer.shap_values(X_explain, nsamples=300)

shap_values = np.asarray(shap_values)

if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 0]

print("SHAP values shape:", shap_values.shape)

shap_records = []

for i, row in selected_xai_df.iterrows():
    vals = shap_values[i]

    temp = pd.DataFrame({
        "feature_scaled": SCALED_FEATURES,
        "feature": FEATURES,
        "shap_value": vals,
        "abs_shap_value": np.abs(vals),
        "feature_value_scaled": X_explain[i],
        "feature_value_original": row[FEATURES].values.astype(float)
    })

    temp["dataset"] = "UK-DALE"
    temp["building"] = row["building"]
    temp["meter"] = row["meter"]
    temp["primary_category"] = row["primary_category"]
    temp["window_start"] = row["window_start"]
    temp["window_end"] = row["window_end"]
    temp["anomaly_score"] = row["anomaly_score"]

    temp = temp.sort_values("abs_shap_value", ascending=False)
    shap_records.append(temp)

shap_long = pd.concat(shap_records, ignore_index=True)

shap_path = XAI_DIR / "ukdale_shap_explanations_selected_test_windows.csv"
shap_long.to_csv(shap_path, index=False)

print("\nTop SHAP features per selected window:")
display(
    shap_long.groupby("primary_category", group_keys=False)
    .head(8)[
        [
            "primary_category", "building", "meter",
            "window_start", "anomaly_score",
            "feature", "feature_value_original",
            "shap_value", "abs_shap_value"
        ]
    ]
)

print("\nSaved SHAP explanations:")
print(shap_path)

# ------------------------------------------------------------
# LIME explanations using regression mode
# ------------------------------------------------------------

print("\nRunning LIME explanations...")

lime_explainer = LimeTabularExplainer(
    training_data=X_train,
    feature_names=SCALED_FEATURES,
    mode="regression",
    discretize_continuous=True,
    random_state=42
)

lime_records = []

for i, row in selected_xai_df.iterrows():
    x = X_explain[i]

    explanation = lime_explainer.explain_instance(
        data_row=x,
        predict_fn=anomaly_score_fn,
        num_features=10
    )

    lime_list = explanation.as_list()

    for rank, (condition, weight) in enumerate(lime_list, start=1):
        lime_records.append({
            "dataset": "UK-DALE",
            "building": row["building"],
            "meter": row["meter"],
            "primary_category": row["primary_category"],
            "window_start": row["window_start"],
            "window_end": row["window_end"],
            "anomaly_score": row["anomaly_score"],
            "rank": rank,
            "lime_condition": condition,
            "lime_weight": weight,
            "abs_lime_weight": abs(weight)
        })

lime_df = pd.DataFrame(lime_records)

lime_path = XAI_DIR / "ukdale_lime_explanations_selected_test_windows.csv"
lime_df.to_csv(lime_path, index=False)

print("\nTop LIME features per selected window:")
display(lime_df)

print("\nSaved LIME explanations:")
print(lime_path)

# ------------------------------------------------------------
# Compact XAI tables
# ------------------------------------------------------------

shap_top5 = (
    shap_long.groupby("primary_category", group_keys=False)
    .head(5)
    .copy()
)

lime_top5 = (
    lime_df.sort_values(["primary_category", "rank"])
    .groupby("primary_category", group_keys=False)
    .head(5)
    .copy()
)

shap_top5_path = XAI_DIR / "ukdale_shap_top5_per_category.csv"
lime_top5_path = XAI_DIR / "ukdale_lime_top5_per_category.csv"

shap_top5.to_csv(shap_top5_path, index=False)
lime_top5.to_csv(lime_top5_path, index=False)

print("\nSaved compact XAI tables:")
print(shap_top5_path)
print(lime_top5_path)

print("\nXAI step completed successfully.")

In [ ]:
# ============================================================
# EXP_AMPDS2_Selected_Pipeline
# Cell 1: AMPds2 discovery and file inventory
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import os
from datetime import datetime

# Mount Drive if needed
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped or already mounted:", e)

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

EXP_DIR = BASE / "experiments" / "EXP_AMPDS2_Selected_Pipeline"
OUT_DIR = BASE / "processed_data" / "AMPDS2_Selected_Pipeline"

EXP_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("EXP_DIR:", EXP_DIR)
print("OUT_DIR:", OUT_DIR)
print("BASE exists:", BASE.exists())

# ------------------------------------------------------------
# Search for AMPds2-related files
# ------------------------------------------------------------

keywords = [
    "ampds",
    "ampds2",
    "almanac",
    "electricity",
    "Electricity",
    "AMPD"
]

allowed_suffixes = {
    ".csv", ".txt", ".tsv", ".xlsx", ".xls", ".h5", ".hdf5", ".json", ".md"
}

exclude_keywords = [
    ".ipynb_checkpoints",
    "__MACOSX",
    "trash",
    "checkpoint"
]

candidates = []

for root, dirs, files in os.walk(BASE):
    root_path = Path(root)
    root_str = str(root_path)

    if any(ex.lower() in root_str.lower() for ex in exclude_keywords):
        continue

    for file in files:
        p = root_path / file

        if p.suffix.lower() not in allowed_suffixes:
            continue

        p_str = str(p)

        if any(k.lower() in p_str.lower() for k in keywords):
            try:
                stat = p.stat()
                candidates.append({
                    "path": str(p),
                    "filename": p.name,
                    "parent": str(p.parent),
                    "suffix": p.suffix.lower(),
                    "size_mb": round(stat.st_size / (1024 ** 2), 3),
                    "modified": datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M:%S")
                })
            except Exception:
                pass

candidates_df = pd.DataFrame(candidates)

print("\nAMPds2-related candidate files found:", len(candidates_df))

if len(candidates_df) > 0:
    candidates_df = candidates_df.sort_values(["size_mb", "path"], ascending=[False, True]).reset_index(drop=True)
    display(candidates_df.head(100))

    candidates_path = OUT_DIR / "ampds2_discovered_candidate_files.csv"
    candidates_df.to_csv(candidates_path, index=False)
    print("\nSaved candidate inventory:")
    print(candidates_path)
else:
    print("No AMPds2-related files found. We may need to check folder names manually.")

# ------------------------------------------------------------
# Inspect likely CSV files
# ------------------------------------------------------------

csv_candidates = candidates_df[candidates_df["suffix"].isin([".csv", ".txt", ".tsv"])].copy() if len(candidates_df) > 0 else pd.DataFrame()

schema_records = []

print("\nInspecting CSV/TXT/TSV schemas...")

for _, row in csv_candidates.head(80).iterrows():
    p = Path(row["path"])

    try:
        sep = "," if p.suffix.lower() == ".csv" else None

        sample = pd.read_csv(p, nrows=5, sep=sep, engine="python")

        cols = list(sample.columns)

        lower_cols = [str(c).lower() for c in cols]

        has_timestamp = any(
            c in lower_cols
            for c in ["timestamp", "date", "datetime", "time", "unix_ts", "unix", "ts"]
        )

        has_standard_schema = all(
            c in lower_cols
            for c in ["timestamp", "mean_power", "max_power", "std_power"]
        )

        numeric_cols = sample.select_dtypes(include=[np.number]).columns.tolist()

        schema_records.append({
            "path": str(p),
            "filename": p.name,
            "size_mb": row["size_mb"],
            "n_preview_rows": len(sample),
            "n_columns": len(cols),
            "columns_preview": cols[:30],
            "has_timestamp_like_column": has_timestamp,
            "has_standard_ph2_schema": has_standard_schema,
            "n_numeric_preview_columns": len(numeric_cols),
            "numeric_columns_preview": numeric_cols[:30]
        })

    except Exception as e:
        schema_records.append({
            "path": str(p),
            "filename": p.name,
            "size_mb": row["size_mb"],
            "n_preview_rows": None,
            "n_columns": None,
            "columns_preview": f"READ_ERROR: {e}",
            "has_timestamp_like_column": False,
            "has_standard_ph2_schema": False,
            "n_numeric_preview_columns": None,
            "numeric_columns_preview": None
        })

schema_df = pd.DataFrame(schema_records)

print("\nSchema inspection results:")
if len(schema_df) > 0:
    display(schema_df)

    schema_path = OUT_DIR / "ampds2_schema_inspection.csv"
    schema_df.to_csv(schema_path, index=False)
    print("\nSaved schema inspection:")
    print(schema_path)
else:
    print("No CSV/TXT/TSV candidates to inspect.")

# ------------------------------------------------------------
# Highlight likely usable files
# ------------------------------------------------------------

if len(schema_df) > 0:
    print("\nLikely already-standardized Phase 2 files:")
    ph2_like = schema_df[schema_df["has_standard_ph2_schema"] == True].copy()
    display(ph2_like[["filename", "size_mb", "n_columns", "path"]])

    print("\nLikely raw AMPds2 files with many numeric channels:")
    raw_like = schema_df[
        (schema_df["has_timestamp_like_column"] == True) &
        (schema_df["n_numeric_preview_columns"].fillna(0) >= 3)
    ].copy()
    display(raw_like[["filename", "size_mb", "n_columns", "columns_preview", "path"]])

print("\nDiscovery complete.")

In [ ]:
# ============================================================
# EXP_AMPDS2_Selected_Pipeline
# Cell 1: AMPds2 discovery and file inventory
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import os
from datetime import datetime

# Mount Drive if needed
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped or already mounted:", e)

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

EXP_DIR = BASE / "experiments" / "EXP_AMPDS2_Selected_Pipeline"
OUT_DIR = BASE / "processed_data" / "AMPDS2_Selected_Pipeline"

EXP_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("EXP_DIR:", EXP_DIR)
print("OUT_DIR:", OUT_DIR)
print("BASE exists:", BASE.exists())

# ------------------------------------------------------------
# Search for AMPds2-related files
# ------------------------------------------------------------

keywords = [
    "ampds",
    "ampds2",
    "almanac",
    "electricity",
    "Electricity",
    "AMPD"
]

allowed_suffixes = {
    ".csv", ".txt", ".tsv", ".xlsx", ".xls", ".h5", ".hdf5", ".json", ".md"
}

exclude_keywords = [
    ".ipynb_checkpoints",
    "__MACOSX",
    "trash",
    "checkpoint"
]

candidates = []

for root, dirs, files in os.walk(BASE):
    root_path = Path(root)
    root_str = str(root_path)

    if any(ex.lower() in root_str.lower() for ex in exclude_keywords):
        continue

    for file in files:
        p = root_path / file

        if p.suffix.lower() not in allowed_suffixes:
            continue

        p_str = str(p)

        if any(k.lower() in p_str.lower() for k in keywords):
            try:
                stat = p.stat()
                candidates.append({
                    "path": str(p),
                    "filename": p.name,
                    "parent": str(p.parent),
                    "suffix": p.suffix.lower(),
                    "size_mb": round(stat.st_size / (1024 ** 2), 3),
                    "modified": datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M:%S")
                })
            except Exception:
                pass

candidates_df = pd.DataFrame(candidates)

print("\nAMPds2-related candidate files found:", len(candidates_df))

if len(candidates_df) > 0:
    candidates_df = candidates_df.sort_values(["size_mb", "path"], ascending=[False, True]).reset_index(drop=True)
    display(candidates_df.head(100))

    candidates_path = OUT_DIR / "ampds2_discovered_candidate_files.csv"
    candidates_df.to_csv(candidates_path, index=False)
    print("\nSaved candidate inventory:")
    print(candidates_path)
else:
    print("No AMPds2-related files found. We may need to check folder names manually.")

# ------------------------------------------------------------
# Inspect likely CSV files
# ------------------------------------------------------------

csv_candidates = candidates_df[candidates_df["suffix"].isin([".csv", ".txt", ".tsv"])].copy() if len(candidates_df) > 0 else pd.DataFrame()

schema_records = []

print("\nInspecting CSV/TXT/TSV schemas...")

for _, row in csv_candidates.head(80).iterrows():
    p = Path(row["path"])

    try:
        sep = "," if p.suffix.lower() == ".csv" else None

        sample = pd.read_csv(p, nrows=5, sep=sep, engine="python")

        cols = list(sample.columns)

        lower_cols = [str(c).lower() for c in cols]

        has_timestamp = any(
            c in lower_cols
            for c in ["timestamp", "date", "datetime", "time", "unix_ts", "unix", "ts"]
        )

        has_standard_schema = all(
            c in lower_cols
            for c in ["timestamp", "mean_power", "max_power", "std_power"]
        )

        numeric_cols = sample.select_dtypes(include=[np.number]).columns.tolist()

        schema_records.append({
            "path": str(p),
            "filename": p.name,
            "size_mb": row["size_mb"],
            "n_preview_rows": len(sample),
            "n_columns": len(cols),
            "columns_preview": cols[:30],
            "has_timestamp_like_column": has_timestamp,
            "has_standard_ph2_schema": has_standard_schema,
            "n_numeric_preview_columns": len(numeric_cols),
            "numeric_columns_preview": numeric_cols[:30]
        })

    except Exception as e:
        schema_records.append({
            "path": str(p),
            "filename": p.name,
            "size_mb": row["size_mb"],
            "n_preview_rows": None,
            "n_columns": None,
            "columns_preview": f"READ_ERROR: {e}",
            "has_timestamp_like_column": False,
            "has_standard_ph2_schema": False,
            "n_numeric_preview_columns": None,
            "numeric_columns_preview": None
        })

schema_df = pd.DataFrame(schema_records)

print("\nSchema inspection results:")
if len(schema_df) > 0:
    display(schema_df)

    schema_path = OUT_DIR / "ampds2_schema_inspection.csv"
    schema_df.to_csv(schema_path, index=False)
    print("\nSaved schema inspection:")
    print(schema_path)
else:
    print("No CSV/TXT/TSV candidates to inspect.")

# ------------------------------------------------------------
# Highlight likely usable files
# ------------------------------------------------------------

if len(schema_df) > 0:
    print("\nLikely already-standardized Phase 2 files:")
    ph2_like = schema_df[schema_df["has_standard_ph2_schema"] == True].copy()
    display(ph2_like[["filename", "size_mb", "n_columns", "path"]])

    print("\nLikely raw AMPds2 files with many numeric channels:")
    raw_like = schema_df[
        (schema_df["has_timestamp_like_column"] == True) &
        (schema_df["n_numeric_preview_columns"].fillna(0) >= 3)
    ].copy()
    display(raw_like[["filename", "size_mb", "n_columns", "columns_preview", "path"]])

print("\nDiscovery complete.")

In [ ]:
# ============================================================
# EXP_AMPDS2_Selected_Pipeline
# Cell 2: AMPds2 standardized 1-minute signal inventory
# Purpose:
#   - Identify standardized AMPds2 files
#   - Parse channel codes and measurement suffixes
#   - Check whether P or Pt is the correct active-power signal
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import os

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

OUT_DIR = BASE / "processed_data" / "AMPDS2_Selected_Pipeline"
EXP_DIR = BASE / "experiments" / "EXP_AMPDS2_Selected_Pipeline"

CANDIDATES_PATH = OUT_DIR / "ampds2_discovered_candidate_files.csv"

print("CANDIDATES_PATH:", CANDIDATES_PATH)
print("Exists:", CANDIDATES_PATH.exists())

candidates = pd.read_csv(CANDIDATES_PATH)

print("\nLoaded candidate inventory:", candidates.shape)

# ------------------------------------------------------------
# Parse standardized AMPds2 1-minute filenames
# Expected pattern:
#   Electricity_WHE_Pt_1min.csv
#   Electricity_FRE_St_1min.csv
#   Electricity_B1E_V_1min.csv
# ------------------------------------------------------------

pattern = re.compile(r"^Electricity_([A-Za-z0-9]+)_([A-Za-z0-9]+)_1min\.csv$")

records = []

for _, row in candidates.iterrows():
    filename = row["filename"]
    match = pattern.match(filename)

    if not match:
        continue

    channel_code = match.group(1)
    measurement = match.group(2)

    path = Path(row["path"])

    # Validate schema cheaply
    try:
        sample = pd.read_csv(path, nrows=5)
        cols = list(sample.columns)
        has_standard_schema = cols == ["timestamp", "mean_power", "max_power", "std_power"]
    except Exception as e:
        cols = [f"READ_ERROR: {e}"]
        has_standard_schema = False

    records.append({
        "path": str(path),
        "filename": filename,
        "channel_code": channel_code,
        "measurement": measurement,
        "size_mb": row["size_mb"],
        "modified": row["modified"],
        "columns": cols,
        "has_standard_schema": has_standard_schema
    })

inventory = pd.DataFrame(records)

print("\nParsed standardized AMPds2 1-minute files:", inventory.shape)

if len(inventory) == 0:
    raise RuntimeError("No standardized AMPds2 1-minute files parsed. Check filenames/paths.")

inventory = inventory.sort_values(["measurement", "channel_code"]).reset_index(drop=True)

inventory_path = OUT_DIR / "ampds2_standardized_1min_inventory.csv"
inventory.to_csv(inventory_path, index=False)

print("\nSaved standardized inventory:")
print(inventory_path)

print("\nMeasurement suffix counts:")
measurement_counts = (
    inventory.groupby("measurement")
    .agg(
        n_files=("filename", "count"),
        total_size_mb=("size_mb", "sum"),
        n_channels=("channel_code", "nunique"),
        standard_schema_files=("has_standard_schema", "sum")
    )
    .reset_index()
    .sort_values("measurement")
)
display(measurement_counts)

print("\nChannel counts:")
channel_counts = (
    inventory.groupby("channel_code")
    .agg(
        n_measurements=("measurement", "nunique"),
        measurements=("measurement", lambda x: ", ".join(sorted(set(x)))),
        total_size_mb=("size_mb", "sum")
    )
    .reset_index()
    .sort_values("channel_code")
)
display(channel_counts)

# ------------------------------------------------------------
# AMPds2 common channel-code helper
# This is only a practical label map for readability.
# It does not affect the experiment.
# ------------------------------------------------------------

channel_name_map = {
    "WHE": "whole_house_electricity",
    "RSE": "rental_suite_electricity",
    "GRE": "garage_electricity",
    "MHE": "main_house_electricity",
    "B1E": "bedroom_1_electricity",
    "B2E": "bedroom_2_electricity",
    "BME": "basement_electricity",
    "CWE": "clothes_washer_electricity",
    "DWE": "dishwasher_electricity",
    "EBE": "electronics_bench_electricity",
    "EQE": "equipment_electricity",
    "FGE": "furnace_gas_electricity",
    "FRE": "fridge_electricity",
    "HPE": "heat_pump_electricity",
    "HTE": "hot_tub_electricity",
    "OFE": "office_electricity",
    "OTE": "outdoor_electricity",
    "TVE": "television_electricity",
    "UTE": "utility_electricity",
    "WOE": "wall_oven_electricity",
    "CDE": "clothes_dryer_electricity",
    "DNE": "dining_room_electricity"
}

inventory["channel_name"] = inventory["channel_code"].map(channel_name_map).fillna("unknown_or_unmapped")
inventory.to_csv(inventory_path, index=False)

print("\nInventory with readable channel names:")
display(inventory[["filename", "channel_code", "channel_name", "measurement", "size_mb", "has_standard_schema", "path"]].head(100))

# ------------------------------------------------------------
# Inspect likely active-power candidates
# We test P and Pt because one may represent instantaneous/active power,
# while the other could be cumulative or transformed depending on processing.
# ------------------------------------------------------------

candidate_measurements = ["P", "Pt"]

candidate_power_files = inventory[
    (inventory["measurement"].isin(candidate_measurements)) &
    (inventory["has_standard_schema"] == True)
].copy()

print("\nCandidate active-power-like files using P or Pt:")
display(candidate_power_files[["filename", "channel_code", "channel_name", "measurement", "size_mb", "path"]])

print("\nCounts for P/Pt candidate files:")
display(
    candidate_power_files.groupby("measurement")
    .agg(
        n_files=("filename", "count"),
        n_channels=("channel_code", "nunique"),
        total_size_mb=("size_mb", "sum")
    )
    .reset_index()
)

# ------------------------------------------------------------
# Signal sanity check:
#   - read a limited sample from selected P/Pt files
#   - check value range, missingness, and monotonic behavior
# If a signal is cumulative, it will often show very high monotonicity.
# ------------------------------------------------------------

def inspect_signal_file(path, nrows=20000):
    temp = pd.read_csv(path, usecols=["timestamp", "mean_power"], nrows=nrows)

    temp["timestamp"] = pd.to_datetime(temp["timestamp"], errors="coerce", utc=True)
    temp["mean_power"] = pd.to_numeric(temp["mean_power"], errors="coerce")

    x = temp["mean_power"].replace([np.inf, -np.inf], np.nan).dropna()

    if len(x) < 2:
        return {
            "n_sample_rows": len(temp),
            "valid_values": len(x),
            "mean": np.nan,
            "std": np.nan,
            "min": np.nan,
            "median": np.nan,
            "max": np.nan,
            "zero_ratio": np.nan,
            "monotonic_non_decreasing_ratio": np.nan,
            "timestamp_min": temp["timestamp"].min(),
            "timestamp_max": temp["timestamp"].max()
        }

    diffs = x.diff().dropna()

    return {
        "n_sample_rows": len(temp),
        "valid_values": len(x),
        "mean": float(x.mean()),
        "std": float(x.std()),
        "min": float(x.min()),
        "median": float(x.median()),
        "max": float(x.max()),
        "zero_ratio": float((x == 0).mean()),
        "monotonic_non_decreasing_ratio": float((diffs >= 0).mean()),
        "timestamp_min": temp["timestamp"].min(),
        "timestamp_max": temp["timestamp"].max()
    }

# Prefer these readable channels for signal sanity inspection
priority_channels = ["WHE", "MHE", "FRE", "DWE", "CWE", "CDE", "WOE", "HPE"]

inspect_rows = []

for measurement in candidate_measurements:
    subset = candidate_power_files[
        candidate_power_files["channel_code"].isin(priority_channels) &
        (candidate_power_files["measurement"] == measurement)
    ].copy()

    # If none found, inspect a few available files for that measurement
    if subset.empty:
        subset = candidate_power_files[candidate_power_files["measurement"] == measurement].head(8).copy()
    else:
        subset = subset.head(8).copy()

    for _, row in subset.iterrows():
        stats = inspect_signal_file(row["path"], nrows=20000)

        inspect_rows.append({
            "filename": row["filename"],
            "channel_code": row["channel_code"],
            "channel_name": row["channel_name"],
            "measurement": row["measurement"],
            "path": row["path"],
            **stats
        })

signal_check = pd.DataFrame(inspect_rows)

signal_check_path = OUT_DIR / "ampds2_power_signal_sanity_check.csv"
signal_check.to_csv(signal_check_path, index=False)

print("\nPower signal sanity check for P/Pt candidates:")
display(signal_check)

print("\nSaved signal sanity check:")
print(signal_check_path)

# ------------------------------------------------------------
# Recommendation helper:
# If P exists and looks non-cumulative, usually prefer P.
# If only Pt exists and is not cumulative, use Pt.
# ------------------------------------------------------------

if len(signal_check) > 0:
    summary_signal = (
        signal_check.groupby("measurement")
        .agg(
            n_checked=("filename", "count"),
            median_mean_power=("mean", "median"),
            median_std_power=("std", "median"),
            median_zero_ratio=("zero_ratio", "median"),
            median_monotonic_ratio=("monotonic_non_decreasing_ratio", "median")
        )
        .reset_index()
    )

    print("\nP/Pt sanity summary:")
    display(summary_signal)

print("\nCell 2 complete.")

In [ ]:
# ============================================================
# EXP_AMPDS2_Selected_Pipeline
# Cell 3: Generate AMPds2 60/30 selected 23-feature windows
# Signal used: P active power only
# No synthetic injection
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
from scipy.stats import skew, kurtosis

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

OUT_DIR = BASE / "processed_data" / "AMPDS2_Selected_Pipeline"
EXP_DIR = BASE / "experiments" / "EXP_AMPDS2_Selected_Pipeline"

INVENTORY_PATH = OUT_DIR / "ampds2_standardized_1min_inventory.csv"

print("INVENTORY_PATH:", INVENTORY_PATH)
print("Exists:", INVENTORY_PATH.exists())

inventory = pd.read_csv(INVENTORY_PATH)

# ------------------------------------------------------------
# Select AMPds2 active-power files only
# ------------------------------------------------------------

p_files = inventory[
    (inventory["measurement"] == "P") &
    (inventory["has_standard_schema"] == True)
].copy()

p_files = p_files.sort_values("channel_code").reset_index(drop=True)

print("\nSelected AMPds2 P active-power files:", p_files.shape)
display(p_files[["filename", "channel_code", "channel_name", "measurement", "size_mb", "path"]])

if len(p_files) == 0:
    raise RuntimeError("No AMPds2 P active-power files found.")

# ------------------------------------------------------------
# Channel category mapping
# ------------------------------------------------------------

channel_category_map = {
    "WHE": "whole_house",
    "RSE": "rental_suite",
    "GRE": "garage",
    "MHE": "main_house",
    "B1E": "bedroom_1",
    "B2E": "bedroom_2",
    "BME": "basement",
    "CWE": "clothes_washer",
    "DWE": "dishwasher",
    "EBE": "electronics_bench",
    "EQE": "equipment",
    "FGE": "furnace_gas",
    "FRE": "fridge",
    "HPE": "heat_pump",
    "HTE": "hot_tub",
    "OFE": "office",
    "OUE": "outdoor_unknown",
    "TVE": "television",
    "UTE": "utility",
    "WOE": "wall_oven",
    "CDE": "clothes_dryer",
    "DNE": "dining_room"
}

# ------------------------------------------------------------
# Selected 23-feature extractor from a 60-value power window
# Same representation family as NAB and UK-DALE
# ------------------------------------------------------------

FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

WINDOW_SIZE = 60
STRIDE = 30

def extract_selected23_from_array(x):
    x = np.asarray(x, dtype=float)

    # Clean invalid values
    x = np.where(np.isfinite(x), x, 0.0)

    diffs = np.diff(x)

    q25 = np.percentile(x, 25)
    q75 = np.percentile(x, 75)
    med = np.median(x)
    mean_val = np.mean(x)
    std_val = np.std(x)
    var_val = np.var(x)

    # Simple local peak count:
    # value greater than both neighbors
    if len(x) >= 3:
        peak_count = int(np.sum((x[1:-1] > x[:-2]) & (x[1:-1] > x[2:])))
    else:
        peak_count = 0

    # Linear slope across the 60-minute window
    try:
        slope_val = float(np.polyfit(np.arange(len(x)), x, 1)[0])
    except Exception:
        slope_val = 0.0

    if std_val == 0:
        skew_val = 0.0
        kurt_val = 0.0
    else:
        skew_val = float(skew(x, bias=False, nan_policy="omit"))
        kurt_val = float(kurtosis(x, bias=False, nan_policy="omit"))

        if not np.isfinite(skew_val):
            skew_val = 0.0
        if not np.isfinite(kurt_val):
            kurt_val = 0.0

    if len(diffs) > 0:
        mean_abs_diff = float(np.mean(np.abs(diffs)))
        max_abs_diff = float(np.max(np.abs(diffs)))
        std_diff = float(np.std(diffs))
    else:
        mean_abs_diff = 0.0
        max_abs_diff = 0.0
        std_diff = 0.0

    return {
        "energy": float(np.sum(x) / WINDOW_SIZE),
        "first_last_diff": float(x[-1] - x[0]),
        "iqr": float(q75 - q25),
        "kurtosis": kurt_val,
        "mad": float(np.median(np.abs(x - med))),
        "max": float(np.max(x)),
        "max_abs_diff": max_abs_diff,
        "mean": float(mean_val),
        "mean_abs_diff": mean_abs_diff,
        "median": float(med),
        "min": float(np.min(x)),
        "n_points": int(len(x)),
        "peak_count": int(peak_count),
        "q25": float(q25),
        "q75": float(q75),
        "range": float(np.max(x) - np.min(x)),
        "rms": float(np.sqrt(np.mean(x ** 2))),
        "skew": skew_val,
        "slope": slope_val,
        "std": float(std_val),
        "std_diff": std_diff,
        "sum": float(np.sum(x)),
        "var": float(var_val)
    }

def generate_windows_for_file(row):
    path = Path(row["path"])
    channel_code = row["channel_code"]
    channel_name = row.get("channel_name", "unknown")
    channel_category = channel_category_map.get(channel_code, channel_name)

    print(f"\nProcessing {path.name} | {channel_code} | {channel_category}")

    raw = pd.read_csv(path)

    required_cols = ["timestamp", "mean_power"]
    missing = [c for c in required_cols if c not in raw.columns]
    if missing:
        raise RuntimeError(f"Missing columns {missing} in {path}")

    raw["timestamp"] = pd.to_datetime(raw["timestamp"], utc=True, errors="coerce")
    raw["mean_power"] = pd.to_numeric(raw["mean_power"], errors="coerce")

    raw = raw.dropna(subset=["timestamp", "mean_power"]).copy()
    raw = raw.sort_values("timestamp").reset_index(drop=True)

    raw["mean_power"] = raw["mean_power"].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    values = raw["mean_power"].values.astype(float)
    timestamps = raw["timestamp"].values

    n = len(raw)

    records = []

    if n < WINDOW_SIZE:
        print("Skipped: fewer rows than window size.")
        return pd.DataFrame(), {
            "filename": path.name,
            "channel_code": channel_code,
            "channel_category": channel_category,
            "n_rows": n,
            "n_windows": 0,
            "status": "too_short"
        }

    for start_idx in range(0, n - WINDOW_SIZE + 1, STRIDE):
        end_idx = start_idx + WINDOW_SIZE

        x = values[start_idx:end_idx]
        start_time = pd.Timestamp(timestamps[start_idx])
        end_time = pd.Timestamp(timestamps[end_idx - 1])

        duration_minutes = ((end_time - start_time) / np.timedelta64(1, "m")) + 1

        feat = extract_selected23_from_array(x)

        rec = {
            "dataset": "AMPds2",
            "channel_code": channel_code,
            "channel_name": channel_name,
            "primary_category": channel_category,
            "measurement": "P",
            "source_raw_file": path.name,
            "window_start": start_time,
            "window_end": end_time,
            "duration_minutes": duration_minutes,
            "window_size_minutes": WINDOW_SIZE,
            "stride_minutes": STRIDE,
            "label": -1,
            "is_synthetic": False
        }

        rec.update(feat)
        records.append(rec)

    out = pd.DataFrame(records)

    summary = {
        "filename": path.name,
        "channel_code": channel_code,
        "channel_name": channel_name,
        "channel_category": channel_category,
        "n_rows": n,
        "n_windows": len(out),
        "timestamp_min": raw["timestamp"].min(),
        "timestamp_max": raw["timestamp"].max(),
        "mean_power_mean": raw["mean_power"].mean(),
        "mean_power_median": raw["mean_power"].median(),
        "mean_power_max": raw["mean_power"].max(),
        "zero_ratio": float((raw["mean_power"] == 0).mean()),
        "duration_median": out["duration_minutes"].median() if len(out) else np.nan,
        "duration_max": out["duration_minutes"].max() if len(out) else np.nan,
        "status": "ok"
    }

    print(f"Generated windows: {len(out):,}")

    return out, summary

# ------------------------------------------------------------
# Generate all AMPds2 P-channel windows
# ------------------------------------------------------------

all_windows = []
summaries = []

for _, row in p_files.iterrows():
    windows, summary = generate_windows_for_file(row)
    summaries.append(summary)

    if len(windows) > 0:
        all_windows.append(windows)

if len(all_windows) == 0:
    raise RuntimeError("No AMPds2 windows generated.")

combined = pd.concat(all_windows, ignore_index=True)
summary_df = pd.DataFrame(summaries)

# ------------------------------------------------------------
# Save raw generated windows before continuity filtering
# ------------------------------------------------------------

WINDOWS_PATH = OUT_DIR / "ampds2_p_60min_stride30_selected23_windows.csv"
SUMMARY_PATH = OUT_DIR / "ampds2_p_60min_stride30_selected23_generation_summary.csv"

combined.to_csv(WINDOWS_PATH, index=False)
summary_df.to_csv(SUMMARY_PATH, index=False)

print("\nSaved generated AMPds2 windows:")
print(WINDOWS_PATH)

print("\nSaved generation summary:")
print(SUMMARY_PATH)

print("\nCombined shape before continuity filtering:", combined.shape)

print("\nGeneration summary:")
display(summary_df)

print("\nWindow counts by category before filtering:")
summary_category = (
    combined.groupby(["primary_category", "channel_code"])
    .agg(
        n_windows=("label", "count"),
        duration_median=("duration_minutes", "median"),
        duration_max=("duration_minutes", "max"),
        mean_energy=("energy", "mean"),
        zero_energy_ratio=("energy", lambda x: float((x == 0).mean()))
    )
    .reset_index()
    .sort_values(["primary_category", "channel_code"])
)
display(summary_category)

summary_category_path = OUT_DIR / "ampds2_p_selected23_window_summary_by_category.csv"
summary_category.to_csv(summary_category_path, index=False)

print("\nSaved category summary:")
print(summary_category_path)

print("\nPreview:")
display(combined.head())

print("\nCell 3 complete.")

In [ ]:
# ============================================================
# EXP_AMPDS2_Selected_Pipeline
# Cell 4: Validate and filter continuous 60-minute AMPds2 windows
# No synthetic injection
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

OUT_DIR = BASE / "processed_data" / "AMPDS2_Selected_Pipeline"
EXP_DIR = BASE / "experiments" / "EXP_AMPDS2_Selected_Pipeline"

WINDOWS_PATH = OUT_DIR / "ampds2_p_60min_stride30_selected23_windows.csv"

print("WINDOWS_PATH:", WINDOWS_PATH)
print("Exists:", WINDOWS_PATH.exists())

df = pd.read_csv(WINDOWS_PATH)

print("\nOriginal AMPds2 generated windows:", df.shape)

# ------------------------------------------------------------
# Convert timestamps and recompute duration
# ------------------------------------------------------------

df["window_start"] = pd.to_datetime(df["window_start"], utc=True, errors="coerce")
df["window_end"] = pd.to_datetime(df["window_end"], utc=True, errors="coerce")

df["duration_minutes_recomputed"] = (
    (df["window_end"] - df["window_start"]) / np.timedelta64(1, "m")
) + 1

print("\nDuration statistics before filtering:")
display(df["duration_minutes_recomputed"].describe())

print("\nDuration value counts:")
display(df["duration_minutes_recomputed"].value_counts().sort_index().head(20))

# ------------------------------------------------------------
# Keep only continuous 60-minute windows
# ------------------------------------------------------------

valid_mask = df["duration_minutes_recomputed"].between(59.9, 60.1)

valid = df[valid_mask].copy()
removed = df[~valid_mask].copy()

print("\nValid continuous windows:", valid.shape)
print("Removed gap-crossing windows:", removed.shape)
print("Removed percentage:", round(len(removed) / len(df) * 100, 4), "%")

if len(removed) > 0:
    print("\nRemoved duration statistics:")
    display(removed["duration_minutes_recomputed"].describe())
else:
    print("\nNo gap-crossing windows removed.")

# Replace duration column with validated duration
valid["duration_minutes"] = valid["duration_minutes_recomputed"]
valid = valid.drop(columns=["duration_minutes_recomputed"])

# ------------------------------------------------------------
# Save valid filtered dataset
# ------------------------------------------------------------

VALID_PATH = OUT_DIR / "ampds2_p_60min_stride30_selected23_valid_windows.csv"
REMOVED_SUMMARY_PATH = OUT_DIR / "ampds2_removed_gap_crossing_windows_summary.csv"

valid.to_csv(VALID_PATH, index=False)

# ------------------------------------------------------------
# Quality summary by channel
# ------------------------------------------------------------

before_summary = (
    df.groupby(["primary_category", "channel_code", "source_raw_file"])
    .agg(before_windows=("label", "count"))
    .reset_index()
)

after_summary = (
    valid.groupby(["primary_category", "channel_code", "source_raw_file"])
    .agg(after_windows=("label", "count"))
    .reset_index()
)

quality_summary = before_summary.merge(
    after_summary,
    on=["primary_category", "channel_code", "source_raw_file"],
    how="left"
)

quality_summary["after_windows"] = quality_summary["after_windows"].fillna(0).astype(int)
quality_summary["removed_windows"] = quality_summary["before_windows"] - quality_summary["after_windows"]
quality_summary["removed_percent"] = (
    quality_summary["removed_windows"] / quality_summary["before_windows"] * 100
).round(4)

quality_summary.to_csv(REMOVED_SUMMARY_PATH, index=False)

print("\nQuality summary by channel:")
display(quality_summary.sort_values(["primary_category", "channel_code"]))

# ------------------------------------------------------------
# Final summaries
# ------------------------------------------------------------

summary_category = (
    valid.groupby(["primary_category", "channel_code"])
    .agg(
        n_windows=("label", "count"),
        duration_median=("duration_minutes", "median"),
        duration_max=("duration_minutes", "max"),
        mean_energy=("energy", "mean"),
        median_energy=("energy", "median"),
        max_energy=("energy", "max"),
        zero_energy_ratio=("energy", lambda x: float((x == 0).mean()))
    )
    .reset_index()
    .sort_values(["primary_category", "channel_code"])
)

summary_overall = pd.DataFrame([{
    "dataset": "AMPds2",
    "measurement": "P",
    "n_channels": valid["channel_code"].nunique(),
    "n_categories": valid["primary_category"].nunique(),
    "n_windows": len(valid),
    "duration_median": valid["duration_minutes"].median(),
    "duration_max": valid["duration_minutes"].max(),
    "window_size_minutes": 60,
    "stride_minutes": 30,
    "synthetic_injection": False
}])

SUMMARY_CATEGORY_PATH = OUT_DIR / "ampds2_p_valid_window_summary_by_category.csv"
SUMMARY_OVERALL_PATH = OUT_DIR / "ampds2_p_valid_window_summary_overall.csv"

summary_category.to_csv(SUMMARY_CATEGORY_PATH, index=False)
summary_overall.to_csv(SUMMARY_OVERALL_PATH, index=False)

print("\nFinal overall summary:")
display(summary_overall)

print("\nFinal valid window counts by category:")
display(summary_category)

print("\nSaved valid dataset:")
print(VALID_PATH)

print("\nSaved summaries:")
print(REMOVED_SUMMARY_PATH)
print(SUMMARY_CATEGORY_PATH)
print(SUMMARY_OVERALL_PATH)

print("\nCell 4 complete.")

In [ ]:
# ============================================================
# EXP_AMPDS2_Selected_Pipeline
# Cell 5: AMPds2 real-data Isolation Forest scoring
# No synthetic injection
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

OUT_DIR = BASE / "processed_data" / "AMPDS2_Selected_Pipeline"
EXP_DIR = BASE / "experiments" / "EXP_AMPDS2_Selected_Pipeline"

EXP_DIR.mkdir(parents=True, exist_ok=True)

VALID_WINDOWS_PATH = OUT_DIR / "ampds2_p_60min_stride30_selected23_valid_windows.csv"

print("VALID_WINDOWS_PATH:", VALID_WINDOWS_PATH)
print("Exists:", VALID_WINDOWS_PATH.exists())

df = pd.read_csv(VALID_WINDOWS_PATH)

print("\nLoaded AMPds2 valid windows:", df.shape)

# ------------------------------------------------------------
# Selected 23 features
# ------------------------------------------------------------

FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

missing = [f for f in FEATURES if f not in df.columns]
print("Missing features:", missing)

if missing:
    raise RuntimeError(f"Missing selected features: {missing}")

# ------------------------------------------------------------
# Clean numeric values and timestamps
# ------------------------------------------------------------

for f in FEATURES:
    df[f] = pd.to_numeric(df[f], errors="coerce")

df[FEATURES] = df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0)

df["window_start"] = pd.to_datetime(df["window_start"], utc=True, errors="coerce")
df["window_end"] = pd.to_datetime(df["window_end"], utc=True, errors="coerce")

df = df.sort_values(["channel_code", "window_start"]).reset_index(drop=True)

df["series_key"] = (
    df["channel_code"].astype(str) + "_" + df["primary_category"].astype(str)
)

print("\nSeries included:")
display(
    df.groupby(["series_key", "channel_code", "primary_category"])
    .agg(n_windows=("label", "count"))
    .reset_index()
    .sort_values(["channel_code"])
)

# ------------------------------------------------------------
# Chronological split per channel: 60/20/20
# ------------------------------------------------------------

split_parts = []

for series_key, g in df.groupby("series_key", sort=False):
    g = g.sort_values("window_start").copy()
    n = len(g)

    train_end = int(n * 0.60)
    val_end = int(n * 0.80)

    split = np.array(["test"] * n, dtype=object)
    split[:train_end] = "train"
    split[train_end:val_end] = "val"

    g["split"] = split
    split_parts.append(g)

df_split = pd.concat(split_parts, ignore_index=True)

print("\nSplit summary:")
split_summary = (
    df_split.groupby("split")
    .agg(n_windows=("label", "count"))
    .reset_index()
)
display(split_summary)

print("\nSplit summary by channel:")
split_channel_summary = (
    df_split.groupby(["primary_category", "channel_code", "split"])
    .agg(n_windows=("label", "count"))
    .reset_index()
    .sort_values(["channel_code", "split"])
)
display(split_channel_summary)

# ------------------------------------------------------------
# Per-channel z-score normalization using TRAIN statistics only
# ------------------------------------------------------------

scaled_parts = []

for series_key, g in df_split.groupby("series_key", sort=False):
    g = g.copy()

    train_mask = g["split"] == "train"

    scaler = StandardScaler()
    scaler.fit(g.loc[train_mask, FEATURES])

    X_scaled = scaler.transform(g[FEATURES])

    scaled_cols = [f"{f}_z" for f in FEATURES]
    scaled_df = pd.DataFrame(X_scaled, columns=scaled_cols, index=g.index)

    g = pd.concat([g, scaled_df], axis=1)

    scaled_parts.append(g)

df_scaled = pd.concat(scaled_parts, ignore_index=True)

SCALED_FEATURES = [f"{f}_z" for f in FEATURES]

print("\nScaled dataset shape:", df_scaled.shape)
print("Scaled feature count:", len(SCALED_FEATURES))

# ------------------------------------------------------------
# Train Isolation Forest on original AMPds2 TRAIN windows only
# ------------------------------------------------------------

train_df = df_scaled[df_scaled["split"] == "train"].copy()
val_df = df_scaled[df_scaled["split"] == "val"].copy()
test_df = df_scaled[df_scaled["split"] == "test"].copy()

print("\nTrain/Val/Test shapes:")
print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

X_train = train_df[SCALED_FEATURES].values
X_all = df_scaled[SCALED_FEATURES].values

if_model = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

print("\nTraining Isolation Forest on AMPds2 train windows only...")
if_model.fit(X_train)

# Lower decision_function = more anomalous.
# Convert to anomaly score where higher = more anomalous.
df_scaled["if_decision_function"] = if_model.decision_function(X_all)
df_scaled["anomaly_score"] = -df_scaled["if_decision_function"]

# Use validation score distribution as operating threshold
THRESHOLD_PERCENTILE = 95
val_scores = df_scaled.loc[df_scaled["split"] == "val", "anomaly_score"]
threshold = np.percentile(val_scores, THRESHOLD_PERCENTILE)

df_scaled["if_threshold_percentile"] = THRESHOLD_PERCENTILE
df_scaled["if_threshold"] = threshold
df_scaled["predicted_anomaly"] = (df_scaled["anomaly_score"] >= threshold).astype(int)

print("\nIsolation Forest scoring complete.")
print("Validation percentile threshold:", THRESHOLD_PERCENTILE)
print("Threshold value:", threshold)

# ------------------------------------------------------------
# Summaries
# ------------------------------------------------------------

overall_summary = (
    df_scaled.groupby("split")
    .agg(
        n_windows=("predicted_anomaly", "count"),
        flagged_windows=("predicted_anomaly", "sum"),
        flagged_ratio=("predicted_anomaly", "mean"),
        score_mean=("anomaly_score", "mean"),
        score_std=("anomaly_score", "std"),
        score_min=("anomaly_score", "min"),
        score_median=("anomaly_score", "median"),
        score_max=("anomaly_score", "max")
    )
    .reset_index()
)

overall_summary["flagged_ratio_percent"] = (overall_summary["flagged_ratio"] * 100).round(2)

category_summary = (
    df_scaled.groupby(["primary_category", "channel_code", "split"])
    .agg(
        n_windows=("predicted_anomaly", "count"),
        flagged_windows=("predicted_anomaly", "sum"),
        flagged_ratio=("predicted_anomaly", "mean"),
        score_mean=("anomaly_score", "mean"),
        score_median=("anomaly_score", "median"),
        score_max=("anomaly_score", "max")
    )
    .reset_index()
    .sort_values(["primary_category", "channel_code", "split"])
)

category_summary["flagged_ratio_percent"] = (category_summary["flagged_ratio"] * 100).round(2)

test_category_summary = (
    category_summary[category_summary["split"] == "test"]
    .copy()
    .sort_values("flagged_ratio", ascending=False)
)

print("\nOverall scoring summary:")
display(overall_summary)

print("\nTest channel/category scoring summary:")
display(test_category_summary)

# Top anomalous windows
top_anomalies = (
    df_scaled
    .sort_values("anomaly_score", ascending=False)
    .head(1000)
    .copy()
)

top_cols = [
    "dataset", "channel_code", "channel_name", "primary_category",
    "measurement", "source_raw_file",
    "window_start", "window_end", "split",
    "anomaly_score", "if_threshold", "predicted_anomaly"
] + FEATURES

print("\nTop anomalous windows preview:")
display(top_anomalies[top_cols].head(30))

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

scored_path = OUT_DIR / "ampds2_p_if_scored_no_injection.csv"

overall_summary_path = EXP_DIR / "ampds2_if_no_injection_overall_summary.csv"
category_summary_path = EXP_DIR / "ampds2_if_no_injection_category_summary.csv"
test_category_summary_path = EXP_DIR / "ampds2_if_no_injection_test_category_summary.csv"
top_anomalies_path = EXP_DIR / "ampds2_if_no_injection_top_anomalous_windows.csv"

df_scaled.to_csv(scored_path, index=False)
overall_summary.to_csv(overall_summary_path, index=False)
category_summary.to_csv(category_summary_path, index=False)
test_category_summary.to_csv(test_category_summary_path, index=False)
top_anomalies[top_cols].to_csv(top_anomalies_path, index=False)

print("\nSaved outputs:")
print(scored_path)
print(overall_summary_path)
print(category_summary_path)
print(test_category_summary_path)
print(top_anomalies_path)

print("\nCell 5 complete.")

In [ ]:
# ============================================================
# EXP_AMPDS2_Selected_Pipeline
# Cell 6: AMPds2 no-injection summaries, figures, and XAI candidates
# Test-only candidate selection
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

OUT_DIR = BASE / "processed_data" / "AMPDS2_Selected_Pipeline"
EXP_DIR = BASE / "experiments" / "EXP_AMPDS2_Selected_Pipeline"

FIG_DIR = EXP_DIR / "figures"
TABLE_DIR = EXP_DIR / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

SCORED_PATH = OUT_DIR / "ampds2_p_if_scored_no_injection.csv"

print("SCORED_PATH:", SCORED_PATH)
print("Exists:", SCORED_PATH.exists())

df = pd.read_csv(SCORED_PATH)

print("\nLoaded scored AMPds2 dataset:", df.shape)

df["window_start"] = pd.to_datetime(df["window_start"], utc=True, errors="coerce")
df["window_end"] = pd.to_datetime(df["window_end"], utc=True, errors="coerce")

# ------------------------------------------------------------
# Summary tables
# ------------------------------------------------------------

overall_summary = (
    df.groupby("split")
    .agg(
        n_windows=("predicted_anomaly", "count"),
        flagged_windows=("predicted_anomaly", "sum"),
        flagged_ratio=("predicted_anomaly", "mean"),
        score_mean=("anomaly_score", "mean"),
        score_std=("anomaly_score", "std"),
        score_min=("anomaly_score", "min"),
        score_median=("anomaly_score", "median"),
        score_max=("anomaly_score", "max")
    )
    .reset_index()
)

overall_summary["flagged_ratio_percent"] = (overall_summary["flagged_ratio"] * 100).round(2)

category_summary = (
    df.groupby(["primary_category", "channel_code", "split"])
    .agg(
        n_windows=("predicted_anomaly", "count"),
        flagged_windows=("predicted_anomaly", "sum"),
        flagged_ratio=("predicted_anomaly", "mean"),
        score_mean=("anomaly_score", "mean"),
        score_std=("anomaly_score", "std"),
        score_median=("anomaly_score", "median"),
        score_max=("anomaly_score", "max")
    )
    .reset_index()
)

category_summary["flagged_ratio_percent"] = (category_summary["flagged_ratio"] * 100).round(2)

test_category_summary = (
    category_summary[category_summary["split"] == "test"]
    .copy()
    .sort_values("flagged_ratio", ascending=False)
    .reset_index(drop=True)
)

print("\nOverall summary:")
display(overall_summary)

print("\nTest category summary:")
display(test_category_summary)

# Save clean summary tables
overall_summary_path = TABLE_DIR / "ampds2_no_injection_overall_summary.csv"
category_summary_path = TABLE_DIR / "ampds2_no_injection_category_summary.csv"
test_category_summary_path = TABLE_DIR / "ampds2_no_injection_test_category_summary.csv"

overall_summary.to_csv(overall_summary_path, index=False)
category_summary.to_csv(category_summary_path, index=False)
test_category_summary.to_csv(test_category_summary_path, index=False)

# ------------------------------------------------------------
# Figures
# ------------------------------------------------------------

# Figure 1: flagged ratio by split
split_order = ["train", "val", "test"]
plot_overall = overall_summary.set_index("split").loc[split_order].reset_index()

plt.figure(figsize=(7, 4.5))
plt.bar(plot_overall["split"], plot_overall["flagged_ratio_percent"])
plt.xlabel("Split")
plt.ylabel("Flagged windows (%)")
plt.title("AMPds2 Isolation Forest flagged-window ratio by split")
plt.tight_layout()

fig1_path = FIG_DIR / "ampds2_if_no_injection_flagged_ratio_by_split.png"
plt.savefig(fig1_path, dpi=300, bbox_inches="tight")
plt.show()

# Figure 2: test flagged ratio by channel/category
plt.figure(figsize=(11, 5.5))
x_labels = test_category_summary["primary_category"] + "\n(" + test_category_summary["channel_code"] + ")"
plt.bar(x_labels, test_category_summary["flagged_ratio_percent"])
plt.xlabel("AMPds2 channel")
plt.ylabel("Flagged windows in test split (%)")
plt.title("AMPds2 test flagged-window ratio by channel")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()

fig2_path = FIG_DIR / "ampds2_if_no_injection_test_flagged_ratio_by_channel.png"
plt.savefig(fig2_path, dpi=300, bbox_inches="tight")
plt.show()

# Figure 3: test anomaly-score distribution by selected top channels
top_channels = test_category_summary.head(8)["channel_code"].tolist()

test_df = df[df["split"] == "test"].copy()
test_top = test_df[test_df["channel_code"].isin(top_channels)].copy()

plt.figure(figsize=(10, 5.5))

# Boxplot per channel
box_data = []
box_labels = []

for ch in top_channels:
    temp = test_top[test_top["channel_code"] == ch]["anomaly_score"].dropna().values
    cat = test_top[test_top["channel_code"] == ch]["primary_category"].iloc[0]
    box_data.append(temp)
    box_labels.append(f"{cat}\n({ch})")

plt.boxplot(box_data, labels=box_labels, showfliers=False)
plt.axhline(df["if_threshold"].iloc[0], linestyle="--", linewidth=1)
plt.xlabel("AMPds2 channel")
plt.ylabel("Anomaly score")
plt.title("AMPds2 test anomaly-score distribution for highest flagged channels")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

fig3_path = FIG_DIR / "ampds2_if_no_injection_test_score_distribution_top_channels.png"
plt.savefig(fig3_path, dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved figures:")
print(fig1_path)
print(fig2_path)
print(fig3_path)

# ------------------------------------------------------------
# XAI candidate selection
# Use TEST split only.
# ------------------------------------------------------------

FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

base_cols = [
    "dataset",
    "channel_code",
    "channel_name",
    "primary_category",
    "measurement",
    "source_raw_file",
    "window_start",
    "window_end",
    "split",
    "anomaly_score",
    "if_threshold",
    "predicted_anomaly"
]

# Top overall TEST flagged windows
top_overall_test = (
    test_df[test_df["predicted_anomaly"] == 1]
    .sort_values("anomaly_score", ascending=False)
    .head(100)
    .copy()
)

# Top one TEST flagged window per channel/category
selected_rows = []

for (category, channel), g in test_df[test_df["predicted_anomaly"] == 1].groupby(["primary_category", "channel_code"]):
    g = g.sort_values("anomaly_score", ascending=False)
    selected_rows.append(g.iloc[0])

top_by_channel_test = (
    pd.DataFrame(selected_rows)
    .sort_values("anomaly_score", ascending=False)
    .reset_index(drop=True)
)

# Representative XAI set:
# Take top 8 channels by flagged ratio, one highest-score flagged test window from each.
top8_channels = test_category_summary.head(8)["channel_code"].tolist()

xai_representative = (
    top_by_channel_test[top_by_channel_test["channel_code"].isin(top8_channels)]
    .copy()
    .sort_values("anomaly_score", ascending=False)
    .reset_index(drop=True)
)

print("\nTop overall TEST flagged windows:")
display(top_overall_test[base_cols + FEATURES].head(20))

print("\nTop one TEST flagged window per channel:")
display(top_by_channel_test[base_cols + FEATURES])

print("\nRepresentative XAI candidates from top 8 flagged-ratio channels:")
display(xai_representative[base_cols + FEATURES])

# Save candidate tables
top_overall_path = TABLE_DIR / "ampds2_xai_candidate_top_overall_test_windows.csv"
top_by_channel_path = TABLE_DIR / "ampds2_xai_candidate_top_by_channel_test_windows.csv"
xai_representative_path = TABLE_DIR / "ampds2_xai_representative_top8_channels_test_windows.csv"

top_overall_test[base_cols + FEATURES].to_csv(top_overall_path, index=False)
top_by_channel_test[base_cols + FEATURES].to_csv(top_by_channel_path, index=False)
xai_representative[base_cols + FEATURES].to_csv(xai_representative_path, index=False)

print("\nSaved tables:")
print(overall_summary_path)
print(category_summary_path)
print(test_category_summary_path)
print(top_overall_path)
print(top_by_channel_path)
print(xai_representative_path)

print("\nCell 6 complete.")

In [ ]:
# ============================================================
# EXP_AMPDS2_Selected_Pipeline
# Cell 7: SHAP and LIME explanations for AMPds2 selected test windows
# No synthetic injection
# Explains model-level reasons for selected flagged windows
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest

# ------------------------------------------------------------
# Optional installs/imports
# ------------------------------------------------------------

try:
    import shap
except Exception:
    !pip -q install shap
    import shap

try:
    from lime.lime_tabular import LimeTabularExplainer
except Exception:
    !pip -q install lime
    from lime.lime_tabular import LimeTabularExplainer

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

OUT_DIR = BASE / "processed_data" / "AMPDS2_Selected_Pipeline"
EXP_DIR = BASE / "experiments" / "EXP_AMPDS2_Selected_Pipeline"
TABLE_DIR = EXP_DIR / "tables"
XAI_DIR = EXP_DIR / "xai"

XAI_DIR.mkdir(parents=True, exist_ok=True)

SCORED_PATH = OUT_DIR / "ampds2_p_if_scored_no_injection.csv"
CANDIDATE_PATH = TABLE_DIR / "ampds2_xai_representative_top8_channels_test_windows.csv"

print("SCORED_PATH:", SCORED_PATH)
print("Exists:", SCORED_PATH.exists())

print("CANDIDATE_PATH:", CANDIDATE_PATH)
print("Exists:", CANDIDATE_PATH.exists())

df = pd.read_csv(SCORED_PATH)
candidates = pd.read_csv(CANDIDATE_PATH)

print("\nLoaded scored dataset:", df.shape)
print("Loaded XAI candidates:", candidates.shape)

# ------------------------------------------------------------
# Feature setup
# ------------------------------------------------------------

FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

SCALED_FEATURES = [f"{f}_z" for f in FEATURES]

missing_scaled = [f for f in SCALED_FEATURES if f not in df.columns]
print("\nMissing scaled features:", missing_scaled)

if missing_scaled:
    raise RuntimeError(f"Missing scaled features in scored dataset: {missing_scaled}")

# Clean timestamps for matching
df["window_start"] = pd.to_datetime(df["window_start"], utc=True, errors="coerce")
df["window_end"] = pd.to_datetime(df["window_end"], utc=True, errors="coerce")

candidates["window_start"] = pd.to_datetime(candidates["window_start"], utc=True, errors="coerce")
candidates["window_end"] = pd.to_datetime(candidates["window_end"], utc=True, errors="coerce")

# ------------------------------------------------------------
# Match candidate rows back to scored dataset
# ------------------------------------------------------------

match_keys = ["channel_code", "primary_category", "window_start", "window_end", "split"]

candidate_keys = candidates[match_keys].copy()
candidate_keys["candidate_order"] = np.arange(len(candidate_keys))

matched = candidate_keys.merge(
    df.reset_index().rename(columns={"index": "df_index"}),
    on=match_keys,
    how="left",
    suffixes=("", "_df")
)

if matched["df_index"].isna().any():
    display(matched[matched["df_index"].isna()])
    raise RuntimeError("Some XAI candidates could not be matched back to the scored dataset.")

matched = matched.sort_values("candidate_order").reset_index(drop=True)

df_indices = matched["df_index"].astype(int).tolist()
explain_df = df.loc[df_indices].copy().reset_index(drop=True)

print("\nMatched explain rows:", explain_df.shape)

display(
    explain_df[
        [
            "channel_code", "primary_category", "window_start", "window_end",
            "split", "anomaly_score", "if_threshold", "predicted_anomaly"
        ]
    ]
)

if not (explain_df["predicted_anomaly"] == 1).all():
    raise RuntimeError("At least one selected XAI candidate is not flagged as anomalous.")

# ------------------------------------------------------------
# Re-train Isolation Forest for explanation consistency
# Same setting as scoring cell
# ------------------------------------------------------------

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()

X_train = train_df[SCALED_FEATURES].values
X_val = val_df[SCALED_FEATURES].values
X_explain = explain_df[SCALED_FEATURES].values

print("\nX_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_explain shape:", X_explain.shape)

if_model = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

print("\nRe-training Isolation Forest for explanation consistency...")
if_model.fit(X_train)

# Recompute validation threshold exactly from the re-trained model
val_scores = -if_model.decision_function(X_val)
threshold = np.percentile(val_scores, 95)

print("Recomputed validation threshold:", threshold)

# Verify candidate scores from retrained model
explain_scores = -if_model.decision_function(X_explain)

score_check = explain_df[
    ["channel_code", "primary_category", "window_start", "window_end", "anomaly_score"]
].copy()

score_check["recomputed_anomaly_score"] = explain_scores
score_check["score_abs_diff"] = np.abs(score_check["anomaly_score"] - score_check["recomputed_anomaly_score"])

print("\nScore consistency check:")
display(score_check)

# ------------------------------------------------------------
# Prediction functions
# ------------------------------------------------------------

def model_score_fn(X):
    """
    Returns Isolation Forest anomaly score.
    Higher value = more anomalous.
    Used for SHAP regression-style explanation.
    """
    X = np.asarray(X)
    return -if_model.decision_function(X)

# LIME needs classifier-like probabilities.
# This is a monotonic pseudo-probability based on anomaly score relative to threshold.
val_score_std = np.std(val_scores)
temperature = val_score_std if val_score_std > 1e-9 else 1.0

def model_lime_proba_fn(X):
    X = np.asarray(X)
    scores = -if_model.decision_function(X)
    z = (scores - threshold) / temperature
    p_anomaly = 1.0 / (1.0 + np.exp(-z))
    p_normal = 1.0 - p_anomaly
    return np.vstack([p_normal, p_anomaly]).T

# ------------------------------------------------------------
# SHAP KernelExplainer
# Use sampled training background for speed.
# ------------------------------------------------------------

rng = np.random.default_rng(42)

background_size = min(100, len(X_train))
background_idx = rng.choice(len(X_train), size=background_size, replace=False)
X_background = X_train[background_idx]

print("\nRunning SHAP KernelExplainer...")
print("Background shape:", X_background.shape)
print("Explain shape:", X_explain.shape)

explainer = shap.KernelExplainer(model_score_fn, X_background)

shap_values = explainer.shap_values(X_explain, nsamples=300)

# SHAP can return list or ndarray depending on version
if isinstance(shap_values, list):
    shap_values = shap_values[0]

shap_values = np.asarray(shap_values)

print("SHAP values shape:", shap_values.shape)

# ------------------------------------------------------------
# Save SHAP explanations in long format
# ------------------------------------------------------------

shap_rows = []

for i in range(X_explain.shape[0]):
    row_meta = explain_df.iloc[i]

    for j, feature_z in enumerate(SCALED_FEATURES):
        raw_feature = FEATURES[j]

        shap_rows.append({
            "candidate_id": i,
            "dataset": "AMPds2",
            "channel_code": row_meta["channel_code"],
            "primary_category": row_meta["primary_category"],
            "window_start": row_meta["window_start"],
            "window_end": row_meta["window_end"],
            "anomaly_score": row_meta["anomaly_score"],
            "if_threshold": row_meta["if_threshold"],
            "feature": raw_feature,
            "scaled_feature": feature_z,
            "feature_value_raw": row_meta[raw_feature],
            "feature_value_scaled": row_meta[feature_z],
            "shap_value": shap_values[i, j],
            "abs_shap_value": abs(shap_values[i, j])
        })

shap_long = pd.DataFrame(shap_rows)

shap_top = (
    shap_long.sort_values(["candidate_id", "abs_shap_value"], ascending=[True, False])
    .groupby("candidate_id")
    .head(10)
    .reset_index(drop=True)
)

SHAP_LONG_PATH = XAI_DIR / "ampds2_shap_explanations_selected_test_windows.csv"
SHAP_TOP_PATH = XAI_DIR / "ampds2_shap_top10_per_candidate.csv"

shap_long.to_csv(SHAP_LONG_PATH, index=False)
shap_top.to_csv(SHAP_TOP_PATH, index=False)

print("\nSHAP top features per candidate:")
display(
    shap_top[
        [
            "candidate_id", "channel_code", "primary_category",
            "feature", "feature_value_raw", "feature_value_scaled",
            "shap_value", "abs_shap_value"
        ]
    ]
)

# ------------------------------------------------------------
# LIME explanations
# ------------------------------------------------------------

print("\nRunning LIME explanations...")

lime_explainer = LimeTabularExplainer(
    training_data=X_train,
    feature_names=SCALED_FEATURES,
    class_names=["normal_like", "flagged_like"],
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

lime_rows = []

for i in range(X_explain.shape[0]):
    row_meta = explain_df.iloc[i]

    exp = lime_explainer.explain_instance(
        data_row=X_explain[i],
        predict_fn=model_lime_proba_fn,
        num_features=10,
        labels=[1]
    )

    local_list = exp.as_list(label=1)

    for rank, (condition, weight) in enumerate(local_list, start=1):
        lime_rows.append({
            "candidate_id": i,
            "dataset": "AMPds2",
            "channel_code": row_meta["channel_code"],
            "primary_category": row_meta["primary_category"],
            "window_start": row_meta["window_start"],
            "window_end": row_meta["window_end"],
            "anomaly_score": row_meta["anomaly_score"],
            "if_threshold": row_meta["if_threshold"],
            "rank": rank,
            "lime_condition": condition,
            "lime_weight": weight,
            "abs_lime_weight": abs(weight)
        })

lime_long = pd.DataFrame(lime_rows)

LIME_PATH = XAI_DIR / "ampds2_lime_explanations_selected_test_windows.csv"
lime_long.to_csv(LIME_PATH, index=False)

print("\nLIME top conditions per candidate:")
display(lime_long)

# ------------------------------------------------------------
# Compact top-5 summary tables for thesis writing
# ------------------------------------------------------------

shap_top5_compact = (
    shap_top.sort_values(["candidate_id", "abs_shap_value"], ascending=[True, False])
    .groupby("candidate_id")
    .head(5)
    .groupby(["candidate_id", "channel_code", "primary_category"])
    .agg(
        top5_shap_features=("feature", lambda x: ", ".join(x)),
        top5_abs_shap=("abs_shap_value", lambda x: ", ".join([f"{v:.4f}" for v in x]))
    )
    .reset_index()
)

lime_top5_compact = (
    lime_long.sort_values(["candidate_id", "abs_lime_weight"], ascending=[True, False])
    .groupby("candidate_id")
    .head(5)
    .groupby(["candidate_id", "channel_code", "primary_category"])
    .agg(
        top5_lime_conditions=("lime_condition", lambda x: " | ".join(x)),
        top5_abs_lime=("abs_lime_weight", lambda x: ", ".join([f"{v:.4f}" for v in x]))
    )
    .reset_index()
)

SHAP_TOP5_COMPACT_PATH = XAI_DIR / "ampds2_shap_top5_per_candidate.csv"
LIME_TOP5_COMPACT_PATH = XAI_DIR / "ampds2_lime_top5_per_candidate.csv"

shap_top5_compact.to_csv(SHAP_TOP5_COMPACT_PATH, index=False)
lime_top5_compact.to_csv(LIME_TOP5_COMPACT_PATH, index=False)

print("\nCompact SHAP top-5 table:")
display(shap_top5_compact)

print("\nCompact LIME top-5 table:")
display(lime_top5_compact)

print("\nSaved XAI outputs:")
print(SHAP_LONG_PATH)
print(SHAP_TOP_PATH)
print(LIME_PATH)
print(SHAP_TOP5_COMPACT_PATH)
print(LIME_TOP5_COMPACT_PATH)

print("\nCell 7 complete.")

In [ ]:
# ============================================================
# Cell 0: Mount Google Drive and restore thesis paths
# ============================================================

from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

# Main thesis directory
BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

# Existing experiment directories
UKDALE_EXP_DIR = BASE / "experiments" / "EXP_UKDALE_Selected_Pipeline"
AMPDS2_EXP_DIR = BASE / "experiments" / "EXP_AMPDS2_Selected_Pipeline"

UKDALE_DATA_DIR = BASE / "processed_data" / "UKDALE_Selected_Pipeline"
AMPDS2_DATA_DIR = BASE / "processed_data" / "AMPDS2_Selected_Pipeline"

# New comparison experiment
COMPARISON_DIR = BASE / "experiments" / "EXP_UKDALE_AMPDS2_Cross_Model_Comparison"
COMPARISON_TABLE_DIR = COMPARISON_DIR / "tables"
COMPARISON_FIGURE_DIR = COMPARISON_DIR / "figures"
COMPARISON_MODEL_DIR = COMPARISON_DIR / "models"

for folder in [
    COMPARISON_DIR,
    COMPARISON_TABLE_DIR,
    COMPARISON_FIGURE_DIR,
    COMPARISON_MODEL_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Google Drive mounted successfully.")
print("BASE:", BASE)
print("BASE exists:", BASE.exists())

print("\nExisting experiment directories:")
print("UK-DALE:", UKDALE_EXP_DIR.exists(), UKDALE_EXP_DIR)
print("AMPds2:", AMPDS2_EXP_DIR.exists(), AMPDS2_EXP_DIR)

print("\nNew comparison directory:")
print(COMPARISON_DIR.exists(), COMPARISON_DIR)

In [ ]:
# ============================================================
# Cell 1: Verify the saved UK-DALE and AMPds2 outputs
# ============================================================

def show_csv_files(folder, label):
    print(f"\n{'=' * 70}")
    print(label)
    print(f"{'=' * 70}")

    if not folder.exists():
        print("Directory not found:", folder)
        return

    files = sorted(folder.rglob("*.csv"))

    print("CSV files found:", len(files))

    for path in files[:30]:
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"- {path.relative_to(BASE)} | {size_mb:.2f} MB")

    if len(files) > 30:
        print(f"... and {len(files) - 30} additional CSV files")


show_csv_files(UKDALE_DATA_DIR, "UK-DALE processed outputs")
show_csv_files(AMPDS2_DATA_DIR, "AMPds2 processed outputs")

In [ ]:
# ============================================================
# Cell 2: Inspect saved scored-file schemas before LOF
# ============================================================

import pandas as pd
from pathlib import Path

UKDALE_SCORED_PATH = (
    UKDALE_DATA_DIR /
    "ukdale_clean_main_if_scored_no_injection.csv"
)

AMPDS2_SCORED_PATH = (
    AMPDS2_DATA_DIR /
    "ampds2_p_if_scored_no_injection.csv"
)

FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

SCALED_FEATURES = [f"{feature}_z" for feature in FEATURES]


def inspect_scored_file(path, label):
    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)
    print("Path:", path)
    print("Exists:", path.exists())

    if not path.exists():
        return

    sample = pd.read_csv(path, nrows=5)

    print("\nShape of preview:", sample.shape)

    print("\nAll columns:")
    for i, col in enumerate(sample.columns, start=1):
        print(f"{i:02d}. {col}")

    print("\nScaled feature check:")
    present_scaled = [col for col in SCALED_FEATURES if col in sample.columns]
    missing_scaled = [col for col in SCALED_FEATURES if col not in sample.columns]

    print("Present scaled features:", len(present_scaled))
    print("Missing scaled features:", missing_scaled)

    likely_identifier_columns = [
        col for col in sample.columns
        if any(term in col.lower() for term in [
            "building",
            "meter",
            "channel",
            "category",
            "appliance",
            "series",
            "window",
            "split"
        ])
    ]

    print("\nLikely identifier/metadata columns:")
    print(likely_identifier_columns)

    print("\nPreview:")
    display(sample)


inspect_scored_file(
    UKDALE_SCORED_PATH,
    "UK-DALE scored dataset"
)

inspect_scored_file(
    AMPDS2_SCORED_PATH,
    "AMPds2 scored dataset"
)

In [ ]:
# ============================================================
# Cell 3: Build reproducible cross-model comparison samples
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import zlib
import gc

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

UKDALE_SCORED_PATH = (
    UKDALE_DATA_DIR /
    "ukdale_clean_main_if_scored_no_injection.csv"
)

AMPDS2_SCORED_PATH = (
    AMPDS2_DATA_DIR /
    "ampds2_p_if_scored_no_injection.csv"
)

COMPARISON_TABLE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Fixed comparison design
# ------------------------------------------------------------

RANDOM_STATE = 42

# LOF training reference per series
REFERENCE_PER_SERIES = 750

# Separate evaluation rows per series
TRAIN_EVAL_PER_SERIES = 250
VAL_EVAL_PER_SERIES = 500
TEST_EVAL_PER_SERIES = 500

FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

SCALED_FEATURES = [f"{feature}_z" for feature in FEATURES]


def stable_series_seed(series_key):
    """
    Produces the same deterministic seed for each series
    across notebook restarts.
    """
    return RANDOM_STATE + zlib.crc32(str(series_key).encode("utf-8"))


def sample_indices(indices, number_required, rng):
    indices = np.asarray(indices)

    if len(indices) == 0:
        return np.array([], dtype=np.int64)

    number_required = min(number_required, len(indices))

    return rng.choice(
        indices,
        size=number_required,
        replace=False
    )


def build_comparison_sample(
    scored_path,
    dataset_slug,
    dataset_label,
    metadata_columns
):
    print("\n" + "=" * 90)
    print(f"Building comparison sample: {dataset_label}")
    print("=" * 90)

    required_columns = (
        metadata_columns
        + [
            "window_start",
            "window_end",
            "series_key",
            "split",
            "anomaly_score",
            "predicted_anomaly"
        ]
        + SCALED_FEATURES
    )

    # Remove accidental duplicate column names
    required_columns = list(dict.fromkeys(required_columns))

    dtype_map = {
        feature: "float32"
        for feature in SCALED_FEATURES
    }

    dtype_map["anomaly_score"] = "float32"
    dtype_map["predicted_anomaly"] = "int8"

    print("Loading compact scored dataset...")

    df = pd.read_csv(
        scored_path,
        usecols=required_columns,
        dtype=dtype_map,
        low_memory=False
    )

    df["source_row_id"] = np.arange(len(df), dtype=np.int64)

    print("Loaded shape:", df.shape)
    print("Series:", df["series_key"].nunique())
    print("\nOriginal split counts:")
    print(df["split"].value_counts().sort_index())

    # --------------------------------------------------------
    # Validate feature matrix
    # --------------------------------------------------------

    feature_matrix = df[SCALED_FEATURES].to_numpy(dtype=np.float32)

    finite_mask = np.isfinite(feature_matrix).all(axis=1)
    invalid_count = int((~finite_mask).sum())

    print("\nRows with non-finite scaled features:", invalid_count)

    if invalid_count > 0:
        df = df.loc[finite_mask].copy()
        print("Rows retained after cleaning:", len(df))

    reference_parts = []
    evaluation_parts = []

    # --------------------------------------------------------
    # Sample independently within each series
    # --------------------------------------------------------

    for series_key, series_df in df.groupby("series_key", sort=True):

        rng = np.random.default_rng(
            stable_series_seed(series_key)
        )

        train_indices = series_df.index[
            series_df["split"] == "train"
        ].to_numpy()

        val_indices = series_df.index[
            series_df["split"] == "val"
        ].to_numpy()

        test_indices = series_df.index[
            series_df["split"] == "test"
        ].to_numpy()

        # Shuffle training rows once, then keep reference and
        # train-evaluation samples completely separate.
        shuffled_train = rng.permutation(train_indices)

        reference_count = min(
            REFERENCE_PER_SERIES,
            len(shuffled_train)
        )

        reference_indices = shuffled_train[:reference_count]

        remaining_train = shuffled_train[reference_count:]

        train_eval_count = min(
            TRAIN_EVAL_PER_SERIES,
            len(remaining_train)
        )

        train_eval_indices = remaining_train[:train_eval_count]

        val_eval_indices = sample_indices(
            val_indices,
            VAL_EVAL_PER_SERIES,
            rng
        )

        test_eval_indices = sample_indices(
            test_indices,
            TEST_EVAL_PER_SERIES,
            rng
        )

        reference_part = df.loc[reference_indices].copy()
        reference_part["sample_role"] = "lof_reference"

        reference_parts.append(reference_part)

        for split_name, selected_indices in [
            ("train", train_eval_indices),
            ("val", val_eval_indices),
            ("test", test_eval_indices)
        ]:
            evaluation_part = df.loc[selected_indices].copy()
            evaluation_part["evaluation_split"] = split_name
            evaluation_part["sample_role"] = "cross_model_evaluation"

            evaluation_parts.append(evaluation_part)

    reference_df = pd.concat(
        reference_parts,
        ignore_index=True
    )

    evaluation_df = pd.concat(
        evaluation_parts,
        ignore_index=True
    )

    # Stable ordering for reproducibility
    reference_df = reference_df.sort_values(
        ["series_key", "window_start", "source_row_id"]
    ).reset_index(drop=True)

    evaluation_df = evaluation_df.sort_values(
        [
            "evaluation_split",
            "series_key",
            "window_start",
            "source_row_id"
        ]
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------

    overlap = set(reference_df["source_row_id"]).intersection(
        set(
            evaluation_df.loc[
                evaluation_df["evaluation_split"] == "train",
                "source_row_id"
            ]
        )
    )

    print("\nReference/evaluation training overlap:", len(overlap))

    if overlap:
        raise RuntimeError(
            "LOF reference and train-evaluation samples overlap."
        )

    reference_path = (
        COMPARISON_TABLE_DIR /
        f"{dataset_slug}_lof_reference_sample.csv"
    )

    evaluation_path = (
        COMPARISON_TABLE_DIR /
        f"{dataset_slug}_cross_model_evaluation_sample.csv"
    )

    reference_df.to_csv(reference_path, index=False)
    evaluation_df.to_csv(evaluation_path, index=False)

    # --------------------------------------------------------
    # Output summaries
    # --------------------------------------------------------

    print("\nLOF reference sample:")
    print("Rows:", len(reference_df))
    print("Series:", reference_df["series_key"].nunique())
    print(
        reference_df.groupby("series_key")
        .size()
        .describe()
    )

    print("\nCross-model evaluation sample by split:")
    print(
        evaluation_df["evaluation_split"]
        .value_counts()
        .reindex(["train", "val", "test"])
    )

    print("\nEvaluation rows by split and series:")
    print(
        evaluation_df.groupby(
            ["evaluation_split", "series_key"]
        ).size().groupby(level=0).describe()
    )

    print("\nSaved:")
    print(reference_path)
    print(evaluation_path)

    summary = pd.DataFrame([
        {
            "dataset": dataset_label,
            "number_of_series": reference_df["series_key"].nunique(),
            "lof_reference_rows": len(reference_df),
            "train_evaluation_rows": int(
                (evaluation_df["evaluation_split"] == "train").sum()
            ),
            "validation_evaluation_rows": int(
                (evaluation_df["evaluation_split"] == "val").sum()
            ),
            "test_evaluation_rows": int(
                (evaluation_df["evaluation_split"] == "test").sum()
            ),
            "reference_per_series_target": REFERENCE_PER_SERIES,
            "train_eval_per_series_target": TRAIN_EVAL_PER_SERIES,
            "val_eval_per_series_target": VAL_EVAL_PER_SERIES,
            "test_eval_per_series_target": TEST_EVAL_PER_SERIES,
            "random_state": RANDOM_STATE
        }
    ])

    del df, feature_matrix
    gc.collect()

    return summary


# ------------------------------------------------------------
# UK-DALE
# ------------------------------------------------------------

ukdale_sample_summary = build_comparison_sample(
    scored_path=UKDALE_SCORED_PATH,
    dataset_slug="ukdale",
    dataset_label="UK-DALE",
    metadata_columns=[
        "dataset",
        "building",
        "meter",
        "primary_category"
    ]
)

# ------------------------------------------------------------
# AMPds2
# ------------------------------------------------------------

ampds2_sample_summary = build_comparison_sample(
    scored_path=AMPDS2_SCORED_PATH,
    dataset_slug="ampds2",
    dataset_label="AMPds2",
    metadata_columns=[
        "dataset",
        "channel_code",
        "channel_name",
        "primary_category"
    ]
)

comparison_sampling_summary = pd.concat(
    [ukdale_sample_summary, ampds2_sample_summary],
    ignore_index=True
)

SUMMARY_PATH = (
    COMPARISON_TABLE_DIR /
    "cross_model_sampling_summary.csv"
)

comparison_sampling_summary.to_csv(
    SUMMARY_PATH,
    index=False
)

print("\n" + "=" * 90)
print("FINAL CROSS-MODEL SAMPLING SUMMARY")
print("=" * 90)

display(comparison_sampling_summary)

print("\nSaved summary:")
print(SUMMARY_PATH)

In [ ]:
# ============================================================
# Cell 4: UK-DALE Local Outlier Factor
# Cross-model comparison — no synthetic injection
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import joblib
import gc

from sklearn.neighbors import LocalOutlierFactor
from scipy.stats import spearmanr

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

UKDALE_REFERENCE_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_lof_reference_sample.csv"
)

UKDALE_EVALUATION_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_cross_model_evaluation_sample.csv"
)

UKDALE_LOF_MODEL_PATH = (
    COMPARISON_MODEL_DIR /
    "ukdale_lof_novelty_model.joblib"
)

UKDALE_LOF_SCORED_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_lof_scored_evaluation.csv"
)

UKDALE_LOF_OVERALL_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_lof_overall_summary.csv"
)

UKDALE_LOF_CATEGORY_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_lof_test_summary_by_category.csv"
)

UKDALE_IF_LOF_AGREEMENT_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_if_lof_agreement_by_split.csv"
)

UKDALE_LOF_TOP_WINDOWS_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_lof_top_test_windows.csv"
)

# ------------------------------------------------------------
# Fixed feature configuration
# ------------------------------------------------------------

FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

SCALED_FEATURES = [f"{feature}_z" for feature in FEATURES]

# ------------------------------------------------------------
# LOF configuration
# ------------------------------------------------------------

LOF_N_NEIGHBORS = 35
LOF_METRIC = "euclidean"

print("=" * 90)
print("UK-DALE LOF CROSS-MODEL EXPERIMENT")
print("=" * 90)

print("Reference file:", UKDALE_REFERENCE_PATH)
print("Evaluation file:", UKDALE_EVALUATION_PATH)

reference_df = pd.read_csv(
    UKDALE_REFERENCE_PATH,
    low_memory=False
)

evaluation_df = pd.read_csv(
    UKDALE_EVALUATION_PATH,
    low_memory=False
)

print("\nReference shape:", reference_df.shape)
print("Evaluation shape:", evaluation_df.shape)

# ------------------------------------------------------------
# Data validation
# ------------------------------------------------------------

missing_reference = [
    column for column in SCALED_FEATURES
    if column not in reference_df.columns
]

missing_evaluation = [
    column for column in SCALED_FEATURES
    if column not in evaluation_df.columns
]

if missing_reference:
    raise RuntimeError(
        f"Missing reference features: {missing_reference}"
    )

if missing_evaluation:
    raise RuntimeError(
        f"Missing evaluation features: {missing_evaluation}"
    )

X_reference = reference_df[
    SCALED_FEATURES
].to_numpy(dtype=np.float32)

X_evaluation = evaluation_df[
    SCALED_FEATURES
].to_numpy(dtype=np.float32)

if not np.isfinite(X_reference).all():
    raise RuntimeError(
        "Reference feature matrix contains NaN or infinite values."
    )

if not np.isfinite(X_evaluation).all():
    raise RuntimeError(
        "Evaluation feature matrix contains NaN or infinite values."
    )

print("\nReference matrix:", X_reference.shape)
print("Evaluation matrix:", X_evaluation.shape)

print("\nEvaluation split counts:")
print(
    evaluation_df["evaluation_split"]
    .value_counts()
    .reindex(["train", "val", "test"])
)

# ------------------------------------------------------------
# Fit novelty-detection LOF
# ------------------------------------------------------------

lof_model = LocalOutlierFactor(
    n_neighbors=LOF_N_NEIGHBORS,
    algorithm="auto",
    metric=LOF_METRIC,
    contamination="auto",
    novelty=True,
    n_jobs=-1
)

print("\nTraining LOF...")
fit_start = time.perf_counter()

lof_model.fit(X_reference)

fit_seconds = time.perf_counter() - fit_start

print(f"LOF training completed in {fit_seconds:.2f} seconds.")

joblib.dump(
    lof_model,
    UKDALE_LOF_MODEL_PATH
)

print("Saved model:")
print(UKDALE_LOF_MODEL_PATH)

# ------------------------------------------------------------
# Score evaluation rows
# Higher score = more anomalous
# ------------------------------------------------------------

print("\nScoring evaluation sample...")
score_start = time.perf_counter()

# score_samples is higher for normal observations.
# Negating it creates an anomaly score where higher = more unusual.
lof_scores = -lof_model.score_samples(X_evaluation)

score_seconds = time.perf_counter() - score_start

evaluation_df["lof_anomaly_score"] = lof_scores.astype(
    np.float32
)

print(f"Scoring completed in {score_seconds:.2f} seconds.")

# ------------------------------------------------------------
# Validation-based threshold
# ------------------------------------------------------------

validation_mask = (
    evaluation_df["evaluation_split"] == "val"
)

validation_scores = evaluation_df.loc[
    validation_mask,
    "lof_anomaly_score"
].to_numpy()

LOF_THRESHOLD_PERCENTILE = 95

lof_threshold = np.percentile(
    validation_scores,
    LOF_THRESHOLD_PERCENTILE
)

evaluation_df["lof_threshold"] = np.float32(
    lof_threshold
)

evaluation_df["lof_predicted_anomaly"] = (
    evaluation_df["lof_anomaly_score"] >
    lof_threshold
).astype(np.int8)

print("\nValidation threshold percentile:",
      LOF_THRESHOLD_PERCENTILE)

print("LOF validation threshold:",
      lof_threshold)

# ------------------------------------------------------------
# Overall summary by split
# ------------------------------------------------------------

overall_rows = []

for split_name in ["train", "val", "test"]:

    split_df = evaluation_df[
        evaluation_df["evaluation_split"] == split_name
    ]

    flagged_count = int(
        split_df["lof_predicted_anomaly"].sum()
    )

    total_count = len(split_df)

    overall_rows.append({
        "dataset": "UK-DALE",
        "model": "Local Outlier Factor",
        "split": split_name,
        "windows": total_count,
        "flagged_windows": flagged_count,
        "flagged_ratio": (
            flagged_count / total_count
            if total_count else np.nan
        ),
        "flagged_ratio_percent": (
            100 * flagged_count / total_count
            if total_count else np.nan
        ),
        "mean_anomaly_score": (
            split_df["lof_anomaly_score"].mean()
        ),
        "median_anomaly_score": (
            split_df["lof_anomaly_score"].median()
        ),
        "maximum_anomaly_score": (
            split_df["lof_anomaly_score"].max()
        ),
        "validation_threshold": lof_threshold,
        "threshold_percentile": LOF_THRESHOLD_PERCENTILE,
        "n_neighbors": LOF_N_NEIGHBORS,
        "reference_rows": len(reference_df),
        "fit_seconds": fit_seconds,
        "score_seconds": score_seconds
    })

overall_summary = pd.DataFrame(overall_rows)

print("\nLOF overall summary:")
display(overall_summary)

# ------------------------------------------------------------
# Test summary by appliance category
# ------------------------------------------------------------

test_df = evaluation_df[
    evaluation_df["evaluation_split"] == "test"
].copy()

category_summary = (
    test_df.groupby(
        "primary_category",
        dropna=False
    )
    .agg(
        test_windows=(
            "lof_predicted_anomaly",
            "size"
        ),
        flagged_windows=(
            "lof_predicted_anomaly",
            "sum"
        ),
        mean_lof_score=(
            "lof_anomaly_score",
            "mean"
        ),
        median_lof_score=(
            "lof_anomaly_score",
            "median"
        ),
        maximum_lof_score=(
            "lof_anomaly_score",
            "max"
        )
    )
    .reset_index()
)

category_summary["flagged_ratio"] = (
    category_summary["flagged_windows"] /
    category_summary["test_windows"]
)

category_summary["flagged_ratio_percent"] = (
    100 * category_summary["flagged_ratio"]
)

category_summary = category_summary.sort_values(
    "flagged_ratio",
    ascending=False
).reset_index(drop=True)

print("\nUK-DALE LOF test summary by category:")
display(category_summary)

# ------------------------------------------------------------
# Isolation Forest versus LOF agreement
# ------------------------------------------------------------

agreement_rows = []

for split_name in ["train", "val", "test"]:

    split_df = evaluation_df[
        evaluation_df["evaluation_split"] == split_name
    ].copy()

    if_flags = (
        split_df["predicted_anomaly"]
        .astype(int)
        .to_numpy()
    )

    lof_flags = (
        split_df["lof_predicted_anomaly"]
        .astype(int)
        .to_numpy()
    )

    both_flagged = int(
        ((if_flags == 1) & (lof_flags == 1)).sum()
    )

    if_only = int(
        ((if_flags == 1) & (lof_flags == 0)).sum()
    )

    lof_only = int(
        ((if_flags == 0) & (lof_flags == 1)).sum()
    )

    neither = int(
        ((if_flags == 0) & (lof_flags == 0)).sum()
    )

    union_count = both_flagged + if_only + lof_only

    jaccard = (
        both_flagged / union_count
        if union_count > 0 else np.nan
    )

    agreement_rate = (
        both_flagged + neither
    ) / len(split_df)

    spearman_result = spearmanr(
        split_df["anomaly_score"],
        split_df["lof_anomaly_score"]
    )

    agreement_rows.append({
        "dataset": "UK-DALE",
        "split": split_name,
        "windows": len(split_df),
        "if_flagged": int(if_flags.sum()),
        "lof_flagged": int(lof_flags.sum()),
        "both_flagged": both_flagged,
        "if_only": if_only,
        "lof_only": lof_only,
        "neither_flagged": neither,
        "flag_agreement_rate": agreement_rate,
        "flag_jaccard_index": jaccard,
        "score_spearman_correlation": (
            spearman_result.statistic
        ),
        "score_spearman_p_value": (
            spearman_result.pvalue
        )
    })

agreement_summary = pd.DataFrame(
    agreement_rows
)

print("\nIsolation Forest versus LOF agreement:")
display(agreement_summary)

# ------------------------------------------------------------
# Highest-scoring LOF test windows
# ------------------------------------------------------------

top_columns = [
    "source_row_id",
    "building",
    "meter",
    "series_key",
    "primary_category",
    "window_start",
    "window_end",
    "lof_anomaly_score",
    "lof_threshold",
    "lof_predicted_anomaly",
    "anomaly_score",
    "predicted_anomaly"
]

top_test_windows = (
    test_df.sort_values(
        "lof_anomaly_score",
        ascending=False
    )
    .head(30)[top_columns]
    .reset_index(drop=True)
)

print("\nTop 30 UK-DALE LOF test windows:")
display(top_test_windows)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

evaluation_df.to_csv(
    UKDALE_LOF_SCORED_PATH,
    index=False
)

overall_summary.to_csv(
    UKDALE_LOF_OVERALL_PATH,
    index=False
)

category_summary.to_csv(
    UKDALE_LOF_CATEGORY_PATH,
    index=False
)

agreement_summary.to_csv(
    UKDALE_IF_LOF_AGREEMENT_PATH,
    index=False
)

top_test_windows.to_csv(
    UKDALE_LOF_TOP_WINDOWS_PATH,
    index=False
)

print("\nSaved outputs:")
print(UKDALE_LOF_SCORED_PATH)
print(UKDALE_LOF_OVERALL_PATH)
print(UKDALE_LOF_CATEGORY_PATH)
print(UKDALE_IF_LOF_AGREEMENT_PATH)
print(UKDALE_LOF_TOP_WINDOWS_PATH)

print("\nCell 4 complete.")

del X_reference, X_evaluation
gc.collect()

In [ ]:
# ============================================================
# Cell 5: UK-DALE LOF stability and duplicate diagnostics
# ============================================================

import pandas as pd
import numpy as np
import time
from itertools import combinations

from sklearn.neighbors import LocalOutlierFactor
from scipy.stats import spearmanr

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

UKDALE_REFERENCE_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_lof_reference_sample.csv"
)

UKDALE_EVALUATION_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_cross_model_evaluation_sample.csv"
)

DUPLICATE_SUMMARY_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_lof_reference_duplicate_summary.csv"
)

SENSITIVITY_SUMMARY_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_lof_neighbor_sensitivity_summary.csv"
)

PAIRWISE_STABILITY_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_lof_pairwise_stability.csv"
)

IF_LOF_SENSITIVITY_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_if_lof_sensitivity_agreement.csv"
)

# ------------------------------------------------------------
# Features
# ------------------------------------------------------------

FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

SCALED_FEATURES = [f"{feature}_z" for feature in FEATURES]

reference_df = pd.read_csv(
    UKDALE_REFERENCE_PATH,
    low_memory=False
)

evaluation_df = pd.read_csv(
    UKDALE_EVALUATION_PATH,
    low_memory=False
)

# Use float64 for this diagnostic.
X_reference = reference_df[
    SCALED_FEATURES
].to_numpy(dtype=np.float64)

X_evaluation = evaluation_df[
    SCALED_FEATURES
].to_numpy(dtype=np.float64)

print("Reference shape:", X_reference.shape)
print("Evaluation shape:", X_evaluation.shape)

# ============================================================
# 1. Duplicate and near-duplicate diagnostics
# ============================================================

duplicate_rows = []

for series_key, group in reference_df.groupby(
    "series_key",
    sort=True
):
    exact_unique = len(
        group[SCALED_FEATURES].drop_duplicates()
    )

    rounded_unique = len(
        group[SCALED_FEATURES]
        .round(6)
        .drop_duplicates()
    )

    total = len(group)

    duplicate_rows.append({
        "series_key": series_key,
        "reference_rows": total,
        "exact_unique_rows": exact_unique,
        "exact_duplicate_rows": total - exact_unique,
        "exact_duplicate_ratio": (
            (total - exact_unique) / total
        ),
        "rounded6_unique_rows": rounded_unique,
        "rounded6_duplicate_rows": total - rounded_unique,
        "rounded6_duplicate_ratio": (
            (total - rounded_unique) / total
        )
    })

duplicate_summary = pd.DataFrame(
    duplicate_rows
).sort_values(
    "rounded6_duplicate_ratio",
    ascending=False
)

print("\nReference duplicate summary:")
display(duplicate_summary)

print("\nOverall exact duplicates:")
print(
    reference_df[SCALED_FEATURES]
    .duplicated()
    .sum()
)

print("\nOverall duplicates after rounding to 6 decimals:")
print(
    reference_df[SCALED_FEATURES]
    .round(6)
    .duplicated()
    .sum()
)

# ============================================================
# 2. LOF neighbor sensitivity
# ============================================================

NEIGHBOR_VALUES = [20, 35, 50, 75]

model_outputs = {}
sensitivity_rows = []

for n_neighbors in NEIGHBOR_VALUES:

    print("\n" + "=" * 80)
    print(f"Running LOF with n_neighbors={n_neighbors}")
    print("=" * 80)

    model = LocalOutlierFactor(
        n_neighbors=n_neighbors,
        novelty=True,
        contamination="auto",
        metric="euclidean",
        n_jobs=-1
    )

    fit_start = time.perf_counter()
    model.fit(X_reference)
    fit_seconds = time.perf_counter() - fit_start

    score_start = time.perf_counter()
    scores = -model.score_samples(X_evaluation)
    score_seconds = time.perf_counter() - score_start

    validation_mask = (
        evaluation_df["evaluation_split"] == "val"
    )

    threshold = np.percentile(
        scores[validation_mask],
        95
    )

    flags = (scores > threshold).astype(np.int8)

    model_outputs[n_neighbors] = {
        "scores": scores,
        "flags": flags,
        "threshold": threshold
    }

    for split_name in ["train", "val", "test"]:

        split_mask = (
            evaluation_df["evaluation_split"] ==
            split_name
        )

        split_scores = scores[split_mask]
        split_flags = flags[split_mask]

        sensitivity_rows.append({
            "n_neighbors": n_neighbors,
            "split": split_name,
            "windows": int(split_mask.sum()),
            "flagged_windows": int(split_flags.sum()),
            "flagged_ratio": float(split_flags.mean()),
            "flagged_ratio_percent": float(
                100 * split_flags.mean()
            ),
            "threshold": float(threshold),
            "score_median": float(
                np.median(split_scores)
            ),
            "score_q95": float(
                np.percentile(split_scores, 95)
            ),
            "score_q99": float(
                np.percentile(split_scores, 99)
            ),
            "score_q999": float(
                np.percentile(split_scores, 99.9)
            ),
            "score_maximum": float(
                np.max(split_scores)
            ),
            "scores_above_1000": int(
                (split_scores > 1e3).sum()
            ),
            "scores_above_1million": int(
                (split_scores > 1e6).sum()
            ),
            "fit_seconds": fit_seconds,
            "score_seconds": score_seconds
        })

sensitivity_summary = pd.DataFrame(
    sensitivity_rows
)

print("\nLOF neighbor sensitivity summary:")
display(sensitivity_summary)

# ============================================================
# 3. Pairwise LOF stability
# ============================================================

test_mask = (
    evaluation_df["evaluation_split"] == "test"
)

pairwise_rows = []

for k1, k2 in combinations(
    NEIGHBOR_VALUES,
    2
):
    scores_1 = model_outputs[k1]["scores"][test_mask]
    scores_2 = model_outputs[k2]["scores"][test_mask]

    flags_1 = model_outputs[k1]["flags"][test_mask]
    flags_2 = model_outputs[k2]["flags"][test_mask]

    both = int(
        ((flags_1 == 1) & (flags_2 == 1)).sum()
    )

    union = int(
        ((flags_1 == 1) | (flags_2 == 1)).sum()
    )

    jaccard = (
        both / union
        if union > 0 else np.nan
    )

    rank_corr = spearmanr(
        scores_1,
        scores_2
    ).statistic

    pairwise_rows.append({
        "neighbors_1": k1,
        "neighbors_2": k2,
        "test_flagged_1": int(flags_1.sum()),
        "test_flagged_2": int(flags_2.sum()),
        "both_flagged": both,
        "flag_union": union,
        "flag_jaccard_index": jaccard,
        "score_spearman_correlation": rank_corr
    })

pairwise_stability = pd.DataFrame(
    pairwise_rows
)

print("\nPairwise LOF stability:")
display(pairwise_stability)

# ============================================================
# 4. Isolation Forest versus each LOF setting
# ============================================================

if_flags_test = (
    evaluation_df.loc[
        test_mask,
        "predicted_anomaly"
    ]
    .astype(int)
    .to_numpy()
)

if_scores_test = (
    evaluation_df.loc[
        test_mask,
        "anomaly_score"
    ]
    .to_numpy()
)

agreement_rows = []

for n_neighbors in NEIGHBOR_VALUES:

    lof_scores_test = model_outputs[
        n_neighbors
    ]["scores"][test_mask]

    lof_flags_test = model_outputs[
        n_neighbors
    ]["flags"][test_mask]

    both = int(
        (
            (if_flags_test == 1) &
            (lof_flags_test == 1)
        ).sum()
    )

    union = int(
        (
            (if_flags_test == 1) |
            (lof_flags_test == 1)
        ).sum()
    )

    agreement_rows.append({
        "n_neighbors": n_neighbors,
        "if_flagged": int(
            if_flags_test.sum()
        ),
        "lof_flagged": int(
            lof_flags_test.sum()
        ),
        "both_flagged": both,
        "flag_union": union,
        "flag_jaccard_index": (
            both / union
            if union > 0 else np.nan
        ),
        "score_spearman_correlation": (
            spearmanr(
                if_scores_test,
                lof_scores_test
            ).statistic
        )
    })

if_lof_sensitivity = pd.DataFrame(
    agreement_rows
)

print("\nIsolation Forest versus LOF sensitivity:")
display(if_lof_sensitivity)

# ============================================================
# Save diagnostics
# ============================================================

duplicate_summary.to_csv(
    DUPLICATE_SUMMARY_PATH,
    index=False
)

sensitivity_summary.to_csv(
    SENSITIVITY_SUMMARY_PATH,
    index=False
)

pairwise_stability.to_csv(
    PAIRWISE_STABILITY_PATH,
    index=False
)

if_lof_sensitivity.to_csv(
    IF_LOF_SENSITIVITY_PATH,
    index=False
)

print("\nSaved diagnostic outputs:")
print(DUPLICATE_SUMMARY_PATH)
print(SENSITIVITY_SUMMARY_PATH)
print(PAIRWISE_STABILITY_PATH)
print(IF_LOF_SENSITIVITY_PATH)

print("\nCell 5 complete.")

In [ ]:
# ============================================================
# Cell 6: AMPds2 Local Outlier Factor
# Fixed transfer from UK-DALE — no synthetic injection
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import joblib
import gc

from sklearn.neighbors import LocalOutlierFactor
from scipy.stats import spearmanr

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

AMPDS2_REFERENCE_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_lof_reference_sample.csv"
)

AMPDS2_EVALUATION_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_cross_model_evaluation_sample.csv"
)

AMPDS2_LOF_MODEL_PATH = (
    COMPARISON_MODEL_DIR /
    "ampds2_lof_novelty_model.joblib"
)

AMPDS2_LOF_SCORED_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_lof_scored_evaluation.csv"
)

AMPDS2_LOF_OVERALL_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_lof_overall_summary.csv"
)

AMPDS2_LOF_CATEGORY_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_lof_test_summary_by_category.csv"
)

AMPDS2_IF_LOF_AGREEMENT_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_if_lof_agreement_by_split.csv"
)

AMPDS2_LOF_TOP_WINDOWS_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_lof_top_test_windows.csv"
)

AMPDS2_DUPLICATE_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_lof_reference_duplicate_summary.csv"
)

# ------------------------------------------------------------
# Features and fixed configuration
# ------------------------------------------------------------

FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

SCALED_FEATURES = [f"{feature}_z" for feature in FEATURES]

LOF_N_NEIGHBORS = 35
LOF_THRESHOLD_PERCENTILE = 95

print("=" * 90)
print("AMPds2 LOF CROSS-MODEL EXPERIMENT")
print("=" * 90)

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------

reference_df = pd.read_csv(
    AMPDS2_REFERENCE_PATH,
    low_memory=False
)

evaluation_df = pd.read_csv(
    AMPDS2_EVALUATION_PATH,
    low_memory=False
)

print("Reference shape:", reference_df.shape)
print("Evaluation shape:", evaluation_df.shape)

print("\nEvaluation split counts:")
print(
    evaluation_df["evaluation_split"]
    .value_counts()
    .reindex(["train", "val", "test"])
)

X_reference = reference_df[
    SCALED_FEATURES
].to_numpy(dtype=np.float64)

X_evaluation = evaluation_df[
    SCALED_FEATURES
].to_numpy(dtype=np.float64)

if not np.isfinite(X_reference).all():
    raise RuntimeError(
        "AMPds2 reference contains non-finite values."
    )

if not np.isfinite(X_evaluation).all():
    raise RuntimeError(
        "AMPds2 evaluation sample contains non-finite values."
    )

# ------------------------------------------------------------
# Reference duplicate diagnostic
# ------------------------------------------------------------

duplicate_rows = []

for series_key, group in reference_df.groupby(
    "series_key",
    sort=True
):
    total = len(group)

    exact_unique = len(
        group[SCALED_FEATURES].drop_duplicates()
    )

    rounded_unique = len(
        group[SCALED_FEATURES]
        .round(6)
        .drop_duplicates()
    )

    duplicate_rows.append({
        "series_key": series_key,
        "reference_rows": total,
        "exact_unique_rows": exact_unique,
        "exact_duplicate_rows": total - exact_unique,
        "exact_duplicate_ratio": (
            (total - exact_unique) / total
        ),
        "rounded6_unique_rows": rounded_unique,
        "rounded6_duplicate_rows": total - rounded_unique,
        "rounded6_duplicate_ratio": (
            (total - rounded_unique) / total
        )
    })

duplicate_summary = (
    pd.DataFrame(duplicate_rows)
    .sort_values(
        "rounded6_duplicate_ratio",
        ascending=False
    )
    .reset_index(drop=True)
)

overall_duplicate_count = int(
    reference_df[SCALED_FEATURES]
    .duplicated()
    .sum()
)

overall_duplicate_ratio = (
    overall_duplicate_count /
    len(reference_df)
)

print("\nAMPds2 reference duplicate summary:")
display(duplicate_summary)

print("\nOverall duplicate rows:",
      overall_duplicate_count)

print(
    "Overall duplicate ratio:",
    f"{100 * overall_duplicate_ratio:.2f}%"
)

# ------------------------------------------------------------
# Fit LOF
# ------------------------------------------------------------

lof_model = LocalOutlierFactor(
    n_neighbors=LOF_N_NEIGHBORS,
    algorithm="auto",
    metric="euclidean",
    contamination="auto",
    novelty=True,
    n_jobs=-1
)

print("\nTraining AMPds2 LOF...")

fit_start = time.perf_counter()

lof_model.fit(X_reference)

fit_seconds = time.perf_counter() - fit_start

print(
    f"Training completed in {fit_seconds:.2f} seconds."
)

joblib.dump(
    lof_model,
    AMPDS2_LOF_MODEL_PATH
)

# ------------------------------------------------------------
# Score evaluation rows
# ------------------------------------------------------------

print("\nScoring AMPds2 evaluation sample...")

score_start = time.perf_counter()

lof_scores = -lof_model.score_samples(
    X_evaluation
)

score_seconds = time.perf_counter() - score_start

evaluation_df["lof_anomaly_score"] = (
    lof_scores.astype(np.float64)
)

# Transformation for future plotting only.
# It does not change ranking or threshold decisions.
evaluation_df["lof_log1p_score"] = np.log1p(
    np.maximum(
        evaluation_df["lof_anomaly_score"],
        0
    )
)

print(
    f"Scoring completed in {score_seconds:.2f} seconds."
)

# ------------------------------------------------------------
# Validation threshold
# ------------------------------------------------------------

validation_mask = (
    evaluation_df["evaluation_split"] == "val"
)

validation_scores = evaluation_df.loc[
    validation_mask,
    "lof_anomaly_score"
].to_numpy()

lof_threshold = np.percentile(
    validation_scores,
    LOF_THRESHOLD_PERCENTILE
)

evaluation_df["lof_threshold"] = lof_threshold

evaluation_df["lof_predicted_anomaly"] = (
    evaluation_df["lof_anomaly_score"] >
    lof_threshold
).astype(np.int8)

print("\nLOF threshold:", lof_threshold)

# ------------------------------------------------------------
# Overall summary
# ------------------------------------------------------------

overall_rows = []

for split_name in ["train", "val", "test"]:

    split_df = evaluation_df[
        evaluation_df["evaluation_split"] ==
        split_name
    ]

    flagged_count = int(
        split_df["lof_predicted_anomaly"].sum()
    )

    total_count = len(split_df)

    overall_rows.append({
        "dataset": "AMPds2",
        "model": "Local Outlier Factor",
        "split": split_name,
        "windows": total_count,
        "flagged_windows": flagged_count,
        "flagged_ratio": (
            flagged_count / total_count
        ),
        "flagged_ratio_percent": (
            100 * flagged_count / total_count
        ),
        "median_anomaly_score": (
            split_df["lof_anomaly_score"].median()
        ),
        "score_q95": (
            split_df["lof_anomaly_score"]
            .quantile(0.95)
        ),
        "score_q99": (
            split_df["lof_anomaly_score"]
            .quantile(0.99)
        ),
        "maximum_anomaly_score": (
            split_df["lof_anomaly_score"].max()
        ),
        "validation_threshold": lof_threshold,
        "threshold_percentile": (
            LOF_THRESHOLD_PERCENTILE
        ),
        "n_neighbors": LOF_N_NEIGHBORS,
        "reference_rows": len(reference_df),
        "fit_seconds": fit_seconds,
        "score_seconds": score_seconds
    })

overall_summary = pd.DataFrame(
    overall_rows
)

print("\nAMPds2 LOF overall summary:")
display(overall_summary)

# ------------------------------------------------------------
# Test summary by channel/category
# ------------------------------------------------------------

test_df = evaluation_df[
    evaluation_df["evaluation_split"] == "test"
].copy()

category_summary = (
    test_df.groupby(
        [
            "channel_code",
            "channel_name",
            "primary_category"
        ],
        dropna=False
    )
    .agg(
        test_windows=(
            "lof_predicted_anomaly",
            "size"
        ),
        flagged_windows=(
            "lof_predicted_anomaly",
            "sum"
        ),
        mean_lof_score=(
            "lof_anomaly_score",
            "mean"
        ),
        median_lof_score=(
            "lof_anomaly_score",
            "median"
        ),
        maximum_lof_score=(
            "lof_anomaly_score",
            "max"
        )
    )
    .reset_index()
)

category_summary["flagged_ratio"] = (
    category_summary["flagged_windows"] /
    category_summary["test_windows"]
)

category_summary["flagged_ratio_percent"] = (
    100 * category_summary["flagged_ratio"]
)

category_summary = category_summary.sort_values(
    "flagged_ratio",
    ascending=False
).reset_index(drop=True)

print("\nAMPds2 LOF test summary by channel:")
display(category_summary)

# ------------------------------------------------------------
# Isolation Forest versus LOF agreement
# ------------------------------------------------------------

agreement_rows = []

for split_name in ["train", "val", "test"]:

    split_df = evaluation_df[
        evaluation_df["evaluation_split"] ==
        split_name
    ]

    if_flags = (
        split_df["predicted_anomaly"]
        .astype(int)
        .to_numpy()
    )

    lof_flags = (
        split_df["lof_predicted_anomaly"]
        .astype(int)
        .to_numpy()
    )

    both_flagged = int(
        (
            (if_flags == 1) &
            (lof_flags == 1)
        ).sum()
    )

    if_only = int(
        (
            (if_flags == 1) &
            (lof_flags == 0)
        ).sum()
    )

    lof_only = int(
        (
            (if_flags == 0) &
            (lof_flags == 1)
        ).sum()
    )

    neither = int(
        (
            (if_flags == 0) &
            (lof_flags == 0)
        ).sum()
    )

    union_count = (
        both_flagged +
        if_only +
        lof_only
    )

    rank_correlation = spearmanr(
        split_df["anomaly_score"],
        split_df["lof_anomaly_score"]
    )

    agreement_rows.append({
        "dataset": "AMPds2",
        "split": split_name,
        "windows": len(split_df),
        "if_flagged": int(if_flags.sum()),
        "lof_flagged": int(lof_flags.sum()),
        "both_flagged": both_flagged,
        "if_only": if_only,
        "lof_only": lof_only,
        "neither_flagged": neither,
        "flag_agreement_rate": (
            (both_flagged + neither) /
            len(split_df)
        ),
        "flag_jaccard_index": (
            both_flagged / union_count
            if union_count > 0
            else np.nan
        ),
        "score_spearman_correlation": (
            rank_correlation.statistic
        ),
        "score_spearman_p_value": (
            rank_correlation.pvalue
        )
    })

agreement_summary = pd.DataFrame(
    agreement_rows
)

print("\nIsolation Forest versus LOF agreement:")
display(agreement_summary)

# ------------------------------------------------------------
# Highest-ranked LOF test windows
# ------------------------------------------------------------

top_columns = [
    "source_row_id",
    "channel_code",
    "channel_name",
    "series_key",
    "primary_category",
    "window_start",
    "window_end",
    "lof_anomaly_score",
    "lof_log1p_score",
    "lof_threshold",
    "lof_predicted_anomaly",
    "anomaly_score",
    "predicted_anomaly"
]

top_test_windows = (
    test_df.sort_values(
        "lof_anomaly_score",
        ascending=False
    )
    .head(30)[top_columns]
    .reset_index(drop=True)
)

print("\nTop 30 AMPds2 LOF test windows:")
display(top_test_windows)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

evaluation_df.to_csv(
    AMPDS2_LOF_SCORED_PATH,
    index=False
)

overall_summary.to_csv(
    AMPDS2_LOF_OVERALL_PATH,
    index=False
)

category_summary.to_csv(
    AMPDS2_LOF_CATEGORY_PATH,
    index=False
)

agreement_summary.to_csv(
    AMPDS2_IF_LOF_AGREEMENT_PATH,
    index=False
)

top_test_windows.to_csv(
    AMPDS2_LOF_TOP_WINDOWS_PATH,
    index=False
)

duplicate_summary.to_csv(
    AMPDS2_DUPLICATE_PATH,
    index=False
)

print("\nSaved outputs:")
print(AMPDS2_LOF_SCORED_PATH)
print(AMPDS2_LOF_OVERALL_PATH)
print(AMPDS2_LOF_CATEGORY_PATH)
print(AMPDS2_IF_LOF_AGREEMENT_PATH)
print(AMPDS2_LOF_TOP_WINDOWS_PATH)
print(AMPDS2_DUPLICATE_PATH)

print("\nCell 6 complete.")

del X_reference, X_evaluation
gc.collect()

In [ ]:
# ============================================================
# Cell 7: UK-DALE Dense Autoencoder
# Cross-model comparison — no synthetic injection
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import random
import gc

from scipy.stats import spearmanr
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except Exception as exc:
    print("Deterministic operations could not be fully enabled:", exc)

tf.keras.backend.clear_session()

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

UKDALE_REFERENCE_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_lof_reference_sample.csv"
)

# This file already contains the saved IF and LOF outputs
UKDALE_LOF_SCORED_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_lof_scored_evaluation.csv"
)

UKDALE_AE_MODEL_PATH = (
    COMPARISON_MODEL_DIR /
    "ukdale_dense_autoencoder.keras"
)

UKDALE_AE_SCORED_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_if_lof_ae_scored_evaluation.csv"
)

UKDALE_AE_HISTORY_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_autoencoder_training_history.csv"
)

UKDALE_AE_OVERALL_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_autoencoder_overall_summary.csv"
)

UKDALE_AE_CATEGORY_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_autoencoder_test_summary_by_category.csv"
)

UKDALE_AE_AGREEMENT_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_three_model_agreement_by_split.csv"
)

UKDALE_AE_TOP_WINDOWS_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_autoencoder_top_test_windows.csv"
)

UKDALE_AE_TRAINING_FIGURE_PATH = (
    COMPARISON_FIGURE_DIR /
    "ukdale_autoencoder_training_loss.png"
)

# ------------------------------------------------------------
# Features
# ------------------------------------------------------------

FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

SCALED_FEATURES = [f"{feature}_z" for feature in FEATURES]

# ------------------------------------------------------------
# Autoencoder settings
# ------------------------------------------------------------

INTERNAL_VALIDATION_RATIO = 0.15
MAX_EPOCHS = 150
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
THRESHOLD_PERCENTILE = 95

print("=" * 90)
print("UK-DALE DENSE AUTOENCODER EXPERIMENT")
print("=" * 90)

# ------------------------------------------------------------
# Load reference and evaluation data
# ------------------------------------------------------------

reference_df = pd.read_csv(
    UKDALE_REFERENCE_PATH,
    low_memory=False
)

evaluation_df = pd.read_csv(
    UKDALE_LOF_SCORED_PATH,
    low_memory=False
)

print("Reference shape:", reference_df.shape)
print("Evaluation shape:", evaluation_df.shape)

required_evaluation_columns = [
    "evaluation_split",
    "anomaly_score",
    "predicted_anomaly",
    "lof_anomaly_score",
    "lof_predicted_anomaly"
]

missing_columns = [
    column for column in required_evaluation_columns
    if column not in evaluation_df.columns
]

if missing_columns:
    raise RuntimeError(
        f"Missing saved IF/LOF columns: {missing_columns}"
    )

X_reference_all = reference_df[
    SCALED_FEATURES
].to_numpy(dtype=np.float32)

X_evaluation = evaluation_df[
    SCALED_FEATURES
].to_numpy(dtype=np.float32)

if not np.isfinite(X_reference_all).all():
    raise RuntimeError(
        "Reference features contain NaN or infinite values."
    )

if not np.isfinite(X_evaluation).all():
    raise RuntimeError(
        "Evaluation features contain NaN or infinite values."
    )

print("\nReference matrix:", X_reference_all.shape)
print("Evaluation matrix:", X_evaluation.shape)

# ------------------------------------------------------------
# Series-balanced internal train/validation split
# ------------------------------------------------------------

internal_train_indices = []
internal_val_indices = []

for series_key, group in reference_df.groupby(
    "series_key",
    sort=True
):
    group_indices = group.index.to_numpy()

    series_seed = (
        RANDOM_STATE +
        sum(bytearray(str(series_key).encode("utf-8")))
    )

    rng = np.random.default_rng(series_seed)
    shuffled = rng.permutation(group_indices)

    validation_count = max(
        1,
        int(round(
            len(shuffled) *
            INTERNAL_VALIDATION_RATIO
        ))
    )

    internal_val_indices.extend(
        shuffled[:validation_count]
    )

    internal_train_indices.extend(
        shuffled[validation_count:]
    )

internal_train_indices = np.asarray(
    internal_train_indices,
    dtype=np.int64
)

internal_val_indices = np.asarray(
    internal_val_indices,
    dtype=np.int64
)

X_ae_train = reference_df.loc[
    internal_train_indices,
    SCALED_FEATURES
].to_numpy(dtype=np.float32)

X_ae_internal_val = reference_df.loc[
    internal_val_indices,
    SCALED_FEATURES
].to_numpy(dtype=np.float32)

print("\nInternal Autoencoder split:")
print("Training rows:", len(X_ae_train))
print("Internal validation rows:", len(X_ae_internal_val))

print("\nSeries represented in AE training:")
print(
    reference_df.loc[
        internal_train_indices,
        "series_key"
    ].nunique()
)

print("Series represented in AE internal validation:")
print(
    reference_df.loc[
        internal_val_indices,
        "series_key"
    ].nunique()
)

# ------------------------------------------------------------
# Build dense Autoencoder
# ------------------------------------------------------------

input_dim = len(SCALED_FEATURES)

inputs = keras.Input(
    shape=(input_dim,),
    name="window_features"
)

encoded = layers.Dense(
    16,
    activation="relu",
    kernel_regularizer=regularizers.l2(1e-5)
)(inputs)

encoded = layers.Dense(
    8,
    activation="relu"
)(encoded)

bottleneck = layers.Dense(
    4,
    activation="relu",
    name="bottleneck"
)(encoded)

decoded = layers.Dense(
    8,
    activation="relu"
)(bottleneck)

decoded = layers.Dense(
    16,
    activation="relu"
)(decoded)

outputs = layers.Dense(
    input_dim,
    activation="linear",
    name="reconstruction"
)(decoded)

autoencoder = keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="ukdale_dense_autoencoder"
)

autoencoder.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss="mse"
)

autoencoder.summary()

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=12,
        restore_best_weights=True,
        min_delta=1e-6,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=6,
        min_lr=1e-6,
        verbose=1
    )
]

# ------------------------------------------------------------
# Train Autoencoder
# ------------------------------------------------------------

print("\nTraining UK-DALE Autoencoder...")

fit_start = time.perf_counter()

history = autoencoder.fit(
    X_ae_train,
    X_ae_train,
    validation_data=(
        X_ae_internal_val,
        X_ae_internal_val
    ),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    callbacks=callbacks,
    verbose=1
)

fit_seconds = time.perf_counter() - fit_start

print(
    f"\nAutoencoder training completed in "
    f"{fit_seconds:.2f} seconds."
)

autoencoder.save(
    UKDALE_AE_MODEL_PATH
)

print("Saved model:")
print(UKDALE_AE_MODEL_PATH)

# ------------------------------------------------------------
# Save and display training history
# ------------------------------------------------------------

history_df = pd.DataFrame(
    history.history
)

history_df["epoch"] = np.arange(
    1,
    len(history_df) + 1
)

history_df.to_csv(
    UKDALE_AE_HISTORY_PATH,
    index=False
)

print("\nTraining history:")
display(history_df.tail(15))

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(
    history_df["epoch"],
    history_df["loss"],
    label="Training loss"
)
plt.plot(
    history_df["epoch"],
    history_df["val_loss"],
    label="Internal validation loss"
)
plt.xlabel("Epoch")
plt.ylabel("Mean squared reconstruction loss")
plt.title("UK-DALE Autoencoder Training History")
plt.legend()
plt.tight_layout()
plt.savefig(
    UKDALE_AE_TRAINING_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()

# ------------------------------------------------------------
# Reconstruction-error scoring
# ------------------------------------------------------------

print("\nScoring evaluation windows...")

score_start = time.perf_counter()

reconstructed = autoencoder.predict(
    X_evaluation,
    batch_size=1024,
    verbose=1
)

ae_scores = np.mean(
    np.square(
        X_evaluation - reconstructed
    ),
    axis=1
)

score_seconds = time.perf_counter() - score_start

evaluation_df["ae_anomaly_score"] = (
    ae_scores.astype(np.float64)
)

evaluation_df["ae_log1p_score"] = np.log1p(
    np.maximum(
        evaluation_df["ae_anomaly_score"],
        0
    )
)

print(
    f"Autoencoder scoring completed in "
    f"{score_seconds:.2f} seconds."
)

# ------------------------------------------------------------
# Validation-based operating threshold
# ------------------------------------------------------------

validation_mask = (
    evaluation_df["evaluation_split"] == "val"
)

ae_threshold = np.percentile(
    evaluation_df.loc[
        validation_mask,
        "ae_anomaly_score"
    ],
    THRESHOLD_PERCENTILE
)

evaluation_df["ae_threshold"] = ae_threshold

evaluation_df["ae_predicted_anomaly"] = (
    evaluation_df["ae_anomaly_score"] >
    ae_threshold
).astype(np.int8)

print("\nAutoencoder validation threshold:")
print(ae_threshold)

# ------------------------------------------------------------
# Overall summary by split
# ------------------------------------------------------------

overall_rows = []

for split_name in ["train", "val", "test"]:

    split_df = evaluation_df[
        evaluation_df["evaluation_split"] ==
        split_name
    ]

    flagged_count = int(
        split_df["ae_predicted_anomaly"].sum()
    )

    total_count = len(split_df)

    overall_rows.append({
        "dataset": "UK-DALE",
        "model": "Dense Autoencoder",
        "split": split_name,
        "windows": total_count,
        "flagged_windows": flagged_count,
        "flagged_ratio": (
            flagged_count / total_count
        ),
        "flagged_ratio_percent": (
            100 * flagged_count / total_count
        ),
        "median_reconstruction_error": (
            split_df["ae_anomaly_score"].median()
        ),
        "score_q95": (
            split_df["ae_anomaly_score"]
            .quantile(0.95)
        ),
        "score_q99": (
            split_df["ae_anomaly_score"]
            .quantile(0.99)
        ),
        "maximum_reconstruction_error": (
            split_df["ae_anomaly_score"].max()
        ),
        "validation_threshold": ae_threshold,
        "threshold_percentile": (
            THRESHOLD_PERCENTILE
        ),
        "reference_rows": len(reference_df),
        "internal_training_rows": len(X_ae_train),
        "internal_validation_rows": (
            len(X_ae_internal_val)
        ),
        "epochs_completed": len(history_df),
        "fit_seconds": fit_seconds,
        "score_seconds": score_seconds
    })

overall_summary = pd.DataFrame(
    overall_rows
)

print("\nUK-DALE Autoencoder overall summary:")
display(overall_summary)

# ------------------------------------------------------------
# Test summary by category
# ------------------------------------------------------------

test_df = evaluation_df[
    evaluation_df["evaluation_split"] == "test"
].copy()

category_summary = (
    test_df.groupby(
        "primary_category",
        dropna=False
    )
    .agg(
        test_windows=(
            "ae_predicted_anomaly",
            "size"
        ),
        flagged_windows=(
            "ae_predicted_anomaly",
            "sum"
        ),
        mean_reconstruction_error=(
            "ae_anomaly_score",
            "mean"
        ),
        median_reconstruction_error=(
            "ae_anomaly_score",
            "median"
        ),
        maximum_reconstruction_error=(
            "ae_anomaly_score",
            "max"
        )
    )
    .reset_index()
)

category_summary["flagged_ratio"] = (
    category_summary["flagged_windows"] /
    category_summary["test_windows"]
)

category_summary["flagged_ratio_percent"] = (
    100 * category_summary["flagged_ratio"]
)

category_summary = category_summary.sort_values(
    "flagged_ratio",
    ascending=False
).reset_index(drop=True)

print("\nUK-DALE Autoencoder test summary by category:")
display(category_summary)

# ------------------------------------------------------------
# Pairwise and three-model agreement
# ------------------------------------------------------------

def binary_agreement_metrics(
    flags_a,
    flags_b
):
    flags_a = np.asarray(flags_a).astype(int)
    flags_b = np.asarray(flags_b).astype(int)

    both = int(
        ((flags_a == 1) & (flags_b == 1)).sum()
    )

    only_a = int(
        ((flags_a == 1) & (flags_b == 0)).sum()
    )

    only_b = int(
        ((flags_a == 0) & (flags_b == 1)).sum()
    )

    neither = int(
        ((flags_a == 0) & (flags_b == 0)).sum()
    )

    union = both + only_a + only_b

    return {
        "both_flagged": both,
        "only_model_a": only_a,
        "only_model_b": only_b,
        "neither_flagged": neither,
        "flag_agreement_rate": (
            (both + neither) / len(flags_a)
        ),
        "flag_jaccard_index": (
            both / union
            if union > 0 else np.nan
        )
    }


agreement_rows = []

for split_name in ["train", "val", "test"]:

    split_df = evaluation_df[
        evaluation_df["evaluation_split"] ==
        split_name
    ]

    if_flags = split_df[
        "predicted_anomaly"
    ].to_numpy()

    lof_flags = split_df[
        "lof_predicted_anomaly"
    ].to_numpy()

    ae_flags = split_df[
        "ae_predicted_anomaly"
    ].to_numpy()

    if_scores = split_df[
        "anomaly_score"
    ].to_numpy()

    lof_scores = split_df[
        "lof_anomaly_score"
    ].to_numpy()

    ae_scores_split = split_df[
        "ae_anomaly_score"
    ].to_numpy()

    pair_definitions = [
        (
            "Isolation Forest",
            "LOF",
            if_flags,
            lof_flags,
            if_scores,
            lof_scores
        ),
        (
            "Isolation Forest",
            "Autoencoder",
            if_flags,
            ae_flags,
            if_scores,
            ae_scores_split
        ),
        (
            "LOF",
            "Autoencoder",
            lof_flags,
            ae_flags,
            lof_scores,
            ae_scores_split
        )
    ]

    for (
        model_a,
        model_b,
        flags_a,
        flags_b,
        scores_a,
        scores_b
    ) in pair_definitions:

        metrics = binary_agreement_metrics(
            flags_a,
            flags_b
        )

        rank_result = spearmanr(
            scores_a,
            scores_b
        )

        agreement_rows.append({
            "dataset": "UK-DALE",
            "split": split_name,
            "comparison": (
                f"{model_a} vs {model_b}"
            ),
            "windows": len(split_df),
            "model_a_flagged": int(
                np.sum(flags_a)
            ),
            "model_b_flagged": int(
                np.sum(flags_b)
            ),
            **metrics,
            "score_spearman_correlation": (
                rank_result.statistic
            ),
            "score_spearman_p_value": (
                rank_result.pvalue
            )
        })

    three_flagged = int(
        (
            (if_flags == 1) &
            (lof_flags == 1) &
            (ae_flags == 1)
        ).sum()
    )

    at_least_two_flagged = int(
        (
            if_flags +
            lof_flags +
            ae_flags
        >= 2
        ).sum()
    )

    any_flagged = int(
        (
            if_flags +
            lof_flags +
            ae_flags
        >= 1
        ).sum()
    )

    agreement_rows.append({
        "dataset": "UK-DALE",
        "split": split_name,
        "comparison": "Three-model consensus",
        "windows": len(split_df),
        "model_a_flagged": np.nan,
        "model_b_flagged": np.nan,
        "both_flagged": three_flagged,
        "only_model_a": np.nan,
        "only_model_b": np.nan,
        "neither_flagged": (
            len(split_df) - any_flagged
        ),
        "flag_agreement_rate": np.nan,
        "flag_jaccard_index": (
            three_flagged / any_flagged
            if any_flagged > 0 else np.nan
        ),
        "score_spearman_correlation": np.nan,
        "score_spearman_p_value": np.nan,
        "three_models_flagged": three_flagged,
        "at_least_two_models_flagged": (
            at_least_two_flagged
        ),
        "any_model_flagged": any_flagged
    })

agreement_summary = pd.DataFrame(
    agreement_rows
)

print("\nUK-DALE pairwise and consensus agreement:")
display(agreement_summary)

# ------------------------------------------------------------
# Highest-scoring Autoencoder test windows
# ------------------------------------------------------------

top_columns = [
    "source_row_id",
    "building",
    "meter",
    "series_key",
    "primary_category",
    "window_start",
    "window_end",
    "ae_anomaly_score",
    "ae_log1p_score",
    "ae_threshold",
    "ae_predicted_anomaly",
    "anomaly_score",
    "predicted_anomaly",
    "lof_anomaly_score",
    "lof_predicted_anomaly"
]

top_test_windows = (
    test_df.sort_values(
        "ae_anomaly_score",
        ascending=False
    )
    .head(30)[top_columns]
    .reset_index(drop=True)
)

print("\nTop 30 UK-DALE Autoencoder test windows:")
display(top_test_windows)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

evaluation_df.to_csv(
    UKDALE_AE_SCORED_PATH,
    index=False
)

overall_summary.to_csv(
    UKDALE_AE_OVERALL_PATH,
    index=False
)

category_summary.to_csv(
    UKDALE_AE_CATEGORY_PATH,
    index=False
)

agreement_summary.to_csv(
    UKDALE_AE_AGREEMENT_PATH,
    index=False
)

top_test_windows.to_csv(
    UKDALE_AE_TOP_WINDOWS_PATH,
    index=False
)

print("\nSaved outputs:")
print(UKDALE_AE_SCORED_PATH)
print(UKDALE_AE_HISTORY_PATH)
print(UKDALE_AE_OVERALL_PATH)
print(UKDALE_AE_CATEGORY_PATH)
print(UKDALE_AE_AGREEMENT_PATH)
print(UKDALE_AE_TOP_WINDOWS_PATH)
print(UKDALE_AE_TRAINING_FIGURE_PATH)

print("\nCell 7 complete.")

del (
    X_reference_all,
    X_ae_train,
    X_ae_internal_val,
    X_evaluation,
    reconstructed
)

gc.collect()

In [ ]:
# ============================================================
# Cell 8: UK-DALE balanced-reference Isolation Forest
# Fair IF–LOF–Autoencoder cross-model comparison
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import joblib

from sklearn.ensemble import IsolationForest
from scipy.stats import spearmanr

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

UKDALE_REFERENCE_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_lof_reference_sample.csv"
)

UKDALE_THREE_MODEL_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_if_lof_ae_scored_evaluation.csv"
)

BALANCED_IF_MODEL_PATH = (
    COMPARISON_MODEL_DIR /
    "ukdale_balanced_reference_isolation_forest.joblib"
)

BALANCED_IF_SCORED_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_balanced_if_lof_ae_scored_evaluation.csv"
)

BALANCED_IF_OVERALL_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_balanced_if_overall_summary.csv"
)

BALANCED_AGREEMENT_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_balanced_three_model_agreement.csv"
)

ORIGINAL_BALANCED_IF_PATH = (
    COMPARISON_TABLE_DIR /
    "ukdale_original_vs_balanced_if_comparison.csv"
)

# ------------------------------------------------------------
# Features
# ------------------------------------------------------------

FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

SCALED_FEATURES = [f"{feature}_z" for feature in FEATURES]

THRESHOLD_PERCENTILE = 95
RANDOM_STATE = 42

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------

reference_df = pd.read_csv(
    UKDALE_REFERENCE_PATH,
    low_memory=False
)

evaluation_df = pd.read_csv(
    UKDALE_THREE_MODEL_PATH,
    low_memory=False
)

required_columns = [
    "evaluation_split",
    "anomaly_score",
    "predicted_anomaly",
    "lof_anomaly_score",
    "lof_predicted_anomaly",
    "ae_anomaly_score",
    "ae_predicted_anomaly"
]

missing_columns = [
    column
    for column in required_columns
    if column not in evaluation_df.columns
]

if missing_columns:
    raise RuntimeError(
        f"Missing required model outputs: {missing_columns}"
    )

X_reference = reference_df[
    SCALED_FEATURES
].to_numpy(dtype=np.float32)

X_evaluation = evaluation_df[
    SCALED_FEATURES
].to_numpy(dtype=np.float32)

if not np.isfinite(X_reference).all():
    raise RuntimeError(
        "Reference features contain non-finite values."
    )

if not np.isfinite(X_evaluation).all():
    raise RuntimeError(
        "Evaluation features contain non-finite values."
    )

print("=" * 90)
print("UK-DALE BALANCED-REFERENCE ISOLATION FOREST")
print("=" * 90)

print("Reference matrix:", X_reference.shape)
print("Evaluation matrix:", X_evaluation.shape)

# ------------------------------------------------------------
# Train balanced-reference Isolation Forest
# ------------------------------------------------------------

balanced_if = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("\nTraining balanced-reference Isolation Forest...")

fit_start = time.perf_counter()

balanced_if.fit(X_reference)

fit_seconds = time.perf_counter() - fit_start

print(
    f"Training completed in {fit_seconds:.2f} seconds."
)

joblib.dump(
    balanced_if,
    BALANCED_IF_MODEL_PATH
)

# ------------------------------------------------------------
# Score same evaluation rows
# ------------------------------------------------------------

score_start = time.perf_counter()

balanced_scores = -balanced_if.decision_function(
    X_evaluation
)

score_seconds = time.perf_counter() - score_start

evaluation_df[
    "if_balanced_anomaly_score"
] = balanced_scores.astype(np.float64)

validation_mask = (
    evaluation_df["evaluation_split"] == "val"
)

balanced_threshold = np.percentile(
    evaluation_df.loc[
        validation_mask,
        "if_balanced_anomaly_score"
    ],
    THRESHOLD_PERCENTILE
)

evaluation_df[
    "if_balanced_threshold"
] = balanced_threshold

evaluation_df[
    "if_balanced_predicted_anomaly"
] = (
    evaluation_df["if_balanced_anomaly_score"] >
    balanced_threshold
).astype(np.int8)

print(
    f"Scoring completed in {score_seconds:.2f} seconds."
)

print(
    "Balanced IF validation threshold:",
    balanced_threshold
)

# ------------------------------------------------------------
# Overall balanced IF summary
# ------------------------------------------------------------

overall_rows = []

for split_name in ["train", "val", "test"]:

    split_df = evaluation_df[
        evaluation_df["evaluation_split"] ==
        split_name
    ]

    total = len(split_df)

    flagged = int(
        split_df[
            "if_balanced_predicted_anomaly"
        ].sum()
    )

    overall_rows.append({
        "dataset": "UK-DALE",
        "model": "Balanced-reference Isolation Forest",
        "split": split_name,
        "windows": total,
        "flagged_windows": flagged,
        "flagged_ratio": flagged / total,
        "flagged_ratio_percent": 100 * flagged / total,
        "median_anomaly_score": split_df[
            "if_balanced_anomaly_score"
        ].median(),
        "score_q95": split_df[
            "if_balanced_anomaly_score"
        ].quantile(0.95),
        "score_q99": split_df[
            "if_balanced_anomaly_score"
        ].quantile(0.99),
        "maximum_anomaly_score": split_df[
            "if_balanced_anomaly_score"
        ].max(),
        "validation_threshold": balanced_threshold,
        "threshold_percentile": THRESHOLD_PERCENTILE,
        "reference_rows": len(reference_df),
        "fit_seconds": fit_seconds,
        "score_seconds": score_seconds
    })

balanced_overall = pd.DataFrame(
    overall_rows
)

print("\nBalanced Isolation Forest overall summary:")
display(balanced_overall)

# ------------------------------------------------------------
# Original full-data IF versus balanced-reference IF
# ------------------------------------------------------------

def jaccard_from_flags(flags_a, flags_b):

    flags_a = np.asarray(flags_a).astype(int)
    flags_b = np.asarray(flags_b).astype(int)

    intersection = int(
        ((flags_a == 1) & (flags_b == 1)).sum()
    )

    union = int(
        ((flags_a == 1) | (flags_b == 1)).sum()
    )

    return (
        intersection,
        union,
        intersection / union if union else np.nan
    )


original_balanced_rows = []

for split_name in ["train", "val", "test"]:

    split_df = evaluation_df[
        evaluation_df["evaluation_split"] ==
        split_name
    ]

    original_flags = split_df[
        "predicted_anomaly"
    ].to_numpy()

    balanced_flags = split_df[
        "if_balanced_predicted_anomaly"
    ].to_numpy()

    intersection, union, jaccard = (
        jaccard_from_flags(
            original_flags,
            balanced_flags
        )
    )

    rank_result = spearmanr(
        split_df["anomaly_score"],
        split_df["if_balanced_anomaly_score"]
    )

    original_balanced_rows.append({
        "dataset": "UK-DALE",
        "split": split_name,
        "windows": len(split_df),
        "original_if_flagged": int(
            original_flags.sum()
        ),
        "balanced_if_flagged": int(
            balanced_flags.sum()
        ),
        "both_flagged": intersection,
        "flag_union": union,
        "flag_jaccard_index": jaccard,
        "score_spearman_correlation": (
            rank_result.statistic
        ),
        "score_spearman_p_value": (
            rank_result.pvalue
        )
    })

original_balanced_comparison = pd.DataFrame(
    original_balanced_rows
)

print(
    "\nOriginal full-data IF versus balanced-reference IF:"
)

display(original_balanced_comparison)

# ------------------------------------------------------------
# Fair pairwise and three-model agreement
# ------------------------------------------------------------

def pairwise_metrics(
    flags_a,
    flags_b,
    scores_a,
    scores_b
):
    flags_a = np.asarray(flags_a).astype(int)
    flags_b = np.asarray(flags_b).astype(int)

    both = int(
        ((flags_a == 1) & (flags_b == 1)).sum()
    )

    only_a = int(
        ((flags_a == 1) & (flags_b == 0)).sum()
    )

    only_b = int(
        ((flags_a == 0) & (flags_b == 1)).sum()
    )

    neither = int(
        ((flags_a == 0) & (flags_b == 0)).sum()
    )

    union = both + only_a + only_b

    rank_result = spearmanr(
        scores_a,
        scores_b
    )

    return {
        "both_flagged": both,
        "only_model_a": only_a,
        "only_model_b": only_b,
        "neither_flagged": neither,
        "flag_agreement_rate": (
            (both + neither) / len(flags_a)
        ),
        "flag_jaccard_index": (
            both / union if union else np.nan
        ),
        "score_spearman_correlation": (
            rank_result.statistic
        ),
        "score_spearman_p_value": (
            rank_result.pvalue
        )
    }


agreement_rows = []

for split_name in ["train", "val", "test"]:

    split_df = evaluation_df[
        evaluation_df["evaluation_split"] ==
        split_name
    ]

    balanced_if_flags = split_df[
        "if_balanced_predicted_anomaly"
    ].to_numpy()

    lof_flags = split_df[
        "lof_predicted_anomaly"
    ].to_numpy()

    ae_flags = split_df[
        "ae_predicted_anomaly"
    ].to_numpy()

    balanced_if_scores = split_df[
        "if_balanced_anomaly_score"
    ].to_numpy()

    lof_scores = split_df[
        "lof_anomaly_score"
    ].to_numpy()

    ae_scores = split_df[
        "ae_anomaly_score"
    ].to_numpy()

    comparisons = [
        (
            "Balanced Isolation Forest",
            "LOF",
            balanced_if_flags,
            lof_flags,
            balanced_if_scores,
            lof_scores
        ),
        (
            "Balanced Isolation Forest",
            "Autoencoder",
            balanced_if_flags,
            ae_flags,
            balanced_if_scores,
            ae_scores
        ),
        (
            "LOF",
            "Autoencoder",
            lof_flags,
            ae_flags,
            lof_scores,
            ae_scores
        )
    ]

    for (
        model_a,
        model_b,
        flags_a,
        flags_b,
        scores_a,
        scores_b
    ) in comparisons:

        metrics = pairwise_metrics(
            flags_a,
            flags_b,
            scores_a,
            scores_b
        )

        agreement_rows.append({
            "dataset": "UK-DALE",
            "split": split_name,
            "comparison": (
                f"{model_a} vs {model_b}"
            ),
            "windows": len(split_df),
            "model_a_flagged": int(
                flags_a.sum()
            ),
            "model_b_flagged": int(
                flags_b.sum()
            ),
            **metrics,
            "three_models_flagged": np.nan,
            "at_least_two_models_flagged": np.nan,
            "any_model_flagged": np.nan
        })

    model_flag_count = (
        balanced_if_flags.astype(int) +
        lof_flags.astype(int) +
        ae_flags.astype(int)
    )

    three_models_flagged = int(
        (model_flag_count == 3).sum()
    )

    at_least_two_flagged = int(
        (model_flag_count >= 2).sum()
    )

    any_model_flagged = int(
        (model_flag_count >= 1).sum()
    )

    agreement_rows.append({
        "dataset": "UK-DALE",
        "split": split_name,
        "comparison": "Three-model consensus",
        "windows": len(split_df),
        "model_a_flagged": np.nan,
        "model_b_flagged": np.nan,
        "both_flagged": three_models_flagged,
        "only_model_a": np.nan,
        "only_model_b": np.nan,
        "neither_flagged": (
            len(split_df) - any_model_flagged
        ),
        "flag_agreement_rate": np.nan,
        "flag_jaccard_index": (
            three_models_flagged /
            any_model_flagged
            if any_model_flagged else np.nan
        ),
        "score_spearman_correlation": np.nan,
        "score_spearman_p_value": np.nan,
        "three_models_flagged": (
            three_models_flagged
        ),
        "at_least_two_models_flagged": (
            at_least_two_flagged
        ),
        "any_model_flagged": (
            any_model_flagged
        )
    })

balanced_agreement = pd.DataFrame(
    agreement_rows
)

print(
    "\nFair balanced-reference model agreement:"
)

display(balanced_agreement)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

evaluation_df.to_csv(
    BALANCED_IF_SCORED_PATH,
    index=False
)

balanced_overall.to_csv(
    BALANCED_IF_OVERALL_PATH,
    index=False
)

balanced_agreement.to_csv(
    BALANCED_AGREEMENT_PATH,
    index=False
)

original_balanced_comparison.to_csv(
    ORIGINAL_BALANCED_IF_PATH,
    index=False
)

print("\nSaved outputs:")
print(BALANCED_IF_MODEL_PATH)
print(BALANCED_IF_SCORED_PATH)
print(BALANCED_IF_OVERALL_PATH)
print(BALANCED_AGREEMENT_PATH)
print(ORIGINAL_BALANCED_IF_PATH)

print("\nCell 8 complete.")

In [ ]:
# ============================================================
# Cell 9: AMPds2 Dense Autoencoder
# Fixed transfer from UK-DALE — no synthetic injection
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import random
import gc

from scipy.stats import spearmanr
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except Exception as exc:
    print("Deterministic operations could not be fully enabled:", exc)

tf.keras.backend.clear_session()

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

AMPDS2_REFERENCE_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_lof_reference_sample.csv"
)

AMPDS2_LOF_SCORED_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_lof_scored_evaluation.csv"
)

AMPDS2_AE_MODEL_PATH = (
    COMPARISON_MODEL_DIR /
    "ampds2_dense_autoencoder.keras"
)

AMPDS2_AE_SCORED_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_if_lof_ae_scored_evaluation.csv"
)

AMPDS2_AE_HISTORY_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_autoencoder_training_history.csv"
)

AMPDS2_AE_OVERALL_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_autoencoder_overall_summary.csv"
)

AMPDS2_AE_CATEGORY_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_autoencoder_test_summary_by_channel.csv"
)

AMPDS2_AE_AGREEMENT_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_preliminary_three_model_agreement.csv"
)

AMPDS2_AE_TOP_WINDOWS_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_autoencoder_top_test_windows.csv"
)

AMPDS2_AE_TRAINING_FIGURE_PATH = (
    COMPARISON_FIGURE_DIR /
    "ampds2_autoencoder_training_loss.png"
)

# ------------------------------------------------------------
# Features
# ------------------------------------------------------------

FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

SCALED_FEATURES = [
    f"{feature}_z"
    for feature in FEATURES
]

# ------------------------------------------------------------
# Fixed Autoencoder configuration
# Identical to UK-DALE
# ------------------------------------------------------------

INTERNAL_VALIDATION_RATIO = 0.15
MAX_EPOCHS = 150
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
THRESHOLD_PERCENTILE = 95

print("=" * 90)
print("AMPds2 DENSE AUTOENCODER EXPERIMENT")
print("=" * 90)

# ------------------------------------------------------------
# Load reference and evaluation data
# ------------------------------------------------------------

reference_df = pd.read_csv(
    AMPDS2_REFERENCE_PATH,
    low_memory=False
)

evaluation_df = pd.read_csv(
    AMPDS2_LOF_SCORED_PATH,
    low_memory=False
)

print("Reference shape:", reference_df.shape)
print("Evaluation shape:", evaluation_df.shape)

required_evaluation_columns = [
    "evaluation_split",
    "anomaly_score",
    "predicted_anomaly",
    "lof_anomaly_score",
    "lof_predicted_anomaly"
]

missing_columns = [
    column
    for column in required_evaluation_columns
    if column not in evaluation_df.columns
]

if missing_columns:
    raise RuntimeError(
        f"Missing saved IF/LOF columns: {missing_columns}"
    )

X_reference_all = reference_df[
    SCALED_FEATURES
].to_numpy(dtype=np.float32)

X_evaluation = evaluation_df[
    SCALED_FEATURES
].to_numpy(dtype=np.float32)

if not np.isfinite(X_reference_all).all():
    raise RuntimeError(
        "Reference features contain NaN or infinite values."
    )

if not np.isfinite(X_evaluation).all():
    raise RuntimeError(
        "Evaluation features contain NaN or infinite values."
    )

print("\nReference matrix:", X_reference_all.shape)
print("Evaluation matrix:", X_evaluation.shape)

print("\nEvaluation split counts:")
print(
    evaluation_df["evaluation_split"]
    .value_counts()
    .reindex(["train", "val", "test"])
)

# ------------------------------------------------------------
# Series-balanced internal Autoencoder split
# ------------------------------------------------------------

internal_train_indices = []
internal_val_indices = []

for series_key, group in reference_df.groupby(
    "series_key",
    sort=True
):
    group_indices = group.index.to_numpy()

    series_seed = (
        RANDOM_STATE +
        sum(
            bytearray(
                str(series_key).encode("utf-8")
            )
        )
    )

    rng = np.random.default_rng(series_seed)

    shuffled_indices = rng.permutation(
        group_indices
    )

    validation_count = max(
        1,
        int(
            round(
                len(shuffled_indices) *
                INTERNAL_VALIDATION_RATIO
            )
        )
    )

    internal_val_indices.extend(
        shuffled_indices[:validation_count]
    )

    internal_train_indices.extend(
        shuffled_indices[validation_count:]
    )

internal_train_indices = np.asarray(
    internal_train_indices,
    dtype=np.int64
)

internal_val_indices = np.asarray(
    internal_val_indices,
    dtype=np.int64
)

X_ae_train = reference_df.loc[
    internal_train_indices,
    SCALED_FEATURES
].to_numpy(dtype=np.float32)

X_ae_internal_val = reference_df.loc[
    internal_val_indices,
    SCALED_FEATURES
].to_numpy(dtype=np.float32)

print("\nInternal Autoencoder split:")
print("Training rows:", len(X_ae_train))
print(
    "Internal validation rows:",
    len(X_ae_internal_val)
)

print(
    "Training series:",
    reference_df.loc[
        internal_train_indices,
        "series_key"
    ].nunique()
)

print(
    "Internal validation series:",
    reference_df.loc[
        internal_val_indices,
        "series_key"
    ].nunique()
)

# ------------------------------------------------------------
# Build identical dense Autoencoder
# ------------------------------------------------------------

input_dim = len(SCALED_FEATURES)

inputs = keras.Input(
    shape=(input_dim,),
    name="window_features"
)

encoded = layers.Dense(
    16,
    activation="relu",
    kernel_regularizer=regularizers.l2(1e-5)
)(inputs)

encoded = layers.Dense(
    8,
    activation="relu"
)(encoded)

bottleneck = layers.Dense(
    4,
    activation="relu",
    name="bottleneck"
)(encoded)

decoded = layers.Dense(
    8,
    activation="relu"
)(bottleneck)

decoded = layers.Dense(
    16,
    activation="relu"
)(decoded)

outputs = layers.Dense(
    input_dim,
    activation="linear",
    name="reconstruction"
)(decoded)

autoencoder = keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="ampds2_dense_autoencoder"
)

autoencoder.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss="mse"
)

autoencoder.summary()

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=12,
        restore_best_weights=True,
        min_delta=1e-6,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=6,
        min_lr=1e-6,
        verbose=1
    )
]

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

print("\nTraining AMPds2 Autoencoder...")

fit_start = time.perf_counter()

history = autoencoder.fit(
    X_ae_train,
    X_ae_train,
    validation_data=(
        X_ae_internal_val,
        X_ae_internal_val
    ),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    callbacks=callbacks,
    verbose=1
)

fit_seconds = (
    time.perf_counter() -
    fit_start
)

print(
    f"\nTraining completed in "
    f"{fit_seconds:.2f} seconds."
)

autoencoder.save(
    AMPDS2_AE_MODEL_PATH
)

# ------------------------------------------------------------
# Training history
# ------------------------------------------------------------

history_df = pd.DataFrame(
    history.history
)

history_df["epoch"] = np.arange(
    1,
    len(history_df) + 1
)

history_df.to_csv(
    AMPDS2_AE_HISTORY_PATH,
    index=False
)

print("\nFinal training-history rows:")
display(history_df.tail(15))

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(
    history_df["epoch"],
    history_df["loss"],
    label="Training loss"
)

plt.plot(
    history_df["epoch"],
    history_df["val_loss"],
    label="Internal validation loss"
)

plt.xlabel("Epoch")
plt.ylabel("Mean squared reconstruction loss")
plt.title("AMPds2 Autoencoder Training History")
plt.legend()
plt.tight_layout()

plt.savefig(
    AMPDS2_AE_TRAINING_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

# ------------------------------------------------------------
# Reconstruction-error scoring
# ------------------------------------------------------------

print("\nScoring AMPds2 evaluation windows...")

score_start = time.perf_counter()

reconstructed = autoencoder.predict(
    X_evaluation,
    batch_size=1024,
    verbose=1
)

ae_scores = np.mean(
    np.square(
        X_evaluation -
        reconstructed
    ),
    axis=1
)

score_seconds = (
    time.perf_counter() -
    score_start
)

evaluation_df[
    "ae_anomaly_score"
] = ae_scores.astype(np.float64)

evaluation_df[
    "ae_log1p_score"
] = np.log1p(
    np.maximum(
        evaluation_df[
            "ae_anomaly_score"
        ],
        0
    )
)

print(
    f"Scoring completed in "
    f"{score_seconds:.2f} seconds."
)

# ------------------------------------------------------------
# Validation threshold
# ------------------------------------------------------------

validation_mask = (
    evaluation_df["evaluation_split"] ==
    "val"
)

ae_threshold = np.percentile(
    evaluation_df.loc[
        validation_mask,
        "ae_anomaly_score"
    ],
    THRESHOLD_PERCENTILE
)

evaluation_df["ae_threshold"] = (
    ae_threshold
)

evaluation_df[
    "ae_predicted_anomaly"
] = (
    evaluation_df["ae_anomaly_score"] >
    ae_threshold
).astype(np.int8)

print("\nAutoencoder threshold:")
print(ae_threshold)

# ------------------------------------------------------------
# Overall summary
# ------------------------------------------------------------

overall_rows = []

for split_name in [
    "train",
    "val",
    "test"
]:

    split_df = evaluation_df[
        evaluation_df["evaluation_split"] ==
        split_name
    ]

    total_count = len(split_df)

    flagged_count = int(
        split_df[
            "ae_predicted_anomaly"
        ].sum()
    )

    overall_rows.append({
        "dataset": "AMPds2",
        "model": "Dense Autoencoder",
        "split": split_name,
        "windows": total_count,
        "flagged_windows": flagged_count,
        "flagged_ratio": (
            flagged_count /
            total_count
        ),
        "flagged_ratio_percent": (
            100 *
            flagged_count /
            total_count
        ),
        "median_reconstruction_error": (
            split_df[
                "ae_anomaly_score"
            ].median()
        ),
        "score_q95": (
            split_df[
                "ae_anomaly_score"
            ].quantile(0.95)
        ),
        "score_q99": (
            split_df[
                "ae_anomaly_score"
            ].quantile(0.99)
        ),
        "maximum_reconstruction_error": (
            split_df[
                "ae_anomaly_score"
            ].max()
        ),
        "validation_threshold": (
            ae_threshold
        ),
        "threshold_percentile": (
            THRESHOLD_PERCENTILE
        ),
        "reference_rows": (
            len(reference_df)
        ),
        "internal_training_rows": (
            len(X_ae_train)
        ),
        "internal_validation_rows": (
            len(X_ae_internal_val)
        ),
        "epochs_completed": (
            len(history_df)
        ),
        "fit_seconds": fit_seconds,
        "score_seconds": score_seconds
    })

overall_summary = pd.DataFrame(
    overall_rows
)

print(
    "\nAMPds2 Autoencoder overall summary:"
)

display(overall_summary)

# ------------------------------------------------------------
# Test summary by channel
# ------------------------------------------------------------

test_df = evaluation_df[
    evaluation_df["evaluation_split"] ==
    "test"
].copy()

category_summary = (
    test_df.groupby(
        [
            "channel_code",
            "channel_name",
            "primary_category"
        ],
        dropna=False
    )
    .agg(
        test_windows=(
            "ae_predicted_anomaly",
            "size"
        ),
        flagged_windows=(
            "ae_predicted_anomaly",
            "sum"
        ),
        mean_reconstruction_error=(
            "ae_anomaly_score",
            "mean"
        ),
        median_reconstruction_error=(
            "ae_anomaly_score",
            "median"
        ),
        maximum_reconstruction_error=(
            "ae_anomaly_score",
            "max"
        )
    )
    .reset_index()
)

category_summary[
    "flagged_ratio"
] = (
    category_summary[
        "flagged_windows"
    ] /
    category_summary[
        "test_windows"
    ]
)

category_summary[
    "flagged_ratio_percent"
] = (
    100 *
    category_summary[
        "flagged_ratio"
    ]
)

category_summary = (
    category_summary
    .sort_values(
        "flagged_ratio",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "\nAMPds2 Autoencoder test summary by channel:"
)

display(category_summary)

# ------------------------------------------------------------
# Preliminary agreement using original IF
# Final fair agreement follows balanced IF
# ------------------------------------------------------------

def pairwise_metrics(
    flags_a,
    flags_b,
    scores_a,
    scores_b
):
    flags_a = np.asarray(
        flags_a
    ).astype(int)

    flags_b = np.asarray(
        flags_b
    ).astype(int)

    both = int(
        (
            (flags_a == 1) &
            (flags_b == 1)
        ).sum()
    )

    only_a = int(
        (
            (flags_a == 1) &
            (flags_b == 0)
        ).sum()
    )

    only_b = int(
        (
            (flags_a == 0) &
            (flags_b == 1)
        ).sum()
    )

    neither = int(
        (
            (flags_a == 0) &
            (flags_b == 0)
        ).sum()
    )

    union = (
        both +
        only_a +
        only_b
    )

    rank_result = spearmanr(
        scores_a,
        scores_b
    )

    return {
        "both_flagged": both,
        "only_model_a": only_a,
        "only_model_b": only_b,
        "neither_flagged": neither,
        "flag_agreement_rate": (
            (both + neither) /
            len(flags_a)
        ),
        "flag_jaccard_index": (
            both / union
            if union > 0
            else np.nan
        ),
        "score_spearman_correlation": (
            rank_result.statistic
        ),
        "score_spearman_p_value": (
            rank_result.pvalue
        )
    }


agreement_rows = []

for split_name in [
    "train",
    "val",
    "test"
]:

    split_df = evaluation_df[
        evaluation_df[
            "evaluation_split"
        ] == split_name
    ]

    comparisons = [
        (
            "Original Isolation Forest",
            "LOF",
            split_df[
                "predicted_anomaly"
            ].to_numpy(),
            split_df[
                "lof_predicted_anomaly"
            ].to_numpy(),
            split_df[
                "anomaly_score"
            ].to_numpy(),
            split_df[
                "lof_anomaly_score"
            ].to_numpy()
        ),
        (
            "Original Isolation Forest",
            "Autoencoder",
            split_df[
                "predicted_anomaly"
            ].to_numpy(),
            split_df[
                "ae_predicted_anomaly"
            ].to_numpy(),
            split_df[
                "anomaly_score"
            ].to_numpy(),
            split_df[
                "ae_anomaly_score"
            ].to_numpy()
        ),
        (
            "LOF",
            "Autoencoder",
            split_df[
                "lof_predicted_anomaly"
            ].to_numpy(),
            split_df[
                "ae_predicted_anomaly"
            ].to_numpy(),
            split_df[
                "lof_anomaly_score"
            ].to_numpy(),
            split_df[
                "ae_anomaly_score"
            ].to_numpy()
        )
    ]

    for (
        model_a,
        model_b,
        flags_a,
        flags_b,
        scores_a,
        scores_b
    ) in comparisons:

        metrics = pairwise_metrics(
            flags_a,
            flags_b,
            scores_a,
            scores_b
        )

        agreement_rows.append({
            "dataset": "AMPds2",
            "split": split_name,
            "comparison": (
                f"{model_a} vs {model_b}"
            ),
            "windows": len(split_df),
            "model_a_flagged": int(
                flags_a.sum()
            ),
            "model_b_flagged": int(
                flags_b.sum()
            ),
            **metrics
        })

preliminary_agreement = pd.DataFrame(
    agreement_rows
)

print(
    "\nPreliminary AMPds2 model agreement:"
)

display(preliminary_agreement)

# ------------------------------------------------------------
# Highest Autoencoder test windows
# ------------------------------------------------------------

top_columns = [
    "source_row_id",
    "channel_code",
    "channel_name",
    "series_key",
    "primary_category",
    "window_start",
    "window_end",
    "ae_anomaly_score",
    "ae_log1p_score",
    "ae_threshold",
    "ae_predicted_anomaly",
    "anomaly_score",
    "predicted_anomaly",
    "lof_anomaly_score",
    "lof_predicted_anomaly"
]

top_test_windows = (
    test_df.sort_values(
        "ae_anomaly_score",
        ascending=False
    )
    .head(30)[top_columns]
    .reset_index(drop=True)
)

print(
    "\nTop 30 AMPds2 Autoencoder test windows:"
)

display(top_test_windows)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

evaluation_df.to_csv(
    AMPDS2_AE_SCORED_PATH,
    index=False
)

history_df.to_csv(
    AMPDS2_AE_HISTORY_PATH,
    index=False
)

overall_summary.to_csv(
    AMPDS2_AE_OVERALL_PATH,
    index=False
)

category_summary.to_csv(
    AMPDS2_AE_CATEGORY_PATH,
    index=False
)

preliminary_agreement.to_csv(
    AMPDS2_AE_AGREEMENT_PATH,
    index=False
)

top_test_windows.to_csv(
    AMPDS2_AE_TOP_WINDOWS_PATH,
    index=False
)

print("\nSaved outputs:")
print(AMPDS2_AE_MODEL_PATH)
print(AMPDS2_AE_SCORED_PATH)
print(AMPDS2_AE_HISTORY_PATH)
print(AMPDS2_AE_OVERALL_PATH)
print(AMPDS2_AE_CATEGORY_PATH)
print(AMPDS2_AE_AGREEMENT_PATH)
print(AMPDS2_AE_TOP_WINDOWS_PATH)
print(AMPDS2_AE_TRAINING_FIGURE_PATH)

print("\nCell 9 complete.")

del (
    X_reference_all,
    X_ae_train,
    X_ae_internal_val,
    X_evaluation,
    reconstructed
)

gc.collect()

In [ ]:
# ============================================================
# Cell 10: AMPds2 balanced-reference Isolation Forest
# Final fair IF–LOF–Autoencoder comparison
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import joblib

from sklearn.ensemble import IsolationForest
from scipy.stats import spearmanr

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

AMPDS2_REFERENCE_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_lof_reference_sample.csv"
)

AMPDS2_THREE_MODEL_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_if_lof_ae_scored_evaluation.csv"
)

BALANCED_IF_MODEL_PATH = (
    COMPARISON_MODEL_DIR /
    "ampds2_balanced_reference_isolation_forest.joblib"
)

BALANCED_IF_SCORED_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_balanced_if_lof_ae_scored_evaluation.csv"
)

BALANCED_IF_OVERALL_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_balanced_if_overall_summary.csv"
)

BALANCED_IF_CHANNEL_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_balanced_if_test_summary_by_channel.csv"
)

BALANCED_AGREEMENT_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_balanced_three_model_agreement.csv"
)

ORIGINAL_BALANCED_IF_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_original_vs_balanced_if_comparison.csv"
)

COMBINED_MODEL_SUMMARY_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_final_three_model_split_summary.csv"
)

COMBINED_CHANNEL_SUMMARY_PATH = (
    COMPARISON_TABLE_DIR /
    "ampds2_final_three_model_test_channel_summary.csv"
)

# ------------------------------------------------------------
# Features and fixed settings
# ------------------------------------------------------------

FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var"
]

SCALED_FEATURES = [
    f"{feature}_z"
    for feature in FEATURES
]

THRESHOLD_PERCENTILE = 95
RANDOM_STATE = 42

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------

reference_df = pd.read_csv(
    AMPDS2_REFERENCE_PATH,
    low_memory=False
)

evaluation_df = pd.read_csv(
    AMPDS2_THREE_MODEL_PATH,
    low_memory=False
)

required_columns = [
    "evaluation_split",
    "anomaly_score",
    "predicted_anomaly",
    "lof_anomaly_score",
    "lof_predicted_anomaly",
    "ae_anomaly_score",
    "ae_predicted_anomaly",
    "channel_code",
    "channel_name",
    "primary_category"
]

missing_columns = [
    column
    for column in required_columns
    if column not in evaluation_df.columns
]

if missing_columns:
    raise RuntimeError(
        f"Missing required columns: {missing_columns}"
    )

X_reference = reference_df[
    SCALED_FEATURES
].to_numpy(dtype=np.float32)

X_evaluation = evaluation_df[
    SCALED_FEATURES
].to_numpy(dtype=np.float32)

if not np.isfinite(X_reference).all():
    raise RuntimeError(
        "Reference features contain non-finite values."
    )

if not np.isfinite(X_evaluation).all():
    raise RuntimeError(
        "Evaluation features contain non-finite values."
    )

print("=" * 90)
print("AMPds2 BALANCED-REFERENCE ISOLATION FOREST")
print("=" * 90)

print("Reference matrix:", X_reference.shape)
print("Evaluation matrix:", X_evaluation.shape)

print("\nEvaluation split counts:")
print(
    evaluation_df["evaluation_split"]
    .value_counts()
    .reindex(["train", "val", "test"])
)

# ------------------------------------------------------------
# Train balanced-reference Isolation Forest
# ------------------------------------------------------------

balanced_if = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("\nTraining balanced-reference Isolation Forest...")

fit_start = time.perf_counter()

balanced_if.fit(X_reference)

fit_seconds = time.perf_counter() - fit_start

print(
    f"Training completed in {fit_seconds:.2f} seconds."
)

joblib.dump(
    balanced_if,
    BALANCED_IF_MODEL_PATH
)

# ------------------------------------------------------------
# Score identical evaluation rows
# ------------------------------------------------------------

score_start = time.perf_counter()

balanced_scores = -balanced_if.decision_function(
    X_evaluation
)

score_seconds = time.perf_counter() - score_start

evaluation_df[
    "if_balanced_anomaly_score"
] = balanced_scores.astype(np.float64)

validation_mask = (
    evaluation_df["evaluation_split"] == "val"
)

balanced_threshold = np.percentile(
    evaluation_df.loc[
        validation_mask,
        "if_balanced_anomaly_score"
    ],
    THRESHOLD_PERCENTILE
)

evaluation_df[
    "if_balanced_threshold"
] = balanced_threshold

evaluation_df[
    "if_balanced_predicted_anomaly"
] = (
    evaluation_df["if_balanced_anomaly_score"] >
    balanced_threshold
).astype(np.int8)

print(
    f"Scoring completed in {score_seconds:.2f} seconds."
)

print(
    "Balanced IF validation threshold:",
    balanced_threshold
)

# ------------------------------------------------------------
# Balanced IF overall summary
# ------------------------------------------------------------

balanced_overall_rows = []

for split_name in ["train", "val", "test"]:

    split_df = evaluation_df[
        evaluation_df["evaluation_split"] ==
        split_name
    ]

    total = len(split_df)

    flagged = int(
        split_df[
            "if_balanced_predicted_anomaly"
        ].sum()
    )

    balanced_overall_rows.append({
        "dataset": "AMPds2",
        "model": "Balanced-reference Isolation Forest",
        "split": split_name,
        "windows": total,
        "flagged_windows": flagged,
        "flagged_ratio": flagged / total,
        "flagged_ratio_percent": (
            100 * flagged / total
        ),
        "median_anomaly_score": split_df[
            "if_balanced_anomaly_score"
        ].median(),
        "score_q95": split_df[
            "if_balanced_anomaly_score"
        ].quantile(0.95),
        "score_q99": split_df[
            "if_balanced_anomaly_score"
        ].quantile(0.99),
        "maximum_anomaly_score": split_df[
            "if_balanced_anomaly_score"
        ].max(),
        "validation_threshold": balanced_threshold,
        "threshold_percentile": (
            THRESHOLD_PERCENTILE
        ),
        "reference_rows": len(reference_df),
        "fit_seconds": fit_seconds,
        "score_seconds": score_seconds
    })

balanced_overall = pd.DataFrame(
    balanced_overall_rows
)

print("\nBalanced Isolation Forest overall summary:")
display(balanced_overall)

# ------------------------------------------------------------
# Original full-data IF versus balanced-reference IF
# ------------------------------------------------------------

def binary_flag_metrics(flags_a, flags_b):

    flags_a = np.asarray(flags_a).astype(int)
    flags_b = np.asarray(flags_b).astype(int)

    both = int(
        ((flags_a == 1) & (flags_b == 1)).sum()
    )

    only_a = int(
        ((flags_a == 1) & (flags_b == 0)).sum()
    )

    only_b = int(
        ((flags_a == 0) & (flags_b == 1)).sum()
    )

    neither = int(
        ((flags_a == 0) & (flags_b == 0)).sum()
    )

    union = both + only_a + only_b

    return {
        "both_flagged": both,
        "only_model_a": only_a,
        "only_model_b": only_b,
        "neither_flagged": neither,
        "flag_agreement_rate": (
            (both + neither) / len(flags_a)
        ),
        "flag_jaccard_index": (
            both / union
            if union > 0 else np.nan
        )
    }


original_balanced_rows = []

for split_name in ["train", "val", "test"]:

    split_df = evaluation_df[
        evaluation_df["evaluation_split"] ==
        split_name
    ]

    original_flags = split_df[
        "predicted_anomaly"
    ].to_numpy()

    balanced_flags = split_df[
        "if_balanced_predicted_anomaly"
    ].to_numpy()

    metrics = binary_flag_metrics(
        original_flags,
        balanced_flags
    )

    rank_result = spearmanr(
        split_df["anomaly_score"],
        split_df["if_balanced_anomaly_score"]
    )

    original_balanced_rows.append({
        "dataset": "AMPds2",
        "split": split_name,
        "windows": len(split_df),
        "original_if_flagged": int(
            original_flags.sum()
        ),
        "balanced_if_flagged": int(
            balanced_flags.sum()
        ),
        **metrics,
        "score_spearman_correlation": (
            rank_result.statistic
        ),
        "score_spearman_p_value": (
            rank_result.pvalue
        )
    })

original_balanced_comparison = pd.DataFrame(
    original_balanced_rows
)

print(
    "\nOriginal full-data IF versus balanced-reference IF:"
)

display(original_balanced_comparison)

# ------------------------------------------------------------
# Fair pairwise and three-model agreement
# ------------------------------------------------------------

agreement_rows = []

for split_name in ["train", "val", "test"]:

    split_df = evaluation_df[
        evaluation_df["evaluation_split"] ==
        split_name
    ]

    if_flags = split_df[
        "if_balanced_predicted_anomaly"
    ].to_numpy(dtype=int)

    lof_flags = split_df[
        "lof_predicted_anomaly"
    ].to_numpy(dtype=int)

    ae_flags = split_df[
        "ae_predicted_anomaly"
    ].to_numpy(dtype=int)

    if_scores = split_df[
        "if_balanced_anomaly_score"
    ].to_numpy()

    lof_scores = split_df[
        "lof_anomaly_score"
    ].to_numpy()

    ae_scores = split_df[
        "ae_anomaly_score"
    ].to_numpy()

    comparisons = [
        (
            "Balanced Isolation Forest",
            "LOF",
            if_flags,
            lof_flags,
            if_scores,
            lof_scores
        ),
        (
            "Balanced Isolation Forest",
            "Autoencoder",
            if_flags,
            ae_flags,
            if_scores,
            ae_scores
        ),
        (
            "LOF",
            "Autoencoder",
            lof_flags,
            ae_flags,
            lof_scores,
            ae_scores
        )
    ]

    for (
        model_a,
        model_b,
        flags_a,
        flags_b,
        scores_a,
        scores_b
    ) in comparisons:

        metrics = binary_flag_metrics(
            flags_a,
            flags_b
        )

        rank_result = spearmanr(
            scores_a,
            scores_b
        )

        agreement_rows.append({
            "dataset": "AMPds2",
            "split": split_name,
            "comparison": (
                f"{model_a} vs {model_b}"
            ),
            "windows": len(split_df),
            "model_a_flagged": int(
                flags_a.sum()
            ),
            "model_b_flagged": int(
                flags_b.sum()
            ),
            **metrics,
            "score_spearman_correlation": (
                rank_result.statistic
            ),
            "score_spearman_p_value": (
                rank_result.pvalue
            ),
            "three_models_flagged": np.nan,
            "at_least_two_models_flagged": np.nan,
            "any_model_flagged": np.nan
        })

    model_flag_count = (
        if_flags +
        lof_flags +
        ae_flags
    )

    all_three = int(
        (model_flag_count == 3).sum()
    )

    at_least_two = int(
        (model_flag_count >= 2).sum()
    )

    any_model = int(
        (model_flag_count >= 1).sum()
    )

    agreement_rows.append({
        "dataset": "AMPds2",
        "split": split_name,
        "comparison": "Three-model consensus",
        "windows": len(split_df),
        "model_a_flagged": np.nan,
        "model_b_flagged": np.nan,
        "both_flagged": all_three,
        "only_model_a": np.nan,
        "only_model_b": np.nan,
        "neither_flagged": (
            len(split_df) - any_model
        ),
        "flag_agreement_rate": np.nan,
        "flag_jaccard_index": (
            all_three / any_model
            if any_model > 0 else np.nan
        ),
        "score_spearman_correlation": np.nan,
        "score_spearman_p_value": np.nan,
        "three_models_flagged": all_three,
        "at_least_two_models_flagged": (
            at_least_two
        ),
        "any_model_flagged": any_model
    })

balanced_agreement = pd.DataFrame(
    agreement_rows
)

print(
    "\nFinal fair balanced-reference model agreement:"
)

display(balanced_agreement)

# ------------------------------------------------------------
# Balanced IF test summary by channel
# ------------------------------------------------------------

test_df = evaluation_df[
    evaluation_df["evaluation_split"] == "test"
].copy()

balanced_if_channel_summary = (
    test_df.groupby(
        [
            "channel_code",
            "channel_name",
            "primary_category"
        ],
        dropna=False
    )
    .agg(
        test_windows=(
            "if_balanced_predicted_anomaly",
            "size"
        ),
        flagged_windows=(
            "if_balanced_predicted_anomaly",
            "sum"
        ),
        mean_anomaly_score=(
            "if_balanced_anomaly_score",
            "mean"
        ),
        median_anomaly_score=(
            "if_balanced_anomaly_score",
            "median"
        ),
        maximum_anomaly_score=(
            "if_balanced_anomaly_score",
            "max"
        )
    )
    .reset_index()
)

balanced_if_channel_summary[
    "flagged_ratio"
] = (
    balanced_if_channel_summary[
        "flagged_windows"
    ] /
    balanced_if_channel_summary[
        "test_windows"
    ]
)

balanced_if_channel_summary[
    "flagged_ratio_percent"
] = (
    100 *
    balanced_if_channel_summary[
        "flagged_ratio"
    ]
)

balanced_if_channel_summary = (
    balanced_if_channel_summary
    .sort_values(
        "flagged_ratio",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "\nBalanced IF test summary by channel:"
)

display(balanced_if_channel_summary)

# ------------------------------------------------------------
# Final three-model summary by split
# ------------------------------------------------------------

model_definitions = [
    (
        "Isolation Forest",
        "if_balanced_predicted_anomaly",
        "if_balanced_anomaly_score"
    ),
    (
        "Local Outlier Factor",
        "lof_predicted_anomaly",
        "lof_anomaly_score"
    ),
    (
        "Dense Autoencoder",
        "ae_predicted_anomaly",
        "ae_anomaly_score"
    )
]

combined_split_rows = []

for model_name, flag_column, score_column in model_definitions:

    for split_name in ["train", "val", "test"]:

        split_df = evaluation_df[
            evaluation_df["evaluation_split"] ==
            split_name
        ]

        flagged = int(
            split_df[flag_column].sum()
        )

        combined_split_rows.append({
            "dataset": "AMPds2",
            "model": model_name,
            "split": split_name,
            "windows": len(split_df),
            "flagged_windows": flagged,
            "flagged_ratio": (
                flagged / len(split_df)
            ),
            "flagged_ratio_percent": (
                100 * flagged / len(split_df)
            ),
            "median_model_score": (
                split_df[score_column].median()
            ),
            "score_q95": (
                split_df[score_column]
                .quantile(0.95)
            ),
            "score_q99": (
                split_df[score_column]
                .quantile(0.99)
            )
        })

combined_model_summary = pd.DataFrame(
    combined_split_rows
)

print("\nFinal three-model split summary:")
display(combined_model_summary)

# ------------------------------------------------------------
# Final three-model test summary by channel
# ------------------------------------------------------------

combined_channel_rows = []

for (
    channel_code,
    channel_name,
    category
), group in test_df.groupby(
    [
        "channel_code",
        "channel_name",
        "primary_category"
    ],
    dropna=False
):

    row = {
        "channel_code": channel_code,
        "channel_name": channel_name,
        "primary_category": category,
        "test_windows": len(group)
    }

    for model_short, flag_column in [
        (
            "if",
            "if_balanced_predicted_anomaly"
        ),
        (
            "lof",
            "lof_predicted_anomaly"
        ),
        (
            "ae",
            "ae_predicted_anomaly"
        )
    ]:
        flagged = int(
            group[flag_column].sum()
        )

        row[f"{model_short}_flagged_windows"] = flagged
        row[f"{model_short}_flagged_ratio_percent"] = (
            100 * flagged / len(group)
        )

    flag_count = (
        group[
            "if_balanced_predicted_anomaly"
        ].astype(int) +
        group[
            "lof_predicted_anomaly"
        ].astype(int) +
        group[
            "ae_predicted_anomaly"
        ].astype(int)
    )

    row["all_three_models_flagged"] = int(
        (flag_count == 3).sum()
    )

    row["at_least_two_models_flagged"] = int(
        (flag_count >= 2).sum()
    )

    row[
        "at_least_two_models_ratio_percent"
    ] = (
        100 *
        row["at_least_two_models_flagged"] /
        len(group)
    )

    combined_channel_rows.append(row)

combined_channel_summary = pd.DataFrame(
    combined_channel_rows
)

combined_channel_summary = (
    combined_channel_summary
    .sort_values(
        "at_least_two_models_ratio_percent",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "\nFinal three-model test-channel summary:"
)

display(combined_channel_summary)

# ------------------------------------------------------------
# Save all outputs
# ------------------------------------------------------------

evaluation_df.to_csv(
    BALANCED_IF_SCORED_PATH,
    index=False
)

balanced_overall.to_csv(
    BALANCED_IF_OVERALL_PATH,
    index=False
)

balanced_if_channel_summary.to_csv(
    BALANCED_IF_CHANNEL_PATH,
    index=False
)

balanced_agreement.to_csv(
    BALANCED_AGREEMENT_PATH,
    index=False
)

original_balanced_comparison.to_csv(
    ORIGINAL_BALANCED_IF_PATH,
    index=False
)

combined_model_summary.to_csv(
    COMBINED_MODEL_SUMMARY_PATH,
    index=False
)

combined_channel_summary.to_csv(
    COMBINED_CHANNEL_SUMMARY_PATH,
    index=False
)

print("\nSaved outputs:")
print(BALANCED_IF_MODEL_PATH)
print(BALANCED_IF_SCORED_PATH)
print(BALANCED_IF_OVERALL_PATH)
print(BALANCED_IF_CHANNEL_PATH)
print(BALANCED_AGREEMENT_PATH)
print(ORIGINAL_BALANCED_IF_PATH)
print(COMBINED_MODEL_SUMMARY_PATH)
print(COMBINED_CHANNEL_SUMMARY_PATH)

print("\nCell 10 complete.")

In [ ]:
# ============================================================
# Cell 11: Final UK-DALE and AMPds2 cross-model synthesis
# Produces thesis-ready tables and figures
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

COMPARISON_DIR = (
    BASE /
    "experiments" /
    "EXP_UKDALE_AMPDS2_Cross_Model_Comparison"
)

TABLE_DIR = COMPARISON_DIR / "tables"
FIGURE_DIR = COMPARISON_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

UKDALE_PATH = (
    TABLE_DIR /
    "ukdale_balanced_if_lof_ae_scored_evaluation.csv"
)

AMPDS2_PATH = (
    TABLE_DIR /
    "ampds2_balanced_if_lof_ae_scored_evaluation.csv"
)

# ------------------------------------------------------------
# Load final controlled evaluation files
# ------------------------------------------------------------

ukdale = pd.read_csv(
    UKDALE_PATH,
    low_memory=False
)

ampds2 = pd.read_csv(
    AMPDS2_PATH,
    low_memory=False
)

print("=" * 90)
print("FINAL CROSS-DATASET MODEL SYNTHESIS")
print("=" * 90)

print("UK-DALE:", ukdale.shape)
print("AMPds2:", ampds2.shape)

MODEL_DEFINITIONS = {
    "Isolation Forest": {
        "flag": "if_balanced_predicted_anomaly",
        "score": "if_balanced_anomaly_score"
    },
    "Local Outlier Factor": {
        "flag": "lof_predicted_anomaly",
        "score": "lof_anomaly_score"
    },
    "Dense Autoencoder": {
        "flag": "ae_predicted_anomaly",
        "score": "ae_anomaly_score"
    }
}

# ------------------------------------------------------------
# 1. Final model summary by dataset and split
# ------------------------------------------------------------

split_rows = []

for dataset_name, df in [
    ("UK-DALE", ukdale),
    ("AMPds2", ampds2)
]:
    for model_name, columns in MODEL_DEFINITIONS.items():
        for split_name in ["train", "val", "test"]:

            split_df = df[
                df["evaluation_split"] == split_name
            ]

            total = len(split_df)

            flagged = int(
                split_df[columns["flag"]].sum()
            )

            split_rows.append({
                "dataset": dataset_name,
                "model": model_name,
                "split": split_name,
                "windows": total,
                "flagged_windows": flagged,
                "flagged_ratio": flagged / total,
                "flagged_ratio_percent": (
                    100 * flagged / total
                ),
                "median_score": split_df[
                    columns["score"]
                ].median(),
                "score_q95": split_df[
                    columns["score"]
                ].quantile(0.95),
                "score_q99": split_df[
                    columns["score"]
                ].quantile(0.99)
            })

final_split_summary = pd.DataFrame(split_rows)

FINAL_SPLIT_PATH = (
    TABLE_DIR /
    "final_cross_dataset_three_model_split_summary.csv"
)

final_split_summary.to_csv(
    FINAL_SPLIT_PATH,
    index=False
)

print("\nFinal model summary by dataset and split:")
display(final_split_summary)

# ------------------------------------------------------------
# 2. Pairwise test-model agreement
# ------------------------------------------------------------

def calculate_pairwise_agreement(
    dataset_name,
    test_df,
    model_a,
    model_b
):
    definition_a = MODEL_DEFINITIONS[model_a]
    definition_b = MODEL_DEFINITIONS[model_b]

    flags_a = test_df[
        definition_a["flag"]
    ].astype(int).to_numpy()

    flags_b = test_df[
        definition_b["flag"]
    ].astype(int).to_numpy()

    scores_a = test_df[
        definition_a["score"]
    ].to_numpy()

    scores_b = test_df[
        definition_b["score"]
    ].to_numpy()

    both = int(
        ((flags_a == 1) & (flags_b == 1)).sum()
    )

    only_a = int(
        ((flags_a == 1) & (flags_b == 0)).sum()
    )

    only_b = int(
        ((flags_a == 0) & (flags_b == 1)).sum()
    )

    neither = int(
        ((flags_a == 0) & (flags_b == 0)).sum()
    )

    union = both + only_a + only_b

    correlation = spearmanr(
        scores_a,
        scores_b
    )

    return {
        "dataset": dataset_name,
        "model_a": model_a,
        "model_b": model_b,
        "test_windows": len(test_df),
        "model_a_flagged": int(flags_a.sum()),
        "model_b_flagged": int(flags_b.sum()),
        "both_flagged": both,
        "model_a_only": only_a,
        "model_b_only": only_b,
        "neither_flagged": neither,
        "flag_jaccard_index": (
            both / union if union else np.nan
        ),
        "score_spearman_correlation": (
            correlation.statistic
        ),
        "score_spearman_p_value": (
            correlation.pvalue
        )
    }


model_pairs = [
    (
        "Isolation Forest",
        "Local Outlier Factor"
    ),
    (
        "Isolation Forest",
        "Dense Autoencoder"
    ),
    (
        "Local Outlier Factor",
        "Dense Autoencoder"
    )
]

agreement_rows = []

for dataset_name, df in [
    ("UK-DALE", ukdale),
    ("AMPds2", ampds2)
]:
    test_df = df[
        df["evaluation_split"] == "test"
    ].copy()

    for model_a, model_b in model_pairs:
        agreement_rows.append(
            calculate_pairwise_agreement(
                dataset_name,
                test_df,
                model_a,
                model_b
            )
        )

pairwise_agreement = pd.DataFrame(
    agreement_rows
)

PAIRWISE_PATH = (
    TABLE_DIR /
    "final_cross_dataset_test_pairwise_agreement.csv"
)

pairwise_agreement.to_csv(
    PAIRWISE_PATH,
    index=False
)

print("\nFinal test pairwise model agreement:")
display(pairwise_agreement)

# ------------------------------------------------------------
# 3. Dataset-level consensus summary
# ------------------------------------------------------------

consensus_rows = []

for dataset_name, df in [
    ("UK-DALE", ukdale),
    ("AMPds2", ampds2)
]:
    test_df = df[
        df["evaluation_split"] == "test"
    ].copy()

    flag_count = (
        test_df[
            "if_balanced_predicted_anomaly"
        ].astype(int)
        +
        test_df[
            "lof_predicted_anomaly"
        ].astype(int)
        +
        test_df[
            "ae_predicted_anomaly"
        ].astype(int)
    )

    total = len(test_df)

    all_three = int(
        (flag_count == 3).sum()
    )

    at_least_two = int(
        (flag_count >= 2).sum()
    )

    any_model = int(
        (flag_count >= 1).sum()
    )

    consensus_rows.append({
        "dataset": dataset_name,
        "test_windows": total,
        "all_three_models_flagged": all_three,
        "all_three_ratio_percent": (
            100 * all_three / total
        ),
        "at_least_two_models_flagged": (
            at_least_two
        ),
        "at_least_two_ratio_percent": (
            100 * at_least_two / total
        ),
        "any_model_flagged": any_model,
        "any_model_ratio_percent": (
            100 * any_model / total
        )
    })

consensus_summary = pd.DataFrame(
    consensus_rows
)

CONSENSUS_PATH = (
    TABLE_DIR /
    "final_cross_dataset_test_consensus_summary.csv"
)

consensus_summary.to_csv(
    CONSENSUS_PATH,
    index=False
)

print("\nFinal test consensus summary:")
display(consensus_summary)

# ------------------------------------------------------------
# 4. UK-DALE final category summary
# ------------------------------------------------------------

ukdale_test = ukdale[
    ukdale["evaluation_split"] == "test"
].copy()

ukdale_category_rows = []

for category, group in ukdale_test.groupby(
    "primary_category",
    dropna=False
):
    flag_count = (
        group[
            "if_balanced_predicted_anomaly"
        ].astype(int)
        +
        group[
            "lof_predicted_anomaly"
        ].astype(int)
        +
        group[
            "ae_predicted_anomaly"
        ].astype(int)
    )

    row = {
        "primary_category": category,
        "test_windows": len(group)
    }

    for short_name, flag_column in [
        (
            "if",
            "if_balanced_predicted_anomaly"
        ),
        (
            "lof",
            "lof_predicted_anomaly"
        ),
        (
            "ae",
            "ae_predicted_anomaly"
        )
    ]:
        flagged = int(
            group[flag_column].sum()
        )

        row[
            f"{short_name}_flagged_windows"
        ] = flagged

        row[
            f"{short_name}_flagged_ratio_percent"
        ] = (
            100 * flagged / len(group)
        )

    row["all_three_models_flagged"] = int(
        (flag_count == 3).sum()
    )

    row["at_least_two_models_flagged"] = int(
        (flag_count >= 2).sum()
    )

    row[
        "at_least_two_models_ratio_percent"
    ] = (
        100 *
        row["at_least_two_models_flagged"] /
        len(group)
    )

    ukdale_category_rows.append(row)

ukdale_category_summary = pd.DataFrame(
    ukdale_category_rows
).sort_values(
    "at_least_two_models_ratio_percent",
    ascending=False
).reset_index(drop=True)

UKDALE_CATEGORY_PATH = (
    TABLE_DIR /
    "ukdale_final_three_model_test_category_summary.csv"
)

ukdale_category_summary.to_csv(
    UKDALE_CATEGORY_PATH,
    index=False
)

print("\nUK-DALE final three-model category summary:")
display(ukdale_category_summary)

# ------------------------------------------------------------
# 5. AMPds2 final channel summary
# ------------------------------------------------------------

ampds2_test = ampds2[
    ampds2["evaluation_split"] == "test"
].copy()

ampds2_channel_rows = []

for (
    channel_code,
    channel_name,
    category
), group in ampds2_test.groupby(
    [
        "channel_code",
        "channel_name",
        "primary_category"
    ],
    dropna=False
):
    flag_count = (
        group[
            "if_balanced_predicted_anomaly"
        ].astype(int)
        +
        group[
            "lof_predicted_anomaly"
        ].astype(int)
        +
        group[
            "ae_predicted_anomaly"
        ].astype(int)
    )

    row = {
        "channel_code": channel_code,
        "channel_name": channel_name,
        "primary_category": category,
        "test_windows": len(group)
    }

    for short_name, flag_column in [
        (
            "if",
            "if_balanced_predicted_anomaly"
        ),
        (
            "lof",
            "lof_predicted_anomaly"
        ),
        (
            "ae",
            "ae_predicted_anomaly"
        )
    ]:
        flagged = int(
            group[flag_column].sum()
        )

        row[
            f"{short_name}_flagged_windows"
        ] = flagged

        row[
            f"{short_name}_flagged_ratio_percent"
        ] = (
            100 * flagged / len(group)
        )

    row["all_three_models_flagged"] = int(
        (flag_count == 3).sum()
    )

    row["at_least_two_models_flagged"] = int(
        (flag_count >= 2).sum()
    )

    row[
        "at_least_two_models_ratio_percent"
    ] = (
        100 *
        row["at_least_two_models_flagged"] /
        len(group)
    )

    ampds2_channel_rows.append(row)

ampds2_channel_summary = pd.DataFrame(
    ampds2_channel_rows
).sort_values(
    "at_least_two_models_ratio_percent",
    ascending=False
).reset_index(drop=True)

AMPDS2_CHANNEL_PATH = (
    TABLE_DIR /
    "ampds2_final_three_model_test_channel_summary.csv"
)

ampds2_channel_summary.to_csv(
    AMPDS2_CHANNEL_PATH,
    index=False
)

print("\nAMPds2 final three-model channel summary:")
display(ampds2_channel_summary)

# ============================================================
# FIGURES
# ============================================================

model_order = [
    "Isolation Forest",
    "Local Outlier Factor",
    "Dense Autoencoder"
]

split_order = [
    "train",
    "val",
    "test"
]

# ------------------------------------------------------------
# Figure 1: Flagged ratio by split and model
# ------------------------------------------------------------

for dataset_name in [
    "UK-DALE",
    "AMPds2"
]:
    plotting_df = final_split_summary[
        final_split_summary["dataset"] ==
        dataset_name
    ].copy()

    pivot_df = plotting_df.pivot(
        index="split",
        columns="model",
        values="flagged_ratio_percent"
    ).reindex(
        split_order
    )[model_order]

    ax = pivot_df.plot(
        kind="bar",
        figsize=(9, 5)
    )

    ax.set_xlabel("Split")
    ax.set_ylabel("Flagged windows (%)")
    ax.set_title(
        f"{dataset_name} flagged-window ratio by model and split"
    )

    plt.xticks(rotation=0)
    plt.legend(
        title="Model"
    )
    plt.tight_layout()

    figure_name = (
        dataset_name
        .lower()
        .replace("-", "")
        .replace(" ", "_")
    )

    output_path = (
        FIGURE_DIR /
        f"{figure_name}_three_model_flagged_ratio_by_split.png"
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

# ------------------------------------------------------------
# Figure 2: UK-DALE test-category comparison
# ------------------------------------------------------------

ukdale_plot = (
    ukdale_category_summary[
        [
            "primary_category",
            "if_flagged_ratio_percent",
            "lof_flagged_ratio_percent",
            "ae_flagged_ratio_percent"
        ]
    ]
    .set_index("primary_category")
)

ukdale_plot.columns = [
    "Isolation Forest",
    "Local Outlier Factor",
    "Dense Autoencoder"
]

ax = ukdale_plot.plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_xlabel("Appliance category")
ax.set_ylabel("Flagged test windows (%)")
ax.set_title(
    "UK-DALE test flagged-window ratio by model and category"
)

plt.xticks(
    rotation=35,
    ha="right"
)

plt.legend(
    title="Model"
)

plt.tight_layout()

UKDALE_FIGURE_PATH = (
    FIGURE_DIR /
    "ukdale_three_model_test_ratio_by_category.png"
)

plt.savefig(
    UKDALE_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

# ------------------------------------------------------------
# Figure 3: AMPds2 strongest consensus channels
# ------------------------------------------------------------

ampds2_top = (
    ampds2_channel_summary
    .head(10)
    .copy()
)

ampds2_top["channel_label"] = (
    ampds2_top["primary_category"]
    + "\n(" +
    ampds2_top["channel_code"]
    + ")"
)

ax = ampds2_top.plot(
    x="channel_label",
    y="at_least_two_models_ratio_percent",
    kind="bar",
    figsize=(11, 5),
    legend=False
)

ax.set_xlabel("AMPds2 channel")
ax.set_ylabel(
    "Windows flagged by at least two models (%)"
)
ax.set_title(
    "AMPds2 strongest cross-model test-window consensus"
)

plt.xticks(
    rotation=40,
    ha="right"
)

plt.tight_layout()

AMPDS2_CONSENSUS_FIGURE_PATH = (
    FIGURE_DIR /
    "ampds2_test_at_least_two_model_consensus_top_channels.png"
)

plt.savefig(
    AMPDS2_CONSENSUS_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

# ------------------------------------------------------------
# Figure 4: Pairwise test score correlation
# ------------------------------------------------------------

pairwise_plot = pairwise_agreement.copy()

pairwise_plot["comparison"] = (
    pairwise_plot["model_a"]
    .str.replace(
        "Local Outlier Factor",
        "LOF",
        regex=False
    )
    +
    " vs "
    +
    pairwise_plot["model_b"]
    .str.replace(
        "Local Outlier Factor",
        "LOF",
        regex=False
    )
)

correlation_pivot = pairwise_plot.pivot(
    index="comparison",
    columns="dataset",
    values="score_spearman_correlation"
)

ax = correlation_pivot.plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_xlabel("Model comparison")
ax.set_ylabel("Spearman score correlation")
ax.set_ylim(0, 1)
ax.set_title(
    "Cross-model anomaly-score agreement on test windows"
)

plt.xticks(
    rotation=25,
    ha="right"
)

plt.legend(
    title="Dataset"
)

plt.tight_layout()

CORRELATION_FIGURE_PATH = (
    FIGURE_DIR /
    "cross_dataset_test_score_spearman_comparison.png"
)

plt.savefig(
    CORRELATION_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

# ------------------------------------------------------------
# Final saved-output list
# ------------------------------------------------------------

print("\nSaved final tables:")
print(FINAL_SPLIT_PATH)
print(PAIRWISE_PATH)
print(CONSENSUS_PATH)
print(UKDALE_CATEGORY_PATH)
print(AMPDS2_CHANNEL_PATH)

print("\nSaved final figures:")
print(
    FIGURE_DIR /
    "ukdale_three_model_flagged_ratio_by_split.png"
)
print(
    FIGURE_DIR /
    "ampds2_three_model_flagged_ratio_by_split.png"
)
print(UKDALE_FIGURE_PATH)
print(AMPDS2_CONSENSUS_FIGURE_PATH)
print(CORRELATION_FIGURE_PATH)

print("\nCell 11 complete.")

In [ ]:
# ============================================================
# Cell 12: Final code-freeze audit and experiment manifest
# No model training
# ============================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import subprocess
import os

# ------------------------------------------------------------
# Main directories
# ------------------------------------------------------------

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

PROCESSED_DIR = BASE / "processed_data"
EXPERIMENTS_DIR = BASE / "experiments"

UKDALE_DATA_DIR = (
    PROCESSED_DIR /
    "UKDALE_Selected_Pipeline"
)

AMPDS2_DATA_DIR = (
    PROCESSED_DIR /
    "AMPDS2_Selected_Pipeline"
)

UKDALE_EXP_DIR = (
    EXPERIMENTS_DIR /
    "EXP_UKDALE_Selected_Pipeline"
)

AMPDS2_EXP_DIR = (
    EXPERIMENTS_DIR /
    "EXP_AMPDS2_Selected_Pipeline"
)

COMPARISON_DIR = (
    EXPERIMENTS_DIR /
    "EXP_UKDALE_AMPDS2_Cross_Model_Comparison"
)

TABLE_DIR = COMPARISON_DIR / "tables"
FIGURE_DIR = COMPARISON_DIR / "figures"
MODEL_DIR = COMPARISON_DIR / "models"

FREEZE_DIR = BASE / "documentation" / "CODE_FREEZE"
FREEZE_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 90)
print("FINAL THESIS CODE-FREEZE AUDIT")
print("=" * 90)

print("Base directory:", BASE)
print("Base exists:", BASE.exists())

# ------------------------------------------------------------
# Critical artifacts
# ------------------------------------------------------------

critical_artifacts = {
    # Full no-injection Isolation Forest outputs
    "UK-DALE full IF scored data":
        UKDALE_DATA_DIR /
        "ukdale_clean_main_if_scored_no_injection.csv",

    "AMPds2 full IF scored data":
        AMPDS2_DATA_DIR /
        "ampds2_p_if_scored_no_injection.csv",

    # Controlled comparison samples
    "UK-DALE reference sample":
        TABLE_DIR /
        "ukdale_lof_reference_sample.csv",

    "UK-DALE evaluation sample":
        TABLE_DIR /
        "ukdale_cross_model_evaluation_sample.csv",

    "AMPds2 reference sample":
        TABLE_DIR /
        "ampds2_lof_reference_sample.csv",

    "AMPds2 evaluation sample":
        TABLE_DIR /
        "ampds2_cross_model_evaluation_sample.csv",

    # Final controlled scored files
    "UK-DALE final three-model scored data":
        TABLE_DIR /
        "ukdale_balanced_if_lof_ae_scored_evaluation.csv",

    "AMPds2 final three-model scored data":
        TABLE_DIR /
        "ampds2_balanced_if_lof_ae_scored_evaluation.csv",

    # Final synthesis tables
    "Cross-dataset split summary":
        TABLE_DIR /
        "final_cross_dataset_three_model_split_summary.csv",

    "Cross-dataset pairwise agreement":
        TABLE_DIR /
        "final_cross_dataset_test_pairwise_agreement.csv",

    "Cross-dataset consensus summary":
        TABLE_DIR /
        "final_cross_dataset_test_consensus_summary.csv",

    "UK-DALE category summary":
        TABLE_DIR /
        "ukdale_final_three_model_test_category_summary.csv",

    "AMPds2 channel summary":
        TABLE_DIR /
        "ampds2_final_three_model_test_channel_summary.csv",

    # Training histories
    "UK-DALE Autoencoder history":
        TABLE_DIR /
        "ukdale_autoencoder_training_history.csv",

    "AMPds2 Autoencoder history":
        TABLE_DIR /
        "ampds2_autoencoder_training_history.csv",

    # Models
    "UK-DALE LOF model":
        MODEL_DIR /
        "ukdale_lof_novelty_model.joblib",

    "AMPds2 LOF model":
        MODEL_DIR /
        "ampds2_lof_novelty_model.joblib",

    "UK-DALE Autoencoder model":
        MODEL_DIR /
        "ukdale_dense_autoencoder.keras",

    "AMPds2 Autoencoder model":
        MODEL_DIR /
        "ampds2_dense_autoencoder.keras",

    "UK-DALE balanced Isolation Forest model":
        MODEL_DIR /
        "ukdale_balanced_reference_isolation_forest.joblib",

    "AMPds2 balanced Isolation Forest model":
        MODEL_DIR /
        "ampds2_balanced_reference_isolation_forest.joblib",

    # Final figures
    "UK-DALE split comparison figure":
        FIGURE_DIR /
        "ukdale_three_model_flagged_ratio_by_split.png",

    "AMPds2 split comparison figure":
        FIGURE_DIR /
        "ampds2_three_model_flagged_ratio_by_split.png",

    "UK-DALE category comparison figure":
        FIGURE_DIR /
        "ukdale_three_model_test_ratio_by_category.png",

    "AMPds2 consensus figure":
        FIGURE_DIR /
        "ampds2_test_at_least_two_model_consensus_top_channels.png",

    "Cross-dataset score correlation figure":
        FIGURE_DIR /
        "cross_dataset_test_score_spearman_comparison.png"
}

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def count_csv_rows(path):
    """
    Counts CSV data rows without loading the full file.
    Header row is excluded.
    """
    try:
        result = subprocess.run(
            ["wc", "-l", str(path)],
            capture_output=True,
            text=True,
            check=True
        )

        line_count = int(
            result.stdout.strip().split()[0]
        )

        return max(line_count - 1, 0)

    except Exception:
        return None


def format_size(size_bytes):
    if size_bytes < 1024:
        return f"{size_bytes} B"

    if size_bytes < 1024 ** 2:
        return f"{size_bytes / 1024:.2f} KB"

    if size_bytes < 1024 ** 3:
        return f"{size_bytes / (1024 ** 2):.2f} MB"

    return f"{size_bytes / (1024 ** 3):.2f} GB"


# ------------------------------------------------------------
# Audit critical files
# ------------------------------------------------------------

audit_rows = []

for artifact_name, path in critical_artifacts.items():

    exists = path.exists()

    size_bytes = (
        path.stat().st_size
        if exists else None
    )

    modified_time = (
        datetime.fromtimestamp(
            path.stat().st_mtime
        ).isoformat(timespec="seconds")
        if exists else None
    )

    csv_rows = (
        count_csv_rows(path)
        if exists and path.suffix.lower() == ".csv"
        else None
    )

    audit_rows.append({
        "artifact": artifact_name,
        "path": str(path),
        "exists": exists,
        "size_bytes": size_bytes,
        "size_readable": (
            format_size(size_bytes)
            if size_bytes is not None
            else None
        ),
        "csv_data_rows": csv_rows,
        "modified_time": modified_time
    })

audit_df = pd.DataFrame(audit_rows)

print("\nCritical-artifact audit:")
display(audit_df)

missing_df = audit_df[
    audit_df["exists"] == False
].copy()

print("\nMissing critical artifacts:", len(missing_df))

if len(missing_df) > 0:
    display(
        missing_df[
            ["artifact", "path"]
        ]
    )

# ------------------------------------------------------------
# Validate expected controlled sample sizes
# ------------------------------------------------------------

expected_rows = {
    "UK-DALE reference sample": 12000,
    "UK-DALE evaluation sample": 19694,
    "AMPds2 reference sample": 15750,
    "AMPds2 evaluation sample": 26250,
    "UK-DALE final three-model scored data": 19694,
    "AMPds2 final three-model scored data": 26250
}

row_check_rows = []

for artifact_name, expected_count in expected_rows.items():

    match = audit_df[
        audit_df["artifact"] == artifact_name
    ]

    actual_count = (
        int(match.iloc[0]["csv_data_rows"])
        if (
            len(match) == 1 and
            pd.notna(match.iloc[0]["csv_data_rows"])
        )
        else None
    )

    row_check_rows.append({
        "artifact": artifact_name,
        "expected_rows": expected_count,
        "actual_rows": actual_count,
        "matches_expected": (
            actual_count == expected_count
            if actual_count is not None
            else False
        )
    })

row_check_df = pd.DataFrame(
    row_check_rows
)

print("\nControlled sample-size validation:")
display(row_check_df)

# ------------------------------------------------------------
# Discover XAI outputs automatically
# ------------------------------------------------------------

xai_files = []

for search_dir in [
    UKDALE_EXP_DIR,
    AMPDS2_EXP_DIR
]:
    if search_dir.exists():
        for pattern in [
            "*shap*.csv",
            "*lime*.csv",
            "*SHAP*.csv",
            "*LIME*.csv"
        ]:
            xai_files.extend(
                search_dir.rglob(pattern)
            )

xai_files = sorted(
    set(xai_files)
)

xai_rows = []

for path in xai_files:
    xai_rows.append({
        "artifact_type": "XAI output",
        "path": str(path),
        "filename": path.name,
        "size_readable": format_size(
            path.stat().st_size
        )
    })

xai_df = pd.DataFrame(xai_rows)

print("\nDiscovered SHAP/LIME outputs:", len(xai_df))

if len(xai_df) > 0:
    display(xai_df)
else:
    print(
        "No SHAP/LIME CSV files were discovered automatically."
    )

# ------------------------------------------------------------
# Inventory all experiment files
# ------------------------------------------------------------

inventory_rows = []

for root_dir in [
    UKDALE_DATA_DIR,
    AMPDS2_DATA_DIR,
    UKDALE_EXP_DIR,
    AMPDS2_EXP_DIR,
    COMPARISON_DIR
]:
    if not root_dir.exists():
        continue

    for path in root_dir.rglob("*"):

        if not path.is_file():
            continue

        inventory_rows.append({
            "root_section": root_dir.name,
            "relative_path": str(
                path.relative_to(BASE)
            ),
            "extension": path.suffix.lower(),
            "size_bytes": path.stat().st_size,
            "size_readable": format_size(
                path.stat().st_size
            ),
            "modified_time": datetime.fromtimestamp(
                path.stat().st_mtime
            ).isoformat(timespec="seconds")
        })

inventory_df = pd.DataFrame(
    inventory_rows
).sort_values(
    ["root_section", "relative_path"]
).reset_index(drop=True)

print("\nTotal inventoried thesis artifacts:", len(inventory_df))

# ------------------------------------------------------------
# Save audit and inventory
# ------------------------------------------------------------

AUDIT_PATH = (
    FREEZE_DIR /
    "critical_artifact_audit.csv"
)

ROW_CHECK_PATH = (
    FREEZE_DIR /
    "controlled_sample_size_validation.csv"
)

INVENTORY_PATH = (
    FREEZE_DIR /
    "complete_experiment_artifact_manifest.csv"
)

XAI_INVENTORY_PATH = (
    FREEZE_DIR /
    "xai_artifact_manifest.csv"
)

audit_df.to_csv(
    AUDIT_PATH,
    index=False
)

row_check_df.to_csv(
    ROW_CHECK_PATH,
    index=False
)

inventory_df.to_csv(
    INVENTORY_PATH,
    index=False
)

xai_df.to_csv(
    XAI_INVENTORY_PATH,
    index=False
)

# ------------------------------------------------------------
# Read final result summaries
# ------------------------------------------------------------

split_summary_path = (
    TABLE_DIR /
    "final_cross_dataset_three_model_split_summary.csv"
)

agreement_path = (
    TABLE_DIR /
    "final_cross_dataset_test_pairwise_agreement.csv"
)

consensus_path = (
    TABLE_DIR /
    "final_cross_dataset_test_consensus_summary.csv"
)

split_summary = (
    pd.read_csv(split_summary_path)
    if split_summary_path.exists()
    else pd.DataFrame()
)

agreement_summary = (
    pd.read_csv(agreement_path)
    if agreement_path.exists()
    else pd.DataFrame()
)

consensus_summary = (
    pd.read_csv(consensus_path)
    if consensus_path.exists()
    else pd.DataFrame()
)

# ------------------------------------------------------------
# Create code-freeze README
# ------------------------------------------------------------

freeze_timestamp = datetime.now().isoformat(
    timespec="seconds"
)

all_critical_exist = (
    audit_df["exists"].all()
)

all_row_checks_pass = (
    row_check_df["matches_expected"].all()
)

freeze_status = (
    "PASSED"
    if (
        all_critical_exist and
        all_row_checks_pass
    )
    else "REVIEW REQUIRED"
)

readme_lines = [
    "# Thesis Experimental Code Freeze",
    "",
    f"**Freeze timestamp:** {freeze_timestamp}",
    f"**Audit status:** {freeze_status}",
    "",
    "## Final implemented scope",
    "",
    "- Datasets: NAB, UK-DALE, and AMPds2.",
    "- Window size: 60 minutes.",
    "- Window stride: 30 minutes.",
    "- Feature representation: 23 window-level statistical features.",
    "- Normalization: per-series/per-channel z-score using training statistics only.",
    "- Chronological split: 60% training, 20% validation, and 20% test.",
    "- Anomaly-detection models: Isolation Forest, Local Outlier Factor, and dense Autoencoder.",
    "- Operating threshold: 95th percentile of validation anomaly scores.",
    "- Explainability methods: SHAP and LIME.",
    "- Synthetic anomaly injection was not used for UK-DALE or AMPds2.",
    "",
    "## Controlled cross-model comparison",
    "",
    "- UK-DALE reference sample: 12,000 windows.",
    "- UK-DALE evaluation sample: 19,694 windows.",
    "- AMPds2 reference sample: 15,750 windows.",
    "- AMPds2 evaluation sample: 26,250 windows.",
    "- Random state: 42.",
    "- LOF neighbors: 35.",
    "- Isolation Forest estimators: 300.",
    "- Autoencoder architecture: 23-16-8-4-8-16-23.",
    "- Autoencoder maximum epochs: 150.",
    "- Autoencoder batch size: 256.",
    "",
    "## Interpretation constraints",
    "",
    "- UK-DALE and AMPds2 do not provide ground-truth anomaly labels for the evaluated windows.",
    "- Their results must be described as model-flagged windows or anomalous-score patterns.",
    "- Precision, Recall, F1-score, and false positive rate must not be reported for these unlabeled experiments.",
    "- Raw LOF score magnitudes must not be compared directly with Isolation Forest or Autoencoder scores.",
    "- Cross-model comparisons should use flagged ratios, rankings, Spearman correlation, Jaccard overlap, and consensus.",
    "",
    "## Code-freeze rule",
    "",
    "No additional model training, hyperparameter expansion, dataset expansion, or feature redesign should be performed unless explicitly requested by the supervisor.",
    "",
    "Permitted post-freeze changes:",
    "",
    "- correcting figure labels;",
    "- regenerating presentation-quality figures;",
    "- verifying table values;",
    "- correcting implementation errors;",
    "- exporting additional summaries from already completed results.",
    "",
    "## Final artifact locations",
    "",
    f"- Processed datasets: `{PROCESSED_DIR}`",
    f"- UK-DALE experiment: `{UKDALE_EXP_DIR}`",
    f"- AMPds2 experiment: `{AMPDS2_EXP_DIR}`",
    f"- Cross-model comparison: `{COMPARISON_DIR}`",
    f"- Final tables: `{TABLE_DIR}`",
    f"- Final figures: `{FIGURE_DIR}`",
    f"- Saved models: `{MODEL_DIR}`",
    f"- Code-freeze documentation: `{FREEZE_DIR}`",
    "",
    "## Final cross-model summaries",
    ""
]

if not split_summary.empty:
    readme_lines.extend([
        "### Split-level flagged-window ratios",
        "",
        split_summary[
            [
                "dataset",
                "model",
                "split",
                "windows",
                "flagged_windows",
                "flagged_ratio_percent"
            ]
        ].to_markdown(index=False),
        ""
    ])

if not agreement_summary.empty:
    readme_lines.extend([
        "### Test pairwise agreement",
        "",
        agreement_summary[
            [
                "dataset",
                "model_a",
                "model_b",
                "flag_jaccard_index",
                "score_spearman_correlation"
            ]
        ].to_markdown(index=False),
        ""
    ])

if not consensus_summary.empty:
    readme_lines.extend([
        "### Test consensus",
        "",
        consensus_summary.to_markdown(
            index=False
        ),
        ""
    ])

README_PATH = (
    FREEZE_DIR /
    "README_CODE_FREEZE.md"
)

README_PATH.write_text(
    "\n".join(readme_lines),
    encoding="utf-8"
)

# ------------------------------------------------------------
# Create final status marker
# ------------------------------------------------------------

STATUS_PATH = (
    FREEZE_DIR /
    "CODE_FREEZE_STATUS.txt"
)

STATUS_PATH.write_text(
    "\n".join([
        f"STATUS={freeze_status}",
        f"TIMESTAMP={freeze_timestamp}",
        f"CRITICAL_ARTIFACTS={len(audit_df)}",
        f"MISSING_ARTIFACTS={len(missing_df)}",
        f"SAMPLE_SIZE_CHECKS_PASSED={int(row_check_df['matches_expected'].sum())}/{len(row_check_df)}",
        f"XAI_FILES_DISCOVERED={len(xai_df)}",
        f"TOTAL_FILES_INVENTORIED={len(inventory_df)}"
    ]),
    encoding="utf-8"
)

# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CODE-FREEZE RESULT")
print("=" * 90)

print("Status:", freeze_status)
print("Missing critical artifacts:", len(missing_df))
print(
    "Sample-size checks passed:",
    f"{int(row_check_df['matches_expected'].sum())}/{len(row_check_df)}"
)
print("XAI files discovered:", len(xai_df))
print("Total artifacts inventoried:", len(inventory_df))

print("\nSaved code-freeze files:")
print(AUDIT_PATH)
print(ROW_CHECK_PATH)
print(INVENTORY_PATH)
print(XAI_INVENTORY_PATH)
print(README_PATH)
print(STATUS_PATH)

if freeze_status == "PASSED":
    print(
        "\nThe experimental implementation is ready for code freeze."
    )
else:
    print(
        "\nReview the missing files or failed row-count checks before freezing."
    )

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import os
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
)

import tensorflow as tf

warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# ---------------------------------------------------------
# Main thesis paths
# ---------------------------------------------------------
THESIS_ROOT = Path("/content/drive/MyDrive/Master/Thesis/Code")

PROCESSED_ROOT = THESIS_ROOT / "processed_data"
NAB_ROOT = PROCESSED_ROOT / "PH3" / "NAB"

NAB_FEATURES_DIR = NAB_ROOT / "raw_windows" / "features"
NAB_SEQUENCES_DIR = NAB_ROOT / "raw_windows" / "sequence"

EXPERIMENTS_ROOT = THESIS_ROOT / "experiments"
NEW_EXPERIMENT_ROOT = (
    EXPERIMENTS_ROOT / "EXP_NAB_OCSVM_LSTM_Autoencoder"
)

OCSVM_OUTPUT_DIR = NEW_EXPERIMENT_ROOT / "OneClassSVM"
LSTM_OUTPUT_DIR = NEW_EXPERIMENT_ROOT / "LSTM_Autoencoder"
COMMON_OUTPUT_DIR = NEW_EXPERIMENT_ROOT / "common"

for folder in [
    NEW_EXPERIMENT_ROOT,
    OCSVM_OUTPUT_DIR,
    LSTM_OUTPUT_DIR,
    COMMON_OUTPUT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Confirm paths
# ---------------------------------------------------------
paths_to_check = {
    "Thesis root": THESIS_ROOT,
    "Processed root": PROCESSED_ROOT,
    "NAB root": NAB_ROOT,
    "NAB feature windows": NAB_FEATURES_DIR,
    "NAB sequence windows": NAB_SEQUENCES_DIR,
    "New experiment folder": NEW_EXPERIMENT_ROOT,
}

print("Environment")
print("-" * 70)
print("Python/TensorFlow setup ready")
print(f"TensorFlow version: {tf.__version__}")
print(f"Random state: {RANDOM_STATE}")

print("\nPath verification")
print("-" * 70)

all_required_exist = True

for name, path in paths_to_check.items():
    exists = path.exists()
    print(f"{name:25s}: {exists} | {path}")

    if name != "New experiment folder" and not exists:
        all_required_exist = False

if not all_required_exist:
    raise FileNotFoundError(
        "One or more required thesis folders were not found. "
        "Check the printed paths before continuing."
    )

print("\nDrive mounted and experiment folders prepared successfully.")

In [ ]:
def list_candidate_files(
    root: Path,
    suffixes=(".csv", ".parquet", ".npy", ".npz", ".pkl", ".pickle"),
    maximum_files: int = 300,
) -> pd.DataFrame:
    """Return a compact inventory of candidate experiment files."""

    records = []

    if not root.exists():
        return pd.DataFrame(
            columns=["relative_path", "suffix", "size_mb"]
        )

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in suffixes:
            continue

        records.append(
            {
                "relative_path": str(path.relative_to(root)),
                "suffix": path.suffix.lower(),
                "size_mb": round(path.stat().st_size / (1024 ** 2), 3),
            }
        )

        if len(records) >= maximum_files:
            break

    return pd.DataFrame(records)


feature_inventory = list_candidate_files(NAB_FEATURES_DIR)
sequence_inventory = list_candidate_files(NAB_SEQUENCES_DIR)

print("NAB feature-window files")
print("=" * 100)
print(feature_inventory.to_string(index=False))

print("\nNAB sequence-window files")
print("=" * 100)
print(sequence_inventory.to_string(index=False))

print("\nSummary")
print("-" * 60)
print(f"Feature files discovered : {len(feature_inventory):,}")
print(f"Sequence files discovered: {len(sequence_inventory):,}")

In [ ]:
from pathlib import Path

# ---------------------------------------------------------
# Locate the final consolidated NAB experiment files
# ---------------------------------------------------------
final_window_candidates = sorted(
    THESIS_ROOT.rglob("nab_windows_60min_stride30.csv")
)

previous_metrics_candidates = sorted(
    THESIS_ROOT.rglob("metrics_nab_60min_stride30_all_models.csv")
)

previous_prediction_candidates = sorted(
    THESIS_ROOT.rglob("test_predictions_nab_60min_stride30.csv")
)

raw_nab_directory_candidates = sorted(
    path
    for path in THESIS_ROOT.rglob("nab_all_series")
    if path.is_dir()
)

def print_candidates(title, candidates):
    print(f"\n{title}")
    print("=" * 100)

    if not candidates:
        print("No candidates found.")
        return

    for index, path in enumerate(candidates, start=1):
        if path.is_file():
            size_mb = path.stat().st_size / (1024 ** 2)
            print(f"{index:02d}. {path} | {size_mb:.3f} MB")
        else:
            print(f"{index:02d}. {path}")


print_candidates(
    "Final 60-minute / 30-minute NAB window files",
    final_window_candidates
)

print_candidates(
    "Previous five-model comparison source metrics",
    previous_metrics_candidates
)

print_candidates(
    "Previous NAB test prediction files",
    previous_prediction_candidates
)

print_candidates(
    "Processed NAB source-series directories",
    raw_nab_directory_candidates
)

if not final_window_candidates:
    raise FileNotFoundError(
        "The final consolidated NAB window file was not found."
    )

if not raw_nab_directory_candidates:
    raise FileNotFoundError(
        "The processed NAB source-series directory was not found."
    )

# Choose the largest matching consolidated file.
# This avoids selecting an empty or incomplete duplicate folder.
FINAL_NAB_WINDOWS_PATH = max(
    final_window_candidates,
    key=lambda path: path.stat().st_size
)

RAW_NAB_SERIES_DIR = raw_nab_directory_candidates[0]

PREVIOUS_METRICS_PATH = (
    max(previous_metrics_candidates, key=lambda path: path.stat().st_size)
    if previous_metrics_candidates
    else None
)

PREVIOUS_PREDICTIONS_PATH = (
    max(previous_prediction_candidates, key=lambda path: path.stat().st_size)
    if previous_prediction_candidates
    else None
)

print("\nSelected paths")
print("-" * 100)
print(f"Final NAB windows   : {FINAL_NAB_WINDOWS_PATH}")
print(f"Raw NAB series      : {RAW_NAB_SERIES_DIR}")
print(f"Previous metrics    : {PREVIOUS_METRICS_PATH}")
print(f"Previous predictions: {PREVIOUS_PREDICTIONS_PATH}")

In [ ]:
# ---------------------------------------------------------
# Load the final consolidated 60/30 NAB representation
# ---------------------------------------------------------
nab_windows = pd.read_csv(
    FINAL_NAB_WINDOWS_PATH,
    low_memory=False
)

print("Final NAB consolidated table")
print("=" * 100)
print(f"Rows   : {len(nab_windows):,}")
print(f"Columns: {nab_windows.shape[1]:,}")
print(
    f"Memory : "
    f"{nab_windows.memory_usage(deep=True).sum() / (1024 ** 2):.2f} MB"
)

print("\nAll columns")
print("-" * 100)

for index, column in enumerate(nab_windows.columns, start=1):
    print(f"{index:02d}. {column}")


# ---------------------------------------------------------
# Expected final feature representation
# ---------------------------------------------------------
FINAL_FEATURES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var",
]

present_features = [
    feature
    for feature in FINAL_FEATURES
    if feature in nab_windows.columns
]

missing_features = [
    feature
    for feature in FINAL_FEATURES
    if feature not in nab_windows.columns
]

print("\nFinal-feature audit")
print("-" * 100)
print(f"Expected features: {len(FINAL_FEATURES)}")
print(f"Present features : {len(present_features)}")
print(f"Missing features : {missing_features}")


# ---------------------------------------------------------
# Detect important metadata columns
# ---------------------------------------------------------
def find_columns(candidate_names):
    candidate_names = {name.lower() for name in candidate_names}

    return [
        column
        for column in nab_windows.columns
        if column.lower() in candidate_names
    ]


label_candidates = find_columns([
    "label",
    "is_anomaly",
    "anomaly",
    "anomaly_label",
    "window_label",
    "target",
    "y",
])

series_candidates = find_columns([
    "series",
    "series_id",
    "series_name",
    "source",
    "source_file",
    "filename",
    "file",
])

split_candidates = find_columns([
    "split",
    "data_split",
    "dataset_split",
    "partition",
])

time_candidates = find_columns([
    "timestamp",
    "window_start",
    "window_end",
    "start_time",
    "end_time",
])

print("\nDetected metadata")
print("-" * 100)
print(f"Label candidates : {label_candidates}")
print(f"Series candidates: {series_candidates}")
print(f"Split candidates : {split_candidates}")
print(f"Time candidates  : {time_candidates}")


# ---------------------------------------------------------
# Show important distributions where available
# ---------------------------------------------------------
for column in label_candidates:
    print(f"\nValue counts for label column: {column}")
    print(nab_windows[column].value_counts(dropna=False).sort_index())

for column in split_candidates:
    print(f"\nValue counts for split column: {column}")
    print(nab_windows[column].value_counts(dropna=False))

for column in series_candidates:
    print(f"\nUnique values in series column '{column}':")
    print(nab_windows[column].nunique(dropna=True))


print("\nFirst three rows")
print("-" * 100)
display(nab_windows.head(3))

In [ ]:
# ---------------------------------------------------------
# Locate the processed one-minute source series
# ---------------------------------------------------------
raw_series_files = sorted(
    path
    for path in RAW_NAB_SERIES_DIR.rglob("*.csv")
    if "summary" not in path.name.lower()
)

print("Processed NAB source-series audit")
print("=" * 100)
print(f"CSV files found: {len(raw_series_files):,}")

print("\nFirst 15 files")
print("-" * 100)

for path in raw_series_files[:15]:
    print(path.relative_to(RAW_NAB_SERIES_DIR))


# ---------------------------------------------------------
# Inspect representative schemas
# ---------------------------------------------------------
print("\nRepresentative raw-series schemas")
print("=" * 100)

for path in raw_series_files[:5]:
    sample = pd.read_csv(path, nrows=5)

    print(f"\nFile: {path.name}")
    print(f"Columns: {list(sample.columns)}")
    display(sample.head(3))

In [ ]:
# =========================================================
# CELL 7 — EXACT CHRONOLOGICAL 60/20/20 SPLIT
# =========================================================

nab_model_df = nab_windows.copy()

# Stable identifier tied to the consolidated table
nab_model_df["window_id"] = np.arange(len(nab_model_df), dtype=np.int64)

# Parse timestamps
nab_model_df["window_start"] = pd.to_datetime(
    nab_model_df["window_start"],
    utc=True,
    errors="raise"
)

nab_model_df["window_end"] = pd.to_datetime(
    nab_model_df["window_end"],
    utc=True,
    errors="raise"
)

# Sort independently inside each source series
nab_model_df = (
    nab_model_df
    .sort_values(
        ["series", "window_start", "window_end", "window_id"]
    )
    .reset_index(drop=True)
)

nab_model_df["split"] = None

# Chronological 60% / 20% / 20% split inside every series
for series_name, group in nab_model_df.groupby(
    "series",
    sort=False
):
    group_indices = group.index.to_numpy()
    number_of_windows = len(group_indices)

    train_end = int(np.floor(0.60 * number_of_windows))
    validation_end = int(np.floor(0.80 * number_of_windows))

    train_indices = group_indices[:train_end]
    validation_indices = group_indices[
        train_end:validation_end
    ]
    test_indices = group_indices[validation_end:]

    nab_model_df.loc[train_indices, "split"] = "train"
    nab_model_df.loc[validation_indices, "split"] = "val"
    nab_model_df.loc[test_indices, "split"] = "test"


# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------
assert nab_model_df["split"].notna().all()
assert nab_model_df["window_id"].nunique() == len(nab_model_df)

split_summary = (
    nab_model_df
    .groupby("split", sort=False)
    .agg(
        windows=("window_id", "size"),
        normal_windows=("label", lambda x: int((x == 0).sum())),
        anomalous_windows=("label", lambda x: int((x == 1).sum())),
        anomaly_ratio=("label", "mean"),
        unique_series=("series", "nunique"),
    )
    .reset_index()
)

split_summary["anomaly_ratio"] *= 100

print("NAB chronological split summary")
print("=" * 90)
display(split_summary)

print("\nExpected historical split counts")
print("-" * 90)
print("Train      : 62,096")
print("Validation : 20,705")
print("Test       : 20,725")

actual_counts = (
    nab_model_df["split"]
    .value_counts()
    .to_dict()
)

expected_counts = {
    "train": 62096,
    "val": 20705,
    "test": 20725,
}

print("\nActual counts")
print(actual_counts)

if actual_counts != expected_counts:
    raise ValueError(
        "The recreated split does not match the previous NAB "
        "experiment. Do not continue until the split rule is resolved."
    )

print("\nSplit recreation passed.")

In [ ]:
# =========================================================
# CELL 8 — PER-SERIES TRAINING-ONLY NORMALIZATION
# =========================================================

FEATURE_COLUMNS = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var",
]

assert len(FEATURE_COLUMNS) == 23
assert all(
    column in nab_model_df.columns
    for column in FEATURE_COLUMNS
)

# Replace infinities with missing values before checking
nab_model_df[FEATURE_COLUMNS] = (
    nab_model_df[FEATURE_COLUMNS]
    .replace([np.inf, -np.inf], np.nan)
)

missing_counts = (
    nab_model_df[FEATURE_COLUMNS]
    .isna()
    .sum()
)

if missing_counts.sum() > 0:
    print("Missing feature values detected:")
    print(missing_counts[missing_counts > 0])

    raise ValueError(
        "Missing feature values exist. Do not silently impute "
        "before inspecting them."
    )


# ---------------------------------------------------------
# Calculate normalization statistics from TRAIN only
# ---------------------------------------------------------
normalization_records = []
normalized_parts = []

for series_name, series_df in nab_model_df.groupby(
    "series",
    sort=False
):
    series_df = series_df.copy()

    series_train = series_df[
        series_df["split"] == "train"
    ]

    if series_train.empty:
        raise ValueError(
            f"No training windows found for series: {series_name}"
        )

    train_means = series_train[FEATURE_COLUMNS].mean()
    train_stds = series_train[FEATURE_COLUMNS].std(ddof=0)

    # Constant features cannot be divided by zero.
    safe_stds = train_stds.mask(
        train_stds.abs() < 1e-12,
        1.0
    )

    normalized_values = (
        series_df[FEATURE_COLUMNS] - train_means
    ) / safe_stds

    normalized_values.columns = [
        f"z_{column}"
        for column in FEATURE_COLUMNS
    ]

    series_df = pd.concat(
        [
            series_df.reset_index(drop=True),
            normalized_values.reset_index(drop=True),
        ],
        axis=1
    )

    normalized_parts.append(series_df)

    for feature in FEATURE_COLUMNS:
        normalization_records.append(
            {
                "series": series_name,
                "feature": feature,
                "train_mean": float(train_means[feature]),
                "train_std_original": float(train_stds[feature]),
                "train_std_used": float(safe_stds[feature]),
                "constant_feature": bool(
                    abs(train_stds[feature]) < 1e-12
                ),
            }
        )


nab_normalized_df = pd.concat(
    normalized_parts,
    ignore_index=True
)

Z_FEATURE_COLUMNS = [
    f"z_{column}"
    for column in FEATURE_COLUMNS
]

normalization_stats_df = pd.DataFrame(
    normalization_records
)

# ---------------------------------------------------------
# Final matrices
# Labels are kept separately and never passed into fit()
# ---------------------------------------------------------
train_df = nab_normalized_df[
    nab_normalized_df["split"] == "train"
].copy()

validation_df = nab_normalized_df[
    nab_normalized_df["split"] == "val"
].copy()

test_df = nab_normalized_df[
    nab_normalized_df["split"] == "test"
].copy()

X_train = train_df[Z_FEATURE_COLUMNS].to_numpy(
    dtype=np.float32
)

X_validation = validation_df[
    Z_FEATURE_COLUMNS
].to_numpy(dtype=np.float32)

X_test = test_df[Z_FEATURE_COLUMNS].to_numpy(
    dtype=np.float32
)

# Stored only for evaluation after training and threshold locking
y_train_hidden = train_df["label"].to_numpy(
    dtype=np.int8
)

y_validation_hidden = validation_df[
    "label"
].to_numpy(dtype=np.int8)

y_test_hidden = test_df["label"].to_numpy(
    dtype=np.int8
)

print("Normalized matrix shapes")
print("=" * 70)
print(f"X_train      : {X_train.shape}")
print(f"X_validation : {X_validation.shape}")
print(f"X_test       : {X_test.shape}")

print("\nFinite-value audit")
print("-" * 70)
print(f"Train finite      : {np.isfinite(X_train).all()}")
print(f"Validation finite : {np.isfinite(X_validation).all()}")
print(f"Test finite       : {np.isfinite(X_test).all()}")

print("\nLabels remain separate from model inputs.")

In [ ]:
# =========================================================
# CELL 9 — LABEL-BLIND BALANCED OCSVM REFERENCE POOL
# =========================================================

OCSVM_MAX_WINDOWS_PER_SERIES = 300

reference_parts = []

for series_name, series_train_df in train_df.groupby(
    "series",
    sort=False
):
    number_available = len(series_train_df)

    number_selected = min(
        OCSVM_MAX_WINDOWS_PER_SERIES,
        number_available
    )

    sampled_series_df = series_train_df.sample(
        n=number_selected,
        replace=False,
        random_state=RANDOM_STATE
    )

    reference_parts.append(sampled_series_df)


ocsvm_reference_df = pd.concat(
    reference_parts,
    ignore_index=True
)

X_ocsvm_reference = (
    ocsvm_reference_df[Z_FEATURE_COLUMNS]
    .to_numpy(dtype=np.float32)
)

reference_summary = (
    ocsvm_reference_df
    .groupby("series")
    .size()
    .describe()
)

print("One-Class SVM reference pool")
print("=" * 80)
print(f"Reference windows : {len(ocsvm_reference_df):,}")
print(
    f"Reference series  : "
    f"{ocsvm_reference_df['series'].nunique():,}"
)
print(f"Feature dimension : {X_ocsvm_reference.shape[1]}")

print("\nPer-series reference-window summary")
print("-" * 80)
print(reference_summary)

print(
    "\nImportant: labels were not used to construct "
    "the reference pool."
)

# Save the exact reference-window identities
ocsvm_reference_df[
    [
        "window_id",
        "series",
        "domain",
        "window_start",
        "window_end",
        "split",
    ]
].to_csv(
    COMMON_OUTPUT_DIR /
    "ocsvm_training_reference_windows.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 10 — ONE-CLASS SVM TRAINING AND EVALUATION
# =========================================================

import joblib

OCSVM_CONFIG = {
    "model": "One-Class SVM",
    "kernel": "rbf",
    "gamma": "scale",
    "nu": 0.05,
    "cache_size_mb": 2048,
    "threshold_percentile": 95.0,
    "random_state": RANDOM_STATE,
    "window_minutes": 60,
    "stride_minutes": 30,
    "feature_count": len(FEATURE_COLUMNS),
    "normalization": (
        "Per-series z-score using training statistics only"
    ),
    "training_labels_used": False,
    "threshold_labels_used": False,
    "reference_sampling": (
        f"Maximum {OCSVM_MAX_WINDOWS_PER_SERIES} "
        "training windows per series"
    ),
}

with open(
    OCSVM_OUTPUT_DIR / "config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        OCSVM_CONFIG,
        file,
        indent=2
    )


ocsvm_model = OneClassSVM(
    kernel=OCSVM_CONFIG["kernel"],
    gamma=OCSVM_CONFIG["gamma"],
    nu=OCSVM_CONFIG["nu"],
    cache_size=OCSVM_CONFIG["cache_size_mb"],
)

print("Training One-Class SVM...")
training_start = time.perf_counter()

ocsvm_model.fit(X_ocsvm_reference)

training_seconds = (
    time.perf_counter() - training_start
)

print(
    f"Training completed in "
    f"{training_seconds:.2f} seconds."
)


# ---------------------------------------------------------
# Generate continuous anomaly scores
# Higher score = more anomalous
# ---------------------------------------------------------
print("\nScoring validation and test windows...")

validation_scoring_start = time.perf_counter()

ocsvm_validation_scores = -ocsvm_model.decision_function(
    X_validation
).reshape(-1)

validation_scoring_seconds = (
    time.perf_counter() - validation_scoring_start
)

test_scoring_start = time.perf_counter()

ocsvm_test_scores = -ocsvm_model.decision_function(
    X_test
).reshape(-1)

test_scoring_seconds = (
    time.perf_counter() - test_scoring_start
)


# ---------------------------------------------------------
# Threshold uses scores only — no labels
# ---------------------------------------------------------
ocsvm_threshold = float(
    np.percentile(
        ocsvm_validation_scores,
        OCSVM_CONFIG["threshold_percentile"]
    )
)

ocsvm_validation_predictions = (
    ocsvm_validation_scores > ocsvm_threshold
).astype(np.int8)

ocsvm_test_predictions = (
    ocsvm_test_scores > ocsvm_threshold
).astype(np.int8)


# ---------------------------------------------------------
# Reveal labels only now for evaluation
# ---------------------------------------------------------
def calculate_anomaly_metrics(
    labels,
    predictions,
    scores
):
    tn, fp, fn, tp = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1]
    ).ravel()

    precision = precision_score(
        labels,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        labels,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        labels,
        predictions,
        zero_division=0
    )

    false_positive_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else np.nan
    )

    roc_auc = roc_auc_score(
        labels,
        scores
    )

    pr_auc = average_precision_score(
        labels,
        scores
    )

    return {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "false_positive_rate": float(false_positive_rate),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "flagged_windows": int(predictions.sum()),
        "flagged_ratio_percent": float(
            predictions.mean() * 100
        ),
    }


ocsvm_validation_metrics = calculate_anomaly_metrics(
    y_validation_hidden,
    ocsvm_validation_predictions,
    ocsvm_validation_scores
)

ocsvm_test_metrics = calculate_anomaly_metrics(
    y_test_hidden,
    ocsvm_test_predictions,
    ocsvm_test_scores
)

ocsvm_metrics_df = pd.DataFrame(
    [
        {
            "dataset": "NAB",
            "model": "One-Class SVM",
            "split": "validation",
            "threshold": ocsvm_threshold,
            **ocsvm_validation_metrics,
        },
        {
            "dataset": "NAB",
            "model": "One-Class SVM",
            "split": "test",
            "threshold": ocsvm_threshold,
            **ocsvm_test_metrics,
        },
    ]
)

ocsvm_metrics_df[
    "training_seconds"
] = training_seconds

ocsvm_metrics_df[
    "validation_scoring_seconds"
] = validation_scoring_seconds

ocsvm_metrics_df[
    "test_scoring_seconds"
] = test_scoring_seconds

print("\nOne-Class SVM results")
print("=" * 100)
display(ocsvm_metrics_df)


# ---------------------------------------------------------
# Save metrics and predictions
# ---------------------------------------------------------
ocsvm_metrics_df.to_csv(
    OCSVM_OUTPUT_DIR / "metrics.csv",
    index=False
)

validation_output = validation_df[
    [
        "window_id",
        "series",
        "domain",
        "window_start",
        "window_end",
        "label",
    ]
].copy()

validation_output["anomaly_score"] = (
    ocsvm_validation_scores
)

validation_output["threshold"] = (
    ocsvm_threshold
)

validation_output["prediction"] = (
    ocsvm_validation_predictions
)

validation_output.to_csv(
    OCSVM_OUTPUT_DIR / "validation_scores.csv",
    index=False
)


test_output = test_df[
    [
        "window_id",
        "series",
        "domain",
        "window_start",
        "window_end",
        "label",
    ]
].copy()

test_output["anomaly_score"] = (
    ocsvm_test_scores
)

test_output["threshold"] = (
    ocsvm_threshold
)

test_output["prediction"] = (
    ocsvm_test_predictions
)

test_output.to_csv(
    OCSVM_OUTPUT_DIR / "test_scores.csv",
    index=False
)

normalization_stats_df.to_csv(
    COMMON_OUTPUT_DIR / "normalization_statistics.csv",
    index=False
)

nab_model_df[
    [
        "window_id",
        "series",
        "domain",
        "window_start",
        "window_end",
        "label",
        "split",
    ]
].to_csv(
    COMMON_OUTPUT_DIR / "nab_split_manifest.csv",
    index=False
)

joblib.dump(
    ocsvm_model,
    OCSVM_OUTPUT_DIR / "one_class_svm.joblib"
)

print("\nSaved One-Class SVM artifacts to:")
print(OCSVM_OUTPUT_DIR)

In [ ]:
# =========================================================
# CELL 11 — RECONSTRUCT 60-STEP NAB SEQUENCES
# =========================================================

SEQUENCE_LENGTH = 60

raw_sequence_matrix = np.empty(
    (len(nab_model_df), SEQUENCE_LENGTH),
    dtype=np.float32
)

source_point_counts = np.zeros(
    len(nab_model_df),
    dtype=np.int32
)

sequence_generation_records = []

print("Constructing fixed-length temporal sequences...")
print("=" * 90)

for series_number, (
    series_name,
    window_group
) in enumerate(
    nab_model_df.groupby("series", sort=False),
    start=1
):
    source_path = RAW_NAB_SERIES_DIR / series_name

    if not source_path.exists():
        raise FileNotFoundError(
            f"Source series not found: {source_path}"
        )

    source_df = pd.read_csv(
        source_path,
        low_memory=False
    )

    required_columns = {"timestamp", "value"}

    if not required_columns.issubset(source_df.columns):
        raise ValueError(
            f"Missing required columns in {source_path}: "
            f"{required_columns - set(source_df.columns)}"
        )

    source_df["timestamp"] = pd.to_datetime(
        source_df["timestamp"],
        utc=True,
        errors="coerce"
    )

    source_df["value"] = pd.to_numeric(
        source_df["value"],
        errors="coerce"
    )

    source_df = (
        source_df
        .dropna(subset=["timestamp", "value"])
        .sort_values("timestamp")
        .drop_duplicates(
            subset=["timestamp"],
            keep="last"
        )
        .reset_index(drop=True)
    )

    source_timestamps_ns = (
        source_df["timestamp"]
        .astype("int64")
        .to_numpy()
    )

    source_values = (
        source_df["value"]
        .to_numpy(dtype=np.float64)
    )

    generated_for_series = 0

    for row_index, window_row in window_group.iterrows():
        window_start_ns = int(
            window_row["window_start"].value
        )

        window_end_ns = int(
            window_row["window_end"].value
        )

        left_index = np.searchsorted(
            source_timestamps_ns,
            window_start_ns,
            side="left"
        )

        right_index = np.searchsorted(
            source_timestamps_ns,
            window_end_ns,
            side="left"
        )

        window_timestamps_ns = source_timestamps_ns[
            left_index:right_index
        ]

        window_values = source_values[
            left_index:right_index
        ]

        number_of_source_points = len(window_values)

        source_point_counts[row_index] = (
            number_of_source_points
        )

        if number_of_source_points == 0:
            raise ValueError(
                "No source observations found for window:\n"
                f"Series: {series_name}\n"
                f"Start : {window_row['window_start']}\n"
                f"End   : {window_row['window_end']}"
            )

        window_duration_seconds = (
            window_end_ns - window_start_ns
        ) / 1_000_000_000

        target_seconds = np.linspace(
            0.0,
            window_duration_seconds,
            num=SEQUENCE_LENGTH,
            endpoint=False,
            dtype=np.float64
        )

        if number_of_source_points == 1:
            interpolated_values = np.full(
                SEQUENCE_LENGTH,
                window_values[0],
                dtype=np.float64
            )
        else:
            source_seconds = (
                window_timestamps_ns - window_start_ns
            ) / 1_000_000_000

            interpolated_values = np.interp(
                target_seconds,
                source_seconds,
                window_values,
                left=window_values[0],
                right=window_values[-1]
            )

        raw_sequence_matrix[row_index] = (
            interpolated_values.astype(np.float32)
        )

        generated_for_series += 1

    sequence_generation_records.append(
        {
            "series": series_name,
            "source_rows": len(source_df),
            "generated_windows": generated_for_series,
            "native_median_interval_minutes": (
                source_df["timestamp"]
                .diff()
                .dt.total_seconds()
                .median()
                / 60.0
            ),
        }
    )

    print(
        f"[{series_number:02d}/58] "
        f"{series_name}: "
        f"{generated_for_series:,} windows"
    )


sequence_generation_summary = pd.DataFrame(
    sequence_generation_records
)

print("\nSequence matrix")
print("-" * 90)
print(f"Shape       : {raw_sequence_matrix.shape}")
print(f"Finite      : {np.isfinite(raw_sequence_matrix).all()}")
print(f"Minimum     : {raw_sequence_matrix.min():.6f}")
print(f"Maximum     : {raw_sequence_matrix.max():.6f}")


# ---------------------------------------------------------
# Verify that extracted source-point counts match the
# n_points values used during feature generation
# ---------------------------------------------------------
expected_point_counts = (
    nab_model_df["n_points"]
    .to_numpy(dtype=np.int32)
)

point_count_match = (
    source_point_counts == expected_point_counts
)

print("\nWindow-source alignment audit")
print("-" * 90)
print(
    f"Exact n_points matches: "
    f"{point_count_match.sum():,} / "
    f"{len(point_count_match):,}"
)

print(
    f"Mismatched windows: "
    f"{(~point_count_match).sum():,}"
)

if not point_count_match.all():
    mismatch_indices = np.where(
        ~point_count_match
    )[0][:20]

    mismatch_report = nab_model_df.loc[
        mismatch_indices,
        [
            "series",
            "window_start",
            "window_end",
            "n_points",
        ]
    ].copy()

    mismatch_report[
        "reconstructed_source_points"
    ] = source_point_counts[mismatch_indices]

    display(mismatch_report)

    raise ValueError(
        "Some reconstructed windows do not match the "
        "feature-generation source-point counts."
    )

print("\nAll window boundaries matched successfully.")


# ---------------------------------------------------------
# Save the reconstruction summary
# ---------------------------------------------------------
sequence_generation_summary.to_csv(
    COMMON_OUTPUT_DIR /
    "lstm_sequence_generation_summary.csv",
    index=False
)

np.save(
    COMMON_OUTPUT_DIR /
    "nab_raw_sequences_60_steps.npy",
    raw_sequence_matrix
)

In [ ]:
# =========================================================
# CELL 12 — PER-SERIES SEQUENCE NORMALIZATION
# =========================================================

normalized_sequence_matrix = np.empty_like(
    raw_sequence_matrix,
    dtype=np.float32
)

sequence_normalization_records = []

for series_name, series_group in nab_model_df.groupby(
    "series",
    sort=False
):
    all_indices = series_group.index.to_numpy()

    training_indices = series_group.loc[
        series_group["split"] == "train"
    ].index.to_numpy()

    if len(training_indices) == 0:
        raise ValueError(
            f"No training sequences for {series_name}"
        )

    training_values = raw_sequence_matrix[
        training_indices
    ]

    training_mean = float(
        training_values.mean()
    )

    training_std_original = float(
        training_values.std(ddof=0)
    )

    training_std_used = (
        training_std_original
        if training_std_original >= 1e-8
        else 1.0
    )

    normalized_sequence_matrix[
        all_indices
    ] = (
        raw_sequence_matrix[all_indices]
        - training_mean
    ) / training_std_used

    sequence_normalization_records.append(
        {
            "series": series_name,
            "training_mean": training_mean,
            "training_std_original": (
                training_std_original
            ),
            "training_std_used": training_std_used,
            "constant_series": bool(
                training_std_original < 1e-8
            ),
        }
    )


sequence_normalization_df = pd.DataFrame(
    sequence_normalization_records
)

train_indices = nab_model_df.index[
    nab_model_df["split"] == "train"
].to_numpy()

validation_indices = nab_model_df.index[
    nab_model_df["split"] == "val"
].to_numpy()

test_indices = nab_model_df.index[
    nab_model_df["split"] == "test"
].to_numpy()


X_lstm_train = normalized_sequence_matrix[
    train_indices
][..., np.newaxis]

X_lstm_validation = normalized_sequence_matrix[
    validation_indices
][..., np.newaxis]

X_lstm_test = normalized_sequence_matrix[
    test_indices
][..., np.newaxis]


y_lstm_validation_hidden = nab_model_df.loc[
    validation_indices,
    "label"
].to_numpy(dtype=np.int8)

y_lstm_test_hidden = nab_model_df.loc[
    test_indices,
    "label"
].to_numpy(dtype=np.int8)


print("LSTM sequence tensors")
print("=" * 80)
print(f"Training   : {X_lstm_train.shape}")
print(f"Validation : {X_lstm_validation.shape}")
print(f"Test       : {X_lstm_test.shape}")

print("\nFinite-value audit")
print("-" * 80)
print(
    f"Training finite   : "
    f"{np.isfinite(X_lstm_train).all()}"
)
print(
    f"Validation finite : "
    f"{np.isfinite(X_lstm_validation).all()}"
)
print(
    f"Test finite       : "
    f"{np.isfinite(X_lstm_test).all()}"
)

sequence_normalization_df.to_csv(
    COMMON_OUTPUT_DIR /
    "lstm_sequence_normalization_statistics.csv",
    index=False
)

np.savez_compressed(
    COMMON_OUTPUT_DIR /
    "nab_lstm_sequence_tensors.npz",
    X_train=X_lstm_train,
    X_validation=X_lstm_validation,
    X_test=X_lstm_test,
    train_window_ids=nab_model_df.loc[
        train_indices,
        "window_id"
    ].to_numpy(),
    validation_window_ids=nab_model_df.loc[
        validation_indices,
        "window_id"
    ].to_numpy(),
    test_window_ids=nab_model_df.loc[
        test_indices,
        "window_id"
    ].to_numpy(),
)

In [ ]:
# =========================================================
# CELL 13 — LSTM AUTOENCODER TRAINING
# =========================================================

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    RepeatVector,
    TimeDistributed,
    Dense,
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
)

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(RANDOM_STATE)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


LSTM_CONFIG = {
    "model": "LSTM Autoencoder",
    "sequence_length": SEQUENCE_LENGTH,
    "input_features": 1,
    "encoder_units_1": 32,
    "latent_units": 16,
    "decoder_units_1": 16,
    "decoder_units_2": 32,
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "loss": "mean_squared_error",
    "batch_size": 256,
    "maximum_epochs": 100,
    "early_stopping_patience": 10,
    "threshold_percentile": 95.0,
    "training_labels_used": False,
    "threshold_labels_used": False,
    "normalization": (
        "Per-series z-score using training sequence "
        "values only"
    ),
    "sequence_standardization": (
        "Each original 60-minute window interpolated "
        "to 60 ordered time steps"
    ),
    "random_state": RANDOM_STATE,
}


inputs = Input(
    shape=(SEQUENCE_LENGTH, 1),
    name="sequence_input"
)

encoded_sequence = LSTM(
    32,
    return_sequences=True,
    name="encoder_lstm_1"
)(inputs)

latent_vector = LSTM(
    16,
    return_sequences=False,
    name="latent_lstm"
)(encoded_sequence)

repeated_vector = RepeatVector(
    SEQUENCE_LENGTH,
    name="repeat_latent_vector"
)(latent_vector)

decoded_sequence = LSTM(
    16,
    return_sequences=True,
    name="decoder_lstm_1"
)(repeated_vector)

decoded_sequence = LSTM(
    32,
    return_sequences=True,
    name="decoder_lstm_2"
)(decoded_sequence)

outputs = TimeDistributed(
    Dense(1),
    name="reconstructed_sequence"
)(decoded_sequence)


lstm_autoencoder = Model(
    inputs=inputs,
    outputs=outputs,
    name="NAB_LSTM_Autoencoder"
)

lstm_autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LSTM_CONFIG[
            "learning_rate"
        ]
    ),
    loss="mean_squared_error"
)

lstm_autoencoder.summary()


early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=LSTM_CONFIG[
        "early_stopping_patience"
    ],
    min_delta=1e-5,
    restore_best_weights=True,
    verbose=1
)

reduce_learning_rate = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)


print("\nTraining LSTM Autoencoder...")
training_start = time.perf_counter()

lstm_history = lstm_autoencoder.fit(
    X_lstm_train,
    X_lstm_train,
    validation_data=(
        X_lstm_validation,
        X_lstm_validation
    ),
    epochs=LSTM_CONFIG["maximum_epochs"],
    batch_size=LSTM_CONFIG["batch_size"],
    shuffle=True,
    callbacks=[
        early_stopping,
        reduce_learning_rate
    ],
    verbose=1
)

lstm_training_seconds = (
    time.perf_counter() - training_start
)

print(
    f"\nTraining completed in "
    f"{lstm_training_seconds:.2f} seconds."
)

history_df = pd.DataFrame(
    lstm_history.history
)

history_df.index.name = "epoch"
history_df.to_csv(
    LSTM_OUTPUT_DIR / "training_history.csv"
)

with open(
    LSTM_OUTPUT_DIR / "config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        LSTM_CONFIG,
        file,
        indent=2
    )

lstm_autoencoder.save(
    LSTM_OUTPUT_DIR /
    "lstm_autoencoder.keras"
)

In [ ]:
# =========================================================
# CELL 14 — LSTM AUTOENCODER EVALUATION
# =========================================================

print("Reconstructing validation sequences...")

validation_scoring_start = time.perf_counter()

validation_reconstructions = (
    lstm_autoencoder.predict(
        X_lstm_validation,
        batch_size=512,
        verbose=1
    )
)

lstm_validation_scoring_seconds = (
    time.perf_counter()
    - validation_scoring_start
)


print("\nReconstructing test sequences...")

test_scoring_start = time.perf_counter()

test_reconstructions = (
    lstm_autoencoder.predict(
        X_lstm_test,
        batch_size=512,
        verbose=1
    )
)

lstm_test_scoring_seconds = (
    time.perf_counter()
    - test_scoring_start
)


# Mean reconstruction error per window
lstm_validation_scores = np.mean(
    np.square(
        X_lstm_validation
        - validation_reconstructions
    ),
    axis=(1, 2)
)

lstm_test_scores = np.mean(
    np.square(
        X_lstm_test
        - test_reconstructions
    ),
    axis=(1, 2)
)


# Threshold selected without labels
lstm_threshold = float(
    np.percentile(
        lstm_validation_scores,
        LSTM_CONFIG["threshold_percentile"]
    )
)

lstm_validation_predictions = (
    lstm_validation_scores
    > lstm_threshold
).astype(np.int8)

lstm_test_predictions = (
    lstm_test_scores
    > lstm_threshold
).astype(np.int8)


# Reveal labels only for final evaluation
lstm_validation_metrics = (
    calculate_anomaly_metrics(
        y_lstm_validation_hidden,
        lstm_validation_predictions,
        lstm_validation_scores
    )
)

lstm_test_metrics = (
    calculate_anomaly_metrics(
        y_lstm_test_hidden,
        lstm_test_predictions,
        lstm_test_scores
    )
)


lstm_metrics_df = pd.DataFrame(
    [
        {
            "dataset": "NAB",
            "model": "LSTM Autoencoder",
            "split": "validation",
            "threshold": lstm_threshold,
            **lstm_validation_metrics,
        },
        {
            "dataset": "NAB",
            "model": "LSTM Autoencoder",
            "split": "test",
            "threshold": lstm_threshold,
            **lstm_test_metrics,
        },
    ]
)

lstm_metrics_df[
    "training_seconds"
] = lstm_training_seconds

lstm_metrics_df[
    "validation_scoring_seconds"
] = lstm_validation_scoring_seconds

lstm_metrics_df[
    "test_scoring_seconds"
] = lstm_test_scoring_seconds


print("\nLSTM Autoencoder results")
print("=" * 100)
display(lstm_metrics_df)


# ---------------------------------------------------------
# Save validation and test outputs
# ---------------------------------------------------------
lstm_validation_output = nab_model_df.loc[
    validation_indices,
    [
        "window_id",
        "series",
        "domain",
        "window_start",
        "window_end",
        "label",
    ]
].copy()

lstm_validation_output[
    "anomaly_score"
] = lstm_validation_scores

lstm_validation_output[
    "threshold"
] = lstm_threshold

lstm_validation_output[
    "prediction"
] = lstm_validation_predictions

lstm_validation_output.to_csv(
    LSTM_OUTPUT_DIR /
    "validation_scores.csv",
    index=False
)


lstm_test_output = nab_model_df.loc[
    test_indices,
    [
        "window_id",
        "series",
        "domain",
        "window_start",
        "window_end",
        "label",
    ]
].copy()

lstm_test_output[
    "anomaly_score"
] = lstm_test_scores

lstm_test_output[
    "threshold"
] = lstm_threshold

lstm_test_output[
    "prediction"
] = lstm_test_predictions

lstm_test_output.to_csv(
    LSTM_OUTPUT_DIR /
    "test_scores.csv",
    index=False
)


lstm_metrics_df.to_csv(
    LSTM_OUTPUT_DIR / "metrics.csv",
    index=False
)

print("\nSaved LSTM Autoencoder artifacts to:")
print(LSTM_OUTPUT_DIR)

In [ ]:
# =========================================================
# CELL 15 — CONTROLLED LABEL-BLIND FEATURE-MODEL COMPARISON
# =========================================================

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
import joblib

CONTROLLED_MODELS_DIR = (
    NEW_EXPERIMENT_ROOT / "Controlled_Five_Model_Comparison"
)

CONTROLLED_MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

THRESHOLD_PERCENTILE = 95.0


def run_feature_anomaly_model(
    model_name,
    model,
    train_matrix,
    validation_matrix,
    test_matrix,
):
    """Fit an unsupervised model and evaluate it label-blind."""

    print(f"\n{'=' * 90}")
    print(f"Running: {model_name}")
    print(f"{'=' * 90}")

    start_time = time.perf_counter()

    model.fit(train_matrix)

    training_seconds = time.perf_counter() - start_time

    validation_start = time.perf_counter()

    # All three sklearn models assign larger scores to
    # more normal observations. Reverse the direction.
    validation_scores = -model.score_samples(
        validation_matrix
    ).reshape(-1)

    validation_seconds = (
        time.perf_counter() - validation_start
    )

    test_start = time.perf_counter()

    test_scores = -model.score_samples(
        test_matrix
    ).reshape(-1)

    test_seconds = time.perf_counter() - test_start

    # Labels are not used here.
    threshold = float(
        np.percentile(
            validation_scores,
            THRESHOLD_PERCENTILE
        )
    )

    validation_predictions = (
        validation_scores > threshold
    ).astype(np.int8)

    test_predictions = (
        test_scores > threshold
    ).astype(np.int8)

    # Labels are revealed only for evaluation.
    validation_metrics = calculate_anomaly_metrics(
        y_validation_hidden,
        validation_predictions,
        validation_scores
    )

    test_metrics = calculate_anomaly_metrics(
        y_test_hidden,
        test_predictions,
        test_scores
    )

    metrics = pd.DataFrame(
        [
            {
                "dataset": "NAB",
                "model": model_name,
                "split": "validation",
                "threshold_percentile": (
                    THRESHOLD_PERCENTILE
                ),
                "threshold": threshold,
                **validation_metrics,
                "training_seconds": training_seconds,
                "scoring_seconds": validation_seconds,
            },
            {
                "dataset": "NAB",
                "model": model_name,
                "split": "test",
                "threshold_percentile": (
                    THRESHOLD_PERCENTILE
                ),
                "threshold": threshold,
                **test_metrics,
                "training_seconds": training_seconds,
                "scoring_seconds": test_seconds,
            },
        ]
    )

    safe_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "")
    )

    metrics.to_csv(
        CONTROLLED_MODELS_DIR /
        f"{safe_name}_metrics.csv",
        index=False
    )

    test_result = test_df[
        [
            "window_id",
            "series",
            "domain",
            "window_start",
            "window_end",
            "label",
        ]
    ].copy()

    test_result["anomaly_score"] = test_scores
    test_result["threshold"] = threshold
    test_result["prediction"] = test_predictions

    test_result.to_csv(
        CONTROLLED_MODELS_DIR /
        f"{safe_name}_test_scores.csv",
        index=False
    )

    joblib.dump(
        model,
        CONTROLLED_MODELS_DIR /
        f"{safe_name}.joblib"
    )

    display(metrics)

    return {
        "model": model,
        "metrics": metrics,
        "validation_scores": validation_scores,
        "test_scores": test_scores,
        "test_predictions": test_predictions,
    }


controlled_results = {}


# ---------------------------------------------------------
# 1. Full-training One-Class SVM
# ---------------------------------------------------------
controlled_results["One-Class SVM"] = (
    run_feature_anomaly_model(
        model_name="One-Class SVM",
        model=OneClassSVM(
            kernel="rbf",
            gamma="scale",
            nu=0.05,
            cache_size=2048
        ),
        train_matrix=X_train,
        validation_matrix=X_validation,
        test_matrix=X_test,
    )
)


# ---------------------------------------------------------
# 2. Isolation Forest
# ---------------------------------------------------------
controlled_results["Isolation Forest"] = (
    run_feature_anomaly_model(
        model_name="Isolation Forest",
        model=IsolationForest(
            n_estimators=300,
            contamination="auto",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        train_matrix=X_train,
        validation_matrix=X_validation,
        test_matrix=X_test,
    )
)


# ---------------------------------------------------------
# 3. Local Outlier Factor
# ---------------------------------------------------------
controlled_results["Local Outlier Factor"] = (
    run_feature_anomaly_model(
        model_name="Local Outlier Factor",
        model=LocalOutlierFactor(
            n_neighbors=35,
            novelty=True,
            contamination="auto",
            n_jobs=-1
        ),
        train_matrix=X_train,
        validation_matrix=X_validation,
        test_matrix=X_test,
    )
)

In [ ]:
# =========================================================
# CELL 16 — LABEL-BLIND DENSE AUTOENCODER
# =========================================================

from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(RANDOM_STATE)

dense_input = Input(
    shape=(23,),
    name="feature_input"
)

x = Dense(
    16,
    activation="relu",
    name="encoder_16"
)(dense_input)

x = Dense(
    8,
    activation="relu",
    name="encoder_8"
)(x)

latent = Dense(
    4,
    activation="relu",
    name="latent_4"
)(x)

x = Dense(
    8,
    activation="relu",
    name="decoder_8"
)(latent)

x = Dense(
    16,
    activation="relu",
    name="decoder_16"
)(x)

dense_output = Dense(
    23,
    activation="linear",
    name="reconstruction"
)(x)

dense_autoencoder = Model(
    inputs=dense_input,
    outputs=dense_output,
    name="NAB_Dense_Autoencoder_Controlled"
)

dense_autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mean_squared_error"
)

dense_autoencoder.summary()

dense_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    min_delta=1e-5,
    restore_best_weights=True,
    verbose=1
)

print("\nTraining controlled dense Autoencoder...")

dense_training_start = time.perf_counter()

dense_history = dense_autoencoder.fit(
    X_train,
    X_train,
    validation_data=(
        X_validation,
        X_validation
    ),
    epochs=150,
    batch_size=256,
    shuffle=True,
    callbacks=[dense_early_stopping],
    verbose=1
)

dense_training_seconds = (
    time.perf_counter() - dense_training_start
)

print(
    f"\nTraining completed in "
    f"{dense_training_seconds:.2f} seconds."
)


# ---------------------------------------------------------
# Reconstruction scores
# ---------------------------------------------------------
dense_validation_reconstruction = (
    dense_autoencoder.predict(
        X_validation,
        batch_size=512,
        verbose=1
    )
)

dense_test_reconstruction = (
    dense_autoencoder.predict(
        X_test,
        batch_size=512,
        verbose=1
    )
)

dense_validation_scores = np.mean(
    np.square(
        X_validation -
        dense_validation_reconstruction
    ),
    axis=1
)

dense_test_scores = np.mean(
    np.square(
        X_test -
        dense_test_reconstruction
    ),
    axis=1
)

dense_threshold = float(
    np.percentile(
        dense_validation_scores,
        THRESHOLD_PERCENTILE
    )
)

dense_validation_predictions = (
    dense_validation_scores > dense_threshold
).astype(np.int8)

dense_test_predictions = (
    dense_test_scores > dense_threshold
).astype(np.int8)


dense_validation_metrics = calculate_anomaly_metrics(
    y_validation_hidden,
    dense_validation_predictions,
    dense_validation_scores
)

dense_test_metrics = calculate_anomaly_metrics(
    y_test_hidden,
    dense_test_predictions,
    dense_test_scores
)


dense_metrics_df = pd.DataFrame(
    [
        {
            "dataset": "NAB",
            "model": "Dense Autoencoder",
            "split": "validation",
            "threshold_percentile": (
                THRESHOLD_PERCENTILE
            ),
            "threshold": dense_threshold,
            **dense_validation_metrics,
            "training_seconds": (
                dense_training_seconds
            ),
        },
        {
            "dataset": "NAB",
            "model": "Dense Autoencoder",
            "split": "test",
            "threshold_percentile": (
                THRESHOLD_PERCENTILE
            ),
            "threshold": dense_threshold,
            **dense_test_metrics,
            "training_seconds": (
                dense_training_seconds
            ),
        },
    ]
)

display(dense_metrics_df)

dense_metrics_df.to_csv(
    CONTROLLED_MODELS_DIR /
    "dense_autoencoder_metrics.csv",
    index=False
)

pd.DataFrame(
    dense_history.history
).to_csv(
    CONTROLLED_MODELS_DIR /
    "dense_autoencoder_training_history.csv",
    index_label="epoch"
)

dense_autoencoder.save(
    CONTROLLED_MODELS_DIR /
    "dense_autoencoder.keras"
)

controlled_results["Dense Autoencoder"] = {
    "model": dense_autoencoder,
    "metrics": dense_metrics_df,
    "validation_scores": dense_validation_scores,
    "test_scores": dense_test_scores,
    "test_predictions": dense_test_predictions,
}

In [ ]:
# =========================================================
# CELL 17 — FINAL LABEL-BLIND FIVE-MODEL COMPARISON
# =========================================================

comparison_parts = []

for model_name in [
    "Isolation Forest",
    "Local Outlier Factor",
    "Dense Autoencoder",
    "One-Class SVM",
]:
    model_metrics = controlled_results[
        model_name
    ]["metrics"]

    comparison_parts.append(
        model_metrics[
            model_metrics["split"] == "test"
        ]
    )


# Add the completed LSTM result
comparison_parts.append(
    lstm_metrics_df[
        lstm_metrics_df["split"] == "test"
    ]
)

five_model_comparison = pd.concat(
    comparison_parts,
    ignore_index=True
)

comparison_columns = [
    "model",
    "precision",
    "recall",
    "f1_score",
    "false_positive_rate",
    "roc_auc",
    "pr_auc",
    "tn",
    "fp",
    "fn",
    "tp",
    "flagged_windows",
    "flagged_ratio_percent",
    "training_seconds",
]

five_model_comparison = (
    five_model_comparison[
        comparison_columns
    ]
    .sort_values(
        "f1_score",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Final label-blind five-model NAB comparison")
print("=" * 110)
display(five_model_comparison)

five_model_comparison.to_csv(
    CONTROLLED_MODELS_DIR /
    "nab_label_blind_five_model_comparison.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 18 — FREEZE FINAL NAB FIVE-MODEL RESULTS
# =========================================================

from pathlib import Path
import json
import matplotlib.pyplot as plt

FINAL_OUTPUT_DIR = (
    NEW_EXPERIMENT_ROOT /
    "FINAL_NAB_FIVE_MODEL_RESULTS"
)

FINAL_FIGURES_DIR = FINAL_OUTPUT_DIR / "figures"
FINAL_TABLES_DIR = FINAL_OUTPUT_DIR / "tables"

for folder in [
    FINAL_OUTPUT_DIR,
    FINAL_FIGURES_DIR,
    FINAL_TABLES_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# Save complete comparison
# ---------------------------------------------------------
five_model_comparison.to_csv(
    FINAL_TABLES_DIR /
    "nab_label_blind_five_model_comparison.csv",
    index=False
)


# ---------------------------------------------------------
# Create publication table
# ---------------------------------------------------------
publication_table = (
    five_model_comparison[
        [
            "model",
            "precision",
            "recall",
            "f1_score",
            "false_positive_rate",
            "roc_auc",
            "pr_auc",
            "flagged_ratio_percent",
            "training_seconds",
        ]
    ]
    .copy()
)

publication_table.columns = [
    "Model",
    "Precision",
    "Recall",
    "F1-Score",
    "FPR",
    "ROC-AUC",
    "PR-AUC",
    "Flagged Ratio (%)",
    "Training Time (s)",
]

publication_table.to_csv(
    FINAL_TABLES_DIR /
    "nab_five_model_publication_table.csv",
    index=False
)

latex_table = publication_table.to_latex(
    index=False,
    float_format=lambda value: f"{value:.4f}",
    escape=True,
    column_format="lrrrrrrrr",
    caption=(
        "Controlled label-blind test performance of the "
        "five unsupervised anomaly-detection models on NAB."
    ),
    label="tab:nab_label_blind_five_models",
)

with open(
    FINAL_TABLES_DIR /
    "nab_five_model_publication_table.tex",
    "w",
    encoding="utf-8"
) as file:
    file.write(latex_table)


# ---------------------------------------------------------
# Figure 1: thresholded performance
# ---------------------------------------------------------
plot_df = publication_table.set_index("Model")

ax = plot_df[
    [
        "Precision",
        "Recall",
        "F1-Score",
        "FPR",
    ]
].plot(
    kind="bar",
    figsize=(11, 6)
)

ax.set_title(
    "Label-Blind NAB Test Performance at the "
    "95th-Percentile Validation Threshold"
)
ax.set_xlabel("Model")
ax.set_ylabel("Metric value")
ax.set_ylim(0, 0.40)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()

plt.savefig(
    FINAL_FIGURES_DIR /
    "nab_five_model_threshold_metrics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    FINAL_FIGURES_DIR /
    "nab_five_model_threshold_metrics.pdf",
    bbox_inches="tight"
)

plt.show()


# ---------------------------------------------------------
# Figure 2: ranking performance
# ---------------------------------------------------------
ax = plot_df[
    [
        "ROC-AUC",
        "PR-AUC",
    ]
].plot(
    kind="bar",
    figsize=(10, 6)
)

ax.set_title(
    "NAB Continuous Anomaly-Score Ranking Performance"
)
ax.set_xlabel("Model")
ax.set_ylabel("Area under curve")
ax.set_ylim(0, 0.75)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()

plt.savefig(
    FINAL_FIGURES_DIR /
    "nab_five_model_auc_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    FINAL_FIGURES_DIR /
    "nab_five_model_auc_comparison.pdf",
    bbox_inches="tight"
)

plt.show()


# ---------------------------------------------------------
# Figure 3: computational cost
# ---------------------------------------------------------
ax = plot_df[
    "Training Time (s)"
].sort_values().plot(
    kind="bar",
    figsize=(10, 6)
)

ax.set_title(
    "NAB Model Training-Time Comparison"
)
ax.set_xlabel("Model")
ax.set_ylabel("Training time in seconds — logarithmic scale")
ax.set_yscale("log")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()

plt.savefig(
    FINAL_FIGURES_DIR /
    "nab_five_model_training_time.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    FINAL_FIGURES_DIR /
    "nab_five_model_training_time.pdf",
    bbox_inches="tight"
)

plt.show()


# ---------------------------------------------------------
# Freeze experiment metadata
# ---------------------------------------------------------
final_metadata = {
    "dataset": "NAB",
    "window_minutes": 60,
    "stride_minutes": 30,
    "total_windows": 103526,
    "training_windows": 62096,
    "validation_windows": 20705,
    "test_windows": 20725,
    "test_anomalous_windows": 2543,
    "test_anomaly_ratio_percent": 12.270205,
    "split_method": (
        "Chronological 60/20/20 split performed "
        "independently for each series"
    ),
    "threshold_method": (
        "95th percentile of validation anomaly scores"
    ),
    "labels_used_during_training": False,
    "labels_used_for_threshold": False,
    "labels_used_for_final_evaluation": True,
    "feature_models": [
        "Isolation Forest",
        "Local Outlier Factor",
        "Dense Autoencoder",
        "One-Class SVM",
    ],
    "sequence_model": "LSTM Autoencoder",
    "feature_dimension": 23,
    "lstm_sequence_shape": "60 x 1",
    "best_test_f1_model": "One-Class SVM",
    "best_test_f1": 0.277765,
    "best_roc_auc_model": "Isolation Forest",
    "best_roc_auc": 0.686147,
    "best_pr_auc_model": "Isolation Forest",
    "best_pr_auc": 0.242816,
    "lowest_fpr_model": "LSTM Autoencoder",
    "lowest_fpr": 0.051589,
    "random_state": RANDOM_STATE,
}

with open(
    FINAL_OUTPUT_DIR /
    "experiment_metadata.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_metadata,
        file,
        indent=2
    )


# ---------------------------------------------------------
# Save final interpretation
# ---------------------------------------------------------
summary_text = """
# Controlled Label-Blind Five-Model NAB Comparison

All five models used the same chronological 60/20/20 split.
Labels were hidden during model fitting and threshold selection.
The operating threshold was the 95th percentile of validation
anomaly scores.

## Main results

- One-Class SVM obtained the highest test Precision, Recall,
  and F1-score.
- Isolation Forest obtained the highest ROC-AUC and PR-AUC
  and required the least training time.
- The dense Autoencoder produced competitive F1 performance
  but the highest false-positive rate.
- LOF showed moderate local-density detection performance.
- The LSTM Autoencoder produced the lowest false-positive
  rate but the lowest Recall and F1-score.
- Feature-based methods generally outperformed the
  sequence-based LSTM Autoencoder on the heterogeneous NAB
  collection.
- All test flagged-window ratios exceeded the validation
  operating ratio of approximately 5%, indicating a change
  in score distributions between validation and test periods.
"""

with open(
    FINAL_OUTPUT_DIR /
    "NAB_FIVE_MODEL_RESULTS_SUMMARY.md",
    "w",
    encoding="utf-8"
) as file:
    file.write(summary_text.strip())


print("Final NAB comparison frozen successfully.")
print(f"Tables : {FINAL_TABLES_DIR}")
print(f"Figures: {FINAL_FIGURES_DIR}")
print(f"Summary: {FINAL_OUTPUT_DIR}")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# =========================================================
# CELL 1 — ENVIRONMENT AND PATHS
# =========================================================

from pathlib import Path
import os
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ---------------------------------------------------------
# Thesis directories
# ---------------------------------------------------------
THESIS_ROOT = Path(
    "/content/drive/MyDrive/Master/Thesis"
)

CODE_ROOT = THESIS_ROOT / "Code"

BLUED_PH2_ROOT = (
    CODE_ROOT
    / "processed_data"
    / "PH2"
    / "blued_all_series"
)

BLUED_PH3_ROOT = (
    CODE_ROOT
    / "processed_data"
    / "PH3"
    / "BLUED"
)

BLUED_PH3_FEATURES = (
    BLUED_PH3_ROOT
    / "raw_windows"
    / "features"
)

BLUED_PH3_SEQUENCES = (
    BLUED_PH3_ROOT
    / "raw_windows"
    / "sequence"
)

EXPERIMENT_ROOT = (
    CODE_ROOT
    / "experiments"
    / "EXP_BLUED_FIVE_MODEL_LABEL_BLIND"
)

COMMON_OUTPUT_DIR = EXPERIMENT_ROOT / "common"
FEATURE_OUTPUT_DIR = EXPERIMENT_ROOT / "feature_models"
LSTM_OUTPUT_DIR = EXPERIMENT_ROOT / "lstm_autoencoder"
FINAL_OUTPUT_DIR = EXPERIMENT_ROOT / "final_results"

for folder in [
    EXPERIMENT_ROOT,
    COMMON_OUTPUT_DIR,
    FEATURE_OUTPUT_DIR,
    LSTM_OUTPUT_DIR,
    FINAL_OUTPUT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

paths = {
    "Thesis root": THESIS_ROOT,
    "Code root": CODE_ROOT,
    "BLUED Phase 2": BLUED_PH2_ROOT,
    "BLUED Phase 3": BLUED_PH3_ROOT,
    "Phase 3 features": BLUED_PH3_FEATURES,
    "Phase 3 sequences": BLUED_PH3_SEQUENCES,
    "Experiment output": EXPERIMENT_ROOT,
}

print("BLUED path verification")
print("=" * 100)

for name, path in paths.items():
    print(f"{name:25s}: {path.exists()} | {path}")

if not BLUED_PH2_ROOT.exists():
    raise FileNotFoundError(
        f"BLUED Phase 2 folder not found: {BLUED_PH2_ROOT}"
    )

print("\nEnvironment initialized successfully.")

In [ ]:
# =========================================================
# CELL 2 — PHASE 2 BLUED INVENTORY
# =========================================================

BLUED_SUMMARY_PATH = (
    BLUED_PH2_ROOT
    / "blued_ph2_summary.csv"
)

processed_series_files = sorted(
    path
    for path in BLUED_PH2_ROOT.rglob("*.csv")
    if path.name != "blued_ph2_summary.csv"
)

dataset_directories = sorted(
    path
    for path in BLUED_PH2_ROOT.iterdir()
    if path.is_dir()
)

print("BLUED Phase 2 inventory")
print("=" * 100)
print(f"Dataset directories : {len(dataset_directories):,}")
print(f"Processed CSV files : {len(processed_series_files):,}")
print(f"Summary exists      : {BLUED_SUMMARY_PATH.exists()}")

print("\nDataset directories")
print("-" * 100)

for path in dataset_directories:
    number_of_csvs = len(list(path.glob("*.csv")))

    print(
        f"{path.name:35s} "
        f"{number_of_csvs:>6,} CSV files"
    )

print("\nFirst 20 processed files")
print("-" * 100)

for path in processed_series_files[:20]:
    print(path.relative_to(BLUED_PH2_ROOT))

if not processed_series_files:
    raise FileNotFoundError(
        "No processed BLUED CSV files were found."
    )

In [ ]:
# =========================================================
# CELL 3 — BLUED PHASE 2 SUMMARY AUDIT
# =========================================================

if not BLUED_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"Summary file not found: {BLUED_SUMMARY_PATH}"
    )

blued_summary = pd.read_csv(
    BLUED_SUMMARY_PATH,
    low_memory=False
)

print("BLUED preprocessing summary")
print("=" * 100)
print(f"Rows   : {len(blued_summary):,}")
print(f"Columns: {list(blued_summary.columns)}")

print("\nStatus distribution")
print("-" * 100)

if "status" in blued_summary.columns:
    print(
        blued_summary["status"]
        .value_counts(dropna=False)
    )

print("\nResolution distribution")
print("-" * 100)

if "resolution" in blued_summary.columns:
    print(
        blued_summary["resolution"]
        .value_counts(dropna=False)
    )

print("\nRows-per-series distribution")
print("-" * 100)

if "rows" in blued_summary.columns:
    print(
        pd.to_numeric(
            blued_summary["rows"],
            errors="coerce"
        ).describe()
    )

# Extract source dataset from paths such as:
# location_001_dataset_003/location_001_ivdata_801.txt
if "series" in blued_summary.columns:
    blued_summary["dataset_folder"] = (
        blued_summary["series"]
        .astype(str)
        .str.split("/")
        .str[0]
    )

    print("\nFiles per BLUED dataset folder")
    print("-" * 100)

    folder_summary = (
        blued_summary
        .groupby("dataset_folder")
        .agg(
            files=("series", "size"),
            total_rows=("rows", "sum"),
        )
        .reset_index()
        .sort_values("dataset_folder")
    )

    display(folder_summary)

print("\nFirst five summary records")
print("-" * 100)
display(blued_summary.head())

In [ ]:
# =========================================================
# CELL 4 — REPRESENTATIVE FILE SCHEMA AUDIT
# =========================================================

representative_records = []

# One representative file from each dataset folder
representative_files = []

for dataset_directory in dataset_directories:
    candidates = sorted(dataset_directory.glob("*.csv"))

    if candidates:
        representative_files.append(candidates[0])

print(
    f"Inspecting {len(representative_files)} "
    "representative files..."
)

for path in representative_files:
    sample = pd.read_csv(
        path,
        low_memory=False
    )

    record = {
        "dataset_folder": path.parent.name,
        "file": path.name,
        "rows": len(sample),
        "columns": ", ".join(sample.columns),
    }

    if "timestamp" in sample.columns:
        parsed_time = pd.to_timedelta(
            sample["timestamp"],
            errors="coerce"
        )

        record["valid_timestamps"] = int(
            parsed_time.notna().sum()
        )

        if parsed_time.notna().any():
            record["first_timestamp"] = str(
                parsed_time.min()
            )

            record["last_timestamp"] = str(
                parsed_time.max()
            )

            record["duration_seconds"] = float(
                (
                    parsed_time.max()
                    - parsed_time.min()
                ).total_seconds()
            )

            record["median_interval_seconds"] = float(
                parsed_time
                .sort_values()
                .diff()
                .dt.total_seconds()
                .median()
            )

    representative_records.append(record)


representative_audit = pd.DataFrame(
    representative_records
)

display(representative_audit)

print("\nFirst representative file")
print("=" * 100)

first_file = representative_files[0]
first_sample = pd.read_csv(first_file)

print(first_file)
display(first_sample.head(10))

print("\nData types")
print(first_sample.dtypes)

In [ ]:
# =========================================================
# CELL 5 — LOCATE BLUED LABEL / EVENT METADATA
# =========================================================

SEARCH_ROOTS = [
    THESIS_ROOT / "Datasets",
    THESIS_ROOT / "Dataset",
    THESIS_ROOT / "data",
    CODE_ROOT,
]

KEYWORDS = [
    "label",
    "event",
    "annotation",
    "groundtruth",
    "ground_truth",
    "appliance",
    "location",
]

candidate_records = []

for search_root in SEARCH_ROOTS:
    if not search_root.exists():
        continue

    for root, directories, files in os.walk(search_root):
        root_path = Path(root)

        # Skip large experiment output folders that cannot
        # contain original event annotations.
        directories[:] = [
            directory
            for directory in directories
            if directory not in {
                ".git",
                "__pycache__",
                "models",
                "figures",
            }
        ]

        root_text = str(root_path).lower()

        for filename in files:
            filename_lower = filename.lower()
            full_path = root_path / filename

            # Keep files associated with BLUED or located
            # inside a BLUED directory.
            is_blued_related = (
                "blued" in root_text
                or "blued" in filename_lower
                or "location_001" in filename_lower
            )

            has_label_keyword = any(
                keyword in filename_lower
                for keyword in KEYWORDS
            )

            useful_extension = (
                full_path.suffix.lower()
                in {
                    ".csv",
                    ".txt",
                    ".json",
                    ".xlsx",
                    ".xls",
                    ".mat",
                    ".xml",
                    ".yaml",
                    ".yml",
                }
            )

            if (
                is_blued_related
                and has_label_keyword
                and useful_extension
            ):
                candidate_records.append(
                    {
                        "path": str(full_path),
                        "name": filename,
                        "suffix": full_path.suffix.lower(),
                        "size_mb": round(
                            full_path.stat().st_size
                            / (1024 ** 2),
                            4
                        ),
                    }
                )


label_candidates = (
    pd.DataFrame(candidate_records)
    .drop_duplicates()
    .sort_values(
        ["suffix", "name"],
        ignore_index=True
    )
    if candidate_records
    else pd.DataFrame(
        columns=[
            "path",
            "name",
            "suffix",
            "size_mb",
        ]
    )
)

print("Possible BLUED event/label files")
print("=" * 120)

if label_candidates.empty:
    print("No obvious event-label files were found.")
else:
    display(label_candidates)

label_candidates.to_csv(
    COMMON_OUTPUT_DIR
    / "blued_label_file_candidates.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 6 — RECONCILE PROCESSED BLUED FILE STRUCTURE
# =========================================================

processed_file_records = []

for path in processed_series_files:
    relative_path = path.relative_to(BLUED_PH2_ROOT)
    relative_parts = relative_path.parts

    processed_file_records.append(
        {
            "dataset_folder": relative_parts[0],
            "nested_folder": (
                "/".join(relative_parts[1:-1])
                if len(relative_parts) > 2
                else ""
            ),
            "filename": path.name,
            "relative_path": str(relative_path),
            "size_kb": round(
                path.stat().st_size / 1024,
                3
            ),
        }
    )

processed_file_index = pd.DataFrame(
    processed_file_records
)

processed_folder_counts = (
    processed_file_index
    .groupby("dataset_folder")
    .agg(
        successful_csv_files=("filename", "size"),
        nested_csv_files=(
            "nested_folder",
            lambda values: int((values != "").sum())
        ),
        unique_nested_folders=(
            "nested_folder",
            lambda values: int(
                values[values != ""].nunique()
            )
        ),
    )
    .reset_index()
    .sort_values("dataset_folder")
)

print("Actual processed BLUED files by dataset")
print("=" * 100)
display(processed_folder_counts)

print(
    "\nTotal successful files:",
    processed_folder_counts[
        "successful_csv_files"
    ].sum()
)

print("\nNested-folder distribution")
print("=" * 100)

nested_distribution = (
    processed_file_index[
        processed_file_index["nested_folder"] != ""
    ]
    .groupby(
        ["dataset_folder", "nested_folder"]
    )
    .size()
    .reset_index(name="csv_files")
)

if nested_distribution.empty:
    print("No nested processed CSV folders found.")
else:
    display(nested_distribution)

print("\nFiles under location_001_dataset_016")
print("=" * 100)

display(
    processed_file_index[
        processed_file_index[
            "dataset_folder"
        ] == "location_001_dataset_016"
    ][
        [
            "nested_folder",
            "filename",
            "relative_path",
        ]
    ].head(20)
)

processed_file_index.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_processed_file_index.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 7 — FIND RAW BLUED DATA DIRECTORIES
# =========================================================

likely_raw_roots = [
    CODE_ROOT / "Datasets" / "blued",
    CODE_ROOT / "Datasets" / "BLUED",
    THESIS_ROOT / "Datasets" / "blued",
    THESIS_ROOT / "Datasets" / "BLUED",
    CODE_ROOT / "data" / "blued",
    CODE_ROOT / "data" / "BLUED",
]

raw_blued_roots = []

for path in likely_raw_roots:
    if path.exists() and path.is_dir():
        raw_blued_roots.append(path.resolve())

raw_blued_roots = list(
    dict.fromkeys(raw_blued_roots)
)

print("Raw BLUED directory candidates")
print("=" * 100)

if not raw_blued_roots:
    print("No expected raw BLUED root was found.")
else:
    for index, path in enumerate(
        raw_blued_roots,
        start=1
    ):
        print(f"{index:02d}. {path}")

if not raw_blued_roots:
    raise FileNotFoundError(
        "The raw BLUED directory was not found in "
        "the expected thesis locations."
    )

RAW_BLUED_ROOT = raw_blued_roots[0]

print("\nSelected raw BLUED root:")
print(RAW_BLUED_ROOT)

print("\nTop-level contents")
print("-" * 100)

for path in sorted(RAW_BLUED_ROOT.iterdir()):
    print(
        f"{'DIR ' if path.is_dir() else 'FILE'} "
        f"{path.name}"
    )

In [ ]:
# =========================================================
# CELL 8 — EXACT EVENT-LIST DISCOVERY
# =========================================================

EVENT_FILENAME_PATTERNS = [
    "eventlist",
    "event_list",
    "event-list",
    "eventslist",
    "events_list",
    "annotation",
    "groundtruth",
    "ground_truth",
]

EVENT_EXTENSIONS = {
    ".txt",
    ".csv",
    ".tsv",
    ".xls",
    ".xlsx",
    ".json",
    ".mat",
}

event_file_records = []

for raw_root in raw_blued_roots:
    for path in raw_root.rglob("*"):
        if not path.is_file():
            continue

        filename_lower = path.name.lower()

        matches_event_pattern = any(
            pattern in filename_lower
            for pattern in EVENT_FILENAME_PATTERNS
        )

        valid_extension = (
            path.suffix.lower() in EVENT_EXTENSIONS
        )

        # Exclude raw waveform/IV files.
        is_ivdata = (
            "ivdata" in filename_lower
        )

        if (
            matches_event_pattern
            and valid_extension
            and not is_ivdata
        ):
            event_file_records.append(
                {
                    "path": str(path),
                    "relative_path": str(
                        path.relative_to(raw_root)
                    ),
                    "filename": path.name,
                    "suffix": path.suffix.lower(),
                    "size_kb": round(
                        path.stat().st_size / 1024,
                        3
                    ),
                }
            )

event_files_df = (
    pd.DataFrame(event_file_records)
    .drop_duplicates(subset=["path"])
    .sort_values(
        ["filename", "path"],
        ignore_index=True
    )
    if event_file_records
    else pd.DataFrame(
        columns=[
            "path",
            "relative_path",
            "filename",
            "suffix",
            "size_kb",
        ]
    )
)

print("Actual BLUED event-list candidates")
print("=" * 120)

if event_files_df.empty:
    print("No event-list files found.")
else:
    display(event_files_df)

event_files_df.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_actual_event_file_candidates.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 9 — EVENT-LIST SCHEMA PREVIEW
# =========================================================

def read_text_with_fallback(
    path: Path
) -> tuple[str, str]:
    """Read a text file using common encoding fallbacks."""

    encodings = [
        "utf-8-sig",
        "utf-8",
        "latin-1",
        "cp1252",
    ]

    for encoding in encodings:
        try:
            return (
                path.read_text(encoding=encoding),
                encoding,
            )
        except UnicodeDecodeError:
            continue

    raise UnicodeDecodeError(
        "unknown",
        b"",
        0,
        1,
        f"Unable to decode {path}"
    )


if event_files_df.empty:
    raise FileNotFoundError(
        "No event-list files are available to preview."
    )

for _, event_record in event_files_df.iterrows():
    event_path = Path(event_record["path"])

    print("\n" + "=" * 120)
    print(f"FILE: {event_path}")
    print("=" * 120)

    if event_path.suffix.lower() in {
        ".xls",
        ".xlsx",
    }:
        workbook = pd.ExcelFile(event_path)

        print("Sheets:", workbook.sheet_names)

        for sheet_name in workbook.sheet_names:
            print(f"\nSheet: {sheet_name}")

            sample = pd.read_excel(
                event_path,
                sheet_name=sheet_name,
                nrows=20
            )

            display(sample)

    elif event_path.suffix.lower() in {
        ".csv",
        ".tsv",
    }:
        separator = (
            "\t"
            if event_path.suffix.lower() == ".tsv"
            else None
        )

        try:
            sample = pd.read_csv(
                event_path,
                sep=separator,
                engine="python",
                nrows=20
            )

            display(sample)

        except Exception as error:
            print(
                "Structured read failed:",
                repr(error)
            )

            text, encoding = (
                read_text_with_fallback(event_path)
            )

            print(f"Encoding: {encoding}")

            for line_number, line in enumerate(
                text.splitlines()[:40],
                start=1
            ):
                print(
                    f"{line_number:03d}: {line}"
                )

    elif event_path.suffix.lower() == ".json":
        with open(
            event_path,
            "r",
            encoding="utf-8"
        ) as file:
            data = json.load(file)

        print(
            json.dumps(
                data,
                indent=2
            )[:8000]
        )

    elif event_path.suffix.lower() == ".mat":
        print(
            "MATLAB annotation file detected. "
            "Its variable structure will be inspected next."
        )

    else:
        text, encoding = read_text_with_fallback(
            event_path
        )

        print(f"Encoding: {encoding}")
        print(
            f"Non-empty lines: "
            f"{sum(bool(line.strip()) for line in text.splitlines()):,}"
        )

        print("\nFirst 40 lines")
        print("-" * 120)

        for line_number, line in enumerate(
            text.splitlines()[:40],
            start=1
        ):
            print(
                f"{line_number:03d}: {line}"
            )

In [ ]:
# =========================================================
# CELL 8 — CLEAN BLUED EVENT-LIST DISCOVERY
# =========================================================

EVENT_EXTENSIONS = {
    ".txt",
    ".csv",
    ".tsv",
    ".xls",
    ".xlsx",
    ".json",
    ".mat",
}

event_file_records = []

for path in RAW_BLUED_ROOT.rglob("*"):
    if not path.is_file():
        continue

    relative_path = path.relative_to(RAW_BLUED_ROOT)
    filename_lower = path.name.lower()

    # -----------------------------------------------------
    # Exclude macOS metadata and hidden/system files
    # -----------------------------------------------------
    if path.name.startswith("._"):
        continue

    if path.name in {
        ".DS_Store",
        "Thumbs.db",
    }:
        continue

    if "__MACOSX" in relative_path.parts:
        continue

    # Exclude waveform data files
    if "ivdata" in filename_lower:
        continue

    # Keep only actual event-list files
    is_event_list = (
        "eventlist" in filename_lower
        or "eventslist" in filename_lower
    )

    if not is_event_list:
        continue

    if path.suffix.lower() not in EVENT_EXTENSIONS:
        continue

    # Detect the dataset identifier from the path
    dataset_parts = [
        part
        for part in relative_path.parts
        if part.startswith("location_001_dataset_")
    ]

    dataset_identifier = (
        dataset_parts[-1]
        if dataset_parts
        else relative_path.parts[0]
    )

    stem_lower = path.stem.lower()

    if stem_lower.endswith("_a"):
        event_phase = "A"
    elif stem_lower.endswith("_b"):
        event_phase = "B"
    else:
        event_phase = "combined"

    event_file_records.append(
        {
            "dataset": dataset_identifier,
            "phase": event_phase,
            "path": str(path),
            "relative_path": str(relative_path),
            "filename": path.name,
            "suffix": path.suffix.lower(),
            "size_kb": round(
                path.stat().st_size / 1024,
                3
            ),
        }
    )


event_files_df = pd.DataFrame(
    event_file_records
)

if not event_files_df.empty:
    event_files_df = (
        event_files_df
        .drop_duplicates(subset=["path"])
        .sort_values(
            [
                "dataset",
                "phase",
                "relative_path",
            ],
            ignore_index=True
        )
    )


print("Clean BLUED event-list files")
print("=" * 120)

if event_files_df.empty:
    print("No valid event-list files found.")
else:
    display(event_files_df)

    print("\nSummary by phase")
    print("-" * 80)

    display(
        event_files_df[
            "phase"
        ].value_counts()
        .rename_axis("phase")
        .reset_index(name="files")
    )

    print("\nSummary by dataset")
    print("-" * 80)

    display(
        event_files_df
        .groupby("dataset")
        .agg(
            files=("path", "size"),
            phases=(
                "phase",
                lambda values: ", ".join(
                    sorted(values.unique())
                )
            ),
        )
        .reset_index()
    )

print(
    f"\nValid event-list files found: "
    f"{len(event_files_df):,}"
)

event_files_df.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_clean_event_file_index.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 9 — SAFE EVENT-LIST SCHEMA PREVIEW
# =========================================================

def read_event_text_safely(
    path: Path
) -> tuple[str, str]:
    """
    Read an event-list text file using safe encoding
    fallbacks. This function will not fail on unusual bytes.
    """

    raw_bytes = path.read_bytes()

    encodings = [
        "utf-8-sig",
        "utf-8",
        "cp1252",
        "latin-1",
    ]

    for encoding in encodings:
        try:
            return (
                raw_bytes.decode(encoding),
                encoding
            )
        except UnicodeDecodeError:
            continue

    # Final replacement-based fallback
    return (
        raw_bytes.decode(
            "latin-1",
            errors="replace"
        ),
        "latin-1-with-replacement"
    )


if event_files_df.empty:
    raise FileNotFoundError(
        "No valid BLUED event-list files are available."
    )


preview_records = []
preview_errors = []

for _, event_record in event_files_df.iterrows():
    event_path = Path(
        event_record["path"]
    )

    print("\n" + "=" * 120)
    print(
        f"DATASET: {event_record['dataset']} | "
        f"PHASE: {event_record['phase']}"
    )
    print(f"FILE: {event_path}")
    print("=" * 120)

    try:
        suffix = event_path.suffix.lower()

        if suffix in {".xls", ".xlsx"}:
            workbook = pd.ExcelFile(event_path)

            print(
                "Sheets:",
                workbook.sheet_names
            )

            for sheet_name in workbook.sheet_names:
                sample = pd.read_excel(
                    event_path,
                    sheet_name=sheet_name,
                    nrows=20
                )

                print(f"\nSheet: {sheet_name}")
                display(sample)

                preview_records.append(
                    {
                        "dataset": event_record["dataset"],
                        "phase": event_record["phase"],
                        "file": event_path.name,
                        "encoding": "Excel",
                        "nonempty_lines": len(sample),
                    }
                )

        elif suffix == ".json":
            with open(
                event_path,
                "r",
                encoding="utf-8"
            ) as file:
                data = json.load(file)

            print(
                json.dumps(
                    data,
                    indent=2
                )[:6000]
            )

            preview_records.append(
                {
                    "dataset": event_record["dataset"],
                    "phase": event_record["phase"],
                    "file": event_path.name,
                    "encoding": "JSON",
                    "nonempty_lines": np.nan,
                }
            )

        elif suffix == ".mat":
            print(
                "MATLAB annotation file detected. "
                "It will be inspected separately."
            )

            preview_records.append(
                {
                    "dataset": event_record["dataset"],
                    "phase": event_record["phase"],
                    "file": event_path.name,
                    "encoding": "MATLAB",
                    "nonempty_lines": np.nan,
                }
            )

        else:
            text, encoding = read_event_text_safely(
                event_path
            )

            nonempty_lines = [
                line.strip()
                for line in text.splitlines()
                if line.strip()
            ]

            print(f"Encoding: {encoding}")
            print(
                f"Non-empty lines: "
                f"{len(nonempty_lines):,}"
            )

            print("\nFirst 25 non-empty lines")
            print("-" * 120)

            for line_number, line in enumerate(
                nonempty_lines[:25],
                start=1
            ):
                print(
                    f"{line_number:03d}: {line}"
                )

            preview_records.append(
                {
                    "dataset": event_record["dataset"],
                    "phase": event_record["phase"],
                    "file": event_path.name,
                    "encoding": encoding,
                    "nonempty_lines": len(nonempty_lines),
                }
            )

    except Exception as error:
        preview_errors.append(
            {
                "dataset": event_record["dataset"],
                "phase": event_record["phase"],
                "file": str(event_path),
                "error_type": type(error).__name__,
                "error_message": str(error),
            }
        )

        print(
            f"Preview failed but execution continued: "
            f"{type(error).__name__}: {error}"
        )


event_preview_summary = pd.DataFrame(
    preview_records
)

event_preview_errors = pd.DataFrame(
    preview_errors
)

print("\n" + "=" * 120)
print("EVENT PREVIEW SUMMARY")
print("=" * 120)

display(event_preview_summary)

print("\nPreview errors")
print("-" * 120)

if event_preview_errors.empty:
    print("No preview errors.")
else:
    display(event_preview_errors)


event_preview_summary.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_event_preview_summary.csv",
    index=False
)

event_preview_errors.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_event_preview_errors.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 10 — PARSE COMBINED BLUED EVENT ANNOTATIONS
# =========================================================

combined_event_files = (
    event_files_df[
        event_files_df["phase"] == "combined"
    ]
    .copy()
    .sort_values("dataset")
    .reset_index(drop=True)
)

print("Combined annotation files")
print("=" * 100)
print(f"Files: {len(combined_event_files):,}")

display(
    combined_event_files[
        [
            "dataset",
            "relative_path",
            "size_kb",
        ]
    ]
)

if len(combined_event_files) != 16:
    raise ValueError(
        "Expected 16 combined BLUED event-list files, "
        f"but found {len(combined_event_files)}."
    )


event_tables = []
event_parsing_records = []

for _, file_record in combined_event_files.iterrows():
    dataset_name = file_record["dataset"]
    event_path = Path(file_record["path"])

    event_df = pd.read_csv(
        event_path,
        encoding="utf-8-sig"
    )

    event_df.columns = [
        str(column).strip()
        for column in event_df.columns
    ]

    required_columns = {
        "Timestamp",
        "Label",
        "Phase",
    }

    missing_columns = (
        required_columns - set(event_df.columns)
    )

    if missing_columns:
        raise ValueError(
            f"Missing columns in {event_path}: "
            f"{missing_columns}"
        )

    event_df["event_timestamp"] = pd.to_datetime(
        event_df["Timestamp"],
        errors="coerce"
    )

    event_df["event_code"] = pd.to_numeric(
        event_df["Label"],
        errors="coerce"
    ).astype("Int64")

    event_df["phase"] = (
        event_df["Phase"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    invalid_rows = (
        event_df["event_timestamp"].isna()
        | event_df["event_code"].isna()
        | ~event_df["phase"].isin(["A", "B"])
    )

    invalid_count = int(invalid_rows.sum())

    if invalid_count > 0:
        print(
            f"Warning: dropping {invalid_count} invalid rows "
            f"from {dataset_name}"
        )

        event_df = event_df.loc[
            ~invalid_rows
        ].copy()

    event_df["dataset"] = dataset_name
    event_df["source_file"] = str(event_path)

    event_df = event_df[
        [
            "dataset",
            "event_timestamp",
            "event_code",
            "phase",
            "source_file",
        ]
    ]

    event_tables.append(event_df)

    event_parsing_records.append(
        {
            "dataset": dataset_name,
            "parsed_events": len(event_df),
            "phase_a_events": int(
                (event_df["phase"] == "A").sum()
            ),
            "phase_b_events": int(
                (event_df["phase"] == "B").sum()
            ),
            "unique_event_codes": int(
                event_df["event_code"].nunique()
            ),
            "first_event": event_df[
                "event_timestamp"
            ].min(),
            "last_event": event_df[
                "event_timestamp"
            ].max(),
            "invalid_rows_removed": invalid_count,
        }
    )


blued_events = (
    pd.concat(
        event_tables,
        ignore_index=True
    )
    .sort_values(
        [
            "event_timestamp",
            "dataset",
            "phase",
            "event_code",
        ]
    )
    .reset_index(drop=True)
)

blued_events["event_id"] = np.arange(
    len(blued_events),
    dtype=np.int64
)


event_parsing_summary = pd.DataFrame(
    event_parsing_records
)

print("\nBLUED combined-event summary")
print("=" * 110)
display(event_parsing_summary)

print("\nOverall event statistics")
print("-" * 100)
print(f"Total event records  : {len(blued_events):,}")
print(
    f"Unique event codes   : "
    f"{blued_events['event_code'].nunique():,}"
)
print(
    f"Phase A events       : "
    f"{(blued_events['phase'] == 'A').sum():,}"
)
print(
    f"Phase B events       : "
    f"{(blued_events['phase'] == 'B').sum():,}"
)
print(
    f"First event          : "
    f"{blued_events['event_timestamp'].min()}"
)
print(
    f"Last event           : "
    f"{blued_events['event_timestamp'].max()}"
)


# Exact duplicate audit
duplicate_columns = [
    "dataset",
    "event_timestamp",
    "event_code",
    "phase",
]

exact_duplicate_mask = blued_events.duplicated(
    subset=duplicate_columns,
    keep=False
)

print(
    f"Exact duplicate rows : "
    f"{exact_duplicate_mask.sum():,}"
)

if exact_duplicate_mask.any():
    display(
        blued_events.loc[
            exact_duplicate_mask,
            duplicate_columns
        ].head(30)
    )


blued_events.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_combined_event_annotations.csv",
    index=False
)

event_parsing_summary.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_event_parsing_summary.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 11 — VERIFY COMBINED = PHASE A UNION PHASE B
# =========================================================

def parse_single_event_file(
    event_path: Path,
    dataset_name: str
) -> pd.DataFrame:
    """Parse one BLUED annotation file."""

    frame = pd.read_csv(
        event_path,
        encoding="utf-8-sig"
    )

    frame.columns = [
        str(column).strip()
        for column in frame.columns
    ]

    frame["event_timestamp"] = pd.to_datetime(
        frame["Timestamp"],
        errors="coerce"
    )

    frame["event_code"] = pd.to_numeric(
        frame["Label"],
        errors="coerce"
    ).astype("Int64")

    frame["phase"] = (
        frame["Phase"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    frame["dataset"] = dataset_name

    frame = frame.dropna(
        subset=[
            "event_timestamp",
            "event_code",
        ]
    )

    return frame[
        [
            "dataset",
            "event_timestamp",
            "event_code",
            "phase",
        ]
    ]


verification_records = []
verification_mismatches = []

for dataset_name in sorted(
    event_files_df["dataset"].unique()
):
    dataset_files = event_files_df[
        event_files_df["dataset"] == dataset_name
    ]

    file_lookup = {
        row["phase"]: Path(row["path"])
        for _, row in dataset_files.iterrows()
    }

    required_phases = {
        "A",
        "B",
        "combined",
    }

    if not required_phases.issubset(file_lookup):
        raise ValueError(
            f"Incomplete annotation set for {dataset_name}: "
            f"{set(file_lookup)}"
        )

    phase_a = parse_single_event_file(
        file_lookup["A"],
        dataset_name
    )

    phase_b = parse_single_event_file(
        file_lookup["B"],
        dataset_name
    )

    combined = parse_single_event_file(
        file_lookup["combined"],
        dataset_name
    )

    separate = pd.concat(
        [phase_a, phase_b],
        ignore_index=True
    )

    key_columns = [
        "dataset",
        "event_timestamp",
        "event_code",
        "phase",
    ]

    separate_counts = (
        separate
        .groupby(key_columns)
        .size()
        .rename("separate_count")
        .reset_index()
    )

    combined_counts = (
        combined
        .groupby(key_columns)
        .size()
        .rename("combined_count")
        .reset_index()
    )

    comparison = separate_counts.merge(
        combined_counts,
        on=key_columns,
        how="outer"
    )

    comparison[
        [
            "separate_count",
            "combined_count",
        ]
    ] = comparison[
        [
            "separate_count",
            "combined_count",
        ]
    ].fillna(0).astype(int)

    comparison["count_difference"] = (
        comparison["combined_count"]
        - comparison["separate_count"]
    )

    mismatches = comparison[
        comparison["count_difference"] != 0
    ].copy()

    verification_records.append(
        {
            "dataset": dataset_name,
            "phase_a_rows": len(phase_a),
            "phase_b_rows": len(phase_b),
            "a_plus_b_rows": len(separate),
            "combined_rows": len(combined),
            "mismatched_event_keys": len(mismatches),
            "verification_passed": (
                len(mismatches) == 0
            ),
        }
    )

    if not mismatches.empty:
        verification_mismatches.append(
            mismatches
        )


combined_verification = pd.DataFrame(
    verification_records
)

print("Combined-event verification")
print("=" * 110)
display(combined_verification)

print(
    "\nDatasets passed:",
    int(
        combined_verification[
            "verification_passed"
        ].sum()
    ),
    "/",
    len(combined_verification)
)

if verification_mismatches:
    mismatch_table = pd.concat(
        verification_mismatches,
        ignore_index=True
    )

    print("\nMismatches detected")
    display(mismatch_table.head(50))

    mismatch_table.to_csv(
        COMMON_OUTPUT_DIR /
        "blued_combined_event_mismatches.csv",
        index=False
    )
else:
    print(
        "\nAll combined annotation files exactly match "
        "their Phase A and Phase B files."
    )


combined_verification.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_combined_event_verification.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 12 — PROCESSED SEGMENT TEMPORAL COVERAGE
# =========================================================

segment_records = []
segment_errors = []

print("Auditing 6,430 processed BLUED segments...")
print("=" * 100)

for file_number, segment_path in enumerate(
    processed_series_files,
    start=1
):
    relative_path = segment_path.relative_to(
        BLUED_PH2_ROOT
    )

    dataset_parts = [
        part
        for part in relative_path.parts
        if part.startswith(
            "location_001_dataset_"
        )
    ]

    logical_dataset = (
        dataset_parts[-1]
        if dataset_parts
        else relative_path.parts[0]
    )

    try:
        segment_df = pd.read_csv(
            segment_path,
            usecols=[
                "timestamp",
                "mean_power",
                "max_power",
                "std_power",
            ],
            low_memory=False
        )

        segment_time = pd.to_timedelta(
            segment_df["timestamp"],
            errors="coerce"
        )

        valid_time = segment_time.dropna()

        if valid_time.empty:
            raise ValueError(
                "No valid timestamp values"
            )

        sorted_time = valid_time.sort_values()

        segment_records.append(
            {
                "dataset": logical_dataset,
                "relative_path": str(relative_path),
                "filename": segment_path.name,
                "rows": len(segment_df),
                "valid_timestamp_rows": len(valid_time),
                "elapsed_start": sorted_time.iloc[0],
                "elapsed_end": sorted_time.iloc[-1],
                "median_interval_seconds": float(
                    sorted_time
                    .diff()
                    .dt.total_seconds()
                    .median()
                ),
            }
        )

    except Exception as error:
        segment_errors.append(
            {
                "path": str(segment_path),
                "error_type": type(error).__name__,
                "error_message": str(error),
            }
        )

    if (
        file_number % 500 == 0
        or file_number == len(processed_series_files)
    ):
        print(
            f"Processed "
            f"{file_number:,}/"
            f"{len(processed_series_files):,}"
        )


segment_index = pd.DataFrame(
    segment_records
)

segment_error_df = pd.DataFrame(
    segment_errors
)

print("\nSegment audit")
print("-" * 100)
print(f"Successful segments: {len(segment_index):,}")
print(f"Failed segments    : {len(segment_error_df):,}")


dataset_temporal_coverage = (
    segment_index
    .groupby("dataset")
    .agg(
        segment_files=("filename", "size"),
        total_rows=("rows", "sum"),
        elapsed_start=("elapsed_start", "min"),
        elapsed_end=("elapsed_end", "max"),
        minimum_rows=("rows", "min"),
        median_rows=("rows", "median"),
        maximum_rows=("rows", "max"),
        median_interval_seconds=(
            "median_interval_seconds",
            "median"
        ),
    )
    .reset_index()
    .sort_values("dataset")
)

dataset_temporal_coverage[
    "coverage_hours"
] = (
    dataset_temporal_coverage[
        "elapsed_end"
    ]
    - dataset_temporal_coverage[
        "elapsed_start"
    ]
).dt.total_seconds() / 3600


print("\nProcessed temporal coverage")
print("=" * 120)
display(dataset_temporal_coverage)

print(
    "\nLogical datasets found:",
    dataset_temporal_coverage[
        "dataset"
    ].nunique()
)

segment_index.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_processed_segment_index.csv",
    index=False
)

dataset_temporal_coverage.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_dataset_temporal_coverage.csv",
    index=False
)

segment_error_df.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_segment_audit_errors.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 13 — INFER BLUED RECORDING ORIGIN
# =========================================================

event_coverage = (
    blued_events
    .groupby("dataset")
    .agg(
        event_count=("event_id", "size"),
        first_event=("event_timestamp", "min"),
        last_event=("event_timestamp", "max"),
    )
    .reset_index()
)

alignment_table = (
    dataset_temporal_coverage
    .merge(
        event_coverage,
        on="dataset",
        how="inner",
        validate="one_to_one"
    )
)

# For all events to fit within the corresponding processed
# elapsed-time interval:
#
# origin + elapsed_start <= first_event
# origin + elapsed_end   >= last_event
#
# Therefore:
# origin >= last_event - elapsed_end
# origin <= first_event - elapsed_start

alignment_table[
    "origin_lower_bound"
] = (
    alignment_table["last_event"]
    - alignment_table["elapsed_end"]
)

alignment_table[
    "origin_upper_bound"
] = (
    alignment_table["first_event"]
    - alignment_table["elapsed_start"]
)

alignment_table[
    "dataset_alignment_possible"
] = (
    alignment_table["origin_lower_bound"]
    <= alignment_table["origin_upper_bound"]
)


print("Per-dataset recording-origin constraints")
print("=" * 130)

display(
    alignment_table[
        [
            "dataset",
            "segment_files",
            "event_count",
            "elapsed_start",
            "elapsed_end",
            "first_event",
            "last_event",
            "origin_lower_bound",
            "origin_upper_bound",
            "dataset_alignment_possible",
        ]
    ]
)

global_origin_lower = (
    alignment_table[
        "origin_lower_bound"
    ].max()
)

global_origin_upper = (
    alignment_table[
        "origin_upper_bound"
    ].min()
)

print("\nGlobal recording-origin interval")
print("-" * 100)
print(f"Lower bound: {global_origin_lower}")
print(f"Upper bound: {global_origin_upper}")

if global_origin_lower <= global_origin_upper:
    print("\nA common recording origin exists.")

    inferred_origin_midpoint = (
        global_origin_lower
        + (
            global_origin_upper
            - global_origin_lower
        ) / 2
    )

    print(
        f"Interval midpoint: "
        f"{inferred_origin_midpoint}"
    )
else:
    print(
        "\nNo common origin exists yet. "
        "The dataset-folder mapping or elapsed timestamps "
        "need further inspection."
    )


alignment_table.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_recording_origin_constraints.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 14 — AUDIT DATASET 013 EXTRA SEGMENTS
# =========================================================

DATASET_013 = "location_001_dataset_013"
DATASET_012 = "location_001_dataset_012"

dataset_013_segments = (
    segment_index[
        segment_index["dataset"] == DATASET_013
    ]
    .copy()
    .sort_values(
        ["elapsed_start", "elapsed_end", "relative_path"]
    )
    .reset_index(drop=True)
)

dataset_012_end = (
    dataset_temporal_coverage
    .loc[
        dataset_temporal_coverage["dataset"]
        == DATASET_012,
        "elapsed_end"
    ]
    .iloc[0]
)

print("Dataset 013 segment audit")
print("=" * 110)
print(f"Segments found         : {len(dataset_013_segments):,}")
print(f"Dataset 012 final time : {dataset_012_end}")

# Any legitimate continuation of dataset 012 should be
# close to or later than dataset 012's final elapsed time.
dataset_013_segments["before_dataset_012_end"] = (
    dataset_013_segments["elapsed_end"]
    < dataset_012_end
)

suspected_extra_segments = dataset_013_segments[
    dataset_013_segments["before_dataset_012_end"]
].copy()

likely_continuation_segments = dataset_013_segments[
    ~dataset_013_segments["before_dataset_012_end"]
].copy()

print("\nClassification")
print("-" * 110)
print(
    f"Suspected reset/extra segments : "
    f"{len(suspected_extra_segments):,}"
)
print(
    f"Likely continuation segments   : "
    f"{len(likely_continuation_segments):,}"
)

print("\nSuspected extra segments")
print("-" * 110)

display(
    suspected_extra_segments[
        [
            "relative_path",
            "filename",
            "rows",
            "elapsed_start",
            "elapsed_end",
        ]
    ]
)

print("\nFirst legitimate continuation segments")
print("-" * 110)

display(
    likely_continuation_segments[
        [
            "relative_path",
            "filename",
            "rows",
            "elapsed_start",
            "elapsed_end",
        ]
    ].head(20)
)

print("\nLast legitimate continuation segments")
print("-" * 110)

display(
    likely_continuation_segments[
        [
            "relative_path",
            "filename",
            "rows",
            "elapsed_start",
            "elapsed_end",
        ]
    ].tail(20)
)

suspected_extra_segments.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_dataset_013_suspected_extra_segments.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 15 — DATASET 013 DUPLICATE-CONTENT AUDIT
# =========================================================

import hashlib


def calculate_file_content_signature(
    path: Path
) -> str:
    """
    Create a stable signature from the processed numerical
    content rather than from the filename.
    """

    frame = pd.read_csv(
        path,
        usecols=[
            "timestamp",
            "mean_power",
            "max_power",
            "std_power",
        ],
        low_memory=False
    )

    hashed_values = pd.util.hash_pandas_object(
        frame,
        index=False
    ).values.tobytes()

    return hashlib.sha256(
        hashed_values
    ).hexdigest()


print("Calculating signatures for suspected segments...")
print("=" * 100)

suspected_signature_records = []

for _, row in suspected_extra_segments.iterrows():
    path = BLUED_PH2_ROOT / row["relative_path"]

    suspected_signature_records.append(
        {
            "relative_path": row["relative_path"],
            "signature": (
                calculate_file_content_signature(path)
            ),
        }
    )

suspected_signatures = pd.DataFrame(
    suspected_signature_records
)


# Search for matching content outside dataset 013.
comparison_segments = segment_index[
    segment_index["dataset"] != DATASET_013
].copy()

matching_records = []

suspected_signature_set = set(
    suspected_signatures["signature"]
)

print(
    f"Searching {len(comparison_segments):,} other files..."
)

for file_number, (_, row) in enumerate(
    comparison_segments.iterrows(),
    start=1
):
    path = BLUED_PH2_ROOT / row["relative_path"]

    signature = calculate_file_content_signature(path)

    if signature in suspected_signature_set:
        matching_suspect = suspected_signatures[
            suspected_signatures["signature"] == signature
        ]

        for _, suspect_row in matching_suspect.iterrows():
            matching_records.append(
                {
                    "dataset_013_file": (
                        suspect_row["relative_path"]
                    ),
                    "matching_file": row["relative_path"],
                    "matching_dataset": row["dataset"],
                    "signature": signature,
                }
            )

    if file_number % 500 == 0:
        print(
            f"Checked {file_number:,}/"
            f"{len(comparison_segments):,}"
        )


dataset_013_duplicate_matches = pd.DataFrame(
    matching_records
)

print("\nDuplicate-content matches")
print("=" * 110)

if dataset_013_duplicate_matches.empty:
    print(
        "No exact content duplicates were found outside "
        "dataset 013."
    )
else:
    display(dataset_013_duplicate_matches)

dataset_013_duplicate_matches.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_dataset_013_duplicate_matches.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 16 — INFER TWO BLUED RECORDING ORIGINS
# =========================================================

BLOCK_1_DATASETS = [
    f"location_001_dataset_{number:03d}"
    for number in range(1, 13)
]

BLOCK_2_REFERENCE_DATASETS = [
    "location_001_dataset_014",
    "location_001_dataset_016",
]


def calculate_origin_interval(
    alignment_frame: pd.DataFrame,
    datasets: list[str],
    block_name: str
) -> dict:
    """Calculate the common origin interval for a block."""

    subset = alignment_frame[
        alignment_frame["dataset"].isin(datasets)
    ].copy()

    missing = set(datasets) - set(subset["dataset"])

    if missing:
        raise ValueError(
            f"Missing datasets for {block_name}: {missing}"
        )

    lower_bound = subset[
        "origin_lower_bound"
    ].max()

    upper_bound = subset[
        "origin_upper_bound"
    ].min()

    possible = lower_bound <= upper_bound

    midpoint = (
        lower_bound
        + (upper_bound - lower_bound) / 2
        if possible
        else pd.NaT
    )

    rounded_midpoint = (
        midpoint.round("min")
        if possible
        else pd.NaT
    )

    return {
        "block": block_name,
        "datasets": ", ".join(datasets),
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "alignment_possible": possible,
        "interval_midpoint": midpoint,
        "rounded_origin_candidate": rounded_midpoint,
    }


origin_results = [
    calculate_origin_interval(
        alignment_table,
        BLOCK_1_DATASETS,
        "Block 1"
    ),
    calculate_origin_interval(
        alignment_table,
        BLOCK_2_REFERENCE_DATASETS,
        "Block 2"
    ),
]

origin_summary = pd.DataFrame(origin_results)

print("BLUED recording-block origins")
print("=" * 130)
display(origin_summary)

if not origin_summary[
    "alignment_possible"
].all():
    raise ValueError(
        "At least one recording block does not have a "
        "consistent origin interval."
    )

BLOCK_1_ORIGIN = origin_summary.loc[
    origin_summary["block"] == "Block 1",
    "rounded_origin_candidate"
].iloc[0]

BLOCK_2_ORIGIN = origin_summary.loc[
    origin_summary["block"] == "Block 2",
    "rounded_origin_candidate"
].iloc[0]

print("\nSelected rounded origin candidates")
print("-" * 100)
print(f"Block 1: {BLOCK_1_ORIGIN}")
print(f"Block 2: {BLOCK_2_ORIGIN}")

origin_summary.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_recording_block_origins.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 17 — EVENT-TO-SIGNAL COVERAGE AUDIT
# =========================================================

dataset_origin_map = {}

for dataset_name in BLOCK_1_DATASETS:
    dataset_origin_map[dataset_name] = BLOCK_1_ORIGIN

dataset_origin_map[
    "location_001_dataset_013"
] = BLOCK_1_ORIGIN

for dataset_name in [
    "location_001_dataset_014",
    "location_001_dataset_015",
    "location_001_dataset_016",
]:
    dataset_origin_map[dataset_name] = BLOCK_2_ORIGIN


# Correct dataset 013 coverage temporarily by using only
# the likely continuation segments.
corrected_013_start = (
    likely_continuation_segments[
        "elapsed_start"
    ].min()
)

corrected_013_end = (
    likely_continuation_segments[
        "elapsed_end"
    ].max()
)

corrected_coverage = (
    dataset_temporal_coverage
    .copy()
)

corrected_coverage.loc[
    corrected_coverage["dataset"]
    == "location_001_dataset_013",
    "elapsed_start"
] = corrected_013_start

corrected_coverage.loc[
    corrected_coverage["dataset"]
    == "location_001_dataset_013",
    "elapsed_end"
] = corrected_013_end


coverage_lookup = corrected_coverage.set_index(
    "dataset"
)

event_alignment_audit = blued_events.copy()

event_alignment_audit["recording_origin"] = (
    event_alignment_audit["dataset"]
    .map(dataset_origin_map)
)

if event_alignment_audit[
    "recording_origin"
].isna().any():
    missing_datasets = event_alignment_audit.loc[
        event_alignment_audit[
            "recording_origin"
        ].isna(),
        "dataset"
    ].unique()

    raise ValueError(
        f"Missing origin assignments: {missing_datasets}"
    )


event_alignment_audit["event_elapsed"] = (
    event_alignment_audit["event_timestamp"]
    - event_alignment_audit["recording_origin"]
)

event_alignment_audit["coverage_start"] = (
    event_alignment_audit["dataset"]
    .map(coverage_lookup["elapsed_start"])
)

event_alignment_audit["coverage_end"] = (
    event_alignment_audit["dataset"]
    .map(coverage_lookup["elapsed_end"])
)

event_alignment_audit["inside_signal_coverage"] = (
    (
        event_alignment_audit["event_elapsed"]
        >= event_alignment_audit["coverage_start"]
    )
    &
    (
        event_alignment_audit["event_elapsed"]
        <= event_alignment_audit["coverage_end"]
    )
)


outside_coverage_events = event_alignment_audit[
    ~event_alignment_audit[
        "inside_signal_coverage"
    ]
].copy()

print("BLUED event-to-signal alignment")
print("=" * 120)
print(
    f"Events inside coverage  : "
    f"{event_alignment_audit['inside_signal_coverage'].sum():,}"
)
print(
    f"Events outside coverage : "
    f"{len(outside_coverage_events):,}"
)

print("\nOutside-coverage events")
print("-" * 120)

if outside_coverage_events.empty:
    print("No events fall outside signal coverage.")
else:
    display(
        outside_coverage_events[
            [
                "dataset",
                "event_timestamp",
                "event_code",
                "phase",
                "event_elapsed",
                "coverage_start",
                "coverage_end",
            ]
        ]
    )


# Canonical BLUED events:
# combined event lists, inside corresponding signal coverage,
# and exact duplicates removed.
canonical_blued_events = (
    event_alignment_audit[
        event_alignment_audit[
            "inside_signal_coverage"
        ]
    ]
    .drop_duplicates(
        subset=[
            "dataset",
            "event_timestamp",
            "event_code",
            "phase",
        ],
        keep="first"
    )
    .sort_values("event_timestamp")
    .reset_index(drop=True)
)

canonical_blued_events["canonical_event_id"] = (
    np.arange(
        len(canonical_blued_events),
        dtype=np.int64
    )
)

print(
    f"\nCanonical event records retained: "
    f"{len(canonical_blued_events):,}"
)

outside_coverage_events.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_outside_coverage_events.csv",
    index=False
)

canonical_blued_events.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_canonical_event_annotations.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 18 — CANONICAL GLOBAL BLUED EVENT UNION
# =========================================================

all_event_records = []

for _, file_record in event_files_df.iterrows():
    event_path = Path(file_record["path"])

    frame = pd.read_csv(
        event_path,
        encoding="utf-8-sig"
    )

    frame.columns = [
        str(column).strip()
        for column in frame.columns
    ]

    required_columns = {
        "Timestamp",
        "Label",
        "Phase",
    }

    missing_columns = (
        required_columns - set(frame.columns)
    )

    if missing_columns:
        raise ValueError(
            f"Missing columns in {event_path}: "
            f"{missing_columns}"
        )

    frame["event_timestamp"] = pd.to_datetime(
        frame["Timestamp"],
        errors="coerce"
    )

    frame["event_code"] = pd.to_numeric(
        frame["Label"],
        errors="coerce"
    ).astype("Int64")

    frame["phase"] = (
        frame["Phase"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    frame = frame[
        frame["event_timestamp"].notna()
        & frame["event_code"].notna()
        & frame["phase"].isin(["A", "B"])
    ].copy()

    frame["source_dataset"] = file_record["dataset"]
    frame["source_list"] = file_record["phase"]
    frame["source_file"] = str(event_path)

    all_event_records.append(
        frame[
            [
                "event_timestamp",
                "event_code",
                "phase",
                "source_dataset",
                "source_list",
                "source_file",
            ]
        ]
    )


all_blued_event_records = pd.concat(
    all_event_records,
    ignore_index=True
)

print("Raw annotation records from all 48 files")
print("=" * 100)
print(f"Records: {len(all_blued_event_records):,}")


# ---------------------------------------------------------
# Deduplicate globally.
#
# Dataset-folder identity is deliberately excluded because
# events near file boundaries may appear in neighboring
# annotation lists.
# ---------------------------------------------------------
event_key_columns = [
    "event_timestamp",
    "event_code",
    "phase",
]

canonical_event_union = (
    all_blued_event_records
    .groupby(
        event_key_columns,
        as_index=False,
        dropna=False
    )
    .agg(
        source_datasets=(
            "source_dataset",
            lambda values: ", ".join(
                sorted(set(values))
            )
        ),
        source_lists=(
            "source_list",
            lambda values: ", ".join(
                sorted(set(values))
            )
        ),
        source_files=(
            "source_file",
            lambda values: " | ".join(
                sorted(set(values))
            )
        ),
        source_record_count=(
            "source_file",
            "size"
        ),
    )
    .sort_values("event_timestamp")
    .reset_index(drop=True)
)

canonical_event_union["event_id"] = np.arange(
    len(canonical_event_union),
    dtype=np.int64
)


print("\nCanonical BLUED event union")
print("=" * 100)
print(
    f"Unique events       : "
    f"{len(canonical_event_union):,}"
)
print(
    f"Phase A events      : "
    f"{(canonical_event_union['phase'] == 'A').sum():,}"
)
print(
    f"Phase B events      : "
    f"{(canonical_event_union['phase'] == 'B').sum():,}"
)
print(
    f"Unique event codes  : "
    f"{canonical_event_union['event_code'].nunique():,}"
)
print(
    f"First event         : "
    f"{canonical_event_union['event_timestamp'].min()}"
)
print(
    f"Last event          : "
    f"{canonical_event_union['event_timestamp'].max()}"
)


print("\nEvents appearing in only one annotation source")
print("-" * 100)

single_source_events = canonical_event_union[
    canonical_event_union[
        "source_record_count"
    ] == 1
].copy()

display(
    single_source_events[
        [
            "event_timestamp",
            "event_code",
            "phase",
            "source_datasets",
            "source_lists",
        ]
    ]
)


canonical_event_union.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_canonical_global_event_union.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 19 — TWO-BLOCK BLUED SEGMENT MANIFEST
# =========================================================

blued_segment_manifest = segment_index.copy()

blued_segment_manifest["file_number"] = pd.to_numeric(
    blued_segment_manifest[
        "filename"
    ].str.extract(
        r"ivdata_(\d+)",
        expand=False
    ),
    errors="coerce"
).astype("Int64")


if blued_segment_manifest[
    "file_number"
].isna().any():
    invalid_files = blued_segment_manifest.loc[
        blued_segment_manifest[
            "file_number"
        ].isna(),
        "relative_path"
    ]

    raise ValueError(
        "Unable to extract file numbers from:\n"
        + "\n".join(invalid_files.head(20))
    )


blued_segment_manifest = (
    blued_segment_manifest
    .sort_values("file_number")
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Verify complete file-number sequence
# ---------------------------------------------------------
expected_file_numbers = np.arange(
    1,
    6431,
    dtype=np.int64
)

actual_file_numbers = (
    blued_segment_manifest[
        "file_number"
    ]
    .to_numpy(dtype=np.int64)
)

print("BLUED file-number audit")
print("=" * 100)
print(f"Minimum file number : {actual_file_numbers.min()}")
print(f"Maximum file number : {actual_file_numbers.max()}")
print(f"Unique file numbers : {len(np.unique(actual_file_numbers)):,}")
print(
    f"Exact 1–6430 sequence: "
    f"{np.array_equal(actual_file_numbers, expected_file_numbers)}"
)

if not np.array_equal(
    actual_file_numbers,
    expected_file_numbers
):
    missing_numbers = sorted(
        set(expected_file_numbers)
        - set(actual_file_numbers)
    )

    duplicate_numbers = (
        blued_segment_manifest.loc[
            blued_segment_manifest[
                "file_number"
            ].duplicated(keep=False),
            "file_number"
        ]
        .astype(int)
        .tolist()
    )

    raise ValueError(
        f"Missing file numbers: {missing_numbers[:50]}\n"
        f"Duplicate file numbers: {duplicate_numbers[:50]}"
    )


# ---------------------------------------------------------
# Two recording blocks
# ---------------------------------------------------------
blued_segment_manifest["recording_block"] = np.where(
    blued_segment_manifest["file_number"] <= 5055,
    "Block 1",
    "Block 2"
)

block_segment_summary = (
    blued_segment_manifest
    .groupby("recording_block")
    .agg(
        segments=("file_number", "size"),
        first_file=("file_number", "min"),
        last_file=("file_number", "max"),
        elapsed_start=("elapsed_start", "min"),
        elapsed_end=("elapsed_end", "max"),
        total_rows=("rows", "sum"),
    )
    .reset_index()
)

print("\nTwo-block segment summary")
print("=" * 110)
display(block_segment_summary)


# Expected:
# Block 1 = files 1–5055
# Block 2 = files 5056–6430
expected_block_counts = {
    "Block 1": 5055,
    "Block 2": 1375,
}

actual_block_counts = (
    blued_segment_manifest[
        "recording_block"
    ].value_counts().to_dict()
)

if actual_block_counts != expected_block_counts:
    raise ValueError(
        "Unexpected block counts:\n"
        f"{actual_block_counts}"
    )


blued_segment_manifest.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_two_block_segment_manifest.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 20 — RECONSTRUCT TWO CONTINUOUS BLUED SIGNALS
# =========================================================

# Exact midpoint of the valid Block 2 origin interval
BLOCK_2_ORIGIN = origin_summary.loc[
    origin_summary["block"] == "Block 2",
    "interval_midpoint"
].iloc[0]

# Last elapsed timestamp before the reset at file 5056
BLOCK_1_LAST_ELAPSED = (
    blued_segment_manifest.loc[
        blued_segment_manifest[
            "recording_block"
        ] == "Block 1",
        "elapsed_end"
    ].max()
)

# Derive Block 1 origin so that its final timestamp aligns
# with the beginning of Block 2.
BLOCK_1_ORIGIN = (
    BLOCK_2_ORIGIN
    - BLOCK_1_LAST_ELAPSED
)

print("Selected exact recording origins")
print("=" * 100)
print(f"Block 1 origin       : {BLOCK_1_ORIGIN}")
print(f"Block 1 final elapsed: {BLOCK_1_LAST_ELAPSED}")
print(f"Block 2 origin       : {BLOCK_2_ORIGIN}")

print(
    "\nBlock transition check:"
)
print(
    BLOCK_1_ORIGIN + BLOCK_1_LAST_ELAPSED
)


# ---------------------------------------------------------
# Verify that Block 1 origin remains inside its valid
# origin interval
# ---------------------------------------------------------
block_1_lower = origin_summary.loc[
    origin_summary["block"] == "Block 1",
    "lower_bound"
].iloc[0]

block_1_upper = origin_summary.loc[
    origin_summary["block"] == "Block 1",
    "upper_bound"
].iloc[0]

if not (
    block_1_lower
    <= BLOCK_1_ORIGIN
    <= block_1_upper
):
    raise ValueError(
        "Derived Block 1 origin falls outside its "
        "valid alignment interval."
    )


origin_map = {
    "Block 1": BLOCK_1_ORIGIN,
    "Block 2": BLOCK_2_ORIGIN,
}


signal_parts = []
signal_read_errors = []

print("\nReconstructing BLUED signal...")
print("=" * 100)

for row_number, row in blued_segment_manifest.iterrows():
    segment_path = (
        BLUED_PH2_ROOT
        / row["relative_path"]
    )

    try:
        frame = pd.read_csv(
            segment_path,
            usecols=[
                "timestamp",
                "mean_power",
                "max_power",
                "std_power",
            ],
            low_memory=False
        )

        frame["elapsed_timestamp"] = pd.to_timedelta(
            frame["timestamp"],
            errors="coerce"
        )

        for column in [
            "mean_power",
            "max_power",
            "std_power",
        ]:
            frame[column] = pd.to_numeric(
                frame[column],
                errors="coerce"
            )

        frame = frame.dropna(
            subset=[
                "elapsed_timestamp",
                "mean_power",
                "max_power",
                "std_power",
            ]
        )

        recording_block = row["recording_block"]

        frame["absolute_timestamp"] = (
            origin_map[recording_block]
            + frame["elapsed_timestamp"]
        )

        frame["recording_block"] = recording_block
        frame["file_number"] = int(row["file_number"])

        signal_parts.append(
            frame[
                [
                    "absolute_timestamp",
                    "recording_block",
                    "file_number",
                    "mean_power",
                    "max_power",
                    "std_power",
                ]
            ]
        )

    except Exception as error:
        signal_read_errors.append(
            {
                "relative_path": row["relative_path"],
                "error_type": type(error).__name__,
                "error_message": str(error),
            }
        )

    processed_count = row_number + 1

    if (
        processed_count % 500 == 0
        or processed_count
        == len(blued_segment_manifest)
    ):
        print(
            f"Read {processed_count:,}/"
            f"{len(blued_segment_manifest):,} segments"
        )


if signal_read_errors:
    error_df = pd.DataFrame(signal_read_errors)

    display(error_df)

    raise RuntimeError(
        f"{len(error_df)} signal segments failed."
    )


raw_blued_signal = pd.concat(
    signal_parts,
    ignore_index=True
)


# ---------------------------------------------------------
# Standardize to one-second timestamps
# ---------------------------------------------------------
raw_blued_signal["timestamp_1s"] = (
    raw_blued_signal[
        "absolute_timestamp"
    ].dt.round("1s")
)

blued_signal_1s = (
    raw_blued_signal
    .groupby(
        [
            "recording_block",
            "timestamp_1s",
        ],
        as_index=False
    )
    .agg(
        mean_power=("mean_power", "mean"),
        max_power=("max_power", "max"),
        std_power=("std_power", "mean"),
        contributing_rows=("file_number", "size"),
        first_file=("file_number", "min"),
        last_file=("file_number", "max"),
    )
    .rename(
        columns={
            "timestamp_1s": "timestamp"
        }
    )
    .sort_values(
        [
            "recording_block",
            "timestamp",
        ]
    )
    .reset_index(drop=True)
)


print("\nReconstructed one-second signal")
print("=" * 110)
print(f"Rows             : {len(blued_signal_1s):,}")
print(
    f"Finite values    : "
    f"{np.isfinite(blued_signal_1s[['mean_power', 'max_power', 'std_power']]).all().all()}"
)

signal_block_summary = (
    blued_signal_1s
    .groupby("recording_block")
    .agg(
        rows=("timestamp", "size"),
        start=("timestamp", "min"),
        end=("timestamp", "max"),
        mean_power_min=("mean_power", "min"),
        mean_power_max=("mean_power", "max"),
    )
    .reset_index()
)

display(signal_block_summary)


# ---------------------------------------------------------
# Gap audit
# ---------------------------------------------------------
gap_records = []

for block_name, block_df in blued_signal_1s.groupby(
    "recording_block"
):
    differences = (
        block_df["timestamp"]
        .sort_values()
        .diff()
        .dt.total_seconds()
        .dropna()
    )

    gap_records.append(
        {
            "recording_block": block_name,
            "median_interval_seconds": (
                differences.median()
            ),
            "maximum_interval_seconds": (
                differences.max()
            ),
            "intervals_over_1_second": int(
                (differences > 1).sum()
            ),
            "intervals_over_5_seconds": int(
                (differences > 5).sum()
            ),
            "intervals_over_60_seconds": int(
                (differences > 60).sum()
            ),
        }
    )

gap_summary = pd.DataFrame(gap_records)

print("\nSignal-gap audit")
print("=" * 100)
display(gap_summary)


try:
    blued_signal_1s.to_parquet(
        COMMON_OUTPUT_DIR /
        "blued_two_block_signal_1s.parquet",
        index=False
    )

    print("\nSaved Parquet signal file.")

except Exception as error:
    print(
        "Parquet save unavailable; using compressed CSV."
    )
    print(repr(error))

    blued_signal_1s.to_csv(
        COMMON_OUTPUT_DIR /
        "blued_two_block_signal_1s.csv.gz",
        index=False,
        compression="gzip"
    )

In [ ]:
# =========================================================
# CELL 21 — GLOBAL EVENT-TO-SIGNAL ALIGNMENT
# =========================================================

BLOCK_TRANSITION = BLOCK_2_ORIGIN

aligned_blued_events = canonical_event_union.copy()

aligned_blued_events["recording_block"] = np.where(
    aligned_blued_events["event_timestamp"]
    < BLOCK_TRANSITION,
    "Block 1",
    "Block 2"
)


block_coverage_lookup = (
    signal_block_summary
    .set_index("recording_block")
)

aligned_blued_events["signal_start"] = (
    aligned_blued_events[
        "recording_block"
    ].map(block_coverage_lookup["start"])
)

aligned_blued_events["signal_end"] = (
    aligned_blued_events[
        "recording_block"
    ].map(block_coverage_lookup["end"])
)

aligned_blued_events["inside_signal_coverage"] = (
    (
        aligned_blued_events["event_timestamp"]
        >= aligned_blued_events["signal_start"]
    )
    &
    (
        aligned_blued_events["event_timestamp"]
        <= aligned_blued_events["signal_end"]
    )
)


print("Global BLUED event alignment")
print("=" * 100)
print(
    f"Total canonical events : "
    f"{len(aligned_blued_events):,}"
)
print(
    f"Inside signal coverage : "
    f"{aligned_blued_events['inside_signal_coverage'].sum():,}"
)
print(
    f"Outside coverage       : "
    f"{(~aligned_blued_events['inside_signal_coverage']).sum():,}"
)


outside_events = aligned_blued_events[
    ~aligned_blued_events[
        "inside_signal_coverage"
    ]
].copy()

if not outside_events.empty:
    print("\nEvents outside global coverage")
    print("-" * 110)

    display(
        outside_events[
            [
                "event_timestamp",
                "event_code",
                "phase",
                "source_datasets",
                "source_lists",
                "recording_block",
                "signal_start",
                "signal_end",
            ]
        ]
    )


canonical_aligned_events = (
    aligned_blued_events[
        aligned_blued_events[
            "inside_signal_coverage"
        ]
    ]
    .sort_values("event_timestamp")
    .reset_index(drop=True)
)

canonical_aligned_events[
    "aligned_event_id"
] = np.arange(
    len(canonical_aligned_events),
    dtype=np.int64
)


print("\nEvents by recording block")
print("-" * 100)

display(
    canonical_aligned_events[
        "recording_block"
    ].value_counts()
    .rename_axis("recording_block")
    .reset_index(name="events")
)


canonical_aligned_events.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_canonical_aligned_events.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 22 — BLUED EVENT-WINDOW DENSITY AUDIT
# =========================================================

WINDOW_CONFIGURATIONS = [
    {
        "window_minutes": 1,
        "stride_seconds": 30,
    },
    {
        "window_minutes": 5,
        "stride_seconds": 150,
    },
    {
        "window_minutes": 15,
        "stride_seconds": 450,
    },
    {
        "window_minutes": 30,
        "stride_seconds": 900,
    },
    {
        "window_minutes": 60,
        "stride_seconds": 1800,
    },
]


def generate_event_window_manifest(
    signal_df: pd.DataFrame,
    event_df: pd.DataFrame,
    window_minutes: int,
    stride_seconds: int,
    minimum_coverage_ratio: float = 0.95,
) -> pd.DataFrame:
    """
    Generate label-audit windows without exposing labels
    to any model.
    """

    window_records = []

    window_duration = pd.Timedelta(
        minutes=window_minutes
    )

    stride_duration = pd.Timedelta(
        seconds=stride_seconds
    )

    expected_points = int(
        window_duration.total_seconds()
    )

    for block_name, block_signal in signal_df.groupby(
        "recording_block"
    ):
        block_signal = (
            block_signal
            .sort_values("timestamp")
            .reset_index(drop=True)
        )

        signal_times_ns = (
            block_signal["timestamp"]
            .astype("int64")
            .to_numpy()
        )

        block_events = (
            event_df[
                event_df["recording_block"]
                == block_name
            ]
            .sort_values("event_timestamp")
        )

        event_times_ns = (
            block_events["event_timestamp"]
            .astype("int64")
            .to_numpy()
        )

        first_timestamp = (
            block_signal["timestamp"]
            .min()
            .ceil("min")
        )

        final_timestamp = (
            block_signal["timestamp"]
            .max()
        )

        window_start = first_timestamp

        while (
            window_start + window_duration
            <= final_timestamp
        ):
            window_end = (
                window_start + window_duration
            )

            start_ns = int(window_start.value)
            end_ns = int(window_end.value)

            signal_left = np.searchsorted(
                signal_times_ns,
                start_ns,
                side="left"
            )

            signal_right = np.searchsorted(
                signal_times_ns,
                end_ns,
                side="left"
            )

            observed_points = (
                signal_right - signal_left
            )

            coverage_ratio = (
                observed_points / expected_points
            )

            event_left = np.searchsorted(
                event_times_ns,
                start_ns,
                side="left"
            )

            event_right = np.searchsorted(
                event_times_ns,
                end_ns,
                side="left"
            )

            event_count = (
                event_right - event_left
            )

            window_records.append(
                {
                    "recording_block": block_name,
                    "window_start": window_start,
                    "window_end": window_end,
                    "window_minutes": window_minutes,
                    "stride_seconds": stride_seconds,
                    "observed_points": observed_points,
                    "expected_points": expected_points,
                    "coverage_ratio": coverage_ratio,
                    "coverage_acceptable": (
                        coverage_ratio
                        >= minimum_coverage_ratio
                    ),
                    "event_count": event_count,
                    "event_window_label": int(
                        event_count > 0
                    ),
                }
            )

            window_start += stride_duration

    return pd.DataFrame(window_records)


window_manifests = {}
window_density_records = []

for configuration in WINDOW_CONFIGURATIONS:
    window_minutes = configuration[
        "window_minutes"
    ]

    stride_seconds = configuration[
        "stride_seconds"
    ]

    print(
        f"Auditing {window_minutes}-minute windows "
        f"with {stride_seconds}-second stride..."
    )

    manifest = generate_event_window_manifest(
        signal_df=blued_signal_1s,
        event_df=canonical_aligned_events,
        window_minutes=window_minutes,
        stride_seconds=stride_seconds,
    )

    valid_manifest = manifest[
        manifest["coverage_acceptable"]
    ].copy()

    positive_windows = int(
        valid_manifest[
            "event_window_label"
        ].sum()
    )

    negative_windows = (
        len(valid_manifest) - positive_windows
    )

    window_density_records.append(
        {
            "window_minutes": window_minutes,
            "stride_seconds": stride_seconds,
            "generated_windows": len(manifest),
            "valid_windows": len(valid_manifest),
            "excluded_low_coverage": int(
                (~manifest["coverage_acceptable"]).sum()
            ),
            "event_windows": positive_windows,
            "non_event_windows": negative_windows,
            "event_window_ratio_percent": (
                100.0
                * positive_windows
                / len(valid_manifest)
                if len(valid_manifest)
                else np.nan
            ),
            "median_events_per_window": (
                valid_manifest[
                    "event_count"
                ].median()
                if len(valid_manifest)
                else np.nan
            ),
            "maximum_events_per_window": (
                valid_manifest[
                    "event_count"
                ].max()
                if len(valid_manifest)
                else np.nan
            ),
        }
    )

    window_manifests[
        window_minutes
    ] = valid_manifest

    valid_manifest.to_csv(
        COMMON_OUTPUT_DIR /
        (
            f"blued_event_windows_"
            f"{window_minutes}min_"
            f"stride{stride_seconds}s.csv"
        ),
        index=False
    )


window_density_summary = pd.DataFrame(
    window_density_records
)

print("\nBLUED event-window density")
print("=" * 120)
display(window_density_summary)


window_density_summary.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_event_window_density_summary.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 23 — PURGED CHRONOLOGICAL BLUED SPLIT
# =========================================================

PRIMARY_WINDOW_MINUTES = 1
PRIMARY_STRIDE_SECONDS = 30

blued_window_manifest = (
    window_manifests[
        PRIMARY_WINDOW_MINUTES
    ]
    .copy()
    .sort_values(
        [
            "recording_block",
            "window_start",
        ]
    )
    .reset_index(drop=True)
)

print("Primary BLUED event-window configuration")
print("=" * 100)
print(
    f"Window length : "
    f"{PRIMARY_WINDOW_MINUTES} minute"
)
print(
    f"Stride        : "
    f"{PRIMARY_STRIDE_SECONDS} seconds"
)
print(
    f"Valid windows : "
    f"{len(blued_window_manifest):,}"
)

if not (
    blued_window_manifest[
        "stride_seconds"
    ] == PRIMARY_STRIDE_SECONDS
).all():
    raise ValueError(
        "Unexpected stride values in the selected manifest."
    )


split_records = []
split_boundary_records = []

for block_name, block_windows in (
    blued_window_manifest
    .groupby("recording_block")
):
    block_windows = (
        block_windows
        .sort_values("window_start")
        .copy()
    )

    block_start = (
        block_windows["window_start"].min()
    )

    block_end = (
        block_windows["window_end"].max()
    )

    block_duration = block_end - block_start

    train_boundary = (
        block_start
        + block_duration * 0.60
    )

    validation_boundary = (
        block_start
        + block_duration * 0.80
    )

    block_windows["split"] = "excluded_boundary"

    # Entire window must be contained in the training period.
    train_mask = (
        block_windows["window_start"]
        >= block_start
    ) & (
        block_windows["window_end"]
        <= train_boundary
    )

    # Entire window must be contained in validation.
    validation_mask = (
        block_windows["window_start"]
        >= train_boundary
    ) & (
        block_windows["window_end"]
        <= validation_boundary
    )

    # Entire window must be contained in test.
    test_mask = (
        block_windows["window_start"]
        >= validation_boundary
    ) & (
        block_windows["window_end"]
        <= block_end
    )

    block_windows.loc[
        train_mask,
        "split"
    ] = "train"

    block_windows.loc[
        validation_mask,
        "split"
    ] = "validation"

    block_windows.loc[
        test_mask,
        "split"
    ] = "test"

    split_records.append(block_windows)

    split_boundary_records.append(
        {
            "recording_block": block_name,
            "block_start": block_start,
            "training_end": train_boundary,
            "validation_end": validation_boundary,
            "block_end": block_end,
            "total_duration_hours": (
                block_duration.total_seconds()
                / 3600
            ),
        }
    )


blued_window_manifest = (
    pd.concat(
        split_records,
        ignore_index=True
    )
    .sort_values(
        [
            "recording_block",
            "window_start",
        ]
    )
    .reset_index(drop=True)
)

blued_window_manifest["window_id"] = np.arange(
    len(blued_window_manifest),
    dtype=np.int64
)

split_boundaries = pd.DataFrame(
    split_boundary_records
)


print("\nSplit boundaries")
print("=" * 120)
display(split_boundaries)


split_summary = (
    blued_window_manifest
    .groupby(
        [
            "recording_block",
            "split",
        ]
    )
    .agg(
        windows=("window_id", "size"),
        event_windows=(
            "event_window_label",
            "sum"
        ),
        first_window=("window_start", "min"),
        last_window=("window_end", "max"),
    )
    .reset_index()
)

split_summary["non_event_windows"] = (
    split_summary["windows"]
    - split_summary["event_windows"]
)

split_summary[
    "event_ratio_percent"
] = (
    100.0
    * split_summary["event_windows"]
    / split_summary["windows"]
)


print("\nBLUED split summary")
print("=" * 130)
display(split_summary)


# ---------------------------------------------------------
# Remove boundary-crossing windows
# ---------------------------------------------------------
excluded_windows = blued_window_manifest[
    blued_window_manifest[
        "split"
    ] == "excluded_boundary"
].copy()

model_window_manifest = (
    blued_window_manifest[
        blued_window_manifest[
            "split"
        ].isin(
            [
                "train",
                "validation",
                "test",
            ]
        )
    ]
    .copy()
    .reset_index(drop=True)
)

model_window_manifest[
    "model_window_id"
] = np.arange(
    len(model_window_manifest),
    dtype=np.int64
)


print("\nFinal modeling manifest")
print("-" * 100)
print(
    f"Retained windows : "
    f"{len(model_window_manifest):,}"
)
print(
    f"Boundary windows excluded: "
    f"{len(excluded_windows):,}"
)


# ---------------------------------------------------------
# Raw-observation overlap audit
# ---------------------------------------------------------
for block_name, block_df in (
    model_window_manifest
    .groupby("recording_block")
):
    train_block = block_df[
        block_df["split"] == "train"
    ]

    validation_block = block_df[
        block_df["split"] == "validation"
    ]

    test_block = block_df[
        block_df["split"] == "test"
    ]

    if (
        not train_block.empty
        and not validation_block.empty
    ):
        assert (
            train_block["window_end"].max()
            <= validation_block[
                "window_start"
            ].min()
        )

    if (
        not validation_block.empty
        and not test_block.empty
    ):
        assert (
            validation_block[
                "window_end"
            ].max()
            <= test_block[
                "window_start"
            ].min()
        )


print(
    "\nLeakage audit passed: no overlapping raw time "
    "interval is shared across splits."
)


model_window_manifest.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_1min_stride30s_model_manifest.csv",
    index=False
)

excluded_windows.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_split_boundary_excluded_windows.csv",
    index=False
)

split_summary.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_1min_split_summary.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 24 — BLUED SEQUENCES AND FEATURE EXTRACTION
# =========================================================

from scipy.signal import find_peaks
from scipy.stats import skew, kurtosis

SIGNAL_CHANNELS = [
    "mean_power",
    "max_power",
    "std_power",
]

SEQUENCE_LENGTH = 60

BASE_FEATURE_NAMES = [
    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var",
]


def calculate_signal_features(
    values: np.ndarray,
    observed_points: int,
) -> dict:
    """Calculate the common 23-feature representation."""

    values = np.asarray(
        values,
        dtype=np.float64
    )

    differences = np.diff(values)

    q25 = float(
        np.percentile(values, 25)
    )

    q75 = float(
        np.percentile(values, 75)
    )

    median_value = float(
        np.median(values)
    )

    skew_value = float(
        skew(
            values,
            bias=False
        )
    )

    kurtosis_value = float(
        kurtosis(
            values,
            fisher=True,
            bias=False
        )
    )

    if not np.isfinite(skew_value):
        skew_value = 0.0

    if not np.isfinite(kurtosis_value):
        kurtosis_value = 0.0

    time_index = np.arange(
        len(values),
        dtype=np.float64
    )

    slope_value = float(
        np.polyfit(
            time_index,
            values,
            1
        )[0]
    )

    peak_indices, _ = find_peaks(values)

    return {
        # One-second sampling; result is watt-hours.
        "energy": float(
            np.sum(values) / 3600.0
        ),
        "first_last_diff": float(
            values[-1] - values[0]
        ),
        "iqr": float(q75 - q25),
        "kurtosis": kurtosis_value,
        "mad": float(
            np.mean(
                np.abs(
                    values - median_value
                )
            )
        ),
        "max": float(np.max(values)),
        "max_abs_diff": float(
            np.max(np.abs(differences))
            if len(differences)
            else 0.0
        ),
        "mean": float(np.mean(values)),
        "mean_abs_diff": float(
            np.mean(np.abs(differences))
            if len(differences)
            else 0.0
        ),
        "median": median_value,
        "min": float(np.min(values)),
        "n_points": int(observed_points),
        "peak_count": int(len(peak_indices)),
        "q25": q25,
        "q75": q75,
        "range": float(
            np.max(values) - np.min(values)
        ),
        "rms": float(
            np.sqrt(
                np.mean(
                    np.square(values)
                )
            )
        ),
        "skew": skew_value,
        "slope": slope_value,
        "std": float(
            np.std(
                values,
                ddof=0
            )
        ),
        "std_diff": float(
            np.std(
                differences,
                ddof=0
            )
            if len(differences)
            else 0.0
        ),
        "sum": float(np.sum(values)),
        "var": float(
            np.var(
                values,
                ddof=0
            )
        ),
    }


# ---------------------------------------------------------
# Create complete one-second arrays for fast slicing
# ---------------------------------------------------------
block_signal_arrays = {}

for block_name, block_df in (
    blued_signal_1s
    .groupby("recording_block")
):
    block_df = (
        block_df
        .sort_values("timestamp")
        .set_index("timestamp")
    )

    complete_index = pd.date_range(
        start=block_df.index.min(),
        end=block_df.index.max(),
        freq="1s"
    )

    complete_frame = block_df[
        SIGNAL_CHANNELS
    ].reindex(complete_index)

    block_signal_arrays[block_name] = {
        "start": complete_index[0],
        "values": complete_frame[
            SIGNAL_CHANNELS
        ].to_numpy(
            dtype=np.float64
        ),
    }


sequence_list = []
feature_records = []
alignment_records = []

print("Constructing BLUED windows...")
print("=" * 100)

for row_number, row in (
    model_window_manifest.iterrows()
):
    block_name = row["recording_block"]

    block_information = (
        block_signal_arrays[block_name]
    )

    block_start = block_information["start"]
    block_values = block_information["values"]

    start_offset = int(
        (
            row["window_start"]
            - block_start
        ).total_seconds()
    )

    end_offset = (
        start_offset + SEQUENCE_LENGTH
    )

    window_values = block_values[
        start_offset:end_offset
    ].copy()

    if window_values.shape != (
        SEQUENCE_LENGTH,
        len(SIGNAL_CHANNELS),
    ):
        raise ValueError(
            f"Unexpected sequence shape for window "
            f"{row['model_window_id']}: "
            f"{window_values.shape}"
        )

    observed_mask = np.isfinite(
        window_values
    ).all(axis=1)

    observed_points = int(
        observed_mask.sum()
    )

    expected_observed = int(
        row["observed_points"]
    )

    alignment_records.append(
        {
            "model_window_id": (
                row["model_window_id"]
            ),
            "expected_observed_points": (
                expected_observed
            ),
            "extracted_observed_points": (
                observed_points
            ),
            "exact_match": (
                observed_points
                == expected_observed
            ),
        }
    )

    # At most a small number of timestamps can be missing
    # because low-coverage windows were already excluded.
    if observed_points < (
        SEQUENCE_LENGTH * 0.95
    ):
        raise ValueError(
            f"Window {row['model_window_id']} has only "
            f"{observed_points} observed points."
        )

    # Interpolate missing values independently inside the
    # current window. No information is taken from another
    # split or neighboring window.
    for channel_index in range(
        len(SIGNAL_CHANNELS)
    ):
        channel_values = window_values[
            :,
            channel_index
        ]

        finite_mask = np.isfinite(
            channel_values
        )

        if not finite_mask.any():
            raise ValueError(
                "A complete signal channel is missing in "
                f"window {row['model_window_id']}."
            )

        if not finite_mask.all():
            sample_positions = np.arange(
                SEQUENCE_LENGTH
            )

            channel_values[
                ~finite_mask
            ] = np.interp(
                sample_positions[
                    ~finite_mask
                ],
                sample_positions[
                    finite_mask
                ],
                channel_values[
                    finite_mask
                ],
            )

            window_values[
                :,
                channel_index
            ] = channel_values

    sequence_list.append(
        window_values.astype(np.float32)
    )

    feature_record = {
        "model_window_id": (
            row["model_window_id"]
        )
    }

    for channel_index, channel_name in enumerate(
        SIGNAL_CHANNELS
    ):
        channel_features = (
            calculate_signal_features(
                window_values[
                    :,
                    channel_index
                ],
                observed_points=observed_points,
            )
        )

        for feature_name, feature_value in (
            channel_features.items()
        ):
            feature_record[
                f"{channel_name}__{feature_name}"
            ] = feature_value

    feature_records.append(feature_record)

    completed = row_number + 1

    if (
        completed % 2000 == 0
        or completed
        == len(model_window_manifest)
    ):
        print(
            f"Constructed {completed:,}/"
            f"{len(model_window_manifest):,}"
        )


blued_sequences = np.stack(
    sequence_list,
    axis=0
)

blued_features = pd.DataFrame(
    feature_records
)

sequence_alignment_audit = pd.DataFrame(
    alignment_records
)


print("\nBLUED model inputs")
print("=" * 100)
print(
    f"Sequence shape : "
    f"{blued_sequences.shape}"
)
print(
    f"Feature shape  : "
    f"{blued_features.shape}"
)
print(
    f"Sequence finite: "
    f"{np.isfinite(blued_sequences).all()}"
)

feature_columns = [
    column
    for column in blued_features.columns
    if column != "model_window_id"
]

print(
    f"Feature columns: "
    f"{len(feature_columns)}"
)

if len(feature_columns) != 69:
    raise ValueError(
        f"Expected 69 features, found "
        f"{len(feature_columns)}."
    )

if not np.isfinite(
    blued_features[
        feature_columns
    ].to_numpy(
        dtype=np.float64
    )
).all():
    raise ValueError(
        "Non-finite feature values were detected."
    )


alignment_mismatches = (
    ~sequence_alignment_audit[
        "exact_match"
    ]
).sum()

print(
    f"Observed-point mismatches: "
    f"{alignment_mismatches:,}"
)

if alignment_mismatches:
    display(
        sequence_alignment_audit[
            ~sequence_alignment_audit[
                "exact_match"
            ]
        ].head(30)
    )

    raise ValueError(
        "Window-observation alignment mismatches detected."
    )


np.save(
    COMMON_OUTPUT_DIR /
    "blued_sequences_1min_3channel.npy",
    blued_sequences
)

blued_features.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_features_1min_69features.csv",
    index=False
)

sequence_alignment_audit.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_sequence_alignment_audit.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 25 — TRAINING-ONLY NORMALIZATION
# =========================================================

feature_matrix = (
    blued_features[
        feature_columns
    ]
    .to_numpy(
        dtype=np.float64
    )
)

split_values = (
    model_window_manifest[
        "split"
    ].to_numpy()
)

event_labels = (
    model_window_manifest[
        "event_window_label"
    ]
    .to_numpy(
        dtype=np.int8
    )
)

train_indices = np.flatnonzero(
    split_values == "train"
)

validation_indices = np.flatnonzero(
    split_values == "validation"
)

test_indices = np.flatnonzero(
    split_values == "test"
)


# ---------------------------------------------------------
# Feature normalization
# ---------------------------------------------------------
feature_training_mean = (
    feature_matrix[
        train_indices
    ].mean(axis=0)
)

feature_training_std = (
    feature_matrix[
        train_indices
    ].std(axis=0)
)

constant_feature_mask = (
    feature_training_std < 1e-8
)

feature_training_std[
    constant_feature_mask
] = 1.0

normalized_feature_matrix = (
    feature_matrix
    - feature_training_mean
) / feature_training_std


X_train = normalized_feature_matrix[
    train_indices
].astype(np.float32)

X_validation = normalized_feature_matrix[
    validation_indices
].astype(np.float32)

X_test = normalized_feature_matrix[
    test_indices
].astype(np.float32)


# ---------------------------------------------------------
# Three-channel sequence normalization
# ---------------------------------------------------------
training_sequence_values = (
    blued_sequences[
        train_indices
    ]
)

sequence_training_mean = (
    training_sequence_values
    .mean(
        axis=(0, 1),
        keepdims=True
    )
)

sequence_training_std = (
    training_sequence_values
    .std(
        axis=(0, 1),
        keepdims=True
    )
)

sequence_training_std[
    sequence_training_std < 1e-8
] = 1.0

normalized_sequences = (
    blued_sequences
    - sequence_training_mean
) / sequence_training_std


X_lstm_train = normalized_sequences[
    train_indices
].astype(np.float32)

X_lstm_validation = normalized_sequences[
    validation_indices
].astype(np.float32)

X_lstm_test = normalized_sequences[
    test_indices
].astype(np.float32)


# Labels remain separate and are not model inputs.
y_train_event = event_labels[
    train_indices
]

y_validation_event = event_labels[
    validation_indices
]

y_test_event = event_labels[
    test_indices
]


print("BLUED normalized model matrices")
print("=" * 100)
print(f"Feature train      : {X_train.shape}")
print(f"Feature validation : {X_validation.shape}")
print(f"Feature test       : {X_test.shape}")
print(f"LSTM train         : {X_lstm_train.shape}")
print(f"LSTM validation    : {X_lstm_validation.shape}")
print(f"LSTM test          : {X_lstm_test.shape}")


print("\nFinite-value audit")
print("-" * 100)
print(
    "Feature train finite      :",
    np.isfinite(X_train).all()
)
print(
    "Feature validation finite :",
    np.isfinite(X_validation).all()
)
print(
    "Feature test finite       :",
    np.isfinite(X_test).all()
)
print(
    "LSTM train finite         :",
    np.isfinite(X_lstm_train).all()
)
print(
    "LSTM validation finite    :",
    np.isfinite(X_lstm_validation).all()
)
print(
    "LSTM test finite          :",
    np.isfinite(X_lstm_test).all()
)


# Evaluation-only label audit
label_split_audit = pd.DataFrame(
    [
        {
            "split": "train",
            "windows": len(train_indices),
            "event_windows": int(
                y_train_event.sum()
            ),
        },
        {
            "split": "validation",
            "windows": len(validation_indices),
            "event_windows": int(
                y_validation_event.sum()
            ),
        },
        {
            "split": "test",
            "windows": len(test_indices),
            "event_windows": int(
                y_test_event.sum()
            ),
        },
    ]
)

label_split_audit[
    "non_event_windows"
] = (
    label_split_audit["windows"]
    - label_split_audit["event_windows"]
)

label_split_audit[
    "event_ratio_percent"
] = (
    100.0
    * label_split_audit["event_windows"]
    / label_split_audit["windows"]
)

print("\nEvaluation-only event-label distribution")
print("=" * 100)
display(label_split_audit)


# ---------------------------------------------------------
# Save normalized matrices and normalization parameters
# ---------------------------------------------------------
np.savez_compressed(
    COMMON_OUTPUT_DIR /
    "blued_normalized_model_inputs.npz",
    X_train=X_train,
    X_validation=X_validation,
    X_test=X_test,
    X_lstm_train=X_lstm_train,
    X_lstm_validation=X_lstm_validation,
    X_lstm_test=X_lstm_test,
    train_indices=train_indices,
    validation_indices=validation_indices,
    test_indices=test_indices,
    y_train_event=y_train_event,
    y_validation_event=y_validation_event,
    y_test_event=y_test_event,
)

feature_normalization_table = pd.DataFrame(
    {
        "feature": feature_columns,
        "training_mean": feature_training_mean,
        "training_std": feature_training_std,
        "constant_in_training": (
            constant_feature_mask
        ),
    }
)

feature_normalization_table.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_feature_normalization.csv",
    index=False
)

sequence_normalization_table = pd.DataFrame(
    {
        "channel": SIGNAL_CHANNELS,
        "training_mean": (
            sequence_training_mean
            .reshape(-1)
        ),
        "training_std": (
            sequence_training_std
            .reshape(-1)
        ),
    }
)

sequence_normalization_table.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_sequence_normalization.csv",
    index=False
)

label_split_audit.to_csv(
    COMMON_OUTPUT_DIR /
    "blued_event_label_split_audit.csv",
    index=False
)

In [ ]:
# =========================================================
# CELL 26 — LABEL-BLIND MODELING AND EVALUATION UTILITIES
# =========================================================

import joblib
import tensorflow as tf

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

THRESHOLD_PERCENTILE = 95.0

tf.random.set_seed(RANDOM_STATE)

FEATURE_MODEL_OUTPUT_DIR = (
    EXPERIMENT_ROOT /
    "controlled_feature_models"
)

DENSE_AE_OUTPUT_DIR = (
    EXPERIMENT_ROOT /
    "dense_autoencoder"
)

LSTM_AE_OUTPUT_DIR = (
    EXPERIMENT_ROOT /
    "lstm_autoencoder"
)

FINAL_COMPARISON_DIR = (
    EXPERIMENT_ROOT /
    "final_five_model_comparison"
)

for folder in [
    FEATURE_MODEL_OUTPUT_DIR,
    DENSE_AE_OUTPUT_DIR,
    LSTM_AE_OUTPUT_DIR,
    FINAL_COMPARISON_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


def calculate_event_alignment_metrics(
    y_true: np.ndarray,
    anomaly_scores: np.ndarray,
    predictions: np.ndarray,
) -> dict:
    """
    Calculate event-alignment metrics.

    Labels are used only inside this evaluation function,
    after model fitting and threshold selection.
    """

    y_true = np.asarray(
        y_true,
        dtype=np.int8
    )

    anomaly_scores = np.asarray(
        anomaly_scores,
        dtype=np.float64
    )

    predictions = np.asarray(
        predictions,
        dtype=np.int8
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_true,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        predictions,
        zero_division=0
    )

    false_positive_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else np.nan
    )

    roc_auc = (
        roc_auc_score(
            y_true,
            anomaly_scores
        )
        if len(np.unique(y_true)) == 2
        else np.nan
    )

    pr_auc = (
        average_precision_score(
            y_true,
            anomaly_scores
        )
        if len(np.unique(y_true)) == 2
        else np.nan
    )

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "false_positive_rate": float(
            false_positive_rate
        ),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "flagged_windows": int(
            predictions.sum()
        ),
        "flagged_ratio_percent": float(
            100.0 * predictions.mean()
        ),
        "evaluation_windows": int(
            len(y_true)
        ),
        "event_windows": int(
            y_true.sum()
        ),
        "non_event_windows": int(
            len(y_true) - y_true.sum()
        ),
    }


def make_result_frame(
    model_name: str,
    split_name: str,
    metrics: dict,
    threshold: float,
    training_seconds: float,
    scoring_seconds: float,
    input_representation: str,
) -> pd.DataFrame:
    """Convert one result dictionary to a single-row frame."""

    result = {
        "model": model_name,
        "split": split_name,
        "input_representation": input_representation,
        "threshold_percentile": (
            THRESHOLD_PERCENTILE
        ),
        "threshold": float(threshold),
        "training_seconds": float(
            training_seconds
        ),
        "scoring_seconds": float(
            scoring_seconds
        ),
    }

    result.update(metrics)

    return pd.DataFrame([result])


print("BLUED five-model setup")
print("=" * 100)
print(f"Feature train shape : {X_train.shape}")
print(f"Feature val shape   : {X_validation.shape}")
print(f"Feature test shape  : {X_test.shape}")
print(f"LSTM train shape    : {X_lstm_train.shape}")
print(f"LSTM val shape      : {X_lstm_validation.shape}")
print(f"LSTM test shape     : {X_lstm_test.shape}")
print(
    f"Threshold protocol  : "
    f"{THRESHOLD_PERCENTILE}th percentile of "
    "validation anomaly scores"
)

controlled_results = {}

In [ ]:
# =========================================================
# CELL 27 — THREE LABEL-BLIND FEATURE MODELS
# =========================================================

def run_feature_anomaly_model(
    model_name: str,
    model,
) -> dict:
    """
    Fit an unsupervised feature model.

    No event labels are supplied during model fitting or
    validation-threshold selection.
    """

    safe_name = (
        model_name
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    model_output_dir = (
        FEATURE_MODEL_OUTPUT_DIR /
        safe_name
    )

    model_output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    print("\n" + "=" * 110)
    print(f"Running {model_name}")
    print("=" * 110)

    # -----------------------------------------------------
    # Unsupervised fitting — no labels
    # -----------------------------------------------------
    fit_start = time.perf_counter()

    model.fit(X_train)

    training_seconds = (
        time.perf_counter()
        - fit_start
    )

    # Higher values must mean more unusual.
    validation_score_start = (
        time.perf_counter()
    )

    validation_scores = (
        -model.score_samples(
            X_validation
        )
    )

    validation_scoring_seconds = (
        time.perf_counter()
        - validation_score_start
    )

    threshold = float(
        np.percentile(
            validation_scores,
            THRESHOLD_PERCENTILE
        )
    )

    validation_predictions = (
        validation_scores > threshold
    ).astype(np.int8)

    # Labels are revealed only here.
    validation_metrics = (
        calculate_event_alignment_metrics(
            y_validation_event,
            validation_scores,
            validation_predictions,
        )
    )

    test_score_start = time.perf_counter()

    test_scores = (
        -model.score_samples(
            X_test
        )
    )

    test_scoring_seconds = (
        time.perf_counter()
        - test_score_start
    )

    test_predictions = (
        test_scores > threshold
    ).astype(np.int8)

    # Labels are revealed only here.
    test_metrics = (
        calculate_event_alignment_metrics(
            y_test_event,
            test_scores,
            test_predictions,
        )
    )

    validation_result = make_result_frame(
        model_name=model_name,
        split_name="validation",
        metrics=validation_metrics,
        threshold=threshold,
        training_seconds=training_seconds,
        scoring_seconds=(
            validation_scoring_seconds
        ),
        input_representation=(
            "69 engineered statistical features"
        ),
    )

    test_result = make_result_frame(
        model_name=model_name,
        split_name="test",
        metrics=test_metrics,
        threshold=threshold,
        training_seconds=training_seconds,
        scoring_seconds=(
            test_scoring_seconds
        ),
        input_representation=(
            "69 engineered statistical features"
        ),
    )

    model_metrics = pd.concat(
        [
            validation_result,
            test_result,
        ],
        ignore_index=True
    )

    metadata_columns = [
        "model_window_id",
        "recording_block",
        "window_start",
        "window_end",
        "split",
        "event_count",
        "event_window_label",
    ]

    metadata_columns = [
        column
        for column in metadata_columns
        if column in model_window_manifest.columns
    ]

    test_metadata = (
        model_window_manifest.iloc[
            test_indices
        ][metadata_columns]
        .reset_index(drop=True)
        .copy()
    )

    test_metadata[
        "anomaly_score"
    ] = test_scores

    test_metadata[
        "threshold"
    ] = threshold

    test_metadata[
        "flagged"
    ] = test_predictions

    model_metrics.to_csv(
        model_output_dir /
        "metrics.csv",
        index=False
    )

    test_metadata.to_csv(
        model_output_dir /
        "test_window_scores.csv",
        index=False
    )

    joblib.dump(
        model,
        model_output_dir /
        "model.joblib"
    )

    print(
        f"Training time : "
        f"{training_seconds:.3f} seconds"
    )

    print(
        f"Threshold     : "
        f"{threshold:.6f}"
    )

    print("\nValidation metrics")
    display(validation_result)

    print("\nTest metrics")
    display(test_result)

    return {
        "model": model,
        "threshold": threshold,
        "validation_scores": (
            validation_scores
        ),
        "validation_predictions": (
            validation_predictions
        ),
        "test_scores": test_scores,
        "test_predictions": test_predictions,
        "metrics": model_metrics,
    }


# ---------------------------------------------------------
# Isolation Forest
# ---------------------------------------------------------
controlled_results[
    "Isolation Forest"
] = run_feature_anomaly_model(
    model_name="Isolation Forest",
    model=IsolationForest(
        n_estimators=300,
        contamination="auto",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
)


# ---------------------------------------------------------
# Local Outlier Factor
# ---------------------------------------------------------
controlled_results[
    "Local Outlier Factor"
] = run_feature_anomaly_model(
    model_name="Local Outlier Factor",
    model=LocalOutlierFactor(
        n_neighbors=35,
        novelty=True,
        contamination="auto",
        n_jobs=-1,
    ),
)


# ---------------------------------------------------------
# One-Class SVM
# ---------------------------------------------------------
controlled_results[
    "One-Class SVM"
] = run_feature_anomaly_model(
    model_name="One-Class SVM",
    model=OneClassSVM(
        kernel="rbf",
        gamma="scale",
        nu=0.05,
        cache_size=2048,
    ),
)

In [ ]:
# =========================================================
# CELL 28 — DENSE AUTOENCODER
# =========================================================

from tensorflow.keras import (
    Model,
    Input,
)

from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
)

tf.keras.backend.clear_session()
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

input_dimension = X_train.shape[1]

dense_input = Input(
    shape=(input_dimension,),
    name="feature_input"
)

encoded = Dense(
    48,
    activation="relu",
    name="encoder_48"
)(dense_input)

encoded = Dense(
    24,
    activation="relu",
    name="encoder_24"
)(encoded)

latent = Dense(
    8,
    activation="relu",
    name="latent_8"
)(encoded)

decoded = Dense(
    24,
    activation="relu",
    name="decoder_24"
)(latent)

decoded = Dense(
    48,
    activation="relu",
    name="decoder_48"
)(decoded)

dense_output = Dense(
    input_dimension,
    activation="linear",
    name="reconstruction"
)(decoded)

dense_autoencoder = Model(
    inputs=dense_input,
    outputs=dense_output,
    name="BLUED_Dense_Autoencoder"
)

dense_autoencoder.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),
    loss="mse"
)

dense_autoencoder.summary()


dense_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    ),
]


print("\nTraining dense Autoencoder...")
print("=" * 100)

dense_training_start = (
    time.perf_counter()
)

dense_history = dense_autoencoder.fit(
    X_train,
    X_train,
    validation_data=(
        X_validation,
        X_validation,
    ),
    epochs=150,
    batch_size=256,
    shuffle=True,
    callbacks=dense_callbacks,
    verbose=1,
)

dense_training_seconds = (
    time.perf_counter()
    - dense_training_start
)


# ---------------------------------------------------------
# Reconstruction anomaly scores
# ---------------------------------------------------------
dense_validation_score_start = (
    time.perf_counter()
)

dense_validation_reconstruction = (
    dense_autoencoder.predict(
        X_validation,
        batch_size=256,
        verbose=0,
    )
)

dense_validation_scores = np.mean(
    np.square(
        X_validation
        - dense_validation_reconstruction
    ),
    axis=1,
)

dense_validation_scoring_seconds = (
    time.perf_counter()
    - dense_validation_score_start
)

dense_threshold = float(
    np.percentile(
        dense_validation_scores,
        THRESHOLD_PERCENTILE
    )
)

dense_validation_predictions = (
    dense_validation_scores
    > dense_threshold
).astype(np.int8)

dense_validation_metrics = (
    calculate_event_alignment_metrics(
        y_validation_event,
        dense_validation_scores,
        dense_validation_predictions,
    )
)


dense_test_score_start = (
    time.perf_counter()
)

dense_test_reconstruction = (
    dense_autoencoder.predict(
        X_test,
        batch_size=256,
        verbose=0,
    )
)

dense_test_scores = np.mean(
    np.square(
        X_test
        - dense_test_reconstruction
    ),
    axis=1,
)

dense_test_scoring_seconds = (
    time.perf_counter()
    - dense_test_score_start
)

dense_test_predictions = (
    dense_test_scores
    > dense_threshold
).astype(np.int8)

dense_test_metrics = (
    calculate_event_alignment_metrics(
        y_test_event,
        dense_test_scores,
        dense_test_predictions,
    )
)


dense_validation_result = make_result_frame(
    model_name="Dense Autoencoder",
    split_name="validation",
    metrics=dense_validation_metrics,
    threshold=dense_threshold,
    training_seconds=dense_training_seconds,
    scoring_seconds=(
        dense_validation_scoring_seconds
    ),
    input_representation=(
        "69 engineered statistical features"
    ),
)

dense_test_result = make_result_frame(
    model_name="Dense Autoencoder",
    split_name="test",
    metrics=dense_test_metrics,
    threshold=dense_threshold,
    training_seconds=dense_training_seconds,
    scoring_seconds=(
        dense_test_scoring_seconds
    ),
    input_representation=(
        "69 engineered statistical features"
    ),
)

dense_metrics_df = pd.concat(
    [
        dense_validation_result,
        dense_test_result,
    ],
    ignore_index=True
)


dense_history_df = pd.DataFrame(
    dense_history.history
)

dense_history_df[
    "epoch"
] = np.arange(
    1,
    len(dense_history_df) + 1
)

dense_history_df.to_csv(
    DENSE_AE_OUTPUT_DIR /
    "training_history.csv",
    index=False
)

dense_metrics_df.to_csv(
    DENSE_AE_OUTPUT_DIR /
    "metrics.csv",
    index=False
)

dense_autoencoder.save(
    DENSE_AE_OUTPUT_DIR /
    "dense_autoencoder.keras"
)


dense_test_metadata = (
    model_window_manifest.iloc[
        test_indices
    ][
        [
            "model_window_id",
            "recording_block",
            "window_start",
            "window_end",
            "event_count",
            "event_window_label",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

dense_test_metadata[
    "anomaly_score"
] = dense_test_scores

dense_test_metadata[
    "threshold"
] = dense_threshold

dense_test_metadata[
    "flagged"
] = dense_test_predictions

dense_test_metadata.to_csv(
    DENSE_AE_OUTPUT_DIR /
    "test_window_scores.csv",
    index=False
)


controlled_results[
    "Dense Autoencoder"
] = {
    "model": dense_autoencoder,
    "threshold": dense_threshold,
    "validation_scores": (
        dense_validation_scores
    ),
    "validation_predictions": (
        dense_validation_predictions
    ),
    "test_scores": dense_test_scores,
    "test_predictions": (
        dense_test_predictions
    ),
    "metrics": dense_metrics_df,
}


print("\nDense Autoencoder completed")
print("=" * 100)
print(
    f"Training time: "
    f"{dense_training_seconds:.3f} seconds"
)
print(
    f"Threshold: "
    f"{dense_threshold:.6f}"
)

print("\nValidation result")
display(dense_validation_result)

print("\nTest result")
display(dense_test_result)

In [ ]:
# =========================================================
# CELL 29 — THREE-CHANNEL LSTM AUTOENCODER
# =========================================================

from tensorflow.keras.layers import (
    LSTM,
    RepeatVector,
    TimeDistributed,
)

tf.keras.backend.clear_session()
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

sequence_length = (
    X_lstm_train.shape[1]
)

sequence_channels = (
    X_lstm_train.shape[2]
)


lstm_input = Input(
    shape=(
        sequence_length,
        sequence_channels,
    ),
    name="sequence_input"
)

encoded_sequence = LSTM(
    32,
    return_sequences=True,
    name="encoder_lstm_32"
)(lstm_input)

latent_vector = LSTM(
    16,
    return_sequences=False,
    name="latent_lstm_16"
)(encoded_sequence)

repeated_vector = RepeatVector(
    sequence_length,
    name="repeat_latent"
)(latent_vector)

decoded_sequence = LSTM(
    16,
    return_sequences=True,
    name="decoder_lstm_16"
)(repeated_vector)

decoded_sequence = LSTM(
    32,
    return_sequences=True,
    name="decoder_lstm_32"
)(decoded_sequence)

lstm_output = TimeDistributed(
    Dense(
        sequence_channels,
        activation="linear"
    ),
    name="sequence_reconstruction"
)(decoded_sequence)

lstm_autoencoder = Model(
    inputs=lstm_input,
    outputs=lstm_output,
    name="BLUED_LSTM_Autoencoder"
)

lstm_autoencoder.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),
    loss="mse"
)

lstm_autoencoder.summary()


lstm_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    ),
]


print("\nTraining LSTM Autoencoder...")
print("=" * 100)

lstm_training_start = (
    time.perf_counter()
)

lstm_history = lstm_autoencoder.fit(
    X_lstm_train,
    X_lstm_train,
    validation_data=(
        X_lstm_validation,
        X_lstm_validation,
    ),
    epochs=100,
    batch_size=256,
    shuffle=True,
    callbacks=lstm_callbacks,
    verbose=1,
)

lstm_training_seconds = (
    time.perf_counter()
    - lstm_training_start
)


# ---------------------------------------------------------
# Validation reconstruction scores
# ---------------------------------------------------------
lstm_validation_score_start = (
    time.perf_counter()
)

lstm_validation_reconstruction = (
    lstm_autoencoder.predict(
        X_lstm_validation,
        batch_size=256,
        verbose=0,
    )
)

lstm_validation_scores = np.mean(
    np.square(
        X_lstm_validation
        - lstm_validation_reconstruction
    ),
    axis=(1, 2),
)

lstm_validation_scoring_seconds = (
    time.perf_counter()
    - lstm_validation_score_start
)

lstm_threshold = float(
    np.percentile(
        lstm_validation_scores,
        THRESHOLD_PERCENTILE
    )
)

lstm_validation_predictions = (
    lstm_validation_scores
    > lstm_threshold
).astype(np.int8)

lstm_validation_metrics = (
    calculate_event_alignment_metrics(
        y_validation_event,
        lstm_validation_scores,
        lstm_validation_predictions,
    )
)


# ---------------------------------------------------------
# Test reconstruction scores
# ---------------------------------------------------------
lstm_test_score_start = (
    time.perf_counter()
)

lstm_test_reconstruction = (
    lstm_autoencoder.predict(
        X_lstm_test,
        batch_size=256,
        verbose=0,
    )
)

lstm_test_scores = np.mean(
    np.square(
        X_lstm_test
        - lstm_test_reconstruction
    ),
    axis=(1, 2),
)

lstm_test_scoring_seconds = (
    time.perf_counter()
    - lstm_test_score_start
)

lstm_test_predictions = (
    lstm_test_scores
    > lstm_threshold
).astype(np.int8)

lstm_test_metrics = (
    calculate_event_alignment_metrics(
        y_test_event,
        lstm_test_scores,
        lstm_test_predictions,
    )
)


lstm_validation_result = make_result_frame(
    model_name="LSTM Autoencoder",
    split_name="validation",
    metrics=lstm_validation_metrics,
    threshold=lstm_threshold,
    training_seconds=lstm_training_seconds,
    scoring_seconds=(
        lstm_validation_scoring_seconds
    ),
    input_representation=(
        "60-step three-channel temporal sequence"
    ),
)

lstm_test_result = make_result_frame(
    model_name="LSTM Autoencoder",
    split_name="test",
    metrics=lstm_test_metrics,
    threshold=lstm_threshold,
    training_seconds=lstm_training_seconds,
    scoring_seconds=(
        lstm_test_scoring_seconds
    ),
    input_representation=(
        "60-step three-channel temporal sequence"
    ),
)

lstm_metrics_df = pd.concat(
    [
        lstm_validation_result,
        lstm_test_result,
    ],
    ignore_index=True
)


lstm_history_df = pd.DataFrame(
    lstm_history.history
)

lstm_history_df[
    "epoch"
] = np.arange(
    1,
    len(lstm_history_df) + 1
)

lstm_history_df.to_csv(
    LSTM_AE_OUTPUT_DIR /
    "training_history.csv",
    index=False
)

lstm_metrics_df.to_csv(
    LSTM_AE_OUTPUT_DIR /
    "metrics.csv",
    index=False
)

lstm_autoencoder.save(
    LSTM_AE_OUTPUT_DIR /
    "lstm_autoencoder.keras"
)


lstm_test_metadata = (
    model_window_manifest.iloc[
        test_indices
    ][
        [
            "model_window_id",
            "recording_block",
            "window_start",
            "window_end",
            "event_count",
            "event_window_label",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

lstm_test_metadata[
    "anomaly_score"
] = lstm_test_scores

lstm_test_metadata[
    "threshold"
] = lstm_threshold

lstm_test_metadata[
    "flagged"
] = lstm_test_predictions

lstm_test_metadata.to_csv(
    LSTM_AE_OUTPUT_DIR /
    "test_window_scores.csv",
    index=False
)


controlled_results[
    "LSTM Autoencoder"
] = {
    "model": lstm_autoencoder,
    "threshold": lstm_threshold,
    "validation_scores": (
        lstm_validation_scores
    ),
    "validation_predictions": (
        lstm_validation_predictions
    ),
    "test_scores": lstm_test_scores,
    "test_predictions": (
        lstm_test_predictions
    ),
    "metrics": lstm_metrics_df,
}


print("\nLSTM Autoencoder completed")
print("=" * 100)
print(
    f"Training time: "
    f"{lstm_training_seconds:.3f} seconds"
)
print(
    f"Threshold: "
    f"{lstm_threshold:.6f}"
)

print("\nValidation result")
display(lstm_validation_result)

print("\nTest result")
display(lstm_test_result)

In [ ]:
# =========================================================
# CELL 30 — FINAL FIVE-MODEL BLUED COMPARISON
# =========================================================

all_model_metric_frames = []

for model_name, model_result in (
    controlled_results.items()
):
    all_model_metric_frames.append(
        model_result["metrics"]
    )

all_blued_model_metrics = pd.concat(
    all_model_metric_frames,
    ignore_index=True
)


blued_five_model_comparison = (
    all_blued_model_metrics[
        all_blued_model_metrics[
            "split"
        ] == "test"
    ]
    .copy()
    .sort_values(
        "f1_score",
        ascending=False
    )
    .reset_index(drop=True)
)


comparison_columns = [
    "model",
    "input_representation",
    "precision",
    "recall",
    "f1_score",
    "false_positive_rate",
    "roc_auc",
    "pr_auc",
    "tn",
    "fp",
    "fn",
    "tp",
    "flagged_windows",
    "flagged_ratio_percent",
    "training_seconds",
]

blued_five_model_comparison = (
    blued_five_model_comparison[
        comparison_columns
    ]
)


# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------
expected_test_windows = len(
    y_test_event
)

for _, result_row in (
    blued_five_model_comparison.iterrows()
):
    confusion_total = (
        result_row["tn"]
        + result_row["fp"]
        + result_row["fn"]
        + result_row["tp"]
    )

    if confusion_total != expected_test_windows:
        raise ValueError(
            f"Confusion-matrix total mismatch for "
            f"{result_row['model']}: "
            f"{confusion_total} != "
            f"{expected_test_windows}"
        )


print(
    "Final label-blind five-model BLUED "
    "event-alignment comparison"
)
print("=" * 120)

display(blued_five_model_comparison)


blued_five_model_comparison.to_csv(
    FINAL_COMPARISON_DIR /
    (
        "blued_label_blind_five_model_"
        "event_alignment_comparison.csv"
    ),
    index=False
)

all_blued_model_metrics.to_csv(
    FINAL_COMPARISON_DIR /
    "blued_all_validation_and_test_metrics.csv",
    index=False
)


print("\nVerification")
print("-" * 100)
print(
    f"Test windows per model: "
    f"{expected_test_windows:,}"
)
print(
    "All confusion matrices sum to the complete "
    "BLUED test set."
)
print(
    "Labels were used only for evaluation, not for "
    "training or threshold selection."
)

In [ ]:
# =========================================================
# CELL 31 — FINALIZE BLUED FIVE-MODEL RESULTS
# =========================================================

from pathlib import Path
import json

BLUED_PUBLICATION_DIR = (
    FINAL_COMPARISON_DIR /
    "publication_outputs"
)

BLUED_TABLE_DIR = (
    BLUED_PUBLICATION_DIR /
    "tables"
)

BLUED_FIGURE_DIR = (
    BLUED_PUBLICATION_DIR /
    "figures"
)

for directory in [
    BLUED_PUBLICATION_DIR,
    BLUED_TABLE_DIR,
    BLUED_FIGURE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ---------------------------------------------------------
# Publication table
# ---------------------------------------------------------
publication_columns = [
    "model",
    "precision",
    "recall",
    "f1_score",
    "false_positive_rate",
    "roc_auc",
    "pr_auc",
    "flagged_ratio_percent",
    "training_seconds",
]

blued_publication_table = (
    blued_five_model_comparison[
        publication_columns
    ]
    .copy()
    .rename(
        columns={
            "model": "Model",
            "precision": "Precision",
            "recall": "Recall",
            "f1_score": "F1",
            "false_positive_rate": "FPR",
            "roc_auc": "ROC-AUC",
            "pr_auc": "PR-AUC",
            "flagged_ratio_percent": (
                "Flagged Windows (%)"
            ),
            "training_seconds": (
                "Training Time (s)"
            ),
        }
    )
)

numeric_columns = [
    "Precision",
    "Recall",
    "F1",
    "FPR",
    "ROC-AUC",
    "PR-AUC",
    "Flagged Windows (%)",
    "Training Time (s)",
]

blued_publication_table[
    numeric_columns
] = blued_publication_table[
    numeric_columns
].round(4)


blued_publication_table.to_csv(
    BLUED_TABLE_DIR /
    "blued_five_model_publication_table.csv",
    index=False
)

blued_five_model_comparison.to_csv(
    BLUED_TABLE_DIR /
    "blued_five_model_complete_results.csv",
    index=False
)

all_blued_model_metrics.to_csv(
    BLUED_TABLE_DIR /
    "blued_validation_and_test_results.csv",
    index=False
)


latex_table = (
    blued_publication_table
    .to_latex(
        index=False,
        escape=True,
        float_format="%.4f",
        caption=(
            "Label-blind event-alignment performance "
            "of the five unsupervised models on BLUED."
        ),
        label=(
            "tab:blued_five_model_comparison"
        ),
    )
)

latex_table = latex_table.replace(
    "\\begin{table}",
    "\\begin{table}[htbp]\n\\centering"
)

latex_table = latex_table.replace(
    "\\begin{tabular}",
    "\\resizebox{\\linewidth}{!}{%\n"
    "\\begin{tabular}"
)

latex_table = latex_table.replace(
    "\\end{tabular}",
    "\\end{tabular}%\n}"
)

with open(
    BLUED_TABLE_DIR /
    "blued_five_model_publication_table.tex",
    "w",
    encoding="utf-8",
) as file:
    file.write(latex_table)


print("BLUED publication table")
print("=" * 120)
display(blued_publication_table)


# ---------------------------------------------------------
# Figure 1 — thresholded test metrics
# ---------------------------------------------------------
threshold_metrics = [
    "precision",
    "recall",
    "f1_score",
    "false_positive_rate",
]

threshold_metric_labels = [
    "Precision",
    "Recall",
    "F1",
    "FPR",
]

threshold_plot_data = (
    blued_five_model_comparison
    .set_index("model")[
        threshold_metrics
    ]
    .rename(
        columns=dict(
            zip(
                threshold_metrics,
                threshold_metric_labels,
            )
        )
    )
)

figure, axis = plt.subplots(
    figsize=(12, 6)
)

threshold_plot_data.plot(
    kind="bar",
    ax=axis,
)

axis.set_title(
    "BLUED Label-Blind Event-Alignment Performance"
)

axis.set_xlabel("Model")
axis.set_ylabel("Metric Value")
axis.set_ylim(0, 0.70)
axis.tick_params(
    axis="x",
    rotation=20
)
axis.grid(
    axis="y",
    alpha=0.3
)
axis.legend(
    title="Metric",
    ncol=4
)

figure.tight_layout()

figure.savefig(
    BLUED_FIGURE_DIR /
    "blued_thresholded_metrics.png",
    dpi=300,
    bbox_inches="tight"
)

figure.savefig(
    BLUED_FIGURE_DIR /
    "blued_thresholded_metrics.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(figure)


# ---------------------------------------------------------
# Figure 2 — ranking metrics
# ---------------------------------------------------------
auc_plot_data = (
    blued_five_model_comparison
    .set_index("model")[
        [
            "roc_auc",
            "pr_auc",
        ]
    ]
    .rename(
        columns={
            "roc_auc": "ROC-AUC",
            "pr_auc": "PR-AUC",
        }
    )
)

figure, axis = plt.subplots(
    figsize=(11, 6)
)

auc_plot_data.plot(
    kind="bar",
    ax=axis,
)

axis.axhline(
    y=float(y_test_event.mean()),
    linestyle="--",
    linewidth=1.5,
    label=(
        "Random PR-AUC baseline "
        f"({y_test_event.mean():.3f})"
    ),
)

axis.set_title(
    "BLUED Continuous Anomaly-Score Ranking"
)

axis.set_xlabel("Model")
axis.set_ylabel("Area Under Curve")
axis.set_ylim(0, 0.85)
axis.tick_params(
    axis="x",
    rotation=20
)
axis.grid(
    axis="y",
    alpha=0.3
)
axis.legend()

figure.tight_layout()

figure.savefig(
    BLUED_FIGURE_DIR /
    "blued_auc_metrics.png",
    dpi=300,
    bbox_inches="tight"
)

figure.savefig(
    BLUED_FIGURE_DIR /
    "blued_auc_metrics.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(figure)


# ---------------------------------------------------------
# Figure 3 — computational cost
# ---------------------------------------------------------
training_time_data = (
    blued_five_model_comparison[
        [
            "model",
            "training_seconds",
        ]
    ]
    .sort_values(
        "training_seconds",
        ascending=True
    )
)

figure, axis = plt.subplots(
    figsize=(10, 6)
)

axis.bar(
    training_time_data["model"],
    training_time_data[
        "training_seconds"
    ],
)

axis.set_yscale("log")

axis.set_title(
    "BLUED Model Training Time"
)

axis.set_xlabel("Model")
axis.set_ylabel(
    "Training Time in Seconds "
    "(Logarithmic Scale)"
)

axis.tick_params(
    axis="x",
    rotation=20
)
axis.grid(
    axis="y",
    alpha=0.3
)

for index, value in enumerate(
    training_time_data[
        "training_seconds"
    ]
):
    axis.text(
        index,
        value,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

figure.tight_layout()

figure.savefig(
    BLUED_FIGURE_DIR /
    "blued_training_time.png",
    dpi=300,
    bbox_inches="tight"
)

figure.savefig(
    BLUED_FIGURE_DIR /
    "blued_training_time.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(figure)


# ---------------------------------------------------------
# Experiment metadata
# ---------------------------------------------------------
blued_experiment_metadata = {
    "dataset": "BLUED",
    "evaluation_role": (
        "Event-alignment benchmark; appliance switching "
        "events are not treated as confirmed anomalies."
    ),
    "canonical_events": 2482,
    "signal_rows": int(
        len(blued_signal_1s)
    ),
    "window_minutes": 1,
    "stride_seconds": 30,
    "generated_valid_windows": 20414,
    "modeling_windows": int(
        len(model_window_manifest)
    ),
    "excluded_boundary_windows": int(
        len(excluded_windows)
    ),
    "train_windows": int(
        len(train_indices)
    ),
    "validation_windows": int(
        len(validation_indices)
    ),
    "test_windows": int(
        len(test_indices)
    ),
    "test_event_windows": int(
        y_test_event.sum()
    ),
    "test_non_event_windows": int(
        len(y_test_event)
        - y_test_event.sum()
    ),
    "test_event_prevalence": float(
        y_test_event.mean()
    ),
    "feature_representation": (
        "69 statistical features: 23 features for each "
        "of mean_power, max_power, and std_power."
    ),
    "sequence_representation": (
        "60 time steps by 3 channels."
    ),
    "split_strategy": (
        "Purged chronological 60/20/20 split inside "
        "each recording block."
    ),
    "normalization": (
        "Training-only z-score normalization."
    ),
    "threshold_protocol": (
        "95th percentile of validation anomaly scores; "
        "validation labels were not used."
    ),
    "labels_used_during_training": False,
    "labels_used_for_threshold_selection": False,
    "labels_used_for_final_evaluation": True,
    "random_state": int(RANDOM_STATE),
}

with open(
    BLUED_PUBLICATION_DIR /
    "blued_experiment_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        blued_experiment_metadata,
        file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# Human-readable experiment summary
# ---------------------------------------------------------
best_f1_row = (
    blued_five_model_comparison
    .sort_values(
        "f1_score",
        ascending=False
    )
    .iloc[0]
)

best_auc_row = (
    blued_five_model_comparison
    .sort_values(
        "roc_auc",
        ascending=False
    )
    .iloc[0]
)

best_pr_auc_row = (
    blued_five_model_comparison
    .sort_values(
        "pr_auc",
        ascending=False
    )
    .iloc[0]
)

summary_text = f"""
# Final BLUED Five-Model Event-Alignment Experiment

## Experimental protocol

- One-minute windows
- Thirty-second stride
- Purged chronological 60/20/20 split
- 20,406 retained windows
- 69 engineered features for feature-based models
- 60 x 3 temporal sequences for the LSTM Autoencoder
- Training-only normalization
- Labels hidden during training
- Labels hidden during threshold selection
- Threshold fixed at the 95th percentile of validation scores
- BLUED event labels revealed only for evaluation

## Main results

- Best F1: {best_f1_row['model']} ({best_f1_row['f1_score']:.6f})
- Best ROC-AUC: {best_auc_row['model']} ({best_auc_row['roc_auc']:.6f})
- Best PR-AUC: {best_pr_auc_row['model']} ({best_pr_auc_row['pr_auc']:.6f})
- Test event prevalence: {y_test_event.mean():.6f}

## Interpretation

The LSTM Autoencoder achieved the strongest thresholded
event-alignment balance. Isolation Forest achieved the
strongest continuous score-ranking performance and the
lowest false-positive rate. The results evaluate alignment
with appliance switching events and must not be interpreted
as accuracy against confirmed anomaly labels.
""".strip()

with open(
    BLUED_PUBLICATION_DIR /
    "BLUED_FINAL_RESULTS_SUMMARY.md",
    "w",
    encoding="utf-8",
) as file:
    file.write(summary_text)


print("\nBLUED finalization complete")
print("=" * 100)
print(f"Output directory: {BLUED_PUBLICATION_DIR}")
print(
    "Saved publication table, complete metrics, "
    "three figures, metadata and summary."
)

In [ ]:
# Step 1 — Mount Google Drive and define thesis project paths

from google.colab import drive
from pathlib import Path
import sys

# Mount Google Drive
drive.mount("/content/drive")

# Main thesis project path
BASE_PATH = Path("/content/drive/MyDrive/Master/Thesis/Code")

# Standard project directories
DATASETS_PATH = BASE_PATH / "Datasets"
PROCESSED_DATA_PATH = BASE_PATH / "processed_data"
RESULTS_PATH = BASE_PATH / "results"
EXPERIMENTS_PATH = BASE_PATH / "experiments"
LOGS_PATH = BASE_PATH / "experiment_logs"
SRC_PATH = BASE_PATH / "src"

# Add the source-code directory to Python's import path
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# Verify that the main project directory exists
if not BASE_PATH.exists():
    raise FileNotFoundError(
        f"The thesis project directory was not found:\n{BASE_PATH}"
    )

print("Google Drive mounted successfully.")
print()
print("Thesis project paths:")
print(f"BASE_PATH:           {BASE_PATH}")
print(f"DATASETS_PATH:       {DATASETS_PATH}")
print(f"PROCESSED_DATA_PATH: {PROCESSED_DATA_PATH}")
print(f"RESULTS_PATH:        {RESULTS_PATH}")
print(f"EXPERIMENTS_PATH:    {EXPERIMENTS_PATH}")
print(f"LOGS_PATH:           {LOGS_PATH}")
print(f"SRC_PATH:            {SRC_PATH}")
print()
print("Project directory verified successfully.")

In [ ]:
# Step 2 — Locate and audit all existing BLUED files and directories

from collections import Counter
from pathlib import Path
import os

SEARCH_ROOTS = [
    DATASETS_PATH,
    PROCESSED_DATA_PATH,
    RESULTS_PATH,
    EXPERIMENTS_PATH,
    LOGS_PATH,
]

blued_paths = []

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for current_dir, dir_names, file_names in os.walk(root):
        current_path = Path(current_dir)

        # Record directories containing "BLUED" in their name or path
        if "blued" in str(current_path).lower():
            blued_paths.append(current_path)

        # Record files whose name or full path contains "BLUED"
        for file_name in file_names:
            file_path = current_path / file_name

            if "blued" in str(file_path).lower():
                blued_paths.append(file_path)

# Remove duplicates and sort
blued_paths = sorted(
    set(blued_paths),
    key=lambda path: (str(path).lower(), path.is_file())
)

if not blued_paths:
    raise FileNotFoundError(
        "No BLUED-related files or directories were found under the thesis project."
    )

print(f"Found {len(blued_paths):,} BLUED-related paths.\n")

file_extension_counts = Counter()
file_count = 0
directory_count = 0

for index, path in enumerate(blued_paths, start=1):
    if path.is_dir():
        directory_count += 1
        path_type = "DIR "
        size_text = ""
    else:
        file_count += 1
        path_type = "FILE"

        suffix = path.suffix.lower() if path.suffix else "[no extension]"
        file_extension_counts[suffix] += 1

        try:
            size_mb = path.stat().st_size / (1024 ** 2)
            size_text = f" | {size_mb:,.3f} MB"
        except OSError:
            size_text = " | size unavailable"

    print(f"{index:03d}. [{path_type}] {path}{size_text}")

print("\nBLUED audit summary:")
print(f"Directories: {directory_count:,}")
print(f"Files:       {file_count:,}")

print("\nFile types:")
for suffix, count in sorted(
    file_extension_counts.items(),
    key=lambda item: (-item[1], item[0])
):
    print(f"{suffix:15s}: {count:,}")

In [ ]:
# Step 3 — Inspect the final BLUED window-level and supervised-learning artifacts

from pathlib import Path
import os

PH3_BLUED_PATH = PROCESSED_DATA_PATH / "PH3" / "BLUED"
PH4_SUPERVISED_BLUED_PATH = (
    PROCESSED_DATA_PATH / "PH4" / "supervised" / "BLUED"
)

TARGET_PATHS = [
    PH3_BLUED_PATH,
    PH4_SUPERVISED_BLUED_PATH,
]

for target_path in TARGET_PATHS:
    print("=" * 100)
    print(f"Directory: {target_path}")
    print(f"Exists:    {target_path.exists()}")
    print("=" * 100)

    if not target_path.exists():
        print("Directory not found.\n")
        continue

    files = sorted(
        [
            path
            for path in target_path.rglob("*")
            if path.is_file()
        ],
        key=lambda path: str(path).lower()
    )

    if not files:
        print("No files found.\n")
        continue

    print(f"Files found: {len(files):,}\n")

    for index, file_path in enumerate(files, start=1):
        relative_path = file_path.relative_to(target_path)

        try:
            size_mb = file_path.stat().st_size / (1024 ** 2)
            size_text = f"{size_mb:,.3f} MB"
        except OSError:
            size_text = "size unavailable"

        print(
            f"{index:03d}. "
            f"{relative_path} "
            f"| {size_text}"
        )

    print()

print("Targeted BLUED artifact inspection completed.")

In [ ]:
# Step 4 — Locate the existing BLUED reconstruction, windows, features, labels, and split artifacts

from pathlib import Path
from collections import Counter, defaultdict

# Keywords associated with reusable modeling artifacts
TARGET_KEYWORDS = {
    "reconstruct",
    "signal",
    "window",
    "feature",
    "sequence",
    "split",
    "label",
    "event",
    "metadata",
    "model",
    "supervised",
    "complete",
    "final",
    "summary",
    "scored",
    "prediction",
}

# Directories associated mainly with raw BLUED segments or irrelevant caches
EXCLUDED_PATH_PARTS = {
    ".git",
    "__pycache__",
    "raw_segments",
    "processed_segments",
    "event_lists",
}

candidate_files = []

for file_path in BASE_PATH.rglob("*"):
    if not file_path.is_file():
        continue

    path_lower = str(file_path).lower()
    name_lower = file_path.name.lower()

    # Keep only BLUED-related paths
    if "blued" not in path_lower:
        continue

    # Avoid large collections of raw source segments
    if any(part in path_lower for part in EXCLUDED_PATH_PARTS):
        continue

    # Keep files whose names indicate reusable modeling artifacts
    if any(keyword in name_lower for keyword in TARGET_KEYWORDS):
        candidate_files.append(file_path)

candidate_files = sorted(
    set(candidate_files),
    key=lambda path: str(path).lower()
)

if not candidate_files:
    print("No filtered BLUED modeling artifacts were found.")
    print("A broader directory-level search will be required.")
else:
    print(f"Found {len(candidate_files):,} candidate BLUED modeling artifacts.\n")

    extension_counts = Counter()
    directory_counts = defaultdict(int)

    for index, file_path in enumerate(candidate_files, start=1):
        suffix = file_path.suffix.lower() or "[no extension]"
        extension_counts[suffix] += 1

        parent_relative = file_path.parent.relative_to(BASE_PATH)
        directory_counts[str(parent_relative)] += 1

        try:
            size_mb = file_path.stat().st_size / (1024 ** 2)
            size_text = f"{size_mb:,.3f} MB"
        except OSError:
            size_text = "size unavailable"

        print(
            f"{index:03d}. "
            f"{file_path.relative_to(BASE_PATH)} "
            f"| {size_text}"
        )

    print("\nCandidate file types:")
    for suffix, count in sorted(
        extension_counts.items(),
        key=lambda item: (-item[1], item[0])
    ):
        print(f"{suffix:15s}: {count:,}")

    print("\nDirectories containing candidate artifacts:")
    for directory, count in sorted(
        directory_counts.items(),
        key=lambda item: (-item[1], item[0])
    ):
        print(f"{count:4d} file(s) | {directory}")

In [ ]:
# Step 5 — Load and audit the reusable BLUED supervised-learning inputs

import pandas as pd
import numpy as np
from pathlib import Path

BLUED_COMMON_PATH = (
    EXPERIMENTS_PATH
    / "EXP_BLUED_FIVE_MODEL_LABEL_BLIND"
    / "common"
)

FEATURES_FILE = (
    BLUED_COMMON_PATH
    / "blued_features_1min_69features.csv"
)

MANIFEST_FILE = (
    BLUED_COMMON_PATH
    / "blued_1min_stride30s_model_manifest.csv"
)

EVENT_WINDOWS_FILE = (
    BLUED_COMMON_PATH
    / "blued_event_windows_1min_stride30s.csv"
)

SPLIT_SUMMARY_FILE = (
    BLUED_COMMON_PATH
    / "blued_1min_split_summary.csv"
)

REQUIRED_FILES = [
    FEATURES_FILE,
    MANIFEST_FILE,
    EVENT_WINDOWS_FILE,
    SPLIT_SUMMARY_FILE,
]

print("Required input-file verification:\n")

for file_path in REQUIRED_FILES:
    exists = file_path.exists()

    print(f"{file_path.name}")
    print(f"  Exists: {exists}")

    if exists:
        size_mb = file_path.stat().st_size / (1024 ** 2)
        print(f"  Size:   {size_mb:,.3f} MB")

    print()

missing_files = [
    file_path
    for file_path in REQUIRED_FILES
    if not file_path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following required BLUED files are missing:\n"
        + "\n".join(str(path) for path in missing_files)
    )

# Load the reusable artifacts
features_df = pd.read_csv(FEATURES_FILE)
manifest_df = pd.read_csv(MANIFEST_FILE)
event_windows_df = pd.read_csv(EVENT_WINDOWS_FILE)
split_summary_df = pd.read_csv(SPLIT_SUMMARY_FILE)

loaded_tables = {
    "Feature table": features_df,
    "Model manifest": manifest_df,
    "Event-window table": event_windows_df,
    "Split summary": split_summary_df,
}

for table_name, dataframe in loaded_tables.items():
    print("=" * 100)
    print(table_name)
    print("=" * 100)

    print(f"Shape: {dataframe.shape}")
    print(f"Duplicate rows: {dataframe.duplicated().sum():,}")
    print(f"Total missing values: {dataframe.isna().sum().sum():,}")

    print("\nColumns:")
    for column_number, column_name in enumerate(
        dataframe.columns,
        start=1
    ):
        print(
            f"{column_number:03d}. "
            f"{column_name} "
            f"| dtype={dataframe[column_name].dtype}"
        )

    print("\nFirst three rows:")
    display(dataframe.head(3))

    print()

# Show likely identifier, split, and target columns
all_column_names = sorted(
    set(features_df.columns)
    | set(manifest_df.columns)
    | set(event_windows_df.columns)
)

important_keywords = [
    "window",
    "block",
    "start",
    "end",
    "split",
    "label",
    "event",
    "target",
]

important_columns = [
    column_name
    for column_name in all_column_names
    if any(
        keyword in column_name.lower()
        for keyword in important_keywords
    )
]

print("=" * 100)
print("Potential identifier, split, and target columns")
print("=" * 100)

for column_name in important_columns:
    locations = []

    if column_name in features_df.columns:
        locations.append("features")

    if column_name in manifest_df.columns:
        locations.append("manifest")

    if column_name in event_windows_df.columns:
        locations.append("event_windows")

    print(
        f"{column_name:45s} "
        f"| present in: {', '.join(locations)}"
    )

print("\nBLUED reusable-input audit completed.")

In [ ]:
# Step 6 — Verify BLUED feature-label alignment and prepare the supervised dataset

import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 1. Verify identifier uniqueness
# ------------------------------------------------------------------

if not features_df["model_window_id"].is_unique:
    raise ValueError(
        "The feature table contains duplicate model_window_id values."
    )

if not manifest_df["model_window_id"].is_unique:
    raise ValueError(
        "The model manifest contains duplicate model_window_id values."
    )

feature_ids = set(features_df["model_window_id"])
manifest_ids = set(manifest_df["model_window_id"])

if feature_ids != manifest_ids:
    only_in_features = sorted(feature_ids - manifest_ids)
    only_in_manifest = sorted(manifest_ids - feature_ids)

    raise ValueError(
        "Feature and manifest identifiers do not match.\n"
        f"Only in features: {len(only_in_features):,}\n"
        f"Only in manifest: {len(only_in_manifest):,}"
    )

print("Identifier verification passed.")
print(f"Matching model_window_id values: {len(feature_ids):,}")

# ------------------------------------------------------------------
# 2. Merge the features with labels, timestamps, and split assignments
# ------------------------------------------------------------------

manifest_columns = [
    "model_window_id",
    "window_id",
    "recording_block",
    "window_start",
    "window_end",
    "event_count",
    "event_window_label",
    "split",
]

supervised_df = manifest_df[manifest_columns].merge(
    features_df,
    on="model_window_id",
    how="inner",
    validate="one_to_one",
)

supervised_df["window_start"] = pd.to_datetime(
    supervised_df["window_start"],
    errors="raise",
)

supervised_df["window_end"] = pd.to_datetime(
    supervised_df["window_end"],
    errors="raise",
)

supervised_df = supervised_df.sort_values(
    ["recording_block", "window_start", "model_window_id"]
).reset_index(drop=True)

print(f"Merged supervised dataset shape: {supervised_df.shape}")

# ------------------------------------------------------------------
# 3. Identify and validate the 69 engineered feature columns
# ------------------------------------------------------------------

NON_FEATURE_COLUMNS = {
    "model_window_id",
    "window_id",
    "recording_block",
    "window_start",
    "window_end",
    "event_count",
    "event_window_label",
    "split",
}

feature_columns = [
    column
    for column in supervised_df.columns
    if column not in NON_FEATURE_COLUMNS
]

if len(feature_columns) != 69:
    raise ValueError(
        f"Expected 69 engineered features, but found "
        f"{len(feature_columns)}."
    )

non_numeric_features = [
    column
    for column in feature_columns
    if not pd.api.types.is_numeric_dtype(supervised_df[column])
]

if non_numeric_features:
    raise TypeError(
        "The following feature columns are not numeric:\n"
        + "\n".join(non_numeric_features)
    )

feature_matrix = supervised_df[feature_columns].to_numpy(dtype=np.float64)

if not np.isfinite(feature_matrix).all():
    invalid_count = np.size(feature_matrix) - np.isfinite(feature_matrix).sum()

    raise ValueError(
        f"The feature matrix contains {invalid_count:,} "
        "non-finite values."
    )

print("Feature verification passed.")
print(f"Engineered feature columns: {len(feature_columns)}")
print(f"Feature matrix shape:       {feature_matrix.shape}")

# ------------------------------------------------------------------
# 4. Validate target values and chronological split assignments
# ------------------------------------------------------------------

valid_labels = {0, 1}
observed_labels = set(
    supervised_df["event_window_label"].unique()
)

if observed_labels != valid_labels:
    raise ValueError(
        f"Expected binary labels {valid_labels}, "
        f"but found {observed_labels}."
    )

# Normalize split names so that "validation" is represented consistently as "val"
supervised_df["split"] = (
    supervised_df["split"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(
        {
            "training": "train",
            "validation": "val",
            "testing": "test",
        }
    )
)

expected_splits = {"train", "val", "test"}
observed_splits = set(supervised_df["split"].unique())

if observed_splits != expected_splits:
    raise ValueError(
        f"Expected splits {expected_splits}, "
        f"but found {observed_splits}."
    )

print(f"Normalized split names: {sorted(observed_splits)}")

split_audit = (
    supervised_df
    .groupby("split", sort=False)
    .agg(
        windows=("model_window_id", "size"),
        event_windows=("event_window_label", "sum"),
        first_window=("window_start", "min"),
        last_window=("window_end", "max"),
    )
    .reset_index()
)

split_audit["non_event_windows"] = (
    split_audit["windows"]
    - split_audit["event_windows"]
)

split_audit["event_ratio_percent"] = (
    100.0
    * split_audit["event_windows"]
    / split_audit["windows"]
)

split_order = pd.Categorical(
    split_audit["split"],
    categories=["train", "val", "test"],
    ordered=True,
)

split_audit = (
    split_audit
    .assign(split_order=split_order)
    .sort_values("split_order")
    .drop(columns="split_order")
    .reset_index(drop=True)
)

print("\nSupervised BLUED split audit:")
display(split_audit)

# ------------------------------------------------------------------
# 5. Confirm the expected final split sizes and label counts
# ------------------------------------------------------------------

expected_counts = {
    "train": {
        "windows": 12248,
        "event_windows": 1788,
        "non_event_windows": 10460,
    },
    "val": {
        "windows": 4079,
        "event_windows": 588,
        "non_event_windows": 3491,
    },
    "test": {
        "windows": 4079,
        "event_windows": 450,
        "non_event_windows": 3629,
    },
}

for split_name, expected in expected_counts.items():
    row = split_audit.loc[
        split_audit["split"] == split_name
    ].iloc[0]

    for metric_name, expected_value in expected.items():
        actual_value = int(row[metric_name])

        if actual_value != expected_value:
            raise ValueError(
                f"{split_name} {metric_name}: "
                f"expected {expected_value:,}, "
                f"found {actual_value:,}."
            )

print("Expected split sizes and label counts verified successfully.")

# ------------------------------------------------------------------
# 6. Audit chronology within each recording block
# ------------------------------------------------------------------

chronology_rows = []

for recording_block in sorted(
    supervised_df["recording_block"].unique()
):
    block_df = supervised_df.loc[
        supervised_df["recording_block"] == recording_block
    ]

    for split_name in ["train", "val", "test"]:
        split_df = block_df.loc[
            block_df["split"] == split_name
        ]

        if split_df.empty:
            continue

        chronology_rows.append(
            {
                "recording_block": recording_block,
                "split": split_name,
                "windows": len(split_df),
                "first_start": split_df["window_start"].min(),
                "last_end": split_df["window_end"].max(),
            }
        )

chronology_audit = pd.DataFrame(chronology_rows)

print("\nChronological coverage by recording block and split:")
display(chronology_audit)

for recording_block in chronology_audit["recording_block"].unique():
    block_audit = (
        chronology_audit.loc[
            chronology_audit["recording_block"] == recording_block
        ]
        .set_index("split")
    )

    available_order = [
        split_name
        for split_name in ["train", "val", "test"]
        if split_name in block_audit.index
    ]

    for earlier_split, later_split in zip(
        available_order[:-1],
        available_order[1:]
    ):
        earlier_end = block_audit.loc[
            earlier_split,
            "last_end",
        ]

        later_start = block_audit.loc[
            later_split,
            "first_start",
        ]

        if earlier_end > later_start:
            raise ValueError(
                f"Chronological overlap detected in {recording_block}: "
                f"{earlier_split} ends at {earlier_end}, while "
                f"{later_split} starts at {later_start}."
            )

print("Chronological split verification passed.")

# ------------------------------------------------------------------
# 7. Prepare reusable train, validation, and test partitions
# ------------------------------------------------------------------

train_df = supervised_df.loc[
    supervised_df["split"] == "train"
].copy()

val_df = supervised_df.loc[
    supervised_df["split"] == "val"
].copy()

test_df = supervised_df.loc[
    supervised_df["split"] == "test"
].copy()

X_train = train_df[feature_columns].to_numpy(dtype=np.float64)
X_val = val_df[feature_columns].to_numpy(dtype=np.float64)
X_test = test_df[feature_columns].to_numpy(dtype=np.float64)

y_train = train_df["event_window_label"].to_numpy(dtype=np.int64)
y_val = val_df["event_window_label"].to_numpy(dtype=np.int64)
y_test = test_df["event_window_label"].to_numpy(dtype=np.int64)

print("\nPrepared supervised arrays:")
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape} | y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape} | y_test:  {y_test.shape}")

print("\nBLUED supervised dataset preparation completed successfully.")

In [ ]:
# Step 7 — Create the supervised BLUED experiment structure and freeze its configuration

import json
import random
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

# ------------------------------------------------------------------
# 1. Define the experiment identity
# ------------------------------------------------------------------

EXPERIMENT_ID = "PH4_SUP_BLUED_001"
EXPERIMENT_TITLE = (
    "Supervised BLUED Event-Window Classification Using Four Models"
)

SUPERVISED_BLUED_PATH = (
    PROCESSED_DATA_PATH
    / "PH4"
    / "supervised"
    / "BLUED"
    / EXPERIMENT_ID
)

CONFIG_PATH = SUPERVISED_BLUED_PATH / "configs"
METRICS_PATH = SUPERVISED_BLUED_PATH / "metrics"
PREDICTIONS_PATH = SUPERVISED_BLUED_PATH / "predictions"
MODELS_PATH = SUPERVISED_BLUED_PATH / "models"
FIGURES_PATH = SUPERVISED_BLUED_PATH / "figures"
AUDIT_PATH = SUPERVISED_BLUED_PATH / "audit"
LOG_PATH = SUPERVISED_BLUED_PATH / "logs"

EXPERIMENT_DIRECTORIES = [
    SUPERVISED_BLUED_PATH,
    CONFIG_PATH,
    METRICS_PATH,
    PREDICTIONS_PATH,
    MODELS_PATH,
    FIGURES_PATH,
    AUDIT_PATH,
    LOG_PATH,
]

for directory in EXPERIMENT_DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)

print("Experiment directories created successfully:\n")

for directory in EXPERIMENT_DIRECTORIES:
    print(directory)

# ------------------------------------------------------------------
# 2. Freeze reproducibility settings
# ------------------------------------------------------------------

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ------------------------------------------------------------------
# 3. Calculate class-distribution information
# ------------------------------------------------------------------

train_event_windows = int(y_train.sum())
train_non_event_windows = int(len(y_train) - train_event_windows)

validation_event_windows = int(y_val.sum())
validation_non_event_windows = int(len(y_val) - validation_event_windows)

test_event_windows = int(y_test.sum())
test_non_event_windows = int(len(y_test) - test_event_windows)

scale_pos_weight = (
    train_non_event_windows
    / train_event_windows
)

class_weight_event_0 = (
    len(y_train)
    / (2.0 * train_non_event_windows)
)

class_weight_event_1 = (
    len(y_train)
    / (2.0 * train_event_windows)
)

print("\nTraining class distribution:")
print(f"Non-event windows: {train_non_event_windows:,}")
print(f"Event windows:     {train_event_windows:,}")
print(f"Event ratio:       {100 * y_train.mean():.4f}%")
print(f"scale_pos_weight:  {scale_pos_weight:.6f}")

print("\nBalanced class weights:")
print(f"Class 0: {class_weight_event_0:.6f}")
print(f"Class 1: {class_weight_event_1:.6f}")

# ------------------------------------------------------------------
# 4. Define the planned supervised models and evaluation protocol
# ------------------------------------------------------------------

supervised_models = {
    "Random Forest": {
        "n_estimators": 300,
        "class_weight": "balanced_subsample",
        "random_state": RANDOM_SEED,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "scale_pos_weight": scale_pos_weight,
        "random_state": RANDOM_SEED,
        "n_jobs": -1,
    },
    "Logistic Regression": {
        "max_iter": 3000,
        "class_weight": "balanced",
        "random_state": RANDOM_SEED,
    },
    "MLP": {
        "hidden_layers": [64, 32],
        "hidden_activation": "relu",
        "output_activation": "sigmoid",
        "dropout": 0.2,
        "optimizer": "adam",
        "learning_rate": 0.001,
        "loss": "binary_crossentropy",
        "maximum_epochs": 50,
        "batch_size": 256,
        "early_stopping_patience": 5,
        "class_weights": {
            "0": class_weight_event_0,
            "1": class_weight_event_1,
        },
        "random_seed": RANDOM_SEED,
    },
}

evaluation_protocol = {
    "primary_target": "event_window_label",
    "target_semantics": (
        "A positive label indicates that at least one annotated appliance "
        "switching event occurs within the one-minute window. It is not a "
        "confirmed anomaly label."
    ),
    "split_strategy": (
        "Purged chronological 60/20/20 split applied separately within "
        "each BLUED recording block."
    ),
    "threshold_settings": [
        "Default probability threshold of 0.5",
        (
            "Validation-optimized threshold selected by maximizing "
            "F1-Score on the validation partition"
        ),
    ],
    "test_isolation": (
        "The test labels will not be used for model fitting, feature "
        "scaling, hyperparameter selection, or decision-threshold selection."
    ),
    "metrics": [
        "Precision",
        "Recall",
        "F1-Score",
        "False Positive Rate",
        "ROC-AUC",
        "PR-AUC",
        "True Negatives",
        "False Positives",
        "False Negatives",
        "True Positives",
    ],
}

# ------------------------------------------------------------------
# 5. Build and save the frozen experiment configuration
# ------------------------------------------------------------------

experiment_config = {
    "experiment_id": EXPERIMENT_ID,
    "experiment_title": EXPERIMENT_TITLE,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "research_task": "Supervised BLUED event-window classification",
    "dataset": "BLUED",
    "dataset_role": (
        "Labeled appliance switching-event dataset used for supervised "
        "event-window classification. The labels are not confirmed "
        "anomaly labels."
    ),
    "synthetic_anomaly_injection": False,
    "input_artifacts": {
        "features_file": str(FEATURES_FILE),
        "manifest_file": str(MANIFEST_FILE),
        "event_windows_file": str(EVENT_WINDOWS_FILE),
        "split_summary_file": str(SPLIT_SUMMARY_FILE),
    },
    "window_configuration": {
        "window_duration_seconds": 60,
        "stride_seconds": 30,
        "sequence_resolution_seconds": 1,
        "recording_blocks": 2,
        "boundary_crossing_windows_excluded": 8,
    },
    "representation": {
        "number_of_features": len(feature_columns),
        "feature_columns": feature_columns,
        "feature_models": [
            "Random Forest",
            "XGBoost",
            "Logistic Regression",
            "MLP",
        ],
    },
    "split_counts": {
        "train": {
            "windows": len(y_train),
            "event_windows": train_event_windows,
            "non_event_windows": train_non_event_windows,
            "event_ratio": float(y_train.mean()),
        },
        "validation": {
            "windows": len(y_val),
            "event_windows": validation_event_windows,
            "non_event_windows": validation_non_event_windows,
            "event_ratio": float(y_val.mean()),
        },
        "test": {
            "windows": len(y_test),
            "event_windows": test_event_windows,
            "non_event_windows": test_non_event_windows,
            "event_ratio": float(y_test.mean()),
        },
    },
    "imbalance_handling": {
        "training_scale_pos_weight": scale_pos_weight,
        "balanced_class_weight_0": class_weight_event_0,
        "balanced_class_weight_1": class_weight_event_1,
    },
    "models": supervised_models,
    "evaluation_protocol": evaluation_protocol,
    "reproducibility": {
        "random_seed": RANDOM_SEED,
        "python_version": platform.python_version(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "scikit_learn_version": sklearn.__version__,
    },
}

CONFIG_FILE = CONFIG_PATH / "blued_supervised_experiment_config.json"

with open(CONFIG_FILE, "w", encoding="utf-8") as file:
    json.dump(
        experiment_config,
        file,
        indent=2,
        ensure_ascii=False,
    )

# ------------------------------------------------------------------
# 6. Save the feature schema and split-audit tables
# ------------------------------------------------------------------

FEATURE_SCHEMA_FILE = (
    CONFIG_PATH
    / "blued_supervised_69_feature_schema.json"
)

with open(FEATURE_SCHEMA_FILE, "w", encoding="utf-8") as file:
    json.dump(
        {
            "experiment_id": EXPERIMENT_ID,
            "number_of_features": len(feature_columns),
            "features": feature_columns,
        },
        file,
        indent=2,
    )

SPLIT_AUDIT_FILE = (
    AUDIT_PATH
    / "blued_supervised_split_audit.csv"
)

CHRONOLOGY_AUDIT_FILE = (
    AUDIT_PATH
    / "blued_supervised_chronology_audit.csv"
)

split_audit.to_csv(
    SPLIT_AUDIT_FILE,
    index=False,
)

chronology_audit.to_csv(
    CHRONOLOGY_AUDIT_FILE,
    index=False,
)

# Save the modeling manifest used specifically for this experiment
SUPERVISED_MANIFEST_FILE = (
    AUDIT_PATH
    / "blued_supervised_modeling_manifest.csv"
)

supervised_df[
    [
        "model_window_id",
        "window_id",
        "recording_block",
        "window_start",
        "window_end",
        "event_count",
        "event_window_label",
        "split",
    ]
].to_csv(
    SUPERVISED_MANIFEST_FILE,
    index=False,
)

# ------------------------------------------------------------------
# 7. Verify all saved experiment-control artifacts
# ------------------------------------------------------------------

saved_control_files = [
    CONFIG_FILE,
    FEATURE_SCHEMA_FILE,
    SPLIT_AUDIT_FILE,
    CHRONOLOGY_AUDIT_FILE,
    SUPERVISED_MANIFEST_FILE,
]

print("\nSaved experiment-control artifacts:\n")

for file_path in saved_control_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Expected output was not created: {file_path}"
        )

    size_kb = file_path.stat().st_size / 1024

    print(
        f"{file_path.name:50s} "
        f"| {size_kb:,.2f} KB"
    )

print("\nSupervised BLUED experiment configuration frozen successfully.")
print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Experiment root: {SUPERVISED_BLUED_PATH}")

In [ ]:
# Step 8 — Verify machine-learning libraries and prepare train-only preprocessing pipelines

import os
import json
import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Reduce unnecessary TensorFlow console messages
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# ------------------------------------------------------------------
# 1. Import and verify XGBoost
# ------------------------------------------------------------------

try:
    import xgboost as xgb
    from xgboost import XGBClassifier
except ImportError as error:
    raise ImportError(
        "XGBoost is not installed in the current Colab runtime."
    ) from error

# ------------------------------------------------------------------
# 2. Import and verify TensorFlow
# ------------------------------------------------------------------

try:
    import tensorflow as tf
except ImportError as error:
    raise ImportError(
        "TensorFlow is not installed in the current Colab runtime."
    ) from error

print("Installed library versions:")
print(f"Python / runtime NumPy: {np.__version__}")
print(f"pandas:                 {pd.__version__}")
print(f"scikit-learn:           {sklearn.__version__}")
print(f"XGBoost:                {xgb.__version__}")
print(f"TensorFlow:             {tf.__version__}")

# ------------------------------------------------------------------
# 3. Configure reproducibility
# ------------------------------------------------------------------

np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)

try:
    tf.config.experimental.enable_op_determinism()
    deterministic_status = "Enabled"
except Exception:
    deterministic_status = "Not supported by this runtime"

print(f"\nTensorFlow deterministic operations: {deterministic_status}")

physical_gpus = tf.config.list_physical_devices("GPU")

if physical_gpus:
    print(f"GPU devices detected: {len(physical_gpus)}")

    for gpu_number, gpu_device in enumerate(
        physical_gpus,
        start=1
    ):
        print(f"  GPU {gpu_number}: {gpu_device.name}")
else:
    print("GPU devices detected: 0")
    print("The MLP will run on the CPU.")

# ------------------------------------------------------------------
# 4. Create preprocessing pipelines
# ------------------------------------------------------------------
# Random Forest and XGBoost:
# - median imputation only
# - no feature scaling
#
# Logistic Regression and MLP:
# - median imputation
# - standardization fitted only on the training partition

tree_imputer = SimpleImputer(
    strategy="median"
)

scaled_imputer = SimpleImputer(
    strategy="median"
)

standard_scaler = StandardScaler()

# ------------------------------------------------------------------
# 5. Fit preprocessing objects using training data only
# ------------------------------------------------------------------

X_train_tree = tree_imputer.fit_transform(X_train)
X_val_tree = tree_imputer.transform(X_val)
X_test_tree = tree_imputer.transform(X_test)

X_train_imputed = scaled_imputer.fit_transform(X_train)
X_val_imputed = scaled_imputer.transform(X_val)
X_test_imputed = scaled_imputer.transform(X_test)

X_train_scaled = standard_scaler.fit_transform(
    X_train_imputed
)

X_val_scaled = standard_scaler.transform(
    X_val_imputed
)

X_test_scaled = standard_scaler.transform(
    X_test_imputed
)

# Convert scaled arrays to float32 for TensorFlow efficiency
X_train_scaled_float32 = X_train_scaled.astype(np.float32)
X_val_scaled_float32 = X_val_scaled.astype(np.float32)
X_test_scaled_float32 = X_test_scaled.astype(np.float32)

# ------------------------------------------------------------------
# 6. Verify transformed arrays
# ------------------------------------------------------------------

transformed_arrays = {
    "X_train_tree": X_train_tree,
    "X_val_tree": X_val_tree,
    "X_test_tree": X_test_tree,
    "X_train_scaled": X_train_scaled,
    "X_val_scaled": X_val_scaled,
    "X_test_scaled": X_test_scaled,
}

for array_name, array_value in transformed_arrays.items():
    if array_value.shape[1] != 69:
        raise ValueError(
            f"{array_name} contains {array_value.shape[1]} features "
            "instead of the expected 69."
        )

    if not np.isfinite(array_value).all():
        invalid_values = (
            array_value.size
            - np.isfinite(array_value).sum()
        )

        raise ValueError(
            f"{array_name} contains {invalid_values:,} "
            "non-finite values."
        )

print("\nPrepared preprocessing arrays:")

for array_name, array_value in transformed_arrays.items():
    print(
        f"{array_name:20s} "
        f"| shape={array_value.shape} "
        f"| dtype={array_value.dtype}"
    )

# ------------------------------------------------------------------
# 7. Audit standardized training features
# ------------------------------------------------------------------

scaled_train_means = X_train_scaled.mean(axis=0)
scaled_train_stds = X_train_scaled.std(axis=0)

print("\nStandardized training-feature audit:")
print(
    "Maximum absolute feature mean: "
    f"{np.max(np.abs(scaled_train_means)):.8f}"
)
print(
    "Minimum feature standard deviation: "
    f"{np.min(scaled_train_stds):.8f}"
)
print(
    "Maximum feature standard deviation: "
    f"{np.max(scaled_train_stds):.8f}"
)

constant_feature_indices = np.where(
    scaled_train_stds < 1e-12
)[0]

if len(constant_feature_indices) > 0:
    constant_feature_names = [
        feature_columns[index]
        for index in constant_feature_indices
    ]

    print(
        f"\nConstant or near-constant training features: "
        f"{len(constant_feature_names)}"
    )

    for feature_name in constant_feature_names:
        print(f"  - {feature_name}")
else:
    print("\nNo constant training features were detected.")

# ------------------------------------------------------------------
# 8. Save fitted preprocessing objects
# ------------------------------------------------------------------

TREE_IMPUTER_FILE = (
    MODELS_PATH
    / "tree_models_median_imputer.joblib"
)

SCALED_IMPUTER_FILE = (
    MODELS_PATH
    / "scaled_models_median_imputer.joblib"
)

STANDARD_SCALER_FILE = (
    MODELS_PATH
    / "logistic_mlp_standard_scaler.joblib"
)

joblib.dump(
    tree_imputer,
    TREE_IMPUTER_FILE
)

joblib.dump(
    scaled_imputer,
    SCALED_IMPUTER_FILE
)

joblib.dump(
    standard_scaler,
    STANDARD_SCALER_FILE
)

# ------------------------------------------------------------------
# 9. Save preprocessing metadata
# ------------------------------------------------------------------

PREPROCESSING_METADATA_FILE = (
    CONFIG_PATH
    / "blued_supervised_preprocessing_metadata.json"
)

preprocessing_metadata = {
    "experiment_id": EXPERIMENT_ID,
    "training_only_fitting": True,
    "tree_models": {
        "models": [
            "Random Forest",
            "XGBoost",
        ],
        "imputation": "Median imputation fitted on training data only",
        "scaling": None,
    },
    "scaled_models": {
        "models": [
            "Logistic Regression",
            "MLP",
        ],
        "imputation": "Median imputation fitted on training data only",
        "scaling": (
            "StandardScaler fitted on the imputed training data only"
        ),
    },
    "array_shapes": {
        "X_train": list(X_train.shape),
        "X_validation": list(X_val.shape),
        "X_test": list(X_test.shape),
    },
    "library_versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgb.__version__,
        "tensorflow": tf.__version__,
    },
    "tensorflow_determinism": deterministic_status,
    "gpu_devices_detected": len(physical_gpus),
}

with open(
    PREPROCESSING_METADATA_FILE,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        preprocessing_metadata,
        file,
        indent=2,
    )

# ------------------------------------------------------------------
# 10. Verify saved preprocessing artifacts
# ------------------------------------------------------------------

saved_preprocessing_files = [
    TREE_IMPUTER_FILE,
    SCALED_IMPUTER_FILE,
    STANDARD_SCALER_FILE,
    PREPROCESSING_METADATA_FILE,
]

print("\nSaved preprocessing artifacts:")

for file_path in saved_preprocessing_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Expected preprocessing file was not created: "
            f"{file_path}"
        )

    print(
        f"{file_path.name:50s} "
        f"| {file_path.stat().st_size / 1024:,.2f} KB"
    )

print("\nBLUED supervised preprocessing completed successfully.")

In [ ]:
# Step 9 — Define the common validation-threshold selection and model-evaluation functions

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# ------------------------------------------------------------------
# 1. Validate probability-score arrays
# ------------------------------------------------------------------

def validate_binary_scores(
    y_true,
    y_score,
    dataset_name="dataset",
):
    """
    Validate binary targets and continuous probability scores.
    """

    y_true = np.asarray(y_true, dtype=np.int64).reshape(-1)
    y_score = np.asarray(y_score, dtype=np.float64).reshape(-1)

    if len(y_true) != len(y_score):
        raise ValueError(
            f"{dataset_name}: label and score lengths differ. "
            f"Labels={len(y_true):,}, scores={len(y_score):,}."
        )

    if not np.isfinite(y_score).all():
        invalid_count = (
            y_score.size
            - np.isfinite(y_score).sum()
        )

        raise ValueError(
            f"{dataset_name}: probability scores contain "
            f"{invalid_count:,} non-finite values."
        )

    observed_labels = set(np.unique(y_true))

    if not observed_labels.issubset({0, 1}):
        raise ValueError(
            f"{dataset_name}: expected binary labels, "
            f"but found {observed_labels}."
        )

    if len(observed_labels) < 2:
        raise ValueError(
            f"{dataset_name}: both classes are required "
            "for ROC-AUC and PR-AUC evaluation."
        )

    return y_true, y_score


# ------------------------------------------------------------------
# 2. Select the decision threshold using validation F1-Score only
# ------------------------------------------------------------------

def select_validation_f1_threshold(
    y_validation,
    validation_scores,
):
    """
    Select the probability threshold that maximizes F1-Score on
    the validation partition.

    Tie-breaking order:
    1. Highest validation F1-Score
    2. Highest validation Precision
    3. Highest threshold, providing the more conservative operating point
    """

    y_validation, validation_scores = validate_binary_scores(
        y_validation,
        validation_scores,
        dataset_name="Validation partition",
    )

    precision_values, recall_values, thresholds = (
        precision_recall_curve(
            y_validation,
            validation_scores,
        )
    )

    if len(thresholds) == 0:
        raise ValueError(
            "No candidate validation thresholds were generated."
        )

    # precision_recall_curve returns one additional precision/recall
    # value that has no corresponding threshold.
    candidate_precision = precision_values[:-1]
    candidate_recall = recall_values[:-1]

    denominator = (
        candidate_precision
        + candidate_recall
    )

    candidate_f1 = np.divide(
        2.0
        * candidate_precision
        * candidate_recall,
        denominator,
        out=np.zeros_like(
            denominator,
            dtype=np.float64,
        ),
        where=denominator > 0,
    )

    threshold_table = pd.DataFrame(
        {
            "threshold": thresholds.astype(np.float64),
            "precision": candidate_precision.astype(np.float64),
            "recall": candidate_recall.astype(np.float64),
            "f1_score": candidate_f1.astype(np.float64),
        }
    )

    # Stable deterministic tie-breaking
    threshold_table = (
        threshold_table
        .sort_values(
            by=[
                "f1_score",
                "precision",
                "threshold",
            ],
            ascending=[
                False,
                False,
                False,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    selected_row = threshold_table.iloc[0]

    selection_result = {
        "threshold": float(selected_row["threshold"]),
        "validation_precision": float(
            selected_row["precision"]
        ),
        "validation_recall": float(
            selected_row["recall"]
        ),
        "validation_f1": float(
            selected_row["f1_score"]
        ),
        "candidate_thresholds": int(
            len(threshold_table)
        ),
    }

    return selection_result, threshold_table


# ------------------------------------------------------------------
# 3. Evaluate continuous scores at a fixed decision threshold
# ------------------------------------------------------------------

def evaluate_binary_classifier(
    y_true,
    y_score,
    threshold,
    model_name,
    split_name,
    threshold_source,
):
    """
    Evaluate a supervised binary classifier using continuous scores
    and a specified decision threshold.
    """

    y_true, y_score = validate_binary_scores(
        y_true,
        y_score,
        dataset_name=f"{model_name} — {split_name}",
    )

    threshold = float(threshold)

    if not np.isfinite(threshold):
        raise ValueError(
            f"{model_name}: threshold is not finite."
        )

    y_pred = (
        y_score >= threshold
    ).astype(np.int64)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    false_positive_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0.0
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    roc_auc = roc_auc_score(
        y_true,
        y_score,
    )

    pr_auc = average_precision_score(
        y_true,
        y_score,
    )

    flagged_windows = int(y_pred.sum())
    total_windows = int(len(y_pred))

    result = {
        "experiment_id": EXPERIMENT_ID,
        "dataset": "BLUED",
        "task": "supervised_event_window_classification",
        "model": model_name,
        "split": split_name,
        "threshold_source": threshold_source,
        "threshold": threshold,
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "false_positive_rate": float(
            false_positive_rate
        ),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "flagged_windows": flagged_windows,
        "flagged_ratio": float(
            flagged_windows / total_windows
        ),
        "flagged_ratio_percent": float(
            100.0 * flagged_windows / total_windows
        ),
        "positive_windows": int(y_true.sum()),
        "negative_windows": int(
            len(y_true) - y_true.sum()
        ),
        "positive_prevalence": float(
            y_true.mean()
        ),
        "n_windows": total_windows,
    }

    prediction_table = pd.DataFrame(
        {
            "true_label": y_true,
            "probability_score": y_score,
            "predicted_label": y_pred,
        }
    )

    return result, prediction_table


# ------------------------------------------------------------------
# 4. Evaluate both required threshold settings
# ------------------------------------------------------------------

def evaluate_model_with_two_thresholds(
    model_name,
    validation_scores,
    test_scores,
    y_validation,
    y_test_values,
):
    """
    Evaluate one model at:
    1. The default probability threshold of 0.5
    2. The validation-F1-optimized threshold
    """

    threshold_selection, threshold_table = (
        select_validation_f1_threshold(
            y_validation,
            validation_scores,
        )
    )

    selected_threshold = threshold_selection[
        "threshold"
    ]

    validation_selected_result, _ = (
        evaluate_binary_classifier(
            y_true=y_validation,
            y_score=validation_scores,
            threshold=selected_threshold,
            model_name=model_name,
            split_name="validation",
            threshold_source=(
                "validation_f1_optimized"
            ),
        )
    )

    test_default_result, test_default_predictions = (
        evaluate_binary_classifier(
            y_true=y_test_values,
            y_score=test_scores,
            threshold=0.5,
            model_name=model_name,
            split_name="test",
            threshold_source="default_0.5",
        )
    )

    test_selected_result, test_selected_predictions = (
        evaluate_binary_classifier(
            y_true=y_test_values,
            y_score=test_scores,
            threshold=selected_threshold,
            model_name=model_name,
            split_name="test",
            threshold_source=(
                "validation_f1_optimized"
            ),
        )
    )

    return {
        "threshold_selection": threshold_selection,
        "threshold_table": threshold_table,
        "validation_selected_result": (
            validation_selected_result
        ),
        "test_default_result": (
            test_default_result
        ),
        "test_selected_result": (
            test_selected_result
        ),
        "test_default_predictions": (
            test_default_predictions
        ),
        "test_selected_predictions": (
            test_selected_predictions
        ),
    }


# ------------------------------------------------------------------
# 5. Run a controlled sanity check on the evaluation functions
# ------------------------------------------------------------------

sanity_y_validation = np.array(
    [0, 0, 0, 1, 1, 1],
    dtype=np.int64,
)

sanity_validation_scores = np.array(
    [0.05, 0.15, 0.35, 0.45, 0.75, 0.90],
    dtype=np.float64,
)

sanity_y_test = np.array(
    [0, 0, 1, 1],
    dtype=np.int64,
)

sanity_test_scores = np.array(
    [0.10, 0.40, 0.60, 0.85],
    dtype=np.float64,
)

sanity_output = evaluate_model_with_two_thresholds(
    model_name="Sanity Check Model",
    validation_scores=sanity_validation_scores,
    test_scores=sanity_test_scores,
    y_validation=sanity_y_validation,
    y_test_values=sanity_y_test,
)

print("Evaluation-function sanity check:")
print(
    "Selected validation threshold: "
    f"{sanity_output['threshold_selection']['threshold']:.6f}"
)
print(
    "Validation F1 at selected threshold: "
    f"{sanity_output['threshold_selection']['validation_f1']:.6f}"
)
print(
    "Test F1 at selected threshold: "
    f"{sanity_output['test_selected_result']['f1_score']:.6f}"
)

if (
    sanity_output[
        "test_selected_result"
    ]["f1_score"]
    < 0.0
    or sanity_output[
        "test_selected_result"
    ]["f1_score"]
    > 1.0
):
    raise RuntimeError(
        "The evaluation sanity check produced an invalid F1-Score."
    )

# ------------------------------------------------------------------
# 6. Save the frozen evaluation-protocol description
# ------------------------------------------------------------------

EVALUATION_PROTOCOL_FILE = (
    CONFIG_PATH
    / "blued_supervised_evaluation_protocol.json"
)

evaluation_protocol_metadata = {
    "experiment_id": EXPERIMENT_ID,
    "task": (
        "Supervised BLUED event-window classification"
    ),
    "target": "event_window_label",
    "positive_class_meaning": (
        "At least one annotated appliance switching event "
        "occurs inside the one-minute window."
    ),
    "annotation_warning": (
        "BLUED events are normal appliance switching events, "
        "not confirmed anomalies."
    ),
    "continuous_metrics": {
        "roc_auc": (
            "Calculated from continuous test probability scores"
        ),
        "pr_auc": (
            "Average Precision calculated from continuous "
            "test probability scores"
        ),
    },
    "thresholded_metrics": [
        "Precision",
        "Recall",
        "F1-Score",
        "False Positive Rate",
        "Confusion matrix counts",
    ],
    "thresholds": {
        "default": 0.5,
        "selected": (
            "Threshold maximizing F1-Score on validation "
            "probability scores only"
        ),
        "tie_breaking": [
            "Highest validation F1-Score",
            "Highest validation Precision",
            "Highest threshold",
        ],
    },
    "test_label_usage": (
        "Test labels are used only after model fitting and "
        "validation-threshold selection are complete."
    ),
}

with open(
    EVALUATION_PROTOCOL_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        evaluation_protocol_metadata,
        file,
        indent=2,
    )

if not EVALUATION_PROTOCOL_FILE.exists():
    raise FileNotFoundError(
        "The evaluation-protocol file was not created."
    )

print(
    "\nEvaluation protocol saved:"
)
print(EVALUATION_PROTOCOL_FILE)

print(
    "\nCommon supervised BLUED evaluation functions "
    "defined successfully."
)

In [ ]:
# Step 10 — Train and evaluate the supervised Random Forest model

import json
import time
import joblib

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier

# ------------------------------------------------------------------
# 1. Create the Random Forest using the frozen configuration
# ------------------------------------------------------------------

random_forest_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced_subsample",
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

print("Random Forest configuration:")
print(random_forest_model)
print()

# ------------------------------------------------------------------
# 2. Train the model using the training partition only
# ------------------------------------------------------------------

training_start_time = time.perf_counter()

random_forest_model.fit(
    X_train_tree,
    y_train,
)

random_forest_training_seconds = (
    time.perf_counter()
    - training_start_time
)

print(
    "Random Forest training completed in "
    f"{random_forest_training_seconds:,.4f} seconds."
)

# ------------------------------------------------------------------
# 3. Generate validation and test probability scores
# ------------------------------------------------------------------

inference_start_time = time.perf_counter()

random_forest_validation_scores = (
    random_forest_model.predict_proba(
        X_val_tree
    )[:, 1]
)

random_forest_test_scores = (
    random_forest_model.predict_proba(
        X_test_tree
    )[:, 1]
)

random_forest_inference_seconds = (
    time.perf_counter()
    - inference_start_time
)

if len(random_forest_validation_scores) != len(y_val):
    raise ValueError(
        "Random Forest validation-score length does not "
        "match the validation labels."
    )

if len(random_forest_test_scores) != len(y_test):
    raise ValueError(
        "Random Forest test-score length does not "
        "match the test labels."
    )

if not np.isfinite(
    random_forest_validation_scores
).all():
    raise ValueError(
        "Random Forest validation scores contain "
        "non-finite values."
    )

if not np.isfinite(
    random_forest_test_scores
).all():
    raise ValueError(
        "Random Forest test scores contain "
        "non-finite values."
    )

print(
    "Probability-score generation completed in "
    f"{random_forest_inference_seconds:,.4f} seconds."
)

print(
    "Validation score range: "
    f"{random_forest_validation_scores.min():.6f} "
    f"to {random_forest_validation_scores.max():.6f}"
)

print(
    "Test score range:       "
    f"{random_forest_test_scores.min():.6f} "
    f"to {random_forest_test_scores.max():.6f}"
)

# ------------------------------------------------------------------
# 4. Select the validation threshold and evaluate the test partition
# ------------------------------------------------------------------

random_forest_evaluation = (
    evaluate_model_with_two_thresholds(
        model_name="Random Forest",
        validation_scores=(
            random_forest_validation_scores
        ),
        test_scores=random_forest_test_scores,
        y_validation=y_val,
        y_test_values=y_test,
    )
)

selected_threshold = (
    random_forest_evaluation[
        "threshold_selection"
    ]["threshold"]
)

validation_selected_result = (
    random_forest_evaluation[
        "validation_selected_result"
    ].copy()
)

test_default_result = (
    random_forest_evaluation[
        "test_default_result"
    ].copy()
)

test_selected_result = (
    random_forest_evaluation[
        "test_selected_result"
    ].copy()
)

# Add computational information to every saved result
for result in [
    validation_selected_result,
    test_default_result,
    test_selected_result,
]:
    result["training_seconds"] = float(
        random_forest_training_seconds
    )

    result["inference_seconds"] = float(
        random_forest_inference_seconds
    )

    result["n_features"] = int(
        len(feature_columns)
    )

    result["n_train"] = int(
        len(y_train)
    )

    result["positive_train"] = int(
        y_train.sum()
    )

# ------------------------------------------------------------------
# 5. Prepare the complete metrics table
# ------------------------------------------------------------------

random_forest_metrics_df = pd.DataFrame(
    [
        validation_selected_result,
        test_default_result,
        test_selected_result,
    ]
)

preferred_metric_columns = [
    "experiment_id",
    "dataset",
    "task",
    "model",
    "split",
    "threshold_source",
    "threshold",
    "precision",
    "recall",
    "f1_score",
    "false_positive_rate",
    "roc_auc",
    "pr_auc",
    "tn",
    "fp",
    "fn",
    "tp",
    "flagged_windows",
    "flagged_ratio",
    "flagged_ratio_percent",
    "positive_windows",
    "negative_windows",
    "positive_prevalence",
    "n_windows",
    "n_train",
    "positive_train",
    "n_features",
    "training_seconds",
    "inference_seconds",
]

random_forest_metrics_df = (
    random_forest_metrics_df[
        preferred_metric_columns
    ]
)

# ------------------------------------------------------------------
# 6. Prepare validation-threshold audit data
# ------------------------------------------------------------------

random_forest_threshold_table = (
    random_forest_evaluation[
        "threshold_table"
    ].copy()
)

random_forest_threshold_table.insert(
    0,
    "experiment_id",
    EXPERIMENT_ID,
)

random_forest_threshold_table.insert(
    1,
    "model",
    "Random Forest",
)

random_forest_threshold_table.insert(
    2,
    "split",
    "validation",
)

# ------------------------------------------------------------------
# 7. Prepare validation-score output
# ------------------------------------------------------------------

random_forest_validation_predictions = (
    val_df[
        [
            "model_window_id",
            "window_id",
            "recording_block",
            "window_start",
            "window_end",
            "event_count",
            "event_window_label",
            "split",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

random_forest_validation_predictions[
    "model"
] = "Random Forest"

random_forest_validation_predictions[
    "probability_score"
] = random_forest_validation_scores

random_forest_validation_predictions[
    "selected_threshold"
] = selected_threshold

random_forest_validation_predictions[
    "predicted_label_selected_threshold"
] = (
    random_forest_validation_scores
    >= selected_threshold
).astype(np.int64)

# ------------------------------------------------------------------
# 8. Prepare complete test-prediction output
# ------------------------------------------------------------------

random_forest_test_predictions = (
    test_df[
        [
            "model_window_id",
            "window_id",
            "recording_block",
            "window_start",
            "window_end",
            "event_count",
            "event_window_label",
            "split",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

random_forest_test_predictions[
    "model"
] = "Random Forest"

random_forest_test_predictions[
    "probability_score"
] = random_forest_test_scores

random_forest_test_predictions[
    "default_threshold"
] = 0.5

random_forest_test_predictions[
    "predicted_label_default_0.5"
] = (
    random_forest_test_scores >= 0.5
).astype(np.int64)

random_forest_test_predictions[
    "validation_selected_threshold"
] = selected_threshold

random_forest_test_predictions[
    "predicted_label_validation_f1_optimized"
] = (
    random_forest_test_scores
    >= selected_threshold
).astype(np.int64)

# ------------------------------------------------------------------
# 9. Extract and rank Random Forest feature importance
# ------------------------------------------------------------------

random_forest_feature_importance_df = (
    pd.DataFrame(
        {
            "feature": feature_columns,
            "importance": (
                random_forest_model.feature_importances_
            ),
        }
    )
    .sort_values(
        "importance",
        ascending=False,
    )
    .reset_index(drop=True)
)

random_forest_feature_importance_df[
    "rank"
] = (
    np.arange(
        1,
        len(random_forest_feature_importance_df) + 1,
    )
)

random_forest_feature_importance_df = (
    random_forest_feature_importance_df[
        [
            "rank",
            "feature",
            "importance",
        ]
    ]
)

# ------------------------------------------------------------------
# 10. Define output files
# ------------------------------------------------------------------

RANDOM_FOREST_MODEL_FILE = (
    MODELS_PATH
    / "blued_supervised_random_forest.joblib"
)

RANDOM_FOREST_METRICS_FILE = (
    METRICS_PATH
    / "blued_supervised_random_forest_metrics.csv"
)

RANDOM_FOREST_THRESHOLD_FILE = (
    METRICS_PATH
    / "blued_supervised_random_forest_validation_thresholds.csv"
)

RANDOM_FOREST_VALIDATION_PREDICTIONS_FILE = (
    PREDICTIONS_PATH
    / "blued_supervised_random_forest_validation_predictions.csv"
)

RANDOM_FOREST_TEST_PREDICTIONS_FILE = (
    PREDICTIONS_PATH
    / "blued_supervised_random_forest_test_predictions.csv"
)

RANDOM_FOREST_IMPORTANCE_FILE = (
    METRICS_PATH
    / "blued_supervised_random_forest_feature_importance.csv"
)

RANDOM_FOREST_SUMMARY_FILE = (
    LOG_PATH
    / "blued_supervised_random_forest_summary.json"
)

# ------------------------------------------------------------------
# 11. Save the trained model and all outputs
# ------------------------------------------------------------------

joblib.dump(
    random_forest_model,
    RANDOM_FOREST_MODEL_FILE,
)

random_forest_metrics_df.to_csv(
    RANDOM_FOREST_METRICS_FILE,
    index=False,
)

random_forest_threshold_table.to_csv(
    RANDOM_FOREST_THRESHOLD_FILE,
    index=False,
)

random_forest_validation_predictions.to_csv(
    RANDOM_FOREST_VALIDATION_PREDICTIONS_FILE,
    index=False,
)

random_forest_test_predictions.to_csv(
    RANDOM_FOREST_TEST_PREDICTIONS_FILE,
    index=False,
)

random_forest_feature_importance_df.to_csv(
    RANDOM_FOREST_IMPORTANCE_FILE,
    index=False,
)

random_forest_summary = {
    "experiment_id": EXPERIMENT_ID,
    "model": "Random Forest",
    "configuration": {
        "n_estimators": 300,
        "class_weight": "balanced_subsample",
        "random_state": RANDOM_SEED,
        "n_jobs": -1,
        "preprocessing": (
            "Median imputation fitted on training data only; "
            "no feature scaling"
        ),
    },
    "selected_validation_threshold": float(
        selected_threshold
    ),
    "validation_performance_at_selected_threshold": (
        validation_selected_result
    ),
    "test_performance_default_threshold": (
        test_default_result
    ),
    "test_performance_selected_threshold": (
        test_selected_result
    ),
    "training_seconds": float(
        random_forest_training_seconds
    ),
    "inference_seconds": float(
        random_forest_inference_seconds
    ),
    "top_10_features": (
        random_forest_feature_importance_df
        .head(10)
        .to_dict(orient="records")
    ),
}

with open(
    RANDOM_FOREST_SUMMARY_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        random_forest_summary,
        file,
        indent=2,
    )

# ------------------------------------------------------------------
# 12. Verify saved artifacts
# ------------------------------------------------------------------

random_forest_output_files = [
    RANDOM_FOREST_MODEL_FILE,
    RANDOM_FOREST_METRICS_FILE,
    RANDOM_FOREST_THRESHOLD_FILE,
    RANDOM_FOREST_VALIDATION_PREDICTIONS_FILE,
    RANDOM_FOREST_TEST_PREDICTIONS_FILE,
    RANDOM_FOREST_IMPORTANCE_FILE,
    RANDOM_FOREST_SUMMARY_FILE,
]

for file_path in random_forest_output_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Random Forest output was not created: "
            f"{file_path}"
        )

print("\nSaved Random Forest artifacts:")

for file_path in random_forest_output_files:
    print(
        f"{file_path.name:65s} "
        f"| {file_path.stat().st_size / 1024:,.2f} KB"
    )

# ------------------------------------------------------------------
# 13. Display the main Random Forest results
# ------------------------------------------------------------------

print("\nRandom Forest validation-threshold selection:")
print(
    f"Selected threshold:     "
    f"{selected_threshold:.6f}"
)
print(
    f"Validation Precision:   "
    f"{validation_selected_result['precision']:.6f}"
)
print(
    f"Validation Recall:      "
    f"{validation_selected_result['recall']:.6f}"
)
print(
    f"Validation F1-Score:    "
    f"{validation_selected_result['f1_score']:.6f}"
)

print("\nRandom Forest test results:")
display(
    random_forest_metrics_df.loc[
        random_forest_metrics_df["split"] == "test",
        [
            "threshold_source",
            "threshold",
            "precision",
            "recall",
            "f1_score",
            "false_positive_rate",
            "roc_auc",
            "pr_auc",
            "tn",
            "fp",
            "fn",
            "tp",
            "flagged_ratio_percent",
        ],
    ]
)

print("\nTop 10 Random Forest features:")
display(
    random_forest_feature_importance_df.head(10)
)

print(
    "\nSupervised BLUED Random Forest experiment "
    "completed successfully."
)

In [ ]:
# Step 11 — Train and evaluate the supervised XGBoost model

import json
import time
import joblib

import numpy as np
import pandas as pd

from xgboost import XGBClassifier

# ------------------------------------------------------------------
# 1. Create XGBoost using the frozen experiment configuration
# ------------------------------------------------------------------

xgboost_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

print("XGBoost configuration:")
print(xgboost_model)
print()

# ------------------------------------------------------------------
# 2. Train the model using the training partition only
# ------------------------------------------------------------------

training_start_time = time.perf_counter()

xgboost_model.fit(
    X_train_tree,
    y_train,
    eval_set=[
        (X_val_tree, y_val),
    ],
    verbose=False,
)

xgboost_training_seconds = (
    time.perf_counter()
    - training_start_time
)

print(
    "XGBoost training completed in "
    f"{xgboost_training_seconds:,.4f} seconds."
)

# ------------------------------------------------------------------
# 3. Generate validation and test probability scores
# ------------------------------------------------------------------

inference_start_time = time.perf_counter()

xgboost_validation_scores = (
    xgboost_model.predict_proba(
        X_val_tree
    )[:, 1]
)

xgboost_test_scores = (
    xgboost_model.predict_proba(
        X_test_tree
    )[:, 1]
)

xgboost_inference_seconds = (
    time.perf_counter()
    - inference_start_time
)

if len(xgboost_validation_scores) != len(y_val):
    raise ValueError(
        "XGBoost validation-score length does not "
        "match the validation labels."
    )

if len(xgboost_test_scores) != len(y_test):
    raise ValueError(
        "XGBoost test-score length does not "
        "match the test labels."
    )

if not np.isfinite(
    xgboost_validation_scores
).all():
    raise ValueError(
        "XGBoost validation scores contain "
        "non-finite values."
    )

if not np.isfinite(
    xgboost_test_scores
).all():
    raise ValueError(
        "XGBoost test scores contain "
        "non-finite values."
    )

print(
    "Probability-score generation completed in "
    f"{xgboost_inference_seconds:,.4f} seconds."
)

print(
    "Validation score range: "
    f"{xgboost_validation_scores.min():.6f} "
    f"to {xgboost_validation_scores.max():.6f}"
)

print(
    "Test score range:       "
    f"{xgboost_test_scores.min():.6f} "
    f"to {xgboost_test_scores.max():.6f}"
)

# ------------------------------------------------------------------
# 4. Apply the common two-threshold evaluation protocol
# ------------------------------------------------------------------

xgboost_evaluation = (
    evaluate_model_with_two_thresholds(
        model_name="XGBoost",
        validation_scores=(
            xgboost_validation_scores
        ),
        test_scores=xgboost_test_scores,
        y_validation=y_val,
        y_test_values=y_test,
    )
)

xgboost_selected_threshold = (
    xgboost_evaluation[
        "threshold_selection"
    ]["threshold"]
)

xgboost_validation_selected_result = (
    xgboost_evaluation[
        "validation_selected_result"
    ].copy()
)

xgboost_test_default_result = (
    xgboost_evaluation[
        "test_default_result"
    ].copy()
)

xgboost_test_selected_result = (
    xgboost_evaluation[
        "test_selected_result"
    ].copy()
)

# Add computational and representation information
for result in [
    xgboost_validation_selected_result,
    xgboost_test_default_result,
    xgboost_test_selected_result,
]:
    result["training_seconds"] = float(
        xgboost_training_seconds
    )

    result["inference_seconds"] = float(
        xgboost_inference_seconds
    )

    result["n_features"] = int(
        len(feature_columns)
    )

    result["n_train"] = int(
        len(y_train)
    )

    result["positive_train"] = int(
        y_train.sum()
    )

    result["scale_pos_weight"] = float(
        scale_pos_weight
    )

# ------------------------------------------------------------------
# 5. Prepare the complete XGBoost metrics table
# ------------------------------------------------------------------

xgboost_metrics_df = pd.DataFrame(
    [
        xgboost_validation_selected_result,
        xgboost_test_default_result,
        xgboost_test_selected_result,
    ]
)

xgboost_metric_columns = [
    "experiment_id",
    "dataset",
    "task",
    "model",
    "split",
    "threshold_source",
    "threshold",
    "precision",
    "recall",
    "f1_score",
    "false_positive_rate",
    "roc_auc",
    "pr_auc",
    "tn",
    "fp",
    "fn",
    "tp",
    "flagged_windows",
    "flagged_ratio",
    "flagged_ratio_percent",
    "positive_windows",
    "negative_windows",
    "positive_prevalence",
    "n_windows",
    "n_train",
    "positive_train",
    "n_features",
    "scale_pos_weight",
    "training_seconds",
    "inference_seconds",
]

xgboost_metrics_df = (
    xgboost_metrics_df[
        xgboost_metric_columns
    ]
)

# ------------------------------------------------------------------
# 6. Prepare the validation-threshold audit table
# ------------------------------------------------------------------

xgboost_threshold_table = (
    xgboost_evaluation[
        "threshold_table"
    ].copy()
)

xgboost_threshold_table.insert(
    0,
    "experiment_id",
    EXPERIMENT_ID,
)

xgboost_threshold_table.insert(
    1,
    "model",
    "XGBoost",
)

xgboost_threshold_table.insert(
    2,
    "split",
    "validation",
)

# ------------------------------------------------------------------
# 7. Prepare validation predictions
# ------------------------------------------------------------------

xgboost_validation_predictions = (
    val_df[
        [
            "model_window_id",
            "window_id",
            "recording_block",
            "window_start",
            "window_end",
            "event_count",
            "event_window_label",
            "split",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

xgboost_validation_predictions[
    "model"
] = "XGBoost"

xgboost_validation_predictions[
    "probability_score"
] = xgboost_validation_scores

xgboost_validation_predictions[
    "selected_threshold"
] = xgboost_selected_threshold

xgboost_validation_predictions[
    "predicted_label_selected_threshold"
] = (
    xgboost_validation_scores
    >= xgboost_selected_threshold
).astype(np.int64)

# ------------------------------------------------------------------
# 8. Prepare complete test predictions
# ------------------------------------------------------------------

xgboost_test_predictions = (
    test_df[
        [
            "model_window_id",
            "window_id",
            "recording_block",
            "window_start",
            "window_end",
            "event_count",
            "event_window_label",
            "split",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

xgboost_test_predictions[
    "model"
] = "XGBoost"

xgboost_test_predictions[
    "probability_score"
] = xgboost_test_scores

xgboost_test_predictions[
    "default_threshold"
] = 0.5

xgboost_test_predictions[
    "predicted_label_default_0.5"
] = (
    xgboost_test_scores >= 0.5
).astype(np.int64)

xgboost_test_predictions[
    "validation_selected_threshold"
] = xgboost_selected_threshold

xgboost_test_predictions[
    "predicted_label_validation_f1_optimized"
] = (
    xgboost_test_scores
    >= xgboost_selected_threshold
).astype(np.int64)

# ------------------------------------------------------------------
# 9. Extract XGBoost feature importance
# ------------------------------------------------------------------

xgboost_feature_importance_df = (
    pd.DataFrame(
        {
            "feature": feature_columns,
            "importance": (
                xgboost_model.feature_importances_
            ),
        }
    )
    .sort_values(
        "importance",
        ascending=False,
    )
    .reset_index(drop=True)
)

xgboost_feature_importance_df[
    "rank"
] = np.arange(
    1,
    len(xgboost_feature_importance_df) + 1,
)

xgboost_feature_importance_df = (
    xgboost_feature_importance_df[
        [
            "rank",
            "feature",
            "importance",
        ]
    ]
)

# ------------------------------------------------------------------
# 10. Extract the training evaluation history
# ------------------------------------------------------------------

xgboost_evaluation_history = (
    xgboost_model.evals_result()
)

validation_logloss_values = (
    xgboost_evaluation_history[
        "validation_0"
    ]["logloss"]
)

xgboost_training_history_df = pd.DataFrame(
    {
        "boosting_round": np.arange(
            1,
            len(validation_logloss_values) + 1,
        ),
        "validation_logloss": (
            validation_logloss_values
        ),
    }
)

# ------------------------------------------------------------------
# 11. Define output files
# ------------------------------------------------------------------

XGBOOST_MODEL_FILE = (
    MODELS_PATH
    / "blued_supervised_xgboost.json"
)

XGBOOST_JOBLIB_FILE = (
    MODELS_PATH
    / "blued_supervised_xgboost.joblib"
)

XGBOOST_METRICS_FILE = (
    METRICS_PATH
    / "blued_supervised_xgboost_metrics.csv"
)

XGBOOST_THRESHOLD_FILE = (
    METRICS_PATH
    / "blued_supervised_xgboost_validation_thresholds.csv"
)

XGBOOST_VALIDATION_PREDICTIONS_FILE = (
    PREDICTIONS_PATH
    / "blued_supervised_xgboost_validation_predictions.csv"
)

XGBOOST_TEST_PREDICTIONS_FILE = (
    PREDICTIONS_PATH
    / "blued_supervised_xgboost_test_predictions.csv"
)

XGBOOST_IMPORTANCE_FILE = (
    METRICS_PATH
    / "blued_supervised_xgboost_feature_importance.csv"
)

XGBOOST_HISTORY_FILE = (
    METRICS_PATH
    / "blued_supervised_xgboost_training_history.csv"
)

XGBOOST_SUMMARY_FILE = (
    LOG_PATH
    / "blued_supervised_xgboost_summary.json"
)

# ------------------------------------------------------------------
# 12. Save the trained model and complete outputs
# ------------------------------------------------------------------

xgboost_model.save_model(
    XGBOOST_MODEL_FILE
)

joblib.dump(
    xgboost_model,
    XGBOOST_JOBLIB_FILE,
)

xgboost_metrics_df.to_csv(
    XGBOOST_METRICS_FILE,
    index=False,
)

xgboost_threshold_table.to_csv(
    XGBOOST_THRESHOLD_FILE,
    index=False,
)

xgboost_validation_predictions.to_csv(
    XGBOOST_VALIDATION_PREDICTIONS_FILE,
    index=False,
)

xgboost_test_predictions.to_csv(
    XGBOOST_TEST_PREDICTIONS_FILE,
    index=False,
)

xgboost_feature_importance_df.to_csv(
    XGBOOST_IMPORTANCE_FILE,
    index=False,
)

xgboost_training_history_df.to_csv(
    XGBOOST_HISTORY_FILE,
    index=False,
)

xgboost_summary = {
    "experiment_id": EXPERIMENT_ID,
    "model": "XGBoost",
    "configuration": {
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "scale_pos_weight": float(
            scale_pos_weight
        ),
        "random_state": RANDOM_SEED,
        "n_jobs": -1,
        "preprocessing": (
            "Median imputation fitted on training data only; "
            "no feature scaling"
        ),
    },
    "selected_validation_threshold": float(
        xgboost_selected_threshold
    ),
    "validation_performance_at_selected_threshold": (
        xgboost_validation_selected_result
    ),
    "test_performance_default_threshold": (
        xgboost_test_default_result
    ),
    "test_performance_selected_threshold": (
        xgboost_test_selected_result
    ),
    "training_seconds": float(
        xgboost_training_seconds
    ),
    "inference_seconds": float(
        xgboost_inference_seconds
    ),
    "final_validation_logloss": float(
        validation_logloss_values[-1]
    ),
    "minimum_validation_logloss": float(
        np.min(validation_logloss_values)
    ),
    "minimum_validation_logloss_round": int(
        np.argmin(validation_logloss_values) + 1
    ),
    "top_10_features": (
        xgboost_feature_importance_df
        .head(10)
        .to_dict(orient="records")
    ),
}

with open(
    XGBOOST_SUMMARY_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        xgboost_summary,
        file,
        indent=2,
    )

# ------------------------------------------------------------------
# 13. Verify saved XGBoost artifacts
# ------------------------------------------------------------------

xgboost_output_files = [
    XGBOOST_MODEL_FILE,
    XGBOOST_JOBLIB_FILE,
    XGBOOST_METRICS_FILE,
    XGBOOST_THRESHOLD_FILE,
    XGBOOST_VALIDATION_PREDICTIONS_FILE,
    XGBOOST_TEST_PREDICTIONS_FILE,
    XGBOOST_IMPORTANCE_FILE,
    XGBOOST_HISTORY_FILE,
    XGBOOST_SUMMARY_FILE,
]

for file_path in xgboost_output_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"XGBoost output was not created: "
            f"{file_path}"
        )

print("\nSaved XGBoost artifacts:")

for file_path in xgboost_output_files:
    print(
        f"{file_path.name:65s} "
        f"| {file_path.stat().st_size / 1024:,.2f} KB"
    )

# ------------------------------------------------------------------
# 14. Display the main XGBoost results
# ------------------------------------------------------------------

print("\nXGBoost validation-threshold selection:")
print(
    f"Selected threshold:     "
    f"{xgboost_selected_threshold:.6f}"
)
print(
    f"Validation Precision:   "
    f"{xgboost_validation_selected_result['precision']:.6f}"
)
print(
    f"Validation Recall:      "
    f"{xgboost_validation_selected_result['recall']:.6f}"
)
print(
    f"Validation F1-Score:    "
    f"{xgboost_validation_selected_result['f1_score']:.6f}"
)

print("\nXGBoost validation log-loss:")
print(
    f"Final round:            "
    f"{validation_logloss_values[-1]:.6f}"
)
print(
    f"Minimum value:          "
    f"{np.min(validation_logloss_values):.6f}"
)
print(
    f"Minimum-value round:    "
    f"{np.argmin(validation_logloss_values) + 1}"
)

print("\nXGBoost test results:")
display(
    xgboost_metrics_df.loc[
        xgboost_metrics_df["split"] == "test",
        [
            "threshold_source",
            "threshold",
            "precision",
            "recall",
            "f1_score",
            "false_positive_rate",
            "roc_auc",
            "pr_auc",
            "tn",
            "fp",
            "fn",
            "tp",
            "flagged_ratio_percent",
        ],
    ]
)

print("\nTop 10 XGBoost features:")
display(
    xgboost_feature_importance_df.head(10)
)

print(
    "\nSupervised BLUED XGBoost experiment "
    "completed successfully."
)

In [ ]:
# Step 12 — Train and evaluate the supervised Logistic Regression model

import json
import time
import joblib
import warnings

import numpy as np
import pandas as pd

from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------------
# 1. Create Logistic Regression using the frozen configuration
# ------------------------------------------------------------------

logistic_regression_model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=RANDOM_SEED,
)

print("Logistic Regression configuration:")
print(logistic_regression_model)
print()

# ------------------------------------------------------------------
# 2. Train using standardized training features only
# ------------------------------------------------------------------

training_start_time = time.perf_counter()

with warnings.catch_warnings(record=True) as captured_warnings:
    warnings.simplefilter(
        "always",
        category=ConvergenceWarning,
    )

    logistic_regression_model.fit(
        X_train_scaled,
        y_train,
    )

logistic_regression_training_seconds = (
    time.perf_counter()
    - training_start_time
)

convergence_warnings = [
    str(warning.message)
    for warning in captured_warnings
    if issubclass(
        warning.category,
        ConvergenceWarning,
    )
]

print(
    "Logistic Regression training completed in "
    f"{logistic_regression_training_seconds:,.4f} seconds."
)

print(
    "Optimization iterations used: "
    f"{int(logistic_regression_model.n_iter_[0]):,}"
)

if convergence_warnings:
    print("\nConvergence warnings detected:")

    for warning_message in convergence_warnings:
        print(f"  - {warning_message}")
else:
    print("No convergence warnings were detected.")

# ------------------------------------------------------------------
# 3. Generate validation and test probability scores
# ------------------------------------------------------------------

inference_start_time = time.perf_counter()

logistic_validation_scores = (
    logistic_regression_model.predict_proba(
        X_val_scaled
    )[:, 1]
)

logistic_test_scores = (
    logistic_regression_model.predict_proba(
        X_test_scaled
    )[:, 1]
)

logistic_regression_inference_seconds = (
    time.perf_counter()
    - inference_start_time
)

if len(logistic_validation_scores) != len(y_val):
    raise ValueError(
        "Logistic Regression validation-score length does not "
        "match the validation labels."
    )

if len(logistic_test_scores) != len(y_test):
    raise ValueError(
        "Logistic Regression test-score length does not "
        "match the test labels."
    )

if not np.isfinite(
    logistic_validation_scores
).all():
    raise ValueError(
        "Logistic Regression validation scores contain "
        "non-finite values."
    )

if not np.isfinite(
    logistic_test_scores
).all():
    raise ValueError(
        "Logistic Regression test scores contain "
        "non-finite values."
    )

print(
    "Probability-score generation completed in "
    f"{logistic_regression_inference_seconds:,.4f} seconds."
)

print(
    "Validation score range: "
    f"{logistic_validation_scores.min():.6f} "
    f"to {logistic_validation_scores.max():.6f}"
)

print(
    "Test score range:       "
    f"{logistic_test_scores.min():.6f} "
    f"to {logistic_test_scores.max():.6f}"
)

# ------------------------------------------------------------------
# 4. Apply the common two-threshold evaluation protocol
# ------------------------------------------------------------------

logistic_evaluation = (
    evaluate_model_with_two_thresholds(
        model_name="Logistic Regression",
        validation_scores=(
            logistic_validation_scores
        ),
        test_scores=logistic_test_scores,
        y_validation=y_val,
        y_test_values=y_test,
    )
)

logistic_selected_threshold = (
    logistic_evaluation[
        "threshold_selection"
    ]["threshold"]
)

logistic_validation_selected_result = (
    logistic_evaluation[
        "validation_selected_result"
    ].copy()
)

logistic_test_default_result = (
    logistic_evaluation[
        "test_default_result"
    ].copy()
)

logistic_test_selected_result = (
    logistic_evaluation[
        "test_selected_result"
    ].copy()
)

for result in [
    logistic_validation_selected_result,
    logistic_test_default_result,
    logistic_test_selected_result,
]:
    result["training_seconds"] = float(
        logistic_regression_training_seconds
    )

    result["inference_seconds"] = float(
        logistic_regression_inference_seconds
    )

    result["n_features"] = int(
        len(feature_columns)
    )

    result["n_train"] = int(
        len(y_train)
    )

    result["positive_train"] = int(
        y_train.sum()
    )

    result["optimization_iterations"] = int(
        logistic_regression_model.n_iter_[0]
    )

# ------------------------------------------------------------------
# 5. Prepare the Logistic Regression metrics table
# ------------------------------------------------------------------

logistic_metrics_df = pd.DataFrame(
    [
        logistic_validation_selected_result,
        logistic_test_default_result,
        logistic_test_selected_result,
    ]
)

logistic_metric_columns = [
    "experiment_id",
    "dataset",
    "task",
    "model",
    "split",
    "threshold_source",
    "threshold",
    "precision",
    "recall",
    "f1_score",
    "false_positive_rate",
    "roc_auc",
    "pr_auc",
    "tn",
    "fp",
    "fn",
    "tp",
    "flagged_windows",
    "flagged_ratio",
    "flagged_ratio_percent",
    "positive_windows",
    "negative_windows",
    "positive_prevalence",
    "n_windows",
    "n_train",
    "positive_train",
    "n_features",
    "optimization_iterations",
    "training_seconds",
    "inference_seconds",
]

logistic_metrics_df = logistic_metrics_df[
    logistic_metric_columns
]

# ------------------------------------------------------------------
# 6. Prepare the validation-threshold audit table
# ------------------------------------------------------------------

logistic_threshold_table = (
    logistic_evaluation[
        "threshold_table"
    ].copy()
)

logistic_threshold_table.insert(
    0,
    "experiment_id",
    EXPERIMENT_ID,
)

logistic_threshold_table.insert(
    1,
    "model",
    "Logistic Regression",
)

logistic_threshold_table.insert(
    2,
    "split",
    "validation",
)

# ------------------------------------------------------------------
# 7. Prepare validation predictions
# ------------------------------------------------------------------

logistic_validation_predictions = (
    val_df[
        [
            "model_window_id",
            "window_id",
            "recording_block",
            "window_start",
            "window_end",
            "event_count",
            "event_window_label",
            "split",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

logistic_validation_predictions[
    "model"
] = "Logistic Regression"

logistic_validation_predictions[
    "probability_score"
] = logistic_validation_scores

logistic_validation_predictions[
    "selected_threshold"
] = logistic_selected_threshold

logistic_validation_predictions[
    "predicted_label_selected_threshold"
] = (
    logistic_validation_scores
    >= logistic_selected_threshold
).astype(np.int64)

# ------------------------------------------------------------------
# 8. Prepare complete test predictions
# ------------------------------------------------------------------

logistic_test_predictions = (
    test_df[
        [
            "model_window_id",
            "window_id",
            "recording_block",
            "window_start",
            "window_end",
            "event_count",
            "event_window_label",
            "split",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

logistic_test_predictions[
    "model"
] = "Logistic Regression"

logistic_test_predictions[
    "probability_score"
] = logistic_test_scores

logistic_test_predictions[
    "default_threshold"
] = 0.5

logistic_test_predictions[
    "predicted_label_default_0.5"
] = (
    logistic_test_scores >= 0.5
).astype(np.int64)

logistic_test_predictions[
    "validation_selected_threshold"
] = logistic_selected_threshold

logistic_test_predictions[
    "predicted_label_validation_f1_optimized"
] = (
    logistic_test_scores
    >= logistic_selected_threshold
).astype(np.int64)

# ------------------------------------------------------------------
# 9. Extract standardized Logistic Regression coefficients
# ------------------------------------------------------------------

logistic_coefficients = (
    logistic_regression_model.coef_[0]
)

logistic_coefficient_df = pd.DataFrame(
    {
        "feature": feature_columns,
        "coefficient": logistic_coefficients,
        "absolute_coefficient": np.abs(
            logistic_coefficients
        ),
        "direction": np.where(
            logistic_coefficients > 0,
            "positive_event_association",
            np.where(
                logistic_coefficients < 0,
                "negative_event_association",
                "no_linear_association",
            ),
        ),
        "odds_ratio_per_standard_deviation": np.exp(
            np.clip(
                logistic_coefficients,
                -50,
                50,
            )
        ),
    }
)

logistic_coefficient_df = (
    logistic_coefficient_df
    .sort_values(
        "absolute_coefficient",
        ascending=False,
    )
    .reset_index(drop=True)
)

logistic_coefficient_df[
    "rank"
] = np.arange(
    1,
    len(logistic_coefficient_df) + 1,
)

logistic_coefficient_df = (
    logistic_coefficient_df[
        [
            "rank",
            "feature",
            "coefficient",
            "absolute_coefficient",
            "direction",
            "odds_ratio_per_standard_deviation",
        ]
    ]
)

# ------------------------------------------------------------------
# 10. Define output files
# ------------------------------------------------------------------

LOGISTIC_MODEL_FILE = (
    MODELS_PATH
    / "blued_supervised_logistic_regression.joblib"
)

LOGISTIC_METRICS_FILE = (
    METRICS_PATH
    / "blued_supervised_logistic_regression_metrics.csv"
)

LOGISTIC_THRESHOLD_FILE = (
    METRICS_PATH
    / "blued_supervised_logistic_regression_validation_thresholds.csv"
)

LOGISTIC_VALIDATION_PREDICTIONS_FILE = (
    PREDICTIONS_PATH
    / "blued_supervised_logistic_regression_validation_predictions.csv"
)

LOGISTIC_TEST_PREDICTIONS_FILE = (
    PREDICTIONS_PATH
    / "blued_supervised_logistic_regression_test_predictions.csv"
)

LOGISTIC_COEFFICIENT_FILE = (
    METRICS_PATH
    / "blued_supervised_logistic_regression_coefficients.csv"
)

LOGISTIC_SUMMARY_FILE = (
    LOG_PATH
    / "blued_supervised_logistic_regression_summary.json"
)

# ------------------------------------------------------------------
# 11. Save the model and all Logistic Regression outputs
# ------------------------------------------------------------------

joblib.dump(
    logistic_regression_model,
    LOGISTIC_MODEL_FILE,
)

logistic_metrics_df.to_csv(
    LOGISTIC_METRICS_FILE,
    index=False,
)

logistic_threshold_table.to_csv(
    LOGISTIC_THRESHOLD_FILE,
    index=False,
)

logistic_validation_predictions.to_csv(
    LOGISTIC_VALIDATION_PREDICTIONS_FILE,
    index=False,
)

logistic_test_predictions.to_csv(
    LOGISTIC_TEST_PREDICTIONS_FILE,
    index=False,
)

logistic_coefficient_df.to_csv(
    LOGISTIC_COEFFICIENT_FILE,
    index=False,
)

logistic_summary = {
    "experiment_id": EXPERIMENT_ID,
    "model": "Logistic Regression",
    "configuration": {
        "max_iter": 3000,
        "class_weight": "balanced",
        "random_state": RANDOM_SEED,
        "solver": (
            logistic_regression_model.solver
        ),
        "regularization": (
            logistic_regression_model.penalty
        ),
        "inverse_regularization_strength_C": float(
            logistic_regression_model.C
        ),
        "preprocessing": (
            "Median imputation and StandardScaler fitted "
            "using training data only"
        ),
    },
    "convergence": {
        "optimization_iterations": int(
            logistic_regression_model.n_iter_[0]
        ),
        "convergence_warning_count": len(
            convergence_warnings
        ),
        "convergence_warnings": convergence_warnings,
    },
    "intercept": float(
        logistic_regression_model.intercept_[0]
    ),
    "selected_validation_threshold": float(
        logistic_selected_threshold
    ),
    "validation_performance_at_selected_threshold": (
        logistic_validation_selected_result
    ),
    "test_performance_default_threshold": (
        logistic_test_default_result
    ),
    "test_performance_selected_threshold": (
        logistic_test_selected_result
    ),
    "training_seconds": float(
        logistic_regression_training_seconds
    ),
    "inference_seconds": float(
        logistic_regression_inference_seconds
    ),
    "top_10_absolute_coefficients": (
        logistic_coefficient_df
        .head(10)
        .to_dict(orient="records")
    ),
}

with open(
    LOGISTIC_SUMMARY_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        logistic_summary,
        file,
        indent=2,
    )

# ------------------------------------------------------------------
# 12. Verify saved Logistic Regression artifacts
# ------------------------------------------------------------------

logistic_output_files = [
    LOGISTIC_MODEL_FILE,
    LOGISTIC_METRICS_FILE,
    LOGISTIC_THRESHOLD_FILE,
    LOGISTIC_VALIDATION_PREDICTIONS_FILE,
    LOGISTIC_TEST_PREDICTIONS_FILE,
    LOGISTIC_COEFFICIENT_FILE,
    LOGISTIC_SUMMARY_FILE,
]

for file_path in logistic_output_files:
    if not file_path.exists():
        raise FileNotFoundError(
            "Logistic Regression output was not created: "
            f"{file_path}"
        )

print("\nSaved Logistic Regression artifacts:")

for file_path in logistic_output_files:
    print(
        f"{file_path.name:65s} "
        f"| {file_path.stat().st_size / 1024:,.2f} KB"
    )

# ------------------------------------------------------------------
# 13. Display the main Logistic Regression results
# ------------------------------------------------------------------

print("\nLogistic Regression validation-threshold selection:")
print(
    f"Selected threshold:     "
    f"{logistic_selected_threshold:.6f}"
)
print(
    f"Validation Precision:   "
    f"{logistic_validation_selected_result['precision']:.6f}"
)
print(
    f"Validation Recall:      "
    f"{logistic_validation_selected_result['recall']:.6f}"
)
print(
    f"Validation F1-Score:    "
    f"{logistic_validation_selected_result['f1_score']:.6f}"
)

print("\nLogistic Regression test results:")

display(
    logistic_metrics_df.loc[
        logistic_metrics_df["split"] == "test",
        [
            "threshold_source",
            "threshold",
            "precision",
            "recall",
            "f1_score",
            "false_positive_rate",
            "roc_auc",
            "pr_auc",
            "tn",
            "fp",
            "fn",
            "tp",
            "flagged_ratio_percent",
        ],
    ]
)

print(
    "\nTop 10 standardized Logistic Regression "
    "coefficients by absolute magnitude:"
)

display(
    logistic_coefficient_df.head(10)
)

print(
    "\nSupervised BLUED Logistic Regression experiment "
    "completed successfully."
)

In [ ]:
# Step 13 — Train and evaluate the supervised Multilayer Perceptron model

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

# ------------------------------------------------------------------
# 1. Reset the TensorFlow session and reproducibility state
# ------------------------------------------------------------------

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(RANDOM_SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

# ------------------------------------------------------------------
# 2. Build the frozen MLP architecture
# ------------------------------------------------------------------

mlp_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(
            shape=(len(feature_columns),),
            name="blued_69_feature_input",
        ),
        tf.keras.layers.Dense(
            64,
            activation="relu",
            name="hidden_layer_64",
        ),
        tf.keras.layers.Dropout(
            0.2,
            name="dropout_after_64",
        ),
        tf.keras.layers.Dense(
            32,
            activation="relu",
            name="hidden_layer_32",
        ),
        tf.keras.layers.Dropout(
            0.2,
            name="dropout_after_32",
        ),
        tf.keras.layers.Dense(
            1,
            activation="sigmoid",
            name="event_probability_output",
        ),
    ],
    name="blued_supervised_mlp",
)

mlp_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001,
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="binary_accuracy",
        ),
        tf.keras.metrics.Precision(
            name="precision",
        ),
        tf.keras.metrics.Recall(
            name="recall",
        ),
        tf.keras.metrics.AUC(
            name="roc_auc",
            curve="ROC",
        ),
        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR",
        ),
    ],
)

print("MLP architecture:")
mlp_model.summary()

# ------------------------------------------------------------------
# 3. Define class weights and early stopping
# ------------------------------------------------------------------

mlp_class_weights = {
    0: float(class_weight_event_0),
    1: float(class_weight_event_1),
}

early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

print("\nMLP class weights:")
print(f"Class 0: {mlp_class_weights[0]:.6f}")
print(f"Class 1: {mlp_class_weights[1]:.6f}")

# ------------------------------------------------------------------
# 4. Train on standardized training features only
# ------------------------------------------------------------------

training_start_time = time.perf_counter()

mlp_history = mlp_model.fit(
    X_train_scaled_float32,
    y_train.astype(np.float32),
    validation_data=(
        X_val_scaled_float32,
        y_val.astype(np.float32),
    ),
    epochs=50,
    batch_size=256,
    class_weight=mlp_class_weights,
    callbacks=[
        early_stopping_callback,
    ],
    shuffle=True,
    verbose=2,
)

mlp_training_seconds = (
    time.perf_counter()
    - training_start_time
)

epochs_completed = len(
    mlp_history.history["loss"]
)

best_epoch = int(
    np.argmin(
        mlp_history.history["val_loss"]
    )
    + 1
)

best_validation_loss = float(
    np.min(
        mlp_history.history["val_loss"]
    )
)

print(
    "\nMLP training completed in "
    f"{mlp_training_seconds:,.4f} seconds."
)

print(
    f"Epochs completed:       {epochs_completed:,}"
)

print(
    f"Best validation epoch: {best_epoch:,}"
)

print(
    "Best validation loss:  "
    f"{best_validation_loss:.6f}"
)

# ------------------------------------------------------------------
# 5. Generate validation and test probability scores
# ------------------------------------------------------------------

inference_start_time = time.perf_counter()

mlp_validation_scores = (
    mlp_model.predict(
        X_val_scaled_float32,
        batch_size=256,
        verbose=0,
    )
    .reshape(-1)
    .astype(np.float64)
)

mlp_test_scores = (
    mlp_model.predict(
        X_test_scaled_float32,
        batch_size=256,
        verbose=0,
    )
    .reshape(-1)
    .astype(np.float64)
)

mlp_inference_seconds = (
    time.perf_counter()
    - inference_start_time
)

if len(mlp_validation_scores) != len(y_val):
    raise ValueError(
        "MLP validation-score length does not match "
        "the validation labels."
    )

if len(mlp_test_scores) != len(y_test):
    raise ValueError(
        "MLP test-score length does not match "
        "the test labels."
    )

if not np.isfinite(
    mlp_validation_scores
).all():
    raise ValueError(
        "MLP validation scores contain non-finite values."
    )

if not np.isfinite(
    mlp_test_scores
).all():
    raise ValueError(
        "MLP test scores contain non-finite values."
    )

print(
    "Probability-score generation completed in "
    f"{mlp_inference_seconds:,.4f} seconds."
)

print(
    "Validation score range: "
    f"{mlp_validation_scores.min():.6f} "
    f"to {mlp_validation_scores.max():.6f}"
)

print(
    "Test score range:       "
    f"{mlp_test_scores.min():.6f} "
    f"to {mlp_test_scores.max():.6f}"
)

# ------------------------------------------------------------------
# 6. Apply the common two-threshold evaluation protocol
# ------------------------------------------------------------------

mlp_evaluation = (
    evaluate_model_with_two_thresholds(
        model_name="MLP",
        validation_scores=mlp_validation_scores,
        test_scores=mlp_test_scores,
        y_validation=y_val,
        y_test_values=y_test,
    )
)

mlp_selected_threshold = (
    mlp_evaluation[
        "threshold_selection"
    ]["threshold"]
)

mlp_validation_selected_result = (
    mlp_evaluation[
        "validation_selected_result"
    ].copy()
)

mlp_test_default_result = (
    mlp_evaluation[
        "test_default_result"
    ].copy()
)

mlp_test_selected_result = (
    mlp_evaluation[
        "test_selected_result"
    ].copy()
)

for result in [
    mlp_validation_selected_result,
    mlp_test_default_result,
    mlp_test_selected_result,
]:
    result["training_seconds"] = float(
        mlp_training_seconds
    )

    result["inference_seconds"] = float(
        mlp_inference_seconds
    )

    result["n_features"] = int(
        len(feature_columns)
    )

    result["n_train"] = int(
        len(y_train)
    )

    result["positive_train"] = int(
        y_train.sum()
    )

    result["epochs_completed"] = int(
        epochs_completed
    )

    result["best_validation_epoch"] = int(
        best_epoch
    )

    result["best_validation_loss"] = float(
        best_validation_loss
    )

# ------------------------------------------------------------------
# 7. Prepare the MLP metrics table
# ------------------------------------------------------------------

mlp_metrics_df = pd.DataFrame(
    [
        mlp_validation_selected_result,
        mlp_test_default_result,
        mlp_test_selected_result,
    ]
)

mlp_metric_columns = [
    "experiment_id",
    "dataset",
    "task",
    "model",
    "split",
    "threshold_source",
    "threshold",
    "precision",
    "recall",
    "f1_score",
    "false_positive_rate",
    "roc_auc",
    "pr_auc",
    "tn",
    "fp",
    "fn",
    "tp",
    "flagged_windows",
    "flagged_ratio",
    "flagged_ratio_percent",
    "positive_windows",
    "negative_windows",
    "positive_prevalence",
    "n_windows",
    "n_train",
    "positive_train",
    "n_features",
    "epochs_completed",
    "best_validation_epoch",
    "best_validation_loss",
    "training_seconds",
    "inference_seconds",
]

mlp_metrics_df = mlp_metrics_df[
    mlp_metric_columns
]

# ------------------------------------------------------------------
# 8. Prepare validation-threshold audit data
# ------------------------------------------------------------------

mlp_threshold_table = (
    mlp_evaluation[
        "threshold_table"
    ].copy()
)

mlp_threshold_table.insert(
    0,
    "experiment_id",
    EXPERIMENT_ID,
)

mlp_threshold_table.insert(
    1,
    "model",
    "MLP",
)

mlp_threshold_table.insert(
    2,
    "split",
    "validation",
)

# ------------------------------------------------------------------
# 9. Prepare validation predictions
# ------------------------------------------------------------------

mlp_validation_predictions = (
    val_df[
        [
            "model_window_id",
            "window_id",
            "recording_block",
            "window_start",
            "window_end",
            "event_count",
            "event_window_label",
            "split",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

mlp_validation_predictions[
    "model"
] = "MLP"

mlp_validation_predictions[
    "probability_score"
] = mlp_validation_scores

mlp_validation_predictions[
    "selected_threshold"
] = mlp_selected_threshold

mlp_validation_predictions[
    "predicted_label_selected_threshold"
] = (
    mlp_validation_scores
    >= mlp_selected_threshold
).astype(np.int64)

# ------------------------------------------------------------------
# 10. Prepare complete test predictions
# ------------------------------------------------------------------

mlp_test_predictions = (
    test_df[
        [
            "model_window_id",
            "window_id",
            "recording_block",
            "window_start",
            "window_end",
            "event_count",
            "event_window_label",
            "split",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

mlp_test_predictions[
    "model"
] = "MLP"

mlp_test_predictions[
    "probability_score"
] = mlp_test_scores

mlp_test_predictions[
    "default_threshold"
] = 0.5

mlp_test_predictions[
    "predicted_label_default_0.5"
] = (
    mlp_test_scores >= 0.5
).astype(np.int64)

mlp_test_predictions[
    "validation_selected_threshold"
] = mlp_selected_threshold

mlp_test_predictions[
    "predicted_label_validation_f1_optimized"
] = (
    mlp_test_scores
    >= mlp_selected_threshold
).astype(np.int64)

# ------------------------------------------------------------------
# 11. Prepare the complete training-history table
# ------------------------------------------------------------------

mlp_history_df = pd.DataFrame(
    mlp_history.history
)

mlp_history_df.insert(
    0,
    "epoch",
    np.arange(
        1,
        len(mlp_history_df) + 1,
    ),
)

mlp_history_df[
    "is_best_validation_loss_epoch"
] = (
    mlp_history_df["epoch"] == best_epoch
)

# ------------------------------------------------------------------
# 12. Capture the model architecture summary
# ------------------------------------------------------------------

mlp_summary_lines = []

mlp_model.summary(
    print_fn=lambda line: mlp_summary_lines.append(line)
)

mlp_architecture_text = "\n".join(
    mlp_summary_lines
)

# ------------------------------------------------------------------
# 13. Define output files
# ------------------------------------------------------------------

MLP_MODEL_FILE = (
    MODELS_PATH
    / "blued_supervised_mlp.keras"
)

MLP_METRICS_FILE = (
    METRICS_PATH
    / "blued_supervised_mlp_metrics.csv"
)

MLP_THRESHOLD_FILE = (
    METRICS_PATH
    / "blued_supervised_mlp_validation_thresholds.csv"
)

MLP_VALIDATION_PREDICTIONS_FILE = (
    PREDICTIONS_PATH
    / "blued_supervised_mlp_validation_predictions.csv"
)

MLP_TEST_PREDICTIONS_FILE = (
    PREDICTIONS_PATH
    / "blued_supervised_mlp_test_predictions.csv"
)

MLP_HISTORY_FILE = (
    METRICS_PATH
    / "blued_supervised_mlp_training_history.csv"
)

MLP_ARCHITECTURE_FILE = (
    CONFIG_PATH
    / "blued_supervised_mlp_architecture.txt"
)

MLP_SUMMARY_FILE = (
    LOG_PATH
    / "blued_supervised_mlp_summary.json"
)

# ------------------------------------------------------------------
# 14. Save the model and all MLP outputs
# ------------------------------------------------------------------

mlp_model.save(
    MLP_MODEL_FILE
)

mlp_metrics_df.to_csv(
    MLP_METRICS_FILE,
    index=False,
)

mlp_threshold_table.to_csv(
    MLP_THRESHOLD_FILE,
    index=False,
)

mlp_validation_predictions.to_csv(
    MLP_VALIDATION_PREDICTIONS_FILE,
    index=False,
)

mlp_test_predictions.to_csv(
    MLP_TEST_PREDICTIONS_FILE,
    index=False,
)

mlp_history_df.to_csv(
    MLP_HISTORY_FILE,
    index=False,
)

with open(
    MLP_ARCHITECTURE_FILE,
    "w",
    encoding="utf-8",
) as file:
    file.write(
        mlp_architecture_text
    )

mlp_summary = {
    "experiment_id": EXPERIMENT_ID,
    "model": "MLP",
    "configuration": {
        "input_features": int(
            len(feature_columns)
        ),
        "hidden_layers": [
            64,
            32,
        ],
        "hidden_activation": "relu",
        "dropout_rate": 0.2,
        "output_activation": "sigmoid",
        "optimizer": "Adam",
        "learning_rate": 0.001,
        "loss": "binary_crossentropy",
        "maximum_epochs": 50,
        "batch_size": 256,
        "early_stopping_patience": 5,
        "restore_best_weights": True,
        "class_weights": {
            "0": float(
                mlp_class_weights[0]
            ),
            "1": float(
                mlp_class_weights[1]
            ),
        },
        "random_seed": RANDOM_SEED,
        "preprocessing": (
            "Median imputation and StandardScaler fitted "
            "using training data only"
        ),
    },
    "training": {
        "epochs_completed": int(
            epochs_completed
        ),
        "best_validation_epoch": int(
            best_epoch
        ),
        "best_validation_loss": float(
            best_validation_loss
        ),
        "training_seconds": float(
            mlp_training_seconds
        ),
        "inference_seconds": float(
            mlp_inference_seconds
        ),
    },
    "selected_validation_threshold": float(
        mlp_selected_threshold
    ),
    "validation_performance_at_selected_threshold": (
        mlp_validation_selected_result
    ),
    "test_performance_default_threshold": (
        mlp_test_default_result
    ),
    "test_performance_selected_threshold": (
        mlp_test_selected_result
    ),
}

with open(
    MLP_SUMMARY_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp_summary,
        file,
        indent=2,
    )

# ------------------------------------------------------------------
# 15. Verify saved MLP artifacts
# ------------------------------------------------------------------

mlp_output_files = [
    MLP_MODEL_FILE,
    MLP_METRICS_FILE,
    MLP_THRESHOLD_FILE,
    MLP_VALIDATION_PREDICTIONS_FILE,
    MLP_TEST_PREDICTIONS_FILE,
    MLP_HISTORY_FILE,
    MLP_ARCHITECTURE_FILE,
    MLP_SUMMARY_FILE,
]

for file_path in mlp_output_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"MLP output was not created: {file_path}"
        )

print("\nSaved MLP artifacts:")

for file_path in mlp_output_files:
    print(
        f"{file_path.name:65s} "
        f"| {file_path.stat().st_size / 1024:,.2f} KB"
    )

# ------------------------------------------------------------------
# 16. Display the main MLP results
# ------------------------------------------------------------------

print("\nMLP validation-threshold selection:")

print(
    f"Selected threshold:     "
    f"{mlp_selected_threshold:.6f}"
)

print(
    f"Validation Precision:   "
    f"{mlp_validation_selected_result['precision']:.6f}"
)

print(
    f"Validation Recall:      "
    f"{mlp_validation_selected_result['recall']:.6f}"
)

print(
    f"Validation F1-Score:    "
    f"{mlp_validation_selected_result['f1_score']:.6f}"
)

print("\nMLP test results:")

display(
    mlp_metrics_df.loc[
        mlp_metrics_df["split"] == "test",
        [
            "threshold_source",
            "threshold",
            "precision",
            "recall",
            "f1_score",
            "false_positive_rate",
            "roc_auc",
            "pr_auc",
            "tn",
            "fp",
            "fn",
            "tp",
            "flagged_ratio_percent",
        ],
    ]
)

print("\nMLP training-history summary:")

display(
    mlp_history_df[
        [
            "epoch",
            "loss",
            "val_loss",
            "precision",
            "recall",
            "roc_auc",
            "pr_auc",
            "val_precision",
            "val_recall",
            "val_roc_auc",
            "val_pr_auc",
            "is_best_validation_loss_epoch",
        ]
    ].tail(10)
)

print(
    "\nSupervised BLUED MLP experiment "
    "completed successfully."
)

In [ ]:
# Step 14 — Consolidate and compare all four supervised BLUED models

import json
from pathlib import Path

import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# 1. Define and verify the four model-metrics files
# ------------------------------------------------------------------

MODEL_METRICS_FILES = {
    "Random Forest": (
        METRICS_PATH
        / "blued_supervised_random_forest_metrics.csv"
    ),
    "XGBoost": (
        METRICS_PATH
        / "blued_supervised_xgboost_metrics.csv"
    ),
    "Logistic Regression": (
        METRICS_PATH
        / "blued_supervised_logistic_regression_metrics.csv"
    ),
    "MLP": (
        METRICS_PATH
        / "blued_supervised_mlp_metrics.csv"
    ),
}

for model_name, file_path in MODEL_METRICS_FILES.items():
    if not file_path.exists():
        raise FileNotFoundError(
            f"{model_name} metrics file was not found:\n"
            f"{file_path}"
        )

print("Verified model-metrics files:")

for model_name, file_path in MODEL_METRICS_FILES.items():
    print(
        f"{model_name:22s} "
        f"| {file_path.name}"
    )

# ------------------------------------------------------------------
# 2. Load and combine all model-level metrics
# ------------------------------------------------------------------

combined_metric_frames = []

for model_name, file_path in MODEL_METRICS_FILES.items():
    model_metrics = pd.read_csv(file_path)

    if "model" not in model_metrics.columns:
        raise ValueError(
            f"{model_name} metrics do not contain a model column."
        )

    observed_models = set(
        model_metrics["model"].dropna().unique()
    )

    if observed_models != {model_name}:
        raise ValueError(
            f"{model_name} file contains unexpected model names: "
            f"{observed_models}"
        )

    combined_metric_frames.append(model_metrics)

combined_metrics_df = pd.concat(
    combined_metric_frames,
    ignore_index=True,
    sort=False,
)

expected_models = set(MODEL_METRICS_FILES)
observed_models = set(
    combined_metrics_df["model"].unique()
)

if observed_models != expected_models:
    raise ValueError(
        f"Expected models {expected_models}, "
        f"but found {observed_models}."
    )

rows_per_model = (
    combined_metrics_df
    .groupby("model")
    .size()
)

if not (rows_per_model == 3).all():
    raise ValueError(
        "Each model should have exactly three metric rows:\n"
        "validation-selected, test-default, and test-selected."
    )

print(
    f"\nCombined metric rows: "
    f"{len(combined_metrics_df):,}"
)

print("\nRows per model:")
print(rows_per_model.to_string())

# ------------------------------------------------------------------
# 3. Extract validation-selected performance
# ------------------------------------------------------------------

validation_comparison_df = (
    combined_metrics_df.loc[
        (combined_metrics_df["split"] == "validation")
        & (
            combined_metrics_df["threshold_source"]
            == "validation_f1_optimized"
        )
    ]
    .copy()
)

if len(validation_comparison_df) != 4:
    raise ValueError(
        "Expected one validation-selected row per model."
    )

validation_comparison_df[
    "validation_f1_rank"
] = (
    validation_comparison_df["f1_score"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

validation_comparison_df = (
    validation_comparison_df
    .sort_values(
        [
            "validation_f1_rank",
            "model",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------------
# 4. Extract primary test results
# ------------------------------------------------------------------
# These are the primary operating-point results because each threshold
# was selected using validation labels only.

primary_test_comparison_df = (
    combined_metrics_df.loc[
        (combined_metrics_df["split"] == "test")
        & (
            combined_metrics_df["threshold_source"]
            == "validation_f1_optimized"
        )
    ]
    .copy()
)

if len(primary_test_comparison_df) != 4:
    raise ValueError(
        "Expected one validation-selected test row per model."
    )

primary_test_comparison_df[
    "test_f1_rank"
] = (
    primary_test_comparison_df["f1_score"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

primary_test_comparison_df[
    "test_roc_auc_rank"
] = (
    primary_test_comparison_df["roc_auc"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

primary_test_comparison_df[
    "test_pr_auc_rank"
] = (
    primary_test_comparison_df["pr_auc"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

primary_test_comparison_df[
    "test_fpr_rank"
] = (
    primary_test_comparison_df["false_positive_rate"]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

primary_test_comparison_df = (
    primary_test_comparison_df
    .sort_values(
        [
            "test_f1_rank",
            "model",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------------
# 5. Extract supplementary default-threshold results
# ------------------------------------------------------------------

default_test_comparison_df = (
    combined_metrics_df.loc[
        (combined_metrics_df["split"] == "test")
        & (
            combined_metrics_df["threshold_source"]
            == "default_0.5"
        )
    ]
    .copy()
)

if len(default_test_comparison_df) != 4:
    raise ValueError(
        "Expected one default-threshold test row per model."
    )

default_test_comparison_df[
    "test_f1_rank"
] = (
    default_test_comparison_df["f1_score"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

default_test_comparison_df = (
    default_test_comparison_df
    .sort_values(
        [
            "test_f1_rank",
            "model",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------------
# 6. Quantify threshold sensitivity
# ------------------------------------------------------------------

threshold_sensitivity_df = (
    primary_test_comparison_df[
        [
            "model",
            "threshold",
            "precision",
            "recall",
            "f1_score",
            "false_positive_rate",
            "flagged_ratio_percent",
        ]
    ]
    .rename(
        columns={
            "threshold": "selected_threshold",
            "precision": "selected_precision",
            "recall": "selected_recall",
            "f1_score": "selected_f1",
            "false_positive_rate": "selected_fpr",
            "flagged_ratio_percent": (
                "selected_flagged_ratio_percent"
            ),
        }
    )
    .merge(
        default_test_comparison_df[
            [
                "model",
                "threshold",
                "precision",
                "recall",
                "f1_score",
                "false_positive_rate",
                "flagged_ratio_percent",
            ]
        ].rename(
            columns={
                "threshold": "default_threshold",
                "precision": "default_precision",
                "recall": "default_recall",
                "f1_score": "default_f1",
                "false_positive_rate": "default_fpr",
                "flagged_ratio_percent": (
                    "default_flagged_ratio_percent"
                ),
            }
        ),
        on="model",
        how="inner",
        validate="one_to_one",
    )
)

threshold_sensitivity_df[
    "precision_delta_selected_minus_default"
] = (
    threshold_sensitivity_df["selected_precision"]
    - threshold_sensitivity_df["default_precision"]
)

threshold_sensitivity_df[
    "recall_delta_selected_minus_default"
] = (
    threshold_sensitivity_df["selected_recall"]
    - threshold_sensitivity_df["default_recall"]
)

threshold_sensitivity_df[
    "f1_delta_selected_minus_default"
] = (
    threshold_sensitivity_df["selected_f1"]
    - threshold_sensitivity_df["default_f1"]
)

threshold_sensitivity_df[
    "fpr_delta_selected_minus_default"
] = (
    threshold_sensitivity_df["selected_fpr"]
    - threshold_sensitivity_df["default_fpr"]
)

threshold_sensitivity_df = (
    threshold_sensitivity_df
    .sort_values("model")
    .reset_index(drop=True)
)

# ------------------------------------------------------------------
# 7. Identify the principal comparative findings
# ------------------------------------------------------------------

best_validation_f1_row = (
    validation_comparison_df
    .sort_values(
        "f1_score",
        ascending=False,
    )
    .iloc[0]
)

best_test_selected_f1_row = (
    primary_test_comparison_df
    .sort_values(
        "f1_score",
        ascending=False,
    )
    .iloc[0]
)

best_test_roc_auc_row = (
    primary_test_comparison_df
    .sort_values(
        "roc_auc",
        ascending=False,
    )
    .iloc[0]
)

best_test_pr_auc_row = (
    primary_test_comparison_df
    .sort_values(
        "pr_auc",
        ascending=False,
    )
    .iloc[0]
)

lowest_test_fpr_row = (
    primary_test_comparison_df
    .sort_values(
        "false_positive_rate",
        ascending=True,
    )
    .iloc[0]
)

highest_test_precision_row = (
    primary_test_comparison_df
    .sort_values(
        "precision",
        ascending=False,
    )
    .iloc[0]
)

highest_test_recall_row = (
    primary_test_comparison_df
    .sort_values(
        "recall",
        ascending=False,
    )
    .iloc[0]
)

# ------------------------------------------------------------------
# 8. Define final output files
# ------------------------------------------------------------------

COMBINED_METRICS_FILE = (
    METRICS_PATH
    / "blued_supervised_all_models_complete_metrics.csv"
)

VALIDATION_COMPARISON_FILE = (
    METRICS_PATH
    / "blued_supervised_all_models_validation_comparison.csv"
)

PRIMARY_TEST_COMPARISON_FILE = (
    METRICS_PATH
    / "blued_supervised_all_models_primary_test_comparison.csv"
)

DEFAULT_TEST_COMPARISON_FILE = (
    METRICS_PATH
    / "blued_supervised_all_models_default_test_comparison.csv"
)

THRESHOLD_SENSITIVITY_FILE = (
    METRICS_PATH
    / "blued_supervised_all_models_threshold_sensitivity.csv"
)

FINAL_SUMMARY_FILE = (
    LOG_PATH
    / "blued_supervised_all_models_final_summary.json"
)

# ------------------------------------------------------------------
# 9. Save the consolidated tables
# ------------------------------------------------------------------

combined_metrics_df.to_csv(
    COMBINED_METRICS_FILE,
    index=False,
)

validation_comparison_df.to_csv(
    VALIDATION_COMPARISON_FILE,
    index=False,
)

primary_test_comparison_df.to_csv(
    PRIMARY_TEST_COMPARISON_FILE,
    index=False,
)

default_test_comparison_df.to_csv(
    DEFAULT_TEST_COMPARISON_FILE,
    index=False,
)

threshold_sensitivity_df.to_csv(
    THRESHOLD_SENSITIVITY_FILE,
    index=False,
)

# ------------------------------------------------------------------
# 10. Save the final structured summary
# ------------------------------------------------------------------

final_summary = {
    "experiment_id": EXPERIMENT_ID,
    "task": "Supervised BLUED event-window classification",
    "semantic_boundary": (
        "The positive class represents windows containing at least "
        "one annotated appliance switching event. These annotations "
        "are not confirmed anomaly labels."
    ),
    "primary_reporting_protocol": (
        "Test Precision, Recall, F1-Score, and False Positive Rate "
        "at thresholds selected by maximizing validation F1-Score."
    ),
    "supplementary_reporting_protocol": (
        "Default probability threshold results at 0.5 are retained "
        "for transparency but are not selected using test performance."
    ),
    "best_validation_f1": {
        "model": str(
            best_validation_f1_row["model"]
        ),
        "f1_score": float(
            best_validation_f1_row["f1_score"]
        ),
        "threshold": float(
            best_validation_f1_row["threshold"]
        ),
    },
    "highest_test_f1_at_validation_selected_threshold": {
        "model": str(
            best_test_selected_f1_row["model"]
        ),
        "f1_score": float(
            best_test_selected_f1_row["f1_score"]
        ),
    },
    "highest_test_roc_auc": {
        "model": str(
            best_test_roc_auc_row["model"]
        ),
        "roc_auc": float(
            best_test_roc_auc_row["roc_auc"]
        ),
    },
    "highest_test_pr_auc": {
        "model": str(
            best_test_pr_auc_row["model"]
        ),
        "pr_auc": float(
            best_test_pr_auc_row["pr_auc"]
        ),
    },
    "highest_test_precision_at_selected_threshold": {
        "model": str(
            highest_test_precision_row["model"]
        ),
        "precision": float(
            highest_test_precision_row["precision"]
        ),
    },
    "highest_test_recall_at_selected_threshold": {
        "model": str(
            highest_test_recall_row["model"]
        ),
        "recall": float(
            highest_test_recall_row["recall"]
        ),
    },
    "lowest_test_fpr_at_selected_threshold": {
        "model": str(
            lowest_test_fpr_row["model"]
        ),
        "false_positive_rate": float(
            lowest_test_fpr_row[
                "false_positive_rate"
            ]
        ),
    },
    "interpretation": [
        (
            "XGBoost achieved the highest validation F1-Score and "
            "the highest test ROC-AUC."
        ),
        (
            "Random Forest achieved the highest test F1-Score and "
            "PR-AUC at the validation-selected operating point."
        ),
        (
            "Logistic Regression and MLP benefited from their "
            "validation-selected thresholds on the test F1-Score."
        ),
        (
            "Default-threshold results are supplementary and must "
            "not be used to retrospectively select an operating "
            "threshold from the test set."
        ),
    ],
}

with open(
    FINAL_SUMMARY_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        final_summary,
        file,
        indent=2,
    )

# ------------------------------------------------------------------
# 11. Verify the consolidated outputs
# ------------------------------------------------------------------

consolidated_output_files = [
    COMBINED_METRICS_FILE,
    VALIDATION_COMPARISON_FILE,
    PRIMARY_TEST_COMPARISON_FILE,
    DEFAULT_TEST_COMPARISON_FILE,
    THRESHOLD_SENSITIVITY_FILE,
    FINAL_SUMMARY_FILE,
]

for file_path in consolidated_output_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Consolidated output was not created: "
            f"{file_path}"
        )

print("\nSaved consolidated supervised BLUED outputs:")

for file_path in consolidated_output_files:
    print(
        f"{file_path.name:70s} "
        f"| {file_path.stat().st_size / 1024:,.2f} KB"
    )

# ------------------------------------------------------------------
# 12. Display the validation comparison
# ------------------------------------------------------------------

print("\nValidation-selected operating points:")

display(
    validation_comparison_df[
        [
            "validation_f1_rank",
            "model",
            "threshold",
            "precision",
            "recall",
            "f1_score",
            "false_positive_rate",
            "roc_auc",
            "pr_auc",
        ]
    ]
)

# ------------------------------------------------------------------
# 13. Display the primary test comparison
# ------------------------------------------------------------------

print(
    "\nPrimary test comparison using "
    "validation-selected thresholds:"
)

display(
    primary_test_comparison_df[
        [
            "test_f1_rank",
            "model",
            "threshold",
            "precision",
            "recall",
            "f1_score",
            "false_positive_rate",
            "roc_auc",
            "pr_auc",
            "tn",
            "fp",
            "fn",
            "tp",
            "flagged_ratio_percent",
        ]
    ]
)

# ------------------------------------------------------------------
# 14. Display the supplementary default-threshold comparison
# ------------------------------------------------------------------

print(
    "\nSupplementary test comparison at the "
    "default probability threshold of 0.5:"
)

display(
    default_test_comparison_df[
        [
            "test_f1_rank",
            "model",
            "threshold",
            "precision",
            "recall",
            "f1_score",
            "false_positive_rate",
            "roc_auc",
            "pr_auc",
        ]
    ]
)

# ------------------------------------------------------------------
# 15. Display threshold sensitivity
# ------------------------------------------------------------------

print(
    "\nEffect of validation-selected thresholds "
    "relative to the default threshold:"
)

display(
    threshold_sensitivity_df[
        [
            "model",
            "selected_threshold",
            "f1_delta_selected_minus_default",
            "precision_delta_selected_minus_default",
            "recall_delta_selected_minus_default",
            "fpr_delta_selected_minus_default",
        ]
    ]
)

# ------------------------------------------------------------------
# 16. Print the principal findings
# ------------------------------------------------------------------

print("\nPrincipal supervised BLUED findings:")

print(
    "Highest validation F1-Score: "
    f"{best_validation_f1_row['model']} "
    f"({best_validation_f1_row['f1_score']:.6f})"
)

print(
    "Highest test F1-Score at validation-selected threshold: "
    f"{best_test_selected_f1_row['model']} "
    f"({best_test_selected_f1_row['f1_score']:.6f})"
)

print(
    "Highest test ROC-AUC: "
    f"{best_test_roc_auc_row['model']} "
    f"({best_test_roc_auc_row['roc_auc']:.6f})"
)

print(
    "Highest test PR-AUC: "
    f"{best_test_pr_auc_row['model']} "
    f"({best_test_pr_auc_row['pr_auc']:.6f})"
)

print(
    "Lowest test FPR at validation-selected threshold: "
    f"{lowest_test_fpr_row['model']} "
    f"({lowest_test_fpr_row['false_positive_rate']:.6f})"
)

print(
    "\nAll four supervised BLUED models were consolidated "
    "successfully."
)

In [ ]:
# Step 15 — Perform the final supervised BLUED artifact-integrity and reproducibility audit

import hashlib
import json
import joblib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

# ------------------------------------------------------------------
# 1. Define the required artifact set for each supervised model
# ------------------------------------------------------------------

MODEL_AUDIT_SPECIFICATIONS = {
    "Random Forest": {
        "metrics_file": (
            METRICS_PATH
            / "blued_supervised_random_forest_metrics.csv"
        ),
        "validation_predictions_file": (
            PREDICTIONS_PATH
            / "blued_supervised_random_forest_validation_predictions.csv"
        ),
        "test_predictions_file": (
            PREDICTIONS_PATH
            / "blued_supervised_random_forest_test_predictions.csv"
        ),
        "model_file": (
            MODELS_PATH
            / "blued_supervised_random_forest.joblib"
        ),
        "model_type": "joblib_probability_classifier",
        "test_input": X_test_tree,
    },
    "XGBoost": {
        "metrics_file": (
            METRICS_PATH
            / "blued_supervised_xgboost_metrics.csv"
        ),
        "validation_predictions_file": (
            PREDICTIONS_PATH
            / "blued_supervised_xgboost_validation_predictions.csv"
        ),
        "test_predictions_file": (
            PREDICTIONS_PATH
            / "blued_supervised_xgboost_test_predictions.csv"
        ),
        "model_file": (
            MODELS_PATH
            / "blued_supervised_xgboost.joblib"
        ),
        "model_type": "joblib_probability_classifier",
        "test_input": X_test_tree,
    },
    "Logistic Regression": {
        "metrics_file": (
            METRICS_PATH
            / "blued_supervised_logistic_regression_metrics.csv"
        ),
        "validation_predictions_file": (
            PREDICTIONS_PATH
            / "blued_supervised_logistic_regression_validation_predictions.csv"
        ),
        "test_predictions_file": (
            PREDICTIONS_PATH
            / "blued_supervised_logistic_regression_test_predictions.csv"
        ),
        "model_file": (
            MODELS_PATH
            / "blued_supervised_logistic_regression.joblib"
        ),
        "model_type": "joblib_probability_classifier",
        "test_input": X_test_scaled,
    },
    "MLP": {
        "metrics_file": (
            METRICS_PATH
            / "blued_supervised_mlp_metrics.csv"
        ),
        "validation_predictions_file": (
            PREDICTIONS_PATH
            / "blued_supervised_mlp_validation_predictions.csv"
        ),
        "test_predictions_file": (
            PREDICTIONS_PATH
            / "blued_supervised_mlp_test_predictions.csv"
        ),
        "model_file": (
            MODELS_PATH
            / "blued_supervised_mlp.keras"
        ),
        "model_type": "keras_probability_classifier",
        "test_input": X_test_scaled_float32,
    },
}

# ------------------------------------------------------------------
# 2. Define reusable audit helpers
# ------------------------------------------------------------------

def require_file(file_path, artifact_description):
    """
    Verify that an expected artifact exists and is non-empty.
    """

    if not file_path.exists():
        raise FileNotFoundError(
            f"Missing {artifact_description}:\n{file_path}"
        )

    if file_path.stat().st_size <= 0:
        raise ValueError(
            f"Empty {artifact_description}:\n{file_path}"
        )


def assert_numeric_match(
    saved_value,
    recomputed_value,
    field_name,
    model_name,
    tolerance=1e-10,
):
    """
    Verify numerical agreement between a saved metric and a
    recomputed metric.
    """

    if not np.isclose(
        float(saved_value),
        float(recomputed_value),
        rtol=tolerance,
        atol=tolerance,
    ):
        raise ValueError(
            f"{model_name}: mismatch in {field_name}. "
            f"Saved={saved_value}, "
            f"recomputed={recomputed_value}."
        )


def calculate_sha256(file_path, chunk_size=1024 * 1024):
    """
    Calculate the SHA-256 checksum of one artifact.
    """

    sha256_hash = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            data_chunk = file.read(chunk_size)

            if not data_chunk:
                break

            sha256_hash.update(data_chunk)

    return sha256_hash.hexdigest()


# ------------------------------------------------------------------
# 3. Audit every model independently
# ------------------------------------------------------------------

audit_rows = []
all_integrity_checks = []

for model_name, specification in MODEL_AUDIT_SPECIFICATIONS.items():
    print("=" * 100)
    print(f"Auditing model: {model_name}")
    print("=" * 100)

    metrics_file = specification["metrics_file"]
    validation_predictions_file = specification[
        "validation_predictions_file"
    ]
    test_predictions_file = specification[
        "test_predictions_file"
    ]
    model_file = specification["model_file"]

    require_file(
        metrics_file,
        f"{model_name} metrics file",
    )

    require_file(
        validation_predictions_file,
        f"{model_name} validation-prediction file",
    )

    require_file(
        test_predictions_file,
        f"{model_name} test-prediction file",
    )

    require_file(
        model_file,
        f"{model_name} trained-model file",
    )

    model_metrics = pd.read_csv(metrics_file)
    validation_predictions = pd.read_csv(
        validation_predictions_file
    )
    test_predictions = pd.read_csv(
        test_predictions_file
    )

    # --------------------------------------------------------------
    # 3.1 Verify expected row counts
    # --------------------------------------------------------------

    if len(model_metrics) != 3:
        raise ValueError(
            f"{model_name}: expected 3 metrics rows, "
            f"found {len(model_metrics):,}."
        )

    if len(validation_predictions) != len(y_val):
        raise ValueError(
            f"{model_name}: expected {len(y_val):,} validation "
            f"prediction rows, found "
            f"{len(validation_predictions):,}."
        )

    if len(test_predictions) != len(y_test):
        raise ValueError(
            f"{model_name}: expected {len(y_test):,} test "
            f"prediction rows, found "
            f"{len(test_predictions):,}."
        )

    # --------------------------------------------------------------
    # 3.2 Verify common window identifiers and true labels
    # --------------------------------------------------------------

    expected_validation_ids = (
        val_df["model_window_id"]
        .reset_index(drop=True)
        .astype(str)
        .to_numpy()
    )

    observed_validation_ids = (
        validation_predictions["model_window_id"]
        .astype(str)
        .to_numpy()
    )

    expected_test_ids = (
        test_df["model_window_id"]
        .reset_index(drop=True)
        .astype(str)
        .to_numpy()
    )

    observed_test_ids = (
        test_predictions["model_window_id"]
        .astype(str)
        .to_numpy()
    )

    if not np.array_equal(
        expected_validation_ids,
        observed_validation_ids,
    ):
        raise ValueError(
            f"{model_name}: validation window identifiers "
            "do not match the frozen validation partition."
        )

    if not np.array_equal(
        expected_test_ids,
        observed_test_ids,
    ):
        raise ValueError(
            f"{model_name}: test window identifiers do not "
            "match the frozen test partition."
        )

    if not np.array_equal(
        validation_predictions[
            "event_window_label"
        ].to_numpy(dtype=np.int64),
        y_val,
    ):
        raise ValueError(
            f"{model_name}: validation labels do not match "
            "the frozen validation labels."
        )

    if not np.array_equal(
        test_predictions[
            "event_window_label"
        ].to_numpy(dtype=np.int64),
        y_test,
    ):
        raise ValueError(
            f"{model_name}: test labels do not match "
            "the frozen test labels."
        )

    # --------------------------------------------------------------
    # 3.3 Validate saved probability scores
    # --------------------------------------------------------------

    validation_scores = validation_predictions[
        "probability_score"
    ].to_numpy(dtype=np.float64)

    test_scores = test_predictions[
        "probability_score"
    ].to_numpy(dtype=np.float64)

    if not np.isfinite(validation_scores).all():
        raise ValueError(
            f"{model_name}: validation probability scores "
            "contain non-finite values."
        )

    if not np.isfinite(test_scores).all():
        raise ValueError(
            f"{model_name}: test probability scores contain "
            "non-finite values."
        )

    if (
        validation_scores.min() < 0.0
        or validation_scores.max() > 1.0
    ):
        raise ValueError(
            f"{model_name}: validation probability scores "
            "fall outside [0, 1]."
        )

    if (
        test_scores.min() < 0.0
        or test_scores.max() > 1.0
    ):
        raise ValueError(
            f"{model_name}: test probability scores "
            "fall outside [0, 1]."
        )

    # --------------------------------------------------------------
    # 3.4 Extract the three expected metrics rows
    # --------------------------------------------------------------

    validation_metric_rows = model_metrics.loc[
        (model_metrics["split"] == "validation")
        & (
            model_metrics["threshold_source"]
            == "validation_f1_optimized"
        )
    ]

    test_default_metric_rows = model_metrics.loc[
        (model_metrics["split"] == "test")
        & (
            model_metrics["threshold_source"]
            == "default_0.5"
        )
    ]

    test_selected_metric_rows = model_metrics.loc[
        (model_metrics["split"] == "test")
        & (
            model_metrics["threshold_source"]
            == "validation_f1_optimized"
        )
    ]

    if len(validation_metric_rows) != 1:
        raise ValueError(
            f"{model_name}: invalid validation metrics-row count."
        )

    if len(test_default_metric_rows) != 1:
        raise ValueError(
            f"{model_name}: invalid default test metrics-row count."
        )

    if len(test_selected_metric_rows) != 1:
        raise ValueError(
            f"{model_name}: invalid selected-threshold "
            "test metrics-row count."
        )

    validation_metric_row = (
        validation_metric_rows.iloc[0]
    )

    test_default_metric_row = (
        test_default_metric_rows.iloc[0]
    )

    test_selected_metric_row = (
        test_selected_metric_rows.iloc[0]
    )

    selected_threshold = float(
        validation_metric_row["threshold"]
    )

    assert_numeric_match(
        saved_value=test_selected_metric_row["threshold"],
        recomputed_value=selected_threshold,
        field_name="selected test threshold",
        model_name=model_name,
    )

    # --------------------------------------------------------------
    # 3.5 Independently recompute all evaluation metrics
    # --------------------------------------------------------------

    recomputed_validation_result, _ = (
        evaluate_binary_classifier(
            y_true=y_val,
            y_score=validation_scores,
            threshold=selected_threshold,
            model_name=model_name,
            split_name="validation",
            threshold_source="validation_f1_optimized",
        )
    )

    recomputed_test_default_result, _ = (
        evaluate_binary_classifier(
            y_true=y_test,
            y_score=test_scores,
            threshold=0.5,
            model_name=model_name,
            split_name="test",
            threshold_source="default_0.5",
        )
    )

    recomputed_test_selected_result, _ = (
        evaluate_binary_classifier(
            y_true=y_test,
            y_score=test_scores,
            threshold=selected_threshold,
            model_name=model_name,
            split_name="test",
            threshold_source="validation_f1_optimized",
        )
    )

    numeric_metric_fields = [
        "threshold",
        "precision",
        "recall",
        "f1_score",
        "false_positive_rate",
        "roc_auc",
        "pr_auc",
        "tn",
        "fp",
        "fn",
        "tp",
        "flagged_windows",
        "flagged_ratio",
        "flagged_ratio_percent",
        "positive_windows",
        "negative_windows",
        "positive_prevalence",
        "n_windows",
    ]

    saved_and_recomputed_groups = [
        (
            validation_metric_row,
            recomputed_validation_result,
            "validation-selected",
        ),
        (
            test_default_metric_row,
            recomputed_test_default_result,
            "test-default",
        ),
        (
            test_selected_metric_row,
            recomputed_test_selected_result,
            "test-selected",
        ),
    ]

    for (
        saved_metric_row,
        recomputed_metric_result,
        metric_group_name,
    ) in saved_and_recomputed_groups:
        for metric_field in numeric_metric_fields:
            assert_numeric_match(
                saved_value=saved_metric_row[metric_field],
                recomputed_value=(
                    recomputed_metric_result[metric_field]
                ),
                field_name=(
                    f"{metric_group_name} {metric_field}"
                ),
                model_name=model_name,
                tolerance=1e-9,
            )

    # --------------------------------------------------------------
    # 3.6 Verify saved predicted-label columns
    # --------------------------------------------------------------

    expected_validation_selected_predictions = (
        validation_scores >= selected_threshold
    ).astype(np.int64)

    saved_validation_selected_predictions = (
        validation_predictions[
            "predicted_label_selected_threshold"
        ].to_numpy(dtype=np.int64)
    )

    if not np.array_equal(
        expected_validation_selected_predictions,
        saved_validation_selected_predictions,
    ):
        raise ValueError(
            f"{model_name}: saved validation predictions "
            "do not match the selected threshold."
        )

    expected_test_default_predictions = (
        test_scores >= 0.5
    ).astype(np.int64)

    saved_test_default_predictions = (
        test_predictions[
            "predicted_label_default_0.5"
        ].to_numpy(dtype=np.int64)
    )

    if not np.array_equal(
        expected_test_default_predictions,
        saved_test_default_predictions,
    ):
        raise ValueError(
            f"{model_name}: saved default-threshold test "
            "predictions are inconsistent."
        )

    expected_test_selected_predictions = (
        test_scores >= selected_threshold
    ).astype(np.int64)

    saved_test_selected_predictions = (
        test_predictions[
            "predicted_label_validation_f1_optimized"
        ].to_numpy(dtype=np.int64)
    )

    if not np.array_equal(
        expected_test_selected_predictions,
        saved_test_selected_predictions,
    ):
        raise ValueError(
            f"{model_name}: saved selected-threshold test "
            "predictions are inconsistent."
        )

    # --------------------------------------------------------------
    # 3.7 Reload the saved model and reproduce its test scores
    # --------------------------------------------------------------

    if (
        specification["model_type"]
        == "joblib_probability_classifier"
    ):
        reloaded_model = joblib.load(model_file)

        reproduced_test_scores = (
            reloaded_model.predict_proba(
                specification["test_input"]
            )[:, 1]
        )

    elif (
        specification["model_type"]
        == "keras_probability_classifier"
    ):
        reloaded_model = tf.keras.models.load_model(
            model_file,
            compile=False,
        )

        reproduced_test_scores = (
            reloaded_model.predict(
                specification["test_input"],
                batch_size=256,
                verbose=0,
            )
            .reshape(-1)
        )

    else:
        raise ValueError(
            f"{model_name}: unsupported model artifact type."
        )

    reproduced_test_scores = np.asarray(
        reproduced_test_scores,
        dtype=np.float64,
    )

    maximum_reload_score_difference = float(
        np.max(
            np.abs(
                reproduced_test_scores
                - test_scores
            )
        )
    )

    if not np.allclose(
        reproduced_test_scores,
        test_scores,
        rtol=1e-6,
        atol=1e-6,
    ):
        raise ValueError(
            f"{model_name}: reloaded model scores do not "
            "match the saved test scores. Maximum difference: "
            f"{maximum_reload_score_difference:.12f}"
        )

    audit_rows.append(
        {
            "model": model_name,
            "validation_selected_threshold": (
                selected_threshold
            ),
            "validation_f1": float(
                validation_metric_row["f1_score"]
            ),
            "test_precision": float(
                test_selected_metric_row["precision"]
            ),
            "test_recall": float(
                test_selected_metric_row["recall"]
            ),
            "test_f1": float(
                test_selected_metric_row["f1_score"]
            ),
            "test_false_positive_rate": float(
                test_selected_metric_row[
                    "false_positive_rate"
                ]
            ),
            "test_roc_auc": float(
                test_selected_metric_row["roc_auc"]
            ),
            "test_pr_auc": float(
                test_selected_metric_row["pr_auc"]
            ),
            "test_tn": int(
                test_selected_metric_row["tn"]
            ),
            "test_fp": int(
                test_selected_metric_row["fp"]
            ),
            "test_fn": int(
                test_selected_metric_row["fn"]
            ),
            "test_tp": int(
                test_selected_metric_row["tp"]
            ),
            "validation_prediction_rows": int(
                len(validation_predictions)
            ),
            "test_prediction_rows": int(
                len(test_predictions)
            ),
            "maximum_model_reload_score_difference": (
                maximum_reload_score_difference
            ),
            "integrity_status": "PASS",
        }
    )

    all_integrity_checks.append(
        {
            "model": model_name,
            "status": "PASS",
            "checks": [
                "Required artifacts exist and are non-empty",
                "Prediction-row counts match frozen splits",
                "Window identifiers match frozen splits",
                "True labels match frozen labels",
                "Probability scores are finite and bounded",
                "Saved metrics reproduce exactly",
                "Saved thresholded predictions reproduce exactly",
                "Reloaded model reproduces saved test scores",
            ],
        }
    )

    print(
        f"{model_name}: PASS "
        f"| max reload-score difference = "
        f"{maximum_reload_score_difference:.12f}"
    )

# ------------------------------------------------------------------
# 4. Verify cross-model consistency
# ------------------------------------------------------------------

print("\n" + "=" * 100)
print("Cross-model consistency audit")
print("=" * 100)

reference_test_predictions = pd.read_csv(
    MODEL_AUDIT_SPECIFICATIONS[
        "Random Forest"
    ]["test_predictions_file"]
)

reference_validation_predictions = pd.read_csv(
    MODEL_AUDIT_SPECIFICATIONS[
        "Random Forest"
    ]["validation_predictions_file"]
)

for model_name, specification in MODEL_AUDIT_SPECIFICATIONS.items():
    current_test_predictions = pd.read_csv(
        specification["test_predictions_file"]
    )

    current_validation_predictions = pd.read_csv(
        specification["validation_predictions_file"]
    )

    for identifier_column in [
        "model_window_id",
        "window_id",
        "recording_block",
        "event_window_label",
        "split",
    ]:
        reference_test_values = (
            reference_test_predictions[
                identifier_column
            ]
            .astype(str)
            .to_numpy()
        )

        current_test_values = (
            current_test_predictions[
                identifier_column
            ]
            .astype(str)
            .to_numpy()
        )

        if not np.array_equal(
            reference_test_values,
            current_test_values,
        ):
            raise ValueError(
                f"Cross-model test inconsistency for "
                f"{model_name}: {identifier_column}."
            )

        reference_validation_values = (
            reference_validation_predictions[
                identifier_column
            ]
            .astype(str)
            .to_numpy()
        )

        current_validation_values = (
            current_validation_predictions[
                identifier_column
            ]
            .astype(str)
            .to_numpy()
        )

        if not np.array_equal(
            reference_validation_values,
            current_validation_values,
        ):
            raise ValueError(
                f"Cross-model validation inconsistency for "
                f"{model_name}: {identifier_column}."
            )

print("All models used identical validation and test windows.")
print("All models used identical event-window labels.")

# ------------------------------------------------------------------
# 5. Verify the consolidated comparison files
# ------------------------------------------------------------------

required_consolidated_files = [
    COMBINED_METRICS_FILE,
    VALIDATION_COMPARISON_FILE,
    PRIMARY_TEST_COMPARISON_FILE,
    DEFAULT_TEST_COMPARISON_FILE,
    THRESHOLD_SENSITIVITY_FILE,
    FINAL_SUMMARY_FILE,
]

for file_path in required_consolidated_files:
    require_file(
        file_path,
        "consolidated supervised BLUED artifact",
    )

final_primary_comparison = pd.read_csv(
    PRIMARY_TEST_COMPARISON_FILE
)

if len(final_primary_comparison) != 4:
    raise ValueError(
        "The final primary comparison must contain four models."
    )

if set(final_primary_comparison["model"]) != set(
    MODEL_AUDIT_SPECIFICATIONS
):
    raise ValueError(
        "The final primary comparison contains an unexpected "
        "model set."
    )

print("Consolidated comparison artifacts verified successfully.")

# ------------------------------------------------------------------
# 6. Save the final integrity-audit table
# ------------------------------------------------------------------

final_audit_df = (
    pd.DataFrame(audit_rows)
    .sort_values(
        "test_f1",
        ascending=False,
    )
    .reset_index(drop=True)
)

final_audit_df.insert(
    0,
    "test_f1_rank",
    np.arange(
        1,
        len(final_audit_df) + 1,
    ),
)

FINAL_INTEGRITY_AUDIT_CSV = (
    AUDIT_PATH
    / "blued_supervised_final_integrity_audit.csv"
)

final_audit_df.to_csv(
    FINAL_INTEGRITY_AUDIT_CSV,
    index=False,
)

# ------------------------------------------------------------------
# 7. Save the structured final audit report
# ------------------------------------------------------------------

FINAL_INTEGRITY_AUDIT_JSON = (
    AUDIT_PATH
    / "blued_supervised_final_integrity_audit.json"
)

final_integrity_report = {
    "experiment_id": EXPERIMENT_ID,
    "audit_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "overall_status": "PASS",
    "task": (
        "Supervised BLUED event-window classification"
    ),
    "semantic_boundary": (
        "The positive class denotes a one-minute window "
        "containing at least one annotated appliance switching "
        "event. It does not denote a confirmed anomaly."
    ),
    "frozen_dataset": {
        "total_windows": int(len(supervised_df)),
        "features": int(len(feature_columns)),
        "train_windows": int(len(y_train)),
        "validation_windows": int(len(y_val)),
        "test_windows": int(len(y_test)),
        "train_event_windows": int(y_train.sum()),
        "validation_event_windows": int(y_val.sum()),
        "test_event_windows": int(y_test.sum()),
    },
    "models_audited": list(
        MODEL_AUDIT_SPECIFICATIONS.keys()
    ),
    "model_checks": all_integrity_checks,
    "cross_model_checks": [
        "All models used identical validation windows",
        "All models used identical test windows",
        "All models used identical validation labels",
        "All models used identical test labels",
        "All primary test thresholds were selected using validation data",
    ],
    "primary_results": (
        final_audit_df[
            [
                "model",
                "validation_selected_threshold",
                "test_precision",
                "test_recall",
                "test_f1",
                "test_false_positive_rate",
                "test_roc_auc",
                "test_pr_auc",
            ]
        ].to_dict(orient="records")
    ),
    "conclusion": (
        "All trained models, saved predictions, evaluation metrics, "
        "decision thresholds, split assignments, and consolidated "
        "comparison artifacts passed the final integrity audit."
    ),
}

with open(
    FINAL_INTEGRITY_AUDIT_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        final_integrity_report,
        file,
        indent=2,
    )

# ------------------------------------------------------------------
# 8. Create a final completion marker
# ------------------------------------------------------------------

COMPLETION_MARKER_FILE = (
    SUPERVISED_BLUED_PATH
    / "SUPERVISED_BLUED_EXPERIMENT_COMPLETE.txt"
)

completion_marker_text = f"""
Experiment ID: {EXPERIMENT_ID}
Task: Supervised BLUED event-window classification
Status: COMPLETE
Integrity audit: PASS
Audit timestamp UTC: {final_integrity_report['audit_timestamp_utc']}

Important semantic boundary:
The positive class represents a one-minute window containing at
least one annotated appliance switching event. BLUED event
annotations are not confirmed anomaly labels.

Primary reporting protocol:
Report thresholded test metrics at validation-F1-selected thresholds.
Retain default 0.5-threshold results as supplementary results only.
""".strip()

with open(
    COMPLETION_MARKER_FILE,
    "w",
    encoding="utf-8",
) as file:
    file.write(completion_marker_text)

# ------------------------------------------------------------------
# 9. Generate the experiment artifact checksum manifest
# ------------------------------------------------------------------

ARTIFACT_MANIFEST_FILE = (
    AUDIT_PATH
    / "blued_supervised_artifact_checksum_manifest.csv"
)

artifact_files = sorted(
    [
        file_path
        for file_path in SUPERVISED_BLUED_PATH.rglob("*")
        if (
            file_path.is_file()
            and file_path != ARTIFACT_MANIFEST_FILE
        )
    ],
    key=lambda file_path: str(file_path).lower(),
)

artifact_manifest_rows = []

for file_path in artifact_files:
    relative_path = file_path.relative_to(
        SUPERVISED_BLUED_PATH
    )

    artifact_manifest_rows.append(
        {
            "relative_path": str(relative_path),
            "file_name": file_path.name,
            "file_extension": (
                file_path.suffix.lower()
                if file_path.suffix
                else "[no extension]"
            ),
            "size_bytes": int(
                file_path.stat().st_size
            ),
            "size_kilobytes": float(
                file_path.stat().st_size / 1024
            ),
            "sha256": calculate_sha256(
                file_path
            ),
        }
    )

artifact_manifest_df = pd.DataFrame(
    artifact_manifest_rows
)

artifact_manifest_df.to_csv(
    ARTIFACT_MANIFEST_FILE,
    index=False,
)

# ------------------------------------------------------------------
# 10. Verify final audit outputs
# ------------------------------------------------------------------

final_audit_outputs = [
    FINAL_INTEGRITY_AUDIT_CSV,
    FINAL_INTEGRITY_AUDIT_JSON,
    COMPLETION_MARKER_FILE,
    ARTIFACT_MANIFEST_FILE,
]

for file_path in final_audit_outputs:
    require_file(
        file_path,
        "final integrity-audit output",
    )

if artifact_manifest_df["sha256"].duplicated().any():
    duplicate_hash_count = int(
        artifact_manifest_df[
            "sha256"
        ].duplicated().sum()
    )

    print(
        f"\nNote: {duplicate_hash_count:,} duplicate file "
        "checksum(s) were detected. This can occur when separate "
        "artifacts contain identical content."
    )

print("\nSaved final audit artifacts:")

for file_path in final_audit_outputs:
    print(
        f"{file_path.name:65s} "
        f"| {file_path.stat().st_size / 1024:,.2f} KB"
    )

print(
    f"\nArtifact checksum records: "
    f"{len(artifact_manifest_df):,}"
)

# ------------------------------------------------------------------
# 11. Display the final audited result table
# ------------------------------------------------------------------

print("\nFinal audited primary supervised BLUED results:")

display(
    final_audit_df[
        [
            "test_f1_rank",
            "model",
            "validation_selected_threshold",
            "test_precision",
            "test_recall",
            "test_f1",
            "test_false_positive_rate",
            "test_roc_auc",
            "test_pr_auc",
            "test_tn",
            "test_fp",
            "test_fn",
            "test_tp",
            "maximum_model_reload_score_difference",
            "integrity_status",
        ]
    ]
)

print("\nFINAL AUDIT STATUS: PASS")
print(
    "The supervised BLUED event-window classification "
    "experiment is complete and ready for Chapter 4 reporting."
)

In [ ]:
# Step 15A — Preserve full floating-point precision in saved supervised BLUED artifacts

import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# 1. Collect all in-memory metric, threshold, and prediction tables
# ------------------------------------------------------------------

precision_sensitive_artifacts = [
    # Random Forest
    (
        random_forest_metrics_df,
        RANDOM_FOREST_METRICS_FILE,
    ),
    (
        random_forest_threshold_table,
        RANDOM_FOREST_THRESHOLD_FILE,
    ),
    (
        random_forest_validation_predictions,
        RANDOM_FOREST_VALIDATION_PREDICTIONS_FILE,
    ),
    (
        random_forest_test_predictions,
        RANDOM_FOREST_TEST_PREDICTIONS_FILE,
    ),

    # XGBoost
    (
        xgboost_metrics_df,
        XGBOOST_METRICS_FILE,
    ),
    (
        xgboost_threshold_table,
        XGBOOST_THRESHOLD_FILE,
    ),
    (
        xgboost_validation_predictions,
        XGBOOST_VALIDATION_PREDICTIONS_FILE,
    ),
    (
        xgboost_test_predictions,
        XGBOOST_TEST_PREDICTIONS_FILE,
    ),

    # Logistic Regression
    (
        logistic_metrics_df,
        LOGISTIC_METRICS_FILE,
    ),
    (
        logistic_threshold_table,
        LOGISTIC_THRESHOLD_FILE,
    ),
    (
        logistic_validation_predictions,
        LOGISTIC_VALIDATION_PREDICTIONS_FILE,
    ),
    (
        logistic_test_predictions,
        LOGISTIC_TEST_PREDICTIONS_FILE,
    ),

    # MLP
    (
        mlp_metrics_df,
        MLP_METRICS_FILE,
    ),
    (
        mlp_threshold_table,
        MLP_THRESHOLD_FILE,
    ),
    (
        mlp_validation_predictions,
        MLP_VALIDATION_PREDICTIONS_FILE,
    ),
    (
        mlp_test_predictions,
        MLP_TEST_PREDICTIONS_FILE,
    ),

    # Consolidated comparisons
    (
        combined_metrics_df,
        COMBINED_METRICS_FILE,
    ),
    (
        validation_comparison_df,
        VALIDATION_COMPARISON_FILE,
    ),
    (
        primary_test_comparison_df,
        PRIMARY_TEST_COMPARISON_FILE,
    ),
    (
        default_test_comparison_df,
        DEFAULT_TEST_COMPARISON_FILE,
    ),
    (
        threshold_sensitivity_df,
        THRESHOLD_SENSITIVITY_FILE,
    ),
]

# ------------------------------------------------------------------
# 2. Resave each table with 17 significant digits
# ------------------------------------------------------------------

print("Resaving precision-sensitive artifacts:\n")

for dataframe, file_path in precision_sensitive_artifacts:
    dataframe_to_save = dataframe.copy()

    # Promote float32 columns to float64 before serialization.
    for column_name in dataframe_to_save.columns:
        if pd.api.types.is_float_dtype(
            dataframe_to_save[column_name]
        ):
            dataframe_to_save[column_name] = (
                dataframe_to_save[column_name]
                .astype(np.float64)
            )

    dataframe_to_save.to_csv(
        file_path,
        index=False,
        float_format="%.17g",
    )

    print(
        f"{file_path.name:70s} "
        f"| {file_path.stat().st_size / 1024:,.2f} KB"
    )

# ------------------------------------------------------------------
# 3. Verify the XGBoost validation-threshold boundary
# ------------------------------------------------------------------

reloaded_xgboost_metrics = pd.read_csv(
    XGBOOST_METRICS_FILE
)

reloaded_xgboost_validation = pd.read_csv(
    XGBOOST_VALIDATION_PREDICTIONS_FILE
)

reloaded_xgboost_threshold = float(
    reloaded_xgboost_metrics.loc[
        (
            reloaded_xgboost_metrics["split"]
            == "validation"
        )
        & (
            reloaded_xgboost_metrics["threshold_source"]
            == "validation_f1_optimized"
        ),
        "threshold",
    ].iloc[0]
)

reloaded_xgboost_scores = (
    reloaded_xgboost_validation[
        "probability_score"
    ].to_numpy(dtype=np.float64)
)

saved_xgboost_predictions = (
    reloaded_xgboost_validation[
        "predicted_label_selected_threshold"
    ].to_numpy(dtype=np.int64)
)

recomputed_xgboost_predictions = (
    reloaded_xgboost_scores
    >= reloaded_xgboost_threshold
).astype(np.int64)

prediction_disagreements = int(
    np.sum(
        saved_xgboost_predictions
        != recomputed_xgboost_predictions
    )
)

minimum_threshold_distance = float(
    np.min(
        np.abs(
            reloaded_xgboost_scores
            - reloaded_xgboost_threshold
        )
    )
)

print("\nXGBoost precision-boundary verification:")
print(
    f"Reloaded selected threshold: "
    f"{reloaded_xgboost_threshold:.17g}"
)
print(
    f"Minimum score-to-threshold distance: "
    f"{minimum_threshold_distance:.17g}"
)
print(
    f"Prediction disagreements after full-precision saving: "
    f"{prediction_disagreements:,}"
)

if prediction_disagreements != 0:
    raise ValueError(
        "XGBoost predictions still differ after full-precision "
        "serialization."
    )

print(
    "\nFull-precision artifact serialization completed "
    "successfully."
)
print(
    "Rerun Cell 15 from the beginning without changing it."
)